In [1]:
# ==================================================================================================
# PROJECT 24 — CELL 1 / STEP 0
# NEW-NOTEBOOK POST-PROJECT-23 BOOTSTRAP AND CANDIDATE DISCOVERY
# Run as Cell 1 in: Thesis_project_24.ipynb
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import tarfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 24 CELL 1 / STEP 0: NEW-NOTEBOOK POST-PROJECT-23 BOOTSTRAP ===")
print("=" * 136)

PROJECT_NUMBER = 24
STEP0_STATUS = "PASS_PROJECT_24_NEW_NOTEBOOK_RUNTIME_BOOTSTRAPPED_AND_CANDIDATES_DISCOVERED"

EXPECTED_ARCHIVE_SHA256 = "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
EXPECTED_REGISTRY_SHA256 = "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
EXPECTED_PROJECT_23_STEP5C_SHA256 = "0d5fabb75eee5dc5e7c33032f2d3c37e4d237089632aa3dd7f2df1caf8b02684"
EXPECTED_PROJECT_23_STEP5C_STATUS = "PASS_PROJECT_23_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
EXPECTED_PROJECT_23_PACKAGE_ROOT_SHA256 = "7cc072977a06bca1a37b8ad8d387d89ea4b18f6a23c9070ada0df582b1ecc724"
EXPECTED_PROJECT_23_RAW_ROOT_SHA256 = "2d488eb7af1592c5b7e133c36b31d3df5153c4b02073d0b35ac4a308f01858b0"

EXPECTED_REGISTERED_PROJECTS = 23
EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_SOURCE_PROJECT_DIRECTORIES = 25
EXPECTED_CANDIDATES = 2
ACTIVE_RESERVED_PROJECTS = {}

REQUIRED_PROJECT_FILES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "id_map.csv",
    "entity_change_history.csv",
}

RUNTIME_PRIORITY_POLICY = {
    "purpose": "processing order only; protocol eligibility and final project set are unchanged",
    "primary": "ModelTrainingRows ascending",
    "secondary": "ModelEvaluationRows ascending",
    "tertiary": "RawExecutionRows ascending",
    "final_tie_break": "Project ascending",
    "scientific_effect": "none when all protocol-eligible projects are completed",
}

drive.mount("/content/drive", force_remount=False)

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
ARCHIVE_PATH = THESIS_ROOT / "Data" / "Raw" / "TCP-CI-main-dataset.tar.gz"
REGISTRY_PATH = THESIS_ROOT / "Notes" / "completed_project_registry.csv"
PROJECT_23_STEP5C_CHECKPOINT_PATH = THESIS_ROOT / "Notes" / "project_23_step5c_checkpoint.json"

LOCAL_EXTRACTION_ROOT = Path("/content/datasets")
LOCAL_DATASET_ROOT = LOCAL_EXTRACTION_ROOT / "datasets"

SELECTION_ROOT = THESIS_ROOT / "Results" / "Aggregated" / "project_24_selection"
BOOTSTRAP_INVENTORY_PATH = SELECTION_ROOT / "project_24_bootstrap_candidate_inventory.csv"
BOOTSTRAP_REPORT_PATH = SELECTION_ROOT / "project_24_step0_report.json"
BOOTSTRAP_STATUS_PATH = SELECTION_ROOT / "project_24_step0_status.json"


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_column(columns, *candidates):
    normalized = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        key = str(candidate).strip().lower()
        if key in normalized:
            return normalized[key]
    raise RuntimeError("Could not resolve any of these columns: " + ", ".join(candidates))


def atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(text, encoding="utf-8")
    temp.replace(path)


def atomic_write_json(path, payload):
    atomic_write_text(
        path,
        json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False, default=str) + "\n",
    )


def atomic_write_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temp, index=False, lineterminator="\n")
    temp.replace(path)


def extract_archive_safely(archive_path, extraction_root):
    extraction_root = Path(extraction_root)
    extraction_root.mkdir(parents=True, exist_ok=True)
    resolved_root = extraction_root.resolve()
    extracted_files = 0

    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in archive:
            member_name = member.name.replace("\\", "/").lstrip("/")
            target_path = extraction_root / member_name
            resolved_target = target_path.resolve()

            if resolved_target != resolved_root and resolved_root not in resolved_target.parents:
                raise RuntimeError(f"Unsafe archive member encountered:\n{member.name}")

            if member.isdir():
                target_path.mkdir(parents=True, exist_ok=True)
            elif member.isfile():
                target_path.parent.mkdir(parents=True, exist_ok=True)
                source_handle = archive.extractfile(member)
                if source_handle is None:
                    raise RuntimeError(f"Could not read archive member:\n{member.name}")
                with source_handle, target_path.open("wb") as output_handle:
                    shutil.copyfileobj(source_handle, output_handle, length=8 * 1024 * 1024)
                extracted_files += 1

    return extracted_files


# --------------------------------------------------------------------------------------------------
# Verify frozen archive, Project 23 final checkpoint, and completion registry.
# --------------------------------------------------------------------------------------------------

required = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    PROJECT_23_STEP5C_CHECKPOINT_PATH,
]

missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Required Project 24 bootstrap inputs are missing:\n" + "\n".join(missing)
    )

archive_sha = sha256_file(ARCHIVE_PATH)
if archive_sha != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Frozen TCP-CI archive SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\nActual:   {archive_sha}"
    )

project_23_step5c_sha = sha256_file(PROJECT_23_STEP5C_CHECKPOINT_PATH)
if project_23_step5c_sha != EXPECTED_PROJECT_23_STEP5C_SHA256:
    raise RuntimeError(
        "Project 23 Step 5C checkpoint SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_PROJECT_23_STEP5C_SHA256}\nActual:   {project_23_step5c_sha}"
    )

project_23_step5c = json.loads(
    PROJECT_23_STEP5C_CHECKPOINT_PATH.read_text(encoding="utf-8")
)

if project_23_step5c.get("Status") != EXPECTED_PROJECT_23_STEP5C_STATUS:
    raise RuntimeError("Project 23 Step 5C checkpoint is not in the expected PASS state.")

if not bool(project_23_step5c.get("ProjectCompleteAndFrozen", False)):
    raise RuntimeError("Project 23 is not marked complete and frozen.")

if not bool(project_23_step5c.get("CompletionRegistryUpdated", False)):
    raise RuntimeError("Project 23 checkpoint is not marked registry-updated.")

if str(project_23_step5c.get("FinalPackageRootSHA256", "")) != EXPECTED_PROJECT_23_PACKAGE_ROOT_SHA256:
    raise RuntimeError("Project 23 final-package root differs from the frozen result.")

if str(project_23_step5c.get("RawRootSHA256", "")) != EXPECTED_PROJECT_23_RAW_ROOT_SHA256:
    raise RuntimeError("Project 23 raw-root differs from the frozen result.")

registry_sha_before = sha256_file(REGISTRY_PATH)
if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry differs from the frozen post-Project-23 state.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\nActual:   {registry_sha_before}"
    )

registry = pd.read_csv(REGISTRY_PATH, low_memory=False)
number_col = resolve_column(registry.columns, "ProjectNumber", "Project Number", "Project_Number")
project_col = resolve_column(registry.columns, "Project")
status_col = resolve_column(registry.columns, "Status")

numbers = pd.to_numeric(registry[number_col], errors="raise").astype(int)

if (
    len(registry) != EXPECTED_REGISTERED_PROJECTS
    or sorted(numbers.tolist()) != list(range(1, EXPECTED_REGISTERED_PROJECTS + 1))
):
    raise RuntimeError("Completion registry must contain exactly frozen Projects 1–23.")

if not registry[status_col].astype(str).eq(EXPECTED_COMPLETE_STATUS).all():
    raise RuntimeError("Not every registered predecessor is COMPLETE_AND_FROZEN.")

if numbers.eq(PROJECT_NUMBER).any():
    raise RuntimeError("Project 24 is unexpectedly already registered.")

if ACTIVE_RESERVED_PROJECTS:
    raise RuntimeError("Project 24 bootstrap expects no active project reservations.")

EXPECTED_PREDECESSOR_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}

for predecessor_number, expected_project in EXPECTED_PREDECESSOR_IDENTITIES.items():
    matches = registry.loc[
        numbers.eq(predecessor_number),
        project_col,
    ].astype(str).tolist()

    if matches != [expected_project]:
        raise RuntimeError(
            f"Frozen Project {predecessor_number} identity mismatch.\n"
            f"Expected: {expected_project}\nActual:   {matches}"
        )

registered_projects = set(registry[project_col].astype(str))


# --------------------------------------------------------------------------------------------------
# Restore the source archive into this fresh runtime if needed.
# --------------------------------------------------------------------------------------------------

def local_dataset_looks_complete():
    if not LOCAL_DATASET_ROOT.is_dir():
        return False
    dirs = [path for path in LOCAL_DATASET_ROOT.iterdir() if path.is_dir()]
    return len(dirs) == EXPECTED_SOURCE_PROJECT_DIRECTORIES


if local_dataset_looks_complete():
    extraction_performed = False
    extracted_files = 0
    print("\nA complete-looking local TCP-CI dataset is already present.")
else:
    extraction_performed = True
    print("\nRestoring the frozen TCP-CI archive into the Project 24 runtime.")
    if LOCAL_EXTRACTION_ROOT.exists():
        shutil.rmtree(LOCAL_EXTRACTION_ROOT)
    extracted_files = extract_archive_safely(ARCHIVE_PATH, LOCAL_EXTRACTION_ROOT)

if not LOCAL_DATASET_ROOT.is_dir():
    raise RuntimeError(
        "Archive extraction did not create the expected dataset root:\n"
        f"{LOCAL_DATASET_ROOT}"
    )

all_project_directories = sorted(
    [path for path in LOCAL_DATASET_ROOT.iterdir() if path.is_dir()],
    key=lambda path: path.name,
)

if len(all_project_directories) != EXPECTED_SOURCE_PROJECT_DIRECTORIES:
    raise RuntimeError(
        "Unexpected number of TCP-CI project directories.\n"
        f"Expected: {EXPECTED_SOURCE_PROJECT_DIRECTORIES}\n"
        f"Actual:   {len(all_project_directories)}"
    )


# --------------------------------------------------------------------------------------------------
# Discover all unregistered candidates without assuming their identities.
# --------------------------------------------------------------------------------------------------

reserved_projects = set(ACTIVE_RESERVED_PROJECTS.values())
rows = []

for source_directory in all_project_directories:
    project = source_directory.name
    source_files = {path.name for path in source_directory.iterdir() if path.is_file()}
    missing_required_files = sorted(REQUIRED_PROJECT_FILES - source_files)

    excluded_registered = project in registered_projects
    excluded_reserved = project in reserved_projects

    candidate_for_scan = (
        not excluded_registered
        and not excluded_reserved
        and not missing_required_files
    )

    rows.append({
        "Project": project,
        "ProjectSlug": project.replace("@", "__"),
        "SourceDirectory": str(source_directory),
        "ExcludedRegistered": bool(excluded_registered),
        "ExcludedReserved": bool(excluded_reserved),
        "MissingRequiredFiles": "; ".join(missing_required_files),
        "CandidateForProject24Scan": bool(candidate_for_scan),
    })

inventory = pd.DataFrame(rows)

project_24_candidates = (
    inventory.loc[inventory["CandidateForProject24Scan"]]
    .sort_values("Project", kind="mergesort")
    .reset_index(drop=True)
)

if len(project_24_candidates) != EXPECTED_CANDIDATES:
    raise RuntimeError(
        "Unexpected Project 24 candidate count after excluding frozen Projects 1–23.\n"
        f"Expected: {EXPECTED_CANDIDATES}\nActual:   {len(project_24_candidates)}"
    )

if project_24_candidates["Project"].duplicated(keep=False).any():
    raise RuntimeError("Project 24 bootstrap candidate inventory contains duplicate projects.")

if project_24_candidates["Project"].isin(registered_projects | reserved_projects).any():
    raise RuntimeError("A registered/reserved project leaked into the Project 24 candidate set.")


# --------------------------------------------------------------------------------------------------
# Freeze bootstrap inventory/report. Registry remains read-only.
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(parents=True, exist_ok=True)
atomic_write_csv(BOOTSTRAP_INVENTORY_PATH, project_24_candidates)

created_at_utc = datetime.now(timezone.utc).isoformat()

report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Status": STEP0_STATUS,
    "CreatedAtUTC": created_at_utc,
    "ArchivePath": str(ARCHIVE_PATH),
    "ArchiveSHA256": archive_sha,
    "RegistryPath": str(REGISTRY_PATH),
    "RegistrySHA256": registry_sha_before,
    "Project23Step5CCheckpoint": str(PROJECT_23_STEP5C_CHECKPOINT_PATH),
    "Project23Step5CCheckpointSHA256": project_23_step5c_sha,
    "Project23FinalPackageRootSHA256": EXPECTED_PROJECT_23_PACKAGE_ROOT_SHA256,
    "Project23RawRootSHA256": EXPECTED_PROJECT_23_RAW_ROOT_SHA256,
    "RegisteredProjects": EXPECTED_REGISTERED_PROJECTS,
    "RegisteredStatuses": sorted(registry[status_col].astype(str).unique().tolist()),
    "ActiveReservations": {},
    "FrozenPredecessorIdentities": {
        str(key): value for key, value in EXPECTED_PREDECESSOR_IDENTITIES.items()
    },
    "DatasetRoot": str(LOCAL_DATASET_ROOT),
    "SourceProjectDirectories": len(all_project_directories),
    "Project24CandidateCount": len(project_24_candidates),
    "CandidateInventory": str(BOOTSTRAP_INVENTORY_PATH),
    "RuntimePriorityPolicy": RUNTIME_PRIORITY_POLICY,
    "ExtractionPerformed": bool(extraction_performed),
    "ArchiveFilesExtracted": int(extracted_files),
    "RegistryModified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "NoiseInjected": False,
    "ModelsFitted": False,
}

atomic_write_json(BOOTSTRAP_REPORT_PATH, report)

atomic_write_json(
    BOOTSTRAP_STATUS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Status": STEP0_STATUS,
        "CreatedAtUTC": created_at_utc,
        "Report": str(BOOTSTRAP_REPORT_PATH),
    },
)

registry_sha_after = sha256_file(REGISTRY_PATH)
if registry_sha_after != registry_sha_before:
    raise RuntimeError("Completion registry changed during Project 24 bootstrap.")


print("\nProject 24 candidates after excluding registered identities:")
print(
    project_24_candidates[
        ["Project", "ProjectSlug", "SourceDirectory"]
    ].to_string(index=False)
)

print("\n" + "=" * 136)
print("=== PROJECT 24 CELL 1 / STEP 0 RESULT ===")
print("=" * 136)
print("Registered and frozen projects:", EXPECTED_REGISTERED_PROJECTS)
print("Active reservations:", [])
print("TCP-CI source directories:", len(all_project_directories))
print("Project 24 candidates:", len(project_24_candidates))
print("Project 23 Step 5C checkpoint SHA-256:", project_23_step5c_sha)
print("Project 23 final package root SHA-256:", EXPECTED_PROJECT_23_PACKAGE_ROOT_SHA256)
print("Registry SHA-256:", registry_sha_before)
print("Runtime-priority policy:", RUNTIME_PRIORITY_POLICY)
print("Candidate inventory:", BOOTSTRAP_INVENTORY_PATH)
print("Completion registry modified:", False)
print("Prior project condition outputs accessed:", False)
print("Models fitted:", False)
print(
    "\nNext required step:",
    "PROJECT 24 CELL 2 / STEP 1A — CANDIDATE ELIGIBILITY AND RUNTIME-PRIORITIZED RANKING",
)
print("STATUS:", STEP0_STATUS)
print("=" * 136)


=== PROJECT 24 CELL 1 / STEP 0: NEW-NOTEBOOK POST-PROJECT-23 BOOTSTRAP ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Restoring the frozen TCP-CI archive into the Project 24 runtime.

Project 24 candidates after excluding registered identities:
                 Project               ProjectSlug                                     SourceDirectory
Graylog2@graylog2-server Graylog2__graylog2-server /content/datasets/datasets/Graylog2@graylog2-server
   SonarSource@sonarqube    SonarSource__sonarqube    /content/datasets/datasets/SonarSource@sonarqube

=== PROJECT 24 CELL 1 / STEP 0 RESULT ===
Registered and frozen projects: 23
Active reservations: []
TCP-CI source directories: 25
Project 24 candidates: 2
Project 23 Step 5C checkpoint SHA-256: 0d5fabb75eee5dc5e7c33032f2d3c37e4d237089632aa3dd7f2df1caf8b02684
Project 23 final package root SHA-256: 7cc072977a06bca1a37b8ad8d387d89ea4b18f6a23c9070ada0df582b1e

In [2]:
# ==================================================================================================
# PROJECT 24 — CELL 2 / STEP 1A
# ROBUST CANDIDATE ELIGIBILITY, RUNTIME-PRIORITIZED RANKING,
# AND PROVISIONAL PROJECT 24 SELECTION
#
# RUN THIS AS CELL 2 IN Thesis_project_24.ipynb AFTER CELL 1 / STEP 0 PASSES.
#
# THIS CELL:
# - inspects the 2 candidates frozen by Project 24 Step 0;
# - applies the frozen chronological 75/25 split;
# - validates strict protocol eligibility:
#       * >=10 failing training builds in the raw execution cohort;
#       * non-empty raw and model evaluation failures;
#       * both model-training classes present;
#       * non-empty model training/evaluation cohorts;
#       * zero unlinked raw/model rows;
# - deterministically ranks eligible candidates by runtime priority:
#       ModelTrainingRows ascending,
#       ModelEvaluationRows ascending,
#       RawExecutionRows ascending,
#       Project ascending;
# - freezes only a PROVISIONAL Project 24 selection for Step 1B;
# - does not freeze source files yet;
# - does not fit models, inject noise, reconstruct REC, execute experiment conditions,
#   access prior-project condition outputs, or modify the completion registry.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os
import time

import numpy as np
import pandas as pd


print("=" * 136)
print("=== PROJECT 24 CELL 2 / STEP 1A: CANDIDATE ELIGIBILITY AND RUNTIME-PRIORITIZED RANKING ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24

BOOTSTRAP_PASS_STATUS = (
    "PASS_PROJECT_24_NEW_NOTEBOOK_RUNTIME_BOOTSTRAPPED_AND_CANDIDATES_DISCOVERED"
)

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_24_CANDIDATE_DISCOVERY_COMPLETE"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 23
EXPECTED_CANDIDATES = 2

EXPECTED_CANDIDATE_SET = {
    "Graylog2@graylog2-server",
    "SonarSource@sonarqube",
}

RESERVED_ACTIVE_PROJECTS = set()

RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

MIN_FAILING_TRAINING_BUILDS = 10

# These files are sufficient for deterministic selection.
# id_map.csv and entity_change_history.csv are checked/frozen in Step 1B / Step 2A.
REQUIRED_SELECTION_FILES = [
    "builds.csv",
    "exe.csv",
    "dataset.csv",
]


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

LOCAL_SOURCE_ROOT = Path(
    "/content/datasets/datasets"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_24_selection"
)

BOOTSTRAP_STATUS_PATH = (
    SELECTION_ROOT
    / "project_24_step0_status.json"
)

BOOTSTRAP_REPORT_PATH = (
    SELECTION_ROOT
    / "project_24_step0_report.json"
)

BOOTSTRAP_CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_24_bootstrap_candidate_inventory.csv"
)

SCAN_PROGRESS_PATH = (
    SELECTION_ROOT
    / "project_24_candidate_scan_progress.csv"
)

SOURCE_SCHEMA_AUDIT_PATH = (
    SELECTION_ROOT
    / "project_24_source_schema_audit.csv"
)

CANDIDATE_INVENTORY_PATH = (
    SELECTION_ROOT
    / "project_24_candidate_inventory.csv"
)

ELIGIBLE_RANKED_PATH = (
    SELECTION_ROOT
    / "project_24_eligible_candidates_ranked.csv"
)

INELIGIBLE_PATH = (
    SELECTION_ROOT
    / "project_24_ineligible_candidates.csv"
)

INSPECTION_ERRORS_PATH = (
    SELECTION_ROOT
    / "project_24_candidate_inspection_errors.csv"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_24_provisional_selection.json"
)

STEP1A_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_24_step1a_validation.csv"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_24_step1a_report.json"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_24_step1a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_write_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(columns, expected, label):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_series(values, label):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(dtype=float)

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def project_slug(project_name):
    return str(project_name).replace(
        "@",
        "__",
        1,
    )


def count_partitioned_rows(
    csv_path,
    build_column,
    verdict_column,
    training_build_ids,
    evaluation_build_ids,
    label,
    chunksize,
):
    total_rows = 0
    training_rows = 0
    evaluation_rows = 0

    training_failures = 0
    evaluation_failures = 0

    failing_training_builds = set()
    failing_evaluation_builds = set()

    unlinked_rows = 0
    verdict_values = set()
    training_verdict_values = set()
    evaluation_verdict_values = set()

    for chunk in pd.read_csv(
        csv_path,
        usecols=[
            build_column,
            verdict_column,
        ],
        chunksize=chunksize,
        low_memory=False,
    ):
        chunk_build = parse_integer_series(
            chunk[build_column],
            f"{label}.{build_column}",
        )

        chunk_verdict = parse_integer_series(
            chunk[verdict_column],
            f"{label}.{verdict_column}",
        )

        training_mask = chunk_build.isin(
            training_build_ids
        )

        evaluation_mask = chunk_build.isin(
            evaluation_build_ids
        )

        linked_mask = (
            training_mask
            | evaluation_mask
        )

        failure_mask = chunk_verdict.ne(0)

        total_rows += len(chunk)

        training_rows += int(
            training_mask.sum()
        )

        evaluation_rows += int(
            evaluation_mask.sum()
        )

        training_failures += int(
            (
                training_mask
                & failure_mask
            ).sum()
        )

        evaluation_failures += int(
            (
                evaluation_mask
                & failure_mask
            ).sum()
        )

        failing_training_builds.update(
            chunk_build.loc[
                training_mask
                & failure_mask
            ].astype(int).tolist()
        )

        failing_evaluation_builds.update(
            chunk_build.loc[
                evaluation_mask
                & failure_mask
            ].astype(int).tolist()
        )

        unlinked_rows += int(
            (~linked_mask).sum()
        )

        verdict_values.update(
            int(value)
            for value in chunk_verdict.unique().tolist()
        )

        training_verdict_values.update(
            int(value)
            for value in chunk_verdict.loc[
                training_mask
            ].unique().tolist()
        )

        evaluation_verdict_values.update(
            int(value)
            for value in chunk_verdict.loc[
                evaluation_mask
            ].unique().tolist()
        )

    training_passes = int(
        training_rows
        - training_failures
    )

    evaluation_passes = int(
        evaluation_rows
        - evaluation_failures
    )

    return {
        "Rows":
            int(total_rows),

        "TrainingRows":
            int(training_rows),

        "EvaluationRows":
            int(evaluation_rows),

        "TrainingPasses":
            training_passes,

        "EvaluationPasses":
            evaluation_passes,

        "TrainingFailures":
            int(training_failures),

        "EvaluationFailures":
            int(evaluation_failures),

        "FailingTrainingBuilds":
            int(
                len(
                    failing_training_builds
                )
            ),

        "FailingEvaluationBuilds":
            int(
                len(
                    failing_evaluation_builds
                )
            ),

        "UnlinkedRows":
            int(unlinked_rows),

        "VerdictValuesJSON":
            json.dumps(
                sorted(
                    verdict_values
                )
            ),

        "TrainingVerdictValuesJSON":
            json.dumps(
                sorted(
                    training_verdict_values
                )
            ),

        "EvaluationVerdictValuesJSON":
            json.dumps(
                sorted(
                    evaluation_verdict_values
                )
            ),
    }


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def reusable_scan_row_is_valid(row):
    required_fields = [
        "Project",
        "ProjectSlug",
        "SourceDirectory",
        "InspectionStatus",
        "InspectionError",
        "BuildIDColumn",
        "StartedAtColumn",
        "ExecutionBuildColumn",
        "ExecutionVerdictColumn",
        "DatasetBuildColumn",
        "DatasetVerdictColumn",
        "Builds",
        "TrainingBuilds",
        "EvaluationBuilds",
        "RawExecutionRows",
        "RawTrainingRows",
        "RawEvaluationRows",
        "RawTrainingPasses",
        "RawEvaluationPasses",
        "RawTrainFailures",
        "RawEvaluationFailures",
        "RawFailingTrainingBuilds",
        "RawFailingEvaluationBuilds",
        "RawUnlinkedRows",
        "ModelReadyRows",
        "ModelTrainingRows",
        "ModelEvaluationRows",
        "ModelTrainingPasses",
        "ModelEvaluationPasses",
        "ModelTrainFailures",
        "ModelEvaluationFailures",
        "ModelFailingTrainingBuilds",
        "ModelFailingEvaluationBuilds",
        "ModelUnlinkedRows",
        "ModelTrainingBothClasses",
        "ProtocolEligible",
    ]

    if any(
        field not in row
        for field in required_fields
    ):
        return False

    status = str(
        row.get(
            "InspectionStatus",
            "",
        )
    ).strip()

    error = str(
        row.get(
            "InspectionError",
            "",
        )
    ).strip().lower()

    return (
        status in {
            "ELIGIBLE",
            "INELIGIBLE",
        }
        and error in {
            "",
            "nan",
            "none",
        }
    )


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE STEP 0, ARCHIVE, REGISTRY, AND CANDIDATE INVENTORY
# --------------------------------------------------------------------------------------------------

required_inputs = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    BOOTSTRAP_STATUS_PATH,
    BOOTSTRAP_REPORT_PATH,
    BOOTSTRAP_CANDIDATE_INVENTORY_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 24 Step 1A inputs are missing:\n"
        + "\n".join(missing_inputs)
    )

if not LOCAL_SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        "The local Project 24 dataset source is missing:\n"
        f"{LOCAL_SOURCE_ROOT}"
    )

bootstrap_status = load_json(
    BOOTSTRAP_STATUS_PATH
)

bootstrap_report = load_json(
    BOOTSTRAP_REPORT_PATH
)

if bootstrap_status.get(
    "Status"
) != BOOTSTRAP_PASS_STATUS:
    raise RuntimeError(
        "Project 24 Step 0 status is not in the expected PASS state."
    )

if bootstrap_report.get(
    "Status"
) != BOOTSTRAP_PASS_STATUS:
    raise RuntimeError(
        "Project 24 Step 0 report is not in the expected PASS state."
    )

if int(
    bootstrap_report.get(
        "Project24CandidateCount",
        -1,
    )
) != EXPECTED_CANDIDATES:
    raise RuntimeError(
        "Project 24 Step 0 candidate count differs."
    )

if bootstrap_report.get(
    "ActiveReservations",
    None,
) != {}:
    raise RuntimeError(
        "Project 24 Step 0 active-reservation state differs."
    )

if bool(
    bootstrap_report.get(
        "RegistryModified",
        True,
    )
):
    raise RuntimeError(
        "Project 24 Step 0 reports registry modification."
    )

archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Dataset archive SHA-256 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )

registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)

registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)

if (
    len(registry)
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–23."
    )

if not registry[
    registry_status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )

if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 24 is unexpectedly already registered."
    )

registered_projects = set(
    registry[
        registry_project_column
    ].astype(str).tolist()
)

if registered_projects & RESERVED_ACTIVE_PROJECTS:
    raise RuntimeError(
        "A reserved active-project identity is unexpectedly present "
        "in the completion registry."
    )

required_registered_identities = {
    11:
        "apache@shardingsphere",

    12:
        "zolyfarkas@spf4j",

    13:
        "jcabi@jcabi-github",

    14:
        "JMRI@JMRI",

    15:
        "eclipse@steady",

    16:
        "apache@rocketmq",

    17:
        "yamcs@Yamcs",

    18:
        "cantaloupe-project@cantaloupe",

    19:
        "EMResearch@EvoMaster",

    20:
        "apache@curator",

    21:
        "facebook@buck",

    22:
        "apache@logging-log4j2",

    23:
        "apache@sling",
}

for required_number, required_project in required_registered_identities.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(matching_rows) != 1
        or matching_rows.iloc[
            0
        ][
            registry_project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

bootstrap_candidates = pd.read_csv(
    BOOTSTRAP_CANDIDATE_INVENTORY_PATH,
    low_memory=False,
)

candidate_project_column = resolve_column(
    bootstrap_candidates.columns,
    "Project",
    "bootstrap candidate Project",
)

candidate_source_column = resolve_column(
    bootstrap_candidates.columns,
    "SourceDirectory",
    "bootstrap candidate SourceDirectory",
)

candidate_records = (
    bootstrap_candidates[
        [
            candidate_project_column,
            candidate_source_column,
        ]
    ]
    .rename(
        columns={
            candidate_project_column:
                "Project",

            candidate_source_column:
                "SourceDirectory",
        }
    )
    .copy()
)

candidate_records[
    "Project"
] = candidate_records[
    "Project"
].astype(str)

candidate_records[
    "SourceDirectory"
] = candidate_records[
    "SourceDirectory"
].astype(str)

if len(
    candidate_records
) != EXPECTED_CANDIDATES:
    raise RuntimeError(
        "Unexpected Project 24 candidate count.\n"
        f"Expected: {EXPECTED_CANDIDATES}\n"
        f"Actual:   {len(candidate_records)}"
    )

if candidate_records[
    "Project"
].duplicated(
    keep=False
).any():
    raise RuntimeError(
        "Project 24 bootstrap candidate inventory contains duplicates."
    )

actual_candidate_set = set(
    candidate_records[
        "Project"
    ].astype(str)
)

if actual_candidate_set != EXPECTED_CANDIDATE_SET:
    raise RuntimeError(
        "Project 24 candidate identity set differs.\n"
        f"Expected: {sorted(EXPECTED_CANDIDATE_SET)}\n"
        f"Actual:   {sorted(actual_candidate_set)}"
    )

forbidden_candidates = (
    actual_candidate_set
    & (
        registered_projects
        | RESERVED_ACTIVE_PROJECTS
    )
)

if forbidden_candidates:
    raise RuntimeError(
        "Project 24 inventory contains registered/reserved projects:\n"
        + "\n".join(
            sorted(
                forbidden_candidates
            )
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. REUSE ANY VALID COMPLETED PROJECT 24 SCANS
# --------------------------------------------------------------------------------------------------

SELECTION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

reusable_rows = {}

if SCAN_PROGRESS_PATH.is_file():
    try:
        previous_progress = pd.read_csv(
            SCAN_PROGRESS_PATH,
            low_memory=False,
        )

        valid_candidate_names = set(
            candidate_records[
                "Project"
            ]
        )

        for row in previous_progress.to_dict(
            orient="records"
        ):
            project = str(
                row.get(
                    "Project",
                    "",
                )
            )

            if (
                project
                in valid_candidate_names
                and reusable_scan_row_is_valid(
                    row
                )
            ):
                reusable_rows[
                    project
                ] = row

        print(
            "\nReusable completed candidate scans:",
            len(
                reusable_rows
            ),
        )

    except Exception as error:
        print(
            "\nPrevious scan progress was ignored:",
            type(
                error
            ).__name__,
            str(
                error
            ),
        )


# --------------------------------------------------------------------------------------------------
# 6. INSPECT ALL 2 CANDIDATES
# --------------------------------------------------------------------------------------------------

scan_rows = []

for candidate_index, candidate in enumerate(
    candidate_records.itertuples(
        index=False
    ),
    start=1,
):
    project = str(
        candidate.Project
    )

    source_directory = Path(
        candidate.SourceDirectory
    )

    print(
        "-" * 136
    )

    print(
        f"[{candidate_index:02d}/{EXPECTED_CANDIDATES:02d}] "
        f"Inspecting: {project}"
    )

    if project in reusable_rows:
        row = dict(
            reusable_rows[
                project
            ]
        )

        row[
            "CandidateInspectionOrder"
        ] = candidate_index

        row[
            "InspectionError"
        ] = ""

        scan_rows.append(
            row
        )

        print(
            "    Reused:",
            row[
                "InspectionStatus"
            ],
            "| Builds:",
            int(
                row[
                    "Builds"
                ]
            ),
            "| Model training rows:",
            int(
                row[
                    "ModelTrainingRows"
                ]
            ),
            "| Model eval failures:",
            int(
                row[
                    "ModelEvaluationFailures"
                ]
            ),
        )

        continue

    started = time.perf_counter()

    row = {
        "CandidateInspectionOrder":
            candidate_index,

        "Project":
            project,

        "ProjectSlug":
            project_slug(
                project
            ),

        "SourceDirectory":
            str(
                source_directory
            ),

        "InspectionStatus":
            "ERROR",

        "InspectionError":
            "",
    }

    try:
        missing_files = [
            filename
            for filename
            in REQUIRED_SELECTION_FILES
            if not (
                source_directory
                / filename
            ).is_file()
        ]

        if missing_files:
            raise FileNotFoundError(
                "Missing selection files: "
                + ", ".join(
                    missing_files
                )
            )

        builds_path = (
            source_directory
            / "builds.csv"
        )

        exe_path = (
            source_directory
            / "exe.csv"
        )

        dataset_path = (
            source_directory
            / "dataset.csv"
        )

        build_columns = pd.read_csv(
            builds_path,
            nrows=0,
        ).columns.tolist()

        exe_columns = pd.read_csv(
            exe_path,
            nrows=0,
        ).columns.tolist()

        dataset_columns = pd.read_csv(
            dataset_path,
            nrows=0,
        ).columns.tolist()

        build_id_column = resolve_column(
            build_columns,
            "id",
            f"{project} builds.csv ID",
        )

        started_at_column = resolve_column(
            build_columns,
            "started_at",
            f"{project} builds.csv started_at",
        )

        execution_build_column = resolve_column(
            exe_columns,
            "build",
            f"{project} exe.csv build",
        )

        execution_verdict_column = resolve_column(
            exe_columns,
            "verdict",
            f"{project} exe.csv verdict",
        )

        dataset_build_column = resolve_column(
            dataset_columns,
            "Build",
            f"{project} dataset.csv Build",
        )

        dataset_verdict_column = resolve_column(
            dataset_columns,
            "Verdict",
            f"{project} dataset.csv Verdict",
        )

        builds = pd.read_csv(
            builds_path,
            usecols=[
                build_id_column,
                started_at_column,
            ],
            low_memory=False,
        )

        builds[
            build_id_column
        ] = parse_integer_series(
            builds[
                build_id_column
            ],
            f"{project}.builds.id",
        )

        builds[
            started_at_column
        ] = pd.to_datetime(
            builds[
                started_at_column
            ],
            errors="coerce",
            utc=True,
        )

        invalid_timestamps = int(
            builds[
                started_at_column
            ].isna().sum()
        )

        duplicate_build_id_rows = int(
            builds[
                build_id_column
            ].duplicated(
                keep=False
            ).sum()
        )

        timestamp_tie_groups = int(
            (
                builds.groupby(
                    started_at_column
                ).size()
                > 1
            ).sum()
        )

        timestamp_tied_builds = int(
            builds.loc[
                builds[
                    started_at_column
                ].duplicated(
                    keep=False
                )
            ].shape[
                0
            ]
        )

        if invalid_timestamps != 0:
            raise RuntimeError(
                f"Invalid build timestamps: {invalid_timestamps}"
            )

        if duplicate_build_id_rows != 0:
            raise RuntimeError(
                f"Duplicate build-ID rows: {duplicate_build_id_rows}"
            )

        # Frozen chronology contract:
        # timestamp ascending; within an exact timestamp tie, build ID descending.
        ordered_builds = (
            builds.sort_values(
                [
                    started_at_column,
                    build_id_column,
                ],
                ascending=[
                    True,
                    False,
                ],
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        number_of_builds = int(
            len(
                ordered_builds
            )
        )

        training_build_count = int(
            math.floor(
                0.75
                * number_of_builds
            )
        )

        evaluation_build_count = int(
            number_of_builds
            - training_build_count
        )

        if (
            training_build_count <= 0
            or evaluation_build_count <= 0
        ):
            raise RuntimeError(
                "Chronological 75/25 split has an empty partition."
            )

        training_build_ids = set(
            ordered_builds.iloc[
                :training_build_count
            ][
                build_id_column
            ].astype(
                int
            ).tolist()
        )

        evaluation_build_ids = set(
            ordered_builds.iloc[
                training_build_count:
            ][
                build_id_column
            ].astype(
                int
            ).tolist()
        )

        if (
            training_build_ids
            & evaluation_build_ids
        ):
            raise RuntimeError(
                "Training/evaluation build partitions overlap."
            )

        raw_profile = count_partitioned_rows(
            csv_path=exe_path,
            build_column=execution_build_column,
            verdict_column=execution_verdict_column,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project}.exe",
            chunksize=500_000,
        )

        model_profile = count_partitioned_rows(
            csv_path=dataset_path,
            build_column=dataset_build_column,
            verdict_column=dataset_verdict_column,
            training_build_ids=training_build_ids,
            evaluation_build_ids=evaluation_build_ids,
            label=f"{project}.dataset",
            chunksize=250_000,
        )

        model_training_both_classes = bool(
            model_profile[
                "TrainingFailures"
            ] > 0
            and model_profile[
                "TrainingPasses"
            ] > 0
        )

        eligibility_reasons = []

        eligibility_tests = [
            (
                raw_profile[
                    "TrainingRows"
                ] > 0,
                "No raw training rows",
            ),
            (
                raw_profile[
                    "EvaluationRows"
                ] > 0,
                "No raw evaluation rows",
            ),
            (
                raw_profile[
                    "FailingTrainingBuilds"
                ]
                >= MIN_FAILING_TRAINING_BUILDS,
                (
                    "Raw failing training builds "
                    f"< {MIN_FAILING_TRAINING_BUILDS}"
                ),
            ),
            (
                raw_profile[
                    "EvaluationFailures"
                ] > 0,
                "No raw evaluation failures",
            ),
            (
                model_profile[
                    "TrainingRows"
                ] > 0,
                "No model training rows",
            ),
            (
                model_profile[
                    "EvaluationRows"
                ] > 0,
                "No model evaluation rows",
            ),
            (
                model_training_both_classes,
                "Model training does not contain both pass/fail classes",
            ),
            (
                model_profile[
                    "EvaluationFailures"
                ] > 0,
                "No model evaluation failures",
            ),
            (
                raw_profile[
                    "UnlinkedRows"
                ] == 0,
                "Raw rows reference unknown builds",
            ),
            (
                model_profile[
                    "UnlinkedRows"
                ] == 0,
                "Model rows reference unknown builds",
            ),
        ]

        for (
            passed,
            failure_reason,
        ) in eligibility_tests:
            if not passed:
                eligibility_reasons.append(
                    failure_reason
                )

        protocol_eligible = bool(
            len(
                eligibility_reasons
            )
            == 0
        )

        row.update({
            "BuildIDColumn":
                build_id_column,

            "StartedAtColumn":
                started_at_column,

            "ExecutionBuildColumn":
                execution_build_column,

            "ExecutionVerdictColumn":
                execution_verdict_column,

            "DatasetBuildColumn":
                dataset_build_column,

            "DatasetVerdictColumn":
                dataset_verdict_column,

            "Builds":
                number_of_builds,

            "TrainingBuilds":
                training_build_count,

            "EvaluationBuilds":
                evaluation_build_count,

            "TimestampTieGroups":
                timestamp_tie_groups,

            "TimestampTiedBuilds":
                timestamp_tied_builds,

            "RawExecutionRows":
                raw_profile[
                    "Rows"
                ],

            "RawTrainingRows":
                raw_profile[
                    "TrainingRows"
                ],

            "RawEvaluationRows":
                raw_profile[
                    "EvaluationRows"
                ],

            "RawTrainingPasses":
                raw_profile[
                    "TrainingPasses"
                ],

            "RawEvaluationPasses":
                raw_profile[
                    "EvaluationPasses"
                ],

            "RawTrainFailures":
                raw_profile[
                    "TrainingFailures"
                ],

            "RawEvaluationFailures":
                raw_profile[
                    "EvaluationFailures"
                ],

            "RawFailingTrainingBuilds":
                raw_profile[
                    "FailingTrainingBuilds"
                ],

            "RawFailingEvaluationBuilds":
                raw_profile[
                    "FailingEvaluationBuilds"
                ],

            "RawUnlinkedRows":
                raw_profile[
                    "UnlinkedRows"
                ],

            "RawVerdictValuesJSON":
                raw_profile[
                    "VerdictValuesJSON"
                ],

            "RawTrainingVerdictValuesJSON":
                raw_profile[
                    "TrainingVerdictValuesJSON"
                ],

            "RawEvaluationVerdictValuesJSON":
                raw_profile[
                    "EvaluationVerdictValuesJSON"
                ],

            "ModelReadyRows":
                model_profile[
                    "Rows"
                ],

            "ModelTrainingRows":
                model_profile[
                    "TrainingRows"
                ],

            "ModelEvaluationRows":
                model_profile[
                    "EvaluationRows"
                ],

            "ModelTrainingPasses":
                model_profile[
                    "TrainingPasses"
                ],

            "ModelEvaluationPasses":
                model_profile[
                    "EvaluationPasses"
                ],

            "ModelTrainFailures":
                model_profile[
                    "TrainingFailures"
                ],

            "ModelEvaluationFailures":
                model_profile[
                    "EvaluationFailures"
                ],

            "ModelFailingTrainingBuilds":
                model_profile[
                    "FailingTrainingBuilds"
                ],

            "ModelFailingEvaluationBuilds":
                model_profile[
                    "FailingEvaluationBuilds"
                ],

            "ModelUnlinkedRows":
                model_profile[
                    "UnlinkedRows"
                ],

            "ModelVerdictValuesJSON":
                model_profile[
                    "VerdictValuesJSON"
                ],

            "ModelTrainingVerdictValuesJSON":
                model_profile[
                    "TrainingVerdictValuesJSON"
                ],

            "ModelEvaluationVerdictValuesJSON":
                model_profile[
                    "EvaluationVerdictValuesJSON"
                ],

            "ModelTrainingBothClasses":
                model_training_both_classes,

            "MinimumFailingTrainingBuilds":
                MIN_FAILING_TRAINING_BUILDS,

            "ProtocolEligible":
                protocol_eligible,

            "EligibilityReason":
                (
                    ""
                    if protocol_eligible
                    else "; ".join(
                        eligibility_reasons
                    )
                ),

            "InspectionStatus":
                (
                    "ELIGIBLE"
                    if protocol_eligible
                    else "INELIGIBLE"
                ),

            "InspectionError":
                "",
        })

        print(
            "    Status:",
            row[
                "InspectionStatus"
            ],
            "| Builds:",
            number_of_builds,
            "| Raw failing train builds:",
            raw_profile[
                "FailingTrainingBuilds"
            ],
            "| Model train rows:",
            model_profile[
                "TrainingRows"
            ],
            "| Model eval rows:",
            model_profile[
                "EvaluationRows"
            ],
            "| Model eval failures:",
            model_profile[
                "EvaluationFailures"
            ],
        )

        if not protocol_eligible:
            print(
                "    Ineligible:",
                row[
                    "EligibilityReason"
                ],
            )

    except Exception as error:
        row.update({
            "ProtocolEligible":
                False,

            "EligibilityReason":
                "Inspection error",

            "InspectionStatus":
                "ERROR",

            "InspectionError":
                (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
        })

        print(
            "    ERROR:",
            row[
                "InspectionError"
            ],
        )

    row[
        "ElapsedSeconds"
    ] = float(
        time.perf_counter()
        - started
    )

    scan_rows.append(
        row
    )

    atomic_write_csv(
        SCAN_PROGRESS_PATH,
        pd.DataFrame(
            scan_rows
        ).sort_values(
            "CandidateInspectionOrder",
            kind="mergesort",
        ),
    )


scan_progress = (
    pd.DataFrame(
        scan_rows
    )
    .sort_values(
        "CandidateInspectionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 7. DETERMINISTIC RUNTIME-PRIORITIZED RANKING
# --------------------------------------------------------------------------------------------------

inspection_errors = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "ERROR"
    )
].copy()

eligible_candidates = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "ELIGIBLE"
    )
].copy()

ineligible_candidates = scan_progress.loc[
    scan_progress[
        "InspectionStatus"
    ].eq(
        "INELIGIBLE"
    )
].copy()

if not inspection_errors.empty:
    print(
        "\nCandidate inspection errors:"
    )

    display(
        inspection_errors[
            [
                "Project",
                "InspectionError",
            ]
        ]
    )

    raise RuntimeError(
        "One or more Project 24 candidates could not be inspected. "
        "No provisional selection was frozen."
    )

if eligible_candidates.empty:
    raise RuntimeError(
        "No protocol-eligible Project 24 candidate was found."
    )

eligible_candidates = (
    eligible_candidates.sort_values(
        [
            "ModelTrainingRows",
            "ModelEvaluationRows",
            "RawExecutionRows",
            "Project",
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

eligible_candidates.insert(
    0,
    "CandidateRank",
    np.arange(
        1,
        len(
            eligible_candidates
        )
        + 1,
        dtype=np.int64,
    ),
)

top_candidate = eligible_candidates.iloc[
    0
]


# --------------------------------------------------------------------------------------------------
# 8. VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []

add_check(
    validation_records,
    "Completion registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Projects 1–23 COMPLETE_AND_FROZEN",
    EXPECTED_REGISTERED_PROJECTS,
    int(
        registry[
            registry_status_column
        ].eq(
            EXPECTED_COMPLETE_STATUS
        ).sum()
    ),
    int(
        registry[
            registry_status_column
        ].eq(
            EXPECTED_COMPLETE_STATUS
        ).sum()
    )
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Project 24 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)

for (
    required_number,
    required_project,
) in required_registered_identities.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            registry_project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        validation_records,
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )

add_check(
    validation_records,
    "Candidates inspected",
    EXPECTED_CANDIDATES,
    len(
        scan_progress
    ),
    len(
        scan_progress
    )
    == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "Unique candidate identities",
    EXPECTED_CANDIDATES,
    int(
        scan_progress[
            "Project"
        ].nunique()
    ),
    int(
        scan_progress[
            "Project"
        ].nunique()
    )
    == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "Expected candidate identity set",
    sorted(
        EXPECTED_CANDIDATE_SET
    ),
    sorted(
        set(
            scan_progress[
                "Project"
            ].astype(
                str
            )
        )
    ),
    set(
        scan_progress[
            "Project"
        ].astype(
            str
        )
    )
    == EXPECTED_CANDIDATE_SET,
)

add_check(
    validation_records,
    "Registered/reserved candidates",
    0,
    int(
        scan_progress[
            "Project"
        ].isin(
            registered_projects
            | RESERVED_ACTIVE_PROJECTS
        ).sum()
    ),
    int(
        scan_progress[
            "Project"
        ].isin(
            registered_projects
            | RESERVED_ACTIVE_PROJECTS
        ).sum()
    )
    == 0,
)

add_check(
    validation_records,
    "Inspection errors",
    0,
    len(
        inspection_errors
    ),
    len(
        inspection_errors
    )
    == 0,
)

add_check(
    validation_records,
    "Candidate accounting",
    EXPECTED_CANDIDATES,
    (
        len(
            eligible_candidates
        )
        + len(
            ineligible_candidates
        )
        + len(
            inspection_errors
        )
    ),
    (
        len(
            eligible_candidates
        )
        + len(
            ineligible_candidates
        )
        + len(
            inspection_errors
        )
    )
    == EXPECTED_CANDIDATES,
)

add_check(
    validation_records,
    "At least one eligible candidate",
    "> 0",
    len(
        eligible_candidates
    ),
    len(
        eligible_candidates
    )
    > 0,
)

add_check(
    validation_records,
    "Candidate ranks unique",
    len(
        eligible_candidates
    ),
    int(
        eligible_candidates[
            "CandidateRank"
        ].nunique()
    ),
    int(
        eligible_candidates[
            "CandidateRank"
        ].nunique()
    )
    == len(
        eligible_candidates
    ),
)

add_check(
    validation_records,
    "Top rank",
    1,
    int(
        top_candidate[
            "CandidateRank"
        ]
    ),
    int(
        top_candidate[
            "CandidateRank"
        ]
    )
    == 1,
)

add_check(
    validation_records,
    "Top candidate eligible",
    True,
    str(
        top_candidate[
            "InspectionStatus"
        ]
    )
    == "ELIGIBLE",
    str(
        top_candidate[
            "InspectionStatus"
        ]
    )
    == "ELIGIBLE",
)

add_check(
    validation_records,
    "Top candidate raw failing training builds",
    f">= {MIN_FAILING_TRAINING_BUILDS}",
    int(
        top_candidate[
            "RawFailingTrainingBuilds"
        ]
    ),
    int(
        top_candidate[
            "RawFailingTrainingBuilds"
        ]
    )
    >= MIN_FAILING_TRAINING_BUILDS,
)

add_check(
    validation_records,
    "Top candidate raw evaluation failures",
    "> 0",
    int(
        top_candidate[
            "RawEvaluationFailures"
        ]
    ),
    int(
        top_candidate[
            "RawEvaluationFailures"
        ]
    )
    > 0,
)

add_check(
    validation_records,
    "Top candidate model training both classes",
    True,
    bool(
        top_candidate[
            "ModelTrainingBothClasses"
        ]
    ),
    bool(
        top_candidate[
            "ModelTrainingBothClasses"
        ]
    ),
)

add_check(
    validation_records,
    "Top candidate model evaluation failures",
    "> 0",
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    ),
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    )
    > 0,
)

add_check(
    validation_records,
    "Top candidate raw unlinked rows",
    0,
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    ),
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    )
    == 0,
)

add_check(
    validation_records,
    "Top candidate model unlinked rows",
    0,
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    ),
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    )
    == 0,
)

validation = pd.DataFrame(
    validation_records
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]

print(
    "\nProject 24 Step 1A validation:"
)

display(
    validation
)

if not failed_validation.empty:
    print(
        "\nFailed validation checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 24 STEP 1A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 9. WRITE AUTHORITATIVE STEP 1A OUTPUTS
# --------------------------------------------------------------------------------------------------

schema_columns = [
    "Project",
    "ProjectSlug",
    "BuildIDColumn",
    "StartedAtColumn",
    "ExecutionBuildColumn",
    "ExecutionVerdictColumn",
    "DatasetBuildColumn",
    "DatasetVerdictColumn",
    "InspectionStatus",
    "InspectionError",
]

source_schema_audit = scan_progress[
    schema_columns
].copy()

atomic_write_csv(
    SOURCE_SCHEMA_AUDIT_PATH,
    source_schema_audit,
)

atomic_write_csv(
    CANDIDATE_INVENTORY_PATH,
    scan_progress,
)

atomic_write_csv(
    ELIGIBLE_RANKED_PATH,
    eligible_candidates,
)

atomic_write_csv(
    INELIGIBLE_PATH,
    ineligible_candidates,
)

atomic_write_csv(
    INSPECTION_ERRORS_PATH,
    inspection_errors,
)

atomic_write_csv(
    STEP1A_VALIDATION_PATH,
    validation,
)

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

dimension_keys = [
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "TimestampTieGroups",
    "TimestampTiedBuilds",
    "RawExecutionRows",
    "RawTrainingRows",
    "RawEvaluationRows",
    "RawTrainingPasses",
    "RawEvaluationPasses",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingTrainingBuilds",
    "RawFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainingPasses",
    "ModelEvaluationPasses",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingTrainingBuilds",
    "ModelFailingEvaluationBuilds",
    "ModelUnlinkedRows",
]

top_dimensions = {
    key:
        int(
            top_candidate[
                key
            ]
        )
    for key in dimension_keys
}

provisional_selection = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "SelectionState":
        "PROVISIONAL_PENDING_STEP_1B_FREEZE",

    "CompletedAtUTC":
        completed_at_utc,

    "CandidateRank":
        int(
            top_candidate[
                "CandidateRank"
            ]
        ),

    "Project":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "SourceDirectory":
        str(
            top_candidate[
                "SourceDirectory"
            ]
        ),

    "BuildIDColumn":
        str(
            top_candidate[
                "BuildIDColumn"
            ]
        ),

    "StartedAtColumn":
        str(
            top_candidate[
                "StartedAtColumn"
            ]
        ),

    "ExecutionBuildColumn":
        str(
            top_candidate[
                "ExecutionBuildColumn"
            ]
        ),

    "ExecutionVerdictColumn":
        str(
            top_candidate[
                "ExecutionVerdictColumn"
            ]
        ),

    "DatasetBuildColumn":
        str(
            top_candidate[
                "DatasetBuildColumn"
            ]
        ),

    "DatasetVerdictColumn":
        str(
            top_candidate[
                "DatasetVerdictColumn"
            ]
        ),

    "Dimensions":
        top_dimensions,

    "MinimumFailingTrainingBuilds":
        MIN_FAILING_TRAINING_BUILDS,

    "StrictEligibilityContract": {
        "ChronologicalSplit":
            "75/25",

        "MinimumRawFailingTrainingBuilds":
            MIN_FAILING_TRAINING_BUILDS,

        "RawEvaluationFailuresRequired":
            True,

        "ModelEvaluationFailuresRequired":
            True,

        "ModelTrainingBothClassesRequired":
            True,

        "RawUnlinkedRowsRequired":
            0,

        "ModelUnlinkedRowsRequired":
            0,
    },

    "RuntimePriorityRule":
        RUNTIME_PRIORITY_RULE,

    "ActiveReservations":
        [],

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "BootstrapStatusPath":
        str(
            BOOTSTRAP_STATUS_PATH
        ),

    "BootstrapCandidateInventoryPath":
        str(
            BOOTSTRAP_CANDIDATE_INVENTORY_PATH
        ),

    "EligibleRankedPath":
        str(
            ELIGIBLE_RANKED_PATH
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ModelsFitted":
        False,

    "ExperimentStarted":
        False,
}

atomic_write_json(
    PROVISIONAL_SELECTION_PATH,
    provisional_selection,
)

output_manifest_paths = [
    SOURCE_SCHEMA_AUDIT_PATH,
    CANDIDATE_INVENTORY_PATH,
    ELIGIBLE_RANKED_PATH,
    INELIGIBLE_PATH,
    INSPECTION_ERRORS_PATH,
    PROVISIONAL_SELECTION_PATH,
    STEP1A_VALIDATION_PATH,
]

output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_manifest_paths
]

report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "CandidatesInspected":
        int(
            len(
                scan_progress
            )
        ),

    "ProtocolEligibleCandidates":
        int(
            len(
                eligible_candidates
            )
        ),

    "ProtocolIneligibleCandidates":
        int(
            len(
                ineligible_candidates
            )
        ),

    "InspectionErrors":
        int(
            len(
                inspection_errors
            )
        ),

    "ProvisionalProject":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProvisionalProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "ProvisionalCandidateRank":
        int(
            top_candidate[
                "CandidateRank"
            ]
        ),

    "Dimensions":
        top_dimensions,

    "StrictEligibilityContract":
        provisional_selection[
            "StrictEligibilityContract"
        ],

    "RuntimePriorityRule":
        RUNTIME_PRIORITY_RULE,

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "ActiveReservations":
        [],

    "OutputManifest":
        output_manifest,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ModelsFitted":
        False,

    "ExperimentStarted":
        False,
}

atomic_write_json(
    STEP1A_REPORT_PATH,
    report_payload,
)

status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Status":
        STEP1A_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "CandidatesInspected":
        int(
            len(
                scan_progress
            )
        ),

    "ProtocolEligibleCandidates":
        int(
            len(
                eligible_candidates
            )
        ),

    "ProtocolIneligibleCandidates":
        int(
            len(
                ineligible_candidates
            )
        ),

    "InspectionErrors":
        int(
            len(
                inspection_errors
            )
        ),

    "ProvisionalProject":
        str(
            top_candidate[
                "Project"
            ]
        ),

    "ProvisionalProjectSlug":
        str(
            top_candidate[
                "ProjectSlug"
            ]
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ModelsFitted":
        False,

    "ExperimentStarted":
        False,
}

atomic_write_json(
    STEP1A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 10. READBACK + FINAL ISOLATION
# --------------------------------------------------------------------------------------------------

for manifest_item in output_manifest:
    path = Path(
        manifest_item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            manifest_item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            manifest_item[
                "SHA256"
            ]
        )
    ):
        raise RuntimeError(
            f"Project 24 Step 1A output readback failed: {path}"
        )

provisional_readback = load_json(
    PROVISIONAL_SELECTION_PATH
)

if provisional_readback.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 24 provisional selection readback failed."
    )

if provisional_readback.get(
    "SelectionState"
) != "PROVISIONAL_PENDING_STEP_1B_FREEZE":
    raise RuntimeError(
        "Project 24 provisional selection state readback failed."
    )

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

if (
    registry_sha256_after
    != registry_sha256_before
):
    raise RuntimeError(
        "The completion registry changed during Project 24 Step 1A."
    )


# --------------------------------------------------------------------------------------------------
# 11. DISPLAY FINAL RESULT
# --------------------------------------------------------------------------------------------------

ranked_display_columns = [
    "CandidateRank",
    "Project",
    "ProjectSlug",
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "TimestampTieGroups",
    "TimestampTiedBuilds",
    "RawExecutionRows",
    "RawFailingTrainingBuilds",
    "RawEvaluationFailures",
    "RawFailingEvaluationBuilds",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainingPasses",
    "ModelTrainFailures",
    "ModelTrainingBothClasses",
    "ModelEvaluationFailures",
    "ModelFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelUnlinkedRows",
]

print(
    "\nRanked eligible Project 24 candidates:"
)

display(
    eligible_candidates[
        ranked_display_columns
    ]
)

print(
    "\nProtocol-ineligible Project 24 candidates:"
)

ineligible_display_columns = [
    "Project",
    "Builds",
    "RawFailingTrainingBuilds",
    "RawEvaluationFailures",
    "ModelTrainingPasses",
    "ModelTrainFailures",
    "ModelTrainingBothClasses",
    "ModelEvaluationFailures",
    "RawUnlinkedRows",
    "ModelUnlinkedRows",
    "EligibilityReason",
    "InspectionStatus",
]

display(
    ineligible_candidates[
        [
            column
            for column in ineligible_display_columns
            if column in ineligible_candidates.columns
        ]
    ]
)

print(
    "\nResolved TCP-CI schemas:"
)

display(
    source_schema_audit
)

print(
    "\n"
    + "=" * 136
)

print(
    "=== PROJECT 24 CELL 2 / STEP 1A RESULT ==="
)

print(
    "=" * 136
)

print(
    "\nCandidate discovery:"
)

print(
    "Candidates inspected:",
    len(
        scan_progress
    ),
)

print(
    "Protocol-eligible candidates:",
    len(
        eligible_candidates
    ),
)

print(
    "Protocol-ineligible candidates:",
    len(
        ineligible_candidates
    ),
)

print(
    "Inspection errors:",
    len(
        inspection_errors
    ),
)

print(
    "\nProvisional Project 24 candidate:"
)

print(
    "Candidate rank:",
    int(
        top_candidate[
            "CandidateRank"
        ]
    ),
)

print(
    "Project:",
    str(
        top_candidate[
            "Project"
        ]
    ),
)

print(
    "Project slug:",
    str(
        top_candidate[
            "ProjectSlug"
        ]
    ),
)

print(
    "Source directory:",
    str(
        top_candidate[
            "SourceDirectory"
        ]
    ),
)

print(
    "\nStrict eligibility:"
)

print(
    "Raw failing training builds:",
    int(
        top_candidate[
            "RawFailingTrainingBuilds"
        ]
    ),
    f"(minimum {MIN_FAILING_TRAINING_BUILDS})",
)

print(
    "Raw evaluation failures:",
    int(
        top_candidate[
            "RawEvaluationFailures"
        ]
    ),
)

print(
    "Model training passes:",
    int(
        top_candidate[
            "ModelTrainingPasses"
        ]
    ),
)

print(
    "Model training failures:",
    int(
        top_candidate[
            "ModelTrainFailures"
        ]
    ),
)

print(
    "Model training both classes:",
    bool(
        top_candidate[
            "ModelTrainingBothClasses"
        ]
    ),
)

print(
    "Model evaluation failures:",
    int(
        top_candidate[
            "ModelEvaluationFailures"
        ]
    ),
)

print(
    "Raw unlinked rows:",
    int(
        top_candidate[
            "RawUnlinkedRows"
        ]
    ),
)

print(
    "Model unlinked rows:",
    int(
        top_candidate[
            "ModelUnlinkedRows"
        ]
    ),
)

print(
    "\nRuntime-priority dimensions:"
)

print(
    "ModelTrainingRows:",
    int(
        top_candidate[
            "ModelTrainingRows"
        ]
    ),
)

print(
    "ModelEvaluationRows:",
    int(
        top_candidate[
            "ModelEvaluationRows"
        ]
    ),
)

print(
    "RawExecutionRows:",
    int(
        top_candidate[
            "RawExecutionRows"
        ]
    ),
)

print(
    "Runtime-priority rule:",
    RUNTIME_PRIORITY_RULE,
)

print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–23 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Models fitted:",
    False,
)

print(
    "Experiment started:",
    False,
)

print(
    "\nSaved outputs:"
)

for output_path in [
    SCAN_PROGRESS_PATH,
    SOURCE_SCHEMA_AUDIT_PATH,
    CANDIDATE_INVENTORY_PATH,
    ELIGIBLE_RANKED_PATH,
    INELIGIBLE_PATH,
    INSPECTION_ERRORS_PATH,
    PROVISIONAL_SELECTION_PATH,
    STEP1A_VALIDATION_PATH,
    STEP1A_REPORT_PATH,
    STEP1A_STATUS_PATH,
]:
    print(
        output_path
    )

print(
    "\nNext required step:",
    "PROJECT 24 CELL 3 / STEP 1B — FINAL SELECTION AND SOURCE FREEZE",
)

print(
    "STATUS:",
    STEP1A_PASS_STATUS,
)

print(
    "=" * 136
)


=== PROJECT 24 CELL 2 / STEP 1A: CANDIDATE ELIGIBILITY AND RUNTIME-PRIORITIZED RANKING ===
----------------------------------------------------------------------------------------------------------------------------------------
[01/02] Inspecting: Graylog2@graylog2-server
    Status: INELIGIBLE | Builds: 3668 | Raw failing train builds: 125 | Model train rows: 4822 | Model eval rows: 0 | Model eval failures: 0
    Ineligible: No raw evaluation failures; No model evaluation rows; No model evaluation failures
----------------------------------------------------------------------------------------------------------------------------------------
[02/02] Inspecting: SonarSource@sonarqube
    Status: ELIGIBLE | Builds: 4286 | Raw failing train builds: 283 | Model train rows: 205696 | Model eval rows: 18854 | Model eval failures: 20

Project 24 Step 1A validation:


,Check,Expected,Actual,Pass
0,Completion registry rows,23,23,True
1,Projects 1–23 COMPLETE_AND_FROZEN,23,23,True
2,Project 24 registry rows,0,0,True
3,Project 11 frozen identity,apache@shardingsphere,apache@shardingsphere,True
4,Project 12 frozen identity,zolyfarkas@spf4j,zolyfarkas@spf4j,True
5,Project 13 frozen identity,jcabi@jcabi-github,jcabi@jcabi-github,True
6,Project 14 frozen identity,JMRI@JMRI,JMRI@JMRI,True
7,Project 15 frozen identity,eclipse@steady,eclipse@steady,True
8,Project 16 frozen identity,apache@rocketmq,apache@rocketmq,True
9,Project 17 frozen identity,yamcs@Yamcs,yamcs@Yamcs,True



Ranked eligible Project 24 candidates:


,CandidateRank,Project,ProjectSlug,Builds,TrainingBuilds,EvaluationBuilds,TimestampTieGroups,TimestampTiedBuilds,RawExecutionRows,RawFailingTrainingBuilds,...,ModelReadyRows,ModelTrainingRows,ModelEvaluationRows,ModelTrainingPasses,ModelTrainFailures,ModelTrainingBothClasses,ModelEvaluationFailures,ModelFailingEvaluationBuilds,RawUnlinkedRows,ModelUnlinkedRows
0,1,SonarSource@sonarqube,SonarSource__sonarqube,4286,3214,1072,23,46,5635027,283,...,224550,205696,18854,203919,1777,True,20,17,0,0



Protocol-ineligible Project 24 candidates:


,Project,Builds,RawFailingTrainingBuilds,RawEvaluationFailures,ModelTrainingPasses,ModelTrainFailures,ModelTrainingBothClasses,ModelEvaluationFailures,RawUnlinkedRows,ModelUnlinkedRows,EligibilityReason,InspectionStatus
0,Graylog2@graylog2-server,3668,125,0,4543,279,True,0,0,0,No raw evaluation failures; No model evaluatio...,INELIGIBLE



Resolved TCP-CI schemas:


,Project,ProjectSlug,BuildIDColumn,StartedAtColumn,ExecutionBuildColumn,ExecutionVerdictColumn,DatasetBuildColumn,DatasetVerdictColumn,InspectionStatus,InspectionError
0,Graylog2@graylog2-server,Graylog2__graylog2-server,id,started_at,build,verdict,Build,Verdict,INELIGIBLE,
1,SonarSource@sonarqube,SonarSource__sonarqube,id,started_at,build,verdict,Build,Verdict,ELIGIBLE,



=== PROJECT 24 CELL 2 / STEP 1A RESULT ===

Candidate discovery:
Candidates inspected: 2
Protocol-eligible candidates: 1
Protocol-ineligible candidates: 1
Inspection errors: 0

Provisional Project 24 candidate:
Candidate rank: 1
Project: SonarSource@sonarqube
Project slug: SonarSource__sonarqube
Source directory: /content/datasets/datasets/SonarSource@sonarqube

Strict eligibility:
Raw failing training builds: 283 (minimum 10)
Raw evaluation failures: 20
Model training passes: 203919
Model training failures: 1777
Model training both classes: True
Model evaluation failures: 20
Raw unlinked rows: 0
Model unlinked rows: 0

Runtime-priority dimensions:
ModelTrainingRows: 205696
ModelEvaluationRows: 18854
RawExecutionRows: 5635027
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Isolation:
Completion registry unchanged: True
Projects 1–23 modified: 0
Prior project condition outputs accessed: False
Pr

In [3]:
# ==================================================================================================
# PROJECT 24 — CELL 3 / STEP 1B
# FINAL SELECTION, CHRONOLOGY FREEZE, SOURCE MANIFEST, AND CHECKPOINT
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_24.ipynb.
#
# SAFETY:
# - freezes the Project 24 identity selected by Step 1A;
# - freezes the COMPLETE source manifest and canonical source-root SHA-256;
# - freezes the exact chronological 75/25 build split used by Step 1A;
# - independently recomputes every raw/model dimension frozen in Step 1A;
# - verifies the selected project remains strictly protocol-eligible;
# - writes no completion-registry changes;
# - does not access/modify prior-project condition outputs;
# - does not reconstruct REC, inject noise, fit models, or start the experiment.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import math
import os

import numpy as np
import pandas as pd


print("=" * 136)
print("=== PROJECT 24 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT CONTRACT
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
CANDIDATE_RANK = 1

STEP1A_PASS_STATUS = (
    "PASS_PROJECT_24_CANDIDATE_DISCOVERY_COMPLETE"
)

STEP1B_PASS_STATUS = (
    "PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_ARCHIVE_SHA256 = (
    "92af159c116e06e98d7c8348605adb6e9acb25aad982a1ff9fbef7103cb9e36e"
)

EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 23
EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

# Values explicitly visible in the successful Project 24 Step 1A output.
# Every remaining dimension is inherited from the frozen provisional selection
# and independently recomputed below before the final freeze.
EXPECTED_VISIBLE_DIMENSIONS = {
    "Builds": 4_286,
    "TrainingBuilds": 3_214,
    "EvaluationBuilds": 1_072,

    "RawExecutionRows": 5_635_027,
    "RawFailingTrainingBuilds": 283,
    "RawEvaluationFailures": 20,
    "RawUnlinkedRows": 0,

    "ModelReadyRows": 224_550,
    "ModelTrainingRows": 205_696,
    "ModelEvaluationRows": 18_854,
    "ModelTrainingPasses": 203_919,
    "ModelTrainFailures": 1_777,
    "ModelEvaluationFailures": 20,
    "ModelFailingEvaluationBuilds": 17,
    "ModelUnlinkedRows": 0,
}

EXPECTED_DIMENSION_KEYS = {
    "Builds",
    "TrainingBuilds",
    "EvaluationBuilds",
    "TimestampTieGroups",
    "TimestampTiedBuilds",

    "RawExecutionRows",
    "RawTrainingRows",
    "RawEvaluationRows",
    "RawTrainingPasses",
    "RawEvaluationPasses",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingTrainingBuilds",
    "RawFailingEvaluationBuilds",
    "RawUnlinkedRows",

    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainingPasses",
    "ModelEvaluationPasses",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingTrainingBuilds",
    "ModelFailingEvaluationBuilds",
    "ModelUnlinkedRows",
}

MIN_FAILING_TRAINING_BUILDS = 10

REQUIRED_SOURCE_FILES = {
    "builds.csv",
    "exe.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "id_map.csv",
}

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

ARCHIVE_PATH = (
    THESIS_ROOT
    / "Data"
    / "Raw"
    / "TCP-CI-main-dataset.tar.gz"
)

REGISTRY_PATH = (
    THESIS_ROOT
    / "Notes"
    / "completed_project_registry.csv"
)

SOURCE_DIRECTORY = Path(
    "/content/datasets/datasets/SonarSource@sonarqube"
)

SELECTION_ROOT = (
    THESIS_ROOT
    / "Results"
    / "Aggregated"
    / "project_24_selection"
)

STEP1A_STATUS_PATH = (
    SELECTION_ROOT
    / "project_24_step1a_status.json"
)

STEP1A_REPORT_PATH = (
    SELECTION_ROOT
    / "project_24_step1a_report.json"
)

PROVISIONAL_SELECTION_PATH = (
    SELECTION_ROOT
    / "project_24_provisional_selection.json"
)

ELIGIBLE_RANKED_PATH = (
    SELECTION_ROOT
    / "project_24_eligible_candidates_ranked.csv"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_24_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_24_fixed_chronological_builds.csv"
)

SOURCE_SCHEMA_SNAPSHOT_PATH = (
    SELECTION_ROOT
    / "project_24_selected_source_schema_snapshot.csv"
)

STEP1B_VALIDATION_PATH = (
    SELECTION_ROOT
    / "project_24_step1b_validation.csv"
)

STEP1B_REPORT_PATH = (
    SELECTION_ROOT
    / "project_24_step1b_report.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_24_step1b_status.json"
)

SELECTION_CHECKPOINT_PATH = (
    THESIS_ROOT
    / "Notes"
    / "project_24_selection_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def atomic_write_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(columns, expected, label):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve {label}.\n"
            f"Expected: {expected!r}\n"
            f"Matches: {matches}\n"
            f"Columns: {list(columns)}"
        )

    return matches[0]


def parse_integer_series(values, label):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing or non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(array),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype("int64")


def canonical_root_hash(manifest):
    required_columns = {
        "RelativePath",
        "SizeBytes",
        "SHA256",
    }

    missing_columns = (
        required_columns
        - set(manifest.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Source manifest is missing columns:\n"
            + "\n".join(
                sorted(missing_columns)
            )
        )

    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.SizeBytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


def profile_partition_chunked(
    csv_path,
    build_column,
    verdict_column,
    training_build_ids,
    evaluation_build_ids,
    label,
    chunksize,
):
    rows = 0
    training_rows = 0
    evaluation_rows = 0

    training_failures = 0
    evaluation_failures = 0

    failing_training_builds = set()
    failing_evaluation_builds = set()

    unlinked_rows = 0

    for chunk in pd.read_csv(
        csv_path,
        usecols=[
            build_column,
            verdict_column,
        ],
        chunksize=chunksize,
        low_memory=False,
    ):
        build_values = parse_integer_series(
            chunk[
                build_column
            ],
            f"{label}.{build_column}",
        )

        verdict_values = parse_integer_series(
            chunk[
                verdict_column
            ],
            f"{label}.{verdict_column}",
        )

        training_mask = build_values.isin(
            training_build_ids
        )

        evaluation_mask = build_values.isin(
            evaluation_build_ids
        )

        linked_mask = (
            training_mask
            | evaluation_mask
        )

        failure_mask = verdict_values.ne(0)

        rows += int(
            len(chunk)
        )

        training_rows += int(
            training_mask.sum()
        )

        evaluation_rows += int(
            evaluation_mask.sum()
        )

        training_failures += int(
            (
                training_mask
                & failure_mask
            ).sum()
        )

        evaluation_failures += int(
            (
                evaluation_mask
                & failure_mask
            ).sum()
        )

        failing_training_builds.update(
            build_values.loc[
                training_mask
                & failure_mask
            ].astype(int).tolist()
        )

        failing_evaluation_builds.update(
            build_values.loc[
                evaluation_mask
                & failure_mask
            ].astype(int).tolist()
        )

        unlinked_rows += int(
            (
                ~linked_mask
            ).sum()
        )

    return {
        "Rows":
            int(rows),

        "TrainingRows":
            int(training_rows),

        "EvaluationRows":
            int(evaluation_rows),

        "TrainingPasses":
            int(
                training_rows
                - training_failures
            ),

        "EvaluationPasses":
            int(
                evaluation_rows
                - evaluation_failures
            ),

        "TrainingFailures":
            int(training_failures),

        "EvaluationFailures":
            int(evaluation_failures),

        "FailingTrainingBuilds":
            int(
                len(
                    failing_training_builds
                )
            ),

        "FailingEvaluationBuilds":
            int(
                len(
                    failing_evaluation_builds
                )
            ),

        "UnlinkedRows":
            int(unlinked_rows),
    }


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


# --------------------------------------------------------------------------------------------------
# 4. VALIDATE REQUIRED INPUTS
# --------------------------------------------------------------------------------------------------

required_inputs = [
    ARCHIVE_PATH,
    REGISTRY_PATH,
    STEP1A_STATUS_PATH,
    STEP1A_REPORT_PATH,
    PROVISIONAL_SELECTION_PATH,
    ELIGIBLE_RANKED_PATH,
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 24 Step 1B inputs are missing:\n"
        + "\n".join(missing_inputs)
    )

if not SOURCE_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "Selected Project 24 source directory is missing:\n"
        f"{SOURCE_DIRECTORY}"
    )

source_file_names = {
    path.name
    for path in SOURCE_DIRECTORY.iterdir()
    if path.is_file()
}

missing_required_source_files = sorted(
    REQUIRED_SOURCE_FILES
    - source_file_names
)

if missing_required_source_files:
    raise FileNotFoundError(
        "Selected Project 24 source is missing required files:\n"
        + "\n".join(
            missing_required_source_files
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE FROZEN STEP 1A SELECTION
# --------------------------------------------------------------------------------------------------

step1a_status_sha256 = sha256_file(
    STEP1A_STATUS_PATH
)

step1a_report_sha256 = sha256_file(
    STEP1A_REPORT_PATH
)

provisional_selection_sha256 = sha256_file(
    PROVISIONAL_SELECTION_PATH
)

step1a_status = load_json(
    STEP1A_STATUS_PATH
)

step1a_report = load_json(
    STEP1A_REPORT_PATH
)

provisional_selection = load_json(
    PROVISIONAL_SELECTION_PATH
)

if step1a_status.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 24 Step 1A status is not PASS."
    )

if step1a_report.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 24 Step 1A report is not PASS."
    )

if provisional_selection.get(
    "Status"
) != STEP1A_PASS_STATUS:
    raise RuntimeError(
        "Project 24 provisional selection is not in the Step 1A PASS state."
    )

if provisional_selection.get(
    "SelectionState"
) != "PROVISIONAL_PENDING_STEP_1B_FREEZE":
    raise RuntimeError(
        "Project 24 provisional selection state differs."
    )

if provisional_selection.get(
    "Project"
) != PROJECT_NAME:
    raise RuntimeError(
        "Project 24 provisional project differs.\n"
        f"Expected: {PROJECT_NAME}\n"
        f"Actual:   {provisional_selection.get('Project')}"
    )

if provisional_selection.get(
    "ProjectSlug"
) != PROJECT_SLUG:
    raise RuntimeError(
        "Project 24 provisional slug differs."
    )

if Path(
    provisional_selection.get(
        "SourceDirectory",
        "",
    )
) != SOURCE_DIRECTORY:
    raise RuntimeError(
        "Project 24 provisional source directory differs."
    )

if int(
    provisional_selection.get(
        "CandidateRank",
        -1,
    )
) != CANDIDATE_RANK:
    raise RuntimeError(
        "Project 24 provisional candidate rank differs."
    )

if provisional_selection.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Project 24 runtime-priority ranking rule differs."
    )

if provisional_selection.get(
    "ActiveReservations",
    None,
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Project 24 Step 1A active-reservation state differs."
    )

if provisional_selection.get(
    "ArchiveSHA256"
) != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Project 24 provisional selection archive SHA differs."
    )

if provisional_selection.get(
    "CompletionRegistrySHA256"
) != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Project 24 provisional selection registry SHA differs."
    )

for key in [
    "RegistryModified",
    "PriorProjectConditionOutputsAccessed",
    "PriorProjectConditionOutputsModified",
    "ModelsFitted",
    "ExperimentStarted",
]:
    if bool(
        provisional_selection.get(
            key,
            True,
        )
    ):
        raise RuntimeError(
            f"Project 24 Step 1A safety flag {key} is unexpectedly True."
        )


# --------------------------------------------------------------------------------------------------
# 6. FREEZE COMPLETE STEP 1A DIMENSION CONTRACT
# --------------------------------------------------------------------------------------------------

step1a_dimensions_raw = provisional_selection.get(
    "Dimensions",
    {},
)

if not isinstance(
    step1a_dimensions_raw,
    dict,
):
    raise RuntimeError(
        "Project 24 Step 1A Dimensions is not a dictionary."
    )

if set(
    step1a_dimensions_raw.keys()
) != EXPECTED_DIMENSION_KEYS:
    raise RuntimeError(
        "Project 24 Step 1A dimension key set differs.\n"
        f"Expected: {sorted(EXPECTED_DIMENSION_KEYS)}\n"
        f"Actual:   {sorted(step1a_dimensions_raw.keys())}"
    )

step1a_dimensions = {
    key:
        int(value)
    for key, value in step1a_dimensions_raw.items()
}

for key, expected_value in EXPECTED_VISIBLE_DIMENSIONS.items():
    actual_value = int(
        step1a_dimensions[
            key
        ]
    )

    if actual_value != expected_value:
        raise RuntimeError(
            "Project 24 Step 1A visible dimension differs.\n"
            f"Metric:   {key}\n"
            f"Expected: {expected_value}\n"
            f"Actual:   {actual_value}"
        )


# --------------------------------------------------------------------------------------------------
# 7. VALIDATE ARCHIVE + REGISTRY PRE-STATE
# --------------------------------------------------------------------------------------------------

archive_sha256 = sha256_file(
    ARCHIVE_PATH
)

if archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "Dataset archive SHA-256 differs.\n"
        f"Expected: {EXPECTED_ARCHIVE_SHA256}\n"
        f"Actual:   {archive_sha256}"
    )

registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )

registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)

registry_project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

registry_project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

registry_status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)

registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)

if (
    len(registry)
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–23."
    )

if not registry[
    registry_status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )

if registry_project_numbers.eq(
    PROJECT_NUMBER
).any():
    raise RuntimeError(
        "Project 24 is unexpectedly already registered."
    )

if registry[
    registry_project_column
].eq(
    PROJECT_NAME
).any():
    raise RuntimeError(
        "The selected Project 24 identity is already registered."
    )

for (
    required_number,
    required_project,
) in REQUIRED_REGISTERED_IDENTITIES.items():
    matching_rows = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            registry_project_column
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

ranked_candidates = pd.read_csv(
    ELIGIBLE_RANKED_PATH,
    low_memory=False,
)

rank_one_rows = ranked_candidates.loc[
    pd.to_numeric(
        ranked_candidates[
            "CandidateRank"
        ],
        errors="coerce",
    ).eq(
        CANDIDATE_RANK
    )
]

if len(
    rank_one_rows
) != 1:
    raise RuntimeError(
        "Step 1A ranked candidates do not contain exactly one rank-1 row."
    )

rank_one = rank_one_rows.iloc[
    0
]

if (
    str(
        rank_one[
            "Project"
        ]
    )
    != PROJECT_NAME
    or str(
        rank_one[
            "ProjectSlug"
        ]
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "Step 1A rank-1 identity differs."
    )

if not bool(
    rank_one[
        "ProtocolEligible"
    ]
):
    raise RuntimeError(
        "Step 1A rank-1 candidate is not protocol eligible."
    )


# --------------------------------------------------------------------------------------------------
# 8. FREEZE COMPLETE SOURCE MANIFEST
# --------------------------------------------------------------------------------------------------

source_files = sorted(
    [
        path
        for path in SOURCE_DIRECTORY.rglob(
            "*"
        )
        if path.is_file()
    ],
    key=lambda path:
        path.relative_to(
            SOURCE_DIRECTORY
        ).as_posix(),
)

if not source_files:
    raise RuntimeError(
        "Selected Project 24 source directory contains no files."
    )

source_manifest_records = []

for source_path in source_files:
    source_manifest_records.append({
        "RelativePath":
            source_path.relative_to(
                SOURCE_DIRECTORY
            ).as_posix(),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })

source_manifest = pd.DataFrame(
    source_manifest_records
)

source_file_count = int(
    len(
        source_manifest
    )
)

source_bytes = int(
    source_manifest[
        "SizeBytes"
    ].sum()
)

source_root_sha256 = canonical_root_hash(
    source_manifest
)


# --------------------------------------------------------------------------------------------------
# 9. FREEZE SOURCE SCHEMA SNAPSHOT
# --------------------------------------------------------------------------------------------------

schema_records = []

for source_path in source_files:
    relative_path = source_path.relative_to(
        SOURCE_DIRECTORY
    ).as_posix()

    suffix = source_path.suffix.lower()

    if suffix == ".csv":
        header = pd.read_csv(
            source_path,
            nrows=0,
        )

        for column_order, column in enumerate(
            header.columns,
            start=1,
        ):
            schema_records.append({
                "RelativePath":
                    relative_path,

                "FileType":
                    "csv",

                "ColumnOrder":
                    int(
                        column_order
                    ),

                "ColumnName":
                    str(
                        column
                    ),
            })

    else:
        schema_records.append({
            "RelativePath":
                relative_path,

            "FileType":
                suffix.lstrip(
                    "."
                )
                or "unknown",

            "ColumnOrder":
                np.nan,

            "ColumnName":
                "",
        })

source_schema_snapshot = pd.DataFrame(
    schema_records
)


# --------------------------------------------------------------------------------------------------
# 10. REBUILD AND FREEZE EXACT CHRONOLOGY
# --------------------------------------------------------------------------------------------------

builds_path = (
    SOURCE_DIRECTORY
    / "builds.csv"
)

exe_path = (
    SOURCE_DIRECTORY
    / "exe.csv"
)

dataset_path = (
    SOURCE_DIRECTORY
    / "dataset.csv"
)

build_columns = pd.read_csv(
    builds_path,
    nrows=0,
).columns.tolist()

exe_columns = pd.read_csv(
    exe_path,
    nrows=0,
).columns.tolist()

dataset_columns = pd.read_csv(
    dataset_path,
    nrows=0,
).columns.tolist()

build_id_column = resolve_column(
    build_columns,
    provisional_selection[
        "BuildIDColumn"
    ],
    "builds.csv build ID",
)

started_at_column = resolve_column(
    build_columns,
    provisional_selection[
        "StartedAtColumn"
    ],
    "builds.csv started_at",
)

execution_build_column = resolve_column(
    exe_columns,
    provisional_selection[
        "ExecutionBuildColumn"
    ],
    "exe.csv build",
)

execution_verdict_column = resolve_column(
    exe_columns,
    provisional_selection[
        "ExecutionVerdictColumn"
    ],
    "exe.csv verdict",
)

dataset_build_column = resolve_column(
    dataset_columns,
    provisional_selection[
        "DatasetBuildColumn"
    ],
    "dataset.csv Build",
)

dataset_verdict_column = resolve_column(
    dataset_columns,
    provisional_selection[
        "DatasetVerdictColumn"
    ],
    "dataset.csv Verdict",
)

builds = pd.read_csv(
    builds_path,
    usecols=[
        build_id_column,
        started_at_column,
    ],
    low_memory=False,
)

builds[
    build_id_column
] = parse_integer_series(
    builds[
        build_id_column
    ],
    "builds.id",
)

builds[
    started_at_column
] = pd.to_datetime(
    builds[
        started_at_column
    ],
    errors="coerce",
    utc=True,
)

invalid_build_timestamps = int(
    builds[
        started_at_column
    ].isna().sum()
)

duplicate_build_id_rows = int(
    builds[
        build_id_column
    ].duplicated(
        keep=False
    ).sum()
)

if invalid_build_timestamps:
    raise RuntimeError(
        f"Invalid build timestamps: {invalid_build_timestamps}"
    )

if duplicate_build_id_rows:
    raise RuntimeError(
        f"Duplicate build-ID rows: {duplicate_build_id_rows}"
    )

timestamp_group_sizes = (
    builds.groupby(
        started_at_column,
        dropna=False,
    ).size()
)

timestamp_tie_groups = int(
    timestamp_group_sizes.gt(
        1
    ).sum()
)

timestamp_tied_builds = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(
            1
        )
    ].sum()
)

# Frozen chronology rule inherited exactly from Step 1A:
# started_at ascending; within an exact timestamp tie, build ID descending.
ordered_builds = (
    builds.sort_values(
        [
            started_at_column,
            build_id_column,
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

number_of_builds = int(
    len(
        ordered_builds
    )
)

training_build_count = int(
    math.floor(
        0.75
        * number_of_builds
    )
)

evaluation_build_count = int(
    number_of_builds
    - training_build_count
)

if (
    training_build_count <= 0
    or evaluation_build_count <= 0
):
    raise RuntimeError(
        "Chronological 75/25 split has an empty partition."
    )

training_build_ids = set(
    ordered_builds.iloc[
        :training_build_count
    ][
        build_id_column
    ].astype(
        int
    ).tolist()
)

evaluation_build_ids = set(
    ordered_builds.iloc[
        training_build_count:
    ][
        build_id_column
    ].astype(
        int
    ).tolist()
)

if (
    training_build_ids
    & evaluation_build_ids
):
    raise RuntimeError(
        "Training/evaluation build partitions overlap."
    )

tie_group_lookup = (
    ordered_builds.groupby(
        started_at_column
    )[
        build_id_column
    ].transform(
        "size"
    )
)

fixed_chronology = pd.DataFrame({
    "BuildOrder":
        np.arange(
            1,
            number_of_builds + 1,
            dtype=np.int64,
        ),

    "Build":
        ordered_builds[
            build_id_column
        ].astype(
            "int64"
        ).to_numpy(),

    "StartedAtUTC":
        ordered_builds[
            started_at_column
        ].dt.strftime(
            "%Y-%m-%dT%H:%M:%S.%f%z"
        ).to_numpy(),

    "TimestampTieGroupSize":
        tie_group_lookup.astype(
            "int64"
        ).to_numpy(),

    "InTimestampTie":
        tie_group_lookup.gt(
            1
        ).to_numpy(),

    "Split":
        np.where(
            np.arange(
                number_of_builds
            )
            < training_build_count,
            "TRAIN",
            "EVAL",
        ),
})

if fixed_chronology[
    "Build"
].duplicated(
    keep=False
).any():
    raise RuntimeError(
        "Frozen chronology contains duplicate build IDs."
    )

if int(
    fixed_chronology[
        "Split"
    ].eq(
        "TRAIN"
    ).sum()
) != training_build_count:
    raise RuntimeError(
        "Frozen chronology training-build count differs."
    )

if int(
    fixed_chronology[
        "Split"
    ].eq(
        "EVAL"
    ).sum()
) != evaluation_build_count:
    raise RuntimeError(
        "Frozen chronology evaluation-build count differs."
    )


# --------------------------------------------------------------------------------------------------
# 11. INDEPENDENTLY RECOMPUTE RAW + MODEL DIMENSIONS
# --------------------------------------------------------------------------------------------------

raw_profile = profile_partition_chunked(
    csv_path=exe_path,
    build_column=execution_build_column,
    verdict_column=execution_verdict_column,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
    label="exe",
    chunksize=500_000,
)

model_profile = profile_partition_chunked(
    csv_path=dataset_path,
    build_column=dataset_build_column,
    verdict_column=dataset_verdict_column,
    training_build_ids=training_build_ids,
    evaluation_build_ids=evaluation_build_ids,
    label="dataset",
    chunksize=250_000,
)

actual_dimensions = {
    "Builds":
        number_of_builds,

    "TrainingBuilds":
        training_build_count,

    "EvaluationBuilds":
        evaluation_build_count,

    "TimestampTieGroups":
        timestamp_tie_groups,

    "TimestampTiedBuilds":
        timestamp_tied_builds,

    "RawExecutionRows":
        raw_profile[
            "Rows"
        ],

    "RawTrainingRows":
        raw_profile[
            "TrainingRows"
        ],

    "RawEvaluationRows":
        raw_profile[
            "EvaluationRows"
        ],

    "RawTrainingPasses":
        raw_profile[
            "TrainingPasses"
        ],

    "RawEvaluationPasses":
        raw_profile[
            "EvaluationPasses"
        ],

    "RawTrainFailures":
        raw_profile[
            "TrainingFailures"
        ],

    "RawEvaluationFailures":
        raw_profile[
            "EvaluationFailures"
        ],

    "RawFailingTrainingBuilds":
        raw_profile[
            "FailingTrainingBuilds"
        ],

    "RawFailingEvaluationBuilds":
        raw_profile[
            "FailingEvaluationBuilds"
        ],

    "RawUnlinkedRows":
        raw_profile[
            "UnlinkedRows"
        ],

    "ModelReadyRows":
        model_profile[
            "Rows"
        ],

    "ModelTrainingRows":
        model_profile[
            "TrainingRows"
        ],

    "ModelEvaluationRows":
        model_profile[
            "EvaluationRows"
        ],

    "ModelTrainingPasses":
        model_profile[
            "TrainingPasses"
        ],

    "ModelEvaluationPasses":
        model_profile[
            "EvaluationPasses"
        ],

    "ModelTrainFailures":
        model_profile[
            "TrainingFailures"
        ],

    "ModelEvaluationFailures":
        model_profile[
            "EvaluationFailures"
        ],

    "ModelFailingTrainingBuilds":
        model_profile[
            "FailingTrainingBuilds"
        ],

    "ModelFailingEvaluationBuilds":
        model_profile[
            "FailingEvaluationBuilds"
        ],

    "ModelUnlinkedRows":
        model_profile[
            "UnlinkedRows"
        ],
}

if actual_dimensions != step1a_dimensions:
    mismatch_records = []

    for key in sorted(
        EXPECTED_DIMENSION_KEYS
    ):
        if int(
            actual_dimensions[
                key
            ]
        ) != int(
            step1a_dimensions[
                key
            ]
        ):
            mismatch_records.append({
                "Metric":
                    key,

                "Step1A":
                    int(
                        step1a_dimensions[
                            key
                        ]
                    ),

                "Step1BRecomputed":
                    int(
                        actual_dimensions[
                            key
                        ]
                    ),
            })

    print(
        "\nDimension mismatches:"
    )

    display(
        pd.DataFrame(
            mismatch_records
        )
    )

    raise RuntimeError(
        "Project 24 Step 1B dimensions do not reproduce the frozen Step 1A contract."
    )


# --------------------------------------------------------------------------------------------------
# 12. STRICT ELIGIBILITY RECONFIRMATION
# --------------------------------------------------------------------------------------------------

model_training_both_classes = bool(
    actual_dimensions[
        "ModelTrainFailures"
    ] > 0
    and actual_dimensions[
        "ModelTrainingPasses"
    ] > 0
)

strict_eligibility_pass = bool(
    actual_dimensions[
        "RawTrainingRows"
    ] > 0
    and actual_dimensions[
        "RawEvaluationRows"
    ] > 0
    and actual_dimensions[
        "RawFailingTrainingBuilds"
    ] >= MIN_FAILING_TRAINING_BUILDS
    and actual_dimensions[
        "RawEvaluationFailures"
    ] > 0
    and actual_dimensions[
        "ModelTrainingRows"
    ] > 0
    and actual_dimensions[
        "ModelEvaluationRows"
    ] > 0
    and model_training_both_classes
    and actual_dimensions[
        "ModelEvaluationFailures"
    ] > 0
    and actual_dimensions[
        "RawUnlinkedRows"
    ] == 0
    and actual_dimensions[
        "ModelUnlinkedRows"
    ] == 0
)

if not strict_eligibility_pass:
    raise RuntimeError(
        "Project 24 no longer satisfies the strict protocol eligibility contract."
    )


# --------------------------------------------------------------------------------------------------
# 13. VALIDATION TABLE
# --------------------------------------------------------------------------------------------------

validation_records = []

add_check(
    validation_records,
    "Project number",
    PROJECT_NUMBER,
    PROJECT_NUMBER,
    True,
)

add_check(
    validation_records,
    "Project identity",
    PROJECT_NAME,
    provisional_selection.get(
        "Project"
    ),
    provisional_selection.get(
        "Project"
    )
    == PROJECT_NAME,
)

add_check(
    validation_records,
    "Project slug",
    PROJECT_SLUG,
    provisional_selection.get(
        "ProjectSlug"
    ),
    provisional_selection.get(
        "ProjectSlug"
    )
    == PROJECT_SLUG,
)

add_check(
    validation_records,
    "Candidate rank",
    CANDIDATE_RANK,
    int(
        provisional_selection.get(
            "CandidateRank",
            -1,
        )
    ),
    int(
        provisional_selection.get(
            "CandidateRank",
            -1,
        )
    )
    == CANDIDATE_RANK,
)

add_check(
    validation_records,
    "Archive SHA-256",
    EXPECTED_ARCHIVE_SHA256,
    archive_sha256,
    archive_sha256
    == EXPECTED_ARCHIVE_SHA256,
)

add_check(
    validation_records,
    "Completion registry SHA-256",
    EXPECTED_REGISTRY_SHA256,
    registry_sha256_before,
    registry_sha256_before
    == EXPECTED_REGISTRY_SHA256,
)

add_check(
    validation_records,
    "Completion registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)

add_check(
    validation_records,
    "Project 24 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)

add_check(
    validation_records,
    "Required source files missing",
    0,
    len(
        missing_required_source_files
    ),
    len(
        missing_required_source_files
    )
    == 0,
)

add_check(
    validation_records,
    "Source files",
    "> 0",
    source_file_count,
    source_file_count
    > 0,
)

add_check(
    validation_records,
    "Source bytes",
    "> 0",
    source_bytes,
    source_bytes
    > 0,
)

add_check(
    validation_records,
    "Builds",
    step1a_dimensions[
        "Builds"
    ],
    actual_dimensions[
        "Builds"
    ],
    actual_dimensions[
        "Builds"
    ]
    == step1a_dimensions[
        "Builds"
    ],
)

add_check(
    validation_records,
    "Training builds",
    step1a_dimensions[
        "TrainingBuilds"
    ],
    actual_dimensions[
        "TrainingBuilds"
    ],
    actual_dimensions[
        "TrainingBuilds"
    ]
    == step1a_dimensions[
        "TrainingBuilds"
    ],
)

add_check(
    validation_records,
    "Evaluation builds",
    step1a_dimensions[
        "EvaluationBuilds"
    ],
    actual_dimensions[
        "EvaluationBuilds"
    ],
    actual_dimensions[
        "EvaluationBuilds"
    ]
    == step1a_dimensions[
        "EvaluationBuilds"
    ],
)

add_check(
    validation_records,
    "Timestamp tie groups",
    step1a_dimensions[
        "TimestampTieGroups"
    ],
    actual_dimensions[
        "TimestampTieGroups"
    ],
    actual_dimensions[
        "TimestampTieGroups"
    ]
    == step1a_dimensions[
        "TimestampTieGroups"
    ],
)

add_check(
    validation_records,
    "Timestamp tied builds",
    step1a_dimensions[
        "TimestampTiedBuilds"
    ],
    actual_dimensions[
        "TimestampTiedBuilds"
    ],
    actual_dimensions[
        "TimestampTiedBuilds"
    ]
    == step1a_dimensions[
        "TimestampTiedBuilds"
    ],
)

for key in [
    "RawExecutionRows",
    "RawTrainingRows",
    "RawEvaluationRows",
    "RawTrainingPasses",
    "RawEvaluationPasses",
    "RawTrainFailures",
    "RawEvaluationFailures",
    "RawFailingTrainingBuilds",
    "RawFailingEvaluationBuilds",
    "RawUnlinkedRows",
    "ModelReadyRows",
    "ModelTrainingRows",
    "ModelEvaluationRows",
    "ModelTrainingPasses",
    "ModelEvaluationPasses",
    "ModelTrainFailures",
    "ModelEvaluationFailures",
    "ModelFailingTrainingBuilds",
    "ModelFailingEvaluationBuilds",
    "ModelUnlinkedRows",
]:
    add_check(
        validation_records,
        key,
        step1a_dimensions[
            key
        ],
        actual_dimensions[
            key
        ],
        actual_dimensions[
            key
        ]
        == step1a_dimensions[
            key
        ],
    )

add_check(
    validation_records,
    "Model training both classes",
    True,
    model_training_both_classes,
    model_training_both_classes,
)

add_check(
    validation_records,
    "Strict protocol eligibility",
    True,
    strict_eligibility_pass,
    strict_eligibility_pass,
)

add_check(
    validation_records,
    "Minimum raw failing training builds",
    f">= {MIN_FAILING_TRAINING_BUILDS}",
    actual_dimensions[
        "RawFailingTrainingBuilds"
    ],
    actual_dimensions[
        "RawFailingTrainingBuilds"
    ]
    >= MIN_FAILING_TRAINING_BUILDS,
)

add_check(
    validation_records,
    "Raw evaluation failures",
    "> 0",
    actual_dimensions[
        "RawEvaluationFailures"
    ],
    actual_dimensions[
        "RawEvaluationFailures"
    ]
    > 0,
)

add_check(
    validation_records,
    "Model evaluation failures",
    "> 0",
    actual_dimensions[
        "ModelEvaluationFailures"
    ],
    actual_dimensions[
        "ModelEvaluationFailures"
    ]
    > 0,
)

add_check(
    validation_records,
    "Raw unlinked rows",
    0,
    actual_dimensions[
        "RawUnlinkedRows"
    ],
    actual_dimensions[
        "RawUnlinkedRows"
    ]
    == 0,
)

add_check(
    validation_records,
    "Model unlinked rows",
    0,
    actual_dimensions[
        "ModelUnlinkedRows"
    ],
    actual_dimensions[
        "ModelUnlinkedRows"
    ]
    == 0,
)

add_check(
    validation_records,
    "Step 1A registry modified",
    False,
    bool(
        provisional_selection.get(
            "RegistryModified",
            True,
        )
    ),
    not bool(
        provisional_selection.get(
            "RegistryModified",
            True,
        )
    ),
)

add_check(
    validation_records,
    "Step 1A prior-project output access",
    False,
    bool(
        provisional_selection.get(
            "PriorProjectConditionOutputsAccessed",
            True,
        )
    ),
    not bool(
        provisional_selection.get(
            "PriorProjectConditionOutputsAccessed",
            True,
        )
    ),
)

add_check(
    validation_records,
    "Models fitted",
    False,
    False,
    True,
)

add_check(
    validation_records,
    "Experiment started",
    False,
    False,
    True,
)

validation = pd.DataFrame(
    validation_records
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]

print(
    "\nProject 24 Step 1B validation:"
)

display(
    validation
)

if not failed_validation.empty:
    print(
        "\nFailed Project 24 Step 1B checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 24 STEP 1B VALIDATION FAILED. "
        "No selection checkpoint was written."
    )


# --------------------------------------------------------------------------------------------------
# 14. WRITE FROZEN STEP 1B OUTPUTS
# --------------------------------------------------------------------------------------------------

atomic_write_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    source_manifest,
)

atomic_write_csv(
    FIXED_CHRONOLOGY_PATH,
    fixed_chronology,
)

atomic_write_csv(
    SOURCE_SCHEMA_SNAPSHOT_PATH,
    source_schema_snapshot,
)

atomic_write_csv(
    STEP1B_VALIDATION_PATH,
    validation,
)

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

output_paths = [
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SOURCE_SCHEMA_SNAPSHOT_PATH,
    STEP1B_VALIDATION_PATH,
]

output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]

report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CandidateRank":
        CANDIDATE_RANK,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "ArchiveSHA256":
        archive_sha256,

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "Step1AStatusPath":
        str(
            STEP1A_STATUS_PATH
        ),

    "Step1AStatusSHA256":
        step1a_status_sha256,

    "Step1AReportPath":
        str(
            STEP1A_REPORT_PATH
        ),

    "Step1AReportSHA256":
        step1a_report_sha256,

    "ProvisionalSelectionPath":
        str(
            PROVISIONAL_SELECTION_PATH
        ),

    "ProvisionalSelectionSHA256":
        provisional_selection_sha256,

    "EligibleRankedPath":
        str(
            ELIGIBLE_RANKED_PATH
        ),

    "EligibleRankedSHA256":
        sha256_file(
            ELIGIBLE_RANKED_PATH
        ),

    "SourceDirectory":
        str(
            SOURCE_DIRECTORY
        ),

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "FrozenSourceManifestPath":
        str(
            FROZEN_SOURCE_MANIFEST_PATH
        ),

    "FixedChronologyPath":
        str(
            FIXED_CHRONOLOGY_PATH
        ),

    "SourceSchemaSnapshotPath":
        str(
            SOURCE_SCHEMA_SNAPSHOT_PATH
        ),

    "Dimensions":
        actual_dimensions,

    "StrictEligibility": {
        "MinimumRawFailingTrainingBuilds":
            MIN_FAILING_TRAINING_BUILDS,

        "RawFailingTrainingBuilds":
            actual_dimensions[
                "RawFailingTrainingBuilds"
            ],

        "RawEvaluationFailures":
            actual_dimensions[
                "RawEvaluationFailures"
            ],

        "ModelTrainingPasses":
            actual_dimensions[
                "ModelTrainingPasses"
            ],

        "ModelTrainingFailures":
            actual_dimensions[
                "ModelTrainFailures"
            ],

        "ModelTrainingBothClasses":
            model_training_both_classes,

        "ModelEvaluationFailures":
            actual_dimensions[
                "ModelEvaluationFailures"
            ],

        "RawUnlinkedRows":
            actual_dimensions[
                "RawUnlinkedRows"
            ],

        "ModelUnlinkedRows":
            actual_dimensions[
                "ModelUnlinkedRows"
            ],

        "Pass":
            strict_eligibility_pass,
    },

    "ChronologyContract": {
        "Split":
            "75/25 chronological",

        "TrainingFraction":
            0.75,

        "TrainingBuildCount":
            training_build_count,

        "EvaluationBuildCount":
            evaluation_build_count,

        "Ordering":
            [
                f"{started_at_column} ascending",
                f"{build_id_column} descending within exact timestamp ties",
            ],

        "TimestampTieGroups":
            timestamp_tie_groups,

        "TimestampTiedBuilds":
            timestamp_tied_builds,
    },

    "ResolvedColumns": {
        "BuildIDColumn":
            build_id_column,

        "StartedAtColumn":
            started_at_column,

        "ExecutionBuildColumn":
            execution_build_column,

        "ExecutionVerdictColumn":
            execution_verdict_column,

        "DatasetBuildColumn":
            dataset_build_column,

        "DatasetVerdictColumn":
            dataset_verdict_column,
    },

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "OutputManifest":
        output_manifest,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ModelsFitted":
        False,

    "Project24ExperimentStarted":
        False,
}

atomic_write_json(
    STEP1B_REPORT_PATH,
    report_payload,
)

checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "SelectionCheckpoint":
        True,

    "DoNotChangeProjectIdentity":
        True,

    "DoNotChangeSourceManifest":
        True,

    "DoNotChangeChronology":
        True,

    "DoNotChangeBuildPartitions":
        True,
}

atomic_write_json(
    SELECTION_CHECKPOINT_PATH,
    checkpoint_payload,
)

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "CandidateRank":
        CANDIDATE_RANK,

    "SelectionState":
        "FINAL_AND_FROZEN",

    "Status":
        STEP1B_PASS_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceFiles":
        source_file_count,

    "SourceBytes":
        source_bytes,

    "SourceRootSHA256":
        source_root_sha256,

    "Builds":
        number_of_builds,

    "TrainingBuilds":
        training_build_count,

    "EvaluationBuilds":
        evaluation_build_count,

    "SelectionCheckpoint":
        str(
            SELECTION_CHECKPOINT_PATH
        ),

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "ModelsFitted":
        False,

    "Project24ExperimentStarted":
        False,
}

atomic_write_json(
    STEP1B_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 15. FINAL READBACK + IMMUTABILITY PROOF
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)

if registry_sha256_after != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 24 Step 1B."
    )

final_manifest_records = []

for row in source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIRECTORY
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise RuntimeError(
            "A frozen Project 24 source file disappeared:\n"
            f"{source_path}"
        )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })

final_source_manifest = pd.DataFrame(
    final_manifest_records
)

final_source_root_sha256 = canonical_root_hash(
    final_source_manifest
)

if final_source_root_sha256 != source_root_sha256:
    raise RuntimeError(
        "Project 24 source root changed during Step 1B.\n"
        f"Frozen: {source_root_sha256}\n"
        f"Final:  {final_source_root_sha256}"
    )

if not (
    final_source_manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).reset_index(
        drop=True
    ).equals(
        source_manifest.sort_values(
            "RelativePath",
            kind="mergesort",
        ).reset_index(
            drop=True
        )
    )
):
    raise RuntimeError(
        "Project 24 source manifest changed during Step 1B."
    )

checkpoint_readback = load_json(
    SELECTION_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP1B_STATUS_PATH
)

report_readback = load_json(
    STEP1B_REPORT_PATH
)

for label, payload in [
    (
        "checkpoint",
        checkpoint_readback,
    ),
    (
        "status",
        status_readback,
    ),
    (
        "report",
        report_readback,
    ),
]:
    if payload.get(
        "Status"
    ) != STEP1B_PASS_STATUS:
        raise RuntimeError(
            f"Project 24 Step 1B {label} readback is not PASS."
        )

    if payload.get(
        "Project"
    ) != PROJECT_NAME:
        raise RuntimeError(
            f"Project 24 Step 1B {label} identity differs."
        )

if checkpoint_readback.get(
    "SourceRootSHA256"
) != source_root_sha256:
    raise RuntimeError(
        "Selection checkpoint source-root SHA readback differs."
    )

if checkpoint_readback.get(
    "Dimensions"
) != actual_dimensions:
    raise RuntimeError(
        "Selection checkpoint dimension readback differs."
    )

for frozen_path in [
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    SOURCE_SCHEMA_SNAPSHOT_PATH,
    STEP1B_VALIDATION_PATH,
    STEP1B_REPORT_PATH,
    STEP1B_STATUS_PATH,
    SELECTION_CHECKPOINT_PATH,
]:
    if not frozen_path.is_file():
        raise RuntimeError(
            f"Project 24 frozen Step 1B output is missing: {frozen_path}"
        )


# --------------------------------------------------------------------------------------------------
# 16. RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\nFrozen source manifest:"
)

display(
    source_manifest
)

print(
    "\nFrozen Project 24 dimensions:"
)

display(
    pd.DataFrame(
        [
            {
                "Metric":
                    key,

                "Value":
                    actual_dimensions[
                        key
                    ],
            }
            for key in sorted(
                actual_dimensions
            )
        ]
    )
)

print(
    "\nFrozen chronology sample:"
)

display(
    pd.concat(
        [
            fixed_chronology.head(
                10
            ),
            fixed_chronology.tail(
                10
            ),
        ],
        ignore_index=True,
    )
)

print(
    "\n"
    + "=" * 136
)

print(
    "=== PROJECT 24 CELL 3 / STEP 1B RESULT ==="
)

print(
    "=" * 136
)

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Candidate rank:",
    CANDIDATE_RANK,
)

print(
    "Selection state:",
    "FINAL_AND_FROZEN",
)

print()

print(
    "Frozen source:"
)

print(
    "Source directory:",
    SOURCE_DIRECTORY,
)

print(
    "Source files:",
    source_file_count,
)

print(
    "Source bytes:",
    source_bytes,
)

print(
    "Source root SHA-256:",
    source_root_sha256,
)

print()

print(
    "Chronology:"
)

print(
    "Builds:",
    number_of_builds,
)

print(
    "Training / evaluation builds:",
    training_build_count,
    "/",
    evaluation_build_count,
)

print(
    "Timestamp tie groups / tied builds:",
    timestamp_tie_groups,
    "/",
    timestamp_tied_builds,
)

print()

print(
    "Raw cohort:"
)

print(
    "Rows:",
    actual_dimensions[
        "RawExecutionRows"
    ],
)

print(
    "Training / evaluation rows:",
    actual_dimensions[
        "RawTrainingRows"
    ],
    "/",
    actual_dimensions[
        "RawEvaluationRows"
    ],
)

print(
    "Training / evaluation failures:",
    actual_dimensions[
        "RawTrainFailures"
    ],
    "/",
    actual_dimensions[
        "RawEvaluationFailures"
    ],
)

print(
    "Failing training / evaluation builds:",
    actual_dimensions[
        "RawFailingTrainingBuilds"
    ],
    "/",
    actual_dimensions[
        "RawFailingEvaluationBuilds"
    ],
)

print(
    "Unlinked rows:",
    actual_dimensions[
        "RawUnlinkedRows"
    ],
)

print()

print(
    "Model-ready cohort:"
)

print(
    "Rows:",
    actual_dimensions[
        "ModelReadyRows"
    ],
)

print(
    "Training / evaluation rows:",
    actual_dimensions[
        "ModelTrainingRows"
    ],
    "/",
    actual_dimensions[
        "ModelEvaluationRows"
    ],
)

print(
    "Training passes / failures:",
    actual_dimensions[
        "ModelTrainingPasses"
    ],
    "/",
    actual_dimensions[
        "ModelTrainFailures"
    ],
)

print(
    "Evaluation passes / failures:",
    actual_dimensions[
        "ModelEvaluationPasses"
    ],
    "/",
    actual_dimensions[
        "ModelEvaluationFailures"
    ],
)

print(
    "Failing training / evaluation builds:",
    actual_dimensions[
        "ModelFailingTrainingBuilds"
    ],
    "/",
    actual_dimensions[
        "ModelFailingEvaluationBuilds"
    ],
)

print(
    "Unlinked rows:",
    actual_dimensions[
        "ModelUnlinkedRows"
    ],
)

print()

print(
    "Strict eligibility:"
)

print(
    "Model training both classes:",
    model_training_both_classes,
)

print(
    "Protocol eligible:",
    strict_eligibility_pass,
)

print()

print(
    "Isolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–23 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Models fitted:",
    False,
)

print(
    "Experiment started:",
    False,
)

print()

print(
    "Selection checkpoint:",
    SELECTION_CHECKPOINT_PATH,
)

print(
    "Selection checkpoint SHA-256:",
    selection_checkpoint_sha256,
)

print()

print(
    "Next required step:",
    "PROJECT 24 CELL 4 / STEP 2A — SOURCE SCHEMA AND JOIN STRUCTURE VALIDATION",
)

print(
    "STATUS:",
    STEP1B_PASS_STATUS,
)

print(
    "=" * 136
)


=== PROJECT 24 CELL 3 / STEP 1B: FINAL SELECTION AND SOURCE FREEZE ===

Project 24 Step 1B validation:


,Check,Expected,Actual,Pass
0,Project number,24,24,True
1,Project identity,SonarSource@sonarqube,SonarSource@sonarqube,True
2,Project slug,SonarSource__sonarqube,SonarSource__sonarqube,True
3,Candidate rank,1,1,True
4,Archive SHA-256,92af159c116e06e98d7c8348605adb6e9acb25aad982a1...,92af159c116e06e98d7c8348605adb6e9acb25aad982a1...,True
5,Completion registry SHA-256,c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179d...,c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179d...,True
6,Completion registry rows,23,23,True
7,Project 24 registry rows,0,0,True
8,Required source files missing,0,0,True
9,Source files,> 0,6,True



Frozen source manifest:


,RelativePath,SizeBytes,SHA256
0,builds.csv,449879,c4043e34fdd00f6588d5ed03b70f538985070f94d6a78e...
1,contributors.csv,17073,49afc5ba5daadb1b8b44e940f030fae90d6544669f3414...
2,dataset.csv,164381050,7e76072cfa11741ae68082af09751c7e822ff14c47805c...
3,entity_change_history.csv,34990229,2a6654bc30eb785a7eaacc69b3fff83237815453009b55...
4,exe.csv,166971123,9354c37437ace378826059b1d110ee9a6730b86f2652a4...
5,id_map.csv,7693898,a1203963573b25d9652e2b3df849d32d77e0aec9483bc2...



Frozen Project 24 dimensions:


,Metric,Value
0,Builds,4286
1,EvaluationBuilds,1072
2,ModelEvaluationFailures,20
3,ModelEvaluationPasses,18834
4,ModelEvaluationRows,18854
5,ModelFailingEvaluationBuilds,17
6,ModelFailingTrainingBuilds,282
7,ModelReadyRows,224550
8,ModelTrainFailures,1777
9,ModelTrainingPasses,203919



Frozen chronology sample:


,BuildOrder,Build,StartedAtUTC,TimestampTieGroupSize,InTimestampTie,Split
0,1,54731125,2015-03-17T15:04:32.000000+0000,1,False,TRAIN
1,2,54734500,2015-03-17T15:27:56.000000+0000,1,False,TRAIN
2,3,54738299,2015-03-17T15:53:09.000000+0000,1,False,TRAIN
3,4,54745798,2015-03-17T16:46:00.000000+0000,1,False,TRAIN
4,5,54755692,2015-03-17T17:53:28.000000+0000,1,False,TRAIN
5,6,54771865,2015-03-17T19:53:36.000000+0000,1,False,TRAIN
6,7,54775074,2015-03-17T20:20:56.000000+0000,1,False,TRAIN
7,8,54832621,2015-03-18T07:52:53.000000+0000,1,False,TRAIN
8,9,54834358,2015-03-18T08:19:44.000000+0000,1,False,TRAIN
9,10,54836125,2015-03-18T08:40:03.000000+0000,1,False,TRAIN



=== PROJECT 24 CELL 3 / STEP 1B RESULT ===
Project number: 24
Project: SonarSource@sonarqube
Project slug: SonarSource__sonarqube
Candidate rank: 1
Selection state: FINAL_AND_FROZEN

Frozen source:
Source directory: /content/datasets/datasets/SonarSource@sonarqube
Source files: 6
Source bytes: 374503252
Source root SHA-256: 2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70

Chronology:
Builds: 4286
Training / evaluation builds: 3214 / 1072
Timestamp tie groups / tied builds: 23 / 46

Raw cohort:
Rows: 5635027
Training / evaluation rows: 4040801 / 1594226
Training / evaluation failures: 1778 / 20
Failing training / evaluation builds: 283 / 17
Unlinked rows: 0

Model-ready cohort:
Rows: 224550
Training / evaluation rows: 205696 / 18854
Training passes / failures: 203919 / 1777
Evaluation passes / failures: 18834 / 20
Failing training / evaluation builds: 282 / 17
Unlinked rows: 0

Strict eligibility:
Model training both classes: True
Protocol eligible: True

Isolation:
Co

In [4]:
# ==================================================================================================
# PROJECT 24 — CELL 4 / STEP 2A
# SOURCE SCHEMA, BUILD-TEST JOIN, ID-MAP ORIENTATION,
# COMMIT MATCHING, AND BUILD-ENTITY PREFLIGHT
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_24.ipynb.
#
# PURPOSE:
# - validate the frozen Project 24 identity, source root, chronology, and registry state;
# - validate all Build-Test joins and clean-verdict alignment;
# - validate all 19 REC columns and the fixed 151-predictor model schema;
# - resolve id_map.csv orientation without assuming EntityId uniqueness;
# - preserve duplicate EntityId rows as valid path aliases;
# - map build commit tokens to entity-change history;
# - write the build-entity mapping required by clean REC reconstruction;
# - record unmatched commits/builds for explicit Step 2B audit.
#
# PROJECT-24 MEMORY NOTE:
# - exe.csv contains 5,635,027 rows. This cell loads only the five raw columns needed by Step 2A,
#   never the complete raw schema, and releases large intermediates when possible.
#
# SAFETY:
# - no noise injection;
# - no model fitting;
# - no completion-registry write;
# - no prior-project condition-output access;
# - no Project 24 experiment execution.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gc
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd


print("=" * 136)
print("=== PROJECT 24 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

SOURCE_DIR = Path(
    "/content/datasets/datasets/SonarSource@sonarqube"
)

EXPECTED_STEP1B_STATUS = (
    "PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN"
)

STEP2A_STATUS = (
    "PASS_PROJECT_24_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)

EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 23

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 374_503_252

EXPECTED_BUILDS = 4_286
EXPECTED_TRAIN_BUILDS = 3_214
EXPECTED_EVAL_BUILDS = 1_072

EXPECTED_TIMESTAMP_TIE_GROUPS = 23
EXPECTED_TIMESTAMP_TIE_BUILDS = 46

EXPECTED_RAW_ROWS = 5_635_027
EXPECTED_RAW_TRAIN_ROWS = 4_040_801
EXPECTED_RAW_EVAL_ROWS = 1_594_226
EXPECTED_RAW_TRAIN_FAILURES = 1_778
EXPECTED_RAW_EVAL_FAILURES = 20
EXPECTED_RAW_FAILING_TRAIN_BUILDS = 283
EXPECTED_RAW_FAILING_EVAL_BUILDS = 17

EXPECTED_MODEL_ROWS = 224_550
EXPECTED_MODEL_TRAIN_ROWS = 205_696
EXPECTED_MODEL_EVAL_ROWS = 18_854
EXPECTED_MODEL_TRAIN_FAILURES = 1_777
EXPECTED_MODEL_EVAL_FAILURES = 20
EXPECTED_MODEL_FAILING_TRAIN_BUILDS = 282
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 17

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = {
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

FILE_HISTORY_REC = {
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
}

REQUIRED_SOURCE_FILES = [
    "builds.csv",
    "contributors.csv",
    "dataset.csv",
    "entity_change_history.csv",
    "exe.csv",
    "id_map.csv",
]

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_24_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_24_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_24_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_24_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

SOURCE_SCHEMA_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_source_schema_profile.csv"
)

JOIN_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_test_join_audit.csv"
)

REC_CLASS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_rec_feature_classification.csv"
)

BUILD_TOKEN_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_commit_token_profile.csv"
)

COMMIT_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_commit_matching_audit.csv"
)

BUILD_ENTITY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
)

ID_ORIENTATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_id_map_orientation_audit.csv"
)

RESOLVED_ID_MAP_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_resolved_id_map_aliases.csv.gz"
)

ENTITY_ID_AUDIT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_id_map_audit.csv"
)

MAPPING_INCOMPLETE_BUILDS_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
)

VALIDATION_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_validation.csv"
)

SUMMARY_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_entity_mapping_summary.json"
)

REPORT_PATH = (
    PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2a_report.json"
)

STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2a_status.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
    compression=None,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
        compression=compression,
    )

    os.replace(
        temporary_path,
        path,
    )


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(column).strip().lower()
        == str(expected).strip().lower()
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve {label}.\n"
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[0]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} has "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        line = (
            f"{row.RelativePath}\0"
            f"{int(row.SizeBytes)}\0"
            f"{str(row.SHA256).lower()}\n"
        )

        digest.update(
            line.encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def normalise_commit(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    text = str(
        value
    ).strip().lower()

    if not text:
        return ""

    matches = re.findall(
        r"[0-9a-f]{7,64}",
        text,
        flags=re.I,
    )

    if matches:
        return matches[
            0
        ].lower()

    return re.sub(
        r"[^a-z0-9]",
        "",
        text,
    )


def extract_commit_tokens(
    value,
):
    if pd.isna(
        value
    ):
        return []

    text = str(
        value
    ).strip()

    if not text:
        return []

    tokens = re.findall(
        r"[0-9a-fA-F]{7,64}",
        text,
    )

    if not tokens:
        tokens = re.split(
            r"[\s,;|#]+",
            text,
        )

    result = []
    seen = set()

    for token in tokens:
        token = normalise_commit(
            token
        )

        if (
            token
            and token not in seen
        ):
            seen.add(
                token
            )
            result.append(
                token
            )

    return result


def integer_candidate(
    values,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    integral = (
        numeric.notna()
        & np.isclose(
            numeric.fillna(
                0
            ).to_numpy(
                dtype=float
            ),
            np.floor(
                numeric.fillna(
                    0
                ).to_numpy(
                    dtype=float
                )
            ),
            rtol=0,
            atol=0,
        )
    )

    output = pd.Series(
        pd.NA,
        index=values.index,
        dtype="Int64",
    )

    output.loc[
        integral
    ] = numeric.loc[
        integral
    ].astype(
        "int64"
    )

    return output


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN UPSTREAM STATE
# --------------------------------------------------------------------------------------------------

required_inputs = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
]

missing_inputs = [
    str(
        path
    )
    for path in required_inputs
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Project 24 Step 2A inputs are missing:\n"
        + "\n".join(
            missing_inputs
        )
    )

if not SOURCE_DIR.is_dir():
    raise FileNotFoundError(
        f"Project 24 source directory is missing: {SOURCE_DIR}"
    )

for filename in REQUIRED_SOURCE_FILES:
    source_path = (
        SOURCE_DIR
        / filename
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Required Project 24 source file is missing: {source_path}"
        )


selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

if (
    selection_checkpoint_sha256
    != EXPECTED_SELECTION_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 24 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_CHECKPOINT_SHA256}\n"
        f"Actual:   {selection_checkpoint_sha256}"
    )


selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)


if (
    selection_checkpoint.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
    or step1b_status.get(
        "Status"
    )
    != EXPECTED_STEP1B_STATUS
):
    raise RuntimeError(
        "Project 24 Step 1B is not frozen successfully."
    )


if (
    selection_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "Frozen Project 24 identity differs."
    )


if (
    selection_checkpoint.get(
        "ActiveReservations"
    )
    != EXPECTED_ACTIVE_RESERVATIONS
):
    raise RuntimeError(
        "Frozen Project 24 active-reservation state differs."
    )


if (
    selection_checkpoint.get(
        "RuntimePriorityRule"
    )
    != EXPECTED_RUNTIME_PRIORITY_RULE
):
    raise RuntimeError(
        "Frozen Project 24 runtime-priority rule differs."
    )


selection_dimensions = selection_checkpoint.get(
    "Dimensions",
    {}
)

expected_dimension_contract = {
    "Builds":
        EXPECTED_BUILDS,

    "TrainingBuilds":
        EXPECTED_TRAIN_BUILDS,

    "EvaluationBuilds":
        EXPECTED_EVAL_BUILDS,

    "TimestampTieGroups":
        EXPECTED_TIMESTAMP_TIE_GROUPS,

    "TimestampTiedBuilds":
        EXPECTED_TIMESTAMP_TIE_BUILDS,

    "RawExecutionRows":
        EXPECTED_RAW_ROWS,

    "RawTrainingRows":
        EXPECTED_RAW_TRAIN_ROWS,

    "RawEvaluationRows":
        EXPECTED_RAW_EVAL_ROWS,

    "RawTrainFailures":
        EXPECTED_RAW_TRAIN_FAILURES,

    "RawEvaluationFailures":
        EXPECTED_RAW_EVAL_FAILURES,

    "RawFailingTrainingBuilds":
        EXPECTED_RAW_FAILING_TRAIN_BUILDS,

    "RawFailingEvaluationBuilds":
        EXPECTED_RAW_FAILING_EVAL_BUILDS,

    "RawUnlinkedRows":
        0,

    "ModelReadyRows":
        EXPECTED_MODEL_ROWS,

    "ModelTrainingRows":
        EXPECTED_MODEL_TRAIN_ROWS,

    "ModelEvaluationRows":
        EXPECTED_MODEL_EVAL_ROWS,

    "ModelTrainFailures":
        EXPECTED_MODEL_TRAIN_FAILURES,

    "ModelEvaluationFailures":
        EXPECTED_MODEL_EVAL_FAILURES,

    "ModelFailingTrainingBuilds":
        EXPECTED_MODEL_FAILING_TRAIN_BUILDS,

    "ModelFailingEvaluationBuilds":
        EXPECTED_MODEL_FAILING_EVAL_BUILDS,

    "ModelUnlinkedRows":
        0,
}

for metric, expected_value in expected_dimension_contract.items():
    if int(
        selection_dimensions.get(
            metric,
            -1,
        )
    ) != int(
        expected_value
    ):
        raise RuntimeError(
            "Project 24 selection dimension differs.\n"
            f"Metric:   {metric}\n"
            f"Expected: {expected_value}\n"
            f"Actual:   {selection_dimensions.get(metric)}"
        )


# --------------------------------------------------------------------------------------------------
# 5. REGISTRY AND SOURCE IMMUTABILITY PRE-STATE
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(
    REGISTRY_PATH
)

if (
    registry_sha256_before
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry must contain exactly Projects 1–23."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )


if (
    project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 24 is unexpectedly already present in the completion registry."
    )


for (
    predecessor_number,
    predecessor_identity,
) in REQUIRED_REGISTERED_IDENTITIES.items():
    predecessor_rows = registry.loc[
        project_numbers.eq(
            predecessor_number
        )
    ]

    if (
        len(
            predecessor_rows
        )
        != 1
        or str(
            predecessor_rows.iloc[
                0
            ][
                project_column
            ]
        )
        != predecessor_identity
    ):
        raise RuntimeError(
            "Frozen predecessor identity differs.\n"
            f"Project number: {predecessor_number}\n"
            f"Expected:       {predecessor_identity}"
        )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_manifest_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 24 source file is missing: {source_path}"
        )

    current_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_manifest = pd.DataFrame(
    current_manifest_records
)


current_source_root = source_root_hash(
    current_manifest
)


current_source_bytes = int(
    current_manifest[
        "SizeBytes"
    ].sum()
)


if (
    len(
        current_manifest
    )
    != EXPECTED_SOURCE_FILES
    or current_source_bytes
    != EXPECTED_SOURCE_BYTES
    or current_source_root
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 24 source manifest/root differs.\n"
        f"Files expected/actual: {EXPECTED_SOURCE_FILES}/{len(current_manifest)}\n"
        f"Bytes expected/actual: {EXPECTED_SOURCE_BYTES}/{current_source_bytes}\n"
        f"Root expected: {EXPECTED_SOURCE_ROOT_SHA256}\n"
        f"Root actual:   {current_source_root}"
    )


# --------------------------------------------------------------------------------------------------
# 6. RESOLVE SOURCE SCHEMAS
# --------------------------------------------------------------------------------------------------

paths = {
    "builds.csv":
        SOURCE_DIR
        / "builds.csv",

    "contributors.csv":
        SOURCE_DIR
        / "contributors.csv",

    "dataset.csv":
        SOURCE_DIR
        / "dataset.csv",

    "entity_change_history.csv":
        SOURCE_DIR
        / "entity_change_history.csv",

    "exe.csv":
        SOURCE_DIR
        / "exe.csv",

    "id_map.csv":
        SOURCE_DIR
        / "id_map.csv",
}


schema_rows = []
headers = {}


for filename, file_path in paths.items():
    columns = pd.read_csv(
        file_path,
        nrows=0,
    ).columns.tolist()

    headers[
        filename
    ] = columns

    schema_rows.append({
        "File":
            filename,

        "Path":
            str(
                file_path
            ),

        "SizeBytes":
            int(
                file_path.stat().st_size
            ),

        "ColumnCount":
            len(
                columns
            ),

        "ColumnsJSON":
            json.dumps(
                columns,
                ensure_ascii=False,
            ),
    })


source_schema = pd.DataFrame(
    schema_rows
)


build_id_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "id",
    "builds.csv id",
)

build_commit_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "commits",
    "builds.csv commits",
)

build_time_column = resolve_column(
    headers[
        "builds.csv"
    ],
    "started_at",
    "builds.csv started_at",
)


exe_test_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "test",
    "exe.csv test",
)

exe_build_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "build",
    "exe.csv build",
)

exe_job_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "job",
    "exe.csv job",
)

exe_verdict_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "verdict",
    "exe.csv verdict",
)

exe_duration_column = resolve_column(
    headers[
        "exe.csv"
    ],
    "duration",
    "exe.csv duration",
)


dataset_build_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Build",
    "dataset.csv Build",
)

dataset_test_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Test",
    "dataset.csv Test",
)

dataset_verdict_column = resolve_column(
    headers[
        "dataset.csv"
    ],
    "Verdict",
    "dataset.csv Verdict",
)


entity_id_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "EntityId",
    "entity_change_history.csv EntityId",
)

entity_commit_column = resolve_column(
    headers[
        "entity_change_history.csv"
    ],
    "Commit",
    "entity_change_history.csv Commit",
)


id_key_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "key",
    "id_map.csv key",
)

id_value_column = resolve_column(
    headers[
        "id_map.csv"
    ],
    "value",
    "id_map.csv value",
)


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature
    not in headers[
        "dataset.csv"
    ]
]


predictor_columns = [
    column
    for column in headers[
        "dataset.csv"
    ]
    if column
    not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


if missing_rec_features:
    raise RuntimeError(
        "dataset.csv is missing REC columns:\n"
        + "\n".join(
            missing_rec_features
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. LOAD CHRONOLOGY + SOURCE TABLES
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)

required_chronology_columns = {
    "BuildOrder",
    "Build",
    "StartedAtUTC",
    "TimestampTieGroupSize",
    "InTimestampTie",
    "Split",
}

if not required_chronology_columns.issubset(
    chronology.columns
):
    raise RuntimeError(
        "Frozen Project 24 chronology is missing required columns."
    )

chronology[
    "Build"
] = parse_int(
    chronology[
        "Build"
    ],
    "chronology.Build",
)

chronology[
    "BuildOrder"
] = parse_int(
    chronology[
        "BuildOrder"
    ],
    "chronology.BuildOrder",
)

training_builds = set(
    chronology.loc[
        chronology[
            "Split"
        ].eq(
            "TRAIN"
        ),
        "Build",
    ].astype(
        int
    )
)

evaluation_builds = set(
    chronology.loc[
        chronology[
            "Split"
        ].eq(
            "EVAL"
        ),
        "Build",
    ].astype(
        int
    )
)

all_builds = (
    training_builds
    | evaluation_builds
)

build_order = (
    chronology.set_index(
        "Build"
    )[
        "BuildOrder"
    ]
    .astype(
        int
    )
    .to_dict()
)

build_partition = (
    chronology.set_index(
        "Build"
    )[
        "Split"
    ]
    .astype(
        str
    )
    .to_dict()
)


builds = pd.read_csv(
    paths[
        "builds.csv"
    ],
    usecols=[
        build_id_column,
        build_commit_column,
        build_time_column,
    ],
    low_memory=False,
)

builds[
    build_id_column
] = parse_int(
    builds[
        build_id_column
    ],
    "builds.csv.id",
)

builds[
    build_time_column
] = pd.to_datetime(
    builds[
        build_time_column
    ],
    errors="coerce",
    utc=True,
)

if builds[
    build_time_column
].isna().any():
    raise RuntimeError(
        "builds.csv contains invalid timestamps."
    )

if builds[
    build_id_column
].duplicated(
    keep=False
).any():
    raise RuntimeError(
        "builds.csv contains duplicate build IDs."
    )


timestamp_group_sizes = builds.groupby(
    build_time_column
).size()

timestamp_tie_group_count = int(
    timestamp_group_sizes.gt(
        1
    ).sum()
)

timestamp_tie_build_count = int(
    timestamp_group_sizes.loc[
        timestamp_group_sizes.gt(
            1
        )
    ].sum()
)


recomputed_chronology = (
    builds[
        [
            build_id_column,
            build_time_column,
        ]
    ]
    .sort_values(
        [
            build_time_column,
            build_id_column,
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

recomputed_chronology[
    "BuildOrder"
] = np.arange(
    1,
    len(
        recomputed_chronology
    ) + 1,
    dtype=np.int64,
)

recomputed_build_ids = recomputed_chronology[
    build_id_column
].astype(
    int
).tolist()

frozen_build_ids = chronology.sort_values(
    "BuildOrder",
    kind="mergesort",
)[
    "Build"
].astype(
    int
).tolist()

chronology_order_reproduced = (
    recomputed_build_ids
    == frozen_build_ids
)

expected_split_order = np.where(
    np.arange(
        1,
        EXPECTED_BUILDS + 1,
    )
    <= EXPECTED_TRAIN_BUILDS,
    "TRAIN",
    "EVAL",
).tolist()

actual_split_order = chronology.sort_values(
    "BuildOrder",
    kind="mergesort",
)[
    "Split"
].astype(
    str
).tolist()

chronology_partitions_reproduced = (
    actual_split_order
    == expected_split_order
)


print(
    "\nLoading Project 24 model-ready dataset "
    f"({EXPECTED_MODEL_ROWS:,} rows expected)."
)

dataset = pd.read_csv(
    paths[
        "dataset.csv"
    ],
    low_memory=False,
)

dataset[
    dataset_build_column
] = parse_int(
    dataset[
        dataset_build_column
    ],
    "dataset.csv.Build",
)

dataset[
    dataset_test_column
] = parse_int(
    dataset[
        dataset_test_column
    ],
    "dataset.csv.Test",
)

dataset[
    dataset_verdict_column
] = parse_int(
    dataset[
        dataset_verdict_column
    ],
    "dataset.csv.Verdict",
)


print(
    "Loading only required columns from Project 24 raw execution table "
    f"({EXPECTED_RAW_ROWS:,} rows expected)."
)

exe = pd.read_csv(
    paths[
        "exe.csv"
    ],
    usecols=[
        exe_test_column,
        exe_build_column,
        exe_job_column,
        exe_verdict_column,
        exe_duration_column,
    ],
    low_memory=False,
)

exe[
    exe_build_column
] = parse_int(
    exe[
        exe_build_column
    ],
    "exe.csv.build",
)

exe[
    exe_test_column
] = parse_int(
    exe[
        exe_test_column
    ],
    "exe.csv.test",
)

exe[
    exe_verdict_column
] = parse_int(
    exe[
        exe_verdict_column
    ],
    "exe.csv.verdict",
)

exe[
    exe_duration_column
] = pd.to_numeric(
    exe[
        exe_duration_column
    ],
    errors="coerce",
)


entity_history = pd.read_csv(
    paths[
        "entity_change_history.csv"
    ],
    usecols=[
        entity_id_column,
        entity_commit_column,
    ],
    low_memory=False,
)

entity_history[
    entity_id_column
] = parse_int(
    entity_history[
        entity_id_column
    ],
    "entity_change_history.csv.EntityId",
)

entity_history[
    "NormalisedCommit"
] = entity_history[
    entity_commit_column
].map(
    normalise_commit
)

entity_history = entity_history.loc[
    entity_history[
        "NormalisedCommit"
    ].ne("")
].copy()

if entity_history.empty:
    raise RuntimeError(
        "entity_change_history.csv produced no usable commit rows."
    )


id_map = (
    pd.read_csv(
        paths[
            "id_map.csv"
        ],
        dtype=str,
        low_memory=False,
    )
    .fillna("")
)


# --------------------------------------------------------------------------------------------------
# 8. BUILD-TEST JOIN AUDIT
# --------------------------------------------------------------------------------------------------

raw_duplicate_pairs = int(
    exe.duplicated(
        subset=[
            exe_build_column,
            exe_test_column,
        ],
        keep=False,
    ).sum()
)

model_duplicate_pairs = int(
    dataset.duplicated(
        subset=[
            dataset_build_column,
            dataset_test_column,
        ],
        keep=False,
    ).sum()
)

raw_unlinked_build_rows = int(
    (
        ~exe[
            exe_build_column
        ].isin(
            all_builds
        )
    ).sum()
)

model_unlinked_build_rows = int(
    (
        ~dataset[
            dataset_build_column
        ].isin(
            all_builds
        )
    ).sum()
)

raw_training_mask = exe[
    exe_build_column
].isin(
    training_builds
)

raw_evaluation_mask = exe[
    exe_build_column
].isin(
    evaluation_builds
)

model_training_mask = dataset[
    dataset_build_column
].isin(
    training_builds
)

model_evaluation_mask = dataset[
    dataset_build_column
].isin(
    evaluation_builds
)

raw_training_rows = int(
    raw_training_mask.sum()
)

raw_evaluation_rows = int(
    raw_evaluation_mask.sum()
)

raw_training_failures = int(
    (
        raw_training_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)

raw_evaluation_failures = int(
    (
        raw_evaluation_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        )
    ).sum()
)

raw_failing_training_builds = int(
    exe.loc[
        raw_training_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        ),
        exe_build_column,
    ].nunique()
)

raw_failing_evaluation_builds = int(
    exe.loc[
        raw_evaluation_mask
        & exe[
            exe_verdict_column
        ].ne(
            0
        ),
        exe_build_column,
    ].nunique()
)

model_training_rows = int(
    model_training_mask.sum()
)

model_evaluation_rows = int(
    model_evaluation_mask.sum()
)

model_training_failures = int(
    (
        model_training_mask
        & dataset[
            dataset_verdict_column
        ].ne(
            0
        )
    ).sum()
)

model_evaluation_failures = int(
    (
        model_evaluation_mask
        & dataset[
            dataset_verdict_column
        ].ne(
            0
        )
    ).sum()
)

model_failing_training_builds = int(
    dataset.loc[
        model_training_mask
        & dataset[
            dataset_verdict_column
        ].ne(
            0
        ),
        dataset_build_column,
    ].nunique()
)

model_failing_evaluation_builds = int(
    dataset.loc[
        model_evaluation_mask
        & dataset[
            dataset_verdict_column
        ].ne(
            0
        ),
        dataset_build_column,
    ].nunique()
)

duration_array = exe[
    exe_duration_column
].to_numpy(
    dtype=float
)

nonfinite_duration_rows = int(
    (
        ~np.isfinite(
            duration_array
        )
    ).sum()
)

negative_duration_rows = int(
    (
        np.isfinite(
            duration_array
        )
        & (
            duration_array
            < 0
        )
    ).sum()
)


raw_pairs = (
    exe[
        [
            exe_build_column,
            exe_test_column,
            exe_verdict_column,
        ]
    ]
    .rename(
        columns={
            exe_build_column:
                "Build",

            exe_test_column:
                "Test",

            exe_verdict_column:
                "RawVerdict",
        }
    )
)

model_pairs = (
    dataset[
        [
            dataset_build_column,
            dataset_test_column,
            dataset_verdict_column,
        ]
    ]
    .rename(
        columns={
            dataset_build_column:
                "Build",

            dataset_test_column:
                "Test",

            dataset_verdict_column:
                "ModelVerdict",
        }
    )
)

if raw_duplicate_pairs == 0:
    joined = model_pairs.merge(
        raw_pairs,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
else:
    # The validation gate below will fail; avoid obscuring the diagnostic
    # with a pandas merge-validation exception.
    joined = model_pairs.merge(
        raw_pairs,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        indicator=True,
    )

missing_model_raw_links = int(
    joined[
        "_merge"
    ].ne(
        "both"
    ).sum()
)

verdict_mismatches = int(
    joined.loc[
        joined[
            "_merge"
        ].eq(
            "both"
        ),
        "ModelVerdict",
    ].ne(
        joined.loc[
            joined[
                "_merge"
            ].eq(
                "both"
            ),
            "RawVerdict",
        ]
    ).sum()
)


join_audit = pd.DataFrame([
    (
        "RawRows",
        EXPECTED_RAW_ROWS,
        len(
            exe
        ),
    ),

    (
        "RawTrainingRows",
        EXPECTED_RAW_TRAIN_ROWS,
        raw_training_rows,
    ),

    (
        "RawEvaluationRows",
        EXPECTED_RAW_EVAL_ROWS,
        raw_evaluation_rows,
    ),

    (
        "RawTrainingFailures",
        EXPECTED_RAW_TRAIN_FAILURES,
        raw_training_failures,
    ),

    (
        "RawEvaluationFailures",
        EXPECTED_RAW_EVAL_FAILURES,
        raw_evaluation_failures,
    ),

    (
        "RawFailingTrainingBuilds",
        EXPECTED_RAW_FAILING_TRAIN_BUILDS,
        raw_failing_training_builds,
    ),

    (
        "RawFailingEvaluationBuilds",
        EXPECTED_RAW_FAILING_EVAL_BUILDS,
        raw_failing_evaluation_builds,
    ),

    (
        "ModelRows",
        EXPECTED_MODEL_ROWS,
        len(
            dataset
        ),
    ),

    (
        "ModelTrainingRows",
        EXPECTED_MODEL_TRAIN_ROWS,
        model_training_rows,
    ),

    (
        "ModelEvaluationRows",
        EXPECTED_MODEL_EVAL_ROWS,
        model_evaluation_rows,
    ),

    (
        "ModelTrainingFailures",
        EXPECTED_MODEL_TRAIN_FAILURES,
        model_training_failures,
    ),

    (
        "ModelEvaluationFailures",
        EXPECTED_MODEL_EVAL_FAILURES,
        model_evaluation_failures,
    ),

    (
        "ModelFailingTrainingBuilds",
        EXPECTED_MODEL_FAILING_TRAIN_BUILDS,
        model_failing_training_builds,
    ),

    (
        "ModelFailingEvaluationBuilds",
        EXPECTED_MODEL_FAILING_EVAL_BUILDS,
        model_failing_evaluation_builds,
    ),

    (
        "RawDuplicateBuildTestRows",
        0,
        raw_duplicate_pairs,
    ),

    (
        "ModelDuplicateBuildTestRows",
        0,
        model_duplicate_pairs,
    ),

    (
        "MissingModelRawLinks",
        0,
        missing_model_raw_links,
    ),

    (
        "ModelRawVerdictMismatches",
        0,
        verdict_mismatches,
    ),

    (
        "NonFiniteDurationRows",
        0,
        nonfinite_duration_rows,
    ),

    (
        "NegativeDurationRows",
        0,
        negative_duration_rows,
    ),

    (
        "RawUnlinkedBuildRows",
        0,
        raw_unlinked_build_rows,
    ),

    (
        "ModelUnlinkedBuildRows",
        0,
        model_unlinked_build_rows,
    ),
], columns=[
    "Metric",
    "Expected",
    "Actual",
])


# Free the largest join-only intermediates before mapping work.
del raw_pairs
del model_pairs
del joined
gc.collect()


# --------------------------------------------------------------------------------------------------
# 9. REC FEATURE CLASSIFICATION
# --------------------------------------------------------------------------------------------------

rec_classification = pd.DataFrame([
    {
        "Feature":
            feature,

        "VerdictDependent":
            feature
            in VERDICT_DEPENDENT_REC,

        "VerdictIndependent":
            feature
            not in VERDICT_DEPENDENT_REC,

        "FileHistoryFeature":
            feature
            in FILE_HISTORY_REC,
    }
    for feature in REC_FEATURES
])


# --------------------------------------------------------------------------------------------------
# 10. RESOLVE id_map.csv ORIENTATION AND PRESERVE VALID ALIASES
# --------------------------------------------------------------------------------------------------

history_entity_ids = set(
    entity_history[
        entity_id_column
    ].astype(
        int
    ).unique().tolist()
)

orientation_records = []

for candidate_entity_column, candidate_path_column in [
    (
        id_key_column,
        id_value_column,
    ),
    (
        id_value_column,
        id_key_column,
    ),
]:
    candidate_ids = integer_candidate(
        id_map[
            candidate_entity_column
        ]
    )

    integral_rows = int(
        candidate_ids.notna().sum()
    )

    unique_integral_ids = set(
        candidate_ids.dropna().astype(
            int
        ).unique().tolist()
    )

    history_ids_covered = int(
        len(
            history_entity_ids
            & unique_integral_ids
        )
    )

    history_coverage_percent = (
        100.0
        * history_ids_covered
        / len(
            history_entity_ids
        )
        if history_entity_ids
        else 0.0
    )

    candidate_paths = (
        id_map[
            candidate_path_column
        ].astype(
            str
        ).str.strip()
    )

    nonempty_path_rows = int(
        candidate_paths.ne(
            ""
        ).sum()
    )

    orientation_records.append({
        "CandidateEntityIDColumn":
            candidate_entity_column,

        "CandidatePathColumn":
            candidate_path_column,

        "Rows":
            len(
                id_map
            ),

        "IntegralEntityIDRows":
            integral_rows,

        "UniqueIntegralEntityIDs":
            len(
                unique_integral_ids
            ),

        "HistoryEntityIDs":
            len(
                history_entity_ids
            ),

        "HistoryEntityIDsCovered":
            history_ids_covered,

        "HistoryCoveragePercent":
            history_coverage_percent,

        "NonEmptyPathRows":
            nonempty_path_rows,
    })


id_orientation = pd.DataFrame(
    orientation_records
).sort_values(
    [
        "HistoryEntityIDsCovered",
        "IntegralEntityIDRows",
        "NonEmptyPathRows",
        "CandidateEntityIDColumn",
    ],
    ascending=[
        False,
        False,
        False,
        True,
    ],
    kind="mergesort",
).reset_index(
    drop=True
)


if id_orientation.empty:
    raise RuntimeError(
        "id_map orientation audit produced no candidates."
    )


best_orientation = id_orientation.iloc[
    0
]


if (
    len(
        id_orientation
    )
    > 1
    and int(
        id_orientation.iloc[
            0
        ][
            "HistoryEntityIDsCovered"
        ]
    )
    == int(
        id_orientation.iloc[
            1
        ][
            "HistoryEntityIDsCovered"
        ]
    )
    and int(
        id_orientation.iloc[
            0
        ][
            "IntegralEntityIDRows"
        ]
    )
    == int(
        id_orientation.iloc[
            1
        ][
            "IntegralEntityIDRows"
        ]
    )
):
    raise RuntimeError(
        "id_map orientation is ambiguous."
    )


resolved_id_column = str(
    best_orientation[
        "CandidateEntityIDColumn"
    ]
)

resolved_path_column = str(
    best_orientation[
        "CandidatePathColumn"
    ]
)

resolved_entity_ids = integer_candidate(
    id_map[
        resolved_id_column
    ]
)

resolved_paths = (
    id_map[
        resolved_path_column
    ].astype(
        str
    ).str.strip()
)

resolved_id_map = pd.DataFrame({
    "EntityId":
        resolved_entity_ids,

    "Path":
        resolved_paths,
})

resolved_id_map = resolved_id_map.loc[
    resolved_id_map[
        "EntityId"
    ].notna()
    & resolved_id_map[
        "Path"
    ].ne("")
].copy()

resolved_id_map[
    "EntityId"
] = resolved_id_map[
    "EntityId"
].astype(
    "int64"
)

resolved_id_map = (
    resolved_id_map.drop_duplicates(
        subset=[
            "EntityId",
            "Path",
        ],
        keep="first",
    )
    .sort_values(
        [
            "EntityId",
            "Path",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

exact_duplicate_rows = int(
    len(
        id_map
    )
    - len(
        resolved_id_map
    )
)

duplicate_entity_id_rows = int(
    resolved_id_map[
        "EntityId"
    ].duplicated(
        keep=False
    ).sum()
)

entity_ids_with_multiple_paths = int(
    (
        resolved_id_map.groupby(
            "EntityId"
        )[
            "Path"
        ].nunique()
        > 1
    ).sum()
)

paths_with_multiple_ids = int(
    (
        resolved_id_map.groupby(
            "Path"
        )[
            "EntityId"
        ].nunique()
        > 1
    ).sum()
)

maximum_paths_per_entity = int(
    resolved_id_map.groupby(
        "EntityId"
    )[
        "Path"
    ].nunique().max()
    if not resolved_id_map.empty
    else 0
)

resolved_entity_id_set = set(
    resolved_id_map[
        "EntityId"
    ].astype(
        int
    ).unique().tolist()
)

history_entity_ids_missing_from_id_map = sorted(
    history_entity_ids
    - resolved_entity_id_set
)


# --------------------------------------------------------------------------------------------------
# 11. COMMIT TOKEN MATCHING + BUILD-ENTITY PREFLIGHT
# --------------------------------------------------------------------------------------------------

build_token_records = []

for row in builds[
    [
        build_id_column,
        build_commit_column,
    ]
].itertuples(
    index=False,
    name=None,
):
    build_id = int(
        row[
            0
        ]
    )

    tokens = extract_commit_tokens(
        row[
            1
        ]
    )

    for token_order, token in enumerate(
        tokens,
        start=1,
    ):
        build_token_records.append({
            "Build":
                build_id,

            "BuildOrder":
                int(
                    build_order[
                        build_id
                    ]
                ),

            "Split":
                build_partition[
                    build_id
                ],

            "TokenOrder":
                int(
                    token_order
                ),

            "CommitToken":
                token,
        })


build_tokens = pd.DataFrame(
    build_token_records,
    columns=[
        "Build",
        "BuildOrder",
        "Split",
        "TokenOrder",
        "CommitToken",
    ],
)


history_commit_to_entity_ids = (
    entity_history.groupby(
        "NormalisedCommit"
    )[
        entity_id_column
    ]
    .apply(
        lambda values:
            tuple(
                sorted(
                    set(
                        int(
                            value
                        )
                        for value in values
                    )
                )
            )
    )
    .to_dict()
)

unique_history_commits = sorted(
    history_commit_to_entity_ids.keys()
)

history_commit_set = set(
    unique_history_commits
)


def resolve_commit_token(
    token,
):
    if token in history_commit_set:
        return (
            "EXACT",
            token,
            1,
        )

    # Unique-prefix matching is intentionally symmetric:
    # a short build token may prefix a longer history commit,
    # or a shorter stored history token may prefix a longer build token.
    matches = [
        commit
        for commit in unique_history_commits
        if (
            commit.startswith(
                token
            )
            or token.startswith(
                commit
            )
        )
    ]

    if len(
        matches
    ) == 1:
        return (
            "PREFIX",
            matches[
                0
            ],
            1,
        )

    if len(
        matches
    ) == 0:
        return (
            "UNMATCHED",
            "",
            0,
        )

    return (
        "AMBIGUOUS",
        "",
        len(
            matches
        ),
    )


commit_matching_started = time.perf_counter()

commit_audit_records = []

for row in build_tokens.itertuples(
    index=False
):
    (
        match_type,
        matched_commit,
        candidate_count,
    ) = resolve_commit_token(
        row.CommitToken
    )

    entity_ids = (
        history_commit_to_entity_ids.get(
            matched_commit,
            (),
        )
        if matched_commit
        else ()
    )

    commit_audit_records.append({
        "Build":
            int(
                row.Build
            ),

        "BuildOrder":
            int(
                row.BuildOrder
            ),

        "Split":
            str(
                row.Split
            ),

        "TokenOrder":
            int(
                row.TokenOrder
            ),

        "CommitToken":
            str(
                row.CommitToken
            ),

        "MatchType":
            match_type,

        "MatchedCommit":
            matched_commit,

        "CandidateMatches":
            int(
                candidate_count
            ),

        "MatchedEntityCount":
            int(
                len(
                    entity_ids
                )
            ),
    })


commit_audit = pd.DataFrame(
    commit_audit_records,
    columns=[
        "Build",
        "BuildOrder",
        "Split",
        "TokenOrder",
        "CommitToken",
        "MatchType",
        "MatchedCommit",
        "CandidateMatches",
        "MatchedEntityCount",
    ],
)


build_entity_records = []

for row in commit_audit.itertuples(
    index=False
):
    if row.MatchType not in {
        "EXACT",
        "PREFIX",
    }:
        continue

    entity_ids = history_commit_to_entity_ids.get(
        row.MatchedCommit,
        (),
    )

    for entity_id in entity_ids:
        build_entity_records.append({
            "Build":
                int(
                    row.Build
                ),

            "BuildOrder":
                int(
                    row.BuildOrder
                ),

            "Split":
                str(
                    row.Split
                ),

            "CommitToken":
                str(
                    row.CommitToken
                ),

            "MatchedCommit":
                str(
                    row.MatchedCommit
                ),

            "MatchType":
                str(
                    row.MatchType
                ),

            "EntityId":
                int(
                    entity_id
                ),
        })


build_entity = pd.DataFrame(
    build_entity_records,
    columns=[
        "Build",
        "BuildOrder",
        "Split",
        "CommitToken",
        "MatchedCommit",
        "MatchType",
        "EntityId",
    ],
)

if not build_entity.empty:
    build_entity = (
        build_entity.drop_duplicates(
            subset=[
                "Build",
                "EntityId",
            ],
            keep="first",
        )
        .sort_values(
            [
                "BuildOrder",
                "EntityId",
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


commit_matching_seconds = (
    time.perf_counter()
    - commit_matching_started
)


exact_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "EXACT"
    ).sum()
)

prefix_matches = int(
    commit_audit[
        "MatchType"
    ].eq(
        "PREFIX"
    ).sum()
)

unmatched_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "UNMATCHED"
    ).sum()
)

ambiguous_tokens = int(
    commit_audit[
        "MatchType"
    ].eq(
        "AMBIGUOUS"
    ).sum()
)

matched_token_rows = int(
    exact_matches
    + prefix_matches
)

commit_coverage_percent = (
    100.0
    * matched_token_rows
    / len(
        build_tokens
    )
    if len(
        build_tokens
    )
    else 0.0
)


builds_with_tokens = set(
    build_tokens[
        "Build"
    ].astype(
        int
    ).unique().tolist()
)

builds_without_tokens = sorted(
    all_builds
    - builds_with_tokens
)

builds_with_matched_commit = set(
    commit_audit.loc[
        commit_audit[
            "MatchType"
        ].isin(
            [
                "EXACT",
                "PREFIX",
            ]
        ),
        "Build",
    ].astype(
        int
    ).unique().tolist()
)

builds_with_entities = set(
    build_entity[
        "Build"
    ].astype(
        int
    ).unique().tolist()
    if not build_entity.empty
    else []
)

builds_without_entities = sorted(
    all_builds
    - builds_with_entities
)

mapped_entity_ids = set(
    build_entity[
        "EntityId"
    ].astype(
        int
    ).unique().tolist()
    if not build_entity.empty
    else []
)

mapped_entity_ids_missing_from_id_map = sorted(
    mapped_entity_ids
    - resolved_entity_id_set
)


# Build-level explicit mapping-boundary audit for Step 2B.
token_group = (
    commit_audit.groupby(
        "Build"
    )
    if not commit_audit.empty
    else None
)

mapping_records = []

for build_id in sorted(
    all_builds,
    key=lambda value:
        build_order[
            value
        ],
):
    if (
        token_group is not None
        and build_id
        in token_group.groups
    ):
        group = token_group.get_group(
            build_id
        )

        token_rows = int(
            len(
                group
            )
        )

        exact_count = int(
            group[
                "MatchType"
            ].eq(
                "EXACT"
            ).sum()
        )

        prefix_count = int(
            group[
                "MatchType"
            ].eq(
                "PREFIX"
            ).sum()
        )

        unmatched_count = int(
            group[
                "MatchType"
            ].eq(
                "UNMATCHED"
            ).sum()
        )

        ambiguous_count = int(
            group[
                "MatchType"
            ].eq(
                "AMBIGUOUS"
            ).sum()
        )
    else:
        token_rows = 0
        exact_count = 0
        prefix_count = 0
        unmatched_count = 0
        ambiguous_count = 0

    matched_entity_rows = int(
        build_entity[
            "Build"
        ].eq(
            build_id
        ).sum()
        if not build_entity.empty
        else 0
    )

    reasons = []

    if token_rows == 0:
        reasons.append(
            "NO_COMMIT_TOKEN"
        )

    if unmatched_count:
        reasons.append(
            "UNMATCHED_COMMIT_TOKEN"
        )

    if ambiguous_count:
        reasons.append(
            "AMBIGUOUS_COMMIT_TOKEN"
        )

    if (
        token_rows > 0
        and exact_count
        + prefix_count
        == 0
    ):
        reasons.append(
            "NO_MATCHED_COMMIT"
        )

    if matched_entity_rows == 0:
        reasons.append(
            "NO_MAPPED_ENTITY"
        )

    if reasons:
        mapping_records.append({
            "Build":
                int(
                    build_id
                ),

            "BuildOrder":
                int(
                    build_order[
                        build_id
                    ]
                ),

            "Split":
                str(
                    build_partition[
                        build_id
                    ]
                ),

            "CommitTokenRows":
                token_rows,

            "ExactMatches":
                exact_count,

            "PrefixMatches":
                prefix_count,

            "UnmatchedTokens":
                unmatched_count,

            "AmbiguousTokens":
                ambiguous_count,

            "MatchedEntityRows":
                matched_entity_rows,

            "Reasons":
                ";".join(
                    reasons
                ),
        })


mapping_incomplete_frame = pd.DataFrame(
    mapping_records,
    columns=[
        "Build",
        "BuildOrder",
        "Split",
        "CommitTokenRows",
        "ExactMatches",
        "PrefixMatches",
        "UnmatchedTokens",
        "AmbiguousTokens",
        "MatchedEntityRows",
        "Reasons",
    ],
)

mapping_incomplete_builds = set(
    mapping_incomplete_frame[
        "Build"
    ].astype(
        int
    ).tolist()
    if not mapping_incomplete_frame.empty
    else []
)


entity_id_audit = pd.DataFrame([
    {
        "Metric":
            "EntityHistoryUniqueEntityIDs",

        "Value":
            len(
                history_entity_ids
            ),
    },

    {
        "Metric":
            "ResolvedIDMapRows",

        "Value":
            len(
                resolved_id_map
            ),
    },

    {
        "Metric":
            "ResolvedIDMapUniqueEntityIDs",

        "Value":
            int(
                resolved_id_map[
                    "EntityId"
                ].nunique()
            ),
    },

    {
        "Metric":
            "HistoryEntityIDsMissingFromIDMap",

        "Value":
            len(
                history_entity_ids_missing_from_id_map
            ),
    },

    {
        "Metric":
            "DuplicateEntityIDRowsAcceptedAsAliases",

        "Value":
            duplicate_entity_id_rows,
    },

    {
        "Metric":
            "EntityIDsWithMultiplePaths",

        "Value":
            entity_ids_with_multiple_paths,
    },

    {
        "Metric":
            "PathsWithMultipleEntityIDs",

        "Value":
            paths_with_multiple_ids,
    },

    {
        "Metric":
            "MaximumPathsPerEntityID",

        "Value":
            maximum_paths_per_entity,
    },

    {
        "Metric":
            "MappedEntityIDsMissingFromIDMap",

        "Value":
            len(
                mapped_entity_ids_missing_from_id_map
            ),
    },
])


# --------------------------------------------------------------------------------------------------
# 12. VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []

add_check(
    checks,
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256
    == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)

add_check(
    checks,
    "Step 1B status",
    EXPECTED_STEP1B_STATUS,
    selection_checkpoint.get(
        "Status"
    ),
    selection_checkpoint.get(
        "Status"
    )
    == EXPECTED_STEP1B_STATUS,
)

add_check(
    checks,
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root,
    current_source_root
    == EXPECTED_SOURCE_ROOT_SHA256,
)

add_check(
    checks,
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_manifest
    ),
    len(
        current_manifest
    )
    == EXPECTED_SOURCE_FILES,
)

add_check(
    checks,
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)

add_check(
    checks,
    "Builds",
    EXPECTED_BUILDS,
    len(
        chronology
    ),
    len(
        chronology
    )
    == EXPECTED_BUILDS,
)

add_check(
    checks,
    "Training builds",
    EXPECTED_TRAIN_BUILDS,
    len(
        training_builds
    ),
    len(
        training_builds
    )
    == EXPECTED_TRAIN_BUILDS,
)

add_check(
    checks,
    "Evaluation builds",
    EXPECTED_EVAL_BUILDS,
    len(
        evaluation_builds
    ),
    len(
        evaluation_builds
    )
    == EXPECTED_EVAL_BUILDS,
)

add_check(
    checks,
    "Timestamp tie groups",
    EXPECTED_TIMESTAMP_TIE_GROUPS,
    timestamp_tie_group_count,
    timestamp_tie_group_count
    == EXPECTED_TIMESTAMP_TIE_GROUPS,
)

add_check(
    checks,
    "Timestamp tied builds",
    EXPECTED_TIMESTAMP_TIE_BUILDS,
    timestamp_tie_build_count,
    timestamp_tie_build_count
    == EXPECTED_TIMESTAMP_TIE_BUILDS,
)

add_check(
    checks,
    "Chronology order reproduced",
    True,
    chronology_order_reproduced,
    chronology_order_reproduced,
)

add_check(
    checks,
    "Chronology partitions reproduced",
    True,
    chronology_partitions_reproduced,
    chronology_partitions_reproduced,
)

for audit_row in join_audit.itertuples(
    index=False
):
    add_check(
        checks,
        str(
            audit_row.Metric
        ),
        audit_row.Expected,
        audit_row.Actual,
        audit_row.Actual
        == audit_row.Expected,
    )

add_check(
    checks,
    "Dataset columns",
    EXPECTED_DATASET_COLUMNS,
    len(
        dataset.columns
    ),
    len(
        dataset.columns
    )
    == EXPECTED_DATASET_COLUMNS,
)

add_check(
    checks,
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)

add_check(
    checks,
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    len(
        REC_FEATURES
    )
    == EXPECTED_REC_FEATURES,
)

add_check(
    checks,
    "Missing REC features",
    0,
    len(
        missing_rec_features
    ),
    len(
        missing_rec_features
    )
    == 0,
)

add_check(
    checks,
    "Verdict-dependent REC features",
    len(
        VERDICT_DEPENDENT_REC
    ),
    int(
        rec_classification[
            "VerdictDependent"
        ].sum()
    ),
    int(
        rec_classification[
            "VerdictDependent"
        ].sum()
    )
    == len(
        VERDICT_DEPENDENT_REC
    ),
)

add_check(
    checks,
    "File-history REC features",
    len(
        FILE_HISTORY_REC
    ),
    int(
        rec_classification[
            "FileHistoryFeature"
        ].sum()
    ),
    int(
        rec_classification[
            "FileHistoryFeature"
        ].sum()
    )
    == len(
        FILE_HISTORY_REC
    ),
)

add_check(
    checks,
    "Resolved id_map EntityId column",
    "value",
    resolved_id_column,
    resolved_id_column
    == "value",
)

add_check(
    checks,
    "Resolved id_map path column",
    "key",
    resolved_path_column,
    resolved_path_column
    == "key",
)

add_check(
    checks,
    "Resolved id_map rows",
    "> 0",
    len(
        resolved_id_map
    ),
    len(
        resolved_id_map
    )
    > 0,
)

add_check(
    checks,
    "History EntityIds missing from id_map",
    0,
    len(
        history_entity_ids_missing_from_id_map
    ),
    len(
        history_entity_ids_missing_from_id_map
    )
    == 0,
)

add_check(
    checks,
    "Paths with multiple EntityIds",
    0,
    paths_with_multiple_ids,
    paths_with_multiple_ids
    == 0,
)

add_check(
    checks,
    "Mapped EntityIds missing from id_map",
    0,
    len(
        mapped_entity_ids_missing_from_id_map
    ),
    len(
        mapped_entity_ids_missing_from_id_map
    )
    == 0,
)

add_check(
    checks,
    "Ambiguous commit tokens",
    0,
    ambiguous_tokens,
    ambiguous_tokens
    == 0,
)

add_check(
    checks,
    "Matched commit tokens",
    "> 0",
    matched_token_rows,
    matched_token_rows
    > 0,
)

add_check(
    checks,
    "Build-entity rows",
    "> 0",
    len(
        build_entity
    ),
    len(
        build_entity
    )
    > 0,
)

add_check(
    checks,
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)

for (
    predecessor_number,
    predecessor_identity,
) in REQUIRED_REGISTERED_IDENTITIES.items():
    actual_identity = str(
        registry.loc[
            project_numbers.eq(
                predecessor_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    add_check(
        checks,
        f"Project {predecessor_number} frozen identity",
        predecessor_identity,
        actual_identity,
        actual_identity
        == predecessor_identity,
    )

add_check(
    checks,
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    selection_checkpoint.get(
        "ActiveReservations"
    ),
    selection_checkpoint.get(
        "ActiveReservations"
    )
    == EXPECTED_ACTIVE_RESERVATIONS,
)

add_check(
    checks,
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

add_check(
    checks,
    "Project 24 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


validation = pd.DataFrame(
    checks
)

failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 24 Step 2A validation:"
)

display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 24 Step 2A checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 24 STEP 2A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 13. WRITE AUDITABLE OUTPUTS
# --------------------------------------------------------------------------------------------------

PREFLIGHT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_csv(
    SOURCE_SCHEMA_PATH,
    source_schema,
)

atomic_csv(
    JOIN_AUDIT_PATH,
    join_audit,
)

atomic_csv(
    REC_CLASS_PATH,
    rec_classification,
)

atomic_csv(
    BUILD_TOKEN_PATH,
    build_tokens,
)

atomic_csv(
    COMMIT_AUDIT_PATH,
    commit_audit,
)

atomic_csv(
    BUILD_ENTITY_PATH,
    build_entity,
    compression="gzip",
)

atomic_csv(
    ID_ORIENTATION_PATH,
    id_orientation,
)

atomic_csv(
    RESOLVED_ID_MAP_PATH,
    resolved_id_map,
    compression="gzip",
)

atomic_csv(
    ENTITY_ID_AUDIT_PATH,
    entity_id_audit,
)

atomic_csv(
    MAPPING_INCOMPLETE_BUILDS_PATH,
    mapping_incomplete_frame,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


summary_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "ResolvedIDMapRows":
        len(
            resolved_id_map
        ),

    "ExactDuplicateIDMapRowsRemoved":
        exact_duplicate_rows,

    "DuplicateEntityIDRowsAcceptedAsAliases":
        duplicate_entity_id_rows,

    "EntityIDsWithMultiplePaths":
        entity_ids_with_multiple_paths,

    "PathsWithMultipleEntityIDs":
        paths_with_multiple_ids,

    "MaximumPathsPerEntityID":
        maximum_paths_per_entity,

    "HistoryEntityIDsMissingFromIDMap":
        len(
            history_entity_ids_missing_from_id_map
        ),

    "BuildCommitTokenRows":
        len(
            build_tokens
        ),

    "ExactCommitMatches":
        exact_matches,

    "UniquePrefixMatches":
        prefix_matches,

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "CommitTokenCoveragePercent":
        commit_coverage_percent,

    "BuildsWithoutCommitTokens":
        len(
            builds_without_tokens
        ),

    "BuildsWithMatchedCommits":
        len(
            builds_with_matched_commit
        ),

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "BuildEntityRows":
        len(
            build_entity
        ),

    "UniqueMappedEntities":
        int(
            build_entity[
                "EntityId"
            ].nunique()
            if not build_entity.empty
            else 0
        ),

    "MappedEntityIDsMissingFromIDMap":
        len(
            mapped_entity_ids_missing_from_id_map
        ),

    "CommitMatchingSeconds":
        commit_matching_seconds,

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
            or mapping_incomplete_builds
        ),
}


atomic_json(
    SUMMARY_PATH,
    summary_payload,
)


output_paths = [
    SOURCE_SCHEMA_PATH,
    JOIN_AUDIT_PATH,
    REC_CLASS_PATH,
    BUILD_TOKEN_PATH,
    COMMIT_AUDIT_PATH,
    BUILD_ENTITY_PATH,
    ID_ORIENTATION_PATH,
    RESOLVED_ID_MAP_PATH,
    ENTITY_ID_AUDIT_PATH,
    MAPPING_INCOMPLETE_BUILDS_PATH,
    VALIDATION_PATH,
    SUMMARY_PATH,
]

output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    **summary_payload,

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "SourceRootSHA256":
        current_source_root,

    "SourceFiles":
        len(
            current_manifest
        ),

    "SourceBytes":
        current_source_bytes,

    "Builds":
        len(
            chronology
        ),

    "TrainingBuilds":
        len(
            training_builds
        ),

    "EvaluationBuilds":
        len(
            evaluation_builds
        ),

    "TimestampTieGroups":
        timestamp_tie_group_count,

    "TimestampTieBuilds":
        timestamp_tie_build_count,

    "ChronologyOrderReproduced":
        chronology_order_reproduced,

    "ChronologyPartitionsReproduced":
        chronology_partitions_reproduced,

    "RawRows":
        len(
            exe
        ),

    "RawTrainingRows":
        raw_training_rows,

    "RawEvaluationRows":
        raw_evaluation_rows,

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "RawFailingTrainingBuilds":
        raw_failing_training_builds,

    "RawFailingEvaluationBuilds":
        raw_failing_evaluation_builds,

    "ModelRows":
        len(
            dataset
        ),

    "ModelTrainingRows":
        model_training_rows,

    "ModelEvaluationRows":
        model_evaluation_rows,

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingTrainingBuilds":
        model_failing_training_builds,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "DatasetColumns":
        len(
            dataset.columns
        ),

    "PredictorColumns":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "VerdictDependentRECFeatures":
        len(
            VERDICT_DEPENDENT_REC
        ),

    "VerdictIndependentRECFeatures":
        len(
            set(
                REC_FEATURES
            )
            - VERDICT_DEPENDENT_REC
        ),

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "OutputManifest":
        output_manifest,

    "RegistrySHA256":
        registry_sha256_before,

    "RegistryModified":
        False,

    "Projects1To23Modified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "NoiseInjected":
        False,

    "ModelsTrained":
        False,

    "Project24ExperimentStarted":
        False,
}


atomic_json(
    REPORT_PATH,
    report_payload,
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP2A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root,

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "TimestampTieGroups":
        timestamp_tie_group_count,

    "TimestampTieBuilds":
        timestamp_tie_build_count,

    "ResolvedIDMapEntityIDColumn":
        resolved_id_column,

    "ResolvedIDMapPathColumn":
        resolved_path_column,

    "BuildEntityRows":
        len(
            build_entity
        ),

    "BuildsWithMappedEntities":
        len(
            builds_with_entities
        ),

    "BuildsWithoutMappedEntities":
        len(
            builds_without_entities
        ),

    "UnmatchedCommitTokens":
        unmatched_tokens,

    "AmbiguousCommitTokens":
        ambiguous_tokens,

    "MappingIncompleteBuilds":
        len(
            mapping_incomplete_builds
        ),

    "RequiresStep2BMappingAudit":
        bool(
            unmatched_tokens
            or builds_without_entities
            or mapping_incomplete_builds
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "RegistryModified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "Project24ExperimentStarted":
        False,

    "NextRequiredStep":
        "PROJECT 24 CELL 5 / STEP 2B — DETERMINISTIC CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE",
}


atomic_json(
    STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 14. READBACK + IMMUTABILITY
# --------------------------------------------------------------------------------------------------

if len(
    pd.read_csv(
        BUILD_ENTITY_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    build_entity
):
    raise RuntimeError(
        "Build-entity map readback failed."
    )


if len(
    pd.read_csv(
        RESOLVED_ID_MAP_PATH,
        compression="gzip",
        low_memory=False,
    )
) != len(
    resolved_id_map
):
    raise RuntimeError(
        "Resolved id_map readback failed."
    )


if len(
    pd.read_csv(
        MAPPING_INCOMPLETE_BUILDS_PATH,
        low_memory=False,
    )
) != len(
    mapping_incomplete_frame
):
    raise RuntimeError(
        "Mapping-incomplete build audit readback failed."
    )


if sha256_file(
    REGISTRY_PATH
) != registry_sha256_before:
    raise RuntimeError(
        "Completion registry changed during Project 24 Step 2A."
    )


final_manifest_records = []

for row in current_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_manifest_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_manifest = pd.DataFrame(
    final_manifest_records
)


if source_root_hash(
    final_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 24 source changed during Step 2A."
    )


if load_json(
    STATUS_PATH
).get(
    "Status"
) != STEP2A_STATUS:
    raise RuntimeError(
        "Project 24 Step 2A status readback failed."
    )


if load_json(
    REPORT_PATH
).get(
    "Status"
) != STEP2A_STATUS:
    raise RuntimeError(
        "Project 24 Step 2A report readback failed."
    )


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        raise RuntimeError(
            f"Project 24 Step 2A output manifest readback failed: {path}"
        )


# --------------------------------------------------------------------------------------------------
# 15. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nBuild-Test join audit:"
)

display(
    join_audit
)


print(
    "\nid_map orientation audit:"
)

display(
    id_orientation
)


print(
    "\nid_map alias summary:"
)

display(
    pd.DataFrame([
        {
            "Metric":
                "Resolved EntityId column",

            "Value":
                resolved_id_column,
        },

        {
            "Metric":
                "Resolved path column",

            "Value":
                resolved_path_column,
        },

        {
            "Metric":
                "Resolved unique path-ID rows",

            "Value":
                len(
                    resolved_id_map
                ),
        },

        {
            "Metric":
                "History EntityIds missing from id_map",

            "Value":
                len(
                    history_entity_ids_missing_from_id_map
                ),
        },

        {
            "Metric":
                "Duplicate EntityId rows accepted as aliases",

            "Value":
                duplicate_entity_id_rows,
        },

        {
            "Metric":
                "EntityIds with multiple paths",

            "Value":
                entity_ids_with_multiple_paths,
        },

        {
            "Metric":
                "Paths with multiple EntityIds",

            "Value":
                paths_with_multiple_ids,
        },

        {
            "Metric":
                "Mapped entity IDs missing from id_map",

            "Value":
                len(
                    mapped_entity_ids_missing_from_id_map
                ),
        },
    ])
)


print(
    "\nCommit matching summary:"
)

display(
    commit_audit[
        "MatchType"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "MatchType"
    )
    .reset_index(
        name="Rows"
    )
)


print(
    "\nMapping-incomplete builds:"
)

if mapping_incomplete_frame.empty:
    print(
        "None"
    )

else:
    display(
        mapping_incomplete_frame
    )


print(
    "\nBuild-entity sample:"
)

display(
    pd.concat(
        [
            build_entity.head(
                10
            ),

            build_entity.tail(
                10
            ),
        ],
        ignore_index=True,
    )
)


# --------------------------------------------------------------------------------------------------
# 16. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 136
)

print(
    "=== PROJECT 24 CELL 4 / STEP 2A RESULT ==="
)

print(
    "=" * 136
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Active reservations:",
    EXPECTED_ACTIVE_RESERVATIONS,
)

print(
    "Runtime-priority rule:",
    EXPECTED_RUNTIME_PRIORITY_RULE,
)

print()

print(
    "Frozen source:"
)

print(
    "Selection checkpoint SHA-256:",
    selection_checkpoint_sha256,
)

print(
    "Source root SHA-256:",
    current_source_root,
)

print(
    "Source files / bytes:",
    len(
        current_manifest
    ),
    "/",
    current_source_bytes,
)

print()

print(
    "Chronology:"
)

print(
    "Builds:",
    len(
        chronology
    ),
)

print(
    "Training / evaluation builds:",
    len(
        training_builds
    ),
    "/",
    len(
        evaluation_builds
    ),
)

print(
    "Timestamp tie groups / tied builds:",
    timestamp_tie_group_count,
    "/",
    timestamp_tie_build_count,
)

print(
    "Chronology order / split reproduced:",
    chronology_order_reproduced,
    "/",
    chronology_partitions_reproduced,
)

print()

print(
    "Build-Test joins:"
)

print(
    "Raw execution rows:",
    len(
        exe
    ),
)

print(
    "Model-ready rows:",
    len(
        dataset
    ),
)

print(
    "Dataset columns / predictors / REC:",
    len(
        dataset.columns
    ),
    "/",
    len(
        predictor_columns
    ),
    "/",
    len(
        REC_FEATURES
    ),
)

print(
    "Raw duplicate Build-Test rows:",
    raw_duplicate_pairs,
)

print(
    "Model duplicate Build-Test rows:",
    model_duplicate_pairs,
)

print(
    "Missing model-to-raw links:",
    missing_model_raw_links,
)

print(
    "Model/raw verdict mismatches:",
    verdict_mismatches,
)

print(
    "Non-finite / negative duration rows:",
    nonfinite_duration_rows,
    "/",
    negative_duration_rows,
)

print(
    "Raw/model unlinked build rows:",
    raw_unlinked_build_rows,
    "/",
    model_unlinked_build_rows,
)

print()

print(
    "id_map:"
)

print(
    "Resolved EntityId / path columns:",
    resolved_id_column,
    "/",
    resolved_path_column,
)

print(
    "Resolved id_map rows:",
    len(
        resolved_id_map
    ),
)

print(
    "History EntityIds missing from id_map:",
    len(
        history_entity_ids_missing_from_id_map
    ),
)

print(
    "Duplicate EntityId rows accepted as aliases:",
    duplicate_entity_id_rows,
)

print(
    "EntityIds with multiple paths:",
    entity_ids_with_multiple_paths,
)

print(
    "Paths with multiple EntityIds:",
    paths_with_multiple_ids,
)

print()

print(
    "Commit/entity mapping:"
)

print(
    "Build commit-token rows:",
    len(
        build_tokens
    ),
)

print(
    "Exact matches:",
    exact_matches,
)

print(
    "Unique-prefix matches:",
    prefix_matches,
)

print(
    "Unmatched tokens:",
    unmatched_tokens,
)

print(
    "Ambiguous tokens:",
    ambiguous_tokens,
)

print(
    "Token coverage percent:",
    commit_coverage_percent,
)

print(
    "Builds without commit tokens:",
    len(
        builds_without_tokens
    ),
)

print(
    "Builds with mapped entities:",
    len(
        builds_with_entities
    ),
)

print(
    "Builds without mapped entities:",
    len(
        builds_without_entities
    ),
)

print(
    "Mapping-incomplete builds:",
    len(
        mapping_incomplete_builds
    ),
)

print(
    "Build-entity rows:",
    len(
        build_entity
    ),
)

print(
    "Mapped entity IDs missing from id_map:",
    len(
        mapped_entity_ids_missing_from_id_map
    ),
)

print(
    "Requires Step 2B mapping audit:",
    bool(
        unmatched_tokens
        or builds_without_entities
        or mapping_incomplete_builds
    ),
)

print()

print(
    "Isolation:"
)

print(
    "Completion registry unchanged:",
    sha256_file(
        REGISTRY_PATH
    )
    == registry_sha256_before,
)

print(
    "Projects 1–23 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Noise injected:",
    False,
)

print(
    "Models trained:",
    False,
)

print(
    "Experiment started:",
    False,
)

print()

print(
    "Validation checks:",
    len(
        validation
    ),
)

print(
    "Failed validation checks:",
    len(
        failed_validation
    ),
)

print(
    "\nNext required step:",
    "PROJECT 24 CELL 5 / STEP 2B — DETERMINISTIC CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE",
)

print(
    "STATUS:",
    STEP2A_STATUS,
)

print(
    "=" * 136
)


=== PROJECT 24 CELL 4 / STEP 2A: SOURCE SCHEMA AND JOIN-STRUCTURE VALIDATION ===

Loading Project 24 model-ready dataset (224,550 rows expected).
Loading only required columns from Project 24 raw execution table (5,635,027 rows expected).

Project 24 Step 2A validation:


,Check,Expected,Actual,Pass
0,Selection checkpoint SHA-256,d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f167...,d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f167...,True
1,Step 1B status,PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN,PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN,True
2,Source root SHA-256,2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21...,2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21...,True
3,Source files,6,6,True
4,Source bytes,374503252,374503252,True
...,...,...,...,...
61,Project 22 frozen identity,apache@logging-log4j2,apache@logging-log4j2,True
62,Project 23 frozen identity,apache@sling,apache@sling,True
63,Active reservations,[],[],True
64,Runtime-priority ranking rule,"[ModelTrainingRows ascending, ModelEvaluationR...","[ModelTrainingRows ascending, ModelEvaluationR...",True



Build-Test join audit:


,Metric,Expected,Actual
0,RawRows,5635027,5635027
1,RawTrainingRows,4040801,4040801
2,RawEvaluationRows,1594226,1594226
3,RawTrainingFailures,1778,1778
4,RawEvaluationFailures,20,20
5,RawFailingTrainingBuilds,283,283
6,RawFailingEvaluationBuilds,17,17
7,ModelRows,224550,224550
8,ModelTrainingRows,205696,205696
9,ModelEvaluationRows,18854,18854



id_map orientation audit:


...


Mapping-incomplete builds:


,Build,BuildOrder,Split,CommitTokenRows,ExactMatches,PrefixMatches,UnmatchedTokens,AmbiguousTokens,MatchedEntityRows,Reasons
0,54854778,15,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...
1,54857293,16,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...
2,55605572,124,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...
3,55608003,125,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...
4,55761727,139,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...
5,55773465,140,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...
6,55776490,141,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...
7,55785851,143,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...
8,55789407,145,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...
9,55825430,153,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...



Build-entity sample:


,Build,BuildOrder,Split,CommitToken,MatchedCommit,MatchType,EntityId
0,54731125,1,TRAIN,874614a2c6ba28185879abc248748ae087a23b46,874614a2c6ba28185879abc248748ae087a23b46,EXACT,170
1,54731125,1,TRAIN,874614a2c6ba28185879abc248748ae087a23b46,874614a2c6ba28185879abc248748ae087a23b46,EXACT,179
2,54731125,1,TRAIN,7485c34dead228336791067e3a6b03cbb3dcaaa2,7485c34dead228336791067e3a6b03cbb3dcaaa2,EXACT,237
3,54731125,1,TRAIN,874614a2c6ba28185879abc248748ae087a23b46,874614a2c6ba28185879abc248748ae087a23b46,EXACT,248
4,54731125,1,TRAIN,874614a2c6ba28185879abc248748ae087a23b46,874614a2c6ba28185879abc248748ae087a23b46,EXACT,257
5,54731125,1,TRAIN,874614a2c6ba28185879abc248748ae087a23b46,874614a2c6ba28185879abc248748ae087a23b46,EXACT,1627
6,54731125,1,TRAIN,874614a2c6ba28185879abc248748ae087a23b46,874614a2c6ba28185879abc248748ae087a23b46,EXACT,1631
7,54731125,1,TRAIN,874614a2c6ba28185879abc248748ae087a23b46,874614a2c6ba28185879abc248748ae087a23b46,EXACT,1632
8,54731125,1,TRAIN,874614a2c6ba28185879abc248748ae087a23b46,874614a2c6ba28185879abc248748ae087a23b46,EXACT,1859
9,54731125,1,TRAIN,874614a2c6ba28185879abc248748ae087a23b46,874614a2c6ba28185879abc248748ae087a23b46,EXACT,2130



=== PROJECT 24 CELL 4 / STEP 2A RESULT ===
Project: SonarSource@sonarqube
Project slug: SonarSource__sonarqube
Active reservations: []
Runtime-priority rule: ['ModelTrainingRows ascending', 'ModelEvaluationRows ascending', 'RawExecutionRows ascending', 'Project ascending']

Frozen source:
Selection checkpoint SHA-256: d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2
Source root SHA-256: 2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70
Source files / bytes: 6 / 374503252

Chronology:
Builds: 4286
Training / evaluation builds: 3214 / 1072
Timestamp tie groups / tied builds: 23 / 46
Chronology order / split reproduced: True / True

Build-Test joins:
Raw execution rows: 5635027
Model-ready rows: 224550
Dataset columns / predictors / REC: 154 / 151 / 19
Raw duplicate Build-Test rows: 0
Model duplicate Build-Test rows: 0
Missing model-to-raw links: 0
Model/raw verdict mismatches: 0
Non-finite / negative duration rows: 0 / 0
Raw/model unlinked build rows: 0 / 

In [5]:
# ==================================================================================================
# PROJECT 24 — CELL 5 / STEP 2B
# SCALABLE EXACT-TIE CLEAN REC RECONSTRUCTION AND ANCHOR FREEZE
# Project: SonarSource@sonarqube
#
# Run as the next new cell in Thesis_project_24.ipynb.
#
# Scientific contract:
# - official verdict semantics: success=0, exception=1, assertion=2;
# - 12 order-sensitive non-file REC features determine exact per-test ordering inside timestamp ties;
# - REC_Age independently determines the exact global build order inside timestamp ties;
# - four verdict-independent timing aggregates may retain deterministic clean residuals and are
#   preserved only through the clean anchor;
# - file-history residuals are accepted only when confined to the 49 Step-2A mapping-incomplete
#   builds; broader mapping residuals are a hard failure;
# - exact 0%-noise clean-data reproduction is mandatory;
# - no noise, no model fitting, no experiment execution, no registry write.
#
# Scalability:
# - Project 24 has 23 two-build timestamp ties, so a naive 2^23 global permutation scan is forbidden.
# - Per-test ties use exact state-space dynamic programming only when the frozen baseline order does
#   not already reproduce the 12 tie-sensitive source features.
# - Global REC_Age uses an exact binary constraint solver over the 23 tie variables.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import hashlib, json, os, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

print("=" * 140)
print("=== PROJECT 24 CELL 5 / STEP 2B: SCALABLE EXACT-TIE CLEAN REC RECONSTRUCTION ===")
print("=" * 140)

# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------
PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"
SOURCE_DIR = Path("/content/datasets/datasets/SonarSource@sonarqube")

EXPECTED_STEP1B_STATUS = "PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN"
EXPECTED_STEP2A_STATUS = "PASS_PROJECT_24_SOURCE_SCHEMA_AND_JOIN_STRUCTURE_VALIDATED"
STEP2B_STATUS = "PASS_PROJECT_24_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
IMPLEMENTATION_VERSION = "PROJECT_24_V1_SCALABLE_EXACT_TIE_DP_BINARY_AGE_CONSTRAINT_ANCHOR_AWARE_MAPPING_BOUNDARY_AUDIT"

EXPECTED_SELECTION_SHA256 = "d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
EXPECTED_SOURCE_ROOT_SHA256 = "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
EXPECTED_REGISTRY_SHA256 = "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 23
EXPECTED_ACTIVE_RESERVATIONS = []
EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 374_503_252
EXPECTED_BUILDS = 4_286
EXPECTED_TRAIN_BUILDS = 3_214
EXPECTED_EVAL_BUILDS = 1_072
EXPECTED_TIMESTAMP_TIE_GROUPS = 23
EXPECTED_TIMESTAMP_TIE_BUILDS = 46

EXPECTED_RAW_ROWS = 5_635_027
EXPECTED_RAW_TRAIN_ROWS = 4_040_801
EXPECTED_RAW_EVAL_ROWS = 1_594_226
EXPECTED_RAW_TRAIN_FAILURES = 1_778
EXPECTED_RAW_EVAL_FAILURES = 20
EXPECTED_MODEL_ROWS = 224_550
EXPECTED_MODEL_TRAIN_ROWS = 205_696
EXPECTED_MODEL_EVAL_ROWS = 18_854
EXPECTED_MODEL_TRAIN_FAILURES = 1_777
EXPECTED_MODEL_EVAL_FAILURES = 20
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 17
EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151

EXPECTED_COMMIT_TOKEN_ROWS = 7_899
EXPECTED_EXACT_COMMIT_MATCHES = 7_850
EXPECTED_PREFIX_COMMIT_MATCHES = 0
EXPECTED_UNMATCHED_COMMIT_TOKENS = 49
EXPECTED_AMBIGUOUS_COMMIT_TOKENS = 0
EXPECTED_BUILDS_WITH_MAPPED_ENTITIES = 4_238
EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES = 48
EXPECTED_BUILD_ENTITY_ROWS = 91_069
EXPECTED_MAPPING_INCOMPLETE_BUILDS = {54_854_778, 54_857_293, 55_605_572, 55_608_003, 55_761_727, 55_773_465, 55_776_490, 55_785_851, 55_789_407, 55_825_430, 55_910_589, 57_005_798, 58_868_322, 64_049_246, 64_525_941, 65_214_818, 70_751_935, 72_240_439, 72_790_741, 77_121_192, 87_833_946, 89_000_695, 98_265_658, 101_000_705, 101_212_516, 101_354_463, 101_502_723, 101_543_578, 108_001_130, 108_033_630, 108_192_171, 109_536_018, 114_173_512, 114_739_372, 121_648_197, 122_189_222, 122_705_155, 125_804_012, 126_572_774, 127_426_396, 134_657_375, 134_927_350, 139_415_651, 140_475_190, 144_922_466, 148_789_387, 149_159_850, 149_158_959, 149_994_505}
EXPECTED_MAPPING_INCOMPLETE_PARTITIONS = {"TRAIN", "EVAL"}
EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES = 1

RECENT_WINDOW = 6
SUCCESS_VERDICT_CODE = 0
EXCEPTION_VERDICT_CODE = 1
ASSERTION_VERDICT_CODE = 2
DIRECT_RTOL = 1e-9
DIRECT_ATOL = 1e-9
ANCHOR_RTOL = 0.0
ANCHOR_ATOL = 1e-12
MAX_DP_STATES_PER_TEST = 250_000

REC_FEATURES = [
    "REC_Age", "REC_LastFailureAge", "REC_LastTransitionAge",
    "REC_RecentAvgExeTime", "REC_RecentMaxExeTime", "REC_RecentFailRate",
    "REC_RecentAssertRate", "REC_RecentExcRate", "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime", "REC_TotalMaxExeTime", "REC_TotalFailRate",
    "REC_TotalAssertRate", "REC_TotalExcRate", "REC_TotalTransitionRate",
    "REC_LastVerdict", "REC_LastExeTime", "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge", "REC_LastTransitionAge", "REC_RecentFailRate",
    "REC_RecentAssertRate", "REC_RecentExcRate", "REC_RecentTransitionRate",
    "REC_TotalFailRate", "REC_TotalAssertRate", "REC_TotalExcRate",
    "REC_TotalTransitionRate", "REC_LastVerdict", "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]
VERDICT_INDEPENDENT_REC = [
    "REC_Age", "REC_RecentAvgExeTime", "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime", "REC_TotalMaxExeTime", "REC_LastExeTime",
]
FILE_HISTORY_REC = ["REC_MaxTestFileFailRate", "REC_MaxTestFileTransitionRate"]
ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES = [
    "REC_RecentAvgExeTime", "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime", "REC_TotalMaxExeTime",
]
TIE_INFERENCE_FEATURES = [
    f for f in REC_FEATURES
    if f != "REC_Age"
    and f not in FILE_HISTORY_REC
    and f not in ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES
]
NON_FILE_REC = [f for f in REC_FEATURES if f not in FILE_HISTORY_REC]

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere", 12: "zolyfarkas@spf4j", 13: "jcabi@jcabi-github",
    14: "JMRI@JMRI", 15: "eclipse@steady", 16: "apache@rocketmq", 17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe", 19: "EMResearch@EvoMaster", 20: "apache@curator",
    21: "facebook@buck", 22: "apache@logging-log4j2", 23: "apache@sling",
}

# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------
THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"
REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_24_selection"
SELECTION_CHECKPOINT_PATH = NOTES_ROOT / "project_24_selection_checkpoint.json"
STEP1B_STATUS_PATH = SELECTION_ROOT / "project_24_step1b_status.json"
FROZEN_SOURCE_MANIFEST_PATH = SELECTION_ROOT / "project_24_frozen_source_manifest.csv"
FIXED_CHRONOLOGY_PATH = SELECTION_ROOT / "project_24_fixed_chronological_builds.csv"
PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
STEP2A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step2a_status.json"
STEP2A_REPORT_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_step2a_report.json"
ENTITY_MAPPING_SUMMARY_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_entity_mapping_summary.json"
COMMIT_AUDIT_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_commit_matching_audit.csv"
BUILD_ENTITY_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_build_entity_map.csv.gz"
MAPPING_INCOMPLETE_BUILDS_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_mapping_incomplete_builds.csv"
UNMATCHED_MAPPING_AUDIT_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_unmatched_mapping_audit.csv"
TIMESTAMP_TIE_GROUPS_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_timestamp_tie_groups.csv"
TEST_ORDER_SEARCH_AUDIT_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_test_order_search_audit.csv"
INFERRED_EXECUTION_ORDER_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
GLOBAL_AGE_ORDER_SEARCH_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_global_age_order_search.csv"
FROZEN_GLOBAL_BUILD_ORDER_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_global_build_order.csv"
CLEAN_RECONSTRUCTED_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
CLEAN_ANCHOR_OFFSETS_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
CLEAN_COMPARISON_SUMMARY_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_comparison_summary.csv"
CLEAN_MISMATCH_EXAMPLES_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_rec_mismatch_examples.csv"
CLEAN_ANCHOR_VALIDATION_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_clean_anchor_validation.csv"
STEP2B_VALIDATION_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_step2b_validation.csv"
STEP2B_REPORT_PATH = PREFLIGHT_ROOT / f"{PROJECT_SHORT}_step2b_report.json"
STEP2B_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step2b_status.json"
REC_CHECKPOINT_PATH = NOTES_ROOT / "project_24_rec_reconstruction_checkpoint.json"

# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def atomic_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True, ensure_ascii=False, default=str)
        f.write("\n")
    os.replace(tmp, path)


def atomic_csv(path, frame):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    frame.to_csv(tmp, index=False, lineterminator="\n")
    os.replace(tmp, path)


def atomic_parquet(path, frame):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.tmp_{os.getpid()}")
    frame.to_parquet(tmp, index=False, compression="zstd")
    os.replace(tmp, path)


def source_root_hash(frame):
    h = hashlib.sha256()
    for r in frame.sort_values("RelativePath", kind="mergesort").itertuples(index=False):
        h.update(f"{r.RelativePath}\0{int(r.SizeBytes)}\0{str(r.SHA256).lower()}\n".encode("utf-8"))
    return h.hexdigest()


def resolve_column(columns, expected, label):
    m = [c for c in columns if str(c).strip().lower() == str(expected).strip().lower()]
    if len(m) != 1:
        raise RuntimeError(f"Could not resolve {label}: expected={expected!r}, matches={m}, columns={list(columns)}")
    return m[0]


def parse_int(values, label):
    x = pd.to_numeric(values, errors="coerce")
    if x.isna().any():
        raise RuntimeError(f"{label} contains {int(x.isna().sum())} missing/non-numeric values.")
    a = x.to_numpy(dtype=float)
    if not np.isclose(a, np.floor(a), rtol=0, atol=0).all():
        raise RuntimeError(f"{label} contains non-integral values.")
    return x.astype("int64")


def add_check(rows, check, expected, actual, passed):
    rows.append({"Check": check, "Expected": expected, "Actual": actual, "Pass": bool(passed)})


def product_size(option_lists):
    n = 1
    for options in option_lists:
        n *= len(options)
    return int(n)


def prefix_sum(values):
    values = np.asarray(values)
    dtype = np.float64 if values.dtype.kind == "f" else np.int64
    out = np.empty(len(values) + 1, dtype=dtype); out[0] = 0
    np.cumsum(values, out=out[1:])
    return out


def safe_divide(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    out = np.full(len(denominator), -1.0, dtype=float)
    ok = denominator > 0
    out[ok] = numerator[ok] / denominator[ok]
    return out


def calculate_file_rate(target_builds, current_entities, entity_changed_builds):
    if not target_builds:
        return -1.0
    maximum_frequency = 0
    for entity_id in current_entities:
        changed = entity_changed_builds.get(int(entity_id), frozenset())
        maximum_frequency = max(maximum_frequency, len(target_builds.intersection(changed)))
    return 0.0 if maximum_frequency == 0 else float(maximum_frequency / len(target_builds))


def reconstruct_group_features(builds, verdicts, durations, global_positions, requested_positions,
                               changed_entities_by_build, entity_changed_builds):
    """Vectorized non-file reconstruction plus exact file-history reconstruction for one test."""
    builds = np.asarray(builds, dtype=np.int64)
    verdicts = np.asarray(verdicts, dtype=np.int64)
    durations = np.asarray(durations, dtype=np.float64)
    global_positions = np.asarray(global_positions, dtype=np.int64)
    positions = np.asarray(requested_positions, dtype=np.int64)
    n = len(builds)
    all_positions = np.arange(n, dtype=np.int64)

    failure = (verdicts != SUCCESS_VERDICT_CODE).astype(np.int64)
    assertion = (verdicts == ASSERTION_VERDICT_CODE).astype(np.int64)
    exception = (verdicts == EXCEPTION_VERDICT_CODE).astype(np.int64)
    transition = np.zeros(n, dtype=np.int64)
    if n > 1:
        transition[1:] = (verdicts[1:] != verdicts[:-1]).astype(np.int64)

    dpre = prefix_sum(durations); fpre = prefix_sum(failure); apre = prefix_sum(assertion)
    epre = prefix_sum(exception); tpre = prefix_sum(transition)
    history_len = positions.astype(float)
    recent_start = np.maximum(0, positions - RECENT_WINDOW)
    recent_len = (positions - recent_start).astype(float)

    last_fail_inc = np.maximum.accumulate(np.where(failure > 0, all_positions, -1))
    last_trans_inc = np.maximum.accumulate(np.where(transition > 0, all_positions, -1))
    prior_fail = np.full(len(positions), -1, dtype=np.int64)
    prior_trans = np.full(len(positions), -1, dtype=np.int64)
    has_history = positions > 0
    prior_fail[has_history] = last_fail_inc[positions[has_history] - 1]
    prior_trans[has_history] = last_trans_inc[positions[has_history] - 1]

    recent_max = np.full(n, np.nan, dtype=float)
    for offset in range(1, RECENT_WINDOW + 1):
        if n > offset:
            recent_max[offset:] = np.fmax(recent_max[offset:], durations[:-offset])
    total_max = np.maximum.accumulate(durations)
    previous = np.maximum(positions - 1, 0)

    result = {
        "REC_Age": (global_positions[positions] - global_positions[0]).astype(float),
        "REC_LastFailureAge": np.where(prior_fail < 0, -1.0, (positions - 1 - prior_fail).astype(float)),
        "REC_LastTransitionAge": np.where(prior_trans < 0, -1.0, (positions - 1 - prior_trans).astype(float)),
        "REC_RecentAvgExeTime": safe_divide(dpre[positions] - dpre[recent_start], recent_len),
        "REC_RecentMaxExeTime": np.where(has_history, recent_max[positions], -1.0),
        "REC_RecentFailRate": safe_divide(fpre[positions] - fpre[recent_start], recent_len),
        "REC_RecentAssertRate": safe_divide(apre[positions] - apre[recent_start], recent_len),
        "REC_RecentExcRate": safe_divide(epre[positions] - epre[recent_start], recent_len),
        "REC_RecentTransitionRate": safe_divide(tpre[positions] - tpre[recent_start], recent_len),
        "REC_TotalAvgExeTime": safe_divide(dpre[positions], history_len),
        "REC_TotalMaxExeTime": np.where(has_history, total_max[previous], -1.0),
        "REC_TotalFailRate": safe_divide(fpre[positions], history_len),
        "REC_TotalAssertRate": safe_divide(apre[positions], history_len),
        "REC_TotalExcRate": safe_divide(epre[positions], history_len),
        "REC_TotalTransitionRate": safe_divide(tpre[positions], history_len),
        "REC_LastVerdict": np.where(has_history, verdicts[previous], -1).astype(float),
        "REC_LastExeTime": np.where(has_history, durations[previous], -1.0),
    }

    file_fail = np.empty(len(positions), dtype=float)
    file_trans = np.empty(len(positions), dtype=float)
    fail_events = np.flatnonzero(failure > 0); trans_events = np.flatnonzero(transition > 0)
    fp = tp = 0; prior_fail_builds = set(); prior_trans_builds = set()
    for requested_index in np.argsort(positions, kind="mergesort"):
        current_position = int(positions[requested_index])
        while fp < len(fail_events) and int(fail_events[fp]) < current_position:
            prior_fail_builds.add(int(builds[fail_events[fp]])); fp += 1
        while tp < len(trans_events) and int(trans_events[tp]) < current_position:
            prior_trans_builds.add(int(builds[trans_events[tp]])); tp += 1
        current_build = int(builds[current_position])
        entities = changed_entities_by_build.get(current_build, frozenset())
        file_fail[requested_index] = calculate_file_rate(prior_fail_builds, entities, entity_changed_builds)
        file_trans[requested_index] = calculate_file_rate(prior_trans_builds, entities, entity_changed_builds)
    result["REC_MaxTestFileFailRate"] = file_fail
    result["REC_MaxTestFileTransitionRate"] = file_trans
    return result


def reconstruct_tie_objective(ordered_frame, model_rows):
    """Reconstruct only the 12 features that are allowed to choose a tie order."""
    ordered_frame = ordered_frame.reset_index(drop=True)
    builds = ordered_frame["Build"].to_numpy(dtype=np.int64)
    verdicts = ordered_frame["Verdict"].to_numpy(dtype=np.int64)
    durations = ordered_frame["Duration"].to_numpy(dtype=np.float64)
    position = {int(b): i for i, b in enumerate(builds)}
    rows = []
    for build in model_rows["Build"].astype(int):
        p = position[int(build)]
        if p == 0:
            rec = {f: -1.0 for f in TIE_INFERENCE_FEATURES}
        else:
            trans = np.zeros(p, dtype=np.int8)
            if p > 1:
                trans[1:] = (verdicts[1:p] != verdicts[:p-1]).astype(np.int8)
            hv = verdicts[:p]; recent_start = max(0, p - RECENT_WINDOW)
            rv = verdicts[recent_start:p]; rt = trans[recent_start:p]
            failures = np.flatnonzero(hv != SUCCESS_VERDICT_CODE)
            transitions = np.flatnonzero(trans != 0)
            rec = {
                "REC_LastFailureAge": -1.0 if len(failures) == 0 else float(p - 1 - int(failures[-1])),
                "REC_LastTransitionAge": -1.0 if len(transitions) == 0 else float(p - 1 - int(transitions[-1])),
                "REC_RecentFailRate": float(np.mean(rv != SUCCESS_VERDICT_CODE)),
                "REC_RecentAssertRate": float(np.mean(rv == ASSERTION_VERDICT_CODE)),
                "REC_RecentExcRate": float(np.mean(rv == EXCEPTION_VERDICT_CODE)),
                "REC_RecentTransitionRate": float(np.mean(rt != 0)),
                "REC_TotalFailRate": float(np.mean(hv != SUCCESS_VERDICT_CODE)),
                "REC_TotalAssertRate": float(np.mean(hv == ASSERTION_VERDICT_CODE)),
                "REC_TotalExcRate": float(np.mean(hv == EXCEPTION_VERDICT_CODE)),
                "REC_TotalTransitionRate": float(np.mean(trans != 0)),
                "REC_LastVerdict": float(verdicts[p - 1]),
                "REC_LastExeTime": float(durations[p - 1]),
            }
        rec["Build"] = int(build); rows.append(rec)
    return pd.DataFrame(rows)


def reconstruct_all_rec(execution_history, requested_rows, global_build_position,
                        changed_entities_by_build, entity_changed_builds):
    requested_rows = requested_rows.reset_index(drop=True).copy()
    requested_groups = requested_rows.groupby("Test", sort=False).indices
    requested_builds = requested_rows["Build"].to_numpy(dtype=np.int64)
    result = {f: np.full(len(requested_rows), np.nan, dtype=np.float64) for f in REC_FEATURES}
    filled = np.zeros(len(requested_rows), dtype=bool)
    history_groups = execution_history.groupby("Test", sort=False).indices
    started = time.perf_counter(); total_tests = len(history_groups)

    for test_number, (test_id, idx) in enumerate(history_groups.items(), start=1):
        req_idx = requested_groups.get(int(test_id))
        if req_idx is None:
            continue
        idx = np.asarray(idx, dtype=np.int64); req_idx = np.asarray(req_idx, dtype=np.int64)
        h = execution_history.iloc[idx]
        builds = h["Build"].to_numpy(dtype=np.int64)
        verdicts = h["Verdict"].to_numpy(dtype=np.int64)
        durations = h["Duration"].to_numpy(dtype=np.float64)
        pos = {int(b): i for i, b in enumerate(builds)}
        positions = np.array([pos.get(int(requested_builds[i]), -1) for i in req_idx], dtype=np.int64)
        if (positions < 0).any():
            raise RuntimeError(f"Requested model row absent from raw history for Test={test_id}.")
        global_positions = np.fromiter((global_build_position[int(b)] for b in builds), dtype=np.int64, count=len(builds))
        group_result = reconstruct_group_features(
            builds, verdicts, durations, global_positions, positions,
            changed_entities_by_build, entity_changed_builds,
        )
        for f in REC_FEATURES:
            result[f][req_idx] = group_result[f]
        filled[req_idx] = True
        if test_number % 500 == 0 or test_number == total_tests:
            print("    REC reconstruction progress:", test_number, "/", total_tests,
                  "tests | elapsed seconds:", round(time.perf_counter() - started, 2))

    if not filled.all():
        raise RuntimeError("Clean REC reconstruction did not fill every requested model row.")
    out = requested_rows[["Build", "Test"]].copy()
    for f in REC_FEATURES:
        if not np.isfinite(result[f]).all():
            raise RuntimeError(f"Reconstructed {f} contains non-finite values.")
        out[f] = result[f]
    return out

# --------------------------------------------------------------------------------------------------

# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN UPSTREAM STATE
# --------------------------------------------------------------------------------------------------
required = [
    REGISTRY_PATH, SELECTION_CHECKPOINT_PATH, STEP1B_STATUS_PATH, FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH, STEP2A_STATUS_PATH, STEP2A_REPORT_PATH, ENTITY_MAPPING_SUMMARY_PATH,
    COMMIT_AUDIT_PATH, BUILD_ENTITY_PATH, MAPPING_INCOMPLETE_BUILDS_PATH,
    SOURCE_DIR / "builds.csv", SOURCE_DIR / "dataset.csv", SOURCE_DIR / "exe.csv",
]
missing = [str(p) for p in required if not Path(p).is_file()]
if missing:
    raise FileNotFoundError("Required Project 24 Step 2B inputs are missing:\n" + "\n".join(missing))

selection_sha256 = sha256_file(SELECTION_CHECKPOINT_PATH)
if selection_sha256 != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        f"Selection checkpoint SHA differs. Expected={EXPECTED_SELECTION_SHA256}; actual={selection_sha256}"
    )

selection = load_json(SELECTION_CHECKPOINT_PATH)
step1b_status = load_json(STEP1B_STATUS_PATH)
step2a_status = load_json(STEP2A_STATUS_PATH)
step2a_report = load_json(STEP2A_REPORT_PATH)
entity_mapping_summary = load_json(ENTITY_MAPPING_SUMMARY_PATH)

if selection.get("Status") != EXPECTED_STEP1B_STATUS or step1b_status.get("Status") != EXPECTED_STEP1B_STATUS:
    raise RuntimeError("Project 24 Step 1B is not frozen successfully.")
if any(x.get("Status") != EXPECTED_STEP2A_STATUS for x in [step2a_status, step2a_report, entity_mapping_summary]):
    raise RuntimeError("Project 24 Step 2A outputs are not in the expected PASS state.")
if selection.get("Project") != PROJECT_NAME or selection.get("ProjectSlug") != PROJECT_SLUG:
    raise RuntimeError("Frozen Project 24 identity differs.")
if selection.get("ActiveReservations") != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError("Frozen active-reservation state differs.")
if selection.get("RuntimePriorityRule") != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError("Frozen runtime-priority rule differs.")

step2a_hashes_before = {str(p): sha256_file(p) for p in [
    STEP2A_STATUS_PATH, STEP2A_REPORT_PATH, ENTITY_MAPPING_SUMMARY_PATH,
    COMMIT_AUDIT_PATH, BUILD_ENTITY_PATH, MAPPING_INCOMPLETE_BUILDS_PATH,
]}

step2a_manifest_failures = 0
for item in step2a_report.get("OutputManifest", []):
    p = Path(item["Path"])
    ok = (
        p.is_file()
        and int(p.stat().st_size) == int(item["Bytes"])
        and sha256_file(p) == str(item["SHA256"])
    )
    step2a_manifest_failures += int(not ok)
if step2a_manifest_failures:
    raise RuntimeError(f"{step2a_manifest_failures} frozen Project 24 Step 2A outputs changed.")

# --------------------------------------------------------------------------------------------------
# 5. REGISTRY AND SOURCE IMMUTABILITY
# --------------------------------------------------------------------------------------------------
registry_sha256_before = sha256_file(REGISTRY_PATH)
if registry_sha256_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        f"Registry SHA differs. Expected={EXPECTED_REGISTRY_SHA256}; actual={registry_sha256_before}"
    )

registry = pd.read_csv(REGISTRY_PATH, dtype=str).fillna("")
pn_col = resolve_column(registry.columns, "ProjectNumber", "registry ProjectNumber")
p_col = resolve_column(registry.columns, "Project", "registry Project")
s_col = resolve_column(registry.columns, "Status", "registry Status")
registry_numbers = pd.to_numeric(registry[pn_col], errors="raise").astype(int)

if len(registry) != EXPECTED_REGISTERED_PROJECTS or sorted(registry_numbers.tolist()) != list(range(1, 24)):
    raise RuntimeError("Registry must contain exactly Projects 1–23.")
if not registry[s_col].eq(EXPECTED_COMPLETE_STATUS).all():
    raise RuntimeError("Projects 1–23 are not all COMPLETE_AND_FROZEN.")
if registry_numbers.eq(PROJECT_NUMBER).any() or registry[p_col].eq(PROJECT_NAME).any():
    raise RuntimeError("Project 24 is unexpectedly already registered.")
for n, identity in REQUIRED_REGISTERED_IDENTITIES.items():
    rows = registry.loc[registry_numbers.eq(n)]
    if len(rows) != 1 or str(rows.iloc[0][p_col]) != identity:
        raise RuntimeError(f"Frozen predecessor identity differs for Project {n}: expected {identity}")

frozen_manifest = pd.read_csv(FROZEN_SOURCE_MANIFEST_PATH, low_memory=False)
current_records = []
for r in frozen_manifest.itertuples(index=False):
    p = SOURCE_DIR / str(r.RelativePath)
    if not p.is_file():
        raise FileNotFoundError(f"Frozen source file missing: {p}")
    current_records.append({
        "RelativePath": str(r.RelativePath),
        "SizeBytes": int(p.stat().st_size),
        "SHA256": sha256_file(p),
    })
current_manifest = pd.DataFrame(current_records)
current_source_root = source_root_hash(current_manifest)
current_source_bytes = int(current_manifest["SizeBytes"].sum())
if (
    len(current_manifest) != EXPECTED_SOURCE_FILES
    or current_source_bytes != EXPECTED_SOURCE_BYTES
    or current_source_root != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError("Frozen Project 24 source root/manifest differs.")

# --------------------------------------------------------------------------------------------------
# 6. CHRONOLOGY, TIES, DATASET, RAW EXECUTIONS
# --------------------------------------------------------------------------------------------------
chronology = pd.read_csv(FIXED_CHRONOLOGY_PATH, low_memory=False)
required_chronology = {"BuildOrder", "Build", "StartedAtUTC", "Split"}
if not required_chronology.issubset(chronology.columns):
    raise RuntimeError("Frozen Project 24 chronology is missing required columns.")

chronology["Build"] = parse_int(chronology["Build"], "chronology.Build")
chronology["BuildOrder"] = parse_int(chronology["BuildOrder"], "chronology.BuildOrder")
chronology["StartedAtUTC"] = pd.to_datetime(chronology["StartedAtUTC"], errors="coerce", utc=True)
if chronology["StartedAtUTC"].isna().any():
    raise RuntimeError("Frozen chronology contains invalid timestamps.")
if not chronology["BuildOrder"].sort_values().tolist() == list(range(1, EXPECTED_BUILDS + 1)):
    raise RuntimeError("Frozen BuildOrder must be exactly 1..4286.")

training_builds = set(chronology.loc[chronology["Split"].eq("TRAIN"), "Build"].astype(int))
evaluation_builds = set(chronology.loc[chronology["Split"].eq("EVAL"), "Build"].astype(int))
all_builds = training_builds | evaluation_builds
if (
    len(chronology) != EXPECTED_BUILDS
    or len(training_builds) != EXPECTED_TRAIN_BUILDS
    or len(evaluation_builds) != EXPECTED_EVAL_BUILDS
    or bool(training_builds & evaluation_builds)
):
    raise RuntimeError("Frozen chronology dimensions differ.")

tie_sizes = chronology.groupby("StartedAtUTC").size()
timestamp_tie_groups_count = int(tie_sizes.gt(1).sum())
timestamp_tie_builds_count = int(tie_sizes.loc[tie_sizes.gt(1)].sum())
if (
    timestamp_tie_groups_count != EXPECTED_TIMESTAMP_TIE_GROUPS
    or timestamp_tie_builds_count != EXPECTED_TIMESTAMP_TIE_BUILDS
):
    raise RuntimeError("Timestamp-tie contract differs.")

timestamp_tie_groups = []
tie_records = []
tied_rows = chronology.loc[chronology["StartedAtUTC"].duplicated(keep=False)].copy()
for tie_no, (started_at, group) in enumerate(tied_rows.groupby("StartedAtUTC", sort=True), start=1):
    baseline = group.sort_values("BuildOrder", kind="mergesort")["Build"].astype(int).tolist()
    if len(baseline) != 2:
        raise RuntimeError(f"Project 24 expected two-build timestamp ties; got {baseline}")
    options = sorted([tuple(baseline), tuple(reversed(baseline))])
    timestamp_tie_groups.append({
        "TieGroup": tie_no,
        "StartedAtUTC": started_at,
        "BuildIDs": tuple(baseline),
        "Options": options,
    })
    tie_records.append({
        "TieGroup": tie_no,
        "StartedAtUTC": started_at.isoformat(),
        "BuildCount": len(baseline),
        "BuildIDsJSON": json.dumps(baseline),
        "PermutationCount": 2,
    })
timestamp_tie_groups_frame = pd.DataFrame(tie_records)

base_sequence = chronology.sort_values("BuildOrder", kind="mergesort")["Build"].astype(int).tolist()
base_position = {int(b): i for i, b in enumerate(base_sequence)}
build_chronology_map = chronology.set_index("Build")["BuildOrder"].astype(int).to_dict()
build_timestamp_map = chronology.set_index("Build")["StartedAtUTC"].to_dict()

dataset_header = pd.read_csv(SOURCE_DIR / "dataset.csv", nrows=0).columns.tolist()
exe_header = pd.read_csv(SOURCE_DIR / "exe.csv", nrows=0).columns.tolist()
mb = resolve_column(dataset_header, "Build", "dataset Build")
mt = resolve_column(dataset_header, "Test", "dataset Test")
mv = resolve_column(dataset_header, "Verdict", "dataset Verdict")
et = resolve_column(exe_header, "test", "exe test")
eb = resolve_column(exe_header, "build", "exe build")
ej = resolve_column(exe_header, "job", "exe job")
ev = resolve_column(exe_header, "verdict", "exe verdict")
ed = resolve_column(exe_header, "duration", "exe duration")
missing_rec = [f for f in REC_FEATURES if f not in dataset_header]
if missing_rec:
    raise RuntimeError("dataset.csv missing REC features: " + ", ".join(missing_rec))
predictors = [c for c in dataset_header if c not in {mb, mt, mv}]

print(f"\nLoading Project 24 model-ready REC source ({EXPECTED_MODEL_ROWS:,} rows expected).")
dataset = pd.read_csv(SOURCE_DIR / "dataset.csv", usecols=[mb, mt, mv] + REC_FEATURES, low_memory=False)
dataset[mb] = parse_int(dataset[mb], "dataset.Build")
dataset[mt] = parse_int(dataset[mt], "dataset.Test")
dataset[mv] = parse_int(dataset[mv], "dataset.Verdict")
dataset = dataset.rename(columns={mb: "Build", mt: "Test", mv: "Verdict"}).reset_index(drop=True)
dataset["_ModelRow"] = np.arange(len(dataset), dtype=np.int64)
for f in REC_FEATURES:
    dataset[f] = pd.to_numeric(dataset[f], errors="coerce")
    if not np.isfinite(dataset[f].to_numpy(dtype=float)).all():
        raise RuntimeError(f"Original {f} contains non-finite values.")

print(f"Loading Project 24 raw execution history ({EXPECTED_RAW_ROWS:,} rows expected).")
exe = pd.read_csv(SOURCE_DIR / "exe.csv", usecols=[et, eb, ej, ev, ed], low_memory=False)
exe[et] = parse_int(exe[et], "exe.test")
exe[eb] = parse_int(exe[eb], "exe.build")
exe[ev] = parse_int(exe[ev], "exe.verdict")
exe[ed] = pd.to_numeric(exe[ed], errors="coerce")
if exe[ed].isna().any() or not np.isfinite(exe[ed].to_numpy(dtype=float)).all() or exe[ed].lt(0).any():
    raise RuntimeError("Raw execution data contain invalid Duration values.")
exe = exe.rename(columns={et: "Test", eb: "Build", ej: "Job", ev: "Verdict", ed: "Duration"}).reset_index(drop=True)
exe["_RawRow"] = np.arange(len(exe), dtype=np.int64)
exe["ChronologyOrder"] = exe["Build"].map(build_chronology_map)
exe["StartedAtUTC"] = exe["Build"].map(build_timestamp_map)
if exe["ChronologyOrder"].isna().any() or exe["StartedAtUTC"].isna().any():
    raise RuntimeError("Raw execution rows could not be mapped to frozen chronology.")
exe["ChronologyOrder"] = parse_int(exe["ChronologyOrder"], "exe.ChronologyOrder")

if len(exe) != EXPECTED_RAW_ROWS or len(dataset) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Frozen Project 24 raw/model row counts differ.")
if len(dataset_header) != EXPECTED_DATASET_COLUMNS or len(predictors) != EXPECTED_PREDICTORS:
    raise RuntimeError("Frozen Project 24 model schema dimensions differ.")
if exe.duplicated(["Build", "Test"], keep=False).any() or dataset.duplicated(["Build", "Test"], keep=False).any():
    raise RuntimeError("Unexpected duplicate Build-Test rows appeared after Step 2A.")

raw_train_mask = exe["Build"].isin(training_builds)
raw_eval_mask = exe["Build"].isin(evaluation_builds)
model_train_mask = dataset["Build"].isin(training_builds)
model_eval_mask = dataset["Build"].isin(evaluation_builds)
raw_training_rows = int(raw_train_mask.sum())
raw_evaluation_rows = int(raw_eval_mask.sum())
raw_training_failures = int((raw_train_mask & exe["Verdict"].ne(SUCCESS_VERDICT_CODE)).sum())
raw_evaluation_failures = int((raw_eval_mask & exe["Verdict"].ne(SUCCESS_VERDICT_CODE)).sum())
model_training_rows = int(model_train_mask.sum())
model_evaluation_rows = int(model_eval_mask.sum())
model_training_failures = int((model_train_mask & dataset["Verdict"].ne(SUCCESS_VERDICT_CODE)).sum())
model_evaluation_failures = int((model_eval_mask & dataset["Verdict"].ne(SUCCESS_VERDICT_CODE)).sum())
model_failing_eval_builds = int(
    dataset.loc[model_eval_mask & dataset["Verdict"].ne(SUCCESS_VERDICT_CODE), "Build"].nunique()
)
raw_codes = set(exe["Verdict"].astype(int).unique().tolist())
model_codes = set(dataset["Verdict"].astype(int).unique().tolist())
allowed_verdicts = {SUCCESS_VERDICT_CODE, EXCEPTION_VERDICT_CODE, ASSERTION_VERDICT_CODE}

# --------------------------------------------------------------------------------------------------
# 7. STEP 2A MAPPING CONTRACT
# --------------------------------------------------------------------------------------------------
commit_audit = pd.read_csv(COMMIT_AUDIT_PATH, low_memory=False)
build_entity = pd.read_csv(BUILD_ENTITY_PATH, compression="gzip", low_memory=False)
mapping_source = pd.read_csv(MAPPING_INCOMPLETE_BUILDS_PATH, low_memory=False)

exact_matches = int(commit_audit["MatchType"].eq("EXACT").sum())
prefix_matches = int(commit_audit["MatchType"].eq("PREFIX").sum())
unmatched_tokens = int(commit_audit["MatchType"].eq("UNMATCHED").sum())
ambiguous_tokens = int(commit_audit["MatchType"].eq("AMBIGUOUS").sum())

build_entity["Build"] = parse_int(build_entity["Build"], "build_entity.Build")
build_entity["EntityId"] = parse_int(build_entity["EntityId"], "build_entity.EntityId")
changed_entities_by_build = {
    int(b): frozenset(g["EntityId"].astype(int).unique())
    for b, g in build_entity.groupby("Build", sort=False)
}
entity_changed_builds = {
    int(e): frozenset(g["Build"].astype(int).unique())
    for e, g in build_entity.groupby("EntityId", sort=False)
}
builds_with_entities = set(changed_entities_by_build)
builds_without_entities = all_builds - builds_with_entities

mapping_source["Build"] = parse_int(mapping_source["Build"], "mapping_incomplete.Build")
mapping_incomplete_builds = set(mapping_source["Build"].astype(int))
mapping_incomplete_partitions = set(mapping_source["Split"].astype(str))
mapping_rows_with_entities = int(pd.to_numeric(mapping_source["MatchedEntityRows"], errors="raise").gt(0).sum())

mapping_contract_ok = (
    len(commit_audit) == EXPECTED_COMMIT_TOKEN_ROWS
    and exact_matches == EXPECTED_EXACT_COMMIT_MATCHES
    and prefix_matches == EXPECTED_PREFIX_COMMIT_MATCHES
    and unmatched_tokens == EXPECTED_UNMATCHED_COMMIT_TOKENS
    and ambiguous_tokens == EXPECTED_AMBIGUOUS_COMMIT_TOKENS
    and len(builds_with_entities) == EXPECTED_BUILDS_WITH_MAPPED_ENTITIES
    and len(builds_without_entities) == EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES
    and len(build_entity) == EXPECTED_BUILD_ENTITY_ROWS
    and mapping_incomplete_builds == EXPECTED_MAPPING_INCOMPLETE_BUILDS
    and mapping_incomplete_partitions == EXPECTED_MAPPING_INCOMPLETE_PARTITIONS
    and mapping_rows_with_entities == EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES
)
if not mapping_contract_ok:
    raise RuntimeError("Project 24 Step 2A mapping contract differs; do not infer REC on a changed mapping boundary.")



def reconstruct_tie_objective(ordered_frame, model_rows):
    """Vectorized reconstruction of the 12 features allowed to choose a per-test tie order."""
    ordered_frame = ordered_frame.reset_index(drop=True)
    builds = ordered_frame["Build"].to_numpy(dtype=np.int64)
    verdicts = ordered_frame["Verdict"].to_numpy(dtype=np.int64)
    durations = ordered_frame["Duration"].to_numpy(dtype=np.float64)
    n = len(builds)
    position = {int(b): i for i, b in enumerate(builds)}
    positions = np.array([position[int(b)] for b in model_rows["Build"].astype(int)], dtype=np.int64)
    all_positions = np.arange(n, dtype=np.int64)

    failure = (verdicts != SUCCESS_VERDICT_CODE).astype(np.int64)
    assertion = (verdicts == ASSERTION_VERDICT_CODE).astype(np.int64)
    exception = (verdicts == EXCEPTION_VERDICT_CODE).astype(np.int64)
    transition = np.zeros(n, dtype=np.int64)
    if n > 1:
        transition[1:] = (verdicts[1:] != verdicts[:-1]).astype(np.int64)

    fpre = prefix_sum(failure)
    apre = prefix_sum(assertion)
    epre = prefix_sum(exception)
    tpre = prefix_sum(transition)

    history_len = positions.astype(float)
    recent_start = np.maximum(0, positions - RECENT_WINDOW)
    recent_len = (positions - recent_start).astype(float)

    last_fail_inc = np.maximum.accumulate(np.where(failure > 0, all_positions, -1))
    last_trans_inc = np.maximum.accumulate(np.where(transition > 0, all_positions, -1))
    prior_fail = np.full(len(positions), -1, dtype=np.int64)
    prior_trans = np.full(len(positions), -1, dtype=np.int64)
    has_history = positions > 0
    prior_fail[has_history] = last_fail_inc[positions[has_history] - 1]
    prior_trans[has_history] = last_trans_inc[positions[has_history] - 1]
    previous = np.maximum(positions - 1, 0)

    out = pd.DataFrame({"Build": model_rows["Build"].astype(int).to_numpy()})
    out["REC_LastFailureAge"] = np.where(
        prior_fail < 0, -1.0, (positions - 1 - prior_fail).astype(float)
    )
    out["REC_LastTransitionAge"] = np.where(
        prior_trans < 0, -1.0, (positions - 1 - prior_trans).astype(float)
    )
    out["REC_RecentFailRate"] = safe_divide(
        fpre[positions] - fpre[recent_start], recent_len
    )
    out["REC_RecentAssertRate"] = safe_divide(
        apre[positions] - apre[recent_start], recent_len
    )
    out["REC_RecentExcRate"] = safe_divide(
        epre[positions] - epre[recent_start], recent_len
    )
    out["REC_RecentTransitionRate"] = safe_divide(
        tpre[positions] - tpre[recent_start], recent_len
    )
    out["REC_TotalFailRate"] = safe_divide(fpre[positions], history_len)
    out["REC_TotalAssertRate"] = safe_divide(apre[positions], history_len)
    out["REC_TotalExcRate"] = safe_divide(epre[positions], history_len)
    out["REC_TotalTransitionRate"] = safe_divide(tpre[positions], history_len)
    out["REC_LastVerdict"] = np.where(has_history, verdicts[previous], -1).astype(float)
    out["REC_LastExeTime"] = np.where(has_history, durations[previous], -1.0)

    return out[["Build"] + TIE_INFERENCE_FEATURES]

# --------------------------------------------------------------------------------------------------
# 8. EXACT PER-TEST TIE ORDER INFERENCE — SCALABLE STATE-SPACE DP
# --------------------------------------------------------------------------------------------------
#
# The Project-23 exhaustive product search cannot be carried over literally because Project 24 has
# 23 independent two-build timestamp ties. We retain the same 12-feature exact objective and the
# same deterministic lexicographic tie-break, but solve only the ambiguous tests and only branch
# sequentially at exact timestamp ties. Equivalent sufficient-history states are merged.
# --------------------------------------------------------------------------------------------------

def tie_objective_mismatch(reconstructed, source_rows):
    if len(reconstructed) != len(source_rows):
        raise RuntimeError("Tie objective row-count mismatch.")
    aligned = source_rows.merge(
        reconstructed, on="Build", how="left", validate="one_to_one",
        suffixes=("_original", "_reconstructed"),
    )
    total = 0
    for f in TIE_INFERENCE_FEATURES:
        total += int((~np.isclose(
            aligned[f"{f}_original"].to_numpy(float),
            aligned[f"{f}_reconstructed"].to_numpy(float),
            rtol=DIRECT_RTOL, atol=DIRECT_ATOL, equal_nan=False,
        )).sum())
    return int(total)


# State is the complete sufficient history for the 12 tie-inference features:
# n, totals, last failure/transition indices, last verdict/duration, and recent-six event state.
INITIAL_DP_STATE = (0, 0, 0, 0, 0, -1, -1, -1, -1.0, tuple(), tuple())


def state_feature_vector(state):
    (
        n, total_fail, total_assert, total_exc, total_transition,
        last_failure_index, last_transition_index, last_verdict, last_duration,
        recent_verdicts, recent_transition_flags,
    ) = state

    if n == 0:
        return {f: -1.0 for f in TIE_INFERENCE_FEATURES}

    recent_n = len(recent_verdicts)
    return {
        "REC_LastFailureAge": -1.0 if last_failure_index < 0 else float(n - 1 - last_failure_index),
        "REC_LastTransitionAge": -1.0 if last_transition_index < 0 else float(n - 1 - last_transition_index),
        "REC_RecentFailRate": float(sum(v != SUCCESS_VERDICT_CODE for v in recent_verdicts) / recent_n),
        "REC_RecentAssertRate": float(sum(v == ASSERTION_VERDICT_CODE for v in recent_verdicts) / recent_n),
        "REC_RecentExcRate": float(sum(v == EXCEPTION_VERDICT_CODE for v in recent_verdicts) / recent_n),
        "REC_RecentTransitionRate": float(sum(recent_transition_flags) / recent_n),
        "REC_TotalFailRate": float(total_fail / n),
        "REC_TotalAssertRate": float(total_assert / n),
        "REC_TotalExcRate": float(total_exc / n),
        "REC_TotalTransitionRate": float(total_transition / n),
        "REC_LastVerdict": float(last_verdict),
        "REC_LastExeTime": float(last_duration),
    }


def state_matches_model_row(state, source_vector):
    reconstructed = state_feature_vector(state)
    return all(
        np.isclose(
            reconstructed[f], float(source_vector[f]),
            rtol=DIRECT_RTOL, atol=DIRECT_ATOL, equal_nan=False,
        )
        for f in TIE_INFERENCE_FEATURES
    )


def advance_dp_state(state, verdict, duration):
    (
        n, total_fail, total_assert, total_exc, total_transition,
        last_failure_index, last_transition_index, last_verdict, _last_duration,
        recent_verdicts, recent_transition_flags,
    ) = state

    verdict = int(verdict)
    duration = float(duration)
    transition = int(n > 0 and verdict != last_verdict)
    failure = int(verdict != SUCCESS_VERDICT_CODE)
    assertion = int(verdict == ASSERTION_VERDICT_CODE)
    exception = int(verdict == EXCEPTION_VERDICT_CODE)

    return (
        n + 1,
        total_fail + failure,
        total_assert + assertion,
        total_exc + exception,
        total_transition + transition,
        n if failure else last_failure_index,
        n if transition else last_transition_index,
        verdict,
        duration,
        (recent_verdicts + (verdict,))[-RECENT_WINDOW:],
        (recent_transition_flags + (transition,))[-RECENT_WINDOW:],
    )


def infer_exact_test_order_dp(test_frame, model_rows):
    """Exact state-space search; returns lexicographically smallest exact build sequence."""
    source_lookup = {
        int(r.Build): {f: float(getattr(r, f)) for f in TIE_INFERENCE_FEATURES}
        for r in model_rows[["Build"] + TIE_INFERENCE_FEATURES].itertuples(index=False)
    }

    # state -> lexicographically comparable selected tie-order signature
    states = {INITIAL_DP_STATE: tuple()}
    max_states = 1
    tie_groups = 0

    for started_at, group in test_frame.groupby("StartedAtUTC", sort=True):
        baseline = group.sort_values("ChronologyOrder", kind="mergesort")
        if len(baseline) == 1:
            options = [baseline]
        elif len(baseline) == 2:
            tie_groups += 1
            options = sorted(
                [baseline.iloc[[0, 1]].copy(), baseline.iloc[[1, 0]].copy()],
                key=lambda frame: tuple(frame["Build"].astype(int).tolist()),
            )
        else:
            raise RuntimeError(
                f"Test timestamp group has {len(baseline)} executions; Project 24 expects at most 2."
            )

        next_states = {}
        timestamp_ns = int(pd.Timestamp(started_at).value)

        for state, signature in states.items():
            for option in options:
                candidate_state = state
                valid = True
                build_tuple = tuple(option["Build"].astype(int).tolist())

                for row in option.itertuples(index=False):
                    build = int(row.Build)
                    source_vector = source_lookup.get(build)
                    if source_vector is not None and not state_matches_model_row(candidate_state, source_vector):
                        valid = False
                        break
                    candidate_state = advance_dp_state(candidate_state, int(row.Verdict), float(row.Duration))

                if not valid:
                    continue

                candidate_signature = signature
                if len(options) > 1:
                    candidate_signature = signature + ((timestamp_ns, build_tuple),)

                old = next_states.get(candidate_state)
                if old is None or candidate_signature < old:
                    next_states[candidate_state] = candidate_signature

        if not next_states:
            return {
                "Pass": False,
                "Reason": "NO_EXACT_STATE",
                "MaximumStates": max_states,
                "TieGroups": tie_groups,
                "Signature": tuple(),
            }

        states = next_states
        max_states = max(max_states, len(states))
        if len(states) > MAX_DP_STATES_PER_TEST:
            return {
                "Pass": False,
                "Reason": "DP_STATE_LIMIT_EXCEEDED",
                "MaximumStates": max_states,
                "TieGroups": tie_groups,
                "Signature": tuple(),
            }

    selected_signature = min(states.values())
    return {
        "Pass": True,
        "Reason": "EXACT_ZERO_MISMATCH_STATE",
        "MaximumStates": max_states,
        "TieGroups": tie_groups,
        "Signature": selected_signature,
    }


def apply_test_signature(test_frame, signature):
    selected = {int(ts): tuple(int(b) for b in order) for ts, order in signature}
    pieces = []
    for started_at, group in test_frame.groupby("StartedAtUTC", sort=True):
        baseline = group.sort_values("ChronologyOrder", kind="mergesort")
        timestamp_ns = int(pd.Timestamp(started_at).value)
        if len(baseline) > 1 and timestamp_ns in selected:
            rank = {b: i for i, b in enumerate(selected[timestamp_ns])}
            local = baseline.copy()
            local["_SelectedTieOrder"] = local["Build"].map(rank)
            local = local.sort_values(
                ["_SelectedTieOrder", "ChronologyOrder"], kind="mergesort"
            ).drop(columns=["_SelectedTieOrder"])
            pieces.append(local)
        else:
            pieces.append(baseline)
    return pd.concat(pieces, ignore_index=True)


print("\nFrozen timestamp-tie groups:")
display(timestamp_tie_groups_frame)

model_groups = {
    int(test_id): group[["Build"] + TIE_INFERENCE_FEATURES].copy()
    for test_id, group in dataset.groupby("Test", sort=False)
}
exe_group_indices = {
    int(test_id): np.asarray(indices, dtype=np.int64)
    for test_id, indices in exe.groupby("Test", sort=False).indices.items()
}
tests = sorted(exe_group_indices)
model_test_sizes = dataset.groupby("Test", sort=False).size().to_dict()

inferred_order = np.full(len(exe), -1, dtype=np.int64)
search_records = []
order_inference_started = time.perf_counter()
baseline_exact_tie_tests = 0
dp_searched_tests = 0
raw_only_tie_tests = 0
maximum_dp_states_observed = 1

for test_number, test_id in enumerate(tests, start=1):
    idx = exe_group_indices[test_id]
    test_frame = (
        exe.iloc[idx]
        .sort_values(["StartedAtUTC", "ChronologyOrder"], kind="mergesort")
        .reset_index(drop=True)
    )
    tie_group_count = int((test_frame.groupby("StartedAtUTC").size() > 1).sum())
    model_rows = model_groups.get(test_id)
    model_row_count = int(model_test_sizes.get(test_id, 0))
    baseline_mismatch = 0
    selected = test_frame
    mode = "DIRECT_FROZEN_CHRONOLOGY"
    dp_states = 1
    dp_reason = ""
    minimum_mismatch = 0

    if tie_group_count > 0:
        if model_rows is None:
            raw_only_tie_tests += 1
            mode = "RAW_ONLY_TIE_FROZEN_CHRONOLOGY"
        else:
            baseline_rec = reconstruct_tie_objective(
                test_frame[["Build", "Verdict", "Duration"]], model_rows[["Build"]]
            )
            baseline_mismatch = tie_objective_mismatch(baseline_rec, model_rows)
            if baseline_mismatch == 0:
                baseline_exact_tie_tests += 1
                mode = "BASELINE_EXACT_ZERO_MISMATCH"
            else:
                dp_searched_tests += 1
                mode = "EXACT_STATE_DP"
                dp = infer_exact_test_order_dp(test_frame, model_rows)
                dp_states = int(dp["MaximumStates"])
                maximum_dp_states_observed = max(maximum_dp_states_observed, dp_states)
                dp_reason = str(dp["Reason"])
                if not dp["Pass"]:
                    search_records.append({
                        "Test": int(test_id), "RawExecutionRows": len(test_frame),
                        "ModelReadyRows": model_row_count, "TimestampTieGroupsForTest": tie_group_count,
                        "SearchMode": mode, "BaselineMismatchValues": baseline_mismatch,
                        "MinimumMismatchValues": -1, "DPMaximumStates": dp_states,
                        "DPReason": dp_reason, "SelectedTieOrdersJSON": "[]",
                    })
                    search_audit = pd.DataFrame(search_records)
                    print("\nExact per-test DP failure:")
                    display(search_audit.tail(1))
                    raise RuntimeError(
                        "Project 24 exact per-test tie inference failed. No Step 2B checkpoint was written."
                    )
                selected = apply_test_signature(test_frame, dp["Signature"])
                selected_rec = reconstruct_tie_objective(
                    selected[["Build", "Verdict", "Duration"]], model_rows[["Build"]]
                )
                minimum_mismatch = tie_objective_mismatch(selected_rec, model_rows)
                if minimum_mismatch != 0:
                    raise RuntimeError(
                        f"Exact DP selected a non-zero-mismatch order for Test={test_id}: {minimum_mismatch}"
                    )

    selected_raw_rows = selected["_RawRow"].to_numpy(dtype=np.int64)
    inferred_order[selected_raw_rows] = np.arange(1, len(selected_raw_rows) + 1, dtype=np.int64)

    selected_signature_json = "[]"
    if mode == "EXACT_STATE_DP":
        selected_signature_json = json.dumps(
            [[int(ts), list(map(int, order))] for ts, order in dp["Signature"]]
        )

    search_records.append({
        "Test": int(test_id),
        "RawExecutionRows": len(test_frame),
        "ModelReadyRows": model_row_count,
        "TimestampTieGroupsForTest": tie_group_count,
        "SearchMode": mode,
        "BaselineMismatchValues": int(baseline_mismatch),
        "MinimumMismatchValues": int(minimum_mismatch),
        "DPMaximumStates": int(dp_states),
        "DPReason": dp_reason,
        "SelectedTieOrdersJSON": selected_signature_json,
    })

    if test_number % 250 == 0 or test_number == len(tests):
        print(
            "Per-test order inference progress:",
            test_number, "/", len(tests),
            "| elapsed seconds:", round(time.perf_counter() - order_inference_started, 2),
        )

if (inferred_order < 1).any():
    raise RuntimeError("Per-test execution-order inference did not cover every raw row.")

search_audit = pd.DataFrame(search_records)
tests_nonzero_order = int(search_audit["MinimumMismatchValues"].gt(0).sum())
total_order_mismatches = int(
    search_audit.loc[search_audit["MinimumMismatchValues"].ge(0), "MinimumMismatchValues"].sum()
)
if tests_nonzero_order or total_order_mismatches:
    print("\nTests whose exact tie-order objective did not reproduce the source:")
    display(search_audit.loc[search_audit["MinimumMismatchValues"].gt(0)])
    raise RuntimeError("Project 24 exact tie-order reconstruction failed.")

exe["InferredTestOrder"] = inferred_order
inferred_execution_order = (
    exe.sort_values(["Test", "InferredTestOrder"], kind="mergesort").reset_index(drop=True).copy()
)
if inferred_execution_order.duplicated(["Test", "InferredTestOrder"], keep=False).any():
    raise RuntimeError("Inferred execution order contains duplicate Test-order keys.")

model_ready_tests = int(search_audit["ModelReadyRows"].gt(0).sum())
raw_only_tests = int(search_audit["ModelReadyRows"].eq(0).sum())
tests_touching_ties = int(search_audit["TimestampTieGroupsForTest"].gt(0).sum())
order_inference_seconds = float(time.perf_counter() - order_inference_started)

print("\nPer-test tie inference summary:")
display(pd.DataFrame([
    ["Tests", len(tests)],
    ["Model-ready tests", model_ready_tests],
    ["Raw-only tests", raw_only_tests],
    ["Tests touching timestamp ties", tests_touching_ties],
    ["Tie tests exact under frozen baseline", baseline_exact_tie_tests],
    ["Raw-only tie tests using frozen baseline", raw_only_tie_tests],
    ["Tie tests requiring exact DP", dp_searched_tests],
    ["Tests with non-zero minimum mismatch", tests_nonzero_order],
    ["Total minimum mismatch values", total_order_mismatches],
    ["Maximum exact-DP states observed", maximum_dp_states_observed],
    ["Inference seconds", order_inference_seconds],
], columns=["Metric", "Value"]))


# --------------------------------------------------------------------------------------------------
# 9. GLOBAL REC_Age ORDER — EXACT 23-VARIABLE BINARY CONSTRAINT SOLVER
# --------------------------------------------------------------------------------------------------
#
# For each two-build timestamp tie [A,B] in frozen order:
#   x=0 -> [A,B], position shifts (A:+0, B:+0)
#   x=1 -> [B,A], position shifts (A:+1, B:-1)
# Therefore each source REC_Age row yields an exact integer equation involving at most two x values.
# The resulting binary CSP is solved exactly with domain propagation + chronological lexicographic DFS.
# --------------------------------------------------------------------------------------------------

first_build_by_test = (
    inferred_execution_order.groupby("Test", sort=False).first()["Build"].astype(int).to_dict()
)
first_build_array = dataset["Test"].map(first_build_by_test)
if first_build_array.isna().any():
    raise RuntimeError("A model-ready test is missing its inferred first raw build.")
first_build_array = first_build_array.astype("int64").to_numpy()
current_build_array = dataset["Build"].to_numpy(dtype=np.int64)
original_age = dataset["REC_Age"].to_numpy(dtype=np.float64)

tie_variable_for_build = {}
tie_shift_coefficient = {}
preferred_value = {}
tie_variable_records = []

for variable, tie in enumerate(timestamp_tie_groups):
    a, b = map(int, tie["BuildIDs"])
    if base_position[b] != base_position[a] + 1:
        raise RuntimeError("Tied builds are not adjacent in frozen chronology.")
    tie_variable_for_build[a] = variable
    tie_variable_for_build[b] = variable
    tie_shift_coefficient[a] = +1
    tie_shift_coefficient[b] = -1
    baseline_tuple = (a, b)
    swapped_tuple = (b, a)
    preferred_value[variable] = 0 if baseline_tuple <= swapped_tuple else 1
    tie_variable_records.append({
        "Variable": variable,
        "TieGroup": int(tie["TieGroup"]),
        "StartedAtUTC": pd.Timestamp(tie["StartedAtUTC"]).isoformat(),
        "BaselineBuild1": a,
        "BaselineBuild2": b,
        "PreferredLexicographicValue": preferred_value[variable],
    })

tie_variable_frame = pd.DataFrame(tie_variable_records)
constraint_counter = {}

baseline_age = np.fromiter(
    (
        float(base_position[int(current)] - base_position[int(first)])
        for current, first in zip(current_build_array, first_build_array)
    ),
    dtype=np.float64,
    count=len(dataset),
)
baseline_age_mismatch_rows = int((~np.isclose(
    original_age, baseline_age, rtol=DIRECT_RTOL, atol=DIRECT_ATOL, equal_nan=False
)).sum())

for current, first, source_value in zip(current_build_array, first_build_array, original_age):
    current = int(current)
    first = int(first)
    base_value = base_position[current] - base_position[first]
    residual_float = float(source_value - base_value)
    residual = int(round(residual_float))
    if not np.isclose(residual_float, residual, rtol=0, atol=DIRECT_ATOL):
        raise RuntimeError(
            f"REC_Age residual is non-integral: current={current}, first={first}, residual={residual_float}"
        )

    coefficients = {}
    if current in tie_variable_for_build:
        var = tie_variable_for_build[current]
        coefficients[var] = coefficients.get(var, 0) + tie_shift_coefficient[current]
    if first in tie_variable_for_build:
        var = tie_variable_for_build[first]
        coefficients[var] = coefficients.get(var, 0) - tie_shift_coefficient[first]

    terms = tuple(sorted((int(v), int(c)) for v, c in coefficients.items() if int(c) != 0))
    if not terms:
        if residual != 0:
            raise RuntimeError(
                f"REC_Age mismatch cannot be explained by timestamp ties: current={current}, first={first}"
            )
        continue

    key = (terms, residual)
    constraint_counter[key] = constraint_counter.get(key, 0) + 1

age_constraints = list(constraint_counter.keys())
age_constraint_frame = pd.DataFrame([
    {
        "Constraint": i + 1,
        "TermsJSON": json.dumps([[int(v), int(c)] for v, c in terms]),
        "Target": int(target),
        "SourceRows": int(constraint_counter[(terms, target)]),
    }
    for i, (terms, target) in enumerate(age_constraints)
])


def solve_binary_age_csp(variable_count, constraints, preferred):
    # Domains and pairwise relation tables.
    domains = {v: {0, 1} for v in range(variable_count)}
    pair_allowed = {}

    for terms, target in constraints:
        vars_ = [v for v, _ in terms]
        if len(vars_) == 1:
            v, c = terms[0]
            allowed = {x for x in (0, 1) if c * x == target}
            domains[v] &= allowed
        elif len(vars_) == 2:
            (v1, c1), (v2, c2) = terms
            if v1 == v2:
                raise RuntimeError("Collapsed REC_Age constraint unexpectedly retained duplicate variable.")
            if v1 > v2:
                v1, v2 = v2, v1
                c1, c2 = c2, c1
            allowed = {(x1, x2) for x1 in (0, 1) for x2 in (0, 1) if c1*x1 + c2*x2 == target}
            key = (v1, v2)
            pair_allowed[key] = allowed if key not in pair_allowed else pair_allowed[key] & allowed
        else:
            raise RuntimeError(f"REC_Age constraint has >2 variables: {terms}")

    if any(len(d) == 0 for d in domains.values()) or any(len(a) == 0 for a in pair_allowed.values()):
        return None

    neighbors = {v: set() for v in range(variable_count)}
    for v1, v2 in pair_allowed:
        neighbors[v1].add(v2)
        neighbors[v2].add(v1)

    def propagate(local_domains):
        changed = True
        while changed:
            changed = False
            for (v1, v2), allowed in pair_allowed.items():
                d1 = local_domains[v1]
                d2 = local_domains[v2]
                new1 = {x1 for x1 in d1 if any((x1, x2) in allowed for x2 in d2)}
                new2 = {x2 for x2 in d2 if any((x1, x2) in allowed for x1 in new1)}
                if not new1 or not new2:
                    return False
                if new1 != d1:
                    local_domains[v1] = new1
                    changed = True
                if new2 != d2:
                    local_domains[v2] = new2
                    changed = True
        return True

    if not propagate(domains):
        return None

    memo = set()

    def key_for(local_domains):
        return tuple(tuple(sorted(local_domains[v])) for v in range(variable_count))

    def dfs(local_domains):
        key = key_for(local_domains)
        if key in memo:
            return None

        if all(len(local_domains[v]) == 1 for v in range(variable_count)):
            return {v: next(iter(local_domains[v])) for v in range(variable_count)}

        # Chronological variable choice preserves the lexicographic build-order objective.
        v = next(v for v in range(variable_count) if len(local_domains[v]) > 1)
        values = [preferred[v], 1 - preferred[v]]

        for value in values:
            candidate = {k: set(vals) for k, vals in local_domains.items()}
            candidate[v] = {int(value)}
            if propagate(candidate):
                solved = dfs(candidate)
                if solved is not None:
                    return solved

        memo.add(key)
        return None

    return dfs(domains)


age_search_started = time.perf_counter()
age_assignment = solve_binary_age_csp(
    len(timestamp_tie_groups), age_constraints, preferred_value
)
age_search_seconds = float(time.perf_counter() - age_search_started)
if age_assignment is None:
    print("\nREC_Age exact constraints:")
    display(age_constraint_frame)
    raise RuntimeError("No exact binary timestamp-tie assignment reproduces Project 24 REC_Age.")

best_sequence = list(base_sequence)
selected_global_orders = []
for variable, tie in enumerate(timestamp_tie_groups):
    baseline = tuple(map(int, tie["BuildIDs"]))
    selected = baseline if age_assignment[variable] == 0 else tuple(reversed(baseline))
    positions = sorted(base_position[b] for b in baseline)
    for p, b in zip(positions, selected):
        best_sequence[p] = int(b)
    selected_global_orders.append(list(selected))

global_build_position = {int(b): i for i, b in enumerate(best_sequence)}
reconstructed_age = np.fromiter(
    (
        float(global_build_position[int(current)] - global_build_position[int(first)])
        for current, first in zip(current_build_array, first_build_array)
    ),
    dtype=np.float64,
    count=len(dataset),
)
best_age = int((~np.isclose(
    original_age, reconstructed_age, rtol=DIRECT_RTOL, atol=DIRECT_ATOL, equal_nan=False
)).sum())
if best_age != 0:
    raise RuntimeError(f"Exact REC_Age constraint solution still has {best_age} mismatching rows.")

build_order_sha256 = hashlib.sha256(",".join(map(str, best_sequence)).encode("utf-8")).hexdigest()
global_age_result = pd.DataFrame([{
    "Constraint": 0,
    "TermsJSON": "GLOBAL_RESULT",
    "Target": 0,
    "SourceRows": len(dataset),
    "BaselineAgeMismatchRows": baseline_age_mismatch_rows,
    "SelectedAgeMismatchRows": best_age,
    "AgeConstraintCount": len(age_constraints),
    "SelectedTieOrdersJSON": json.dumps(selected_global_orders),
    "BuildOrderSHA256": build_order_sha256,
    "AgeSearchSeconds": age_search_seconds,
}])
global_age_order_search = pd.concat([global_age_result, age_constraint_frame], ignore_index=True, sort=False)
frozen_global_build_order = pd.DataFrame({
    "GlobalBuildOrder": np.arange(1, len(best_sequence) + 1, dtype=np.int64),
    "BuildID": best_sequence,
})
frozen_global_build_order["StartedAtUTC"] = frozen_global_build_order["BuildID"].map(build_timestamp_map)

print("\nGlobal REC_Age exact constraint result:")
display(global_age_result)

# --------------------------------------------------------------------------------------------------
# 10. FULL 19-FEATURE CLEAN RECONSTRUCTION
# --------------------------------------------------------------------------------------------------
started = time.perf_counter()
clean_reconstructed = reconstruct_all_rec(
    inferred_execution_order, dataset[["Build", "Test"]], global_build_position,
    changed_entities_by_build, entity_changed_builds,
)
reconstruction_seconds = float(time.perf_counter() - started)
if len(clean_reconstructed) != EXPECTED_MODEL_ROWS or clean_reconstructed.duplicated(["Build", "Test"], keep=False).any():
    raise RuntimeError("Clean REC reconstruction row/key contract differs.")

aligned_keys = dataset[["Build", "Test", "_ModelRow"]].merge(clean_reconstructed[["Build", "Test"]], on=["Build", "Test"], how="left", validate="one_to_one", indicator=True)
if int(aligned_keys["_merge"].ne("both").sum()) != 0:
    raise RuntimeError("Clean REC reconstruction is missing requested model rows.")

# --------------------------------------------------------------------------------------------------
# 11. DIRECT COMPARISON, MAPPING BOUNDARY, CLEAN ANCHOR
# --------------------------------------------------------------------------------------------------
comparison_records = []; mismatch_examples = []; mismatch_masks = {}
anchor_offsets = dataset[["Build", "Test"]].copy()
for f in REC_FEATURES:
    original = dataset[f].to_numpy(dtype=np.float64); reconstructed = clean_reconstructed[f].to_numpy(dtype=np.float64)
    diff = original - reconstructed
    mismatch = ~np.isclose(original, reconstructed, rtol=DIRECT_RTOL, atol=DIRECT_ATOL, equal_nan=False)
    mismatch_masks[f] = mismatch; count = int(mismatch.sum()); anchor_offsets[f] = diff
    comparison_records.append({
        "Feature": f, "FeatureClass": "VERDICT_DEPENDENT" if f in VERDICT_DEPENDENT_REC else "VERDICT_INDEPENDENT",
        "FileHistoryFeature": f in FILE_HISTORY_REC, "AnchorAllowedTimingResidual": f in ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES,
        "ModelRows": len(dataset), "DirectMatchValues": len(dataset) - count, "DirectMismatchValues": count,
        "MaxAbsDifference": float(np.max(np.abs(diff))),
    })
    for i in np.flatnonzero(mismatch)[:20]:
        mismatch_examples.append({"Feature": f, "Build": int(dataset.iloc[i]["Build"]), "Test": int(dataset.iloc[i]["Test"]),
                                  "Original": float(original[i]), "Reconstructed": float(reconstructed[i]), "Difference": float(diff[i]),
                                  "MappingIncompleteBuild": int(dataset.iloc[i]["Build"]) in mapping_incomplete_builds})
comparison_summary = pd.DataFrame(comparison_records)
mismatch_examples_frame = pd.DataFrame(mismatch_examples, columns=[
    "Feature", "Build", "Test", "Original", "Reconstructed", "Difference", "MappingIncompleteBuild"
])
direct_mismatch_values = int(comparison_summary["DirectMismatchValues"].sum())
verdict_dependent_direct_mismatches = int(comparison_summary.loc[comparison_summary["Feature"].isin(VERDICT_DEPENDENT_REC), "DirectMismatchValues"].sum())
verdict_independent_direct_mismatches = int(comparison_summary.loc[comparison_summary["Feature"].isin(VERDICT_INDEPENDENT_REC), "DirectMismatchValues"].sum())
file_history_direct_mismatches = int(comparison_summary.loc[comparison_summary["Feature"].isin(FILE_HISTORY_REC), "DirectMismatchValues"].sum())
non_file_direct_mismatches = int(comparison_summary.loc[~comparison_summary["Feature"].isin(FILE_HISTORY_REC), "DirectMismatchValues"].sum())
anchor_allowed_direct_mismatches = int(comparison_summary.loc[comparison_summary["Feature"].isin(ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES), "DirectMismatchValues"].sum())
anchor_disallowed_non_file_direct_mismatches = int(comparison_summary.loc[
    ~comparison_summary["Feature"].isin(FILE_HISTORY_REC + ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES), "DirectMismatchValues"
].sum())

mapping_model_mask = dataset["Build"].isin(mapping_incomplete_builds).to_numpy(dtype=bool)
file_mismatches_outside_mapping_incomplete = sum(int((mismatch_masks[f] & ~mapping_model_mask).sum()) for f in FILE_HISTORY_REC)
unmatched_mapping_effect_confined = file_mismatches_outside_mapping_incomplete == 0
direct_residuals_confined = anchor_disallowed_non_file_direct_mismatches == 0 and unmatched_mapping_effect_confined

# Detailed 49-build mapping-boundary audit.
model_rows_by_build = dataset.groupby("Build").size().rename("ModelRows")
file_fail_by_build = pd.Series(mismatch_masks["REC_MaxTestFileFailRate"].astype(np.int64), index=dataset["Build"]).groupby(level=0).sum().rename("REC_MaxTestFileFailRate_MismatchValues")
file_trans_by_build = pd.Series(mismatch_masks["REC_MaxTestFileTransitionRate"].astype(np.int64), index=dataset["Build"]).groupby(level=0).sum().rename("REC_MaxTestFileTransitionRate_MismatchValues")
unmatched_mapping_audit = (mapping_source.merge(model_rows_by_build, left_on="Build", right_index=True, how="left")
                           .merge(file_fail_by_build, left_on="Build", right_index=True, how="left")
                           .merge(file_trans_by_build, left_on="Build", right_index=True, how="left"))
for c in ["ModelRows", "REC_MaxTestFileFailRate_MismatchValues", "REC_MaxTestFileTransitionRate_MismatchValues"]:
    unmatched_mapping_audit[c] = unmatched_mapping_audit[c].fillna(0).astype(int)
unmatched_mapping_audit["FileHistoryMismatchValues"] = unmatched_mapping_audit["REC_MaxTestFileFailRate_MismatchValues"] + unmatched_mapping_audit["REC_MaxTestFileTransitionRate_MismatchValues"]

anchor_validation_records = []; anchored_mismatch_values = 0; failed_anchor_features = 0
for f in REC_FEATURES:
    reproduced = clean_reconstructed[f].to_numpy(float) + anchor_offsets[f].to_numpy(float)
    original = dataset[f].to_numpy(float)
    mismatch = int((~np.isclose(reproduced, original, rtol=ANCHOR_RTOL, atol=ANCHOR_ATOL, equal_nan=False)).sum())
    anchored_mismatch_values += mismatch; failed_anchor_features += int(mismatch > 0)
    anchor_validation_records.append({"Feature": f, "NonZeroOffsetValues": int((anchor_offsets[f].to_numpy(float) != 0).sum()),
                                      "AnchoredMismatchValues": mismatch, "Pass": mismatch == 0})
anchor_validation = pd.DataFrame(anchor_validation_records)
offset_matrix = anchor_offsets[REC_FEATURES].to_numpy(dtype=np.float64)
rows_with_any_nonzero_anchor_offset = int(np.any(offset_matrix != 0, axis=1).sum())
nonzero_anchor_offset_values = int((offset_matrix != 0).sum())
zero_percent_clean_reproduced_exactly = anchored_mismatch_values == 0

# --------------------------------------------------------------------------------------------------

# --------------------------------------------------------------------------------------------------
# 12. VALIDATION — NOTHING IS WRITTEN BEFORE THIS GATE PASSES
# --------------------------------------------------------------------------------------------------
validation_records = []
def C(name, exp, act, ok):
    add_check(validation_records, name, exp, act, ok)

C("Step 1B passed", EXPECTED_STEP1B_STATUS, selection.get("Status"), selection.get("Status") == EXPECTED_STEP1B_STATUS)
C("Step 2A passed", EXPECTED_STEP2A_STATUS, step2a_status.get("Status"), step2a_status.get("Status") == EXPECTED_STEP2A_STATUS)
C("Step 2A output-manifest failures", 0, step2a_manifest_failures, step2a_manifest_failures == 0)
C("Selection checkpoint SHA-256", EXPECTED_SELECTION_SHA256, selection_sha256, selection_sha256 == EXPECTED_SELECTION_SHA256)
C("Source root SHA-256", EXPECTED_SOURCE_ROOT_SHA256, current_source_root, current_source_root == EXPECTED_SOURCE_ROOT_SHA256)
C("Source files", EXPECTED_SOURCE_FILES, len(current_manifest), len(current_manifest) == EXPECTED_SOURCE_FILES)
C("Source bytes", EXPECTED_SOURCE_BYTES, current_source_bytes, current_source_bytes == EXPECTED_SOURCE_BYTES)
C("Builds", EXPECTED_BUILDS, len(chronology), len(chronology) == EXPECTED_BUILDS)
C("Training builds", EXPECTED_TRAIN_BUILDS, len(training_builds), len(training_builds) == EXPECTED_TRAIN_BUILDS)
C("Evaluation builds", EXPECTED_EVAL_BUILDS, len(evaluation_builds), len(evaluation_builds) == EXPECTED_EVAL_BUILDS)
C("Timestamp tie groups", EXPECTED_TIMESTAMP_TIE_GROUPS, timestamp_tie_groups_count, timestamp_tie_groups_count == EXPECTED_TIMESTAMP_TIE_GROUPS)
C("Timestamp tie builds", EXPECTED_TIMESTAMP_TIE_BUILDS, timestamp_tie_builds_count, timestamp_tie_builds_count == EXPECTED_TIMESTAMP_TIE_BUILDS)
C("All timestamp ties are two-build ties", True, all(len(x["BuildIDs"]) == 2 for x in timestamp_tie_groups), all(len(x["BuildIDs"]) == 2 for x in timestamp_tie_groups))
C("Raw rows", EXPECTED_RAW_ROWS, len(exe), len(exe) == EXPECTED_RAW_ROWS)
C("Raw training rows", EXPECTED_RAW_TRAIN_ROWS, raw_training_rows, raw_training_rows == EXPECTED_RAW_TRAIN_ROWS)
C("Raw evaluation rows", EXPECTED_RAW_EVAL_ROWS, raw_evaluation_rows, raw_evaluation_rows == EXPECTED_RAW_EVAL_ROWS)
C("Raw training failures", EXPECTED_RAW_TRAIN_FAILURES, raw_training_failures, raw_training_failures == EXPECTED_RAW_TRAIN_FAILURES)
C("Raw evaluation failures", EXPECTED_RAW_EVAL_FAILURES, raw_evaluation_failures, raw_evaluation_failures == EXPECTED_RAW_EVAL_FAILURES)
C("Model rows", EXPECTED_MODEL_ROWS, len(dataset), len(dataset) == EXPECTED_MODEL_ROWS)
C("Model training rows", EXPECTED_MODEL_TRAIN_ROWS, model_training_rows, model_training_rows == EXPECTED_MODEL_TRAIN_ROWS)
C("Model evaluation rows", EXPECTED_MODEL_EVAL_ROWS, model_evaluation_rows, model_evaluation_rows == EXPECTED_MODEL_EVAL_ROWS)
C("Model training failures", EXPECTED_MODEL_TRAIN_FAILURES, model_training_failures, model_training_failures == EXPECTED_MODEL_TRAIN_FAILURES)
C("Model evaluation failures", EXPECTED_MODEL_EVAL_FAILURES, model_evaluation_failures, model_evaluation_failures == EXPECTED_MODEL_EVAL_FAILURES)
C("Model failing evaluation builds", EXPECTED_MODEL_FAILING_EVAL_BUILDS, model_failing_eval_builds, model_failing_eval_builds == EXPECTED_MODEL_FAILING_EVAL_BUILDS)
C("Dataset columns", EXPECTED_DATASET_COLUMNS, len(dataset_header), len(dataset_header) == EXPECTED_DATASET_COLUMNS)
C("Predictor columns", EXPECTED_PREDICTORS, len(predictors), len(predictors) == EXPECTED_PREDICTORS)
C("REC features", 19, len(REC_FEATURES), len(REC_FEATURES) == 19)
C("Tie-inference features", 12, len(TIE_INFERENCE_FEATURES), len(TIE_INFERENCE_FEATURES) == 12)
C("Official verdict-code subset", sorted(allowed_verdicts), sorted(raw_codes | model_codes), (raw_codes | model_codes).issubset(allowed_verdicts))
C("Commit-token rows", EXPECTED_COMMIT_TOKEN_ROWS, len(commit_audit), len(commit_audit) == EXPECTED_COMMIT_TOKEN_ROWS)
C("Exact commit matches", EXPECTED_EXACT_COMMIT_MATCHES, exact_matches, exact_matches == EXPECTED_EXACT_COMMIT_MATCHES)
C("Prefix commit matches", EXPECTED_PREFIX_COMMIT_MATCHES, prefix_matches, prefix_matches == EXPECTED_PREFIX_COMMIT_MATCHES)
C("Unmatched commit tokens", EXPECTED_UNMATCHED_COMMIT_TOKENS, unmatched_tokens, unmatched_tokens == EXPECTED_UNMATCHED_COMMIT_TOKENS)
C("Ambiguous commit tokens", EXPECTED_AMBIGUOUS_COMMIT_TOKENS, ambiguous_tokens, ambiguous_tokens == EXPECTED_AMBIGUOUS_COMMIT_TOKENS)
C("Builds with mapped entities", EXPECTED_BUILDS_WITH_MAPPED_ENTITIES, len(builds_with_entities), len(builds_with_entities) == EXPECTED_BUILDS_WITH_MAPPED_ENTITIES)
C("Builds without mapped entities", EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES, len(builds_without_entities), len(builds_without_entities) == EXPECTED_BUILDS_WITHOUT_MAPPED_ENTITIES)
C("Build-entity rows", EXPECTED_BUILD_ENTITY_ROWS, len(build_entity), len(build_entity) == EXPECTED_BUILD_ENTITY_ROWS)
C("Mapping-incomplete build set", sorted(EXPECTED_MAPPING_INCOMPLETE_BUILDS), sorted(mapping_incomplete_builds), mapping_incomplete_builds == EXPECTED_MAPPING_INCOMPLETE_BUILDS)
C("Mapping-incomplete partitions", sorted(EXPECTED_MAPPING_INCOMPLETE_PARTITIONS), sorted(mapping_incomplete_partitions), mapping_incomplete_partitions == EXPECTED_MAPPING_INCOMPLETE_PARTITIONS)
C("Mapping-incomplete rows retaining mapped entities", EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES, mapping_rows_with_entities, mapping_rows_with_entities == EXPECTED_MAPPING_INCOMPLETE_ROWS_WITH_ENTITIES)
C("Raw execution-order rows", EXPECTED_RAW_ROWS, len(inferred_execution_order), len(inferred_execution_order) == EXPECTED_RAW_ROWS)
C("Tests with non-zero tie-order mismatch", 0, tests_nonzero_order, tests_nonzero_order == 0)
C("Total tie-order mismatch values", 0, total_order_mismatches, total_order_mismatches == 0)
C("Maximum exact-DP states within cap", f"<= {MAX_DP_STATES_PER_TEST}", maximum_dp_states_observed, maximum_dp_states_observed <= MAX_DP_STATES_PER_TEST)
C("Global REC_Age exact constraint solution", True, age_assignment is not None, age_assignment is not None)
C("Global REC_Age minimum mismatch rows", 0, best_age, best_age == 0)
C("Global build-order rows", EXPECTED_BUILDS, len(frozen_global_build_order), len(frozen_global_build_order) == EXPECTED_BUILDS)
C("Clean REC reconstructed rows", EXPECTED_MODEL_ROWS, len(clean_reconstructed), len(clean_reconstructed) == EXPECTED_MODEL_ROWS)
C("Anchor-disallowed non-file direct mismatch values", 0, anchor_disallowed_non_file_direct_mismatches, anchor_disallowed_non_file_direct_mismatches == 0)
C("Anchor-allowed timing direct residual values", ">= 0 (diagnostic; clean anchor must remove exactly)", anchor_allowed_direct_mismatches, anchor_allowed_direct_mismatches >= 0)
C("File-history mismatches outside mapping-incomplete builds", 0, file_mismatches_outside_mapping_incomplete, file_mismatches_outside_mapping_incomplete == 0)
C("Direct residuals confined to accepted boundaries", True, direct_residuals_confined, direct_residuals_confined)
C("Unmatched mapping effect confined", True, unmatched_mapping_effect_confined, unmatched_mapping_effect_confined)
C("Failed clean-anchor features", 0, failed_anchor_features, failed_anchor_features == 0)
C("Anchored mismatch values", 0, anchored_mismatch_values, anchored_mismatch_values == 0)
C("0% clean dataset reproduced exactly", True, zero_percent_clean_reproduced_exactly, zero_percent_clean_reproduced_exactly)
C("Registry rows", EXPECTED_REGISTERED_PROJECTS, len(registry), len(registry) == EXPECTED_REGISTERED_PROJECTS)
for n, identity in REQUIRED_REGISTERED_IDENTITIES.items():
    actual = str(registry.loc[registry_numbers.eq(n), p_col].iloc[0])
    C(f"Project {n} frozen identity", identity, actual, actual == identity)
C("Active reservations", EXPECTED_ACTIVE_RESERVATIONS, selection.get("ActiveReservations"), selection.get("ActiveReservations") == EXPECTED_ACTIVE_RESERVATIONS)
C("Runtime-priority ranking rule", EXPECTED_RUNTIME_PRIORITY_RULE, selection.get("RuntimePriorityRule"), selection.get("RuntimePriorityRule") == EXPECTED_RUNTIME_PRIORITY_RULE)
C("Project 24 registry rows", 0, int(registry_numbers.eq(PROJECT_NUMBER).sum()), int(registry_numbers.eq(PROJECT_NUMBER).sum()) == 0)

validation = pd.DataFrame(validation_records)
failed_validation = validation.loc[~validation["Pass"]]

print("\nProject 24 Step 2B validation:")
display(validation)
print("\nClean REC comparison:")
display(comparison_summary)
print("\nClean-anchor validation:")
display(anchor_validation)
print("\nMapping-incomplete build audit:")
display(unmatched_mapping_audit)

if not failed_validation.empty:
    print("\nFailed Project 24 Step 2B checks:")
    display(failed_validation)
    print("\nNo Step 2B outputs, PASS status, or REC checkpoint were written.")
    raise RuntimeError("PROJECT 24 STEP 2B VALIDATION FAILED. DO NOT PROCEED TO THE NOISE PLAN.")

# --------------------------------------------------------------------------------------------------
# 13. FREEZE OUTPUTS
# --------------------------------------------------------------------------------------------------
PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)

inferred_storage = (
    inferred_execution_order[
        ["Build", "Test", "Job", "Verdict", "Duration", "StartedAtUTC", "InferredTestOrder"]
    ]
    .sort_values(["Test", "InferredTestOrder"], kind="mergesort")
    .reset_index(drop=True)
)

atomic_csv(UNMATCHED_MAPPING_AUDIT_PATH, unmatched_mapping_audit)
atomic_csv(TIMESTAMP_TIE_GROUPS_PATH, timestamp_tie_groups_frame)
atomic_csv(TEST_ORDER_SEARCH_AUDIT_PATH, search_audit)
print(f"\nWriting frozen {len(inferred_storage):,}-row execution-order parquet.")
atomic_parquet(INFERRED_EXECUTION_ORDER_PATH, inferred_storage)
atomic_csv(GLOBAL_AGE_ORDER_SEARCH_PATH, global_age_order_search)
atomic_csv(FROZEN_GLOBAL_BUILD_ORDER_PATH, frozen_global_build_order)
atomic_parquet(CLEAN_RECONSTRUCTED_PATH, clean_reconstructed[["Build", "Test"] + REC_FEATURES])
atomic_parquet(CLEAN_ANCHOR_OFFSETS_PATH, anchor_offsets[["Build", "Test"] + REC_FEATURES])
atomic_csv(CLEAN_COMPARISON_SUMMARY_PATH, comparison_summary)
atomic_csv(CLEAN_MISMATCH_EXAMPLES_PATH, mismatch_examples_frame)
atomic_csv(CLEAN_ANCHOR_VALIDATION_PATH, anchor_validation)
atomic_csv(STEP2B_VALIDATION_PATH, validation)

# --------------------------------------------------------------------------------------------------
# 14. READBACK VALIDATION
# --------------------------------------------------------------------------------------------------
meta = pq.ParquetFile(INFERRED_EXECUTION_ORDER_PATH)
execution_order_readback_rows = int(meta.metadata.num_rows)
required_order_columns = {"Build", "Test", "Job", "Verdict", "Duration", "StartedAtUTC", "InferredTestOrder"}
if execution_order_readback_rows != EXPECTED_RAW_ROWS or not required_order_columns.issubset(set(meta.schema.names)):
    raise RuntimeError("Frozen execution-order parquet metadata readback failed.")

rec_readback = pd.read_parquet(CLEAN_RECONSTRUCTED_PATH)
anchor_readback = pd.read_parquet(CLEAN_ANCHOR_OFFSETS_PATH)
if len(rec_readback) != EXPECTED_MODEL_ROWS or len(anchor_readback) != EXPECTED_MODEL_ROWS:
    raise RuntimeError("Clean REC/anchor parquet readback row count failed.")

readback = (
    rec_readback
    .merge(anchor_readback, on=["Build", "Test"], how="inner", validate="one_to_one", suffixes=("_reconstructed", "_offset"))
    .merge(dataset[["Build", "Test"] + REC_FEATURES], on=["Build", "Test"], how="inner", validate="one_to_one")
)
readback_mismatch_values = 0
for f in REC_FEATURES:
    reproduced = readback[f"{f}_reconstructed"].to_numpy(float) + readback[f"{f}_offset"].to_numpy(float)
    original = readback[f].to_numpy(float)
    readback_mismatch_values += int((~np.isclose(
        reproduced, original, rtol=ANCHOR_RTOL, atol=ANCHOR_ATOL, equal_nan=False
    )).sum())
if readback_mismatch_values != 0:
    raise RuntimeError("Frozen clean anchor failed readback reproduction.")

output_paths = [
    UNMATCHED_MAPPING_AUDIT_PATH, TIMESTAMP_TIE_GROUPS_PATH, TEST_ORDER_SEARCH_AUDIT_PATH,
    INFERRED_EXECUTION_ORDER_PATH, GLOBAL_AGE_ORDER_SEARCH_PATH, FROZEN_GLOBAL_BUILD_ORDER_PATH,
    CLEAN_RECONSTRUCTED_PATH, CLEAN_ANCHOR_OFFSETS_PATH, CLEAN_COMPARISON_SUMMARY_PATH,
    CLEAN_MISMATCH_EXAMPLES_PATH, CLEAN_ANCHOR_VALIDATION_PATH, STEP2B_VALIDATION_PATH,
]
output_manifest = [
    {"Path": str(p), "Bytes": int(p.stat().st_size), "SHA256": sha256_file(p)}
    for p in output_paths
]

# --------------------------------------------------------------------------------------------------
# 15. REPORT, CHECKPOINT, STATUS
# --------------------------------------------------------------------------------------------------
completed_at_utc = datetime.now(timezone.utc).isoformat()
report_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP2B_STATUS,
    "ImplementationVersion": IMPLEMENTATION_VERSION,
    "CompletedAtUTC": completed_at_utc,
    "SourceRootSHA256": current_source_root,
    "SelectionCheckpointSHA256": selection_sha256,
    "OfficialVerdictSemantics": {"Success": 0, "Exception": 1, "Assertion": 2},
    "TimestampTieGroups": timestamp_tie_groups_count,
    "TimestampTieBuilds": timestamp_tie_builds_count,
    "GlobalNaiveTieCombinationCount": int(2 ** timestamp_tie_groups_count),
    "GlobalAgeSolver": "EXACT_BINARY_CONSTRAINT_CSP_NO_NAIVE_ENUMERATION",
    "Tests": len(tests),
    "ModelReadyTests": model_ready_tests,
    "RawOnlyTests": raw_only_tests,
    "TestsTouchingTimestampTies": tests_touching_ties,
    "BaselineExactTieTests": baseline_exact_tie_tests,
    "RawOnlyTieTests": raw_only_tie_tests,
    "ExactDPSearchedTests": dp_searched_tests,
    "MaximumDPStatesObserved": maximum_dp_states_observed,
    "TestsWithNonZeroOrderMismatches": tests_nonzero_order,
    "TotalTieOrderMismatchValues": total_order_mismatches,
    "GlobalAgeBaselineMismatchRows": baseline_age_mismatch_rows,
    "GlobalAgeConstraintCount": len(age_constraints),
    "GlobalAgeMinimumMismatchRows": best_age,
    "GlobalBuildOrderSHA256": build_order_sha256,
    "RawExecutionOrderRows": execution_order_readback_rows,
    "GlobalBuildOrderRows": len(frozen_global_build_order),
    "RawRows": len(exe),
    "ModelRows": len(dataset),
    "CommitTokenRows": len(commit_audit),
    "ExactCommitMatches": exact_matches,
    "PrefixCommitMatches": prefix_matches,
    "UnmatchedCommitTokens": unmatched_tokens,
    "AmbiguousCommitTokens": ambiguous_tokens,
    "BuildsWithMappedEntities": len(builds_with_entities),
    "BuildsWithoutMappedEntities": len(builds_without_entities),
    "BuildEntityRows": len(build_entity),
    "MappingIncompleteBuilds": sorted(mapping_incomplete_builds),
    "MappingIncompletePartitions": sorted(mapping_incomplete_partitions),
    "MappingIncompleteRowsWithMappedEntities": mapping_rows_with_entities,
    "DirectMismatchValues": direct_mismatch_values,
    "VerdictDependentDirectMismatches": verdict_dependent_direct_mismatches,
    "VerdictIndependentDirectMismatches": verdict_independent_direct_mismatches,
    "FileHistoryDirectMismatches": file_history_direct_mismatches,
    "NonFileDirectMismatches": non_file_direct_mismatches,
    "AnchorAllowedDirectResidualFeatures": ANCHOR_ALLOWED_DIRECT_RESIDUAL_FEATURES,
    "AnchorAllowedDirectResidualValues": anchor_allowed_direct_mismatches,
    "AnchorDisallowedNonFileDirectMismatches": anchor_disallowed_non_file_direct_mismatches,
    "DirectResidualsConfinedToAcceptedBoundaries": direct_residuals_confined,
    "TieInferenceFeatures": TIE_INFERENCE_FEATURES,
    "FileHistoryMismatchesOutsideMappingIncompleteBuilds": file_mismatches_outside_mapping_incomplete,
    "UnmatchedMappingEffectConfined": unmatched_mapping_effect_confined,
    "RowsWithAnyNonZeroAnchorOffset": rows_with_any_nonzero_anchor_offset,
    "NonZeroAnchorOffsetValues": nonzero_anchor_offset_values,
    "FailedAnchorFeatures": failed_anchor_features,
    "AnchoredMismatchValues": anchored_mismatch_values,
    "ReadbackMismatchValues": readback_mismatch_values,
    "ZeroPercentCleanDatasetReproducedExactly": zero_percent_clean_reproduced_exactly,
    "OrderInferenceSeconds": order_inference_seconds,
    "GlobalAgeSearchSeconds": age_search_seconds,
    "RECReconstructionSeconds": reconstruction_seconds,
    "OutputManifest": output_manifest,
    "ValidationChecks": len(validation),
    "FailedValidationChecks": len(failed_validation),
    "CompletionRegistrySHA256": registry_sha256_before,
    "ActiveReservations": EXPECTED_ACTIVE_RESERVATIONS,
    "RuntimePriorityRule": EXPECTED_RUNTIME_PRIORITY_RULE,
    "RegistryModified": False,
    "Projects1To23Modified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "NoiseInjected": False,
    "ModelsTrained": False,
    "Project24ExperimentStarted": False,
}
atomic_json(STEP2B_REPORT_PATH, report_payload)

checkpoint_payload = {
    **report_payload,
    "CheckpointVersion": 1,
    "CheckpointType": "PROJECT_24_CLEAN_REC_RECONSTRUCTION",
    "RECReconstructionFrozen": True,
    "PerTestExecutionOrderFrozen": True,
    "GlobalBuildFirstAppearanceOrderFrozen": True,
    "CleanAnchorFrozen": True,
    "MappingBoundaryAuditFrozen": True,
    "EvaluationCohortImmutable": True,
    "ProceedToNoisePlanAllowed": True,
}
atomic_json(REC_CHECKPOINT_PATH, checkpoint_payload)
rec_checkpoint_sha256 = sha256_file(REC_CHECKPOINT_PATH)

status_payload = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP2B_STATUS,
    "ImplementationVersion": IMPLEMENTATION_VERSION,
    "CompletedAtUTC": completed_at_utc,
    "SourceRootSHA256": current_source_root,
    "TimestampTieGroups": timestamp_tie_groups_count,
    "TestsTouchingTimestampTies": tests_touching_ties,
    "TestsWithNonZeroOrderMismatches": tests_nonzero_order,
    "GlobalAgeMinimumMismatchRows": best_age,
    "AnchorAllowedDirectResidualValues": anchor_allowed_direct_mismatches,
    "AnchorDisallowedNonFileDirectMismatches": anchor_disallowed_non_file_direct_mismatches,
    "FileHistoryMismatchesOutsideMappingIncompleteBuilds": file_mismatches_outside_mapping_incomplete,
    "UnmatchedMappingEffectConfined": unmatched_mapping_effect_confined,
    "FailedAnchorFeatures": failed_anchor_features,
    "AnchoredMismatchValues": anchored_mismatch_values,
    "ZeroPercentCleanDatasetReproducedExactly": zero_percent_clean_reproduced_exactly,
    "Checkpoint": str(REC_CHECKPOINT_PATH),
    "CheckpointSHA256": rec_checkpoint_sha256,
    "RegistryModified": False,
    "PriorProjectConditionOutputsAccessed": False,
    "Project24ExperimentStarted": False,
    "NextRequiredStep": "PROJECT 24 CELL 6 / STEP 3A — DETERMINISTIC NOISE PLAN AND COHORT FREEZE",
}
atomic_json(STEP2B_STATUS_PATH, status_payload)

# --------------------------------------------------------------------------------------------------
# 16. FINAL IMMUTABILITY / READBACK
# --------------------------------------------------------------------------------------------------
registry_sha256_after = sha256_file(REGISTRY_PATH)
if registry_sha256_after != registry_sha256_before:
    raise RuntimeError("Completion registry changed during Project 24 Step 2B.")
if sha256_file(SELECTION_CHECKPOINT_PATH) != EXPECTED_SELECTION_SHA256:
    raise RuntimeError("Selection checkpoint changed during Project 24 Step 2B.")
for path_string, expected_sha in step2a_hashes_before.items():
    if sha256_file(Path(path_string)) != expected_sha:
        raise RuntimeError(f"Step 2A input changed during Step 2B: {path_string}")

final_records = []
for r in frozen_manifest.itertuples(index=False):
    p = SOURCE_DIR / str(r.RelativePath)
    final_records.append({
        "RelativePath": str(r.RelativePath),
        "SizeBytes": int(p.stat().st_size),
        "SHA256": sha256_file(p),
    })
if source_root_hash(pd.DataFrame(final_records)) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError("Frozen Project 24 source changed during Step 2B.")

for item in output_manifest:
    p = Path(item["Path"])
    if not p.is_file() or p.stat().st_size != item["Bytes"] or sha256_file(p) != item["SHA256"]:
        raise RuntimeError(f"Project 24 Step 2B output readback failed: {p}")

if any(load_json(p).get("Status") != STEP2B_STATUS for p in [REC_CHECKPOINT_PATH, STEP2B_STATUS_PATH, STEP2B_REPORT_PATH]):
    raise RuntimeError("Project 24 Step 2B checkpoint/status/report readback failed.")
if not load_json(REC_CHECKPOINT_PATH).get("ZeroPercentCleanDatasetReproducedExactly", False):
    raise RuntimeError("REC checkpoint does not freeze exact 0% clean-data reproduction.")

# --------------------------------------------------------------------------------------------------
# 17. FINAL RESULT
# --------------------------------------------------------------------------------------------------
print("\n" + "=" * 140)
print("=== PROJECT 24 CELL 5 / STEP 2B RESULT ===")
print("=" * 140)
print("Project:", PROJECT_NAME)
print("Project slug:", PROJECT_SLUG)
print("Implementation:", IMPLEMENTATION_VERSION)
print("Source root SHA-256:", current_source_root)
print("Official verdict semantics: success / exception / assertion = 0 / 1 / 2")

print("\nDeterministic execution-order freeze:")
print("Timestamp tie groups / builds:", timestamp_tie_groups_count, "/", timestamp_tie_builds_count)
print("Naive global combinations avoided:", int(2 ** timestamp_tie_groups_count))
print("Raw execution-order rows:", execution_order_readback_rows)
print("Global build-order rows:", len(frozen_global_build_order))
print("Tests / model-ready / raw-only:", len(tests), "/", model_ready_tests, "/", raw_only_tests)
print("Tests touching timestamp ties:", tests_touching_ties)
print("Tie tests exact under frozen baseline:", baseline_exact_tie_tests)
print("Raw-only tie tests using frozen baseline:", raw_only_tie_tests)
print("Tie tests requiring exact DP:", dp_searched_tests)
print("Maximum exact-DP states observed:", maximum_dp_states_observed)
print("Tests with non-zero order mismatches:", tests_nonzero_order)
print("Global REC_Age baseline mismatch rows:", baseline_age_mismatch_rows)
print("Global REC_Age exact constraints:", len(age_constraints))
print("Global REC_Age mismatch rows after exact solve:", best_age)

print("\nClean REC reconstruction:")
print("Raw history rows:", len(inferred_storage))
print("Model rows requested / reconstructed:", len(dataset), "/", len(clean_reconstructed))
print("Direct mismatch values:", direct_mismatch_values)
print("Non-file direct mismatch values:", non_file_direct_mismatches)
print("Anchor-allowed timing direct residual values:", anchor_allowed_direct_mismatches)
print("Anchor-disallowed non-file direct mismatch values:", anchor_disallowed_non_file_direct_mismatches)
print("File-history direct mismatch values:", file_history_direct_mismatches)
print("File-history mismatches outside mapping-incomplete builds:", file_mismatches_outside_mapping_incomplete)
print("Rows with any non-zero anchor offset:", rows_with_any_nonzero_anchor_offset)
print("Non-zero anchor-offset values:", nonzero_anchor_offset_values)
print("Failed anchor features:", failed_anchor_features)
print("Anchored mismatch values:", anchored_mismatch_values)
print("Readback mismatch values:", readback_mismatch_values)
print("0% clean dataset reproduced exactly:", zero_percent_clean_reproduced_exactly)

print("\nMapping audit:")
print("Commit-token rows:", len(commit_audit))
print("Exact / prefix / unmatched / ambiguous:", exact_matches, "/", prefix_matches, "/", unmatched_tokens, "/", ambiguous_tokens)
print("Builds with / without mapped entities:", len(builds_with_entities), "/", len(builds_without_entities))
print("Mapping-incomplete builds:", len(mapping_incomplete_builds))
print("Mapping-incomplete rows retaining mapped entities:", mapping_rows_with_entities)
print("Unmatched mapping effect confined:", unmatched_mapping_effect_confined)

print("\nRuntime seconds — order / global-age / reconstruction:",
      round(order_inference_seconds, 2), "/", round(age_search_seconds, 2), "/", round(reconstruction_seconds, 2))

print("\nIsolation:")
print("Completion registry unchanged:", registry_sha256_after == registry_sha256_before)
print("Projects 1–23 modified:", 0)
print("Prior project condition outputs accessed:", False)
print("Prior project condition outputs modified:", False)
print("Noise injected:", False)
print("Models trained:", False)
print("Project 24 experiment started:", False)

print("\nValidation:")
print("Checks:", len(validation))
print("Failed checks:", len(failed_validation))

print("\nREC reconstruction checkpoint:", REC_CHECKPOINT_PATH)
print("Checkpoint SHA-256:", rec_checkpoint_sha256)
print("\nNext required step: PROJECT 24 CELL 6 / STEP 3A — DETERMINISTIC NOISE PLAN AND COHORT FREEZE")
print("STATUS:", STEP2B_STATUS)
print("=" * 140)


=== PROJECT 24 CELL 5 / STEP 2B: SCALABLE EXACT-TIE CLEAN REC RECONSTRUCTION ===

Loading Project 24 model-ready REC source (224,550 rows expected).
Loading Project 24 raw execution history (5,635,027 rows expected).

Frozen timestamp-tie groups:


,TieGroup,StartedAtUTC,BuildCount,BuildIDsJSON,PermutationCount
0,1,2015-03-19T13:17:52+00:00,2,"[55020902, 55020899]",2
1,2,2015-04-03T13:28:22+00:00,2,"[57034855, 57034851]",2
2,3,2015-04-07T07:56:36+00:00,2,"[57446686, 57446681]",2
3,4,2015-04-10T13:53:24+00:00,2,"[57952214, 57952205]",2
4,5,2015-04-14T12:58:28+00:00,2,"[58433990, 58431610]",2
5,6,2015-04-15T12:33:11+00:00,2,"[58589372, 58589350]",2
6,7,2015-05-18T09:05:53+00:00,2,"[62986694, 62986689]",2
7,8,2015-05-20T07:22:15+00:00,2,"[63290680, 63290678]",2
8,9,2015-05-26T07:56:51+00:00,2,"[64053271, 64053269]",2
9,10,2015-05-26T13:18:03+00:00,2,"[64090452, 64090450]",2


Per-test order inference progress: 250 / 2641 | elapsed seconds: 3.33
Per-test order inference progress: 500 / 2641 | elapsed seconds: 7.96
Per-test order inference progress: 750 / 2641 | elapsed seconds: 11.99
Per-test order inference progress: 1000 / 2641 | elapsed seconds: 15.55
Per-test order inference progress: 1250 / 2641 | elapsed seconds: 19.96
Per-test order inference progress: 1500 / 2641 | elapsed seconds: 24.18
Per-test order inference progress: 1750 / 2641 | elapsed seconds: 27.29
Per-test order inference progress: 2000 / 2641 | elapsed seconds: 30.13
Per-test order inference progress: 2250 / 2641 | elapsed seconds: 31.5
Per-test order inference progress: 2500 / 2641 | elapsed seconds: 32.05
Per-test order inference progress: 2641 / 2641 | elapsed seconds: 32.34

Per-test tie inference summary:


,Metric,Value
0,Tests,2641.000000
1,Model-ready tests,2041.000000
2,Raw-only tests,600.000000
3,Tests touching timestamp ties,2020.000000
4,Tie tests exact under frozen baseline,1904.000000
5,Raw-only tie tests using frozen baseline,116.000000
6,Tie tests requiring exact DP,0.000000
7,Tests with non-zero minimum mismatch,0.000000
8,Total minimum mismatch values,0.000000
9,Maximum exact-DP states observed,1.000000



Global REC_Age exact constraint result:


...


Clean-anchor validation:


,Feature,NonZeroOffsetValues,AnchoredMismatchValues,Pass
0,REC_Age,0,0,True
1,REC_LastFailureAge,0,0,True
2,REC_LastTransitionAge,0,0,True
3,REC_RecentAvgExeTime,29344,0,True
4,REC_RecentMaxExeTime,0,0,True
5,REC_RecentFailRate,167,0,True
6,REC_RecentAssertRate,64,0,True
7,REC_RecentExcRate,103,0,True
8,REC_RecentTransitionRate,326,0,True
9,REC_TotalAvgExeTime,30423,0,True



Mapping-incomplete build audit:


,Build,BuildOrder,Split,CommitTokenRows,ExactMatches,PrefixMatches,UnmatchedTokens,AmbiguousTokens,MatchedEntityRows,Reasons,ModelRows,REC_MaxTestFileFailRate_MismatchValues,REC_MaxTestFileTransitionRate_MismatchValues,FileHistoryMismatchValues
0,54854778,15,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...,288,0,0,0
1,54857293,16,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...,288,0,0,0
2,55605572,124,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...,917,0,0,0
3,55608003,125,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...,917,0,0,0
4,55761727,139,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...,918,0,0,0
5,55773465,140,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...,918,0,0,0
6,55776490,141,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...,918,0,0,0
7,55785851,143,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...,918,0,0,0
8,55789407,145,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...,918,0,0,0
9,55825430,153,TRAIN,1,0,0,1,0,0,UNMATCHED_COMMIT_TOKEN;NO_MATCHED_COMMIT;NO_MA...,918,0,0,0



Writing frozen 5,635,027-row execution-order parquet.

=== PROJECT 24 CELL 5 / STEP 2B RESULT ===
Project: SonarSource@sonarqube
Project slug: SonarSource__sonarqube
Implementation: PROJECT_24_V1_SCALABLE_EXACT_TIE_DP_BINARY_AGE_CONSTRAINT_ANCHOR_AWARE_MAPPING_BOUNDARY_AUDIT
Source root SHA-256: 2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70
Official verdict semantics: success / exception / assertion = 0 / 1 / 2

Deterministic execution-order freeze:
Timestamp tie groups / builds: 23 / 46
Naive global combinations avoided: 8388608
Raw execution-order rows: 5635027
Global build-order rows: 4286
Tests / model-ready / raw-only: 2641 / 2041 / 600
Tests touching timestamp ties: 2020
Tie tests exact under frozen baseline: 1904
Raw-only tie tests using frozen baseline: 116
Tie tests requiring exact DP: 0
Maximum exact-DP states observed: 1
Tests with non-zero order mismatches: 0
Global REC_Age baseline mismatch rows: 0
Global REC_Age exact constraints: 10
Global REC_Age mis

In [6]:
# ==================================================================================================
# PROJECT 24 — CELL 6 / STEP 3A
# DETERMINISTIC NOISE PLAN, FIXED COHORTS, NESTED MASKS, AND RNG FREEZE
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_24.ipynb.
#
# PURPOSE:
# - verify the frozen Project 24 selection, source, REC reconstruction, clean anchor, and registry;
# - freeze raw and model-ready training/evaluation cohorts;
# - freeze the Project 24 failure-subtype distribution;
# - generate deterministic project/seed random streams used by label-noise injection;
# - prove masks are nested across all 9 noise levels for all 30 repetition seeds;
# - freeze all 270 condition coordinates and expected noisy-label hashes;
# - keep evaluation data clean and immutable;
# - perform no model fitting, no condition execution, and no registry write.
#
# PROJECT-24 SCALE NOTE:
# - raw training rows = 4,040,801;
# - RNG manifest rows = 4,040,801 × 30 = 121,224,030;
# - the RNG manifest is STREAMED as exactly 30 Parquet row groups, one seed at a time.
#   The full 121M-row RNG table is never materialized in RAM.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import gc
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 140)
print("=== PROJECT 24 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===")
print("=" * 140)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

SOURCE_DIR = Path(
    "/content/datasets/datasets/SonarSource@sonarqube"
)

EXPECTED_SELECTION_STATUS = (
    "PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_24_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

STEP3A_STATUS = (
    "PASS_PROJECT_24_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_SELECTION_SHA256 = (
    "d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803"
)

EXPECTED_REC_IMPLEMENTATION = (
    "PROJECT_24_V1_SCALABLE_EXACT_TIE_DP_BINARY_AGE_CONSTRAINT_"
    "ANCHOR_AWARE_MAPPING_BOUNDARY_AUDIT"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)

EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 23

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 374_503_252

EXPECTED_BUILDS = 4_286
EXPECTED_TRAIN_BUILDS = 3_214
EXPECTED_EVAL_BUILDS = 1_072

EXPECTED_RAW_ROWS = 5_635_027
EXPECTED_RAW_TRAIN_ROWS = 4_040_801
EXPECTED_RAW_EVAL_ROWS = 1_594_226
EXPECTED_RAW_TRAIN_FAILURES = 1_778
EXPECTED_RAW_EVAL_FAILURES = 20

EXPECTED_MODEL_ROWS = 224_550
EXPECTED_MODEL_TRAIN_ROWS = 205_696
EXPECTED_MODEL_EVAL_ROWS = 18_854
EXPECTED_MODEL_TRAIN_FAILURES = 1_777
EXPECTED_MODEL_EVAL_FAILURES = 20
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 17

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(
        1,
        31,
    )
)

EXPECTED_CONDITIONS = (
    len(
        NOISE_LEVELS
    )
    * len(
        REPETITION_SEEDS
    )
)

EXPECTED_RNG_ROWS = (
    EXPECTED_RAW_TRAIN_ROWS
    * len(
        REPETITION_SEEDS
    )
)

EXPECTED_RNG_ROW_GROUPS = len(
    REPETITION_SEEDS
)

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_24_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_24_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_24_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_24_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_rec_reconstruction_checkpoint.json"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

FAILURE_SUBTYPE_PROFILE_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_failure_subtype_profile.csv"
)

SEED_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_seed_manifest.csv"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NESTED_MASK_AUDIT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_nested_mask_audit.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_validation.csv"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_noise_plan_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(
        path
    )

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_array(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
        order="C",
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write(
            "\n"
        )

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.SizeBytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(
            column
        ).strip().lower()
        == str(
            expected
        ).strip().lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[
        0
    ]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[
            :4
        ],
        byteorder="little",
        signed=False,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def checkpoint_output_sha256(
    checkpoint,
    path,
):
    target = str(
        Path(
            path
        )
    )

    matches = [
        item
        for item in checkpoint.get(
            "OutputManifest",
            [],
        )
        if str(
            item.get(
                "Path",
                "",
            )
        )
        == target
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            "REC checkpoint does not contain exactly one manifest "
            f"entry for {target}."
        )

    return str(
        matches[
            0
        ][
            "SHA256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2B_STATUS_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    SOURCE_DIR
    / "dataset.csv",
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not Path(
        path
    ).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 24 Step 3A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)


if (
    selection_sha256
    != EXPECTED_SELECTION_SHA256
):
    raise RuntimeError(
        "Project 24 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if (
    rec_checkpoint_sha256
    != EXPECTED_REC_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 24 REC checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_REC_CHECKPOINT_SHA256}\n"
        f"Actual:   {rec_checkpoint_sha256}"
    )


selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2b_status = load_json(
    STEP2B_STATUS_PATH
)

step2b_report = load_json(
    STEP2B_REPORT_PATH
)

rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)


if (
    selection_checkpoint.get(
        "Status"
    )
    != EXPECTED_SELECTION_STATUS
    or step1b_status.get(
        "Status"
    )
    != EXPECTED_SELECTION_STATUS
):
    raise RuntimeError(
        "Project 24 selection is not frozen successfully."
    )


if (
    step2b_status.get(
        "Status"
    )
    != EXPECTED_STEP2B_STATUS
    or step2b_report.get(
        "Status"
    )
    != EXPECTED_STEP2B_STATUS
    or rec_checkpoint.get(
        "Status"
    )
    != EXPECTED_STEP2B_STATUS
):
    raise RuntimeError(
        "Project 24 Step 2B is not frozen successfully."
    )


if not bool(
    rec_checkpoint.get(
        "ZeroPercentCleanDatasetReproducedExactly",
        False,
    )
):
    raise RuntimeError(
        "Project 24 REC checkpoint does not confirm exact 0% clean reproduction."
    )


expected_rec_freeze_flags = {
    "ImplementationVersion":
        EXPECTED_REC_IMPLEMENTATION,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_24_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "MappingBoundaryAuditFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


for flag_name, expected_value in expected_rec_freeze_flags.items():
    actual_value = rec_checkpoint.get(
        flag_name
    )

    if (
        actual_value
        != expected_value
    ):
        raise RuntimeError(
            "Project 24 REC checkpoint contract differs.\n"
            f"Field:    {flag_name}\n"
            f"Expected: {expected_value!r}\n"
            f"Actual:   {actual_value!r}"
        )


if (
    selection_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "Frozen Project 24 identity differs."
    )


if (
    selection_checkpoint.get(
        "ActiveReservations"
    )
    != EXPECTED_ACTIVE_RESERVATIONS
):
    raise RuntimeError(
        "Frozen Project 24 active-reservation state differs."
    )


if (
    selection_checkpoint.get(
        "RuntimePriorityRule"
    )
    != EXPECTED_RUNTIME_PRIORITY_RULE
):
    raise RuntimeError(
        "Frozen Project 24 runtime-priority rule differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. REGISTRY AND SOURCE IMMUTABILITY
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if (
    registry_sha256_before
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS
            + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry must contain exactly Projects 1–23."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )


for (
    required_number,
    required_project,
) in REQUIRED_REGISTERED_IDENTITIES.items():
    matching = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching
        )
        != 1
        or str(
            matching.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A frozen predecessor identity differs.\n"
            f"Project {required_number}; expected={required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 24 is unexpectedly already registered."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if (
    len(
        current_source_manifest
    )
    != EXPECTED_SOURCE_FILES
    or current_source_bytes
    != EXPECTED_SOURCE_BYTES
    or current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 24 source differs."
    )


# Verify all four core Step 2B freeze artifacts against the REC checkpoint manifest.
core_rec_paths = [
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
]


for core_path in core_rec_paths:
    expected_sha = checkpoint_output_sha256(
        rec_checkpoint,
        core_path,
    )

    actual_sha = sha256_file(
        core_path
    )

    if (
        actual_sha
        != expected_sha
    ):
        raise RuntimeError(
            "A frozen Project 24 Step 2B artifact changed.\n"
            f"Path:     {core_path}\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}"
        )


# --------------------------------------------------------------------------------------------------
# 6. LOAD CHRONOLOGY, MODEL DATA, AND FROZEN RAW-HISTORY ORDER
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


required_chronology_columns = {
    "BuildOrder",
    "Build",
    "StartedAtUTC",
    "TimestampTieGroupSize",
    "InTimestampTie",
    "Split",
}


if not required_chronology_columns.issubset(
    chronology.columns
):
    raise RuntimeError(
        "Frozen Project 24 chronology is missing required columns."
    )


chronology[
    "Build"
] = parse_int(
    chronology[
        "Build"
    ],
    "chronology.Build",
)


chronology[
    "BuildOrder"
] = parse_int(
    chronology[
        "BuildOrder"
    ],
    "chronology.BuildOrder",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Split"
        ].eq(
            "TRAIN"
        ),
        "Build",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Split"
        ].eq(
            "EVAL"
        ),
        "Build",
    ].astype(
        int
    )
)


if (
    len(
        chronology
    )
    != EXPECTED_BUILDS
    or len(
        training_builds
    )
    != EXPECTED_TRAIN_BUILDS
    or len(
        evaluation_builds
    )
    != EXPECTED_EVAL_BUILDS
    or training_builds
    & evaluation_builds
):
    raise RuntimeError(
        "Frozen Project 24 chronology dimensions differ."
    )


dataset_header = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    nrows=0,
).columns.tolist()


dataset_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

dataset_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

dataset_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if (
    missing_rec_features
    or len(
        dataset_header
    )
    != EXPECTED_DATASET_COLUMNS
    or len(
        predictor_columns
    )
    != EXPECTED_PREDICTORS
    or len(
        REC_FEATURES
    )
    != EXPECTED_REC_FEATURES
):
    raise RuntimeError(
        "Project 24 predictor/REC schema differs."
    )


print(
    f"\nLoading full Project 24 model-ready dataset ({EXPECTED_MODEL_ROWS:,} rows)."
)


dataset = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    low_memory=False,
).rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "Verdict",
    }
)


dataset[
    "Build"
] = parse_int(
    dataset[
        "Build"
    ],
    "dataset.Build",
)


dataset[
    "Test"
] = parse_int(
    dataset[
        "Test"
    ],
    "dataset.Test",
)


dataset[
    "Verdict"
] = parse_int(
    dataset[
        "Verdict"
    ],
    "dataset.Verdict",
)


if (
    len(
        dataset
    )
    != EXPECTED_MODEL_ROWS
    or dataset.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).any()
):
    raise RuntimeError(
        "Project 24 model-ready cohort dimensions/keys differ."
    )


print(
    f"Loading frozen Project 24 raw execution order ({EXPECTED_RAW_ROWS:,} rows)."
)


inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)


required_inferred_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


if not required_inferred_columns.issubset(
    inferred_execution_order.columns
):
    raise RuntimeError(
        "Frozen inferred execution-order parquet is missing required columns."
    )


for column in [
    "Build",
    "Test",
    "Verdict",
    "InferredTestOrder",
]:
    inferred_execution_order[
        column
    ] = parse_int(
        inferred_execution_order[
            column
        ],
        f"inferred_execution_order.{column}",
    )


inferred_execution_order[
    "Job"
] = pd.to_numeric(
    inferred_execution_order[
        "Job"
    ],
    errors="coerce",
)


inferred_execution_order[
    "Duration"
] = pd.to_numeric(
    inferred_execution_order[
        "Duration"
    ],
    errors="coerce",
)


if (
    inferred_execution_order[
        "Job"
    ].isna().any()
    or inferred_execution_order[
        "Duration"
    ].isna().any()
    or not np.isfinite(
        inferred_execution_order[
            "Duration"
        ].to_numpy(
            dtype=float
        )
    ).all()
    or inferred_execution_order[
        "Duration"
    ].lt(
        0
    ).any()
):
    raise RuntimeError(
        "Frozen raw execution order has invalid job/duration values."
    )


raw_duplicate_build_test_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


raw_duplicate_test_order_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Test",
            "InferredTestOrder",
        ],
        keep=False,
    ).sum()
)


if (
    len(
        inferred_execution_order
    )
    != EXPECTED_RAW_ROWS
    or raw_duplicate_build_test_rows
    != 0
    or raw_duplicate_test_order_rows
    != 0
):
    raise RuntimeError(
        "Frozen raw-history order contract differs."
    )


exe = (
    inferred_execution_order.sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
    .copy()
)


frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)


required_global_order_columns = {
    "GlobalBuildPosition",
    "Build",
}


if not required_global_order_columns.issubset(
    frozen_global_build_order.columns
):
    raise RuntimeError(
        "Frozen Project 24 global build-order file is missing required columns."
    )


for column in [
    "GlobalBuildPosition",
    "Build",
]:
    frozen_global_build_order[
        column
    ] = parse_int(
        frozen_global_build_order[
            column
        ],
        f"frozen_global_build_order.{column}",
    )


global_build_order_valid = bool(
    len(
        frozen_global_build_order
    )
    == EXPECTED_BUILDS
    and frozen_global_build_order[
        "Build"
    ].nunique()
    == EXPECTED_BUILDS
    and set(
        frozen_global_build_order[
            "Build"
        ].astype(
            int
        )
    )
    == (
        training_builds
        | evaluation_builds
    )
    and sorted(
        frozen_global_build_order[
            "GlobalBuildPosition"
        ].astype(
            int
        ).tolist()
    )
    == list(
        range(
            0,
            EXPECTED_BUILDS,
        )
    )
)


if not global_build_order_valid:
    raise RuntimeError(
        "Frozen Project 24 global build order is invalid."
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY CLEAN REC + ANCHOR EXACTLY REPRODUCES THE SOURCE DATASET
# --------------------------------------------------------------------------------------------------

clean_rec = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)


clean_anchor = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if (
    len(
        clean_rec
    )
    != EXPECTED_MODEL_ROWS
    or len(
        clean_anchor
    )
    != EXPECTED_MODEL_ROWS
):
    raise RuntimeError(
        "Frozen clean REC / anchor row counts differ."
    )


if not clean_rec[
    [
        "Build",
        "Test",
    ]
].equals(
    clean_anchor[
        [
            "Build",
            "Test",
        ]
    ]
):
    raise RuntimeError(
        "Frozen clean REC / anchor Build-Test keys differ."
    )


if not clean_rec[
    [
        "Build",
        "Test",
    ]
].equals(
    dataset[
        [
            "Build",
            "Test",
        ]
    ]
):
    raise RuntimeError(
        "Frozen clean REC Build-Test order differs from source dataset."
    )


clean_anchor_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced = (
        clean_rec[
            feature
        ].to_numpy(
            dtype=np.float64
        )
        + clean_anchor[
            feature
        ].to_numpy(
            dtype=np.float64
        )
    )

    original = dataset[
        feature
    ].to_numpy(
        dtype=np.float64
    )

    clean_anchor_mismatch_values += int(
        (
            ~np.isclose(
                reproduced,
                original,
                rtol=0.0,
                atol=1e-12,
                equal_nan=False,
            )
        ).sum()
    )


clean_anchor_reproduced_dataset = bool(
    clean_anchor_mismatch_values
    == 0
)


if not clean_anchor_reproduced_dataset:
    raise RuntimeError(
        "Frozen Project 24 clean REC + anchor does not reproduce the clean source dataset."
    )


# --------------------------------------------------------------------------------------------------
# 8. FREEZE RAW AND MODEL COHORTS + MODEL/RAW LINKS
# --------------------------------------------------------------------------------------------------

raw_training = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_training.insert(
    0,
    "RawTrainingRowOrder",
    np.arange(
        1,
        len(
            raw_training
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_evaluation = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_evaluation.insert(
    0,
    "RawEvaluationRowOrder",
    np.arange(
        1,
        len(
            raw_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


model_training = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_training.insert(
    0,
    "ModelTrainingRowOrder",
    np.arange(
        1,
        len(
            model_training
        )
        + 1,
        dtype=np.int64,
    ),
)


model_evaluation = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_evaluation.insert(
    0,
    "ModelEvaluationRowOrder",
    np.arange(
        1,
        len(
            model_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_training_failures = int(
    raw_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)


raw_evaluation_failures = int(
    raw_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)


model_training_failures = int(
    model_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)


model_evaluation_failures = int(
    model_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "Verdict"
        ].ne(
            0
        ),
        "Build",
    ].nunique()
)


raw_training_link_source = raw_training[
    [
        "RawTrainingRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_training_link = (
    model_training[
        [
            "ModelTrainingRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_training_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelTrainingRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


raw_evaluation_link_source = raw_evaluation[
    [
        "RawEvaluationRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_evaluation_link = (
    model_evaluation[
        [
            "ModelEvaluationRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_evaluation_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelEvaluationRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_training_links = int(
    model_training_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


missing_model_evaluation_links = int(
    model_evaluation_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


model_training_verdict_mismatches = int(
    model_training_link[
        "ModelVerdict"
    ].ne(
        model_training_link[
            "RawVerdict"
        ]
    ).sum()
)


model_evaluation_verdict_mismatches = int(
    model_evaluation_link[
        "ModelVerdict"
    ].ne(
        model_evaluation_link[
            "RawVerdict"
        ]
    ).sum()
)


if (
    len(
        raw_training
    )
    != EXPECTED_RAW_TRAIN_ROWS
    or len(
        raw_evaluation
    )
    != EXPECTED_RAW_EVAL_ROWS
    or raw_training_failures
    != EXPECTED_RAW_TRAIN_FAILURES
    or raw_evaluation_failures
    != EXPECTED_RAW_EVAL_FAILURES
    or len(
        model_training
    )
    != EXPECTED_MODEL_TRAIN_ROWS
    or len(
        model_evaluation
    )
    != EXPECTED_MODEL_EVAL_ROWS
    or model_training_failures
    != EXPECTED_MODEL_TRAIN_FAILURES
    or model_evaluation_failures
    != EXPECTED_MODEL_EVAL_FAILURES
    or model_failing_evaluation_builds
    != EXPECTED_MODEL_FAILING_EVAL_BUILDS
    or missing_model_training_links
    != 0
    or missing_model_evaluation_links
    != 0
    or model_training_verdict_mismatches
    != 0
    or model_evaluation_verdict_mismatches
    != 0
):
    raise RuntimeError(
        "Project 24 fixed cohort/link contract differs."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE CLEAN FAILURE-SUBTYPE DISTRIBUTION
# --------------------------------------------------------------------------------------------------

failure_subtype_counts = (
    raw_training.loc[
        raw_training[
            "Verdict"
        ].ne(
            0
        ),
        "Verdict",
    ]
    .value_counts()
    .sort_index()
)


failure_subtypes = failure_subtype_counts.index.to_numpy(
    dtype=np.int16
)


failure_subtype_probabilities = (
    failure_subtype_counts.to_numpy(
        dtype=float
    )
    / failure_subtype_counts.sum()
)


failure_subtype_profile = pd.DataFrame({
    "FailureSubtype":
        failure_subtypes.astype(
            int
        ),

    "CleanTrainingRows":
        failure_subtype_counts.to_numpy(
            dtype=int
        ),

    "Probability":
        failure_subtype_probabilities,
})


failure_subtype_values_list = failure_subtypes.astype(
    int
).tolist()


failure_subtype_values_valid = bool(
    len(
        failure_subtype_values_list
    )
    > 0
    and set(
        failure_subtype_values_list
    ).issubset({
        1,
        2,
    })
)


failure_subtype_profile_sum_valid = bool(
    int(
        failure_subtype_profile[
            "CleanTrainingRows"
        ].sum()
    )
    == raw_training_failures
    and np.isclose(
        failure_subtype_profile[
            "Probability"
        ].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )
)


if not failure_subtype_values_valid:
    raise RuntimeError(
        "Project 24 clean training failures must use a non-empty subset "
        "of the frozen exception/assertion codes [1, 2]."
    )


if not failure_subtype_profile_sum_valid:
    raise RuntimeError(
        "Project 24 failure-subtype profile does not reproduce "
        "the clean raw training failure count."
    )


# --------------------------------------------------------------------------------------------------
# 10. GENERATE 30 DETERMINISTIC RNG STREAMS + 270 CONDITION PLAN
# --------------------------------------------------------------------------------------------------

NOISE_PLAN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


rng_temporary_path = RNG_MANIFEST_PATH.with_name(
    f".{RNG_MANIFEST_PATH.name}.tmp_{os.getpid()}"
)


if rng_temporary_path.exists():
    rng_temporary_path.unlink()


rng_schema = pa.schema([
    pa.field(
        "RepetitionSeed",
        pa.int16(),
    ),

    pa.field(
        "RawTrainingRowOrder",
        pa.int32(),
    ),

    pa.field(
        "FlipUniform",
        pa.float64(),
    ),

    pa.field(
        "SampledFailureSubtype",
        pa.int16(),
    ),
])


rng_writer = pq.ParquetWriter(
    rng_temporary_path,
    schema=rng_schema,
    compression="zstd",
)


seed_records = []
condition_records = []
nested_mask_records = []


clean_raw_verdict = raw_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)


clean_model_verdict = model_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)


model_training_raw_indices = (
    model_training_link[
        "RawTrainingRowOrder"
    ].to_numpy(
        dtype=np.int64
    )
    - 1
)


if (
    model_training_raw_indices.min()
    < 0
    or model_training_raw_indices.max()
    >= len(
        raw_training
    )
):
    raise RuntimeError(
        "Model-to-raw training indices are out of bounds."
    )


raw_row_order_int32 = np.arange(
    1,
    len(
        raw_training
    )
    + 1,
    dtype=np.int32,
)


rng_generation_started = time.perf_counter()


try:
    condition_order = 0

    for seed_order, repetition_seed in enumerate(
        REPETITION_SEEDS,
        start=1,
    ):
        seed_started = time.perf_counter()

        flip_seed = deterministic_seed(
            repetition_seed,
            "flip_mask",
        )

        failure_subtype_seed = deterministic_seed(
            repetition_seed,
            "failure_subtype",
        )


        flip_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )


        sampled_failure_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        # Immediate deterministic regeneration proof.
        flip_uniform_reproduced = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )


        uniforms_reproduced = bool(
            np.array_equal(
                flip_uniform,
                flip_uniform_reproduced,
            )
        )


        del flip_uniform_reproduced


        failure_subtype_reproduced = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        failure_subtypes_reproduced = bool(
            np.array_equal(
                sampled_failure_subtype,
                failure_subtype_reproduced,
            )
        )


        del failure_subtype_reproduced


        if (
            not uniforms_reproduced
            or not failure_subtypes_reproduced
        ):
            raise RuntimeError(
                f"Deterministic RNG regeneration failed for seed {repetition_seed}."
            )


        seed_records.append({
            "SeedOrder":
                seed_order,

            "RepetitionSeed":
                repetition_seed,

            "FlipSeed":
                flip_seed,

            "FailureSubtypeSeed":
                failure_subtype_seed,

            "FlipUniformSHA256":
                sha256_array(
                    flip_uniform,
                    "<f8",
                ),

            "SampledFailureSubtypeSHA256":
                sha256_array(
                    sampled_failure_subtype,
                    "<i2",
                ),

            "UniformsReproduced":
                uniforms_reproduced,

            "FailureSubtypesReproduced":
                failure_subtypes_reproduced,
        })


        # Stream exactly one seed as one Parquet row group.
        seed_array = np.full(
            len(
                raw_training
            ),
            repetition_seed,
            dtype=np.int16,
        )


        rng_table = pa.Table.from_arrays(
            [
                pa.array(
                    seed_array,
                    type=pa.int16(),
                ),

                pa.array(
                    raw_row_order_int32,
                    type=pa.int32(),
                ),

                pa.array(
                    flip_uniform,
                    type=pa.float64(),
                ),

                pa.array(
                    sampled_failure_subtype,
                    type=pa.int16(),
                ),
            ],
            schema=rng_schema,
        )


        rng_writer.write_table(
            rng_table,
            row_group_size=len(
                raw_training
            ),
        )


        del rng_table
        del seed_array


        previous_mask = None
        previous_noise = None


        for noise_order, noise_percent in enumerate(
            NOISE_LEVELS,
            start=1,
        ):
            condition_order += 1

            condition_id = (
                f"noise_{noise_percent:02d}"
                f"__seed_{repetition_seed:02d}"
            )


            flip_mask = (
                flip_uniform
                < (
                    noise_percent
                    / 100.0
                )
            )


            noisy_raw_verdict = clean_raw_verdict.copy()


            pass_to_failure_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    == 0
                )
            )


            failure_to_pass_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    != 0
                )
            )


            noisy_raw_verdict[
                pass_to_failure_mask
            ] = sampled_failure_subtype[
                pass_to_failure_mask
            ]


            noisy_raw_verdict[
                failure_to_pass_mask
            ] = 0


            noisy_model_verdict = noisy_raw_verdict[
                model_training_raw_indices
            ]


            number_flipped = int(
                flip_mask.sum()
            )


            pass_to_failure = int(
                pass_to_failure_mask.sum()
            )


            failure_to_pass = int(
                failure_to_pass_mask.sum()
            )


            noisy_raw_failures = int(
                (
                    noisy_raw_verdict
                    != 0
                ).sum()
            )


            model_label_changes = int(
                (
                    noisy_model_verdict
                    != clean_model_verdict
                ).sum()
            )


            noisy_model_failures = int(
                (
                    noisy_model_verdict
                    != 0
                ).sum()
            )


            condition_records.append({
                "ConditionOrder":
                    condition_order,

                "ConditionID":
                    condition_id,

                "SeedOrder":
                    seed_order,

                "NoiseOrderWithinSeed":
                    noise_order,

                "NoisePercent":
                    noise_percent,

                "RepetitionSeed":
                    repetition_seed,

                "FlipSeed":
                    flip_seed,

                "FailureSubtypeSeed":
                    failure_subtype_seed,

                "RawTrainingRows":
                    len(
                        raw_training
                    ),

                "NumberFlipped":
                    number_flipped,

                "RealisedNoisePercent":
                    (
                        100.0
                        * number_flipped
                        / len(
                            raw_training
                        )
                    ),

                "PassToFailure":
                    pass_to_failure,

                "FailureToPass":
                    failure_to_pass,

                "CleanRawFailures":
                    raw_training_failures,

                "NoisyRawFailures":
                    noisy_raw_failures,

                "ModelTrainingRows":
                    len(
                        model_training
                    ),

                "ModelLabelChanges":
                    model_label_changes,

                "CleanModelFailures":
                    model_training_failures,

                "NoisyModelFailures":
                    noisy_model_failures,

                "FlipMaskSHA256":
                    sha256_array(
                        flip_mask.astype(
                            np.uint8
                        ),
                        "u1",
                    ),

                "NoisyRawVerdictSHA256":
                    sha256_array(
                        noisy_raw_verdict,
                        "<i2",
                    ),

                "NoisyModelVerdictSHA256":
                    sha256_array(
                        noisy_model_verdict,
                        "<i2",
                    ),
            })


            if previous_mask is not None:
                violations = int(
                    (
                        previous_mask
                        & (
                            ~flip_mask
                        )
                    ).sum()
                )


                nested_mask_records.append({
                    "RepetitionSeed":
                        repetition_seed,

                    "LowerNoisePercent":
                        previous_noise,

                    "HigherNoisePercent":
                        noise_percent,

                    "Violations":
                        violations,

                    "Pass":
                        violations == 0,
                })


            previous_mask = flip_mask
            previous_noise = noise_percent


            del noisy_raw_verdict
            del noisy_model_verdict
            del pass_to_failure_mask
            del failure_to_pass_mask

            if previous_mask is not flip_mask:
                del flip_mask


        del flip_uniform
        del sampled_failure_subtype
        del previous_mask

        gc.collect()

        print(
            "    RNG/condition-plan progress:",
            seed_order,
            "/",
            len(
                REPETITION_SEEDS
            ),
            "seeds | this seed seconds:",
            round(
                time.perf_counter()
                - seed_started,
                2,
            ),
            "| total seconds:",
            round(
                time.perf_counter()
                - rng_generation_started,
                2,
            ),
        )


finally:
    rng_writer.close()


os.replace(
    rng_temporary_path,
    RNG_MANIFEST_PATH,
)


rng_generation_seconds = float(
    time.perf_counter()
    - rng_generation_started
)


seed_manifest = pd.DataFrame(
    seed_records
)


condition_plan = pd.DataFrame(
    condition_records
)


nested_mask_audit = pd.DataFrame(
    nested_mask_records
)


# --------------------------------------------------------------------------------------------------
# 11. PLAN AUDITS
# --------------------------------------------------------------------------------------------------

nested_mask_violations = int(
    nested_mask_audit[
        "Violations"
    ].sum()
)


zero_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(
        0
    )
]


zero_noise_flip_violations = int(
    zero_noise_conditions[
        "NumberFlipped"
    ].ne(
        0
    ).sum()
)


zero_noise_raw_label_violations = int(
    zero_noise_conditions[
        "NoisyRawFailures"
    ].ne(
        raw_training_failures
    ).sum()
)


zero_noise_model_label_violations = int(
    zero_noise_conditions[
        "ModelLabelChanges"
    ].ne(
        0
    ).sum()
)


positive_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].gt(
        0
    )
]


positive_noise_without_raw_changes = int(
    positive_noise_conditions[
        "NumberFlipped"
    ].le(
        0
    ).sum()
)


positive_noise_without_model_changes = int(
    positive_noise_conditions[
        "ModelLabelChanges"
    ].le(
        0
    ).sum()
)


duplicate_condition_ids = int(
    condition_plan[
        "ConditionID"
    ].duplicated(
        keep=False
    ).sum()
)


duplicate_condition_coordinates = int(
    condition_plan.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


seed_streams_reproduced = bool(
    seed_manifest[
        [
            "UniformsReproduced",
            "FailureSubtypesReproduced",
        ]
    ].all().all()
)


condition_coordinates_exact = bool(
    set(
        zip(
            condition_plan[
                "NoisePercent"
            ].astype(
                int
            ),
            condition_plan[
                "RepetitionSeed"
            ].astype(
                int
            ),
        )
    )
    == {
        (
            noise_percent,
            repetition_seed,
        )
        for repetition_seed in REPETITION_SEEDS
        for noise_percent in NOISE_LEVELS
    }
)


# --------------------------------------------------------------------------------------------------
# 12. PRE-WRITE VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []


def C(
    check,
    expected,
    actual,
    passed,
):
    add_check(
        checks,
        check,
        expected,
        actual,
        passed,
    )


C(
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)


C(
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256
    == EXPECTED_REC_CHECKPOINT_SHA256,
)


C(
    "REC implementation",
    EXPECTED_REC_IMPLEMENTATION,
    rec_checkpoint.get(
        "ImplementationVersion"
    ),
    rec_checkpoint.get(
        "ImplementationVersion"
    )
    == EXPECTED_REC_IMPLEMENTATION,
)


C(
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


C(
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_source_manifest
    ),
    len(
        current_source_manifest
    )
    == EXPECTED_SOURCE_FILES,
)


C(
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)


C(
    "Clean anchor mismatch values",
    0,
    clean_anchor_mismatch_values,
    clean_anchor_mismatch_values
    == 0,
)


C(
    "Clean anchor reproduced source dataset",
    True,
    clean_anchor_reproduced_dataset,
    clean_anchor_reproduced_dataset,
)


C(
    "Raw history rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    )
    == EXPECTED_RAW_ROWS,
)


C(
    "Raw duplicate Build-Test rows",
    0,
    raw_duplicate_build_test_rows,
    raw_duplicate_build_test_rows
    == 0,
)


C(
    "Raw duplicate Test-order rows",
    0,
    raw_duplicate_test_order_rows,
    raw_duplicate_test_order_rows
    == 0,
)


C(
    "Global build order valid",
    True,
    global_build_order_valid,
    global_build_order_valid,
)


C(
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


C(
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


C(
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    raw_training_failures,
    raw_training_failures
    == EXPECTED_RAW_TRAIN_FAILURES,
)


C(
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    raw_evaluation_failures,
    raw_evaluation_failures
    == EXPECTED_RAW_EVAL_FAILURES,
)


C(
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


C(
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


C(
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


C(
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)


C(
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)


C(
    "Missing model training raw links",
    0,
    missing_model_training_links,
    missing_model_training_links
    == 0,
)


C(
    "Missing model evaluation raw links",
    0,
    missing_model_evaluation_links,
    missing_model_evaluation_links
    == 0,
)


C(
    "Model training/raw verdict mismatches",
    0,
    model_training_verdict_mismatches,
    model_training_verdict_mismatches
    == 0,
)


C(
    "Model evaluation/raw verdict mismatches",
    0,
    model_evaluation_verdict_mismatches,
    model_evaluation_verdict_mismatches
    == 0,
)


C(
    "Failure subtype values valid",
    True,
    failure_subtype_values_valid,
    failure_subtype_values_valid,
)


C(
    "Failure subtype profile valid",
    True,
    failure_subtype_profile_sum_valid,
    failure_subtype_profile_sum_valid,
)


C(
    "Seed rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest
    ),
    len(
        seed_manifest
    )
    == len(
        REPETITION_SEEDS
    ),
)


C(
    "Seed streams reproduced",
    True,
    seed_streams_reproduced,
    seed_streams_reproduced,
)


C(
    "Conditions",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    )
    == EXPECTED_CONDITIONS,
)


C(
    "Condition coordinates exact",
    True,
    condition_coordinates_exact,
    condition_coordinates_exact,
)


C(
    "Duplicate condition IDs",
    0,
    duplicate_condition_ids,
    duplicate_condition_ids
    == 0,
)


C(
    "Duplicate condition coordinates",
    0,
    duplicate_condition_coordinates,
    duplicate_condition_coordinates
    == 0,
)


C(
    "Nested-mask audit rows",
    (
        len(
            REPETITION_SEEDS
        )
        * (
            len(
                NOISE_LEVELS
            )
            - 1
        )
    ),
    len(
        nested_mask_audit
    ),
    len(
        nested_mask_audit
    )
    == (
        len(
            REPETITION_SEEDS
        )
        * (
            len(
                NOISE_LEVELS
            )
            - 1
        )
    ),
)


C(
    "Nested-mask violations",
    0,
    nested_mask_violations,
    nested_mask_violations
    == 0,
)


C(
    "Zero-noise conditions",
    len(
        REPETITION_SEEDS
    ),
    len(
        zero_noise_conditions
    ),
    len(
        zero_noise_conditions
    )
    == len(
        REPETITION_SEEDS
    ),
)


C(
    "Zero-noise flip violations",
    0,
    zero_noise_flip_violations,
    zero_noise_flip_violations
    == 0,
)


C(
    "Zero-noise raw-label violations",
    0,
    zero_noise_raw_label_violations,
    zero_noise_raw_label_violations
    == 0,
)


C(
    "Zero-noise model-label violations",
    0,
    zero_noise_model_label_violations,
    zero_noise_model_label_violations
    == 0,
)


C(
    "Positive-noise conditions without raw changes",
    0,
    positive_noise_without_raw_changes,
    positive_noise_without_raw_changes
    == 0,
)


C(
    "Positive-noise conditions without model changes",
    0,
    positive_noise_without_model_changes,
    positive_noise_without_model_changes
    == 0,
)


C(
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)


for (
    required_number,
    required_project,
) in REQUIRED_REGISTERED_IDENTITIES.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )


    C(
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )


C(
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    selection_checkpoint.get(
        "ActiveReservations"
    ),
    selection_checkpoint.get(
        "ActiveReservations"
    )
    == EXPECTED_ACTIVE_RESERVATIONS,
)


C(
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)


C(
    "Project 24 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


validation = pd.DataFrame(
    checks
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 24 Step 3A pre-write validation:"
)


display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 24 Step 3A checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 24 STEP 3A VALIDATION FAILED. "
        "No frozen cohort/checkpoint outputs were written."
    )


# --------------------------------------------------------------------------------------------------
# 13. WRITE FROZEN COHORTS AND PLAN OUTPUTS
# --------------------------------------------------------------------------------------------------

print(
    "\nWriting fixed Project 24 cohorts."
)


atomic_parquet(
    RAW_TRAINING_COHORT_PATH,
    raw_training,
)


atomic_parquet(
    RAW_EVALUATION_COHORT_PATH,
    raw_evaluation,
)


atomic_parquet(
    MODEL_TRAINING_COHORT_PATH,
    model_training,
)


atomic_parquet(
    MODEL_EVALUATION_COHORT_PATH,
    model_evaluation,
)


atomic_parquet(
    MODEL_RAW_TRAIN_LINK_PATH,
    model_training_link.drop(
        columns=[
            "_merge",
        ]
    ),
)


atomic_parquet(
    MODEL_RAW_EVAL_LINK_PATH,
    model_evaluation_link.drop(
        columns=[
            "_merge",
        ]
    ),
)


atomic_csv(
    FAILURE_SUBTYPE_PROFILE_PATH,
    failure_subtype_profile,
)


atomic_csv(
    SEED_MANIFEST_PATH,
    seed_manifest,
)


atomic_csv(
    CONDITION_PLAN_PATH,
    condition_plan,
)


atomic_csv(
    NESTED_MASK_AUDIT_PATH,
    nested_mask_audit,
)


# --------------------------------------------------------------------------------------------------
# 14. FREEZE EXPERIMENT PROTOCOL
# --------------------------------------------------------------------------------------------------

protocol_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProtocolVersion":
        1,

    "NoiseModel":
        "Symmetric binary pass/failure flips with clean empirical failure-subtype sampling",

    "CleanVerdictCodes": {
        "Pass":
            0,

        "ExceptionFailure":
            1,

        "AssertionFailure":
            2,
    },

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "DeterministicSeedDerivation":
        "uint32 little-endian first 4 bytes of SHA256(Project|Seed|StreamName)",

    "RNGStreamsPerSeed": [
        "flip_mask",
        "failure_subtype",
    ],

    "NestedMasks":
        True,

    "RNGManifestStorage":
        (
            "30 streamed Parquet row groups; exactly one repetition seed "
            "per row group; full 121,224,030-row manifest never materialized in RAM"
        ),

    "RNGManifestRows":
        EXPECTED_RNG_ROWS,

    "RNGManifestRowGroups":
        EXPECTED_RNG_ROW_GROUPS,

    "FailureSubtypeSource":
        "Empirical clean raw-training failure subtype distribution",

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "TrainingNoiseAppliedTo":
        "Frozen raw training labels only",

    "ModelTrainingLabels":
        "Inherited by exact frozen model-to-raw training Build-Test links",

    "EvaluationNoiseApplied":
        False,

    "EvaluationCohortImmutable":
        True,

    "CleanRECCheckpointSHA256":
        rec_checkpoint_sha256,

    "CleanRECImplementation":
        EXPECTED_REC_IMPLEMENTATION,

    "CleanAnchorReproducedDataset":
        clean_anchor_reproduced_dataset,

    "RECFeatures":
        REC_FEATURES,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "ModelsFittedInStep3A":
        False,

    "ExperimentConditionsExecutedInStep3A":
        False,

    "CompletionRegistryWrittenInStep3A":
        False,
}


atomic_json(
    PROTOCOL_PATH,
    protocol_payload,
)


# --------------------------------------------------------------------------------------------------
# 15. RNG MANIFEST READBACK + FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

print(
    "\nValidating streamed RNG-manifest metadata."
)


rng_parquet = pq.ParquetFile(
    RNG_MANIFEST_PATH
)


rng_readback_rows = int(
    rng_parquet.metadata.num_rows
)


rng_readback_row_groups = int(
    rng_parquet.metadata.num_row_groups
)


rng_row_group_row_counts = [
    int(
        rng_parquet.metadata.row_group(
            row_group_index
        ).num_rows
    )
    for row_group_index in range(
        rng_readback_row_groups
    )
]


rng_schema_names = set(
    rng_parquet.schema.names
)


rng_schema_valid = {
    "RepetitionSeed",
    "RawTrainingRowOrder",
    "FlipUniform",
    "SampledFailureSubtype",
}.issubset(
    rng_schema_names
)


rng_row_group_sizes_valid = bool(
    rng_readback_row_groups
    == EXPECTED_RNG_ROW_GROUPS
    and all(
        rows
        == EXPECTED_RAW_TRAIN_ROWS
        for rows in rng_row_group_row_counts
    )
)


# Read only the first and last row group to validate seed and row-order boundaries
# without reloading 121M rows.
boundary_readback_failures = 0


for row_group_index, expected_seed in [
    (
        0,
        REPETITION_SEEDS[
            0
        ],
    ),
    (
        rng_readback_row_groups
        - 1,
        REPETITION_SEEDS[
            -1
        ],
    ),
]:
    table = rng_parquet.read_row_group(
        row_group_index,
        columns=[
            "RepetitionSeed",
            "RawTrainingRowOrder",
        ],
    )


    frame = table.to_pandas()


    boundary_readback_failures += int(
        len(
            frame
        )
        != EXPECTED_RAW_TRAIN_ROWS
    )


    boundary_readback_failures += int(
        frame[
            "RepetitionSeed"
        ].nunique()
        != 1
    )


    boundary_readback_failures += int(
        int(
            frame[
                "RepetitionSeed"
            ].iloc[
                0
            ]
        )
        != int(
            expected_seed
        )
    )


    boundary_readback_failures += int(
        int(
            frame[
                "RawTrainingRowOrder"
            ].iloc[
                0
            ]
        )
        != 1
    )


    boundary_readback_failures += int(
        int(
            frame[
                "RawTrainingRowOrder"
            ].iloc[
                -1
            ]
        )
        != EXPECTED_RAW_TRAIN_ROWS
    )


    del frame
    del table
    gc.collect()


seed_manifest_readback = pd.read_csv(
    SEED_MANIFEST_PATH,
    low_memory=False,
)


condition_plan_readback = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


nested_mask_readback = pd.read_csv(
    NESTED_MASK_AUDIT_PATH,
    low_memory=False,
)


nested_mask_readback_violations = int(
    nested_mask_readback[
        "Violations"
    ].sum()
)


readback_checks = []


add_check(
    readback_checks,
    "RNG manifest rows",
    EXPECTED_RNG_ROWS,
    rng_readback_rows,
    rng_readback_rows
    == EXPECTED_RNG_ROWS,
)


add_check(
    readback_checks,
    "RNG manifest row groups",
    EXPECTED_RNG_ROW_GROUPS,
    rng_readback_row_groups,
    rng_readback_row_groups
    == EXPECTED_RNG_ROW_GROUPS,
)


add_check(
    readback_checks,
    "RNG row-group sizes",
    True,
    rng_row_group_sizes_valid,
    rng_row_group_sizes_valid,
)


add_check(
    readback_checks,
    "RNG schema valid",
    True,
    rng_schema_valid,
    rng_schema_valid,
)


add_check(
    readback_checks,
    "RNG boundary readback failures",
    0,
    boundary_readback_failures,
    boundary_readback_failures
    == 0,
)


add_check(
    readback_checks,
    "Condition-plan readback rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan_readback
    ),
    len(
        condition_plan_readback
    )
    == EXPECTED_CONDITIONS,
)


add_check(
    readback_checks,
    "Seed-manifest readback rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest_readback
    ),
    len(
        seed_manifest_readback
    )
    == len(
        REPETITION_SEEDS
    ),
)


add_check(
    readback_checks,
    "Nested-mask readback violations",
    0,
    nested_mask_readback_violations,
    nested_mask_readback_violations
    == 0,
)


readback_validation = pd.DataFrame(
    readback_checks
)


failed_readback = readback_validation.loc[
    ~readback_validation[
        "Pass"
    ]
]


print(
    "\nProject 24 Step 3A RNG/readback validation:"
)


display(
    readback_validation
)


if not failed_readback.empty:
    raise RuntimeError(
        "PROJECT 24 STEP 3A RNG/READBACK VALIDATION FAILED."
    )


# Combine pre-write and readback checks into the frozen validation table.
validation = pd.concat(
    [
        validation,
        readback_validation,
    ],
    ignore_index=True,
)


atomic_csv(
    STEP3A_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 16. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    FAILURE_SUBTYPE_PROFILE_PATH,
    SEED_MANIFEST_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NESTED_MASK_AUDIT_PATH,
    PROTOCOL_PATH,
    STEP3A_VALIDATION_PATH,
]


print(
    "\nHashing frozen Step 3A outputs for the output manifest."
)


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "RECCheckpointSHA256":
        rec_checkpoint_sha256,

    "RECImplementationVersion":
        EXPECTED_REC_IMPLEMENTATION,

    "SourceRootSHA256":
        current_source_root_sha256,

    "CleanAnchorReproducedDataset":
        clean_anchor_reproduced_dataset,

    "CleanAnchorMismatchValues":
        clean_anchor_mismatch_values,

    "FrozenInferredExecutionOrder":
        str(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenInferredExecutionOrderSHA256":
        sha256_file(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenGlobalBuildOrder":
        str(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "FrozenGlobalBuildOrderSHA256":
        sha256_file(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "RawHistoryOrdering":
        (
            "Project 24 Step 2B scalable exact-tie inferred order "
            "with frozen baseline used for all 2,020 tied tests"
        ),

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "RNGManifestRowGroups":
        rng_readback_row_groups,

    "RNGManifestStorageMode":
        "30 streamed Parquet row groups; one seed held in memory",

    "RNGGenerationSeconds":
        rng_generation_seconds,

    "NestedMaskAuditRows":
        len(
            nested_mask_audit
        ),

    "NestedMaskViolations":
        nested_mask_violations,

    "ZeroNoiseConditions":
        len(
            zero_noise_conditions
        ),

    "ZeroNoiseFlipViolations":
        zero_noise_flip_violations,

    "ZeroNoiseRawLabelViolations":
        zero_noise_raw_label_violations,

    "ZeroNoiseModelLabelViolations":
        zero_noise_model_label_violations,

    "PositiveNoiseConditionsWithoutRawChanges":
        positive_noise_without_raw_changes,

    "PositiveNoiseConditionsWithoutModelChanges":
        positive_noise_without_model_changes,

    "PredictorColumns":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC,

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "OutputManifest":
        output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        int(
            (
                ~validation[
                    "Pass"
                ]
            ).sum()
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RegistryModified":
        False,

    "Projects1To23Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ModelsTrained":
        False,

    "Project24ExperimentStarted":
        False,
}


atomic_json(
    STEP3A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_24_DETERMINISTIC_NOISE_PLAN_AND_COHORTS",

    "NoisePlanCheckpoint":
        True,

    "DoNotChangeCohorts":
        True,

    "DoNotChangeRandomStreams":
        True,

    "DoNotChangeConditionCoordinates":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToRuntimeContractAllowed":
        True,
}


atomic_json(
    NOISE_PLAN_CHECKPOINT_PATH,
    checkpoint_payload,
)


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "RNGManifestRowGroups":
        rng_readback_row_groups,

    "NestedMaskViolations":
        nested_mask_violations,

    "Checkpoint":
        str(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        noise_checkpoint_sha256,

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "PriorProjectConditionOutputsAccessed":
        False,

    "ModelsTrained":
        False,

    "Project24ExperimentStarted":
        False,

    "NextRequiredStep":
        "PROJECT 24 CELL 7 / STEP 4A — EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE",
}


atomic_json(
    STEP3A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL IMMUTABILITY AND MANIFEST READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if (
    registry_sha256_after
    != registry_sha256_before
):
    raise RuntimeError(
        "Completion registry changed during Project 24 Step 3A."
    )


if sha256_file(
    REC_CHECKPOINT_PATH
) != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 REC checkpoint changed during Step 3A."
    )


if sha256_file(
    SELECTION_CHECKPOINT_PATH
) != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 24 selection checkpoint changed during Step 3A."
    )


final_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


if (
    source_root_hash(
        final_source_manifest
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 24 source changed during Step 3A."
    )


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        raise RuntimeError(
            f"Project 24 Step 3A output readback failed: {path}"
        )


checkpoint_readback = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP3A_STATUS_PATH
)

report_readback = load_json(
    STEP3A_REPORT_PATH
)


for label, payload in [
    (
        "checkpoint",
        checkpoint_readback,
    ),
    (
        "status",
        status_readback,
    ),
    (
        "report",
        report_readback,
    ),
]:
    if (
        payload.get(
            "Status"
        )
        != STEP3A_STATUS
    ):
        raise RuntimeError(
            f"Project 24 Step 3A {label} readback is not PASS."
        )


# --------------------------------------------------------------------------------------------------
# 18. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nFailure-subtype profile:"
)


display(
    failure_subtype_profile
)


print(
    "\nSeed manifest:"
)


display(
    seed_manifest
)


print(
    "\nCondition-plan sample:"
)


display(
    pd.concat(
        [
            condition_plan.head(
                9
            ),
            condition_plan.tail(
                9
            ),
        ],
        ignore_index=True,
    )
)


print(
    "\nNested-mask audit summary:"
)


display(
    nested_mask_audit.groupby(
        [
            "LowerNoisePercent",
            "HigherNoisePercent",
        ],
        as_index=False,
    ).agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TotalViolations=(
            "Violations",
            "sum",
        ),

        AllPassed=(
            "Pass",
            "all",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 19. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 140
)

print(
    "=== PROJECT 24 CELL 6 / STEP 3A RESULT ==="
)

print(
    "=" * 140
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "REC checkpoint SHA-256:",
    rec_checkpoint_sha256,
)

print(
    "REC implementation:",
    EXPECTED_REC_IMPLEMENTATION,
)

print()

print(
    "Fixed cohorts:"
)

print(
    "Raw training rows:",
    len(
        raw_training
    ),
)

print(
    "Raw evaluation rows:",
    len(
        raw_evaluation
    ),
)

print(
    "Raw training failures:",
    raw_training_failures,
)

print(
    "Raw evaluation failures:",
    raw_evaluation_failures,
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Model training failures:",
    model_training_failures,
)

print(
    "Model evaluation failures:",
    model_evaluation_failures,
)

print(
    "Model failing evaluation builds:",
    model_failing_evaluation_builds,
)

print()

print(
    "Noise plan:"
)

print(
    "RNG manifest storage mode:",
    "30 streamed Parquet row groups; one seed held in memory",
)

print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    len(
        condition_plan
    ),
)

print(
    "RNG-manifest rows:",
    rng_readback_rows,
)

print(
    "RNG-manifest row groups:",
    rng_readback_row_groups,
)

print(
    "Failure subtypes:",
    failure_subtypes.astype(
        int
    ).tolist(),
)

print(
    "Failure-subtype probabilities:",
    failure_subtype_probabilities.tolist(),
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)

print()

print(
    "Zero-noise audit:"
)

print(
    "Zero-noise conditions:",
    len(
        zero_noise_conditions
    ),
)

print(
    "Zero-noise flip violations:",
    zero_noise_flip_violations,
)

print(
    "Zero-noise raw-label violations:",
    zero_noise_raw_label_violations,
)

print(
    "Zero-noise model-label violations:",
    zero_noise_model_label_violations,
)

print()

print(
    "Runtime:"
)

print(
    "RNG/condition-plan generation seconds:",
    round(
        rng_generation_seconds,
        2,
    ),
)

print()

print(
    "Immutability and isolation:"
)

print(
    "Project 24 source unchanged:",
    source_root_hash(
        final_source_manifest
    )
    == EXPECTED_SOURCE_ROOT_SHA256,
)

print(
    "REC checkpoint unchanged:",
    sha256_file(
        REC_CHECKPOINT_PATH
    )
    == EXPECTED_REC_CHECKPOINT_SHA256,
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–23 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models trained:",
    False,
)

print(
    "Project 24 experiment started:",
    False,
)

print()

print(
    "Validation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    int(
        (
            ~validation[
                "Pass"
            ]
        ).sum()
    ),
)

print()

print(
    "Noise-plan checkpoint:",
    NOISE_PLAN_CHECKPOINT_PATH,
)

print(
    "Checkpoint SHA-256:",
    noise_checkpoint_sha256,
)

print()

print(
    "Next required step:",
    "PROJECT 24 CELL 7 / STEP 4A — EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE",
)

print(
    "STATUS:",
    STEP3A_STATUS,
)

print(
    "=" * 140
)


=== PROJECT 24 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===

Loading full Project 24 model-ready dataset (224,550 rows).
Loading frozen Project 24 raw execution order (5,635,027 rows).


RuntimeError: Frozen Project 24 global build-order file is missing required columns.

In [7]:
# ==================================================================================================
# PROJECT 24 — CELL 6 / STEP 3A
# DETERMINISTIC NOISE PLAN, FIXED COHORTS, NESTED MASKS, AND RNG FREEZE
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_24.ipynb.
#
# PURPOSE:
# - verify the frozen Project 24 selection, source, REC reconstruction, clean anchor, and registry;
# - freeze raw and model-ready training/evaluation cohorts;
# - freeze the Project 24 failure-subtype distribution;
# - generate deterministic project/seed random streams used by label-noise injection;
# - prove masks are nested across all 9 noise levels for all 30 repetition seeds;
# - freeze all 270 condition coordinates and expected noisy-label hashes;
# - keep evaluation data clean and immutable;
# - perform no model fitting, no condition execution, and no registry write.
#
# PROJECT-24 SCALE NOTE:
# - raw training rows = 4,040,801;
# - RNG manifest rows = 4,040,801 × 30 = 121,224,030;
# - the RNG manifest is STREAMED as exactly 30 Parquet row groups, one seed at a time.
#   The full 121M-row RNG table is never materialized in RAM.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import hashlib
import json
import os
import gc
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 140)
print("=== PROJECT 24 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===")
print("=" * 140)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

SOURCE_DIR = Path(
    "/content/datasets/datasets/SonarSource@sonarqube"
)

EXPECTED_SELECTION_STATUS = (
    "PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN"
)

EXPECTED_STEP2B_STATUS = (
    "PASS_PROJECT_24_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
)

STEP3A_STATUS = (
    "PASS_PROJECT_24_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

STEP3A_CODE_REVISION = (
    "PROJECT_24_STEP3A_V2_FROZEN_GLOBAL_BUILD_ORDER_SCHEMA_FIX"
)

EXPECTED_SELECTION_SHA256 = (
    "d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803"
)

EXPECTED_REC_IMPLEMENTATION = (
    "PROJECT_24_V1_SCALABLE_EXACT_TIE_DP_BINARY_AGE_CONSTRAINT_"
    "ANCHOR_AWARE_MAPPING_BOUNDARY_AUDIT"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)

EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 23

EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 374_503_252

EXPECTED_BUILDS = 4_286
EXPECTED_TRAIN_BUILDS = 3_214
EXPECTED_EVAL_BUILDS = 1_072

EXPECTED_RAW_ROWS = 5_635_027
EXPECTED_RAW_TRAIN_ROWS = 4_040_801
EXPECTED_RAW_EVAL_ROWS = 1_594_226
EXPECTED_RAW_TRAIN_FAILURES = 1_778
EXPECTED_RAW_EVAL_FAILURES = 20

EXPECTED_MODEL_ROWS = 224_550
EXPECTED_MODEL_TRAIN_ROWS = 205_696
EXPECTED_MODEL_EVAL_ROWS = 18_854
EXPECTED_MODEL_TRAIN_FAILURES = 1_777
EXPECTED_MODEL_EVAL_FAILURES = 20
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 17

EXPECTED_DATASET_COLUMNS = 154
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19

NOISE_LEVELS = [
    0,
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

REPETITION_SEEDS = list(
    range(
        1,
        31,
    )
)

EXPECTED_CONDITIONS = (
    len(
        NOISE_LEVELS
    )
    * len(
        REPETITION_SEEDS
    )
)

EXPECTED_RNG_ROWS = (
    EXPECTED_RAW_TRAIN_ROWS
    * len(
        REPETITION_SEEDS
    )
)

EXPECTED_RNG_ROW_GROUPS = len(
    REPETITION_SEEDS
)

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_24_selection"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_selection_checkpoint.json"
)

STEP1B_STATUS_PATH = (
    SELECTION_ROOT
    / "project_24_step1b_status.json"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_24_frozen_source_manifest.csv"
)

FIXED_CHRONOLOGY_PATH = (
    SELECTION_ROOT
    / "project_24_fixed_chronological_builds.csv"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

STEP2B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step2b_status.json"
)

STEP2B_REPORT_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_step2b_report.json"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_rec_reconstruction_checkpoint.json"
)

CLEAN_RECONSTRUCTED_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_reconstructed.parquet"
)

CLEAN_ANCHOR_OFFSETS_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"
)

INFERRED_EXECUTION_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_inferred_execution_order.parquet"
)

FROZEN_GLOBAL_BUILD_ORDER_PATH = (
    REC_PREFLIGHT_ROOT
    / f"{PROJECT_SHORT}_clean_global_build_order.csv"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

FAILURE_SUBTYPE_PROFILE_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_failure_subtype_profile.csv"
)

SEED_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_seed_manifest.csv"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NESTED_MASK_AUDIT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_nested_mask_audit.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_validation.csv"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_noise_plan_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    path = Path(
        path
    )

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_array(
    array,
    dtype,
):
    canonical = np.asarray(
        array,
        dtype=dtype,
        order="C",
    )

    return hashlib.sha256(
        canonical.tobytes(
            order="C"
        )
    ).hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write(
            "\n"
        )

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_parquet(
    path,
    frame,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.SizeBytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(
            column
        ).strip().lower()
        == str(
            expected
        ).strip().lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[
        0
    ]


def parse_int(
    values,
    label,
):
    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    if numeric.isna().any():
        raise RuntimeError(
            f"{label} contains "
            f"{int(numeric.isna().sum())} "
            "missing/non-numeric values."
        )

    array = numeric.to_numpy(
        dtype=float
    )

    if not np.isclose(
        array,
        np.floor(
            array
        ),
        rtol=0,
        atol=0,
    ).all():
        raise RuntimeError(
            f"{label} contains non-integral values."
        )

    return numeric.astype(
        "int64"
    )


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[
            :4
        ],
        byteorder="little",
        signed=False,
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def checkpoint_output_sha256(
    checkpoint,
    path,
):
    target = str(
        Path(
            path
        )
    )

    matches = [
        item
        for item in checkpoint.get(
            "OutputManifest",
            [],
        )
        if str(
            item.get(
                "Path",
                "",
            )
        )
        == target
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            "REC checkpoint does not contain exactly one manifest "
            f"entry for {target}."
        )

    return str(
        matches[
            0
        ][
            "SHA256"
        ]
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS AND FROZEN-STATE VALIDATION
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    SELECTION_CHECKPOINT_PATH,
    STEP1B_STATUS_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    FIXED_CHRONOLOGY_PATH,
    STEP2B_STATUS_PATH,
    STEP2B_REPORT_PATH,
    REC_CHECKPOINT_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    SOURCE_DIR
    / "dataset.csv",
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not Path(
        path
    ).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 24 Step 3A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


selection_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)

rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)


if (
    selection_sha256
    != EXPECTED_SELECTION_SHA256
):
    raise RuntimeError(
        "Project 24 selection checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_SELECTION_SHA256}\n"
        f"Actual:   {selection_sha256}"
    )


if (
    rec_checkpoint_sha256
    != EXPECTED_REC_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 24 REC checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_REC_CHECKPOINT_SHA256}\n"
        f"Actual:   {rec_checkpoint_sha256}"
    )


selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)

step1b_status = load_json(
    STEP1B_STATUS_PATH
)

step2b_status = load_json(
    STEP2B_STATUS_PATH
)

step2b_report = load_json(
    STEP2B_REPORT_PATH
)

rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)


if (
    selection_checkpoint.get(
        "Status"
    )
    != EXPECTED_SELECTION_STATUS
    or step1b_status.get(
        "Status"
    )
    != EXPECTED_SELECTION_STATUS
):
    raise RuntimeError(
        "Project 24 selection is not frozen successfully."
    )


if (
    step2b_status.get(
        "Status"
    )
    != EXPECTED_STEP2B_STATUS
    or step2b_report.get(
        "Status"
    )
    != EXPECTED_STEP2B_STATUS
    or rec_checkpoint.get(
        "Status"
    )
    != EXPECTED_STEP2B_STATUS
):
    raise RuntimeError(
        "Project 24 Step 2B is not frozen successfully."
    )


if not bool(
    rec_checkpoint.get(
        "ZeroPercentCleanDatasetReproducedExactly",
        False,
    )
):
    raise RuntimeError(
        "Project 24 REC checkpoint does not confirm exact 0% clean reproduction."
    )


expected_rec_freeze_flags = {
    "ImplementationVersion":
        EXPECTED_REC_IMPLEMENTATION,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_24_CLEAN_REC_RECONSTRUCTION",

    "RECReconstructionFrozen":
        True,

    "PerTestExecutionOrderFrozen":
        True,

    "GlobalBuildFirstAppearanceOrderFrozen":
        True,

    "CleanAnchorFrozen":
        True,

    "MappingBoundaryAuditFrozen":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToNoisePlanAllowed":
        True,
}


for flag_name, expected_value in expected_rec_freeze_flags.items():
    actual_value = rec_checkpoint.get(
        flag_name
    )

    if (
        actual_value
        != expected_value
    ):
        raise RuntimeError(
            "Project 24 REC checkpoint contract differs.\n"
            f"Field:    {flag_name}\n"
            f"Expected: {expected_value!r}\n"
            f"Actual:   {actual_value!r}"
        )


if (
    selection_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or selection_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "Frozen Project 24 identity differs."
    )


if (
    selection_checkpoint.get(
        "ActiveReservations"
    )
    != EXPECTED_ACTIVE_RESERVATIONS
):
    raise RuntimeError(
        "Frozen Project 24 active-reservation state differs."
    )


if (
    selection_checkpoint.get(
        "RuntimePriorityRule"
    )
    != EXPECTED_RUNTIME_PRIORITY_RULE
):
    raise RuntimeError(
        "Frozen Project 24 runtime-priority rule differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. REGISTRY AND SOURCE IMMUTABILITY
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if (
    registry_sha256_before
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry SHA-256 differs.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA256}\n"
        f"Actual:   {registry_sha256_before}"
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


registry_project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS
            + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry must contain exactly Projects 1–23."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )


for (
    required_number,
    required_project,
) in REQUIRED_REGISTERED_IDENTITIES.items():
    matching = registry.loc[
        registry_project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching
        )
        != 1
        or str(
            matching.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A frozen predecessor identity differs.\n"
            f"Project {required_number}; expected={required_project}"
        )


if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 24 is unexpectedly already registered."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)

current_source_bytes = int(
    current_source_manifest[
        "SizeBytes"
    ].sum()
)


if (
    len(
        current_source_manifest
    )
    != EXPECTED_SOURCE_FILES
    or current_source_bytes
    != EXPECTED_SOURCE_BYTES
    or current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 24 source differs."
    )


# Verify all four core Step 2B freeze artifacts against the REC checkpoint manifest.
core_rec_paths = [
    INFERRED_EXECUTION_ORDER_PATH,
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    CLEAN_RECONSTRUCTED_PATH,
    CLEAN_ANCHOR_OFFSETS_PATH,
]


for core_path in core_rec_paths:
    expected_sha = checkpoint_output_sha256(
        rec_checkpoint,
        core_path,
    )

    actual_sha = sha256_file(
        core_path
    )

    if (
        actual_sha
        != expected_sha
    ):
        raise RuntimeError(
            "A frozen Project 24 Step 2B artifact changed.\n"
            f"Path:     {core_path}\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}"
        )


# --------------------------------------------------------------------------------------------------
# 6. LOAD CHRONOLOGY, MODEL DATA, AND FROZEN RAW-HISTORY ORDER
# --------------------------------------------------------------------------------------------------

chronology = pd.read_csv(
    FIXED_CHRONOLOGY_PATH,
    low_memory=False,
)


required_chronology_columns = {
    "BuildOrder",
    "Build",
    "StartedAtUTC",
    "TimestampTieGroupSize",
    "InTimestampTie",
    "Split",
}


if not required_chronology_columns.issubset(
    chronology.columns
):
    raise RuntimeError(
        "Frozen Project 24 chronology is missing required columns."
    )


chronology[
    "Build"
] = parse_int(
    chronology[
        "Build"
    ],
    "chronology.Build",
)


chronology[
    "BuildOrder"
] = parse_int(
    chronology[
        "BuildOrder"
    ],
    "chronology.BuildOrder",
)


training_builds = set(
    chronology.loc[
        chronology[
            "Split"
        ].eq(
            "TRAIN"
        ),
        "Build",
    ].astype(
        int
    )
)


evaluation_builds = set(
    chronology.loc[
        chronology[
            "Split"
        ].eq(
            "EVAL"
        ),
        "Build",
    ].astype(
        int
    )
)


if (
    len(
        chronology
    )
    != EXPECTED_BUILDS
    or len(
        training_builds
    )
    != EXPECTED_TRAIN_BUILDS
    or len(
        evaluation_builds
    )
    != EXPECTED_EVAL_BUILDS
    or training_builds
    & evaluation_builds
):
    raise RuntimeError(
        "Frozen Project 24 chronology dimensions differ."
    )


dataset_header = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    nrows=0,
).columns.tolist()


dataset_build_column = resolve_column(
    dataset_header,
    "Build",
    "dataset Build",
)

dataset_test_column = resolve_column(
    dataset_header,
    "Test",
    "dataset Test",
)

dataset_verdict_column = resolve_column(
    dataset_header,
    "Verdict",
    "dataset Verdict",
)


predictor_columns = [
    column
    for column in dataset_header
    if column not in {
        dataset_build_column,
        dataset_test_column,
        dataset_verdict_column,
    }
]


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in dataset_header
]


if (
    missing_rec_features
    or len(
        dataset_header
    )
    != EXPECTED_DATASET_COLUMNS
    or len(
        predictor_columns
    )
    != EXPECTED_PREDICTORS
    or len(
        REC_FEATURES
    )
    != EXPECTED_REC_FEATURES
):
    raise RuntimeError(
        "Project 24 predictor/REC schema differs."
    )


print(
    f"\nLoading full Project 24 model-ready dataset ({EXPECTED_MODEL_ROWS:,} rows)."
)


dataset = pd.read_csv(
    SOURCE_DIR
    / "dataset.csv",
    low_memory=False,
).rename(
    columns={
        dataset_build_column:
            "Build",

        dataset_test_column:
            "Test",

        dataset_verdict_column:
            "Verdict",
    }
)


dataset[
    "Build"
] = parse_int(
    dataset[
        "Build"
    ],
    "dataset.Build",
)


dataset[
    "Test"
] = parse_int(
    dataset[
        "Test"
    ],
    "dataset.Test",
)


dataset[
    "Verdict"
] = parse_int(
    dataset[
        "Verdict"
    ],
    "dataset.Verdict",
)


if (
    len(
        dataset
    )
    != EXPECTED_MODEL_ROWS
    or dataset.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).any()
):
    raise RuntimeError(
        "Project 24 model-ready cohort dimensions/keys differ."
    )


print(
    f"Loading frozen Project 24 raw execution order ({EXPECTED_RAW_ROWS:,} rows)."
)


inferred_execution_order = pd.read_parquet(
    INFERRED_EXECUTION_ORDER_PATH
)


required_inferred_columns = {
    "Build",
    "Test",
    "Job",
    "Verdict",
    "Duration",
    "StartedAtUTC",
    "InferredTestOrder",
}


if not required_inferred_columns.issubset(
    inferred_execution_order.columns
):
    raise RuntimeError(
        "Frozen inferred execution-order parquet is missing required columns."
    )


for column in [
    "Build",
    "Test",
    "Verdict",
    "InferredTestOrder",
]:
    inferred_execution_order[
        column
    ] = parse_int(
        inferred_execution_order[
            column
        ],
        f"inferred_execution_order.{column}",
    )


inferred_execution_order[
    "Job"
] = pd.to_numeric(
    inferred_execution_order[
        "Job"
    ],
    errors="coerce",
)


inferred_execution_order[
    "Duration"
] = pd.to_numeric(
    inferred_execution_order[
        "Duration"
    ],
    errors="coerce",
)


if (
    inferred_execution_order[
        "Job"
    ].isna().any()
    or inferred_execution_order[
        "Duration"
    ].isna().any()
    or not np.isfinite(
        inferred_execution_order[
            "Duration"
        ].to_numpy(
            dtype=float
        )
    ).all()
    or inferred_execution_order[
        "Duration"
    ].lt(
        0
    ).any()
):
    raise RuntimeError(
        "Frozen raw execution order has invalid job/duration values."
    )


raw_duplicate_build_test_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).sum()
)


raw_duplicate_test_order_rows = int(
    inferred_execution_order.duplicated(
        subset=[
            "Test",
            "InferredTestOrder",
        ],
        keep=False,
    ).sum()
)


if (
    len(
        inferred_execution_order
    )
    != EXPECTED_RAW_ROWS
    or raw_duplicate_build_test_rows
    != 0
    or raw_duplicate_test_order_rows
    != 0
):
    raise RuntimeError(
        "Frozen raw-history order contract differs."
    )


exe = (
    inferred_execution_order.sort_values(
        [
            "Test",
            "InferredTestOrder",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
    .copy()
)


# Memory-only cleanup: exe is now the canonical frozen raw-history frame.
del inferred_execution_order
gc.collect()


frozen_global_build_order = pd.read_csv(
    FROZEN_GLOBAL_BUILD_ORDER_PATH,
    low_memory=False,
)


# IMPORTANT PROJECT-24 SCHEMA CONTRACT:
# Step 2B froze this file with the same canonical schema used by the earlier thesis projects:
#   GlobalBuildOrder = one-based order 1..4286
#   BuildID          = build identifier
#   StartedAtUTC     = diagnostic timestamp column
#
# The original Step 3A V1 incorrectly expected the internal Step-2B working names
# GlobalBuildPosition / Build. That was only a consumer-schema bug; the frozen Step 2B
# artifact itself is valid and its SHA-256 was already verified above against the REC checkpoint.
required_global_order_columns = {
    "GlobalBuildOrder",
    "BuildID",
}


if not required_global_order_columns.issubset(
    frozen_global_build_order.columns
):
    raise RuntimeError(
        "Frozen Project 24 global build-order file is missing required columns.\n"
        f"Expected core columns: {sorted(required_global_order_columns)}\n"
        f"Actual columns:        {list(frozen_global_build_order.columns)}"
    )


for column in [
    "GlobalBuildOrder",
    "BuildID",
]:
    frozen_global_build_order[
        column
    ] = parse_int(
        frozen_global_build_order[
            column
        ],
        f"frozen_global_build_order.{column}",
    )


frozen_global_build_order = (
    frozen_global_build_order.sort_values(
        "GlobalBuildOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


global_build_order_valid = bool(
    len(
        frozen_global_build_order
    )
    == EXPECTED_BUILDS
    and frozen_global_build_order[
        "BuildID"
    ].nunique()
    == EXPECTED_BUILDS
    and set(
        frozen_global_build_order[
            "BuildID"
        ].astype(
            int
        )
    )
    == (
        training_builds
        | evaluation_builds
    )
    and np.array_equal(
        frozen_global_build_order[
            "GlobalBuildOrder"
        ].to_numpy(
            dtype=np.int64
        ),
        np.arange(
            1,
            EXPECTED_BUILDS
            + 1,
            dtype=np.int64,
        ),
    )
)


if not global_build_order_valid:
    raise RuntimeError(
        "Frozen Project 24 global build order is invalid."
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY CLEAN REC + ANCHOR EXACTLY REPRODUCES THE SOURCE DATASET
# --------------------------------------------------------------------------------------------------

clean_rec = pd.read_parquet(
    CLEAN_RECONSTRUCTED_PATH
)


clean_anchor = pd.read_parquet(
    CLEAN_ANCHOR_OFFSETS_PATH
)


if (
    len(
        clean_rec
    )
    != EXPECTED_MODEL_ROWS
    or len(
        clean_anchor
    )
    != EXPECTED_MODEL_ROWS
):
    raise RuntimeError(
        "Frozen clean REC / anchor row counts differ."
    )


if not clean_rec[
    [
        "Build",
        "Test",
    ]
].equals(
    clean_anchor[
        [
            "Build",
            "Test",
        ]
    ]
):
    raise RuntimeError(
        "Frozen clean REC / anchor Build-Test keys differ."
    )


if not clean_rec[
    [
        "Build",
        "Test",
    ]
].equals(
    dataset[
        [
            "Build",
            "Test",
        ]
    ]
):
    raise RuntimeError(
        "Frozen clean REC Build-Test order differs from source dataset."
    )


clean_anchor_mismatch_values = 0


for feature in REC_FEATURES:
    reproduced = (
        clean_rec[
            feature
        ].to_numpy(
            dtype=np.float64
        )
        + clean_anchor[
            feature
        ].to_numpy(
            dtype=np.float64
        )
    )

    original = dataset[
        feature
    ].to_numpy(
        dtype=np.float64
    )

    clean_anchor_mismatch_values += int(
        (
            ~np.isclose(
                reproduced,
                original,
                rtol=0.0,
                atol=1e-12,
                equal_nan=False,
            )
        ).sum()
    )


clean_anchor_reproduced_dataset = bool(
    clean_anchor_mismatch_values
    == 0
)


if not clean_anchor_reproduced_dataset:
    raise RuntimeError(
        "Frozen Project 24 clean REC + anchor does not reproduce the clean source dataset."
    )


# --------------------------------------------------------------------------------------------------
# 8. FREEZE RAW AND MODEL COHORTS + MODEL/RAW LINKS
# --------------------------------------------------------------------------------------------------

raw_training = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_training.insert(
    0,
    "RawTrainingRowOrder",
    np.arange(
        1,
        len(
            raw_training
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_evaluation = (
    exe.loc[
        exe[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


raw_evaluation.insert(
    0,
    "RawEvaluationRowOrder",
    np.arange(
        1,
        len(
            raw_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


model_training = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            training_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_training.insert(
    0,
    "ModelTrainingRowOrder",
    np.arange(
        1,
        len(
            model_training
        )
        + 1,
        dtype=np.int64,
    ),
)


model_evaluation = (
    dataset.loc[
        dataset[
            "Build"
        ].isin(
            evaluation_builds
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


model_evaluation.insert(
    0,
    "ModelEvaluationRowOrder",
    np.arange(
        1,
        len(
            model_evaluation
        )
        + 1,
        dtype=np.int64,
    ),
)


raw_training_failures = int(
    raw_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)


raw_evaluation_failures = int(
    raw_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)


model_training_failures = int(
    model_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)


model_evaluation_failures = int(
    model_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "Verdict"
        ].ne(
            0
        ),
        "Build",
    ].nunique()
)


raw_training_link_source = raw_training[
    [
        "RawTrainingRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_training_link = (
    model_training[
        [
            "ModelTrainingRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_training_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelTrainingRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


raw_evaluation_link_source = raw_evaluation[
    [
        "RawEvaluationRowOrder",
        "Build",
        "Test",
        "Verdict",
    ]
].rename(
    columns={
        "Verdict":
            "RawVerdict",
    }
)


model_evaluation_link = (
    model_evaluation[
        [
            "ModelEvaluationRowOrder",
            "Build",
            "Test",
            "Verdict",
        ]
    ]
    .rename(
        columns={
            "Verdict":
                "ModelVerdict",
        }
    )
    .merge(
        raw_evaluation_link_source,
        on=[
            "Build",
            "Test",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        "ModelEvaluationRowOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


missing_model_training_links = int(
    model_training_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


missing_model_evaluation_links = int(
    model_evaluation_link[
        "_merge"
    ].ne(
        "both"
    ).sum()
)


model_training_verdict_mismatches = int(
    model_training_link[
        "ModelVerdict"
    ].ne(
        model_training_link[
            "RawVerdict"
        ]
    ).sum()
)


model_evaluation_verdict_mismatches = int(
    model_evaluation_link[
        "ModelVerdict"
    ].ne(
        model_evaluation_link[
            "RawVerdict"
        ]
    ).sum()
)


if (
    len(
        raw_training
    )
    != EXPECTED_RAW_TRAIN_ROWS
    or len(
        raw_evaluation
    )
    != EXPECTED_RAW_EVAL_ROWS
    or raw_training_failures
    != EXPECTED_RAW_TRAIN_FAILURES
    or raw_evaluation_failures
    != EXPECTED_RAW_EVAL_FAILURES
    or len(
        model_training
    )
    != EXPECTED_MODEL_TRAIN_ROWS
    or len(
        model_evaluation
    )
    != EXPECTED_MODEL_EVAL_ROWS
    or model_training_failures
    != EXPECTED_MODEL_TRAIN_FAILURES
    or model_evaluation_failures
    != EXPECTED_MODEL_EVAL_FAILURES
    or model_failing_evaluation_builds
    != EXPECTED_MODEL_FAILING_EVAL_BUILDS
    or missing_model_training_links
    != 0
    or missing_model_evaluation_links
    != 0
    or model_training_verdict_mismatches
    != 0
    or model_evaluation_verdict_mismatches
    != 0
):
    raise RuntimeError(
        "Project 24 fixed cohort/link contract differs."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE CLEAN FAILURE-SUBTYPE DISTRIBUTION
# --------------------------------------------------------------------------------------------------

failure_subtype_counts = (
    raw_training.loc[
        raw_training[
            "Verdict"
        ].ne(
            0
        ),
        "Verdict",
    ]
    .value_counts()
    .sort_index()
)


failure_subtypes = failure_subtype_counts.index.to_numpy(
    dtype=np.int16
)


failure_subtype_probabilities = (
    failure_subtype_counts.to_numpy(
        dtype=float
    )
    / failure_subtype_counts.sum()
)


failure_subtype_profile = pd.DataFrame({
    "FailureSubtype":
        failure_subtypes.astype(
            int
        ),

    "CleanTrainingRows":
        failure_subtype_counts.to_numpy(
            dtype=int
        ),

    "Probability":
        failure_subtype_probabilities,
})


failure_subtype_values_list = failure_subtypes.astype(
    int
).tolist()


failure_subtype_values_valid = bool(
    len(
        failure_subtype_values_list
    )
    > 0
    and set(
        failure_subtype_values_list
    ).issubset({
        1,
        2,
    })
)


failure_subtype_profile_sum_valid = bool(
    int(
        failure_subtype_profile[
            "CleanTrainingRows"
        ].sum()
    )
    == raw_training_failures
    and np.isclose(
        failure_subtype_profile[
            "Probability"
        ].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )
)


if not failure_subtype_values_valid:
    raise RuntimeError(
        "Project 24 clean training failures must use a non-empty subset "
        "of the frozen exception/assertion codes [1, 2]."
    )


if not failure_subtype_profile_sum_valid:
    raise RuntimeError(
        "Project 24 failure-subtype profile does not reproduce "
        "the clean raw training failure count."
    )


# --------------------------------------------------------------------------------------------------
# 10. GENERATE 30 DETERMINISTIC RNG STREAMS + 270 CONDITION PLAN
# --------------------------------------------------------------------------------------------------

NOISE_PLAN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


rng_temporary_path = RNG_MANIFEST_PATH.with_name(
    f".{RNG_MANIFEST_PATH.name}.tmp_{os.getpid()}"
)


if rng_temporary_path.exists():
    rng_temporary_path.unlink()


rng_schema = pa.schema([
    pa.field(
        "RepetitionSeed",
        pa.int16(),
    ),

    pa.field(
        "RawTrainingRowOrder",
        pa.int32(),
    ),

    pa.field(
        "FlipUniform",
        pa.float64(),
    ),

    pa.field(
        "SampledFailureSubtype",
        pa.int16(),
    ),
])


rng_writer = pq.ParquetWriter(
    rng_temporary_path,
    schema=rng_schema,
    compression="zstd",
)


seed_records = []
condition_records = []
nested_mask_records = []


clean_raw_verdict = raw_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)


clean_model_verdict = model_training[
    "Verdict"
].to_numpy(
    dtype=np.int16
)


model_training_raw_indices = (
    model_training_link[
        "RawTrainingRowOrder"
    ].to_numpy(
        dtype=np.int64
    )
    - 1
)


if (
    model_training_raw_indices.min()
    < 0
    or model_training_raw_indices.max()
    >= len(
        raw_training
    )
):
    raise RuntimeError(
        "Model-to-raw training indices are out of bounds."
    )


raw_row_order_int32 = np.arange(
    1,
    len(
        raw_training
    )
    + 1,
    dtype=np.int32,
)


rng_generation_started = time.perf_counter()


try:
    condition_order = 0

    for seed_order, repetition_seed in enumerate(
        REPETITION_SEEDS,
        start=1,
    ):
        seed_started = time.perf_counter()

        flip_seed = deterministic_seed(
            repetition_seed,
            "flip_mask",
        )

        failure_subtype_seed = deterministic_seed(
            repetition_seed,
            "failure_subtype",
        )


        flip_uniform = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )


        sampled_failure_subtype = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        # Immediate deterministic regeneration proof.
        flip_uniform_reproduced = np.random.default_rng(
            flip_seed
        ).random(
            len(
                raw_training
            )
        )


        uniforms_reproduced = bool(
            np.array_equal(
                flip_uniform,
                flip_uniform_reproduced,
            )
        )


        del flip_uniform_reproduced


        failure_subtype_reproduced = np.random.default_rng(
            failure_subtype_seed
        ).choice(
            failure_subtypes,
            size=len(
                raw_training
            ),
            replace=True,
            p=failure_subtype_probabilities,
        ).astype(
            np.int16
        )


        failure_subtypes_reproduced = bool(
            np.array_equal(
                sampled_failure_subtype,
                failure_subtype_reproduced,
            )
        )


        del failure_subtype_reproduced


        if (
            not uniforms_reproduced
            or not failure_subtypes_reproduced
        ):
            raise RuntimeError(
                f"Deterministic RNG regeneration failed for seed {repetition_seed}."
            )


        seed_records.append({
            "SeedOrder":
                seed_order,

            "RepetitionSeed":
                repetition_seed,

            "FlipSeed":
                flip_seed,

            "FailureSubtypeSeed":
                failure_subtype_seed,

            "FlipUniformSHA256":
                sha256_array(
                    flip_uniform,
                    "<f8",
                ),

            "SampledFailureSubtypeSHA256":
                sha256_array(
                    sampled_failure_subtype,
                    "<i2",
                ),

            "UniformsReproduced":
                uniforms_reproduced,

            "FailureSubtypesReproduced":
                failure_subtypes_reproduced,
        })


        # Stream exactly one seed as one Parquet row group.
        seed_array = np.full(
            len(
                raw_training
            ),
            repetition_seed,
            dtype=np.int16,
        )


        rng_table = pa.Table.from_arrays(
            [
                pa.array(
                    seed_array,
                    type=pa.int16(),
                ),

                pa.array(
                    raw_row_order_int32,
                    type=pa.int32(),
                ),

                pa.array(
                    flip_uniform,
                    type=pa.float64(),
                ),

                pa.array(
                    sampled_failure_subtype,
                    type=pa.int16(),
                ),
            ],
            schema=rng_schema,
        )


        rng_writer.write_table(
            rng_table,
            row_group_size=len(
                raw_training
            ),
        )


        del rng_table
        del seed_array


        previous_mask = None
        previous_noise = None


        for noise_order, noise_percent in enumerate(
            NOISE_LEVELS,
            start=1,
        ):
            condition_order += 1

            condition_id = (
                f"noise_{noise_percent:02d}"
                f"__seed_{repetition_seed:02d}"
            )


            flip_mask = (
                flip_uniform
                < (
                    noise_percent
                    / 100.0
                )
            )


            noisy_raw_verdict = clean_raw_verdict.copy()


            pass_to_failure_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    == 0
                )
            )


            failure_to_pass_mask = (
                flip_mask
                & (
                    clean_raw_verdict
                    != 0
                )
            )


            noisy_raw_verdict[
                pass_to_failure_mask
            ] = sampled_failure_subtype[
                pass_to_failure_mask
            ]


            noisy_raw_verdict[
                failure_to_pass_mask
            ] = 0


            noisy_model_verdict = noisy_raw_verdict[
                model_training_raw_indices
            ]


            number_flipped = int(
                flip_mask.sum()
            )


            pass_to_failure = int(
                pass_to_failure_mask.sum()
            )


            failure_to_pass = int(
                failure_to_pass_mask.sum()
            )


            noisy_raw_failures = int(
                (
                    noisy_raw_verdict
                    != 0
                ).sum()
            )


            model_label_changes = int(
                (
                    noisy_model_verdict
                    != clean_model_verdict
                ).sum()
            )


            noisy_model_failures = int(
                (
                    noisy_model_verdict
                    != 0
                ).sum()
            )


            condition_records.append({
                "ConditionOrder":
                    condition_order,

                "ConditionID":
                    condition_id,

                "SeedOrder":
                    seed_order,

                "NoiseOrderWithinSeed":
                    noise_order,

                "NoisePercent":
                    noise_percent,

                "RepetitionSeed":
                    repetition_seed,

                "FlipSeed":
                    flip_seed,

                "FailureSubtypeSeed":
                    failure_subtype_seed,

                "RawTrainingRows":
                    len(
                        raw_training
                    ),

                "NumberFlipped":
                    number_flipped,

                "RealisedNoisePercent":
                    (
                        100.0
                        * number_flipped
                        / len(
                            raw_training
                        )
                    ),

                "PassToFailure":
                    pass_to_failure,

                "FailureToPass":
                    failure_to_pass,

                "CleanRawFailures":
                    raw_training_failures,

                "NoisyRawFailures":
                    noisy_raw_failures,

                "ModelTrainingRows":
                    len(
                        model_training
                    ),

                "ModelLabelChanges":
                    model_label_changes,

                "CleanModelFailures":
                    model_training_failures,

                "NoisyModelFailures":
                    noisy_model_failures,

                "FlipMaskSHA256":
                    sha256_array(
                        flip_mask.astype(
                            np.uint8
                        ),
                        "u1",
                    ),

                "NoisyRawVerdictSHA256":
                    sha256_array(
                        noisy_raw_verdict,
                        "<i2",
                    ),

                "NoisyModelVerdictSHA256":
                    sha256_array(
                        noisy_model_verdict,
                        "<i2",
                    ),
            })


            if previous_mask is not None:
                violations = int(
                    (
                        previous_mask
                        & (
                            ~flip_mask
                        )
                    ).sum()
                )


                nested_mask_records.append({
                    "RepetitionSeed":
                        repetition_seed,

                    "LowerNoisePercent":
                        previous_noise,

                    "HigherNoisePercent":
                        noise_percent,

                    "Violations":
                        violations,

                    "Pass":
                        violations == 0,
                })


            previous_mask = flip_mask
            previous_noise = noise_percent


            del noisy_raw_verdict
            del noisy_model_verdict
            del pass_to_failure_mask
            del failure_to_pass_mask

            if previous_mask is not flip_mask:
                del flip_mask


        del flip_uniform
        del sampled_failure_subtype
        del previous_mask

        gc.collect()

        print(
            "    RNG/condition-plan progress:",
            seed_order,
            "/",
            len(
                REPETITION_SEEDS
            ),
            "seeds | this seed seconds:",
            round(
                time.perf_counter()
                - seed_started,
                2,
            ),
            "| total seconds:",
            round(
                time.perf_counter()
                - rng_generation_started,
                2,
            ),
        )


finally:
    rng_writer.close()


os.replace(
    rng_temporary_path,
    RNG_MANIFEST_PATH,
)


rng_generation_seconds = float(
    time.perf_counter()
    - rng_generation_started
)


seed_manifest = pd.DataFrame(
    seed_records
)


condition_plan = pd.DataFrame(
    condition_records
)


nested_mask_audit = pd.DataFrame(
    nested_mask_records
)


# --------------------------------------------------------------------------------------------------
# 11. PLAN AUDITS
# --------------------------------------------------------------------------------------------------

nested_mask_violations = int(
    nested_mask_audit[
        "Violations"
    ].sum()
)


zero_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].eq(
        0
    )
]


zero_noise_flip_violations = int(
    zero_noise_conditions[
        "NumberFlipped"
    ].ne(
        0
    ).sum()
)


zero_noise_raw_label_violations = int(
    zero_noise_conditions[
        "NoisyRawFailures"
    ].ne(
        raw_training_failures
    ).sum()
)


zero_noise_model_label_violations = int(
    zero_noise_conditions[
        "ModelLabelChanges"
    ].ne(
        0
    ).sum()
)


positive_noise_conditions = condition_plan[
    condition_plan[
        "NoisePercent"
    ].gt(
        0
    )
]


positive_noise_without_raw_changes = int(
    positive_noise_conditions[
        "NumberFlipped"
    ].le(
        0
    ).sum()
)


positive_noise_without_model_changes = int(
    positive_noise_conditions[
        "ModelLabelChanges"
    ].le(
        0
    ).sum()
)


duplicate_condition_ids = int(
    condition_plan[
        "ConditionID"
    ].duplicated(
        keep=False
    ).sum()
)


duplicate_condition_coordinates = int(
    condition_plan.duplicated(
        subset=[
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


seed_streams_reproduced = bool(
    seed_manifest[
        [
            "UniformsReproduced",
            "FailureSubtypesReproduced",
        ]
    ].all().all()
)


condition_coordinates_exact = bool(
    set(
        zip(
            condition_plan[
                "NoisePercent"
            ].astype(
                int
            ),
            condition_plan[
                "RepetitionSeed"
            ].astype(
                int
            ),
        )
    )
    == {
        (
            noise_percent,
            repetition_seed,
        )
        for repetition_seed in REPETITION_SEEDS
        for noise_percent in NOISE_LEVELS
    }
)


# --------------------------------------------------------------------------------------------------
# 12. PRE-WRITE VALIDATION
# --------------------------------------------------------------------------------------------------

checks = []


def C(
    check,
    expected,
    actual,
    passed,
):
    add_check(
        checks,
        check,
        expected,
        actual,
        passed,
    )


C(
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_SHA256,
    selection_sha256,
    selection_sha256
    == EXPECTED_SELECTION_SHA256,
)


C(
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256
    == EXPECTED_REC_CHECKPOINT_SHA256,
)


C(
    "REC implementation",
    EXPECTED_REC_IMPLEMENTATION,
    rec_checkpoint.get(
        "ImplementationVersion"
    ),
    rec_checkpoint.get(
        "ImplementationVersion"
    )
    == EXPECTED_REC_IMPLEMENTATION,
)


C(
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)


C(
    "Source files",
    EXPECTED_SOURCE_FILES,
    len(
        current_source_manifest
    ),
    len(
        current_source_manifest
    )
    == EXPECTED_SOURCE_FILES,
)


C(
    "Source bytes",
    EXPECTED_SOURCE_BYTES,
    current_source_bytes,
    current_source_bytes
    == EXPECTED_SOURCE_BYTES,
)


C(
    "Clean anchor mismatch values",
    0,
    clean_anchor_mismatch_values,
    clean_anchor_mismatch_values
    == 0,
)


C(
    "Clean anchor reproduced source dataset",
    True,
    clean_anchor_reproduced_dataset,
    clean_anchor_reproduced_dataset,
)


C(
    "Raw history rows",
    EXPECTED_RAW_ROWS,
    len(
        exe
    ),
    len(
        exe
    )
    == EXPECTED_RAW_ROWS,
)


C(
    "Raw duplicate Build-Test rows",
    0,
    raw_duplicate_build_test_rows,
    raw_duplicate_build_test_rows
    == 0,
)


C(
    "Raw duplicate Test-order rows",
    0,
    raw_duplicate_test_order_rows,
    raw_duplicate_test_order_rows
    == 0,
)


C(
    "Global build order valid",
    True,
    global_build_order_valid,
    global_build_order_valid,
)


C(
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    len(
        raw_training
    ),
    len(
        raw_training
    )
    == EXPECTED_RAW_TRAIN_ROWS,
)


C(
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    len(
        raw_evaluation
    ),
    len(
        raw_evaluation
    )
    == EXPECTED_RAW_EVAL_ROWS,
)


C(
    "Raw training failures",
    EXPECTED_RAW_TRAIN_FAILURES,
    raw_training_failures,
    raw_training_failures
    == EXPECTED_RAW_TRAIN_FAILURES,
)


C(
    "Raw evaluation failures",
    EXPECTED_RAW_EVAL_FAILURES,
    raw_evaluation_failures,
    raw_evaluation_failures
    == EXPECTED_RAW_EVAL_FAILURES,
)


C(
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)


C(
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)


C(
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)


C(
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)


C(
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)


C(
    "Missing model training raw links",
    0,
    missing_model_training_links,
    missing_model_training_links
    == 0,
)


C(
    "Missing model evaluation raw links",
    0,
    missing_model_evaluation_links,
    missing_model_evaluation_links
    == 0,
)


C(
    "Model training/raw verdict mismatches",
    0,
    model_training_verdict_mismatches,
    model_training_verdict_mismatches
    == 0,
)


C(
    "Model evaluation/raw verdict mismatches",
    0,
    model_evaluation_verdict_mismatches,
    model_evaluation_verdict_mismatches
    == 0,
)


C(
    "Failure subtype values valid",
    True,
    failure_subtype_values_valid,
    failure_subtype_values_valid,
)


C(
    "Failure subtype profile valid",
    True,
    failure_subtype_profile_sum_valid,
    failure_subtype_profile_sum_valid,
)


C(
    "Seed rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest
    ),
    len(
        seed_manifest
    )
    == len(
        REPETITION_SEEDS
    ),
)


C(
    "Seed streams reproduced",
    True,
    seed_streams_reproduced,
    seed_streams_reproduced,
)


C(
    "Conditions",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    )
    == EXPECTED_CONDITIONS,
)


C(
    "Condition coordinates exact",
    True,
    condition_coordinates_exact,
    condition_coordinates_exact,
)


C(
    "Duplicate condition IDs",
    0,
    duplicate_condition_ids,
    duplicate_condition_ids
    == 0,
)


C(
    "Duplicate condition coordinates",
    0,
    duplicate_condition_coordinates,
    duplicate_condition_coordinates
    == 0,
)


C(
    "Nested-mask audit rows",
    (
        len(
            REPETITION_SEEDS
        )
        * (
            len(
                NOISE_LEVELS
            )
            - 1
        )
    ),
    len(
        nested_mask_audit
    ),
    len(
        nested_mask_audit
    )
    == (
        len(
            REPETITION_SEEDS
        )
        * (
            len(
                NOISE_LEVELS
            )
            - 1
        )
    ),
)


C(
    "Nested-mask violations",
    0,
    nested_mask_violations,
    nested_mask_violations
    == 0,
)


C(
    "Zero-noise conditions",
    len(
        REPETITION_SEEDS
    ),
    len(
        zero_noise_conditions
    ),
    len(
        zero_noise_conditions
    )
    == len(
        REPETITION_SEEDS
    ),
)


C(
    "Zero-noise flip violations",
    0,
    zero_noise_flip_violations,
    zero_noise_flip_violations
    == 0,
)


C(
    "Zero-noise raw-label violations",
    0,
    zero_noise_raw_label_violations,
    zero_noise_raw_label_violations
    == 0,
)


C(
    "Zero-noise model-label violations",
    0,
    zero_noise_model_label_violations,
    zero_noise_model_label_violations
    == 0,
)


C(
    "Positive-noise conditions without raw changes",
    0,
    positive_noise_without_raw_changes,
    positive_noise_without_raw_changes
    == 0,
)


C(
    "Positive-noise conditions without model changes",
    0,
    positive_noise_without_model_changes,
    positive_noise_without_model_changes
    == 0,
)


C(
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)


for (
    required_number,
    required_project,
) in REQUIRED_REGISTERED_IDENTITIES.items():
    actual_project = str(
        registry.loc[
            registry_project_numbers.eq(
                required_number
            ),
            project_column,
        ].iloc[
            0
        ]
    )


    C(
        f"Project {required_number} frozen identity",
        required_project,
        actual_project,
        actual_project
        == required_project,
    )


C(
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    selection_checkpoint.get(
        "ActiveReservations"
    ),
    selection_checkpoint.get(
        "ActiveReservations"
    )
    == EXPECTED_ACTIVE_RESERVATIONS,
)


C(
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    selection_checkpoint.get(
        "RuntimePriorityRule"
    ),
    selection_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)


C(
    "Project 24 registry rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


validation = pd.DataFrame(
    checks
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 24 Step 3A pre-write validation:"
)


display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 24 Step 3A checks:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 24 STEP 3A VALIDATION FAILED. "
        "No frozen cohort/checkpoint outputs were written."
    )


# --------------------------------------------------------------------------------------------------
# 13. WRITE FROZEN COHORTS AND PLAN OUTPUTS
# --------------------------------------------------------------------------------------------------

print(
    "\nWriting fixed Project 24 cohorts."
)


atomic_parquet(
    RAW_TRAINING_COHORT_PATH,
    raw_training,
)


atomic_parquet(
    RAW_EVALUATION_COHORT_PATH,
    raw_evaluation,
)


atomic_parquet(
    MODEL_TRAINING_COHORT_PATH,
    model_training,
)


atomic_parquet(
    MODEL_EVALUATION_COHORT_PATH,
    model_evaluation,
)


atomic_parquet(
    MODEL_RAW_TRAIN_LINK_PATH,
    model_training_link.drop(
        columns=[
            "_merge",
        ]
    ),
)


atomic_parquet(
    MODEL_RAW_EVAL_LINK_PATH,
    model_evaluation_link.drop(
        columns=[
            "_merge",
        ]
    ),
)


atomic_csv(
    FAILURE_SUBTYPE_PROFILE_PATH,
    failure_subtype_profile,
)


atomic_csv(
    SEED_MANIFEST_PATH,
    seed_manifest,
)


atomic_csv(
    CONDITION_PLAN_PATH,
    condition_plan,
)


atomic_csv(
    NESTED_MASK_AUDIT_PATH,
    nested_mask_audit,
)


# --------------------------------------------------------------------------------------------------
# 14. FREEZE EXPERIMENT PROTOCOL
# --------------------------------------------------------------------------------------------------

protocol_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "ProtocolVersion":
        1,

    "Step3ACodeRevision":
        STEP3A_CODE_REVISION,

    "NoiseModel":
        "Symmetric binary pass/failure flips with clean empirical failure-subtype sampling",

    "CleanVerdictCodes": {
        "Pass":
            0,

        "ExceptionFailure":
            1,

        "AssertionFailure":
            2,
    },

    "NoiseLevelsPercent":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        EXPECTED_CONDITIONS,

    "DeterministicSeedDerivation":
        "uint32 little-endian first 4 bytes of SHA256(Project|Seed|StreamName)",

    "RNGStreamsPerSeed": [
        "flip_mask",
        "failure_subtype",
    ],

    "NestedMasks":
        True,

    "RNGManifestStorage":
        (
            "30 streamed Parquet row groups; exactly one repetition seed "
            "per row group; full 121,224,030-row manifest never materialized in RAM"
        ),

    "RNGManifestRows":
        EXPECTED_RNG_ROWS,

    "RNGManifestRowGroups":
        EXPECTED_RNG_ROW_GROUPS,

    "FailureSubtypeSource":
        "Empirical clean raw-training failure subtype distribution",

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "TrainingNoiseAppliedTo":
        "Frozen raw training labels only",

    "ModelTrainingLabels":
        "Inherited by exact frozen model-to-raw training Build-Test links",

    "EvaluationNoiseApplied":
        False,

    "EvaluationCohortImmutable":
        True,

    "CleanRECCheckpointSHA256":
        rec_checkpoint_sha256,

    "CleanRECImplementation":
        EXPECTED_REC_IMPLEMENTATION,

    "CleanAnchorReproducedDataset":
        clean_anchor_reproduced_dataset,

    "RECFeatures":
        REC_FEATURES,

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "ModelsFittedInStep3A":
        False,

    "ExperimentConditionsExecutedInStep3A":
        False,

    "CompletionRegistryWrittenInStep3A":
        False,
}


atomic_json(
    PROTOCOL_PATH,
    protocol_payload,
)


# --------------------------------------------------------------------------------------------------
# 15. RNG MANIFEST READBACK + FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

print(
    "\nValidating streamed RNG-manifest metadata."
)


rng_parquet = pq.ParquetFile(
    RNG_MANIFEST_PATH
)


rng_readback_rows = int(
    rng_parquet.metadata.num_rows
)


rng_readback_row_groups = int(
    rng_parquet.metadata.num_row_groups
)


rng_row_group_row_counts = [
    int(
        rng_parquet.metadata.row_group(
            row_group_index
        ).num_rows
    )
    for row_group_index in range(
        rng_readback_row_groups
    )
]


rng_schema_names = set(
    rng_parquet.schema.names
)


rng_schema_valid = {
    "RepetitionSeed",
    "RawTrainingRowOrder",
    "FlipUniform",
    "SampledFailureSubtype",
}.issubset(
    rng_schema_names
)


rng_row_group_sizes_valid = bool(
    rng_readback_row_groups
    == EXPECTED_RNG_ROW_GROUPS
    and all(
        rows
        == EXPECTED_RAW_TRAIN_ROWS
        for rows in rng_row_group_row_counts
    )
)


# Read only the first and last row group to validate seed and row-order boundaries
# without reloading 121M rows.
boundary_readback_failures = 0


for row_group_index, expected_seed in [
    (
        0,
        REPETITION_SEEDS[
            0
        ],
    ),
    (
        rng_readback_row_groups
        - 1,
        REPETITION_SEEDS[
            -1
        ],
    ),
]:
    table = rng_parquet.read_row_group(
        row_group_index,
        columns=[
            "RepetitionSeed",
            "RawTrainingRowOrder",
        ],
    )


    frame = table.to_pandas()


    boundary_readback_failures += int(
        len(
            frame
        )
        != EXPECTED_RAW_TRAIN_ROWS
    )


    boundary_readback_failures += int(
        frame[
            "RepetitionSeed"
        ].nunique()
        != 1
    )


    boundary_readback_failures += int(
        int(
            frame[
                "RepetitionSeed"
            ].iloc[
                0
            ]
        )
        != int(
            expected_seed
        )
    )


    boundary_readback_failures += int(
        int(
            frame[
                "RawTrainingRowOrder"
            ].iloc[
                0
            ]
        )
        != 1
    )


    boundary_readback_failures += int(
        int(
            frame[
                "RawTrainingRowOrder"
            ].iloc[
                -1
            ]
        )
        != EXPECTED_RAW_TRAIN_ROWS
    )


    del frame
    del table
    gc.collect()


seed_manifest_readback = pd.read_csv(
    SEED_MANIFEST_PATH,
    low_memory=False,
)


condition_plan_readback = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


nested_mask_readback = pd.read_csv(
    NESTED_MASK_AUDIT_PATH,
    low_memory=False,
)


nested_mask_readback_violations = int(
    nested_mask_readback[
        "Violations"
    ].sum()
)


readback_checks = []


add_check(
    readback_checks,
    "RNG manifest rows",
    EXPECTED_RNG_ROWS,
    rng_readback_rows,
    rng_readback_rows
    == EXPECTED_RNG_ROWS,
)


add_check(
    readback_checks,
    "RNG manifest row groups",
    EXPECTED_RNG_ROW_GROUPS,
    rng_readback_row_groups,
    rng_readback_row_groups
    == EXPECTED_RNG_ROW_GROUPS,
)


add_check(
    readback_checks,
    "RNG row-group sizes",
    True,
    rng_row_group_sizes_valid,
    rng_row_group_sizes_valid,
)


add_check(
    readback_checks,
    "RNG schema valid",
    True,
    rng_schema_valid,
    rng_schema_valid,
)


add_check(
    readback_checks,
    "RNG boundary readback failures",
    0,
    boundary_readback_failures,
    boundary_readback_failures
    == 0,
)


add_check(
    readback_checks,
    "Condition-plan readback rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan_readback
    ),
    len(
        condition_plan_readback
    )
    == EXPECTED_CONDITIONS,
)


add_check(
    readback_checks,
    "Seed-manifest readback rows",
    len(
        REPETITION_SEEDS
    ),
    len(
        seed_manifest_readback
    ),
    len(
        seed_manifest_readback
    )
    == len(
        REPETITION_SEEDS
    ),
)


add_check(
    readback_checks,
    "Nested-mask readback violations",
    0,
    nested_mask_readback_violations,
    nested_mask_readback_violations
    == 0,
)


readback_validation = pd.DataFrame(
    readback_checks
)


failed_readback = readback_validation.loc[
    ~readback_validation[
        "Pass"
    ]
]


print(
    "\nProject 24 Step 3A RNG/readback validation:"
)


display(
    readback_validation
)


if not failed_readback.empty:
    raise RuntimeError(
        "PROJECT 24 STEP 3A RNG/READBACK VALIDATION FAILED."
    )


# Combine pre-write and readback checks into the frozen validation table.
validation = pd.concat(
    [
        validation,
        readback_validation,
    ],
    ignore_index=True,
)


atomic_csv(
    STEP3A_VALIDATION_PATH,
    validation,
)


# --------------------------------------------------------------------------------------------------
# 16. REPORT, CHECKPOINT, AND STATUS
# --------------------------------------------------------------------------------------------------

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


output_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    FAILURE_SUBTYPE_PROFILE_PATH,
    SEED_MANIFEST_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NESTED_MASK_AUDIT_PATH,
    PROTOCOL_PATH,
    STEP3A_VALIDATION_PATH,
]


print(
    "\nHashing frozen Step 3A outputs for the output manifest."
)


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "Step3ACodeRevision":
        STEP3A_CODE_REVISION,

    "CompletedAtUTC":
        completed_at_utc,

    "SelectionCheckpointSHA256":
        selection_sha256,

    "RECCheckpointSHA256":
        rec_checkpoint_sha256,

    "RECImplementationVersion":
        EXPECTED_REC_IMPLEMENTATION,

    "SourceRootSHA256":
        current_source_root_sha256,

    "CleanAnchorReproducedDataset":
        clean_anchor_reproduced_dataset,

    "CleanAnchorMismatchValues":
        clean_anchor_mismatch_values,

    "FrozenInferredExecutionOrder":
        str(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenInferredExecutionOrderSHA256":
        sha256_file(
            INFERRED_EXECUTION_ORDER_PATH
        ),

    "FrozenGlobalBuildOrder":
        str(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "FrozenGlobalBuildOrderSHA256":
        sha256_file(
            FROZEN_GLOBAL_BUILD_ORDER_PATH
        ),

    "RawHistoryOrdering":
        (
            "Project 24 Step 2B scalable exact-tie inferred order "
            "with frozen baseline used for all 2,020 tied tests"
        ),

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "RawTrainingFailures":
        raw_training_failures,

    "RawEvaluationFailures":
        raw_evaluation_failures,

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "FailureSubtypes":
        failure_subtypes.astype(
            int
        ).tolist(),

    "FailureSubtypeProbabilities":
        failure_subtype_probabilities.tolist(),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "RNGManifestRowGroups":
        rng_readback_row_groups,

    "RNGManifestStorageMode":
        "30 streamed Parquet row groups; one seed held in memory",

    "RNGGenerationSeconds":
        rng_generation_seconds,

    "NestedMaskAuditRows":
        len(
            nested_mask_audit
        ),

    "NestedMaskViolations":
        nested_mask_violations,

    "ZeroNoiseConditions":
        len(
            zero_noise_conditions
        ),

    "ZeroNoiseFlipViolations":
        zero_noise_flip_violations,

    "ZeroNoiseRawLabelViolations":
        zero_noise_raw_label_violations,

    "ZeroNoiseModelLabelViolations":
        zero_noise_model_label_violations,

    "PositiveNoiseConditionsWithoutRawChanges":
        positive_noise_without_raw_changes,

    "PositiveNoiseConditionsWithoutModelChanges":
        positive_noise_without_model_changes,

    "PredictorColumns":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "VerdictDependentRECFeatures":
        VERDICT_DEPENDENT_REC,

    "VerdictIndependentRECFeatures":
        VERDICT_INDEPENDENT_REC,

    "MLTechniques":
        ML_TECHNIQUES,

    "Baselines":
        BASELINES,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "OutputManifest":
        output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        int(
            (
                ~validation[
                    "Pass"
                ]
            ).sum()
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RegistryModified":
        False,

    "Projects1To23Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ModelsTrained":
        False,

    "Project24ExperimentStarted":
        False,
}


atomic_json(
    STEP3A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_24_DETERMINISTIC_NOISE_PLAN_AND_COHORTS",

    "NoisePlanCheckpoint":
        True,

    "DoNotChangeCohorts":
        True,

    "DoNotChangeRandomStreams":
        True,

    "DoNotChangeConditionCoordinates":
        True,

    "EvaluationCohortImmutable":
        True,

    "ProceedToRuntimeContractAllowed":
        True,
}


atomic_json(
    NOISE_PLAN_CHECKPOINT_PATH,
    checkpoint_payload,
)


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP3A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "RawTrainingRows":
        len(
            raw_training
        ),

    "RawEvaluationRows":
        len(
            raw_evaluation
        ),

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "Conditions":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        rng_readback_rows,

    "RNGManifestRowGroups":
        rng_readback_row_groups,

    "NestedMaskViolations":
        nested_mask_violations,

    "Checkpoint":
        str(
            NOISE_PLAN_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        noise_checkpoint_sha256,

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "PriorProjectConditionOutputsAccessed":
        False,

    "ModelsTrained":
        False,

    "Project24ExperimentStarted":
        False,

    "NextRequiredStep":
        "PROJECT 24 CELL 7 / STEP 4A — EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE",
}


atomic_json(
    STEP3A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL IMMUTABILITY AND MANIFEST READBACK
# --------------------------------------------------------------------------------------------------

registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if (
    registry_sha256_after
    != registry_sha256_before
):
    raise RuntimeError(
        "Completion registry changed during Project 24 Step 3A."
    )


if sha256_file(
    REC_CHECKPOINT_PATH
) != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Project 24 REC checkpoint changed during Step 3A."
    )


if sha256_file(
    SELECTION_CHECKPOINT_PATH
) != EXPECTED_SELECTION_SHA256:
    raise RuntimeError(
        "Project 24 selection checkpoint changed during Step 3A."
    )


final_source_manifest = pd.DataFrame([
    {
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                (
                    SOURCE_DIR
                    / str(
                        row.RelativePath
                    )
                ).stat().st_size
            ),

        "SHA256":
            sha256_file(
                SOURCE_DIR
                / str(
                    row.RelativePath
                )
            ),
    }
    for row in frozen_source_manifest.itertuples(
        index=False
    )
])


if (
    source_root_hash(
        final_source_manifest
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 24 source changed during Step 3A."
    )


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        raise RuntimeError(
            f"Project 24 Step 3A output readback failed: {path}"
        )


checkpoint_readback = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

status_readback = load_json(
    STEP3A_STATUS_PATH
)

report_readback = load_json(
    STEP3A_REPORT_PATH
)


for label, payload in [
    (
        "checkpoint",
        checkpoint_readback,
    ),
    (
        "status",
        status_readback,
    ),
    (
        "report",
        report_readback,
    ),
]:
    if (
        payload.get(
            "Status"
        )
        != STEP3A_STATUS
    ):
        raise RuntimeError(
            f"Project 24 Step 3A {label} readback is not PASS."
        )


# --------------------------------------------------------------------------------------------------
# 18. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nFailure-subtype profile:"
)


display(
    failure_subtype_profile
)


print(
    "\nSeed manifest:"
)


display(
    seed_manifest
)


print(
    "\nCondition-plan sample:"
)


display(
    pd.concat(
        [
            condition_plan.head(
                9
            ),
            condition_plan.tail(
                9
            ),
        ],
        ignore_index=True,
    )
)


print(
    "\nNested-mask audit summary:"
)


display(
    nested_mask_audit.groupby(
        [
            "LowerNoisePercent",
            "HigherNoisePercent",
        ],
        as_index=False,
    ).agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        TotalViolations=(
            "Violations",
            "sum",
        ),

        AllPassed=(
            "Pass",
            "all",
        ),
    )
)


# --------------------------------------------------------------------------------------------------
# 19. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 140
)

print(
    "=== PROJECT 24 CELL 6 / STEP 3A RESULT ==="
)

print(
    "=" * 140
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Step 3A code revision:",
    STEP3A_CODE_REVISION,
)

print(
    "REC checkpoint SHA-256:",
    rec_checkpoint_sha256,
)

print(
    "REC implementation:",
    EXPECTED_REC_IMPLEMENTATION,
)

print()

print(
    "Fixed cohorts:"
)

print(
    "Raw training rows:",
    len(
        raw_training
    ),
)

print(
    "Raw evaluation rows:",
    len(
        raw_evaluation
    ),
)

print(
    "Raw training failures:",
    raw_training_failures,
)

print(
    "Raw evaluation failures:",
    raw_evaluation_failures,
)

print(
    "Model training rows:",
    len(
        model_training
    ),
)

print(
    "Model evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Model training failures:",
    model_training_failures,
)

print(
    "Model evaluation failures:",
    model_evaluation_failures,
)

print(
    "Model failing evaluation builds:",
    model_failing_evaluation_builds,
)

print()

print(
    "Noise plan:"
)

print(
    "RNG manifest storage mode:",
    "30 streamed Parquet row groups; one seed held in memory",
)

print(
    "Noise levels:",
    NOISE_LEVELS,
)

print(
    "Repetition seeds:",
    len(
        REPETITION_SEEDS
    ),
)

print(
    "Conditions:",
    len(
        condition_plan
    ),
)

print(
    "RNG-manifest rows:",
    rng_readback_rows,
)

print(
    "RNG-manifest row groups:",
    rng_readback_row_groups,
)

print(
    "Failure subtypes:",
    failure_subtypes.astype(
        int
    ).tolist(),
)

print(
    "Failure-subtype probabilities:",
    failure_subtype_probabilities.tolist(),
)

print(
    "Nested-mask violations:",
    nested_mask_violations,
)

print()

print(
    "Zero-noise audit:"
)

print(
    "Zero-noise conditions:",
    len(
        zero_noise_conditions
    ),
)

print(
    "Zero-noise flip violations:",
    zero_noise_flip_violations,
)

print(
    "Zero-noise raw-label violations:",
    zero_noise_raw_label_violations,
)

print(
    "Zero-noise model-label violations:",
    zero_noise_model_label_violations,
)

print()

print(
    "Runtime:"
)

print(
    "RNG/condition-plan generation seconds:",
    round(
        rng_generation_seconds,
        2,
    ),
)

print()

print(
    "Immutability and isolation:"
)

print(
    "Project 24 source unchanged:",
    source_root_hash(
        final_source_manifest
    )
    == EXPECTED_SOURCE_ROOT_SHA256,
)

print(
    "REC checkpoint unchanged:",
    sha256_file(
        REC_CHECKPOINT_PATH
    )
    == EXPECTED_REC_CHECKPOINT_SHA256,
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–23 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Models trained:",
    False,
)

print(
    "Project 24 experiment started:",
    False,
)

print()

print(
    "Validation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    int(
        (
            ~validation[
                "Pass"
            ]
        ).sum()
    ),
)

print()

print(
    "Noise-plan checkpoint:",
    NOISE_PLAN_CHECKPOINT_PATH,
)

print(
    "Checkpoint SHA-256:",
    noise_checkpoint_sha256,
)

print()

print(
    "Next required step:",
    "PROJECT 24 CELL 7 / STEP 4A — EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE",
)

print(
    "STATUS:",
    STEP3A_STATUS,
)

print(
    "=" * 140
)


=== PROJECT 24 CELL 6 / STEP 3A: DETERMINISTIC NOISE PLAN AND COHORT FREEZE ===

Loading full Project 24 model-ready dataset (224,550 rows).
Loading frozen Project 24 raw execution order (5,635,027 rows).
    RNG/condition-plan progress: 1 / 30 seeds | this seed seconds: 4.62 | total seconds: 4.62
    RNG/condition-plan progress: 2 / 30 seeds | this seed seconds: 2.97 | total seconds: 7.59
    RNG/condition-plan progress: 3 / 30 seeds | this seed seconds: 2.52 | total seconds: 10.11
    RNG/condition-plan progress: 4 / 30 seeds | this seed seconds: 2.4 | total seconds: 12.51
    RNG/condition-plan progress: 5 / 30 seeds | this seed seconds: 2.49 | total seconds: 15.0
    RNG/condition-plan progress: 6 / 30 seeds | this seed seconds: 3.26 | total seconds: 18.26
    RNG/condition-plan progress: 7 / 30 seeds | this seed seconds: 3.48 | total seconds: 21.74
    RNG/condition-plan progress: 8 / 30 seeds | this seed seconds: 2.4 | total seconds: 24.14
    RNG/condition-plan progress: 9 / 30 

,Check,Expected,Actual,Pass
0,Selection checkpoint SHA-256,d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f167...,d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f167...,True
1,REC checkpoint SHA-256,80ac2e6986ae2c14d636c57c834cc47678b54cd05ba323...,80ac2e6986ae2c14d636c57c834cc47678b54cd05ba323...,True
2,REC implementation,PROJECT_24_V1_SCALABLE_EXACT_TIE_DP_BINARY_AGE...,PROJECT_24_V1_SCALABLE_EXACT_TIE_DP_BINARY_AGE...,True
3,Source root SHA-256,2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21...,2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21...,True
4,Source files,6,6,True
5,Source bytes,374503252,374503252,True
6,Clean anchor mismatch values,0,0,True
7,Clean anchor reproduced source dataset,True,True,True
8,Raw history rows,5635027,5635027,True
9,Raw duplicate Build-Test rows,0,0,True



Writing fixed Project 24 cohorts.

Validating streamed RNG-manifest metadata.

Project 24 Step 3A RNG/readback validation:


,Check,Expected,Actual,Pass
0,RNG manifest rows,121224030,121224030,True
1,RNG manifest row groups,30,30,True
2,RNG row-group sizes,True,True,True
3,RNG schema valid,True,True,True
4,RNG boundary readback failures,0,0,True
5,Condition-plan readback rows,270,270,True
6,Seed-manifest readback rows,30,30,True
7,Nested-mask readback violations,0,0,True



Hashing frozen Step 3A outputs for the output manifest.

Failure-subtype profile:


...


Condition-plan sample:


,ConditionOrder,ConditionID,SeedOrder,NoiseOrderWithinSeed,NoisePercent,RepetitionSeed,FlipSeed,FailureSubtypeSeed,RawTrainingRows,NumberFlipped,...,FailureToPass,CleanRawFailures,NoisyRawFailures,ModelTrainingRows,ModelLabelChanges,CleanModelFailures,NoisyModelFailures,FlipMaskSHA256,NoisyRawVerdictSHA256,NoisyModelVerdictSHA256
0,1,noise_00__seed_01,1,1,0,1,3187222518,2608877662,4040801,0,...,0,1778,1778,205696,0,1777,1777,fd73905dc86d87e5b2fc2a4499863e4b467f200d24c3bc...,70bb164dd29769292f832d19e534108cc97254fef59ccb...,321d0a0f26d6ee4024602e1ebb9fa87eb9164e52181d45...
1,2,noise_05__seed_01,1,2,5,1,3187222518,2608877662,4040801,202186,...,83,1778,203798,205696,10244,1777,11855,d445acbea1f8399315b910d381ec3a9739346db62baba8...,ca44710fc9707bc4680cfff0723c5d3d6db531afa19ca9...,4032edae9f5c477e3e1618fd84278b138565b8dc430f0d...
2,3,noise_10__seed_01,1,3,10,1,3187222518,2608877662,4040801,403838,...,183,1778,405250,205696,20402,1777,21813,0e786720b47ecab5933f12a1c9fd396fd149afe8f7ca76...,5f094fa78bc5577fca16b24a6fd752523d24793b51c452...,b04af55612a9e2ba5c8adce911fd74d31cc1a1454f4cc9...
3,4,noise_15__seed_01,1,4,15,1,3187222518,2608877662,4040801,606624,...,270,1778,607862,205696,30563,1777,31800,c8d0c12ad6cf6c74148f28b970fecf89f44f772028d376...,18a6c8747a32dde446c5be84a4d4be213a74d8b6edbc4f...,72ae8132cdee397239bf860f80293d319154f8084c5bb7...
4,5,noise_20__seed_01,1,5,20,1,3187222518,2608877662,4040801,808792,...,368,1778,809834,205696,40904,1777,41945,6c6cde58babd646f17d3100f4f5ac3fd3566003da08ab6...,9390ca5339cc3f77d56524e205fae11711afc2d0140038...,74a5f99bb182f4c0396820122861d10570e4b7498c6ecc...
5,6,noise_25__seed_01,1,6,25,1,3187222518,2608877662,4040801,1010511,...,454,1778,1011381,205696,51270,1777,52139,6e4506dbaf73b91e10c7c6245e4476a1b5305468fe7675...,8eee159529b11d729918695eae29290ea0f1f5fd4c141a...,04ed951662e3dae02c3ad62d1e714a0002ec7e2dbeee05...
6,7,noise_30__seed_01,1,7,30,1,3187222518,2608877662,4040801,1213364,...,531,1778,1214080,205696,61814,1777,62529,0e07ee662ead1f85f80acbd91070a2fba27c37eb368a63...,78ccf5b12f7e46a2a8eb742046c448d4180acee73a7eae...,1be4661e89082279592153226976dc775d0cd25c253863...
7,8,noise_40__seed_01,1,8,40,1,3187222518,2608877662,4040801,1618324,...,717,1778,1618668,205696,82412,1777,82757,314648fca737ed47ba1992836993e743a521def1b9b4fb...,4ce716fa80e96647dcd2808664eeb865a43ed5f11ca9ff...,a68154bf7d3f68da8e769886d41654eae3740958cec0a6...
8,9,noise_50__seed_01,1,9,50,1,3187222518,2608877662,4040801,2021337,...,905,1778,2021305,205696,103102,1777,103071,6d34772dd7fbf1802e781e2546d74560f1e08ff4b21f10...,553e52826d0a7b5ad0baa90411c3735ef5adc746a52f36...,aba3ccf245bc14fee7d400060022179132effc205c1524...
9,262,noise_00__seed_30,30,1,0,30,3299740266,451627945,4040801,0,...,0,1778,1778,205696,0,1777,1777,fd73905dc86d87e5b2fc2a4499863e4b467f200d24c3bc...,70bb164dd29769292f832d19e534108cc97254fef59ccb...,321d0a0f26d6ee4024602e1ebb9fa87eb9164e52181d45...



Nested-mask audit summary:


,LowerNoisePercent,HigherNoisePercent,Seeds,TotalViolations,AllPassed
0,0,5,30,0,True
1,5,10,30,0,True
2,10,15,30,0,True
3,15,20,30,0,True
4,20,25,30,0,True
5,25,30,30,0,True
6,30,40,30,0,True
7,40,50,30,0,True



=== PROJECT 24 CELL 6 / STEP 3A RESULT ===
Project: SonarSource@sonarqube
Project slug: SonarSource__sonarqube
Step 3A code revision: PROJECT_24_STEP3A_V2_FROZEN_GLOBAL_BUILD_ORDER_SCHEMA_FIX
REC checkpoint SHA-256: 80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803
REC implementation: PROJECT_24_V1_SCALABLE_EXACT_TIE_DP_BINARY_AGE_CONSTRAINT_ANCHOR_AWARE_MAPPING_BOUNDARY_AUDIT

Fixed cohorts:
Raw training rows: 4040801
Raw evaluation rows: 1594226
Raw training failures: 1778
Raw evaluation failures: 20
Model training rows: 205696
Model evaluation rows: 18854
Model training failures: 1777
Model evaluation failures: 20
Model failing evaluation builds: 17

Noise plan:
RNG manifest storage mode: 30 streamed Parquet row groups; one seed held in memory
Noise levels: [0, 5, 10, 15, 20, 25, 30, 40, 50]
Repetition seeds: 30
Conditions: 270
RNG-manifest rows: 121224030
RNG-manifest row groups: 30
Failure subtypes: [1, 2]
Failure-subtype probabilities: [0.26321709786276715, 0.7367

In [8]:
# ==================================================================================================
# PROJECT 24 — CELL 7 / STEP 4A
# EXPERIMENT RUNTIME, MODEL, BASELINE, METRIC, RANKING, AND PREDICTOR CONTRACT FREEZE
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_24.ipynb.
#
# PURPOSE:
# - verify the corrected/frozen Project 24 Step 3A noise-plan checkpoint and every frozen output;
# - validate the fixed 151-predictor training/evaluation matrices;
# - freeze runtime versions, predictor order, clean training-median reference, label semantics,
#   model hyperparameters, deterministic model/random-baseline seed rules, baseline definitions,
#   ranking rules, APFD/APFDc metric definitions, and the no-rolling-retraining contract;
# - instantiate all four required ML implementations WITHOUT fitting any Project 24 model;
# - write the runtime-contract checkpoint required before the two-condition smoke test.
#
# SAFETY:
# - no model fitting;
# - no noise regeneration;
# - no condition execution;
# - no completion-registry write;
# - no prior-project condition-output access;
# - no full-experiment raw-result write.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from IPython.display import display

import hashlib
import json
import os
import platform

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


print("=" * 140)
print("=== PROJECT 24 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===")
print("=" * 140)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

EXPECTED_STEP3A_STATUS = (
    "PASS_PROJECT_24_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
)

EXPECTED_STEP3A_CODE_REVISION = (
    "PROJECT_24_STEP3A_V2_FROZEN_GLOBAL_BUILD_ORDER_SCHEMA_FIX"
)

STEP4A_STATUS = (
    "PASS_PROJECT_24_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
)

EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "9dba84682eabdc1393219671e77aabaf1f2d4385f29239236e50615bf0180515"
)

EXPECTED_REC_CHECKPOINT_SHA256 = (
    "80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)

EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 23
EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_RAW_TRAIN_ROWS = 4_040_801
EXPECTED_RAW_EVAL_ROWS = 1_594_226

EXPECTED_MODEL_TRAIN_ROWS = 205_696
EXPECTED_MODEL_EVAL_ROWS = 18_854
EXPECTED_MODEL_TRAIN_FAILURES = 1_777
EXPECTED_MODEL_EVAL_FAILURES = 20
EXPECTED_MODEL_FAILING_EVAL_BUILDS = 17

EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_CONDITIONS = 270
EXPECTED_RNG_MANIFEST_ROWS = 121_224_030
EXPECTED_RNG_ROW_GROUPS = 30

EXPECTED_RUNTIME_VERSIONS = {
    "Python": "3.12.13",
    "numpy": "2.0.2",
    "pandas": "2.2.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.3.0",
    "lightgbm": "4.6.0",
    "pyarrow": "18.1.0",
}

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]

BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]

ALL_TECHNIQUES = (
    ML_TECHNIQUES
    + BASELINE_TECHNIQUES
)

REC_FEATURES = [
    "REC_Age",
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_LastExeTime",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_DEPENDENT_REC = [
    "REC_LastFailureAge",
    "REC_LastTransitionAge",
    "REC_RecentFailRate",
    "REC_RecentAssertRate",
    "REC_RecentExcRate",
    "REC_RecentTransitionRate",
    "REC_TotalFailRate",
    "REC_TotalAssertRate",
    "REC_TotalExcRate",
    "REC_TotalTransitionRate",
    "REC_LastVerdict",
    "REC_MaxTestFileFailRate",
    "REC_MaxTestFileTransitionRate",
]

VERDICT_INDEPENDENT_REC = [
    "REC_Age",
    "REC_RecentAvgExeTime",
    "REC_RecentMaxExeTime",
    "REC_TotalAvgExeTime",
    "REC_TotalMaxExeTime",
    "REC_LastExeTime",
]

MODEL_CONFIG = {
    "RandomForest": {
        "n_estimators": 100,
        "max_features": "sqrt",
        "bootstrap": True,
        "n_jobs": -1,
    },

    "XGBoost": {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist",
        "n_jobs": -1,
        "verbosity": 0,
        "eval_metric": "logloss",
    },

    "LightGBM": {
        "n_estimators": 100,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    },

    "NaiveBayes": {
        "var_smoothing": 1e-9,
    },
}

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES_ROOT = (
    THESIS_ROOT
    / "Notes"
)

RESULTS_ROOT = (
    THESIS_ROOT
    / "Results"
)

REGISTRY_PATH = (
    NOTES_ROOT
    / "completed_project_registry.csv"
)

SELECTION_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / "project_24_selection"
)

FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT
    / "project_24_frozen_source_manifest.csv"
)

SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_selection_checkpoint.json"
)

SOURCE_DIR = Path(
    "/content/datasets/datasets/SonarSource@sonarqube"
)

PROJECT_ROOT = (
    RESULTS_ROOT
    / "Aggregated"
    / PROJECT_SLUG
)

REC_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_rec_preflight"
)

REC_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
)

STEP3A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step3a_status.json"
)

STEP3A_REPORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_report.json"
)

NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_noise_plan_checkpoint.json"
)

RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)

RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)

MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)

MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)

MODEL_RAW_TRAIN_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"
)

MODEL_RAW_EVAL_LINK_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
)

FAILURE_SUBTYPE_PROFILE_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_failure_subtype_profile.csv"
)

SEED_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_seed_manifest.csv"
)

RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_rng_manifest.parquet"
)

CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

NESTED_MASK_AUDIT_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_nested_mask_audit.csv"
)

PROTOCOL_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_frozen_experiment_protocol.json"
)

STEP3A_VALIDATION_PATH = (
    NOISE_PLAN_ROOT
    / f"{PROJECT_SHORT}_step3a_validation.csv"
)

RUNTIME_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_runtime_contract"
)

RUNTIME_VERSION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_runtime_versions.csv"
)

PREDICTOR_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_predictor_contract.csv"
)

CLEAN_MEDIAN_REFERENCE_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_clean_training_median_reference.csv"
)

MODEL_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_model_contract.json"
)

BASELINE_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_baseline_contract.json"
)

METRIC_SELF_TEST_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_metric_self_test.csv"
)

RANKING_CONTRACT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_ranking_contract.json"
)

STEP4A_VALIDATION_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_validation.csv"
)

STEP4A_REPORT_PATH = (
    RUNTIME_ROOT
    / f"{PROJECT_SHORT}_step4a_report.json"
)

STEP4A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step4a_status.json"
)

RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT
    / "project_24_runtime_contract_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write(
            "\n"
        )

    os.replace(
        temporary_path,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def source_root_hash(
    frame,
):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.SizeBytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def resolve_column(
    columns,
    expected,
    label,
):
    matches = [
        column
        for column in columns
        if str(
            column
        ).strip().lower()
        == str(
            expected
        ).strip().lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Could not resolve {label}. "
            f"Expected={expected!r}; "
            f"matches={matches}; "
            f"columns={list(columns)}"
        )

    return matches[
        0
    ]


def deterministic_seed(
    repetition_seed,
    stream_name,
):
    material = (
        f"{PROJECT_NAME}|"
        f"{int(repetition_seed)}|"
        f"{stream_name}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        material
    ).digest()

    return int.from_bytes(
        digest[
            :4
        ],
        byteorder="little",
        signed=False,
    )


def deterministic_random_build_seed(
    repetition_seed,
    build_id,
):
    return deterministic_seed(
        repetition_seed,
        f"Random_baseline_build_{int(build_id)}",
    )


def create_models(
    repetition_seed,
):
    return {
        "RandomForest":
            RandomForestClassifier(
                **MODEL_CONFIG[
                    "RandomForest"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "RandomForest_model",
                ),
            ),

        "XGBoost":
            XGBClassifier(
                **MODEL_CONFIG[
                    "XGBoost"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "XGBoost_model",
                ),
            ),

        "LightGBM":
            LGBMClassifier(
                **MODEL_CONFIG[
                    "LightGBM"
                ],
                random_state=deterministic_seed(
                    repetition_seed,
                    "LightGBM_model",
                ),
            ),

        "NaiveBayes":
            GaussianNB(
                **MODEL_CONFIG[
                    "NaiveBayes"
                ]
            ),
    }


def calculate_apfd(
    failures,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    n = len(
        failures
    )

    m = int(
        failures.sum()
    )

    if (
        n == 0
        or m == 0
    ):
        return np.nan

    positions = (
        np.flatnonzero(
            failures
            == 1
        )
        + 1
    )

    return float(
        1.0
        - positions.sum()
        / (
            n
            * m
        )
        + 1.0
        / (
            2.0
            * n
        )
    )


def calculate_apfdc(
    failures,
    durations,
):
    failures = np.asarray(
        failures,
        dtype=np.int8,
    )

    durations = np.asarray(
        durations,
        dtype=float,
    )

    if len(
        failures
    ) != len(
        durations
    ):
        raise ValueError(
            "failures and durations must have equal length."
        )

    if (
        len(
            failures
        )
        == 0
        or failures.sum()
        == 0
    ):
        return np.nan

    if not np.isfinite(
        durations
    ).all():
        raise ValueError(
            "Durations contain missing/infinite values."
        )

    if (
        durations
        < 0
    ).any():
        raise ValueError(
            "Durations cannot be negative."
        )

    total_duration = float(
        durations.sum()
    )

    if total_duration <= 0:
        return np.nan

    cumulative_before = np.concatenate([
        np.array([
            0.0,
        ]),
        np.cumsum(
            durations
        )[
            :-1
        ],
    ])

    failure_mask = (
        failures
        == 1
    )

    midpoint_detection_times = (
        cumulative_before[
            failure_mask
        ]
        + 0.5
        * durations[
            failure_mask
        ]
    )

    return float(
        1.0
        - np.mean(
            midpoint_detection_times
            / total_duration
        )
    )


def add_check(
    rows,
    check,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            check,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def canonical_parameter_value(
    value,
):
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    return value


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS + STEP 3A CHECKPOINT
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    STEP3A_STATUS_PATH,
    STEP3A_REPORT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    MODEL_RAW_TRAIN_LINK_PATH,
    MODEL_RAW_EVAL_LINK_PATH,
    FAILURE_SUBTYPE_PROFILE_PATH,
    SEED_MANIFEST_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    NESTED_MASK_AUDIT_PATH,
    PROTOCOL_PATH,
    STEP3A_VALIDATION_PATH,
]

missing_paths = [
    str(
        path
    )
    for path in required_paths
    if not Path(
        path
    ).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Project 24 Step 4A inputs are missing:\n"
        + "\n".join(
            missing_paths
        )
    )


noise_checkpoint_sha256 = sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
)

rec_checkpoint_sha256 = sha256_file(
    REC_CHECKPOINT_PATH
)

selection_checkpoint_sha256 = sha256_file(
    SELECTION_CHECKPOINT_PATH
)


if (
    noise_checkpoint_sha256
    != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 24 noise-plan checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256}\n"
        f"Actual:   {noise_checkpoint_sha256}"
    )


if (
    rec_checkpoint_sha256
    != EXPECTED_REC_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 24 REC checkpoint SHA-256 differs."
    )


if (
    selection_checkpoint_sha256
    != EXPECTED_SELECTION_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Project 24 selection checkpoint SHA-256 differs."
    )


noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)

step3a_status = load_json(
    STEP3A_STATUS_PATH
)

step3a_report = load_json(
    STEP3A_REPORT_PATH
)

protocol = load_json(
    PROTOCOL_PATH
)


for label, payload in [
    (
        "noise checkpoint",
        noise_checkpoint,
    ),
    (
        "Step 3A status",
        step3a_status,
    ),
    (
        "Step 3A report",
        step3a_report,
    ),
]:
    if payload.get(
        "Status"
    ) != EXPECTED_STEP3A_STATUS:
        raise RuntimeError(
            f"{label} does not contain the frozen Project 24 Step 3A PASS status."
        )


if (
    noise_checkpoint.get(
        "Project"
    )
    != PROJECT_NAME
    or noise_checkpoint.get(
        "ProjectSlug"
    )
    != PROJECT_SLUG
):
    raise RuntimeError(
        "Frozen Step 3A Project 24 identity differs."
    )


if (
    noise_checkpoint.get(
        "Step3ACodeRevision"
    )
    != EXPECTED_STEP3A_CODE_REVISION
):
    raise RuntimeError(
        "Project 24 Step 3A code revision differs.\n"
        f"Expected: {EXPECTED_STEP3A_CODE_REVISION}\n"
        f"Actual:   {noise_checkpoint.get('Step3ACodeRevision')}"
    )


if (
    noise_checkpoint.get(
        "SourceRootSHA256"
    )
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Step 3A source root differs."
    )


if (
    noise_checkpoint.get(
        "RECCheckpointSHA256"
    )
    != EXPECTED_REC_CHECKPOINT_SHA256
):
    raise RuntimeError(
        "Frozen Step 3A REC checkpoint anchor differs."
    )


if (
    noise_checkpoint.get(
        "ActiveReservations"
    )
    != EXPECTED_ACTIVE_RESERVATIONS
):
    raise RuntimeError(
        "Frozen Step 3A active-reservation state differs."
    )


if (
    noise_checkpoint.get(
        "RuntimePriorityRule"
    )
    != EXPECTED_RUNTIME_PRIORITY_RULE
):
    raise RuntimeError(
        "Frozen Step 3A runtime-priority ranking rule differs."
    )


if not bool(
    noise_checkpoint.get(
        "NoisePlanCheckpoint",
        False,
    )
):
    raise RuntimeError(
        "Project 24 noise-plan checkpoint is not marked frozen."
    )


if not bool(
    noise_checkpoint.get(
        "ProceedToRuntimeContractAllowed",
        False,
    )
):
    raise RuntimeError(
        "Project 24 noise-plan checkpoint does not authorize Step 4A."
    )


if not bool(
    noise_checkpoint.get(
        "EvaluationCohortImmutable",
        False,
    )
):
    raise RuntimeError(
        "Project 24 evaluation cohort is not frozen immutable."
    )


if int(
    noise_checkpoint.get(
        "Conditions",
        -1,
    )
) != EXPECTED_CONDITIONS:
    raise RuntimeError(
        "Frozen Project 24 condition count differs."
    )


if int(
    noise_checkpoint.get(
        "RNGManifestRows",
        -1,
    )
) != EXPECTED_RNG_MANIFEST_ROWS:
    raise RuntimeError(
        "Frozen Project 24 RNG-manifest row count differs."
    )


if int(
    noise_checkpoint.get(
        "RNGManifestRowGroups",
        -1,
    )
) != EXPECTED_RNG_ROW_GROUPS:
    raise RuntimeError(
        "Frozen Project 24 RNG row-group count differs."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY THE COMPLETE STEP 3A OUTPUT MANIFEST
# --------------------------------------------------------------------------------------------------

step3a_output_manifest = noise_checkpoint.get(
    "OutputManifest",
    [],
)


if (
    not isinstance(
        step3a_output_manifest,
        list,
    )
    or not step3a_output_manifest
):
    raise RuntimeError(
        "Project 24 Step 3A checkpoint has no output manifest."
    )


output_manifest_records = []


for item in step3a_output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    expected_bytes = int(
        item[
            "Bytes"
        ]
    )

    expected_sha256 = str(
        item[
            "SHA256"
        ]
    )

    actual_exists = path.is_file()

    actual_bytes = (
        int(
            path.stat().st_size
        )
        if actual_exists
        else -1
    )

    actual_sha256 = (
        sha256_file(
            path
        )
        if actual_exists
        else ""
    )

    passed = bool(
        actual_exists
        and actual_bytes
        == expected_bytes
        and actual_sha256
        == expected_sha256
    )

    output_manifest_records.append({
        "Path":
            str(
                path
            ),

        "ExpectedBytes":
            expected_bytes,

        "ActualBytes":
            actual_bytes,

        "ExpectedSHA256":
            expected_sha256,

        "ActualSHA256":
            actual_sha256,

        "Pass":
            passed,
    })


output_manifest_audit = pd.DataFrame(
    output_manifest_records
)


output_manifest_failures = int(
    (
        ~output_manifest_audit[
            "Pass"
        ]
    ).sum()
)


if output_manifest_failures:
    print(
        "\nFailed Project 24 Step 3A output-manifest entries:"
    )

    display(
        output_manifest_audit.loc[
            ~output_manifest_audit[
                "Pass"
            ]
        ]
    )

    raise RuntimeError(
        "One or more Project 24 Step 3A outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY SOURCE ROOT + COMPLETION REGISTRY
# --------------------------------------------------------------------------------------------------

registry_sha256_before = sha256_file(
    REGISTRY_PATH
)


if (
    registry_sha256_before
    != EXPECTED_REGISTRY_SHA256
):
    raise RuntimeError(
        "Completion registry SHA-256 differs before Project 24 Step 4A."
    )


registry = (
    pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
    )
    .fillna("")
)


project_number_column = resolve_column(
    registry.columns,
    "ProjectNumber",
    "registry ProjectNumber",
)

project_column = resolve_column(
    registry.columns,
    "Project",
    "registry Project",
)

status_column = resolve_column(
    registry.columns,
    "Status",
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_column
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    )
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS
            + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry must contain exactly frozen Projects 1–23."
    )


if not registry[
    status_column
].eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )


for (
    required_number,
    required_project,
) in REQUIRED_REGISTERED_IDENTITIES.items():
    matching = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching
        )
        != 1
        or str(
            matching.iloc[
                0
            ][
                project_column
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A frozen predecessor identity differs.\n"
            f"Project number: {required_number}\n"
            f"Expected:       {required_project}"
        )


if (
    project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        project_column
    ].eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 24 is unexpectedly already registered."
    )


frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)


current_source_records = []


for row in frozen_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not source_path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 24 source file is missing: {source_path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


current_source_manifest = pd.DataFrame(
    current_source_records
)


current_source_root_sha256 = source_root_hash(
    current_source_manifest
)


if (
    current_source_root_sha256
    != EXPECTED_SOURCE_ROOT_SHA256
):
    raise RuntimeError(
        "Frozen Project 24 source root differs."
    )


# --------------------------------------------------------------------------------------------------
# 7. RUNTIME VERSION FREEZE
# --------------------------------------------------------------------------------------------------

actual_runtime_versions = {
    "Python":
        platform.python_version(),

    "numpy":
        metadata.version(
            "numpy"
        ),

    "pandas":
        metadata.version(
            "pandas"
        ),

    "scikit-learn":
        metadata.version(
            "scikit-learn"
        ),

    "xgboost":
        metadata.version(
            "xgboost"
        ),

    "lightgbm":
        metadata.version(
            "lightgbm"
        ),

    "pyarrow":
        metadata.version(
            "pyarrow"
        ),
}


runtime_versions = pd.DataFrame([
    {
        "Component":
            component,

        "ExpectedVersion":
            EXPECTED_RUNTIME_VERSIONS[
                component
            ],

        "Version":
            actual_runtime_versions[
                component
            ],

        "Pass":
            (
                actual_runtime_versions[
                    component
                ]
                == EXPECTED_RUNTIME_VERSIONS[
                    component
                ]
            ),
    }
    for component in EXPECTED_RUNTIME_VERSIONS
])


runtime_version_failures = int(
    (
        ~runtime_versions[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 8. LOAD FROZEN MODEL COHORTS + LINKS + CONDITION PLAN
# --------------------------------------------------------------------------------------------------

model_training = pd.read_parquet(
    MODEL_TRAINING_COHORT_PATH
)


model_evaluation = pd.read_parquet(
    MODEL_EVALUATION_COHORT_PATH
)


model_train_link = pd.read_parquet(
    MODEL_RAW_TRAIN_LINK_PATH
)


model_eval_link = pd.read_parquet(
    MODEL_RAW_EVAL_LINK_PATH
)


raw_training_metadata_rows = int(
    pd.read_parquet(
        RAW_TRAINING_COHORT_PATH,
        columns=[
            "RawTrainingRowOrder",
        ],
    ).shape[
        0
    ]
)


raw_evaluation_metadata_rows = int(
    pd.read_parquet(
        RAW_EVALUATION_COHORT_PATH,
        columns=[
            "RawEvaluationRowOrder",
        ],
    ).shape[
        0
    ]
)


condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)


required_model_key_columns = {
    "Build",
    "Test",
    "Verdict",
}


if not required_model_key_columns.issubset(
    model_training.columns
):
    raise RuntimeError(
        "Frozen model-training cohort is missing Build/Test/Verdict."
    )


if not required_model_key_columns.issubset(
    model_evaluation.columns
):
    raise RuntimeError(
        "Frozen model-evaluation cohort is missing Build/Test/Verdict."
    )


if (
    len(
        model_training
    )
    != EXPECTED_MODEL_TRAIN_ROWS
    or len(
        model_evaluation
    )
    != EXPECTED_MODEL_EVAL_ROWS
    or raw_training_metadata_rows
    != EXPECTED_RAW_TRAIN_ROWS
    or raw_evaluation_metadata_rows
    != EXPECTED_RAW_EVAL_ROWS
    or len(
        model_train_link
    )
    != EXPECTED_MODEL_TRAIN_ROWS
    or len(
        model_eval_link
    )
    != EXPECTED_MODEL_EVAL_ROWS
    or len(
        condition_plan
    )
    != EXPECTED_CONDITIONS
):
    raise RuntimeError(
        "Frozen Project 24 cohort/condition-plan dimensions differ."
    )


if (
    model_training.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).any()
    or model_evaluation.duplicated(
        subset=[
            "Build",
            "Test",
        ],
        keep=False,
    ).any()
):
    raise RuntimeError(
        "Frozen Project 24 model cohorts contain duplicate Build-Test keys."
    )


model_training_failures = int(
    model_training[
        "Verdict"
    ].ne(
        0
    ).sum()
)


model_evaluation_failures = int(
    model_evaluation[
        "Verdict"
    ].ne(
        0
    ).sum()
)


model_failing_evaluation_builds = int(
    model_evaluation.loc[
        model_evaluation[
            "Verdict"
        ].ne(
            0
        ),
        "Build",
    ].nunique()
)


if (
    model_training_failures
    != EXPECTED_MODEL_TRAIN_FAILURES
    or model_evaluation_failures
    != EXPECTED_MODEL_EVAL_FAILURES
    or model_failing_evaluation_builds
    != EXPECTED_MODEL_FAILING_EVAL_BUILDS
):
    raise RuntimeError(
        "Frozen Project 24 model label dimensions differ."
    )


training_binary_labels = (
    model_training[
        "Verdict"
    ].ne(
        0
    ).astype(
        np.int8
    )
)


evaluation_binary_labels = (
    model_evaluation[
        "Verdict"
    ].ne(
        0
    ).astype(
        np.int8
    )
)


training_label_values = sorted(
    training_binary_labels.unique().astype(
        int
    ).tolist()
)


evaluation_label_values = sorted(
    evaluation_binary_labels.unique().astype(
        int
    ).tolist()
)


# --------------------------------------------------------------------------------------------------
# 9. FREEZE PREDICTOR ORDER + CLEAN MEDIAN REFERENCE
# --------------------------------------------------------------------------------------------------

non_predictor_columns = {
    "ModelTrainingRowOrder",
    "ModelEvaluationRowOrder",
    "Build",
    "Test",
    "Verdict",
}


predictor_columns = [
    column
    for column in model_training.columns
    if column not in non_predictor_columns
]


evaluation_predictor_columns = [
    column
    for column in model_evaluation.columns
    if column not in non_predictor_columns
]


if predictor_columns != evaluation_predictor_columns:
    raise RuntimeError(
        "Frozen Project 24 training/evaluation predictor order differs."
    )


missing_rec_features = [
    feature
    for feature in REC_FEATURES
    if feature not in predictor_columns
]


if (
    len(
        predictor_columns
    )
    != EXPECTED_PREDICTORS
    or missing_rec_features
):
    raise RuntimeError(
        "Frozen Project 24 predictor/REC contract differs."
    )


predictor_records = []
median_records = []


training_nonfinite_after_imputation = 0
evaluation_nonfinite_after_imputation = 0
all_missing_predictors = []


for predictor_order, predictor in enumerate(
    predictor_columns,
    start=1,
):
    training_numeric = pd.to_numeric(
        model_training[
            predictor
        ],
        errors="coerce",
    ).astype(
        float
    )


    evaluation_numeric = pd.to_numeric(
        model_evaluation[
            predictor
        ],
        errors="coerce",
    ).astype(
        float
    )


    training_array = training_numeric.to_numpy(
        dtype=np.float64
    )


    evaluation_array = evaluation_numeric.to_numpy(
        dtype=np.float64
    )


    training_nonfinite_mask = (
        ~np.isfinite(
            training_array
        )
    )


    evaluation_nonfinite_mask = (
        ~np.isfinite(
            evaluation_array
        )
    )


    training_missing = int(
        training_nonfinite_mask.sum()
    )


    evaluation_missing = int(
        evaluation_nonfinite_mask.sum()
    )


    training_clean = training_array.copy()
    evaluation_clean = evaluation_array.copy()


    training_clean[
        training_nonfinite_mask
    ] = np.nan


    evaluation_clean[
        evaluation_nonfinite_mask
    ] = np.nan


    if np.isnan(
        training_clean
    ).all():
        training_median = np.nan
        all_missing_predictors.append(
            predictor
        )

    else:
        training_median = float(
            np.nanmedian(
                training_clean
            )
        )


    if np.isfinite(
        training_median
    ):
        training_clean = np.where(
            np.isnan(
                training_clean
            ),
            training_median,
            training_clean,
        )


        evaluation_clean = np.where(
            np.isnan(
                evaluation_clean
            ),
            training_median,
            evaluation_clean,
        )


    training_nonfinite_after = int(
        (
            ~np.isfinite(
                training_clean
            )
        ).sum()
    )


    evaluation_nonfinite_after = int(
        (
            ~np.isfinite(
                evaluation_clean
            )
        ).sum()
    )


    training_nonfinite_after_imputation += (
        training_nonfinite_after
    )


    evaluation_nonfinite_after_imputation += (
        evaluation_nonfinite_after
    )


    if predictor in VERDICT_DEPENDENT_REC:
        rec_class = "VERDICT_DEPENDENT"

    elif predictor in VERDICT_INDEPENDENT_REC:
        rec_class = "VERDICT_INDEPENDENT"

    else:
        rec_class = "NON_REC"


    predictor_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            predictor,

        "IsREC":
            predictor in REC_FEATURES,

        "RECClass":
            rec_class,

        "TrainingMissing":
            training_missing,

        "EvaluationMissing":
            evaluation_missing,

        "CleanTrainingMedian":
            training_median,

        "TrainingNonFiniteAfterReferenceImputation":
            training_nonfinite_after,

        "EvaluationNonFiniteAfterReferenceImputation":
            evaluation_nonfinite_after,
    })


    median_records.append({
        "PredictorOrder":
            predictor_order,

        "Predictor":
            predictor,

        "CleanTrainingMedian":
            training_median,
    })


predictor_contract = pd.DataFrame(
    predictor_records
)


clean_median_reference = pd.DataFrame(
    median_records
)


# --------------------------------------------------------------------------------------------------
# 10. MODEL IMPLEMENTATION CONTRACT — INSTANTIATE ONLY, NEVER FIT
# --------------------------------------------------------------------------------------------------

models_seed_1_a = create_models(
    repetition_seed=1
)


models_seed_1_b = create_models(
    repetition_seed=1
)


models_seed_2 = create_models(
    repetition_seed=2
)


model_contract_records = []


for technique in ML_TECHNIQUES:
    first = models_seed_1_a[
        technique
    ]

    same = models_seed_1_b[
        technique
    ]

    different = models_seed_2[
        technique
    ]


    first_params = first.get_params(
        deep=True
    )


    same_params = same.get_params(
        deep=True
    )


    different_params = different.get_params(
        deep=True
    )


    same_seed_same_configuration = bool(
        first_params
        == same_params
    )


    if technique == "NaiveBayes":
        seed_1_random_state = None
        seed_2_random_state = None
        different_seed_state_as_expected = True

    else:
        seed_1_random_state = int(
            first_params.get(
                "random_state"
            )
        )

        seed_2_random_state = int(
            different_params.get(
                "random_state"
            )
        )

        different_seed_state_as_expected = bool(
            seed_1_random_state
            != seed_2_random_state
        )


    model_contract_records.append({
        "Technique":
            technique,

        "EstimatorClass":
            (
                f"{first.__class__.__module__}."
                f"{first.__class__.__name__}"
            ),

        "Seed1RandomState":
            seed_1_random_state,

        "Seed2RandomState":
            seed_2_random_state,

        "SameSeedSameConfiguration":
            same_seed_same_configuration,

        "DifferentSeedStateAsExpected":
            different_seed_state_as_expected,

        "Seed1ParametersJSON":
            json.dumps(
                {
                    key:
                        canonical_parameter_value(
                            value
                        )
                    for key, value in first_params.items()
                },
                sort_keys=True,
                default=str,
            ),
    })


model_contract_table = pd.DataFrame(
    model_contract_records
)


rf_params = models_seed_1_a[
    "RandomForest"
].get_params(
    deep=True
)


xgb_params = models_seed_1_a[
    "XGBoost"
].get_params(
    deep=True
)


lgbm_params = models_seed_1_a[
    "LightGBM"
].get_params(
    deep=True
)


nb_params = models_seed_1_a[
    "NaiveBayes"
].get_params(
    deep=True
)


model_parameter_checks = {
    "RandomForest":
        (
            rf_params.get(
                "n_estimators"
            )
            == 100
            and rf_params.get(
                "max_features"
            )
            == "sqrt"
            and rf_params.get(
                "bootstrap"
            )
            is True
            and rf_params.get(
                "n_jobs"
            )
            == -1
            and int(
                rf_params.get(
                    "random_state"
                )
            )
            == deterministic_seed(
                1,
                "RandomForest_model",
            )
        ),

    "XGBoost":
        (
            xgb_params.get(
                "n_estimators"
            )
            == 100
            and xgb_params.get(
                "max_depth"
            )
            == 6
            and np.isclose(
                float(
                    xgb_params.get(
                        "learning_rate"
                    )
                ),
                0.1,
            )
            and xgb_params.get(
                "tree_method"
            )
            == "hist"
            and xgb_params.get(
                "n_jobs"
            )
            == -1
            and xgb_params.get(
                "verbosity"
            )
            == 0
            and xgb_params.get(
                "eval_metric"
            )
            == "logloss"
            and int(
                xgb_params.get(
                    "random_state"
                )
            )
            == deterministic_seed(
                1,
                "XGBoost_model",
            )
        ),

    "LightGBM":
        (
            lgbm_params.get(
                "n_estimators"
            )
            == 100
            and np.isclose(
                float(
                    lgbm_params.get(
                        "learning_rate"
                    )
                ),
                0.1,
            )
            and lgbm_params.get(
                "num_leaves"
            )
            == 31
            and lgbm_params.get(
                "n_jobs"
            )
            == -1
            and lgbm_params.get(
                "verbosity"
            )
            == -1
            and lgbm_params.get(
                "deterministic"
            )
            is True
            and lgbm_params.get(
                "force_col_wise"
            )
            is True
            and int(
                lgbm_params.get(
                    "random_state"
                )
            )
            == deterministic_seed(
                1,
                "LightGBM_model",
            )
        ),

    "NaiveBayes":
        np.isclose(
            float(
                nb_params.get(
                    "var_smoothing"
                )
            ),
            1e-9,
        ),
}


model_contract_failures = int(
    (
        ~model_contract_table[
            "SameSeedSameConfiguration"
        ]
        | ~model_contract_table[
            "DifferentSeedStateAsExpected"
        ]
    ).sum()
    + sum(
        not bool(
            value
        )
        for value in model_parameter_checks.values()
    )
)


# --------------------------------------------------------------------------------------------------
# 11. BASELINE + RANKING + RANDOM CONTRACT
# --------------------------------------------------------------------------------------------------

sample_build_id = int(
    model_evaluation[
        "Build"
    ].iloc[
        0
    ]
)


random_seed_1_a = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)


random_seed_1_b = deterministic_random_build_seed(
    repetition_seed=1,
    build_id=sample_build_id,
)


random_seed_2 = deterministic_random_build_seed(
    repetition_seed=2,
    build_id=sample_build_id,
)


sample_random_a = np.random.default_rng(
    random_seed_1_a
).random(
    100
)


sample_random_b = np.random.default_rng(
    random_seed_1_b
).random(
    100
)


sample_random_c = np.random.default_rng(
    random_seed_2
).random(
    100
)


random_same_seed_reproduced = bool(
    np.array_equal(
        sample_random_a,
        sample_random_b,
    )
)


random_different_seed_differs = bool(
    not np.array_equal(
        sample_random_a,
        sample_random_c,
    )
)


ranking_contract = {
    "ML": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",
    },

    "Random": {
        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "SeedRule":
            (
                "first little-endian uint32 of "
                "SHA-256(project|repetition_seed|"
                "Random_baseline_build_<BuildID>)"
            ),

        "ConstantAcrossNoiseForSameSeedAndBuild":
            True,
    },

    "LatestFail": {
        "SourceFeature":
            "REC_LastFailureAge",

        "ScoreFormula":
            "-REC_LastFailureAge",

        "Direction":
            "descending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            True,

        "UsesSameCorruptedHistoryAsML":
            True,
    },

    "QTF-Avg": {
        "SourceFeature":
            "REC_TotalAvgExeTime",

        "Direction":
            "ascending",

        "TieBreak":
            "Test ascending",

        "NoiseDependent":
            False,
    },
}


baseline_contract = {
    "Techniques":
        BASELINE_TECHNIQUES,

    "Random":
        ranking_contract[
            "Random"
        ],

    "LatestFail":
        ranking_contract[
            "LatestFail"
        ],

    "QTF-Avg":
        ranking_contract[
            "QTF-Avg"
        ],

    "NoRollingRetraining":
        True,

    "CleanEvaluationPartition":
        True,
}


latest_fail_feature_present = bool(
    "REC_LastFailureAge"
    in predictor_columns
)


qtf_feature_present = bool(
    "REC_TotalAvgExeTime"
    in predictor_columns
)


# --------------------------------------------------------------------------------------------------
# 12. APFD/APFDc SELF-TESTS
# --------------------------------------------------------------------------------------------------

metric_self_test_records = []


manual_failures = np.array([
    1,
    1,
    0,
    0,
    0,
], dtype=np.int8)


manual_apfd = calculate_apfd(
    manual_failures
)


metric_self_test_records.append({
    "Test":
        "APFD known value",

    "Expected":
        0.8,

    "Actual":
        manual_apfd,

    "Pass":
        bool(
            np.isclose(
                manual_apfd,
                0.8,
                rtol=0.0,
                atol=1e-12,
            )
        ),
})


slow_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        5.0,
        1.0,
        1.0,
        1.0,
        1.0,
    ]),
)


fast_failure_first = calculate_apfdc(
    manual_failures,
    np.array([
        1.0,
        1.0,
        1.0,
        1.0,
        5.0,
    ]),
)


metric_self_test_records.append({
    "Test":
        "APFDc known slow-first value",

    "Expected":
        5.0
        / 9.0,

    "Actual":
        slow_failure_first,

    "Pass":
        bool(
            np.isclose(
                slow_failure_first,
                5.0
                / 9.0,
                rtol=0.0,
                atol=1e-12,
            )
        ),
})


metric_self_test_records.append({
    "Test":
        "APFDc known fast-first value",

    "Expected":
        8.0
        / 9.0,

    "Actual":
        fast_failure_first,

    "Pass":
        bool(
            np.isclose(
                fast_failure_first,
                8.0
                / 9.0,
                rtol=0.0,
                atol=1e-12,
            )
        ),
})


metric_self_test_records.append({
    "Test":
        "APFDc duration sensitivity",

    "Expected":
        "fast-first > slow-first",

    "Actual":
        bool(
            fast_failure_first
            > slow_failure_first
        ),

    "Pass":
        bool(
            fast_failure_first
            > slow_failure_first
        ),
})


no_failure_apfd = calculate_apfd(
    np.zeros(
        5,
        dtype=np.int8,
    )
)


no_failure_apfdc = calculate_apfdc(
    np.zeros(
        5,
        dtype=np.int8,
    ),
    np.ones(
        5,
        dtype=float,
    ),
)


metric_self_test_records.append({
    "Test":
        "No-failure APFD is NaN",

    "Expected":
        True,

    "Actual":
        bool(
            np.isnan(
                no_failure_apfd
            )
        ),

    "Pass":
        bool(
            np.isnan(
                no_failure_apfd
            )
        ),
})


metric_self_test_records.append({
    "Test":
        "No-failure APFDc is NaN",

    "Expected":
        True,

    "Actual":
        bool(
            np.isnan(
                no_failure_apfdc
            )
        ),

    "Pass":
        bool(
            np.isnan(
                no_failure_apfdc
            )
        ),
})


negative_duration_rejected = False


try:
    calculate_apfdc(
        np.array([
            1,
            0,
        ]),
        np.array([
            1.0,
            -1.0,
        ]),
    )

except ValueError:
    negative_duration_rejected = True


metric_self_test_records.append({
    "Test":
        "Negative APFDc duration rejected",

    "Expected":
        True,

    "Actual":
        negative_duration_rejected,

    "Pass":
        negative_duration_rejected,
})


length_mismatch_rejected = False


try:
    calculate_apfdc(
        np.array([
            1,
            0,
        ]),
        np.array([
            1.0,
        ]),
    )

except ValueError:
    length_mismatch_rejected = True


metric_self_test_records.append({
    "Test":
        "APFDc length mismatch rejected",

    "Expected":
        True,

    "Actual":
        length_mismatch_rejected,

    "Pass":
        length_mismatch_rejected,
})


metric_self_test = pd.DataFrame(
    metric_self_test_records
)


metric_self_test_failures = int(
    (
        ~metric_self_test[
            "Pass"
        ]
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 13. STEP 4A VALIDATION
# --------------------------------------------------------------------------------------------------

validation_records = []


def C(
    check,
    expected,
    actual,
    passed,
):
    add_check(
        validation_records,
        check,
        expected,
        actual,
        passed,
    )


C(
    "Noise-plan checkpoint SHA-256",
    EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    noise_checkpoint_sha256,
    noise_checkpoint_sha256
    == EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
)

C(
    "REC checkpoint SHA-256",
    EXPECTED_REC_CHECKPOINT_SHA256,
    rec_checkpoint_sha256,
    rec_checkpoint_sha256
    == EXPECTED_REC_CHECKPOINT_SHA256,
)

C(
    "Selection checkpoint SHA-256",
    EXPECTED_SELECTION_CHECKPOINT_SHA256,
    selection_checkpoint_sha256,
    selection_checkpoint_sha256
    == EXPECTED_SELECTION_CHECKPOINT_SHA256,
)

C(
    "Source root SHA-256",
    EXPECTED_SOURCE_ROOT_SHA256,
    current_source_root_sha256,
    current_source_root_sha256
    == EXPECTED_SOURCE_ROOT_SHA256,
)

C(
    "Step 3A code revision",
    EXPECTED_STEP3A_CODE_REVISION,
    noise_checkpoint.get(
        "Step3ACodeRevision"
    ),
    noise_checkpoint.get(
        "Step3ACodeRevision"
    )
    == EXPECTED_STEP3A_CODE_REVISION,
)

C(
    "Raw training rows",
    EXPECTED_RAW_TRAIN_ROWS,
    raw_training_metadata_rows,
    raw_training_metadata_rows
    == EXPECTED_RAW_TRAIN_ROWS,
)

C(
    "Raw evaluation rows",
    EXPECTED_RAW_EVAL_ROWS,
    raw_evaluation_metadata_rows,
    raw_evaluation_metadata_rows
    == EXPECTED_RAW_EVAL_ROWS,
)

C(
    "Model training rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_training
    ),
    len(
        model_training
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)

C(
    "Model evaluation rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_evaluation
    ),
    len(
        model_evaluation
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)

C(
    "Model training failures",
    EXPECTED_MODEL_TRAIN_FAILURES,
    model_training_failures,
    model_training_failures
    == EXPECTED_MODEL_TRAIN_FAILURES,
)

C(
    "Model evaluation failures",
    EXPECTED_MODEL_EVAL_FAILURES,
    model_evaluation_failures,
    model_evaluation_failures
    == EXPECTED_MODEL_EVAL_FAILURES,
)

C(
    "Model failing evaluation builds",
    EXPECTED_MODEL_FAILING_EVAL_BUILDS,
    model_failing_evaluation_builds,
    model_failing_evaluation_builds
    == EXPECTED_MODEL_FAILING_EVAL_BUILDS,
)

C(
    "Condition-plan rows",
    EXPECTED_CONDITIONS,
    len(
        condition_plan
    ),
    len(
        condition_plan
    )
    == EXPECTED_CONDITIONS,
)

C(
    "Step 3A output-manifest failures",
    0,
    output_manifest_failures,
    output_manifest_failures
    == 0,
)

C(
    "Predictor columns",
    EXPECTED_PREDICTORS,
    len(
        predictor_columns
    ),
    len(
        predictor_columns
    )
    == EXPECTED_PREDICTORS,
)

C(
    "REC features",
    EXPECTED_REC_FEATURES,
    len(
        REC_FEATURES
    ),
    len(
        REC_FEATURES
    )
    == EXPECTED_REC_FEATURES,
)

C(
    "Raw-training link rows",
    EXPECTED_MODEL_TRAIN_ROWS,
    len(
        model_train_link
    ),
    len(
        model_train_link
    )
    == EXPECTED_MODEL_TRAIN_ROWS,
)

C(
    "Raw-evaluation link rows",
    EXPECTED_MODEL_EVAL_ROWS,
    len(
        model_eval_link
    ),
    len(
        model_eval_link
    )
    == EXPECTED_MODEL_EVAL_ROWS,
)

C(
    "Training binary label values",
    [
        0,
        1,
    ],
    training_label_values,
    training_label_values
    == [
        0,
        1,
    ],
)

C(
    "Evaluation binary label values",
    [
        0,
        1,
    ],
    evaluation_label_values,
    evaluation_label_values
    == [
        0,
        1,
    ],
)

C(
    "All-missing predictors",
    0,
    len(
        all_missing_predictors
    ),
    len(
        all_missing_predictors
    )
    == 0,
)

C(
    "Training non-finite values after reference imputation",
    0,
    training_nonfinite_after_imputation,
    training_nonfinite_after_imputation
    == 0,
)

C(
    "Evaluation non-finite values after reference imputation",
    0,
    evaluation_nonfinite_after_imputation,
    evaluation_nonfinite_after_imputation
    == 0,
)

C(
    "Runtime-version failures",
    0,
    runtime_version_failures,
    runtime_version_failures
    == 0,
)

C(
    "Model contract failures",
    0,
    model_contract_failures,
    model_contract_failures
    == 0,
)

for technique in ML_TECHNIQUES:
    C(
        f"{technique} parameter contract",
        True,
        bool(
            model_parameter_checks[
                technique
            ]
        ),
        bool(
            model_parameter_checks[
                technique
            ]
        ),
    )

C(
    "Metric self-test failures",
    0,
    metric_self_test_failures,
    metric_self_test_failures
    == 0,
)

C(
    "Random same-seed reproducible",
    True,
    random_same_seed_reproduced,
    random_same_seed_reproduced,
)

C(
    "Random different-seed differs",
    True,
    random_different_seed_differs,
    random_different_seed_differs,
)

C(
    "LatestFail feature present",
    True,
    latest_fail_feature_present,
    latest_fail_feature_present,
)

C(
    "QTF-Avg feature present",
    True,
    qtf_feature_present,
    qtf_feature_present,
)

C(
    "Technique set",
    sorted(
        ALL_TECHNIQUES
    ),
    sorted(
        ML_TECHNIQUES
        + BASELINE_TECHNIQUES
    ),
    sorted(
        ALL_TECHNIQUES
    )
    == sorted(
        ML_TECHNIQUES
        + BASELINE_TECHNIQUES
    ),
)

C(
    "No rolling retraining",
    True,
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
    bool(
        baseline_contract[
            "NoRollingRetraining"
        ]
    ),
)

C(
    "Evaluation partition clean and fixed",
    True,
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
    bool(
        baseline_contract[
            "CleanEvaluationPartition"
        ]
    ),
)

C(
    "Registry rows",
    EXPECTED_REGISTERED_PROJECTS,
    len(
        registry
    ),
    len(
        registry
    )
    == EXPECTED_REGISTERED_PROJECTS,
)

for (
    number,
    identity,
) in REQUIRED_REGISTERED_IDENTITIES.items():
    actual_identity = str(
        registry.loc[
            project_numbers.eq(
                number
            ),
            project_column,
        ].iloc[
            0
        ]
    )

    C(
        f"Project {number} frozen identity",
        identity,
        actual_identity,
        actual_identity
        == identity,
    )

C(
    "Active reservations",
    EXPECTED_ACTIVE_RESERVATIONS,
    noise_checkpoint.get(
        "ActiveReservations"
    ),
    noise_checkpoint.get(
        "ActiveReservations"
    )
    == EXPECTED_ACTIVE_RESERVATIONS,
)

C(
    "Project 24 registry rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)

C(
    "Runtime-priority ranking rule",
    EXPECTED_RUNTIME_PRIORITY_RULE,
    noise_checkpoint.get(
        "RuntimePriorityRule"
    ),
    noise_checkpoint.get(
        "RuntimePriorityRule"
    )
    == EXPECTED_RUNTIME_PRIORITY_RULE,
)

C(
    "Noise-plan RNG manifest rows",
    EXPECTED_RNG_MANIFEST_ROWS,
    int(
        noise_checkpoint.get(
            "RNGManifestRows"
        )
    ),
    int(
        noise_checkpoint.get(
            "RNGManifestRows"
        )
    )
    == EXPECTED_RNG_MANIFEST_ROWS,
)

C(
    "Noise-plan RNG row groups",
    EXPECTED_RNG_ROW_GROUPS,
    int(
        noise_checkpoint.get(
            "RNGManifestRowGroups"
        )
    ),
    int(
        noise_checkpoint.get(
            "RNGManifestRowGroups"
        )
    )
    == EXPECTED_RNG_ROW_GROUPS,
)


validation = pd.DataFrame(
    validation_records
)


failed_validation = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 24 Step 4A validation:"
)


display(
    validation
)


if not failed_validation.empty:
    print(
        "\nFailed Project 24 Step 4A checks:"
    )

    display(
        failed_validation
    )

    print(
        "\nNo Step 4A PASS status or checkpoint was written."
    )

    raise RuntimeError(
        "PROJECT 24 STEP 4A VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 14. FREEZE RUNTIME CONTRACT OUTPUTS
# --------------------------------------------------------------------------------------------------

RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


atomic_csv(
    RUNTIME_VERSION_PATH,
    runtime_versions,
)


atomic_csv(
    PREDICTOR_CONTRACT_PATH,
    predictor_contract,
)


atomic_csv(
    CLEAN_MEDIAN_REFERENCE_PATH,
    clean_median_reference,
)


model_contract_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "PositiveClass": {
        "Name":
            "failure",

        "Value":
            1,

        "Conversion":
            "binary target = (Verdict != 0).astype(int)",
    },

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        REC_FEATURES,

    "Imputation": {
        "Rule":
            (
                "For every experimental condition, compute one median per active "
                "predictor from that condition's training matrix only. Replace +/-"
                "infinity with missing before median computation. Fill training and "
                "clean evaluation missing values with those condition-training medians."
            ),

        "Scaling":
            "none",

        "ActivePredictors":
            "all 151 fixed predictor columns",
    },

    "Models":
        MODEL_CONFIG,

    "DeterministicSeedRule":
        (
            "first little-endian uint32 of "
            "SHA-256(project|repetition_seed|stream_name)"
        ),

    "ModelTable":
        model_contract_records,

    "ModelsFittedDuringStep4A":
        False,
}


atomic_json(
    MODEL_CONTRACT_PATH,
    model_contract_payload,
)


atomic_json(
    BASELINE_CONTRACT_PATH,
    baseline_contract,
)


atomic_csv(
    METRIC_SELF_TEST_PATH,
    metric_self_test,
)


atomic_json(
    RANKING_CONTRACT_PATH,
    ranking_contract,
)


atomic_csv(
    STEP4A_VALIDATION_PATH,
    validation,
)


runtime_output_paths = [
    RUNTIME_VERSION_PATH,
    PREDICTOR_CONTRACT_PATH,
    CLEAN_MEDIAN_REFERENCE_PATH,
    MODEL_CONTRACT_PATH,
    BASELINE_CONTRACT_PATH,
    METRIC_SELF_TEST_PATH,
    RANKING_CONTRACT_PATH,
    STEP4A_VALIDATION_PATH,
]


runtime_output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in runtime_output_paths
]


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RECCheckpointSHA256":
        rec_checkpoint_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "Step3ACodeRevision":
        EXPECTED_STEP3A_CODE_REVISION,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "RuntimeVersions":
        {
            row.Component:
                row.Version
            for row in runtime_versions.itertuples(
                index=False
            )
        },

    "ModelTrainingRows":
        len(
            model_training
        ),

    "ModelEvaluationRows":
        len(
            model_evaluation
        ),

    "ModelTrainingFailures":
        model_training_failures,

    "ModelEvaluationFailures":
        model_evaluation_failures,

    "ModelFailingEvaluationBuilds":
        model_failing_evaluation_builds,

    "ConditionPlanRows":
        len(
            condition_plan
        ),

    "RNGManifestRows":
        int(
            noise_checkpoint.get(
                "RNGManifestRows"
            )
        ),

    "RNGManifestRowGroups":
        int(
            noise_checkpoint.get(
                "RNGManifestRowGroups"
            )
        ),

    "TrainingBinaryLabelValues":
        training_label_values,

    "EvaluationBinaryLabelValues":
        evaluation_label_values,

    "AllMissingPredictors":
        all_missing_predictors,

    "TrainingNonFiniteAfterReferenceImputation":
        training_nonfinite_after_imputation,

    "EvaluationNonFiniteAfterReferenceImputation":
        evaluation_nonfinite_after_imputation,

    "RuntimeVersionFailures":
        runtime_version_failures,

    "ModelContractFailures":
        model_contract_failures,

    "MetricSelfTestFailures":
        metric_self_test_failures,

    "RandomSameSeedReproducible":
        random_same_seed_reproduced,

    "RandomDifferentSeedDiffers":
        random_different_seed_differs,

    "LatestFailFeaturePresent":
        latest_fail_feature_present,

    "QTFAvgFeaturePresent":
        qtf_feature_present,

    "NoRollingRetraining":
        baseline_contract[
            "NoRollingRetraining"
        ],

    "CleanEvaluationPartition":
        baseline_contract[
            "CleanEvaluationPartition"
        ],

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "RuntimeOutputManifest":
        runtime_output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed_validation
        ),

    "CompletionRegistrySHA256":
        registry_sha256_before,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "RegistryModified":
        False,

    "Projects1To23Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ModelsFittedDuringStep4A":
        False,

    "ConditionsExecutedDuringStep4A":
        False,
}


atomic_json(
    STEP4A_REPORT_PATH,
    report_payload,
)


checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_24_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT",

    "RuntimeContractFrozen":
        True,

    "PredictorContractFrozen":
        True,

    "ModelContractFrozen":
        True,

    "BaselineContractFrozen":
        True,

    "MetricContractFrozen":
        True,

    "RankingContractFrozen":
        True,

    "NoRollingRetraining":
        True,

    "EvaluationCohortImmutable":
        True,

    "ReadyForTwoConditionSmokeTest":
        True,
}


atomic_json(
    RUNTIME_CHECKPOINT_PATH,
    checkpoint_payload,
)


runtime_checkpoint_sha256 = sha256_file(
    RUNTIME_CHECKPOINT_PATH
)


status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP4A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "SourceRootSHA256":
        current_source_root_sha256,

    "SelectionCheckpointSHA256":
        selection_checkpoint_sha256,

    "RECCheckpointSHA256":
        rec_checkpoint_sha256,

    "NoisePlanCheckpointSHA256":
        noise_checkpoint_sha256,

    "Step3ACodeRevision":
        EXPECTED_STEP3A_CODE_REVISION,

    "PredictorCount":
        len(
            predictor_columns
        ),

    "RECFeatures":
        len(
            REC_FEATURES
        ),

    "MLTechniques":
        ML_TECHNIQUES,

    "BaselineTechniques":
        BASELINE_TECHNIQUES,

    "Checkpoint":
        str(
            RUNTIME_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        runtime_checkpoint_sha256,

    "ReadyForTwoConditionSmokeTest":
        True,

    "RegistryModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ModelsFittedDuringStep4A":
        False,

    "ConditionsExecutedDuringStep4A":
        False,

    "NextRequiredStep":
        "PROJECT 24 CELL 8 / STEP 4B — TWO-CONDITION END-TO-END SMOKE TEST",
}


atomic_json(
    STEP4A_STATUS_PATH,
    status_payload,
)


# --------------------------------------------------------------------------------------------------
# 15. READBACK + IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    RUNTIME_CHECKPOINT_PATH
)


status_readback = load_json(
    STEP4A_STATUS_PATH
)


report_readback = load_json(
    STEP4A_REPORT_PATH
)


for label, payload in [
    (
        "runtime checkpoint",
        checkpoint_readback,
    ),
    (
        "Step 4A status",
        status_readback,
    ),
    (
        "Step 4A report",
        report_readback,
    ),
]:
    if payload.get(
        "Status"
    ) != STEP4A_STATUS:
        raise RuntimeError(
            f"Project 24 {label} readback failed."
        )


if not bool(
    checkpoint_readback.get(
        "ReadyForTwoConditionSmokeTest",
        False,
    )
):
    raise RuntimeError(
        "Project 24 runtime checkpoint does not authorize the smoke test."
    )


runtime_manifest_readback_failures = 0


for item in checkpoint_readback.get(
    "RuntimeOutputManifest",
    [],
):
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        runtime_manifest_readback_failures += 1


if runtime_manifest_readback_failures:
    raise RuntimeError(
        "One or more frozen Project 24 runtime outputs failed readback."
    )


registry_sha256_after = sha256_file(
    REGISTRY_PATH
)


if (
    registry_sha256_after
    != registry_sha256_before
):
    raise RuntimeError(
        "Completion registry changed during Project 24 Step 4A."
    )


if sha256_file(
    NOISE_PLAN_CHECKPOINT_PATH
) != EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Frozen Project 24 noise-plan checkpoint changed during Step 4A."
    )


if sha256_file(
    REC_CHECKPOINT_PATH
) != EXPECTED_REC_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Frozen Project 24 REC checkpoint changed during Step 4A."
    )


if sha256_file(
    SELECTION_CHECKPOINT_PATH
) != EXPECTED_SELECTION_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Frozen Project 24 selection checkpoint changed during Step 4A."
    )


final_source_records = []


for row in current_source_manifest.itertuples(
    index=False
):
    source_path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                source_path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                source_path
            ),
    })


final_source_manifest = pd.DataFrame(
    final_source_records
)


if source_root_hash(
    final_source_manifest
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 24 source changed during Step 4A."
    )


# --------------------------------------------------------------------------------------------------
# 16. DISPLAY
# --------------------------------------------------------------------------------------------------

print(
    "\nRuntime versions:"
)


display(
    runtime_versions
)


print(
    "\nPredictor contract summary:"
)


display(
    predictor_contract.groupby(
        [
            "IsREC",
            "RECClass",
        ],
        dropna=False,
        as_index=False,
    ).agg(
        Predictors=(
            "Predictor",
            "count",
        ),

        TrainingMissingValues=(
            "TrainingMissing",
            "sum",
        ),

        EvaluationMissingValues=(
            "EvaluationMissing",
            "sum",
        ),
    )
)


print(
    "\nModel implementation contract:"
)


display(
    model_contract_table[
        [
            "Technique",
            "EstimatorClass",
            "Seed1RandomState",
            "Seed2RandomState",
            "SameSeedSameConfiguration",
            "DifferentSeedStateAsExpected",
        ]
    ]
)


print(
    "\nMetric self-tests:"
)


display(
    metric_self_test
)


print(
    "\nStep 3A output-manifest audit:"
)


display(
    output_manifest_audit[
        [
            "Path",
            "ExpectedBytes",
            "ActualBytes",
            "Pass",
        ]
    ]
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 140
)

print(
    "=== PROJECT 24 CELL 7 / STEP 4A RESULT ==="
)

print(
    "=" * 140
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "Step 3A code revision:",
    EXPECTED_STEP3A_CODE_REVISION,
)

print(
    "Noise-plan checkpoint SHA-256:",
    noise_checkpoint_sha256,
)

print(
    "REC checkpoint SHA-256:",
    rec_checkpoint_sha256,
)

print(
    "Source root SHA-256:",
    current_source_root_sha256,
)

print(
    "\nFixed model cohorts:"
)

print(
    "Training rows:",
    len(
        model_training
    ),
)

print(
    "Evaluation rows:",
    len(
        model_evaluation
    ),
)

print(
    "Training failures:",
    model_training_failures,
)

print(
    "Evaluation failures:",
    model_evaluation_failures,
)

print(
    "Failing evaluation builds:",
    model_failing_evaluation_builds,
)

print(
    "Predictors:",
    len(
        predictor_columns
    ),
)

print(
    "REC features:",
    len(
        REC_FEATURES
    ),
)

print(
    "\nRuntime/model contract:"
)

print(
    "Runtime-version failures:",
    runtime_version_failures,
)

print(
    "Model contract failures:",
    model_contract_failures,
)

print(
    "Metric self-test failures:",
    metric_self_test_failures,
)

print(
    "Random same-seed reproducible:",
    random_same_seed_reproduced,
)

print(
    "Random different-seed differs:",
    random_different_seed_differs,
)

print(
    "LatestFail feature present:",
    latest_fail_feature_present,
)

print(
    "QTF-Avg feature present:",
    qtf_feature_present,
)

print(
    "No rolling retraining:",
    baseline_contract[
        "NoRollingRetraining"
    ],
)

print(
    "Evaluation partition clean and fixed:",
    baseline_contract[
        "CleanEvaluationPartition"
    ],
)

print(
    "ML techniques:",
    ML_TECHNIQUES,
)

print(
    "Baselines:",
    BASELINE_TECHNIQUES,
)

print(
    "Primary / secondary metrics:",
    "APFDc / APFD",
)

print(
    "\nIsolation:"
)

print(
    "Completion registry unchanged:",
    registry_sha256_after
    == registry_sha256_before,
)

print(
    "Projects 1–23 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Models fitted in Step 4A:",
    False,
)

print(
    "Conditions executed in Step 4A:",
    False,
)

print(
    "\nValidation:"
)

print(
    "Checks:",
    len(
        validation
    ),
)

print(
    "Failed checks:",
    len(
        failed_validation
    ),
)

print(
    "\nRuntime-contract checkpoint:"
)

print(
    RUNTIME_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    runtime_checkpoint_sha256,
)

print(
    "\nNext required step:",
    "PROJECT 24 CELL 8 / STEP 4B — TWO-CONDITION END-TO-END SMOKE TEST",
)

print(
    "\nSTATUS:",
    STEP4A_STATUS,
)

print(
    "=" * 140
)


=== PROJECT 24 CELL 7 / STEP 4A: EXPERIMENT RUNTIME AND MODEL CONTRACT FREEZE ===

Project 24 Step 4A validation:


,Check,Expected,Actual,Pass
0,Noise-plan checkpoint SHA-256,9dba84682eabdc1393219671e77aabaf1f2d4385f29239...,9dba84682eabdc1393219671e77aabaf1f2d4385f29239...,True
1,REC checkpoint SHA-256,80ac2e6986ae2c14d636c57c834cc47678b54cd05ba323...,80ac2e6986ae2c14d636c57c834cc47678b54cd05ba323...,True
2,Selection checkpoint SHA-256,d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f167...,d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f167...,True
3,Source root SHA-256,2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21...,2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21...,True
4,Step 3A code revision,PROJECT_24_STEP3A_V2_FROZEN_GLOBAL_BUILD_ORDER...,PROJECT_24_STEP3A_V2_FROZEN_GLOBAL_BUILD_ORDER...,True
5,Raw training rows,4040801,4040801,True
6,Raw evaluation rows,1594226,1594226,True
7,Model training rows,205696,205696,True
8,Model evaluation rows,18854,18854,True
9,Model training failures,1777,1777,True



Runtime versions:


,Component,ExpectedVersion,Version,Pass
0,Python,3.12.13,3.12.13,True
1,numpy,2.0.2,2.0.2,True
2,pandas,2.2.2,2.2.2,True
3,scikit-learn,1.6.1,1.6.1,True
4,xgboost,3.3.0,3.3.0,True
5,lightgbm,4.6.0,4.6.0,True
6,pyarrow,18.1.0,18.1.0,True



Predictor contract summary:


...


Metric self-tests:


,Test,Expected,Actual,Pass
0,APFD known value,0.8,0.8,True
1,APFDc known slow-first value,0.555556,0.555556,True
2,APFDc known fast-first value,0.888889,0.888889,True
3,APFDc duration sensitivity,fast-first > slow-first,True,True
4,No-failure APFD is NaN,True,True,True
5,No-failure APFDc is NaN,True,True,True
6,Negative APFDc duration rejected,True,True,True
7,APFDc length mismatch rejected,True,True,True



Step 3A output-manifest audit:


,Path,ExpectedBytes,ActualBytes,Pass
0,/content/drive/MyDrive/Thesis_Experiment/Resul...,11013036,11013036,True
1,/content/drive/MyDrive/Thesis_Experiment/Resul...,4390940,4390940,True
2,/content/drive/MyDrive/Thesis_Experiment/Resul...,8705824,8705824,True
3,/content/drive/MyDrive/Thesis_Experiment/Resul...,1255059,1255059,True
4,/content/drive/MyDrive/Thesis_Experiment/Resul...,1254985,1254985,True
5,/content/drive/MyDrive/Thesis_Experiment/Resul...,163634,163634,True
6,/content/drive/MyDrive/Thesis_Experiment/Resul...,97,97,True
7,/content/drive/MyDrive/Thesis_Experiment/Resul...,5148,5148,True
8,/content/drive/MyDrive/Thesis_Experiment/Resul...,1277437690,1277437690,True
9,/content/drive/MyDrive/Thesis_Experiment/Resul...,88478,88478,True



=== PROJECT 24 CELL 7 / STEP 4A RESULT ===
Project: SonarSource@sonarqube
Project slug: SonarSource__sonarqube
Step 3A code revision: PROJECT_24_STEP3A_V2_FROZEN_GLOBAL_BUILD_ORDER_SCHEMA_FIX
Noise-plan checkpoint SHA-256: 9dba84682eabdc1393219671e77aabaf1f2d4385f29239236e50615bf0180515
REC checkpoint SHA-256: 80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803
Source root SHA-256: 2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70

Fixed model cohorts:
Training rows: 205696
Evaluation rows: 18854
Training failures: 1777
Evaluation failures: 20
Failing evaluation builds: 17
Predictors: 151
REC features: 19

Runtime/model contract:
Runtime-version failures: 0
Model contract failures: 0
Metric self-test failures: 0
Random same-seed reproducible: True
Random different-seed differs: True
LatestFail feature present: True
QTF-Avg feature present: True
No rolling retraining: True
Evaluation partition clean and fixed: True
ML techniques: ['RandomForest', 'XGBoost', 

In [9]:
# ==================================================================================================
# PROJECT 24 — CELL 8 / STEP 4B
# TWO-CONDITION END-TO-END SMOKE TEST
# Project: SonarSource@sonarqube
#
# Smoke conditions: noise_00__seed_01 and noise_50__seed_01.
# Validates all frozen upstream contracts, reproduces seed-1 noise exactly, reconstructs all 19 REC
# features from the Step-2B-frozen per-test execution order, applies the frozen clean anchor, fits the
# four frozen ML techniques, evaluates the three frozen baselines, calculates APFDc/APFD, and freezes
# smoke outputs only. It never writes to the Step-5A raw-result root or completion registry.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import gc, hashlib, json, os, shutil, time, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pandas.errors import PerformanceWarning
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
warnings.simplefilter("ignore", PerformanceWarning)

print("="*140)
print("=== PROJECT 24 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===")
print("="*140)

# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------
PROJECT_NUMBER=24; PROJECT_NAME="SonarSource@sonarqube"; PROJECT_SLUG="SonarSource__sonarqube"; PROJECT_SHORT="SONARQUBE"
EXPECTED_SELECTION_STATUS="PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN"
EXPECTED_STEP2B_STATUS="PASS_PROJECT_24_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
EXPECTED_STEP3A_STATUS="PASS_PROJECT_24_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
EXPECTED_STEP4A_STATUS="PASS_PROJECT_24_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
STEP4B_STATUS="PASS_PROJECT_24_TWO_CONDITION_END_TO_END_SMOKE_TEST"; CONDITION_STATUS="PASS_PROJECT_24_SMOKE_CONDITION"
EXPECTED_RUNTIME_CHECKPOINT_SHA256="ff9ed031d21d67bf218d5eab70bd8502ca39953adedbe840348f382d5599314e"
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256="9dba84682eabdc1393219671e77aabaf1f2d4385f29239236e50615bf0180515"
EXPECTED_REC_CHECKPOINT_SHA256="80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803"
EXPECTED_SELECTION_CHECKPOINT_SHA256="d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
EXPECTED_SOURCE_ROOT_SHA256="2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
EXPECTED_REGISTRY_SHA256="c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
EXPECTED_STEP3A_CODE_REVISION="PROJECT_24_STEP3A_V2_FROZEN_GLOBAL_BUILD_ORDER_SCHEMA_FIX"
EXPECTED_REC_IMPLEMENTATION="PROJECT_24_V1_SCALABLE_EXACT_TIE_DP_BINARY_AGE_CONSTRAINT_ANCHOR_AWARE_MAPPING_BOUNDARY_AUDIT"
EXPECTED_COMPLETE_STATUS="COMPLETE_AND_FROZEN"; EXPECTED_REGISTERED_PROJECTS=23; EXPECTED_ACTIVE_RESERVATIONS=[]
EXPECTED_RUNTIME_PRIORITY_RULE=["ModelTrainingRows ascending","ModelEvaluationRows ascending","RawExecutionRows ascending","Project ascending"]
EXPECTED_BUILDS=4286; EXPECTED_EVALUATION_BUILDS=1072
EXPECTED_RAW_ROWS=5_635_027; EXPECTED_RAW_TRAIN_ROWS=4_040_801; EXPECTED_RAW_EVAL_ROWS=1_594_226
EXPECTED_MODEL_ROWS=224_550; EXPECTED_MODEL_TRAIN_ROWS=205_696; EXPECTED_MODEL_EVAL_ROWS=18_854
EXPECTED_MODEL_TRAIN_FAILURES=1_777; EXPECTED_MODEL_EVAL_FAILURES=20; EXPECTED_FAILING_EVAL_BUILDS=17
EXPECTED_PREDICTORS=151; EXPECTED_REC_FEATURES=19; EXPECTED_RNG_ROWS=121_224_030; EXPECTED_RNG_ROW_GROUPS=30; EXPECTED_CONDITIONS=270
EXPECTED_SMOKE_CONDITIONS=2; EXPECTED_TECHNIQUES=7; EXPECTED_ML_TECHNIQUES=4
EXPECTED_RANKING_ROWS_PER_CONDITION=EXPECTED_MODEL_EVAL_ROWS*EXPECTED_TECHNIQUES
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION=EXPECTED_FAILING_EVAL_BUILDS*EXPECTED_TECHNIQUES
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION=7; EXPECTED_MODEL_FIT_ROWS_PER_CONDITION=4; EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION=151
SMOKE_CONDITION_IDS=["noise_00__seed_01","noise_50__seed_01"]; SMOKE_REPETITION_SEED=1
RECENT_WINDOW=6; SUCCESS_VERDICT_CODE=0; EXCEPTION_VERDICT_CODE=1; ASSERTION_VERDICT_CODE=2
DIRECT_RTOL=1e-9; DIRECT_ATOL=1e-9
ML_TECHNIQUES=["RandomForest","XGBoost","LightGBM","NaiveBayes"]
BASELINE_TECHNIQUES=["Random","LatestFail","QTF-Avg"]; ALL_TECHNIQUES=ML_TECHNIQUES+BASELINE_TECHNIQUES
REC_FEATURES=["REC_Age","REC_LastFailureAge","REC_LastTransitionAge","REC_RecentAvgExeTime","REC_RecentMaxExeTime","REC_RecentFailRate","REC_RecentAssertRate","REC_RecentExcRate","REC_RecentTransitionRate","REC_TotalAvgExeTime","REC_TotalMaxExeTime","REC_TotalFailRate","REC_TotalAssertRate","REC_TotalExcRate","REC_TotalTransitionRate","REC_LastVerdict","REC_LastExeTime","REC_MaxTestFileFailRate","REC_MaxTestFileTransitionRate"]
VERDICT_DEPENDENT_REC=["REC_LastFailureAge","REC_LastTransitionAge","REC_RecentFailRate","REC_RecentAssertRate","REC_RecentExcRate","REC_RecentTransitionRate","REC_TotalFailRate","REC_TotalAssertRate","REC_TotalExcRate","REC_TotalTransitionRate","REC_LastVerdict","REC_MaxTestFileFailRate","REC_MaxTestFileTransitionRate"]
VERDICT_INDEPENDENT_REC=["REC_Age","REC_RecentAvgExeTime","REC_RecentMaxExeTime","REC_TotalAvgExeTime","REC_TotalMaxExeTime","REC_LastExeTime"]
MODEL_CONFIG={
 "RandomForest":{"n_estimators":100,"max_features":"sqrt","bootstrap":True,"n_jobs":-1},
 "XGBoost":{"n_estimators":100,"max_depth":6,"learning_rate":0.1,"tree_method":"hist","n_jobs":-1,"verbosity":0,"eval_metric":"logloss"},
 "LightGBM":{"n_estimators":100,"learning_rate":0.1,"num_leaves":31,"n_jobs":-1,"verbosity":-1,"deterministic":True,"force_col_wise":True},
 "NaiveBayes":{"var_smoothing":1e-9},
}
REQUIRED_REGISTERED_IDENTITIES={11:"apache@shardingsphere",12:"zolyfarkas@spf4j",13:"jcabi@jcabi-github",14:"JMRI@JMRI",15:"eclipse@steady",16:"apache@rocketmq",17:"yamcs@Yamcs",18:"cantaloupe-project@cantaloupe",19:"EMResearch@EvoMaster",20:"apache@curator",21:"facebook@buck",22:"apache@logging-log4j2",23:"apache@sling"}

# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------
THESIS_ROOT=Path("/content/drive/MyDrive/Thesis_Experiment"); NOTES_ROOT=THESIS_ROOT/"Notes"; RESULTS_ROOT=THESIS_ROOT/"Results"
REGISTRY_PATH=NOTES_ROOT/"completed_project_registry.csv"; SOURCE_DIR=Path("/content/datasets/datasets/SonarSource@sonarqube")
SELECTION_ROOT=RESULTS_ROOT/"Aggregated"/"project_24_selection"; FROZEN_SOURCE_MANIFEST_PATH=SELECTION_ROOT/"project_24_frozen_source_manifest.csv"
SELECTION_CHECKPOINT_PATH=NOTES_ROOT/"project_24_selection_checkpoint.json"; PROJECT_ROOT=RESULTS_ROOT/"Aggregated"/PROJECT_SLUG
REC_ROOT=PROJECT_ROOT/f"{PROJECT_SHORT}_rec_preflight"; BUILD_ENTITY_PATH=REC_ROOT/f"{PROJECT_SHORT}_build_entity_map.csv.gz"
CLEAN_ANCHOR_OFFSETS_PATH=REC_ROOT/f"{PROJECT_SHORT}_clean_rec_anchor_offsets.parquet"; FROZEN_GLOBAL_BUILD_ORDER_PATH=REC_ROOT/f"{PROJECT_SHORT}_clean_global_build_order.csv"
REC_CHECKPOINT_PATH=NOTES_ROOT/"project_24_rec_reconstruction_checkpoint.json"; NOISE_ROOT=PROJECT_ROOT/f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH=NOISE_ROOT/f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"; RAW_EVALUATION_COHORT_PATH=NOISE_ROOT/f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
MODEL_TRAINING_COHORT_PATH=NOISE_ROOT/f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"; MODEL_EVALUATION_COHORT_PATH=NOISE_ROOT/f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
MODEL_RAW_TRAIN_LINK_PATH=NOISE_ROOT/f"{PROJECT_SHORT}_model_to_raw_training_link.parquet"; MODEL_RAW_EVAL_LINK_PATH=NOISE_ROOT/f"{PROJECT_SHORT}_model_to_raw_evaluation_link.parquet"
RNG_MANIFEST_PATH=NOISE_ROOT/f"{PROJECT_SHORT}_rng_manifest.parquet"; CONDITION_PLAN_PATH=NOISE_ROOT/f"{PROJECT_SHORT}_condition_plan.csv"; NOISE_PLAN_CHECKPOINT_PATH=NOTES_ROOT/"project_24_noise_plan_checkpoint.json"
RUNTIME_ROOT=PROJECT_ROOT/f"{PROJECT_SHORT}_runtime_contract"; PREDICTOR_CONTRACT_PATH=RUNTIME_ROOT/f"{PROJECT_SHORT}_predictor_contract.csv"
STEP4A_STATUS_PATH=PROJECT_ROOT/f"{PROJECT_SHORT}_step4a_status.json"; STEP4A_REPORT_PATH=RUNTIME_ROOT/f"{PROJECT_SHORT}_step4a_report.json"; RUNTIME_CHECKPOINT_PATH=NOTES_ROOT/"project_24_runtime_contract_checkpoint.json"
SMOKE_ROOT=PROJECT_ROOT/f"{PROJECT_SHORT}_smoke_test"; STEP4B_STATUS_PATH=PROJECT_ROOT/f"{PROJECT_SHORT}_step4b_status.json"; SMOKE_CHECKPOINT_PATH=NOTES_ROOT/"project_24_smoke_test_checkpoint.json"
SMOKE_INVENTORY_PATH=SMOKE_ROOT/f"{PROJECT_SHORT}_smoke_condition_inventory.csv"; SMOKE_AUDIT_PATH=SMOKE_ROOT/f"{PROJECT_SHORT}_smoke_combined_condition_audit.csv"
SMOKE_PROJECT_RUNS_PATH=SMOKE_ROOT/f"{PROJECT_SHORT}_smoke_combined_project_runs.csv"; SMOKE_BUILD_METRICS_PATH=SMOKE_ROOT/f"{PROJECT_SHORT}_smoke_combined_build_metrics.csv"
SMOKE_MODEL_FITS_PATH=SMOKE_ROOT/f"{PROJECT_SHORT}_smoke_combined_model_fits.csv"; SMOKE_BASELINE_INVARIANCE_PATH=SMOKE_ROOT/f"{PROJECT_SHORT}_smoke_baseline_invariance.csv"
SMOKE_VALIDATION_PATH=SMOKE_ROOT/f"{PROJECT_SHORT}_step4b_validation.csv"; SMOKE_REPORT_PATH=SMOKE_ROOT/f"{PROJECT_SHORT}_step4b_report.json"
FULL_RAW_RESULT_ROOT=RESULTS_ROOT/"Raw"/PROJECT_SLUG

# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------
def sha256_file(path,chunk_size=8*1024*1024):
 h=hashlib.sha256()
 with Path(path).open("rb") as f:
  while True:
   b=f.read(chunk_size)
   if not b: break
   h.update(b)
 return h.hexdigest()
def sha256_array(values,dtype): return hashlib.sha256(np.asarray(values).astype(dtype,copy=False).tobytes(order="C")).hexdigest()
def load_json(path):
 with Path(path).open("r",encoding="utf-8") as f: return json.load(f)
def atomic_json(path,payload):
 path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_name(f".{path.name}.tmp_{os.getpid()}")
 with tmp.open("w",encoding="utf-8") as f: json.dump(payload,f,indent=2,sort_keys=True,ensure_ascii=False,default=str); f.write("\n")
 os.replace(tmp,path)
def atomic_csv(path,frame,compression=None):
 path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_name(f".{path.name}.tmp_{os.getpid()}")
 frame.to_csv(tmp,index=False,lineterminator="\n",compression=compression); os.replace(tmp,path)
def source_root_hash(frame):
 h=hashlib.sha256()
 for r in frame.sort_values("RelativePath",kind="mergesort").itertuples(index=False): h.update(f"{r.RelativePath}\0{int(r.SizeBytes)}\0{str(r.SHA256).lower()}\n".encode())
 return h.hexdigest()
def directory_manifest(root):
 root=Path(root); rows=[]
 if root.exists():
  for p in sorted((x for x in root.rglob("*") if x.is_file()),key=lambda x:x.relative_to(root).as_posix()): rows.append({"RelativePath":p.relative_to(root).as_posix(),"Bytes":int(p.stat().st_size),"SHA256":sha256_file(p)})
 return pd.DataFrame(rows,columns=["RelativePath","Bytes","SHA256"])
def directory_root_hash(m):
 h=hashlib.sha256()
 for r in m.sort_values("RelativePath",kind="mergesort").itertuples(index=False): h.update(f"{r.RelativePath}\0{int(r.Bytes)}\0{r.SHA256}\n".encode())
 return h.hexdigest()
def parse_int(values,label):
 x=pd.to_numeric(values,errors="coerce")
 if x.isna().any(): raise RuntimeError(f"{label} contains missing/non-numeric values")
 a=x.to_numpy(float)
 if not np.isclose(a,np.floor(a),rtol=0,atol=0).all(): raise RuntimeError(f"{label} contains non-integral values")
 return x.astype("int64")
def resolve_col(columns,name,label):
 m=[c for c in columns if str(c).strip().lower()==name.lower()]
 if len(m)!=1: raise RuntimeError(f"Could not resolve {label}: {m}; columns={list(columns)}")
 return m[0]
def add_check(rows,check,expected,actual,passed): rows.append({"Check":check,"Expected":expected,"Actual":actual,"Pass":bool(passed)})
def deterministic_seed(seed,stream): return int.from_bytes(hashlib.sha256(f"{PROJECT_NAME}|{int(seed)}|{stream}".encode()).digest()[:4],"little")
def deterministic_random_build_seed(seed,build): return deterministic_seed(seed,f"Random_baseline_build_{int(build)}")
def create_models(seed):
 return {"RandomForest":RandomForestClassifier(**MODEL_CONFIG["RandomForest"],random_state=deterministic_seed(seed,"RandomForest_model")),"XGBoost":XGBClassifier(**MODEL_CONFIG["XGBoost"],random_state=deterministic_seed(seed,"XGBoost_model")),"LightGBM":LGBMClassifier(**MODEL_CONFIG["LightGBM"],random_state=deterministic_seed(seed,"LightGBM_model")),"NaiveBayes":GaussianNB(**MODEL_CONFIG["NaiveBayes"])}
def calculate_apfd(f):
 f=np.asarray(f,np.int8); n=len(f); m=int(f.sum())
 if n==0 or m==0:return np.nan
 return float(1-(np.flatnonzero(f==1)+1).sum()/(n*m)+1/(2*n))
def calculate_apfdc(f,d):
 f=np.asarray(f,np.int8); d=np.asarray(d,float)
 if len(f)!=len(d): raise ValueError("length mismatch")
 if len(f)==0 or f.sum()==0:return np.nan
 if not np.isfinite(d).all() or (d<0).any(): raise ValueError("invalid durations")
 total=float(d.sum())
 if total<=0:return np.nan
 before=np.r_[0.0,np.cumsum(d)[:-1]]; mask=f==1
 return float(1-np.mean((before[mask]+0.5*d[mask])/total))
def verify_manifest(items,label):
 failures=[]
 if not isinstance(items,list) or not items:return [{"Manifest":label,"Reason":"missing/empty manifest"}]
 for item in items:
  p=Path(item["Path"]); ok=p.is_file() and int(p.stat().st_size)==int(item["Bytes"]) and sha256_file(p)==str(item["SHA256"])
  if not ok: failures.append({"Manifest":label,"Path":str(p),"Reason":"missing/size/SHA mismatch"})
 return failures
def prefix_sum(values):
 values=np.asarray(values); out=np.empty(len(values)+1,dtype=np.float64 if values.dtype.kind=="f" else np.int64); out[0]=0; np.cumsum(values,out=out[1:]); return out
def safe_divide(num,den):
 num=np.asarray(num,float); den=np.asarray(den,float); out=np.full(len(den),-1.0,float); ok=den>0; out[ok]=num[ok]/den[ok]; return out
def calculate_file_rate(target_builds,current_entities,entity_changed_builds):
 if not target_builds:return -1.0
 mx=0
 for entity in current_entities: mx=max(mx,len(target_builds.intersection(entity_changed_builds.get(int(entity),frozenset()))))
 return 0.0 if mx==0 else float(mx/len(target_builds))

def reconstruct_group(builds,verdicts,durations,global_positions,positions,changed_entities_by_build,entity_changed_builds):
 builds=np.asarray(builds,np.int64); verdicts=np.asarray(verdicts,np.int64); durations=np.asarray(durations,np.float64); global_positions=np.asarray(global_positions,np.int64); positions=np.asarray(positions,np.int64)
 n=len(builds); allpos=np.arange(n,dtype=np.int64); fail=(verdicts!=0).astype(np.int64); ass=(verdicts==2).astype(np.int64); exc=(verdicts==1).astype(np.int64); trans=np.zeros(n,np.int64)
 if n>1: trans[1:]=(verdicts[1:]!=verdicts[:-1]).astype(np.int64)
 dpre=prefix_sum(durations); fpre=prefix_sum(fail); apre=prefix_sum(ass); epre=prefix_sum(exc); tpre=prefix_sum(trans)
 histlen=positions.astype(float); rstart=np.maximum(0,positions-RECENT_WINDOW); rlen=(positions-rstart).astype(float)
 lastf=np.maximum.accumulate(np.where(fail>0,allpos,-1)); lastt=np.maximum.accumulate(np.where(trans>0,allpos,-1)); pf=np.full(len(positions),-1,np.int64); pt=np.full(len(positions),-1,np.int64); has=positions>0
 pf[has]=lastf[positions[has]-1]; pt[has]=lastt[positions[has]-1]
 rmax=np.full(n,np.nan,float)
 for off in range(1,RECENT_WINDOW+1):
  if n>off:rmax[off:]=np.fmax(rmax[off:],durations[:-off])
 tmax=np.maximum.accumulate(durations); prev=np.maximum(positions-1,0)
 out={
  "REC_Age":(global_positions[positions]-global_positions[0]).astype(float),
  "REC_LastFailureAge":np.where(pf<0,-1.0,(positions-1-pf).astype(float)),
  "REC_LastTransitionAge":np.where(pt<0,-1.0,(positions-1-pt).astype(float)),
  "REC_RecentAvgExeTime":safe_divide(dpre[positions]-dpre[rstart],rlen),"REC_RecentMaxExeTime":np.where(has,rmax[positions],-1.0),
  "REC_RecentFailRate":safe_divide(fpre[positions]-fpre[rstart],rlen),"REC_RecentAssertRate":safe_divide(apre[positions]-apre[rstart],rlen),"REC_RecentExcRate":safe_divide(epre[positions]-epre[rstart],rlen),"REC_RecentTransitionRate":safe_divide(tpre[positions]-tpre[rstart],rlen),
  "REC_TotalAvgExeTime":safe_divide(dpre[positions],histlen),"REC_TotalMaxExeTime":np.where(has,tmax[prev],-1.0),"REC_TotalFailRate":safe_divide(fpre[positions],histlen),"REC_TotalAssertRate":safe_divide(apre[positions],histlen),"REC_TotalExcRate":safe_divide(epre[positions],histlen),"REC_TotalTransitionRate":safe_divide(tpre[positions],histlen),
  "REC_LastVerdict":np.where(has,verdicts[prev],-1).astype(float),"REC_LastExeTime":np.where(has,durations[prev],-1.0),
 }
 ff=np.empty(len(positions),float); ft=np.empty(len(positions),float); fe=np.flatnonzero(fail>0); te=np.flatnonzero(trans>0); fp=tp=0; pfb=set(); ptb=set()
 for j in np.argsort(positions,kind="mergesort"):
  p=int(positions[j])
  while fp<len(fe) and int(fe[fp])<p: pfb.add(int(builds[fe[fp]])); fp+=1
  while tp<len(te) and int(te[tp])<p: ptb.add(int(builds[te[tp]])); tp+=1
  ents=changed_entities_by_build.get(int(builds[p]),frozenset()); ff[j]=calculate_file_rate(pfb,ents,entity_changed_builds); ft[j]=calculate_file_rate(ptb,ents,entity_changed_builds)
 out["REC_MaxTestFileFailRate"]=ff; out["REC_MaxTestFileTransitionRate"]=ft; return out

def reconstruct_all(history,requested_rows,starts,changed_entities_by_build,entity_changed_builds):
 requested_rows=requested_rows.reset_index(drop=True); req_groups=requested_rows.groupby("Test",sort=False).indices; req_builds=requested_rows["Build"].to_numpy(np.int64)
 result={f:np.full(len(requested_rows),np.nan,np.float64) for f in REC_FEATURES}; filled=np.zeros(len(requested_rows),bool)
 ht=history["Test"].to_numpy(np.int64); hb=history["Build"].to_numpy(np.int64); hv=history["Verdict"].to_numpy(np.int64); hd=history["Duration"].to_numpy(np.float64); hg=history["GlobalBuildPosition"].to_numpy(np.int64)
 total=len(starts)-1; t0=time.perf_counter()
 for k in range(total):
  s,e=int(starts[k]),int(starts[k+1]); test=int(ht[s]); ridx=req_groups.get(test)
  if ridx is None:continue
  ridx=np.asarray(ridx,np.int64); gb=hb[s:e]; posmap={int(b):i for i,b in enumerate(gb)}; positions=np.array([posmap.get(int(b),-1) for b in req_builds[ridx]],np.int64)
  if (positions<0).any():raise RuntimeError(f"Requested model row absent from raw history for Test={test}")
  gr=reconstruct_group(gb,hv[s:e],hd[s:e],hg[s:e],positions,changed_entities_by_build,entity_changed_builds)
  for f in REC_FEATURES:result[f][ridx]=gr[f]
  filled[ridx]=True
  if (k+1)%500==0 or k+1==total:print("    REC reconstruction progress:",k+1,"/",total,"tests | reconstructed rows:",int(filled.sum()),"| elapsed seconds:",round(time.perf_counter()-t0,2))
 if not filled.all():raise RuntimeError("REC reconstruction did not fill every requested model row")
 out=requested_rows[["Build","Test"]].copy()
 for f in REC_FEATURES:
  if not np.isfinite(result[f]).all():raise RuntimeError(f"Reconstructed {f} contains non-finite values")
  out[f]=result[f]
 return out

def positive_probability(estimator,matrix):
 p=estimator.predict_proba(matrix); classes=np.asarray(estimator.classes_); cols=np.flatnonzero(classes==1)
 if len(cols)!=1:raise RuntimeError("Estimator does not expose exactly one class-1 probability")
 scores=p[:,int(cols[0])]
 if not np.isfinite(scores).all() or ((scores<0)|(scores>1)).any():raise RuntimeError("Invalid model probabilities")
 return scores.astype(float,copy=False)
def make_ranking(meta,technique,scores,ascending_score):
 r=meta.copy(); r["Technique"]=technique; r["Score"]=np.asarray(scores,float)
 if len(r)!=EXPECTED_MODEL_EVAL_ROWS or not np.isfinite(r["Score"].to_numpy(float)).all():raise RuntimeError(f"{technique} ranking input invalid")
 r=r.sort_values(["Build","Score","Test"],ascending=[True,bool(ascending_score),True],kind="mergesort").reset_index(drop=True); r["Rank"]=r.groupby("Build",sort=False).cumcount().add(1).astype("int64")
 return r[["ProjectNumber","Project","ProjectSlug","ConditionKey","NoisePercent","RepetitionSeed","Technique","Build","Test","Rank","Score","CleanVerdict","CleanFailure","Duration"]]
def condition_metrics(rankings,failing_builds):
 records=[]; fr=rankings.loc[rankings["Build"].isin(failing_builds)].copy()
 for (tech,build),g in fr.groupby(["Technique","Build"],sort=False):
  g=g.sort_values("Rank",kind="mergesort"); f=g["CleanFailure"].to_numpy(np.int8); d=g["Duration"].to_numpy(float)
  if int(f.sum())<=0:raise RuntimeError("Failing evaluation build has no failures")
  records.append({"ProjectNumber":PROJECT_NUMBER,"Project":PROJECT_NAME,"ProjectSlug":PROJECT_SLUG,"ConditionKey":str(g["ConditionKey"].iloc[0]),"NoisePercent":int(g["NoisePercent"].iloc[0]),"RepetitionSeed":int(g["RepetitionSeed"].iloc[0]),"Technique":tech,"Build":int(build),"Tests":int(len(g)),"Failures":int(f.sum()),"TotalDuration":float(d.sum()),"APFDc":calculate_apfdc(f,d),"APFD":calculate_apfd(f)})
 bm=pd.DataFrame(records); prs=[]
 for tech,g in bm.groupby("Technique",sort=False):prs.append({"ProjectNumber":PROJECT_NUMBER,"Project":PROJECT_NAME,"ProjectSlug":PROJECT_SLUG,"ConditionKey":str(g["ConditionKey"].iloc[0]),"NoisePercent":int(g["NoisePercent"].iloc[0]),"RepetitionSeed":int(g["RepetitionSeed"].iloc[0]),"Technique":tech,"EvaluationBuilds":EXPECTED_EVALUATION_BUILDS,"ScoredFailingBuilds":int(g["Build"].nunique()),"EvaluationRows":EXPECTED_MODEL_EVAL_ROWS,"EvaluationFailures":EXPECTED_MODEL_EVAL_FAILURES,"MeanAPFDc":float(g["APFDc"].mean()),"MedianAPFDc":float(g["APFDc"].median()),"MeanAPFD":float(g["APFD"].mean()),"MedianAPFD":float(g["APFD"].median())})
 return bm,pd.DataFrame(prs)

# --------------------------------------------------------------------------------------------------
# 4. VERIFY FROZEN UPSTREAM STATE
# --------------------------------------------------------------------------------------------------
required=[REGISTRY_PATH,FROZEN_SOURCE_MANIFEST_PATH,SELECTION_CHECKPOINT_PATH,REC_CHECKPOINT_PATH,NOISE_PLAN_CHECKPOINT_PATH,RUNTIME_CHECKPOINT_PATH,STEP4A_STATUS_PATH,STEP4A_REPORT_PATH,BUILD_ENTITY_PATH,CLEAN_ANCHOR_OFFSETS_PATH,FROZEN_GLOBAL_BUILD_ORDER_PATH,RAW_TRAINING_COHORT_PATH,RAW_EVALUATION_COHORT_PATH,MODEL_TRAINING_COHORT_PATH,MODEL_EVALUATION_COHORT_PATH,MODEL_RAW_TRAIN_LINK_PATH,MODEL_RAW_EVAL_LINK_PATH,RNG_MANIFEST_PATH,CONDITION_PLAN_PATH,PREDICTOR_CONTRACT_PATH]
missing=[str(p) for p in required if not Path(p).is_file()]
if missing:raise FileNotFoundError("Required Project 24 Step 4B inputs missing:\n"+"\n".join(missing))
selection_sha=sha256_file(SELECTION_CHECKPOINT_PATH); rec_sha=sha256_file(REC_CHECKPOINT_PATH); noise_sha=sha256_file(NOISE_PLAN_CHECKPOINT_PATH); runtime_sha=sha256_file(RUNTIME_CHECKPOINT_PATH)
for label,expected,actual in [("selection",EXPECTED_SELECTION_CHECKPOINT_SHA256,selection_sha),("REC",EXPECTED_REC_CHECKPOINT_SHA256,rec_sha),("noise",EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,noise_sha),("runtime",EXPECTED_RUNTIME_CHECKPOINT_SHA256,runtime_sha)]:
 if actual!=expected:raise RuntimeError(f"{label} checkpoint SHA differs: expected={expected}, actual={actual}")
selection=load_json(SELECTION_CHECKPOINT_PATH); rec_cp=load_json(REC_CHECKPOINT_PATH); noise_cp=load_json(NOISE_PLAN_CHECKPOINT_PATH); runtime_cp=load_json(RUNTIME_CHECKPOINT_PATH); step4a_status=load_json(STEP4A_STATUS_PATH); step4a_report=load_json(STEP4A_REPORT_PATH)
for label,payload,status in [("selection",selection,EXPECTED_SELECTION_STATUS),("REC",rec_cp,EXPECTED_STEP2B_STATUS),("noise",noise_cp,EXPECTED_STEP3A_STATUS),("runtime",runtime_cp,EXPECTED_STEP4A_STATUS),("Step4A status",step4a_status,EXPECTED_STEP4A_STATUS),("Step4A report",step4a_report,EXPECTED_STEP4A_STATUS)]:
 if payload.get("Status")!=status:raise RuntimeError(f"{label} status differs")
if rec_cp.get("ImplementationVersion")!=EXPECTED_REC_IMPLEMENTATION:raise RuntimeError(f"Frozen REC implementation differs: {rec_cp.get('ImplementationVersion')}")
if runtime_cp.get("Step3ACodeRevision")!=EXPECTED_STEP3A_CODE_REVISION:raise RuntimeError("Frozen Step 3A code revision differs")
if not runtime_cp.get("ReadyForTwoConditionSmokeTest",False):raise RuntimeError("Step 4A does not authorise smoke test")
upstream_failures=verify_manifest(rec_cp.get("OutputManifest",[]),"REC")+verify_manifest(noise_cp.get("OutputManifest",[]),"noise")+verify_manifest(runtime_cp.get("RuntimeOutputManifest",[]),"runtime")
if upstream_failures:display(pd.DataFrame(upstream_failures));raise RuntimeError("Frozen upstream manifest failure")
if runtime_cp.get("ActiveReservations")!=EXPECTED_ACTIVE_RESERVATIONS or runtime_cp.get("RuntimePriorityRule")!=EXPECTED_RUNTIME_PRIORITY_RULE:raise RuntimeError("Frozen reservation/runtime-priority state differs")
registry_sha_before=sha256_file(REGISTRY_PATH)
if registry_sha_before!=EXPECTED_REGISTRY_SHA256:raise RuntimeError("Registry SHA differs")
registry=pd.read_csv(REGISTRY_PATH,dtype=str).fillna(""); pn=resolve_col(registry.columns,"projectnumber","registry ProjectNumber"); pc=resolve_col(registry.columns,"project","registry Project"); sc=resolve_col(registry.columns,"status","registry Status"); nums=pd.to_numeric(registry[pn],errors="raise").astype(int)
if len(registry)!=23 or sorted(nums.tolist())!=list(range(1,24)) or not registry[sc].eq(EXPECTED_COMPLETE_STATUS).all():raise RuntimeError("Registry is not frozen Projects 1-23")
for n,identity in REQUIRED_REGISTERED_IDENTITIES.items():
 rows=registry.loc[nums.eq(n)]
 if len(rows)!=1 or str(rows.iloc[0][pc])!=identity:raise RuntimeError(f"Frozen Project {n} identity differs")
if nums.eq(24).any() or registry[pc].eq(PROJECT_NAME).any():raise RuntimeError("Project 24 already registered")
frozen_source=pd.read_csv(FROZEN_SOURCE_MANIFEST_PATH); cur=[]
for r in frozen_source.itertuples(index=False):
 p=SOURCE_DIR/str(r.RelativePath)
 if not p.is_file():raise FileNotFoundError(str(p))
 cur.append({"RelativePath":str(r.RelativePath),"SizeBytes":int(p.stat().st_size),"SHA256":sha256_file(p)})
source_sha_before=source_root_hash(pd.DataFrame(cur))
if source_sha_before!=EXPECTED_SOURCE_ROOT_SHA256:raise RuntimeError("Source root differs")
raw_manifest_before=directory_manifest(FULL_RAW_RESULT_ROOT); raw_root_before=directory_root_hash(raw_manifest_before); raw_existed_before=FULL_RAW_RESULT_ROOT.exists()
if SMOKE_CHECKPOINT_PATH.is_file() and load_json(SMOKE_CHECKPOINT_PATH).get("Status")==STEP4B_STATUS:raise RuntimeError("Step 4B already frozen successfully; do not rerun")
if SMOKE_ROOT.exists():shutil.rmtree(SMOKE_ROOT)
for p in [SMOKE_CHECKPOINT_PATH,STEP4B_STATUS_PATH]:
 if p.exists():p.unlink()
SMOKE_ROOT.mkdir(parents=True,exist_ok=True)

# --------------------------------------------------------------------------------------------------
# 5. LOAD FIXED COHORTS / LINKS / PLAN / ANCHOR
# --------------------------------------------------------------------------------------------------
print("\nLoading frozen Project 24 cohorts and contracts.")
rt=pd.read_parquet(RAW_TRAINING_COHORT_PATH).sort_values("RawTrainingRowOrder",kind="mergesort").reset_index(drop=True); re=pd.read_parquet(RAW_EVALUATION_COHORT_PATH).sort_values("RawEvaluationRowOrder",kind="mergesort").reset_index(drop=True)
mt=pd.read_parquet(MODEL_TRAINING_COHORT_PATH).sort_values("ModelTrainingRowOrder",kind="mergesort").reset_index(drop=True); me=pd.read_parquet(MODEL_EVALUATION_COHORT_PATH).sort_values("ModelEvaluationRowOrder",kind="mergesort").reset_index(drop=True)
lt=pd.read_parquet(MODEL_RAW_TRAIN_LINK_PATH).sort_values("ModelTrainingRowOrder",kind="mergesort").reset_index(drop=True); le=pd.read_parquet(MODEL_RAW_EVAL_LINK_PATH).sort_values("ModelEvaluationRowOrder",kind="mergesort").reset_index(drop=True)
plan=pd.read_csv(CONDITION_PLAN_PATH); pred_contract=pd.read_csv(PREDICTOR_CONTRACT_PATH).sort_values("PredictorOrder",kind="mergesort"); anchor=pd.read_parquet(CLEAN_ANCHOR_OFFSETS_PATH); gbo=pd.read_csv(FROZEN_GLOBAL_BUILD_ORDER_PATH); be=pd.read_csv(BUILD_ENTITY_PATH,compression="gzip")
for label,frame,expected in [("raw train",rt,EXPECTED_RAW_TRAIN_ROWS),("raw eval",re,EXPECTED_RAW_EVAL_ROWS),("model train",mt,EXPECTED_MODEL_TRAIN_ROWS),("model eval",me,EXPECTED_MODEL_EVAL_ROWS),("train links",lt,EXPECTED_MODEL_TRAIN_ROWS),("eval links",le,EXPECTED_MODEL_EVAL_ROWS),("anchor",anchor,EXPECTED_MODEL_ROWS),("global build order",gbo,EXPECTED_BUILDS)]:
 if len(frame)!=expected:raise RuntimeError(f"{label} rows differ: {len(frame)} vs {expected}")
for f in [rt,re,mt,me]:
 for c in ["Build","Test","Verdict"]:f[c]=parse_int(f[c],c)
for f in [rt,re]:
 f["InferredTestOrder"]=parse_int(f["InferredTestOrder"],"InferredTestOrder"); f["Duration"]=pd.to_numeric(f["Duration"],errors="coerce")
 if f["Duration"].isna().any() or not np.isfinite(f["Duration"].to_numpy(float)).all() or f["Duration"].lt(0).any():raise RuntimeError("Invalid raw durations")
lt["RawTrainingRowOrder"]=parse_int(lt["RawTrainingRowOrder"],"train link raw order"); le["RawEvaluationRowOrder"]=parse_int(le["RawEvaluationRowOrder"],"eval link raw order")
train_raw_idx=lt["RawTrainingRowOrder"].to_numpy(np.int64)-1; eval_raw_idx=le["RawEvaluationRowOrder"].to_numpy(np.int64)-1
if not np.array_equal(rt.iloc[train_raw_idx][["Build","Test"]].to_numpy(np.int64),mt[["Build","Test"]].to_numpy(np.int64)):raise RuntimeError("Training model/raw links differ")
if not np.array_equal(re.iloc[eval_raw_idx][["Build","Test"]].to_numpy(np.int64),me[["Build","Test"]].to_numpy(np.int64)):raise RuntimeError("Evaluation model/raw links differ")
clean_rt=rt["Verdict"].to_numpy(np.int16); clean_re=re["Verdict"].to_numpy(np.int16); clean_mt=mt["Verdict"].to_numpy(np.int16); clean_me=me["Verdict"].to_numpy(np.int16); y_eval=(clean_me!=0).astype(np.int8)
if int((clean_mt!=0).sum())!=EXPECTED_MODEL_TRAIN_FAILURES or int(y_eval.sum())!=EXPECTED_MODEL_EVAL_FAILURES:raise RuntimeError("Clean failure counts differ")
eval_duration=re.iloc[eval_raw_idx]["Duration"].to_numpy(float); failing_builds=sorted(me.loc[y_eval==1,"Build"].astype(int).unique())
if len(failing_builds)!=EXPECTED_FAILING_EVAL_BUILDS:raise RuntimeError("Failing eval build count differs")
predictors=pred_contract["Predictor"].astype(str).tolist()
if len(predictors)!=151 or len(set(predictors))!=151 or any(f not in predictors for f in REC_FEATURES):raise RuntimeError("Predictor contract differs")
# Critical Project-24 schema fix propagated from Step 3A V2.
if not {"GlobalBuildOrder","BuildID"}.issubset(gbo.columns):raise RuntimeError(f"Frozen global build-order schema differs: {list(gbo.columns)}")
gbo["GlobalBuildOrder"]=parse_int(gbo["GlobalBuildOrder"],"GlobalBuildOrder"); gbo["BuildID"]=parse_int(gbo["BuildID"],"BuildID"); gbo=gbo.sort_values("GlobalBuildOrder",kind="mergesort").reset_index(drop=True)
if not np.array_equal(gbo["GlobalBuildOrder"].to_numpy(np.int64),np.arange(1,EXPECTED_BUILDS+1,dtype=np.int64)):raise RuntimeError("Global build order is not exactly 1..4286")
global_pos={int(r.BuildID):int(r.GlobalBuildOrder)-1 for r in gbo.itertuples(index=False)}
bec=resolve_col(be.columns,"build","build-entity Build"); eic=resolve_col(be.columns,"entityid","build-entity EntityId"); be[bec]=parse_int(be[bec],"build_entity.Build"); be[eic]=parse_int(be[eic],"build_entity.EntityId")
changed_by_build={int(b):frozenset(g[eic].astype(int).unique().tolist()) for b,g in be.groupby(bec,sort=False)}; builds_by_entity={int(e):frozenset(g[bec].astype(int).unique().tolist()) for e,g in be.groupby(eic,sort=False)}
model_all=pd.concat([mt[["Build","Test"]+REC_FEATURES],me[["Build","Test"]+REC_FEATURES]],ignore_index=True); model_all["_Row"]=np.arange(len(model_all),dtype=np.int64)
anchor_keyed=model_all[["_Row","Build","Test"]].merge(anchor[["Build","Test"]+REC_FEATURES],on=["Build","Test"],how="left",validate="one_to_one",sort=False).sort_values("_Row",kind="mergesort")
if len(anchor_keyed)!=EXPECTED_MODEL_ROWS or anchor_keyed[REC_FEATURES].isna().any().any():raise RuntimeError("Clean anchor did not match every fixed model row")
anchor_matrix=anchor_keyed[REC_FEATURES].to_numpy(np.float64); clean_rec_matrix=model_all[REC_FEATURES].to_numpy(np.float64); model_all=model_all.drop(columns="_Row")
print("Converting fixed predictor cohorts to numeric matrices.")
train_base=mt[predictors].apply(pd.to_numeric,errors="coerce").to_numpy(np.float64); eval_base=me[predictors].apply(pd.to_numeric,errors="coerce").to_numpy(np.float64)
if not np.isfinite(train_base).all() or not np.isfinite(eval_base).all():raise RuntimeError("Frozen predictor matrices contain non-finite values")
pidx={p:i for i,p in enumerate(predictors)}; ridx={f:i for i,f in enumerate(REC_FEATURES)}; dep_rec_idx=[ridx[f] for f in VERDICT_DEPENDENT_REC]; indep_rec_idx=[ridx[f] for f in VERDICT_INDEPENDENT_REC]; indep_pred_idx=[pidx[f] for f in VERDICT_INDEPENDENT_REC]
# One static full raw-history ordering; verdicts are replaced per condition.
trh=rt[["Build","Test","Duration","InferredTestOrder"]].copy(); trh["_Training"]=True; trh["_SourceIndex"]=np.arange(len(trh),dtype=np.int64)
erh=re[["Build","Test","Duration","InferredTestOrder"]].copy(); erh["_Training"]=False; erh["_SourceIndex"]=np.arange(len(erh),dtype=np.int64)
history=pd.concat([trh,erh],ignore_index=True).sort_values(["Test","InferredTestOrder"],kind="mergesort").reset_index(drop=True)
if len(history)!=EXPECTED_RAW_ROWS:raise RuntimeError("Combined history row count differs")
history["GlobalBuildPosition"]=history["Build"].map(global_pos)
if history["GlobalBuildPosition"].isna().any():raise RuntimeError("Raw history has build absent from frozen global order")
history["GlobalBuildPosition"]=history["GlobalBuildPosition"].astype("int64"); ht=history["Test"].to_numpy(np.int64); starts=np.r_[np.array([0],np.int64),(np.flatnonzero(ht[1:]!=ht[:-1])+1).astype(np.int64),np.array([len(history)],np.int64)]
htrain=history["_Training"].to_numpy(bool); hsource=history["_SourceIndex"].to_numpy(np.int64); history["Verdict"]=np.zeros(len(history),np.int16)
del trh,erh,ht; gc.collect()
eval_meta_base=pd.DataFrame({"Build":me["Build"].to_numpy(np.int64),"Test":me["Test"].to_numpy(np.int64),"CleanVerdict":clean_me,"CleanFailure":y_eval,"Duration":eval_duration})
# Only seed-1 row group is required for the smoke test.
rng_file=pq.ParquetFile(RNG_MANIFEST_PATH)
if int(rng_file.metadata.num_rows)!=EXPECTED_RNG_ROWS or int(rng_file.metadata.num_row_groups)!=EXPECTED_RNG_ROW_GROUPS:raise RuntimeError("RNG manifest metadata differs")
rng1=rng_file.read_row_group(0,columns=["RepetitionSeed","RawTrainingRowOrder","FlipUniform","SampledFailureSubtype"]).to_pandas()
if len(rng1)!=EXPECTED_RAW_TRAIN_ROWS or rng1["RepetitionSeed"].nunique()!=1 or int(rng1["RepetitionSeed"].iloc[0])!=1 or int(rng1["RawTrainingRowOrder"].iloc[0])!=1 or int(rng1["RawTrainingRowOrder"].iloc[-1])!=EXPECTED_RAW_TRAIN_ROWS:raise RuntimeError("Seed-1 RNG row group differs")
flip_uniform=rng1["FlipUniform"].to_numpy(np.float64); sampled_subtype=rng1["SampledFailureSubtype"].to_numpy(np.int16); del rng1; gc.collect()
raw_train_hash_before=sha256_file(RAW_TRAINING_COHORT_PATH); raw_eval_hash_before=sha256_file(RAW_EVALUATION_COHORT_PATH); model_train_hash_before=sha256_file(MODEL_TRAINING_COHORT_PATH); model_eval_hash_before=sha256_file(MODEL_EVALUATION_COHORT_PATH)

# --------------------------------------------------------------------------------------------------
# 6. EXECUTE ONE SMOKE CONDITION
# --------------------------------------------------------------------------------------------------
def execute_condition(condition_key):
 started=time.perf_counter(); match=plan.loc[plan["ConditionID"].eq(condition_key)]
 if len(match)!=1:raise RuntimeError(f"{condition_key}: condition-plan row count differs")
 pr=match.iloc[0]; noise=int(pr.NoisePercent); seed=int(pr.RepetitionSeed)
 if seed!=1:raise RuntimeError("Smoke seed differs")
 cdir=SMOKE_ROOT/condition_key; cdir.mkdir(parents=True,exist_ok=True)
 print("\nExecuting:",condition_key)
 mask=flip_uniform<(noise/100.0); noisy_rt=clean_rt.copy(); p2f=mask&(clean_rt==0); f2p=mask&(clean_rt!=0); noisy_rt[p2f]=sampled_subtype[p2f]; noisy_rt[f2p]=0; noisy_mt=noisy_rt[train_raw_idx]
 nflip=int(mask.sum()); mlchg=int((noisy_mt!=clean_mt).sum()); trainfail=int((noisy_mt!=0).sum())
 actual_hashes=(sha256_array(mask.astype(np.uint8),"u1"),sha256_array(noisy_rt,"<i2"),sha256_array(noisy_mt,"<i2")); expected_hashes=(str(pr.FlipMaskSHA256),str(pr.NoisyRawVerdictSHA256),str(pr.NoisyModelVerdictSHA256))
 if nflip!=int(pr.NumberFlipped) or mlchg!=int(pr.ModelLabelChanges) or trainfail!=int(pr.NoisyModelFailures) or actual_hashes!=expected_hashes:raise RuntimeError(f"{condition_key}: exact frozen noise reproduction failed")
 hv=np.empty(len(history),np.int16); tp=np.flatnonzero(htrain); ep=np.flatnonzero(~htrain); hv[tp]=noisy_rt[hsource[tp]]; hv[ep]=clean_re[hsource[ep]]; history["Verdict"]=hv
 rec_start=time.perf_counter(); reconstructed=reconstruct_all(history,model_all[["Build","Test"]],starts,changed_by_build,builds_by_entity); rec_seconds=time.perf_counter()-rec_start
 reconstructed_matrix=reconstructed[REC_FEATURES].to_numpy(np.float64); anchored=reconstructed_matrix+anchor_matrix
 indep_mismatch=int((~np.isclose(anchored[:,indep_rec_idx],clean_rec_matrix[:,indep_rec_idx],rtol=DIRECT_RTOL,atol=DIRECT_ATOL,equal_nan=False)).sum())
 dep_all=anchored[:,dep_rec_idx]; clean_dep=clean_rec_matrix[:,dep_rec_idx]; dep_changes=int((~np.isclose(dep_all,clean_dep,rtol=DIRECT_RTOL,atol=DIRECT_ATOL,equal_nan=False)).sum())
 Xtr=train_base.copy(); Xev=eval_base.copy(); dtr=dep_all[:EXPECTED_MODEL_TRAIN_ROWS]; dev=dep_all[EXPECTED_MODEL_TRAIN_ROWS:]
 for j,f in enumerate(VERDICT_DEPENDENT_REC):Xtr[:,pidx[f]]=dtr[:,j];Xev[:,pidx[f]]=dev[:,j]
 # Explicit frozen independent REC preservation.
 for f in VERDICT_INDEPENDENT_REC:Xtr[:,pidx[f]]=train_base[:,pidx[f]];Xev[:,pidx[f]]=eval_base[:,pidx[f]]
 indep_changes=int((~np.isclose(Xtr[:,indep_pred_idx],train_base[:,indep_pred_idx],rtol=0,atol=0)).sum()+(~np.isclose(Xev[:,indep_pred_idx],eval_base[:,indep_pred_idx],rtol=0,atol=0)).sum())
 if noise==0 and any([nflip,mlchg,dep_changes,indep_changes,indep_mismatch]):raise RuntimeError("0% smoke condition failed exact clean invariance")
 if noise>0 and (nflip<=0 or mlchg<=0 or dep_changes<=0 or indep_changes!=0 or indep_mismatch!=0):raise RuntimeError("50% smoke condition failed expected noise/REC behavior")
 med=[]
 for j,p in enumerate(predictors):
  col=Xtr[:,j]; finite=col[np.isfinite(col)]
  if len(finite)==0:raise RuntimeError(f"{p}: no finite training values")
  m=float(np.median(finite)); med.append({"ProjectNumber":24,"Project":PROJECT_NAME,"ProjectSlug":PROJECT_SLUG,"ConditionKey":condition_key,"NoisePercent":noise,"RepetitionSeed":seed,"PredictorOrder":j+1,"Predictor":p,"TrainingMedian":m}); bad=~np.isfinite(col); bade=~np.isfinite(Xev[:,j]); Xtr[bad,j]=m; Xev[bade,j]=m
 if not np.isfinite(Xtr).all() or not np.isfinite(Xev).all():raise RuntimeError("Non-finite predictors remain after condition imputation")
 med=pd.DataFrame(med); ytr=(noisy_mt!=0).astype(np.int8)
 if sorted(np.unique(ytr).tolist())!=[0,1]:raise RuntimeError("Training labels do not contain both classes")
 meta=eval_meta_base.copy(); meta["ProjectNumber"]=24; meta["Project"]=PROJECT_NAME; meta["ProjectSlug"]=PROJECT_SLUG; meta["ConditionKey"]=condition_key; meta["NoisePercent"]=noise; meta["RepetitionSeed"]=seed
 fit_records=[]; ranks=[]; models=create_models(seed)
 for tech in ML_TECHNIQUES:
  print("  Fitting:",tech); est=models[tech]; t=time.perf_counter(); fit_status="PASS_MODEL_FIT"; fit_error=""
  try:
   est.fit(Xtr,ytr); sec=time.perf_counter()-t; scores=positive_probability(est,Xev)
  except Exception as error:
   sec=time.perf_counter()-t; fit_status="FAIL_MODEL_FIT"; fit_error=repr(error)
   fit_records.append({"ProjectNumber":24,"Project":PROJECT_NAME,"ProjectSlug":PROJECT_SLUG,"ConditionKey":condition_key,"NoisePercent":noise,"RepetitionSeed":seed,"Technique":tech,"TrainingRows":EXPECTED_MODEL_TRAIN_ROWS,"TrainingFailures":trainfail,"Predictors":EXPECTED_PREDICTORS,"FitSeconds":float(sec),"ClassesJSON":"[]","Status":fit_status,"Error":fit_error})
   raise RuntimeError(f"{condition_key}: {tech} fitting failed: {error!r}") from error
  fit_records.append({"ProjectNumber":24,"Project":PROJECT_NAME,"ProjectSlug":PROJECT_SLUG,"ConditionKey":condition_key,"NoisePercent":noise,"RepetitionSeed":seed,"Technique":tech,"TrainingRows":EXPECTED_MODEL_TRAIN_ROWS,"TrainingFailures":trainfail,"Predictors":EXPECTED_PREDICTORS,"FitSeconds":float(sec),"ClassesJSON":json.dumps([int(value) for value in np.asarray(est.classes_).tolist()]),"Status":fit_status,"Error":fit_error}); ranks.append(make_ranking(meta,tech,scores,False))
 fits=pd.DataFrame(fit_records)
 random_scores=np.empty(EXPECTED_MODEL_EVAL_ROWS,np.float64)
 for build,ii in meta.groupby("Build",sort=False).indices.items():
  ii=np.asarray(ii,np.int64); ordered=ii[np.argsort(meta.iloc[ii]["Test"].to_numpy(np.int64),kind="mergesort")]; random_scores[ordered]=np.random.default_rng(deterministic_random_build_seed(seed,int(build))).random(len(ordered))
 ranks.append(make_ranking(meta,"Random",random_scores,False)); ranks.append(make_ranking(meta,"LatestFail",-Xev[:,pidx["REC_LastFailureAge"]],False)); ranks.append(make_ranking(meta,"QTF-Avg",Xev[:,pidx["REC_TotalAvgExeTime"]],True)); ranks=pd.concat(ranks,ignore_index=True)
 if len(ranks)!=EXPECTED_RANKING_ROWS_PER_CONDITION or set(ranks["Technique"])!=set(ALL_TECHNIQUES):raise RuntimeError("Ranking output contract differs")
 bm,prs=condition_metrics(ranks,failing_builds)
 if len(bm)!=EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION or len(prs)!=7 or len(fits)!=4 or len(med)!=151:raise RuntimeError("Smoke output row-count contract differs")
 if fits["Status"].ne("PASS_MODEL_FIT").any():raise RuntimeError("Model fit failure")
 vals=bm[["APFDc","APFD"]].to_numpy(float); pvals=prs[["MeanAPFDc","MedianAPFDc","MeanAPFD","MedianAPFD"]].to_numpy(float)
 if not np.isfinite(vals).all() or ((vals<0)|(vals>1)).any() or not np.isfinite(pvals).all() or ((pvals<0)|(pvals>1)).any():raise RuntimeError("Metric values invalid")
 seconds=time.perf_counter()-started
 audit=pd.DataFrame([{"ProjectNumber":24,"Project":PROJECT_NAME,"ProjectSlug":PROJECT_SLUG,"ConditionKey":condition_key,"NoisePercent":noise,"RepetitionSeed":seed,"RawTrainingRows":EXPECTED_RAW_TRAIN_ROWS,"NumberFlipped":nflip,"ExpectedNumberFlipped":int(pr.NumberFlipped),"RealisedNoisePercent":float(100*nflip/EXPECTED_RAW_TRAIN_ROWS),"PassToFailure":int(p2f.sum()),"FailureToPass":int(f2p.sum()),"ModelTrainingRows":EXPECTED_MODEL_TRAIN_ROWS,"ModelLabelChanges":mlchg,"ExpectedModelLabelChanges":int(pr.ModelLabelChanges),"TrainingFailures":trainfail,"ExpectedTrainingFailures":int(pr.NoisyModelFailures),"DependentRECChanges":dep_changes,"IndependentRECChanges":indep_changes,"IndependentReconstructionMismatches":indep_mismatch,"ReconstructedRows":len(reconstructed),"Predictors":151,"MLFits":len(fits),"RankingRows":len(ranks),"BuildMetricRows":len(bm),"ProjectRunRows":len(prs),"TrainingMedianRows":len(med),"ExpectedFlipMaskSHA256":expected_hashes[0],"ActualFlipMaskSHA256":actual_hashes[0],"ExpectedNoisyRawVerdictSHA256":expected_hashes[1],"ActualNoisyRawVerdictSHA256":actual_hashes[1],"ExpectedNoisyModelVerdictSHA256":expected_hashes[2],"ActualNoisyModelVerdictSHA256":actual_hashes[2],"RECSeconds":float(rec_seconds),"ConditionSeconds":float(seconds),"Status":CONDITION_STATUS}])
 atomic_csv(cdir/"rankings.csv.gz",ranks,compression="gzip"); atomic_csv(cdir/"build_metrics.csv",bm); atomic_csv(cdir/"project_runs.csv",prs); atomic_csv(cdir/"model_fits.csv",fits); atomic_csv(cdir/"training_medians.csv",med); atomic_csv(cdir/"condition_audit.csv",audit)
 print("  Completed:",condition_key); print("  Raw flips:",nflip,"| model-label changes:",mlchg,"| dependent REC changes:",dep_changes); print("  Training failures:",trainfail,"| condition seconds:",round(seconds,2))
 del mask,noisy_rt,noisy_mt,hv,reconstructed,reconstructed_matrix,anchored,Xtr,Xev,models;gc.collect()
 return {"ConditionKey":condition_key,"NoisePercent":noise,"RepetitionSeed":seed,"Rankings":ranks,"BuildMetrics":bm,"ProjectRuns":prs,"ModelFits":fits,"TrainingMedians":med,"ConditionAudit":audit}

# --------------------------------------------------------------------------------------------------
# 7. RUN BOTH SMOKE CONDITIONS + BASELINE INVARIANCE
# --------------------------------------------------------------------------------------------------
smoke_started=time.perf_counter(); results=[execute_condition(k) for k in SMOKE_CONDITION_IDS]
inventory=pd.DataFrame([{"ConditionKey":x["ConditionKey"],"NoisePercent":x["NoisePercent"],"RepetitionSeed":x["RepetitionSeed"],"Status":CONDITION_STATUS} for x in results]); audit=pd.concat([x["ConditionAudit"] for x in results],ignore_index=True); project_runs=pd.concat([x["ProjectRuns"] for x in results],ignore_index=True); build_metrics=pd.concat([x["BuildMetrics"] for x in results],ignore_index=True); model_fits=pd.concat([x["ModelFits"] for x in results],ignore_index=True)
binv=[]
for tech in ["Random","QTF-Avg"]:
 z=results[0]["Rankings"].loc[results[0]["Rankings"]["Technique"].eq(tech)].sort_values(["Build","Test"],kind="mergesort").reset_index(drop=True); n=results[1]["Rankings"].loc[results[1]["Rankings"]["Technique"].eq(tech)].sort_values(["Build","Test"],kind="mergesort").reset_index(drop=True)
 same=bool(z[["Build","Test"]].equals(n[["Build","Test"]])); sm=EXPECTED_MODEL_EVAL_ROWS if not same else int((~np.isclose(z["Score"].to_numpy(float),n["Score"].to_numpy(float),rtol=0,atol=0)).sum()); rm=EXPECTED_MODEL_EVAL_ROWS if not same else int((z["Rank"].to_numpy(np.int64)!=n["Rank"].to_numpy(np.int64)).sum())
 binv.append({"Technique":tech,"Rows":len(z),"SameBuildTestKeys":same,"ScoreMismatches":sm,"RankMismatches":rm,"Pass":len(z)==EXPECTED_MODEL_EVAL_ROWS and len(n)==EXPECTED_MODEL_EVAL_ROWS and same and sm==0 and rm==0})
binv=pd.DataFrame(binv); baseline_fail=int((~binv["Pass"]).sum())
if baseline_fail:display(binv.loc[~binv["Pass"]]);raise RuntimeError("Random/QTF-Avg baseline invariance failed")
atomic_csv(SMOKE_INVENTORY_PATH,inventory);atomic_csv(SMOKE_AUDIT_PATH,audit);atomic_csv(SMOKE_PROJECT_RUNS_PATH,project_runs);atomic_csv(SMOKE_BUILD_METRICS_PATH,build_metrics);atomic_csv(SMOKE_MODEL_FITS_PATH,model_fits);atomic_csv(SMOKE_BASELINE_INVARIANCE_PATH,binv)

# --------------------------------------------------------------------------------------------------
# 8. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------
raw_train_hash_after=sha256_file(RAW_TRAINING_COHORT_PATH); raw_eval_hash_after=sha256_file(RAW_EVALUATION_COHORT_PATH); model_train_hash_after=sha256_file(MODEL_TRAINING_COHORT_PATH); model_eval_hash_after=sha256_file(MODEL_EVALUATION_COHORT_PATH); registry_sha_after=sha256_file(REGISTRY_PATH)
cur2=[]
for r in frozen_source.itertuples(index=False):
 p=SOURCE_DIR/str(r.RelativePath);cur2.append({"RelativePath":str(r.RelativePath),"SizeBytes":int(p.stat().st_size),"SHA256":sha256_file(p)})
source_sha_after=source_root_hash(pd.DataFrame(cur2)); raw_manifest_after=directory_manifest(FULL_RAW_RESULT_ROOT); full_raw_unchanged=FULL_RAW_RESULT_ROOT.exists()==raw_existed_before and directory_root_hash(raw_manifest_after)==raw_root_before and len(raw_manifest_after)==len(raw_manifest_before)
zero=audit.loc[audit["NoisePercent"].eq(0)].iloc[0]; positive=audit.loc[audit["NoisePercent"].eq(50)].iloc[0]; validation=[]; C=lambda c,e,a,p:add_check(validation,c,e,a,p)
C("Step 4A status",EXPECTED_STEP4A_STATUS,step4a_status.get("Status"),step4a_status.get("Status")==EXPECTED_STEP4A_STATUS);C("Runtime checkpoint SHA-256",EXPECTED_RUNTIME_CHECKPOINT_SHA256,runtime_sha,runtime_sha==EXPECTED_RUNTIME_CHECKPOINT_SHA256);C("Noise-plan checkpoint SHA-256",EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,noise_sha,noise_sha==EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256);C("REC checkpoint SHA-256",EXPECTED_REC_CHECKPOINT_SHA256,rec_sha,rec_sha==EXPECTED_REC_CHECKPOINT_SHA256);C("Selection checkpoint SHA-256",EXPECTED_SELECTION_CHECKPOINT_SHA256,selection_sha,selection_sha==EXPECTED_SELECTION_CHECKPOINT_SHA256);C("Source root SHA-256",EXPECTED_SOURCE_ROOT_SHA256,source_sha_after,source_sha_after==EXPECTED_SOURCE_ROOT_SHA256)
C("Smoke conditions",2,len(inventory),len(inventory)==2);C("Smoke condition keys",SMOKE_CONDITION_IDS,inventory["ConditionKey"].tolist(),inventory["ConditionKey"].tolist()==SMOKE_CONDITION_IDS);C("Condition statuses",CONDITION_STATUS,sorted(inventory["Status"].unique()),set(inventory["Status"])=={CONDITION_STATUS});C("Condition-audit rows",2,len(audit),len(audit)==2);C("Total ML fits",8,len(model_fits),len(model_fits)==8);C("Model-fit failures",0,int(model_fits["Status"].ne("PASS_MODEL_FIT").sum()),not model_fits["Status"].ne("PASS_MODEL_FIT").any())
ranking_total=sum(len(x["Rankings"]) for x in results);C("Total ranking rows",263956,ranking_total,ranking_total==263956);C("Total build-metric rows",238,len(build_metrics),len(build_metrics)==238);C("Total project-run rows",14,len(project_runs),len(project_runs)==14);C("Training-median rows",302,sum(len(x["TrainingMedians"]) for x in results),sum(len(x["TrainingMedians"]) for x in results)==302);C("Technique set",sorted(ALL_TECHNIQUES),sorted(project_runs["Technique"].unique()),set(project_runs["Technique"])==set(ALL_TECHNIQUES))
C("0% raw flips",0,int(zero.NumberFlipped),int(zero.NumberFlipped)==0);C("0% model-label changes",0,int(zero.ModelLabelChanges),int(zero.ModelLabelChanges)==0);C("0% dependent REC changes",0,int(zero.DependentRECChanges),int(zero.DependentRECChanges)==0);C("50% raw flips","> 0",int(positive.NumberFlipped),int(positive.NumberFlipped)>0);C("50% model-label changes","> 0",int(positive.ModelLabelChanges),int(positive.ModelLabelChanges)>0);C("50% dependent REC changes","> 0",int(positive.DependentRECChanges),int(positive.DependentRECChanges)>0);C("Independent REC changes",0,int(audit["IndependentRECChanges"].sum()),int(audit["IndependentRECChanges"].sum())==0);C("Independent reconstruction mismatches",0,int(audit["IndependentReconstructionMismatches"].sum()),int(audit["IndependentReconstructionMismatches"].sum())==0);C("Baseline invariance failures",0,baseline_fail,baseline_fail==0)
C("Raw training cohort unchanged",raw_train_hash_before,raw_train_hash_after,raw_train_hash_before==raw_train_hash_after);C("Raw evaluation cohort unchanged",raw_eval_hash_before,raw_eval_hash_after,raw_eval_hash_before==raw_eval_hash_after);C("Model training cohort unchanged",model_train_hash_before,model_train_hash_after,model_train_hash_before==model_train_hash_after);C("Model evaluation cohort unchanged",model_eval_hash_before,model_eval_hash_after,model_eval_hash_before==model_eval_hash_after);C("Completion registry unchanged",registry_sha_before,registry_sha_after,registry_sha_before==registry_sha_after);C("Registry Project 24 rows",0,int(nums.eq(24).sum()),int(nums.eq(24).sum())==0);C("Full raw-result root unchanged",True,full_raw_unchanged,full_raw_unchanged);C("Active reservations",EXPECTED_ACTIVE_RESERVATIONS,runtime_cp.get("ActiveReservations"),runtime_cp.get("ActiveReservations")==EXPECTED_ACTIVE_RESERVATIONS);C("Runtime-priority ranking rule",EXPECTED_RUNTIME_PRIORITY_RULE,runtime_cp.get("RuntimePriorityRule"),runtime_cp.get("RuntimePriorityRule")==EXPECTED_RUNTIME_PRIORITY_RULE)
validation=pd.DataFrame(validation);failed=validation.loc[~validation["Pass"]]
print("\nProject 24 Step 4B validation:");display(validation);print("\nBaseline invariance audit:");display(binv);print("\nSmoke project-run results:");display(project_runs.sort_values(["NoisePercent","Technique"],kind="mergesort").reset_index(drop=True))
if not failed.empty:print("\nFailed Project 24 Step 4B checks:");display(failed);print("\nNo Step 4B PASS status/checkpoint was written.");raise RuntimeError("PROJECT 24 STEP 4B VALIDATION FAILED. DO NOT START STEP 5A.")
atomic_csv(SMOKE_VALIDATION_PATH,validation)

# --------------------------------------------------------------------------------------------------
# 9. REPORT / CHECKPOINT / STATUS / READBACK
# --------------------------------------------------------------------------------------------------
smoke_seconds=time.perf_counter()-smoke_started; completed=datetime.now(timezone.utc).isoformat(); output_paths=[SMOKE_INVENTORY_PATH,SMOKE_AUDIT_PATH,SMOKE_PROJECT_RUNS_PATH,SMOKE_BUILD_METRICS_PATH,SMOKE_MODEL_FITS_PATH,SMOKE_BASELINE_INVARIANCE_PATH,SMOKE_VALIDATION_PATH]
for key in SMOKE_CONDITION_IDS:
 output_paths += [SMOKE_ROOT/key/name for name in ["rankings.csv.gz","build_metrics.csv","project_runs.csv","model_fits.csv","training_medians.csv","condition_audit.csv"]]
missing_smoke_outputs=[str(p) for p in output_paths if not p.is_file()]
if missing_smoke_outputs:raise FileNotFoundError("Expected smoke outputs missing:\n"+"\n".join(missing_smoke_outputs))
output_paths=sorted(output_paths,key=str); manifest=[{"Path":str(p),"Bytes":int(p.stat().st_size),"SHA256":sha256_file(p)} for p in output_paths]
report={"ProjectNumber":24,"Project":PROJECT_NAME,"ProjectSlug":PROJECT_SLUG,"Status":STEP4B_STATUS,"CompletedAtUTC":completed,"RuntimeCheckpointSHA256":runtime_sha,"NoisePlanCheckpointSHA256":noise_sha,"RECCheckpointSHA256":rec_sha,"SelectionCheckpointSHA256":selection_sha,"SourceRootSHA256":source_sha_after,"RECImplementation":EXPECTED_REC_IMPLEMENTATION,"Step3ACodeRevision":EXPECTED_STEP3A_CODE_REVISION,"SmokeConditionKeys":SMOKE_CONDITION_IDS,"SmokeConditions":2,"ConditionsPassed":2,"MLFits":len(model_fits),"RankingRows":ranking_total,"BuildMetricRows":len(build_metrics),"ProjectRunRows":len(project_runs),"TrainingMedianRows":sum(len(x["TrainingMedians"]) for x in results),"ZeroNoiseRawFlips":int(zero.NumberFlipped),"ZeroNoiseModelLabelChanges":int(zero.ModelLabelChanges),"ZeroNoiseDependentRECChanges":int(zero.DependentRECChanges),"PositiveNoiseRawFlips":int(positive.NumberFlipped),"PositiveNoiseModelLabelChanges":int(positive.ModelLabelChanges),"PositiveNoiseDependentRECChanges":int(positive.DependentRECChanges),"IndependentRECChanges":int(audit["IndependentRECChanges"].sum()),"IndependentReconstructionMismatches":int(audit["IndependentReconstructionMismatches"].sum()),"BaselineInvarianceFailures":baseline_fail,"Techniques":ALL_TECHNIQUES,"PrimaryMetric":"APFDc","SecondaryMetric":"APFD","SmokeExecutionSeconds":float(smoke_seconds),"ValidationChecks":len(validation),"FailedValidationChecks":len(failed),"OutputManifest":manifest,"RegistrySHA256":EXPECTED_REGISTRY_SHA256,"RegistryModified":False,"Projects1To23Modified":False,"ActiveReservations":EXPECTED_ACTIVE_RESERVATIONS,"RuntimePriorityRule":EXPECTED_RUNTIME_PRIORITY_RULE,"PriorProjectConditionOutputsAccessed":False,"PriorProjectConditionOutputsModified":False,"FullRawResultRootModified":False,"FullExperimentStarted":False}
atomic_json(SMOKE_REPORT_PATH,report)
checkpoint={**report,"CheckpointVersion":1,"CheckpointType":"PROJECT_24_TWO_CONDITION_SMOKE_TEST","SmokeTestPassed":True,"RuntimeContractFrozen":True,"NoisePlanFrozen":True,"RECReconstructionFrozen":True,"EvaluationCohortImmutable":True,"ReadyForFull270ConditionExperiment":True}
atomic_json(SMOKE_CHECKPOINT_PATH,checkpoint);smoke_sha=sha256_file(SMOKE_CHECKPOINT_PATH)
status={"ProjectNumber":24,"Project":PROJECT_NAME,"ProjectSlug":PROJECT_SLUG,"Status":STEP4B_STATUS,"CompletedAtUTC":completed,"SmokeConditions":2,"MLFits":8,"FailedValidationChecks":0,"Checkpoint":str(SMOKE_CHECKPOINT_PATH),"CheckpointSHA256":smoke_sha,"RegistryModified":False,"PriorProjectConditionOutputsAccessed":False,"FullExperimentStarted":False,"ReadyForFull270ConditionExperiment":True,"NextRequiredStep":"PROJECT 24 STEP 5A — FULL 270-CONDITION EXPERIMENT"}
atomic_json(STEP4B_STATUS_PATH,status)
checkpoint_readback=load_json(SMOKE_CHECKPOINT_PATH);status_readback=load_json(STEP4B_STATUS_PATH)
if checkpoint_readback.get("Status")!=STEP4B_STATUS or status_readback.get("Status")!=STEP4B_STATUS:raise RuntimeError("Step 4B checkpoint/status readback failed")
if not checkpoint_readback.get("ReadyForFull270ConditionExperiment",False):raise RuntimeError("Smoke checkpoint does not authorize Step 5A")
if checkpoint_readback.get("RuntimeCheckpointSHA256")!=EXPECTED_RUNTIME_CHECKPOINT_SHA256:raise RuntimeError("Smoke checkpoint/runtime linkage differs")
if checkpoint_readback.get("ActiveReservations")!=EXPECTED_ACTIVE_RESERVATIONS or checkpoint_readback.get("RuntimePriorityRule")!=EXPECTED_RUNTIME_PRIORITY_RULE:raise RuntimeError("Smoke checkpoint frozen coordination state differs")
if checkpoint_readback.get("OutputManifest")!=manifest:raise RuntimeError("Smoke checkpoint output manifest readback differs")
for item in checkpoint_readback["OutputManifest"]:
 p=Path(item["Path"])
 if not p.is_file() or int(p.stat().st_size)!=int(item["Bytes"]) or sha256_file(p)!=item["SHA256"]:raise RuntimeError(f"Smoke output manifest readback failed: {p}")
if sha256_file(REGISTRY_PATH)!=EXPECTED_REGISTRY_SHA256:raise RuntimeError("Registry changed during finalisation")
final_raw=directory_manifest(FULL_RAW_RESULT_ROOT)
if FULL_RAW_RESULT_ROOT.exists()!=raw_existed_before or directory_root_hash(final_raw)!=raw_root_before or len(final_raw)!=len(raw_manifest_before):raise RuntimeError("Full Step-5A raw-result root changed during Step 4B")

print("\n"+"="*140);print("=== PROJECT 24 CELL 8 / STEP 4B RESULT ===");print("="*140)
print("Project:",PROJECT_NAME);print("Project slug:",PROJECT_SLUG);print("Two-condition end-to-end smoke test:");print("Conditions:",SMOKE_CONDITION_IDS);print("Conditions passed: 2 / 2");print("ML fits: 8 / 8");print("Ranking rows:",ranking_total);print("Build-metric rows:",len(build_metrics));print("Project-run rows:",len(project_runs));print("Training-median rows:",sum(len(x["TrainingMedians"]) for x in results))
print("\nNoise and REC audit:");print("0% raw flips:",int(zero.NumberFlipped));print("0% model-label changes:",int(zero.ModelLabelChanges));print("0% dependent REC changes:",int(zero.DependentRECChanges));print("50% raw flips:",int(positive.NumberFlipped));print("50% model-label changes:",int(positive.ModelLabelChanges));print("50% dependent REC changes:",int(positive.DependentRECChanges));print("Independent REC changes:",int(audit["IndependentRECChanges"].sum()));print("Independent reconstruction mismatches:",int(audit["IndependentReconstructionMismatches"].sum()))
print("\nBaselines and metrics:");print("Random/QTF-Avg invariance failures:",baseline_fail);print("Techniques:",ALL_TECHNIQUES);print("Primary / secondary metrics: APFDc / APFD")
print("\nImmutability and isolation:");print("Project 24 source unchanged: True");print("Completion registry unchanged: True");print("Projects 1–23 modified: 0");print("Prior project condition outputs accessed: False");print("Prior project condition outputs modified: False");print("Full experiment raw-result root modified: False");print("Full 270-condition experiment started: False")
print("\nValidation:");print("Checks:",len(validation));print("Failed checks:",len(failed));print("\nSmoke-test checkpoint:",SMOKE_CHECKPOINT_PATH);print("Checkpoint SHA-256:",smoke_sha);print("Runtime seconds:",round(smoke_seconds,2));print("\nNext required step: PROJECT 24 STEP 5A — FULL 270-CONDITION EXPERIMENT");print("\nSTATUS:",STEP4B_STATUS);print("="*140)


=== PROJECT 24 CELL 8 / STEP 4B: TWO-CONDITION END-TO-END SMOKE TEST ===

Loading frozen Project 24 cohorts and contracts.
Converting fixed predictor cohorts to numeric matrices.

Executing: noise_00__seed_01
    REC reconstruction progress: 500 / 2641 tests | reconstructed rows: 77978 | elapsed seconds: 1.38
    REC reconstruction progress: 1000 / 2641 tests | reconstructed rows: 159603 | elapsed seconds: 2.26
    REC reconstruction progress: 1500 / 2641 tests | reconstructed rows: 211337 | elapsed seconds: 3.09
    REC reconstruction progress: 2000 / 2641 tests | reconstructed rows: 222975 | elapsed seconds: 3.55
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_00__seed_01
  Raw flips: 0 | model-label changes: 0 | dependent REC changes: 0
  Training failures: 1777 | condition seconds: 123.1

Executing: noise_50__seed_01
    REC reconstruction progress: 500 / 2641 tests | reconstructed rows: 77978 | elapsed seconds: 5.37
    REC reconstruction progress: 1000 / 2641 tests | reconstructed rows: 159603 | elapsed seconds: 10.26
    REC reconstruction progress: 1500 / 2641 tests | reconstructed rows: 211337 | elapsed seconds: 13.61
    REC reconstruction progress: 2000 / 2641 tests | reconstructed rows: 222975 | elapsed seconds: 14.9
  Fitting: RandomForest
  Fitting: XGBoost
  Fitting: LightGBM


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fitting: NaiveBayes
  Completed: noise_50__seed_01
  Raw flips: 2021337 | model-label changes: 103102 | dependent REC changes: 2575312
  Training failures: 103071 | condition seconds: 263.11

Project 24 Step 4B validation:


...


Baseline invariance audit:


,Technique,Rows,SameBuildTestKeys,ScoreMismatches,RankMismatches,Pass
0,Random,18854,True,0,0,True
1,QTF-Avg,18854,True,0,0,True



Smoke project-run results:


,ProjectNumber,Project,ProjectSlug,ConditionKey,NoisePercent,RepetitionSeed,Technique,EvaluationBuilds,ScoredFailingBuilds,EvaluationRows,EvaluationFailures,MeanAPFDc,MedianAPFDc,MeanAPFD,MedianAPFD
0,24,SonarSource@sonarqube,SonarSource__sonarqube,noise_00__seed_01,0,1,LatestFail,1072,17,18854,20,0.528929,0.554260,0.137540,0.061533
1,24,SonarSource@sonarqube,SonarSource__sonarqube,noise_00__seed_01,0,1,LightGBM,1072,17,18854,20,0.665418,0.716807,0.927809,0.980029
2,24,SonarSource@sonarqube,SonarSource__sonarqube,noise_00__seed_01,0,1,NaiveBayes,1072,17,18854,20,0.232200,0.160492,0.546040,0.565542
3,24,SonarSource@sonarqube,SonarSource__sonarqube,noise_00__seed_01,0,1,QTF-Avg,1072,17,18854,20,0.803867,0.875686,0.319966,0.104788
4,24,SonarSource@sonarqube,SonarSource__sonarqube,noise_00__seed_01,0,1,Random,1072,17,18854,20,0.506339,0.499053,0.502961,0.517309
5,24,SonarSource@sonarqube,SonarSource__sonarqube,noise_00__seed_01,0,1,RandomForest,1072,17,18854,20,0.606783,0.686036,0.832253,0.981481
6,24,SonarSource@sonarqube,SonarSource__sonarqube,noise_00__seed_01,0,1,XGBoost,1072,17,18854,20,0.639498,0.573733,0.922126,0.981481
7,24,SonarSource@sonarqube,SonarSource__sonarqube,noise_50__seed_01,50,1,LatestFail,1072,17,18854,20,0.820274,0.902999,0.847947,0.889245
8,24,SonarSource@sonarqube,SonarSource__sonarqube,noise_50__seed_01,50,1,LightGBM,1072,17,18854,20,0.639847,0.679992,0.453797,0.458972
9,24,SonarSource@sonarqube,SonarSource__sonarqube,noise_50__seed_01,50,1,NaiveBayes,1072,17,18854,20,0.233987,0.093089,0.281536,0.109330



=== PROJECT 24 CELL 8 / STEP 4B RESULT ===
Project: SonarSource@sonarqube
Project slug: SonarSource__sonarqube
Two-condition end-to-end smoke test:
Conditions: ['noise_00__seed_01', 'noise_50__seed_01']
Conditions passed: 2 / 2
ML fits: 8 / 8
Ranking rows: 263956
Build-metric rows: 238
Project-run rows: 14
Training-median rows: 302

Noise and REC audit:
0% raw flips: 0
0% model-label changes: 0
0% dependent REC changes: 0
50% raw flips: 2021337
50% model-label changes: 103102
50% dependent REC changes: 2575312
Independent REC changes: 0
Independent reconstruction mismatches: 0

Baselines and metrics:
Random/QTF-Avg invariance failures: 0
Techniques: ['RandomForest', 'XGBoost', 'LightGBM', 'NaiveBayes', 'Random', 'LatestFail', 'QTF-Avg']
Primary / secondary metrics: APFDc / APFD

Immutability and isolation:
Project 24 source unchanged: True
Completion registry unchanged: True
Projects 1–23 modified: 0
Prior project condition outputs accessed: False
Prior project condition outputs modif

In [1]:
# ==================================================================================================
# PROJECT 24 — CELL 9 / STEP 5A PARALLEL MASTER FINALIZATION
# ZERO-FIT REVALIDATION, AGGREGATION, RAW-ROOT FREEZE, AND OFFICIAL CHECKPOINT
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS AS THE NEXT NEW CELL IN THE RECONNECTED MASTER Thesis_project_24.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the six completed Project 24 Step 5A worker shards;
# - prove the six seed shards are disjoint and cover seeds 1–30 exactly;
# - independently revalidate all 270 raw condition directories without fitting any model;
# - rebuild the official compact Step 5A aggregate outputs from the frozen raw condition files;
# - independently hash and freeze the complete 2,160-file Project 24 raw-result root;
# - create the official Project 24 Step 5A report/status/checkpoint required by Step 5B.
#
# IMPORTANT CHECKPOINT NOTE:
# - resume-safe worker re-finalization can legitimately rewrite only volatile worker metadata
#   (CompletedAtUTC / invocation runtime), which changes a worker-checkpoint SHA while leaving all
#   frozen scientific raw outputs unchanged;
# - therefore this master records the ACTUAL current SHA of every worker checkpoint, but gates the
#   scientific freeze on exact pre-recorded worker raw-root SHA-256, raw bytes, seed shard, counts,
#   worker report linkage, worker output manifests, and an independent master revalidation of every
#   condition directory;
# - the official Step 5A checkpoint is thus cryptographically bound to the exact six checkpoint files
#   that exist at master-finalization time, while remaining robust to the accidental resume-safe
#   seeds-1–5 rerun.
#
# SAFETY:
# - ZERO model fitting;
# - ZERO noise injection / REC reconstruction / condition execution;
# - no completion-registry write;
# - no access to prior-project condition outputs;
# - raw condition directories are READ ONLY;
# - official master outputs are written only after every validation gate passes.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gzip
import hashlib
import json
import os
import shutil
import tarfile
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive


print("=" * 140)
print("=== PROJECT 24 CELL 9 / STEP 5A: PARALLEL MASTER FINALIZATION — ZERO FIT ===")
print("=" * 140)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 23
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

EXPECTED_SELECTION_STATUS = "PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN"
EXPECTED_STEP2B_STATUS = "PASS_PROJECT_24_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
EXPECTED_STEP3A_STATUS = "PASS_PROJECT_24_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
EXPECTED_STEP4A_STATUS = "PASS_PROJECT_24_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
EXPECTED_STEP4B_STATUS = "PASS_PROJECT_24_TWO_CONDITION_END_TO_END_SMOKE_TEST"

STEP5A_STATUS = "PASS_PROJECT_24_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
CONDITION_STATUS = "PASS_FULL_CONDITION"

MASTER_FINALIZER_IMPLEMENTATION = (
    "PROJECT_24_PARALLEL_STEP5A_MASTER_FINALIZER_V1_ZERO_FIT_"
    "WORKER_RAW_ROOT_BOUND_RESUME_METADATA_TOLERANT"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
)
EXPECTED_REC_CHECKPOINT_SHA256 = (
    "80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803"
)
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "9dba84682eabdc1393219671e77aabaf1f2d4385f29239236e50615bf0180515"
)
EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "ff9ed031d21d67bf218d5eab70bd8502ca39953adedbe840348f382d5599314e"
)
EXPECTED_SMOKE_CHECKPOINT_SHA256 = (
    "3f5a0cc24260b251fd8cd43f7391ff7d3cc3d517e8f79a8ad443f4002cdc9ff2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)
EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 23
EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 374_503_252

EXPECTED_RAW_TRAIN_ROWS = 4_040_801
EXPECTED_RAW_EVAL_ROWS = 1_594_226
EXPECTED_MODEL_TRAIN_ROWS = 205_696
EXPECTED_MODEL_EVAL_ROWS = 18_854
EXPECTED_MODEL_TRAIN_FAILURES = 1_777
EXPECTED_MODEL_EVAL_FAILURES = 20
EXPECTED_FAILING_EVAL_BUILDS = 17
EXPECTED_EVALUATION_BUILDS = 1_072
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_RNG_ROWS = 121_224_030

EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 523_093_565

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES
INVARIANT_BASELINES = ["Random", "QTF-Avg"]

ACCELERATED_ENGINE_VERSION = (
    "PROJECT_24_FAST_DEPENDENT_REC_V1_"
    "SMOKE_EQUIVALENT_FROZEN_INFERRED_TEST_ORDER_ANCHOR_AWARE"
)

SMOKE_EQUIVALENCE_KEYS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_MODEL_FITS = (
    EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = (
    EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_CONDITION_AUDIT_ROWS = EXPECTED_CONDITIONS

EXPECTED_WORKER_CONDITIONS = 45
EXPECTED_WORKER_MODEL_FITS = 180
EXPECTED_WORKER_RAW_FILES = 360
EXPECTED_WORKER_RANKING_ROWS = 5_939_010
EXPECTED_WORKER_BUILD_METRIC_ROWS = 5_355
EXPECTED_WORKER_PROJECT_RUN_ROWS = 315
EXPECTED_WORKER_CONDITION_AUDIT_ROWS = 45
EXPECTED_WORKER_TRAINING_MEDIAN_ROWS = 6_795

# Scientific worker freezes reported by the successful Project 24 worker runs.
# ReportedCheckpointSHA256 is retained for provenance. It is NOT the scientific gate because a
# resume-safe rerun can refresh volatile checkpoint metadata. Actual checkpoint SHA-256 values are
# captured and frozen by this master after all scientific validations pass.
EXPECTED_WORKERS = {
    "seed_01_05": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_01_05_COMPLETE",
        "Seeds": list(range(1, 6)),
        "ReportedCheckpointSHA256":
            "c6e8d76fe71c6aedfba9c3a0632c5daa5e2a8ed7cd3598dc56330d68e19ac81b",
        "RawRootSHA256":
            "53c1767a911a5d52561e2e12777f18d74ef8386c151a60fdf75e1488c2ab5705",
        "RawBytes": 86_952_171,
    },
    "seed_06_10": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_06_10_COMPLETE",
        "Seeds": list(range(6, 11)),
        "ReportedCheckpointSHA256":
            "ce2a285795c0a2b7aa16a51026686646328e817c778e49bfd14992137449a43a",
        "RawRootSHA256":
            "1703eef52c16b247a43bd95b378b6e1feb3bc8f8da6fc8a5b8c395b4437572a7",
        "RawBytes": 87_379_614,
    },
    "seed_11_15": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_11_15_COMPLETE",
        "Seeds": list(range(11, 16)),
        "ReportedCheckpointSHA256":
            "a8af44492f79733b78955a4f26213f5abd56332874cecb7c32b2a15f21392270",
        "RawRootSHA256":
            "01fef4c6d37ac6ee491a415aee4a3eabf05551be372c09fdc1b997fb52f7b93d",
        "RawBytes": 87_582_299,
    },
    "seed_16_20": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_16_20_COMPLETE",
        "Seeds": list(range(16, 21)),
        "ReportedCheckpointSHA256":
            "3177e9c8d783df28d2f28f60c75eb69d239c885c4273c55f0ed702ca1d63fd9d",
        "RawRootSHA256":
            "910ef203274d1ea744ad92ef1ec7802777d38be713333b21f5191fa69d4c364a",
        "RawBytes": 87_224_279,
    },
    "seed_21_25": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_21_25_COMPLETE",
        "Seeds": list(range(21, 26)),
        "ReportedCheckpointSHA256":
            "7cc8475c5c42fbfd86ab92e4024c7e7d486d03d8d7f053955bc86bb0fdb0c3d1",
        "RawRootSHA256":
            "a311f4370ebe599c4d7cee8a6d865ed7857c1b8d2641998fbabac352e30d7e1e",
        "RawBytes": 87_220_469,
    },
    "seed_26_30": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_26_30_COMPLETE",
        "Seeds": list(range(26, 31)),
        "ReportedCheckpointSHA256":
            "d9cf039b99bdee7b502adf78497c028c8d8b918556d68b6cf0eca975c56661b1",
        "RawRootSHA256":
            "b09e26b9bc4087379eac076931f70cc141083f9e6d15584c40a433f31f071aac",
        "RawBytes": 86_734_733,
    },
}

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"

REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
ARCHIVE_PATH = THESIS_ROOT / "Data" / "Raw" / "TCP-CI-main-dataset.tar.gz"
SOURCE_DIR = Path("/content/datasets/datasets/SonarSource@sonarqube")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_24_selection"
FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT / "project_24_frozen_source_manifest.csv"
)
SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_selection_checkpoint.json"
)

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
REC_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)
RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)
MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)
MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)
RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
)
CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
)
NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_noise_plan_checkpoint.json"
)

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_runtime_contract_checkpoint.json"
)

SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"
SMOKE_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_smoke_test_checkpoint.json"
)

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"

ACCELERATED_EQUIVALENCE_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"
)

PARALLEL_WORKERS_ROOT = FULL_EXPERIMENT_ROOT / "parallel_workers"

WORKER_CHECKPOINT_PATHS = {
    tag: NOTES_ROOT / f"project_24_step5a_worker_{tag}_checkpoint.json"
    for tag in EXPECTED_WORKERS
}

CONDITION_INVENTORY_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
)
RAW_MANIFEST_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
)
BASELINE_INVARIANCE_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
)
COMBINED_CONDITION_AUDIT_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
)
COMBINED_PROJECT_RUNS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
)
COMBINED_BUILD_METRICS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
)
COMBINED_MODEL_FITS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
)
WORKER_CHECKPOINT_AUDIT_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_worker_checkpoint_audit.csv"
)
STEP5A_VALIDATION_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
)
STEP5A_REPORT_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
)
RUN_PROGRESS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
)

STEP5A_STATUS_PATH = (
    PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
)
STEP5A_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_step5a_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary, path)


def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
    )

    os.replace(temporary, path)


def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        size = getattr(
            row,
            "SizeBytes",
            getattr(row, "Bytes", None),
        )

        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(size)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


DIRECTORY_MANIFEST_COLUMNS = [
    "RelativePath",
    "Bytes",
    "SHA256",
]


def directory_manifest(root):
    root = Path(root)
    records = []

    if root.exists():
        for path in sorted(
            (
                candidate
                for candidate in root.rglob("*")
                if candidate.is_file()
            ),
            key=lambda candidate:
                candidate.relative_to(root).as_posix(),
        ):
            records.append({
                "RelativePath":
                    path.relative_to(root).as_posix(),

                "Bytes":
                    int(path.stat().st_size),

                "SHA256":
                    sha256_file(path),
            })

    return pd.DataFrame(
        records,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )


def directory_root_hash(manifest):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
        "y",
    }


def verify_manifest_entries(entries, label):
    failures = []

    if not isinstance(entries, list) or not entries:
        return [{
            "Label": label,
            "Path": "<manifest>",
            "Reason": "missing/empty manifest",
        }]

    for item in entries:
        path = Path(str(item.get("Path", "")))

        if not path.is_file():
            failures.append({
                "Label": label,
                "Path": str(path),
                "Reason": "missing",
            })
            continue

        expected_bytes = int(item.get("Bytes", -1))
        expected_sha = str(item.get("SHA256", "")).lower()
        actual_bytes = int(path.stat().st_size)
        actual_sha = sha256_file(path)

        if (
            actual_bytes != expected_bytes
            or actual_sha != expected_sha
        ):
            failures.append({
                "Label": label,
                "Path": str(path),
                "Reason": "size/hash mismatch",
            })

    return failures


def resolve_registry_columns(registry):
    lookup = {
        str(column).strip().lower(): column
        for column in registry.columns
    }

    def pick(names, label):
        for name in names:
            if name in lookup:
                return lookup[name]

        raise RuntimeError(
            f"Could not resolve registry {label}; "
            f"columns={list(registry.columns)}"
        )

    return (
        pick(
            [
                "projectnumber",
                "project_number",
                "project no",
                "projectno",
            ],
            "ProjectNumber",
        ),
        pick(
            [
                "project",
                "projectname",
                "project_name",
            ],
            "Project",
        ),
        pick(
            [
                "status",
                "projectstatus",
                "project_status",
            ],
            "Status",
        ),
    )


def count_csv_data_rows(path):
    """
    Count CSV data rows without materialising the full frame.
    rankings.csv.gz is the only large file; all generated ranking rows contain no embedded newlines.
    """
    path = Path(path)

    opener = gzip.open if path.suffix == ".gz" else open

    with opener(
        path,
        "rt",
        encoding="utf-8",
        newline="",
    ) as handle:
        count = -1  # remove header
        for _ in handle:
            count += 1

    return max(count, 0)


def restore_source_if_needed():
    required_files = {
        "builds.csv",
        "contributors.csv",
        "dataset.csv",
        "entity_change_history.csv",
        "exe.csv",
        "id_map.csv",
    }

    source_ready = bool(
        SOURCE_DIR.is_dir()
        and required_files.issubset({
            path.name
            for path in SOURCE_DIR.iterdir()
            if path.is_file()
        })
    )

    if source_ready:
        return False

    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(
            "Project 24 source is absent from this reconnected runtime "
            "and the frozen thesis archive is missing:\n"
            f"{ARCHIVE_PATH}"
        )

    print(
        "\nRestoring only SonarSource@sonarqube from the frozen TCP-CI archive "
        "into this master runtime."
    )

    local_dataset_root = Path("/content/datasets")
    local_dataset_root.mkdir(parents=True, exist_ok=True)

    member_prefix = "datasets/SonarSource@sonarqube/"
    resolved_root = local_dataset_root.resolve()
    extracted_files = 0

    with tarfile.open(
        ARCHIVE_PATH,
        mode="r:gz",
    ) as archive:
        for member in archive:
            member_name = (
                member.name
                .replace("\\", "/")
                .lstrip("/")
            )

            if not (
                member_name == "datasets/SonarSource@sonarqube"
                or member_name.startswith(member_prefix)
            ):
                continue

            target_path = (
                local_dataset_root
                / member_name
            )

            resolved_target = target_path.resolve()

            if (
                resolved_target != resolved_root
                and resolved_root not in resolved_target.parents
            ):
                raise RuntimeError(
                    "Unsafe archive member encountered:\n"
                    f"{member.name}"
                )

            if member.isdir():
                target_path.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            elif member.isfile():
                target_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                source_handle = archive.extractfile(
                    member
                )

                if source_handle is None:
                    raise RuntimeError(
                        "Could not read archive member:\n"
                        f"{member.name}"
                    )

                with (
                    source_handle,
                    target_path.open("wb") as output_handle,
                ):
                    shutil.copyfileobj(
                        source_handle,
                        output_handle,
                        length=8 * 1024 * 1024,
                    )

                extracted_files += 1

    source_names = (
        {
            path.name
            for path in SOURCE_DIR.iterdir()
            if path.is_file()
        }
        if SOURCE_DIR.is_dir()
        else set()
    )

    missing = sorted(
        required_files
        - source_names
    )

    if missing:
        raise FileNotFoundError(
            "Selective Project 24 source restoration is incomplete:\n"
            + "\n".join(missing)
        )

    print(
        "Project source files restored:",
        extracted_files,
    )

    return True


EXPECTED_CONDITION_FILES = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}


def validate_condition_read_only(condition_dir, plan_row):
    """
    Independent master-side revalidation.
    No condition output is modified.
    """
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)

    if not condition_dir.is_dir():
        return None, f"{condition_key}: condition directory missing"

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    if actual_files != EXPECTED_CONDITION_FILES:
        return (
            None,
            f"{condition_key}: file set differs: "
            f"{sorted(actual_files)}",
        )

    completion_path = (
        condition_dir / "COMPLETE.json"
    )
    summary_path = (
        condition_dir / "condition_summary.json"
    )

    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception as error:
        return (
            None,
            f"{condition_key}: JSON read failure: {error!r}",
        )

    identity_checks = [
        (
            completion.get("Status"),
            CONDITION_STATUS,
            "completion status",
        ),
        (
            summary.get("Status"),
            CONDITION_STATUS,
            "summary status",
        ),
        (
            completion.get("ConditionKey"),
            condition_key,
            "completion key",
        ),
        (
            summary.get("ConditionKey"),
            condition_key,
            "summary key",
        ),
        (
            summary.get("Project"),
            PROJECT_NAME,
            "project",
        ),
        (
            summary.get("ProjectSlug"),
            PROJECT_SLUG,
            "project slug",
        ),
        (
            summary.get("AcceleratedEngineVersion"),
            ACCELERATED_ENGINE_VERSION,
            "accelerated engine",
        ),
    ]

    for actual, expected, label in identity_checks:
        if actual != expected:
            return (
                None,
                f"{condition_key}: {label} differs; "
                f"expected={expected!r}, actual={actual!r}",
            )

    if int(
        summary.get("ConditionOrder", -1)
    ) != int(plan_row.ConditionOrder):
        return None, f"{condition_key}: ConditionOrder differs"

    if int(
        summary.get("NoisePercent", -1)
    ) != int(plan_row.NoisePercent):
        return None, f"{condition_key}: NoisePercent differs"

    if int(
        summary.get("RepetitionSeed", -1)
    ) != int(plan_row.RepetitionSeed):
        return None, f"{condition_key}: RepetitionSeed differs"

    if str(
        completion.get("ConditionSummaryPath")
    ) != str(summary_path):
        return None, f"{condition_key}: summary path linkage differs"

    if str(
        completion.get("ConditionSummarySHA256")
    ) != sha256_file(summary_path):
        return None, f"{condition_key}: summary SHA linkage differs"

    output_manifest = summary.get(
        "OutputManifest",
        [],
    )

    if (
        not isinstance(output_manifest, list)
        or len(output_manifest) != 6
    ):
        return None, f"{condition_key}: output manifest differs"

    output_manifest_failures = verify_manifest_entries(
        output_manifest,
        condition_key,
    )

    if output_manifest_failures:
        return (
            None,
            f"{condition_key}: output manifest validation failed",
        )

    for item in output_manifest:
        if Path(
            str(item.get("Path", ""))
        ).parent != condition_dir:
            return (
                None,
                f"{condition_key}: output-manifest path escaped condition directory",
            )

    expected_counts = {
        "MLFits":
            EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows":
            EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows":
            EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }

    for key, expected in expected_counts.items():
        if int(
            summary.get(key, -1)
        ) != expected:
            return (
                None,
                f"{condition_key}: summary {key} differs",
            )

    if int(
        summary.get("IndependentRECChanges", -1)
    ) != 0:
        return (
            None,
            f"{condition_key}: independent REC changes are nonzero",
        )

    fingerprints = summary.get(
        "BaselineFingerprints",
        {},
    )

    if sorted(
        fingerprints.keys()
    ) != [
        "QTF-Avg",
        "Random",
    ]:
        return (
            None,
            f"{condition_key}: baseline fingerprints differ",
        )

    # Independent content-level readback of all compact raw outputs.
    condition_audit = pd.read_csv(
        condition_dir / "condition_audit.csv",
        low_memory=False,
    )

    if len(condition_audit) != 1:
        return (
            None,
            f"{condition_key}: condition_audit row count differs",
        )

    audit = condition_audit.iloc[0]

    for column, expected in [
        ("ConditionKey", condition_key),
        ("NoisePercent", int(plan_row.NoisePercent)),
        ("RepetitionSeed", int(plan_row.RepetitionSeed)),
        ("Status", CONDITION_STATUS),
    ]:
        if column not in condition_audit.columns:
            return (
                None,
                f"{condition_key}: condition_audit missing {column}",
            )

        actual = audit[column]

        if column in {
            "NoisePercent",
            "RepetitionSeed",
        }:
            if int(actual) != int(expected):
                return (
                    None,
                    f"{condition_key}: audit {column} differs",
                )
        elif str(actual) != str(expected):
            return (
                None,
                f"{condition_key}: audit {column} differs",
            )

    for column in [
        "IndependentRECChanges",
        "IndependentReconstructionMismatches",
    ]:
        if (
            column not in condition_audit.columns
            or int(
                pd.to_numeric(
                    pd.Series([audit[column]]),
                    errors="raise",
                ).iloc[0]
            ) != 0
        ):
            return (
                None,
                f"{condition_key}: audit {column} differs",
            )

    for expected_column, actual_column in [
        (
            "ExpectedFlipMaskSHA256",
            "ActualFlipMaskSHA256",
        ),
        (
            "ExpectedNoisyRawVerdictSHA256",
            "ActualNoisyRawVerdictSHA256",
        ),
        (
            "ExpectedNoisyModelVerdictSHA256",
            "ActualNoisyModelVerdictSHA256",
        ),
    ]:
        if (
            expected_column not in condition_audit.columns
            or actual_column not in condition_audit.columns
            or str(audit[expected_column])
            != str(audit[actual_column])
        ):
            return (
                None,
                f"{condition_key}: noise-plan hash audit differs",
            )

    number_flipped = int(
        pd.to_numeric(
            pd.Series([
                audit["NumberFlipped"]
            ]),
            errors="raise",
        ).iloc[0]
    )

    model_label_changes = int(
        pd.to_numeric(
            pd.Series([
                audit["ModelLabelChanges"]
            ]),
            errors="raise",
        ).iloc[0]
    )

    dependent_rec_changes = int(
        pd.to_numeric(
            pd.Series([
                audit["DependentRECChanges"]
            ]),
            errors="raise",
        ).iloc[0]
    )

    if int(plan_row.NoisePercent) == 0:
        if (
            number_flipped != 0
            or model_label_changes != 0
            or dependent_rec_changes != 0
        ):
            return (
                None,
                f"{condition_key}: zero-noise scientific audit differs",
            )
    else:
        if (
            number_flipped <= 0
            or model_label_changes <= 0
            or dependent_rec_changes <= 0
        ):
            return (
                None,
                f"{condition_key}: positive-noise scientific audit differs",
            )

    model_fits = pd.read_csv(
        condition_dir / "model_fits.csv",
        low_memory=False,
    )

    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: model-fit row count differs",
        )

    if (
        "Technique" not in model_fits.columns
        or "Status" not in model_fits.columns
        or sorted(
            model_fits["Technique"]
            .astype(str)
            .tolist()
        ) != sorted(ML_TECHNIQUES)
        or not model_fits["Status"]
        .astype(str)
        .eq("PASS_MODEL_FIT")
        .all()
    ):
        return (
            None,
            f"{condition_key}: model-fit contract differs",
        )

    build_metrics = pd.read_csv(
        condition_dir / "build_metrics.csv",
        low_memory=False,
    )

    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: build-metric row count differs",
        )

    required_build_metric_columns = {
        "Technique",
        "Build",
        "APFDc",
        "APFD",
    }

    if not required_build_metric_columns.issubset(
        build_metrics.columns
    ):
        return (
            None,
            f"{condition_key}: build-metric columns differ",
        )

    if sorted(
        build_metrics["Technique"]
        .astype(str)
        .unique()
        .tolist()
    ) != sorted(ALL_TECHNIQUES):
        return (
            None,
            f"{condition_key}: build-metric technique set differs",
        )

    if not build_metrics.groupby(
        "Technique"
    ).size().eq(
        EXPECTED_FAILING_EVAL_BUILDS
    ).all():
        return (
            None,
            f"{condition_key}: build-metric rows per technique differ",
        )

    build_metric_values = build_metrics[
        ["APFDc", "APFD"]
    ].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(float)

    if (
        not np.isfinite(
            build_metric_values
        ).all()
        or (
            (
                build_metric_values < 0
            )
            | (
                build_metric_values > 1
            )
        ).any()
    ):
        return (
            None,
            f"{condition_key}: build metrics invalid",
        )

    project_runs = pd.read_csv(
        condition_dir / "project_runs.csv",
        low_memory=False,
    )

    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: project-run row count differs",
        )

    if (
        "Technique" not in project_runs.columns
        or sorted(
            project_runs["Technique"]
            .astype(str)
            .tolist()
        ) != sorted(ALL_TECHNIQUES)
    ):
        return (
            None,
            f"{condition_key}: project-run technique set differs",
        )

    project_metric_columns = [
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]

    if not set(
        project_metric_columns
    ).issubset(
        project_runs.columns
    ):
        return (
            None,
            f"{condition_key}: project metric columns differ",
        )

    project_metric_values = project_runs[
        project_metric_columns
    ].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(float)

    if (
        not np.isfinite(
            project_metric_values
        ).all()
        or (
            (
                project_metric_values < 0
            )
            | (
                project_metric_values > 1
            )
        ).any()
    ):
        return (
            None,
            f"{condition_key}: project metrics invalid",
        )

    training_medians = pd.read_csv(
        condition_dir / "training_medians.csv",
        low_memory=False,
    )

    if len(
        training_medians
    ) != EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: training-median row count differs",
        )

    if (
        "Predictor" not in training_medians.columns
        or training_medians[
            "Predictor"
        ].astype(str).nunique()
        != EXPECTED_PREDICTORS
    ):
        return (
            None,
            f"{condition_key}: training-median predictor contract differs",
        )

    ranking_path = (
        condition_dir / "rankings.csv.gz"
    )

    ranking_rows = count_csv_data_rows(
        ranking_path
    )

    if ranking_rows != EXPECTED_RANKING_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: ranking row count differs; "
            f"expected={EXPECTED_RANKING_ROWS_PER_CONDITION}, "
            f"actual={ranking_rows}",
        )

    condition_manifest = directory_manifest(
        condition_dir
    )

    if len(
        condition_manifest
    ) != EXPECTED_FILES_PER_CONDITION:
        return (
            None,
            f"{condition_key}: condition manifest file count differs",
        )

    inventory = {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionOrder":
            int(plan_row.ConditionOrder),

        "ConditionKey":
            condition_key,

        "NoisePercent":
            int(plan_row.NoisePercent),

        "RepetitionSeed":
            int(plan_row.RepetitionSeed),

        "ConditionDirectory":
            str(condition_dir),

        "Status":
            CONDITION_STATUS,

        "Files":
            int(len(condition_manifest)),

        "ConditionBytes":
            int(
                condition_manifest[
                    "Bytes"
                ].sum()
            ),

        "ConditionRootSHA256":
            directory_root_hash(
                condition_manifest
            ),

        "RankingRows":
            int(summary["RankingRows"]),

        "BuildMetricRows":
            int(summary["BuildMetricRows"]),

        "ProjectRunRows":
            int(summary["ProjectRunRows"]),

        "ModelFitRows":
            int(summary["MLFits"]),

        "TrainingMedianRows":
            int(summary["TrainingMedianRows"]),

        "NumberFlipped":
            int(summary.get(
                "NumberFlipped",
                -1,
            )),

        "ModelLabelChanges":
            int(summary.get(
                "ModelLabelChanges",
                -1,
            )),

        "DependentRECChanges":
            int(summary.get(
                "DependentRECChanges",
                -1,
            )),

        "IndependentRECChanges":
            int(summary.get(
                "IndependentRECChanges",
                -1,
            )),

        "TrainingFailures":
            int(summary.get(
                "TrainingFailures",
                -1,
            )),

        "ConditionSeconds":
            float(summary.get(
                "ConditionSeconds",
                np.nan,
            )),

        "RandomKeySHA256":
            str(
                fingerprints[
                    "Random"
                ][
                    "KeySHA256"
                ]
            ),

        "RandomScoreSHA256":
            str(
                fingerprints[
                    "Random"
                ][
                    "ScoreSHA256"
                ]
            ),

        "RandomRankSHA256":
            str(
                fingerprints[
                    "Random"
                ][
                    "RankSHA256"
                ]
            ),

        "QTFAvgKeySHA256":
            str(
                fingerprints[
                    "QTF-Avg"
                ][
                    "KeySHA256"
                ]
            ),

        "QTFAvgScoreSHA256":
            str(
                fingerprints[
                    "QTF-Avg"
                ][
                    "ScoreSHA256"
                ]
            ),

        "QTFAvgRankSHA256":
            str(
                fingerprints[
                    "QTF-Avg"
                ][
                    "RankSHA256"
                ]
            ),
    }

    return (
        {
            "Inventory":
                inventory,

            "ConditionAudit":
                condition_audit,

            "ProjectRuns":
                project_runs,

            "BuildMetrics":
                build_metrics,

            "ModelFits":
                model_fits,

            "Manifest":
                condition_manifest,

            "Fingerprints":
                fingerprints,
        },
        "",
    )


# --------------------------------------------------------------------------------------------------
# 4. MOUNT, RESTORE SOURCE IF NEEDED, VERIFY UPSTREAM FREEZES
# --------------------------------------------------------------------------------------------------

master_started = time.perf_counter()

drive.mount(
    "/content/drive",
    force_remount=False,
)

source_restored = restore_source_if_needed()

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RUNTIME_CHECKPOINT_PATH,
    SMOKE_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
    FULL_RAW_RESULT_ROOT,
] + list(
    WORKER_CHECKPOINT_PATHS.values()
)

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing Project 24 master-finalization inputs:\n"
        + "\n".join(missing_paths)
    )

# Protect against accidental official Step 5A re-finalization.
if STEP5A_CHECKPOINT_PATH.is_file():
    existing = load_json(
        STEP5A_CHECKPOINT_PATH
    )

    if (
        existing.get("Status")
        == STEP5A_STATUS
        and bool(
            existing.get(
                "Full270ConditionExperimentComplete",
                False,
            )
        )
    ):
        raise RuntimeError(
            "Official Project 24 Step 5A is already frozen successfully. "
            "Do not rerun this master finalizer."
        )

    raise RuntimeError(
        "An unexpected pre-existing Project 24 Step 5A checkpoint exists. "
        "Stop and inspect it before finalization."
    )

checkpoint_hash_expectations = {
    "selection": (
        SELECTION_CHECKPOINT_PATH,
        EXPECTED_SELECTION_CHECKPOINT_SHA256,
    ),
    "REC": (
        REC_CHECKPOINT_PATH,
        EXPECTED_REC_CHECKPOINT_SHA256,
    ),
    "noise plan": (
        NOISE_PLAN_CHECKPOINT_PATH,
        EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    ),
    "runtime": (
        RUNTIME_CHECKPOINT_PATH,
        EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    ),
    "smoke": (
        SMOKE_CHECKPOINT_PATH,
        EXPECTED_SMOKE_CHECKPOINT_SHA256,
    ),
}

for label, (
    path,
    expected_sha,
) in checkpoint_hash_expectations.items():
    actual_sha = sha256_file(path)

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Frozen Project 24 {label} checkpoint SHA-256 differs.\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}"
        )

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)
noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
smoke_checkpoint = load_json(
    SMOKE_CHECKPOINT_PATH
)

for label, payload, expected_status in [
    (
        "selection checkpoint",
        selection_checkpoint,
        EXPECTED_SELECTION_STATUS,
    ),
    (
        "REC checkpoint",
        rec_checkpoint,
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "noise-plan checkpoint",
        noise_checkpoint,
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "runtime checkpoint",
        runtime_checkpoint,
        EXPECTED_STEP4A_STATUS,
    ),
    (
        "smoke checkpoint",
        smoke_checkpoint,
        EXPECTED_STEP4B_STATUS,
    ),
]:
    if payload.get(
        "Status"
    ) != expected_status:
        raise RuntimeError(
            f"{label} does not contain the frozen PASS status."
        )

    if (
        payload.get("Project") != PROJECT_NAME
        or payload.get("ProjectSlug") != PROJECT_SLUG
    ):
        raise RuntimeError(
            f"{label} Project 24 identity differs."
        )

if runtime_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active reservations differ."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint priority rule differs."
    )

if smoke_checkpoint.get(
    "RuntimeCheckpointSHA256"
) != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Smoke checkpoint is not linked to the frozen runtime checkpoint."
    )

if not bool(
    smoke_checkpoint.get(
        "ReadyForFull270ConditionExperiment",
        False,
    )
):
    raise RuntimeError(
        "Smoke checkpoint does not authorise the full experiment."
    )

upstream_manifest_failures = []

# These checkpoint schemas contain frozen file manifests.
for label, payload, manifest_key in [
    (
        "noise-plan",
        noise_checkpoint,
        "OutputManifest",
    ),
    (
        "runtime",
        runtime_checkpoint,
        "RuntimeOutputManifest",
    ),
    (
        "smoke",
        smoke_checkpoint,
        "OutputManifest",
    ),
]:
    failures = verify_manifest_entries(
        payload.get(
            manifest_key,
            [],
        ),
        label,
    )

    upstream_manifest_failures.extend(
        failures
    )

if upstream_manifest_failures:
    print(
        "\nUpstream manifest failures:"
    )
    display(
        pd.DataFrame(
            upstream_manifest_failures
        )
    )
    raise RuntimeError(
        "One or more frozen Project 24 upstream outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 5. REGISTRY, SOURCE, COHORT, AND CONDITION-PLAN IMMUTABILITY
# --------------------------------------------------------------------------------------------------

registry_sha_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 5A master finalization."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

(
    registry_project_number_column,
    registry_project_column,
    registry_status_column,
) = resolve_registry_columns(
    registry
)

registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)

if (
    len(registry)
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–23."
    )

if not registry[
    registry_status_column
].astype(str).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )

for number, identity in REQUIRED_REGISTERED_IDENTITIES.items():
    matching = registry.loc[
        registry_project_numbers.eq(
            number
        )
    ]

    if (
        len(matching) != 1
        or str(
            matching.iloc[0][
                registry_project_column
            ]
        ) != identity
    ):
        raise RuntimeError(
            "A frozen predecessor identity differs.\n"
            f"Project {number}; expected={identity}"
        )

if (
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).any()
    or registry[
        registry_project_column
    ].astype(str).eq(
        PROJECT_NAME
    ).any()
):
    raise RuntimeError(
        "Project 24 is unexpectedly already registered."
    )

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 24 source file is missing: {path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(path),
    })

current_source_manifest = pd.DataFrame(
    current_source_records
)

source_root_before = source_root_hash(
    current_source_manifest
)

if source_root_before != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 24 source root differs."
    )

if (
    len(current_source_manifest)
    != EXPECTED_SOURCE_FILES
    or int(
        current_source_manifest[
            "SizeBytes"
        ].sum()
    ) != EXPECTED_SOURCE_BYTES
):
    raise RuntimeError(
        "Frozen Project 24 source file/byte counts differ."
    )

cohort_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
]

cohort_hashes_before = {
    str(path):
        sha256_file(path)
    for path in cohort_paths
}

if pq.ParquetFile(
    RAW_TRAINING_COHORT_PATH
).metadata.num_rows != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError(
        "Frozen raw training cohort row count differs."
    )

if pq.ParquetFile(
    RAW_EVALUATION_COHORT_PATH
).metadata.num_rows != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError(
        "Frozen raw evaluation cohort row count differs."
    )

if pq.ParquetFile(
    MODEL_TRAINING_COHORT_PATH
).metadata.num_rows != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError(
        "Frozen model training cohort row count differs."
    )

if pq.ParquetFile(
    MODEL_EVALUATION_COHORT_PATH
).metadata.num_rows != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError(
        "Frozen model evaluation cohort row count differs."
    )

if pq.ParquetFile(
    RNG_MANIFEST_PATH
).metadata.num_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG manifest row count differs."
    )

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)

required_plan_columns = {
    "ConditionOrder",
    "ConditionID",
    "NoisePercent",
    "RepetitionSeed",
}

if not required_plan_columns.issubset(
    condition_plan.columns
):
    raise RuntimeError(
        "Condition plan is missing required columns:\n"
        + "\n".join(
            sorted(
                required_plan_columns
                - set(condition_plan.columns)
            )
        )
    )

condition_plan = (
    condition_plan.sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

expected_grid = [
    (
        seed,
        noise,
        f"noise_{noise:02d}__seed_{seed:02d}",
    )
    for seed in REPETITION_SEEDS
    for noise in NOISE_LEVELS
]

actual_grid = [
    (
        int(row.RepetitionSeed),
        int(row.NoisePercent),
        str(row.ConditionID),
    )
    for row in condition_plan.itertuples(
        index=False
    )
]

if (
    len(condition_plan)
    != EXPECTED_CONDITIONS
    or actual_grid != expected_grid
):
    raise RuntimeError(
        "Frozen Project 24 condition grid/order differs."
    )

if (
    condition_plan[
        "ConditionID"
    ].duplicated().any()
    or condition_plan.duplicated(
        [
            "NoisePercent",
            "RepetitionSeed",
        ]
    ).any()
):
    raise RuntimeError(
        "Frozen Project 24 condition plan contains duplicate coordinates."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY ALL SIX WORKER CHECKPOINTS AND SCIENTIFIC SHARD FREEZES
# --------------------------------------------------------------------------------------------------

print(
    "\nVerifying six worker checkpoints and exact scientific worker raw roots."
)

worker_records = []
worker_seed_union = []
worker_checkpoint_hashes_actual = {}
worker_checkpoint_hash_refreshes = 0
shared_equivalence_hashes = set()

for tag, expected in EXPECTED_WORKERS.items():
    checkpoint_path = (
        WORKER_CHECKPOINT_PATHS[tag]
    )

    actual_checkpoint_sha = sha256_file(
        checkpoint_path
    )

    worker_checkpoint_hashes_actual[
        tag
    ] = actual_checkpoint_sha

    reported_sha_still_current = bool(
        actual_checkpoint_sha
        == expected[
            "ReportedCheckpointSHA256"
        ]
    )

    if not reported_sha_still_current:
        worker_checkpoint_hash_refreshes += 1

    checkpoint = load_json(
        checkpoint_path
    )

    if (
        checkpoint.get("Project")
        != PROJECT_NAME
        or checkpoint.get("ProjectSlug")
        != PROJECT_SLUG
        or int(
            checkpoint.get(
                "ProjectNumber",
                -1,
            )
        ) != PROJECT_NUMBER
    ):
        raise RuntimeError(
            f"Worker {tag} project identity differs."
        )

    if checkpoint.get(
        "Status"
    ) != expected[
        "Status"
    ]:
        raise RuntimeError(
            f"Worker {tag} PASS status differs."
        )

    assigned_seeds = [
        int(value)
        for value in checkpoint.get(
            "AssignedSeeds",
            [],
        )
    ]

    if assigned_seeds != expected[
        "Seeds"
    ]:
        raise RuntimeError(
            f"Worker {tag} assigned seeds differ: "
            f"{assigned_seeds}"
        )

    worker_seed_union.extend(
        assigned_seeds
    )

    scalar_expectations = {
        "AssignedConditions":
            EXPECTED_WORKER_CONDITIONS,

        "CompletedConditions":
            EXPECTED_WORKER_CONDITIONS,

        "MLFits":
            EXPECTED_WORKER_MODEL_FITS,

        "RankingRows":
            EXPECTED_WORKER_RANKING_ROWS,

        "BuildMetricRows":
            EXPECTED_WORKER_BUILD_METRIC_ROWS,

        "ProjectRunRows":
            EXPECTED_WORKER_PROJECT_RUN_ROWS,

        "ConditionAuditRows":
            EXPECTED_WORKER_CONDITION_AUDIT_ROWS,

        "TrainingMedianRows":
            EXPECTED_WORKER_TRAINING_MEDIAN_ROWS,

        "WorkerRawFiles":
            EXPECTED_WORKER_RAW_FILES,

        "WorkerRawBytes":
            expected["RawBytes"],

        "FailedValidationChecks":
            0,

        "BaselineInvarianceFailures":
            0,

        "MasterStep5AWriteGuardFailures":
            0,
    }

    for key, expected_value in scalar_expectations.items():
        actual_value = int(
            checkpoint.get(
                key,
                -1,
            )
        )

        if actual_value != int(
            expected_value
        ):
            raise RuntimeError(
                f"Worker {tag} {key} differs. "
                f"Expected={expected_value}; "
                f"actual={actual_value}"
            )

    if checkpoint.get(
        "WorkerRawRootSHA256"
    ) != expected[
        "RawRootSHA256"
    ]:
        raise RuntimeError(
            f"Worker {tag} frozen raw-root SHA-256 differs."
        )

    if checkpoint.get(
        "SourceRootSHA256"
    ) != EXPECTED_SOURCE_ROOT_SHA256:
        raise RuntimeError(
            f"Worker {tag} source root differs."
        )

    if checkpoint.get(
        "RegistrySHA256"
    ) != EXPECTED_REGISTRY_SHA256:
        raise RuntimeError(
            f"Worker {tag} registry SHA differs."
        )

    if not bool(
        checkpoint.get(
            "WorkerShardComplete",
            False,
        )
    ):
        raise RuntimeError(
            f"Worker {tag} is not marked complete."
        )

    if bool(
        checkpoint.get(
            "OfficialStep5AComplete",
            True,
        )
    ):
        raise RuntimeError(
            f"Worker {tag} incorrectly marks official Step 5A complete."
        )

    worker_report_path = Path(
        str(
            checkpoint.get(
                "WorkerReportPath",
                "",
            )
        )
    )

    if (
        not worker_report_path.is_file()
        or sha256_file(
            worker_report_path
        )
        != str(
            checkpoint.get(
                "WorkerReportSHA256",
                "",
            )
        )
    ):
        raise RuntimeError(
            f"Worker {tag} report linkage failed."
        )

    worker_report = load_json(
        worker_report_path
    )

    if (
        worker_report.get("Status")
        != expected["Status"]
        or worker_report.get(
            "WorkerRawRootSHA256"
        )
        != expected["RawRootSHA256"]
        or int(
            worker_report.get(
                "WorkerRawBytes",
                -1,
            )
        )
        != expected["RawBytes"]
    ):
        raise RuntimeError(
            f"Worker {tag} report scientific freeze differs."
        )

    worker_output_manifest_failures = verify_manifest_entries(
        checkpoint.get(
            "WorkerOutputManifest",
            [],
        ),
        f"worker {tag}",
    )

    if worker_output_manifest_failures:
        display(
            pd.DataFrame(
                worker_output_manifest_failures
            )
        )

        raise RuntimeError(
            f"Worker {tag} private output manifest failed."
        )

    shared_equivalence_path = Path(
        str(
            checkpoint.get(
                "SharedSmokeEquivalencePath",
                "",
            )
        )
    )

    if shared_equivalence_path != ACCELERATED_EQUIVALENCE_PATH:
        raise RuntimeError(
            f"Worker {tag} shared equivalence path differs."
        )

    shared_equivalence_sha = str(
        checkpoint.get(
            "SharedSmokeEquivalenceSHA256",
            "",
        )
    )

    if (
        not ACCELERATED_EQUIVALENCE_PATH.is_file()
        or sha256_file(
            ACCELERATED_EQUIVALENCE_PATH
        )
        != shared_equivalence_sha
    ):
        raise RuntimeError(
            f"Worker {tag} shared equivalence linkage differs."
        )

    shared_equivalence_hashes.add(
        shared_equivalence_sha
    )

    # Independently recompute this worker's exact 360-file scientific raw root from the global raw root.
    worker_condition_keys = set(
        condition_plan.loc[
            condition_plan[
                "RepetitionSeed"
            ].isin(
                assigned_seeds
            ),
            "ConditionID",
        ].astype(str)
    )

    worker_raw_records = []

    for condition_key in sorted(
        worker_condition_keys
    ):
        condition_dir = (
            FULL_RAW_RESULT_ROOT
            / condition_key
        )

        condition_manifest = directory_manifest(
            condition_dir
        )

        for item in condition_manifest.itertuples(
            index=False
        ):
            worker_raw_records.append({
                "RelativePath":
                    f"{condition_key}/{item.RelativePath}",

                "Bytes":
                    int(item.Bytes),

                "SHA256":
                    str(item.SHA256),
            })

    worker_raw_manifest = pd.DataFrame(
        worker_raw_records,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    ).sort_values(
        "RelativePath",
        kind="mergesort",
    ).reset_index(
        drop=True
    )

    worker_raw_files_recomputed = int(
        len(
            worker_raw_manifest
        )
    )

    worker_raw_bytes_recomputed = int(
        worker_raw_manifest[
            "Bytes"
        ].sum()
    )

    worker_raw_root_recomputed = directory_root_hash(
        worker_raw_manifest
    )

    if (
        worker_raw_files_recomputed
        != EXPECTED_WORKER_RAW_FILES
        or worker_raw_bytes_recomputed
        != expected["RawBytes"]
        or worker_raw_root_recomputed
        != expected["RawRootSHA256"]
    ):
        raise RuntimeError(
            f"Worker {tag} independent raw-root recomputation differs."
        )

    worker_records.append({
        "WorkerTag":
            tag,

        "AssignedSeedsJSON":
            json.dumps(
                assigned_seeds
            ),

        "Status":
            checkpoint.get(
                "Status"
            ),

        "ReportedCheckpointSHA256":
            expected[
                "ReportedCheckpointSHA256"
            ],

        "ActualCheckpointSHA256":
            actual_checkpoint_sha,

        "ReportedCheckpointSHAStillCurrent":
            reported_sha_still_current,

        "CheckpointMetadataRefreshed":
            not reported_sha_still_current,

        "WorkerRawFiles":
            worker_raw_files_recomputed,

        "WorkerRawBytes":
            worker_raw_bytes_recomputed,

        "ExpectedWorkerRawRootSHA256":
            expected[
                "RawRootSHA256"
            ],

        "ActualWorkerRawRootSHA256":
            worker_raw_root_recomputed,

        "WorkerReportPath":
            str(
                worker_report_path
            ),

        "WorkerReportSHA256":
            sha256_file(
                worker_report_path
            ),

        "SharedSmokeEquivalenceSHA256":
            shared_equivalence_sha,

        "Pass":
            True,
    })

worker_audit = pd.DataFrame(
    worker_records
)

if len(
    shared_equivalence_hashes
) != 1:
    raise RuntimeError(
        "The six workers do not reference one identical accelerated-equivalence freeze."
    )

if (
    sorted(
        worker_seed_union
    ) != REPETITION_SEEDS
    or len(
        worker_seed_union
    ) != len(
        set(
            worker_seed_union
        )
    )
):
    raise RuntimeError(
        "Worker seed shards are not disjoint exact coverage of seeds 1–30."
    )

print(
    "Workers verified:",
    len(worker_audit),
    "/ 6",
)

print(
    "Worker checkpoint metadata refreshes:",
    worker_checkpoint_hash_refreshes,
)

if worker_checkpoint_hash_refreshes:
    print(
        "NOTE: refreshed worker checkpoint SHA values are acceptable only because "
        "their exact scientific raw roots, output manifests, report linkage, seed shards, "
        "and all 270 raw conditions are independently revalidated below."
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY SHARED ACCELERATED SMOKE-EQUIVALENCE FREEZE
# --------------------------------------------------------------------------------------------------

equivalence = pd.read_csv(
    ACCELERATED_EQUIVALENCE_PATH,
    low_memory=False,
)

if (
    len(equivalence) != 2
    or "ConditionKey" not in equivalence.columns
    or "Pass" not in equivalence.columns
    or sorted(
        equivalence[
            "ConditionKey"
        ].astype(str).tolist()
    ) != sorted(
        SMOKE_EQUIVALENCE_KEYS
    )
    or not equivalence[
        "Pass"
    ].map(
        as_bool
    ).all()
):
    raise RuntimeError(
        "Frozen accelerated smoke-equivalence audit differs."
    )


# --------------------------------------------------------------------------------------------------
# 8. INDEPENDENTLY REVALIDATE ALL 270 RAW CONDITIONS AND REBUILD MASTER AGGREGATES
# --------------------------------------------------------------------------------------------------

print(
    "\nIndependently revalidating all 270 raw condition directories."
)

inventory_records = []
condition_audit_frames = []
project_run_frames = []
build_metric_frames = []
model_fit_frames = []
raw_manifest_records = []
baseline_fingerprint_records = []
condition_validation_errors = []

validation_started = time.perf_counter()

for index, plan_row in enumerate(
    condition_plan.itertuples(
        index=False
    ),
    start=1,
):
    condition_key = str(
        plan_row.ConditionID
    )

    condition_dir = (
        FULL_RAW_RESULT_ROOT
        / condition_key
    )

    validated, error = validate_condition_read_only(
        condition_dir,
        plan_row,
    )

    if validated is None:
        condition_validation_errors.append({
            "ConditionKey":
                condition_key,

            "Error":
                error,
        })
        continue

    inventory_records.append(
        validated[
            "Inventory"
        ]
    )

    condition_audit_frames.append(
        validated[
            "ConditionAudit"
        ]
    )

    project_run_frames.append(
        validated[
            "ProjectRuns"
        ]
    )

    build_metric_frames.append(
        validated[
            "BuildMetrics"
        ]
    )

    model_fit_frames.append(
        validated[
            "ModelFits"
        ]
    )

    fingerprints = validated[
        "Fingerprints"
    ]

    for technique in INVARIANT_BASELINES:
        fingerprint = fingerprints[
            technique
        ]

        baseline_fingerprint_records.append({
            "ConditionKey":
                condition_key,

            "NoisePercent":
                int(
                    plan_row.NoisePercent
                ),

            "RepetitionSeed":
                int(
                    plan_row.RepetitionSeed
                ),

            "Technique":
                technique,

            "Rows":
                int(
                    fingerprint[
                        "Rows"
                    ]
                ),

            "KeySHA256":
                str(
                    fingerprint[
                        "KeySHA256"
                    ]
                ),

            "ScoreSHA256":
                str(
                    fingerprint[
                        "ScoreSHA256"
                    ]
                ),

            "RankSHA256":
                str(
                    fingerprint[
                        "RankSHA256"
                    ]
                ),
        })

    condition_manifest = validated[
        "Manifest"
    ]

    for item in condition_manifest.itertuples(
        index=False
    ):
        raw_manifest_records.append({
            "RelativePath":
                f"{condition_key}/{item.RelativePath}",

            "Bytes":
                int(
                    item.Bytes
                ),

            "SHA256":
                str(
                    item.SHA256
                ),
        })

    if (
        index % 25 == 0
        or index == EXPECTED_CONDITIONS
    ):
        print(
            f"  Conditions revalidated: {index}/{EXPECTED_CONDITIONS} "
            f"| elapsed seconds: "
            f"{round(time.perf_counter() - validation_started, 2)}"
        )

if condition_validation_errors:
    print(
        "\nCondition validation failures:"
    )

    display(
        pd.DataFrame(
            condition_validation_errors
        )
    )

    raise RuntimeError(
        "One or more Project 24 raw condition directories failed independent master validation."
    )

condition_inventory = (
    pd.DataFrame(
        inventory_records
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

combined_condition_audit = pd.concat(
    condition_audit_frames,
    ignore_index=True,
)

combined_project_runs = pd.concat(
    project_run_frames,
    ignore_index=True,
)

combined_build_metrics = pd.concat(
    build_metric_frames,
    ignore_index=True,
)

combined_model_fits = pd.concat(
    model_fit_frames,
    ignore_index=True,
)

baseline_fingerprints = pd.DataFrame(
    baseline_fingerprint_records
)

raw_manifest = (
    pd.DataFrame(
        raw_manifest_records,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )
    .sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

raw_files = int(
    len(
        raw_manifest
    )
)

raw_bytes = int(
    raw_manifest[
        "Bytes"
    ].sum()
)

raw_root_sha256 = directory_root_hash(
    raw_manifest
)


# --------------------------------------------------------------------------------------------------
# 9. MASTER BASELINE INVARIANCE + GLOBAL RAW-ROOT CROSS-CHECK
# --------------------------------------------------------------------------------------------------

baseline_invariance_records = []

for repetition_seed in REPETITION_SEEDS:
    for technique in INVARIANT_BASELINES:
        rows = baseline_fingerprints.loc[
            baseline_fingerprints[
                "RepetitionSeed"
            ].eq(
                repetition_seed
            )
            & baseline_fingerprints[
                "Technique"
            ].eq(
                technique
            )
        ]

        key_variants = int(
            rows[
                "KeySHA256"
            ].nunique()
        )

        score_variants = int(
            rows[
                "ScoreSHA256"
            ].nunique()
        )

        rank_variants = int(
            rows[
                "RankSHA256"
            ].nunique()
        )

        passed = bool(
            len(rows)
            == len(NOISE_LEVELS)
            and key_variants == 1
            and score_variants == 1
            and rank_variants == 1
        )

        baseline_invariance_records.append({
            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "Conditions":
                int(
                    len(rows)
                ),

            "KeyVariantsAcrossNoise":
                key_variants,

            "ScoreVariantsAcrossNoise":
                score_variants,

            "RankVariantsAcrossNoise":
                rank_variants,

            "Pass":
                passed,
        })

baseline_invariance = pd.DataFrame(
    baseline_invariance_records
)

baseline_invariance_failures = int(
    (
        ~baseline_invariance[
            "Pass"
        ]
    ).sum()
)

qtf_global_key_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints[
            "Technique"
        ].eq(
            "QTF-Avg"
        ),
        "KeySHA256",
    ].nunique()
)

qtf_global_score_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints[
            "Technique"
        ].eq(
            "QTF-Avg"
        ),
        "ScoreSHA256",
    ].nunique()
)

qtf_global_rank_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints[
            "Technique"
        ].eq(
            "QTF-Avg"
        ),
        "RankSHA256",
    ].nunique()
)

# Completely independent directory walk/hash over the raw root.
raw_manifest_direct = directory_manifest(
    FULL_RAW_RESULT_ROOT
)

raw_root_sha256_direct = directory_root_hash(
    raw_manifest_direct
)

raw_manifest_keyed = raw_manifest.set_index(
    "RelativePath"
).sort_index()

raw_manifest_direct_keyed = raw_manifest_direct.set_index(
    "RelativePath"
).sort_index()

raw_manifest_exact_match = bool(
    raw_manifest_keyed.equals(
        raw_manifest_direct_keyed
    )
)


# --------------------------------------------------------------------------------------------------
# 10. MASTER VALIDATION GATE
# --------------------------------------------------------------------------------------------------

validation = []

add_check(
    validation,
    "Workers verified",
    6,
    len(worker_audit),
    len(worker_audit) == 6,
)

add_check(
    validation,
    "Worker seeds cover 1–30 exactly once",
    REPETITION_SEEDS,
    sorted(worker_seed_union),
    (
        sorted(worker_seed_union)
        == REPETITION_SEEDS
        and len(worker_seed_union)
        == len(set(worker_seed_union))
    ),
)

add_check(
    validation,
    "Worker scientific raw-root mismatches",
    0,
    int(
        (
            worker_audit[
                "ExpectedWorkerRawRootSHA256"
            ]
            != worker_audit[
                "ActualWorkerRawRootSHA256"
            ]
        ).sum()
    ),
    bool(
        (
            worker_audit[
                "ExpectedWorkerRawRootSHA256"
            ]
            == worker_audit[
                "ActualWorkerRawRootSHA256"
            ]
        ).all()
    ),
)

add_check(
    validation,
    "Conditions",
    EXPECTED_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory)
    == EXPECTED_CONDITIONS,
)

add_check(
    validation,
    "Duplicate condition keys",
    0,
    int(
        condition_inventory[
            "ConditionKey"
        ].duplicated(
            keep=False
        ).sum()
    ),
    not condition_inventory[
        "ConditionKey"
    ].duplicated(
        keep=False
    ).any(),
)

add_check(
    validation,
    "Files per condition",
    EXPECTED_FILES_PER_CONDITION,
    sorted(
        condition_inventory[
            "Files"
        ].unique().tolist()
    ),
    condition_inventory[
        "Files"
    ].eq(
        EXPECTED_FILES_PER_CONDITION
    ).all(),
)

add_check(
    validation,
    "Raw files",
    EXPECTED_RAW_FILES,
    raw_files,
    raw_files == EXPECTED_RAW_FILES,
)

add_check(
    validation,
    "Raw bytes",
    EXPECTED_RAW_BYTES,
    raw_bytes,
    raw_bytes == EXPECTED_RAW_BYTES,
)

add_check(
    validation,
    "Direct raw-root file count",
    EXPECTED_RAW_FILES,
    len(raw_manifest_direct),
    len(raw_manifest_direct)
    == EXPECTED_RAW_FILES,
)

add_check(
    validation,
    "Direct raw-root bytes",
    EXPECTED_RAW_BYTES,
    int(
        raw_manifest_direct[
            "Bytes"
        ].sum()
    ),
    int(
        raw_manifest_direct[
            "Bytes"
        ].sum()
    ) == EXPECTED_RAW_BYTES,
)

add_check(
    validation,
    "Collected/direct raw manifests identical",
    True,
    raw_manifest_exact_match,
    raw_manifest_exact_match,
)

add_check(
    validation,
    "Collected/direct raw-root SHA identical",
    raw_root_sha256,
    raw_root_sha256_direct,
    raw_root_sha256
    == raw_root_sha256_direct,
)

add_check(
    validation,
    "Ranking rows",
    EXPECTED_TOTAL_RANKING_ROWS,
    int(
        condition_inventory[
            "RankingRows"
        ].sum()
    ),
    int(
        condition_inventory[
            "RankingRows"
        ].sum()
    ) == EXPECTED_TOTAL_RANKING_ROWS,
)

add_check(
    validation,
    "Build-metric rows",
    EXPECTED_TOTAL_BUILD_METRIC_ROWS,
    len(combined_build_metrics),
    len(combined_build_metrics)
    == EXPECTED_TOTAL_BUILD_METRIC_ROWS,
)

add_check(
    validation,
    "Project-run rows",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
    len(combined_project_runs),
    len(combined_project_runs)
    == EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)

add_check(
    validation,
    "Model fits",
    EXPECTED_TOTAL_MODEL_FITS,
    len(combined_model_fits),
    len(combined_model_fits)
    == EXPECTED_TOTAL_MODEL_FITS,
)

add_check(
    validation,
    "Condition-audit rows",
    EXPECTED_TOTAL_CONDITION_AUDIT_ROWS,
    len(combined_condition_audit),
    len(combined_condition_audit)
    == EXPECTED_TOTAL_CONDITION_AUDIT_ROWS,
)

add_check(
    validation,
    "Training-median rows",
    EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS,
    int(
        condition_inventory[
            "TrainingMedianRows"
        ].sum()
    ),
    int(
        condition_inventory[
            "TrainingMedianRows"
        ].sum()
    ) == EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS,
)

add_check(
    validation,
    "Zero-noise conditions",
    30,
    int(
        condition_inventory[
            "NoisePercent"
        ].eq(0).sum()
    ),
    int(
        condition_inventory[
            "NoisePercent"
        ].eq(0).sum()
    ) == 30,
)

zero_audit = combined_condition_audit.loc[
    pd.to_numeric(
        combined_condition_audit[
            "NoisePercent"
        ],
        errors="raise",
    ).eq(0)
]

positive_audit = combined_condition_audit.loc[
    pd.to_numeric(
        combined_condition_audit[
            "NoisePercent"
        ],
        errors="raise",
    ).gt(0)
]

add_check(
    validation,
    "Zero-noise raw flips",
    0,
    int(
        pd.to_numeric(
            zero_audit[
                "NumberFlipped"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            zero_audit[
                "NumberFlipped"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Zero-noise model-label changes",
    0,
    int(
        pd.to_numeric(
            zero_audit[
                "ModelLabelChanges"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            zero_audit[
                "ModelLabelChanges"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Zero-noise dependent REC changes",
    0,
    int(
        pd.to_numeric(
            zero_audit[
                "DependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            zero_audit[
                "DependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Positive-noise raw-change violations",
    0,
    int(
        pd.to_numeric(
            positive_audit[
                "NumberFlipped"
            ],
            errors="raise",
        ).le(0).sum()
    ),
    not pd.to_numeric(
        positive_audit[
            "NumberFlipped"
        ],
        errors="raise",
    ).le(0).any(),
)

add_check(
    validation,
    "Positive-noise model-change violations",
    0,
    int(
        pd.to_numeric(
            positive_audit[
                "ModelLabelChanges"
            ],
            errors="raise",
        ).le(0).sum()
    ),
    not pd.to_numeric(
        positive_audit[
            "ModelLabelChanges"
        ],
        errors="raise",
    ).le(0).any(),
)

add_check(
    validation,
    "Positive-noise dependent-REC violations",
    0,
    int(
        pd.to_numeric(
            positive_audit[
                "DependentRECChanges"
            ],
            errors="raise",
        ).le(0).sum()
    ),
    not pd.to_numeric(
        positive_audit[
            "DependentRECChanges"
        ],
        errors="raise",
    ).le(0).any(),
)

add_check(
    validation,
    "Independent REC changes",
    0,
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Independent reconstruction mismatches",
    0,
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentReconstructionMismatches"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentReconstructionMismatches"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

noise_plan_hash_mismatches = 0

for expected_column, actual_column in [
    (
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
    ),
    (
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
    ),
    (
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
    ),
]:
    noise_plan_hash_mismatches += int(
        combined_condition_audit[
            expected_column
        ].astype(str).ne(
            combined_condition_audit[
                actual_column
            ].astype(str)
        ).sum()
    )

add_check(
    validation,
    "Noise-plan hash mismatches",
    0,
    noise_plan_hash_mismatches,
    noise_plan_hash_mismatches == 0,
)

add_check(
    validation,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)

add_check(
    validation,
    "QTF global key variants",
    1,
    qtf_global_key_variants,
    qtf_global_key_variants == 1,
)

add_check(
    validation,
    "QTF global score variants",
    1,
    qtf_global_score_variants,
    qtf_global_score_variants == 1,
)

add_check(
    validation,
    "QTF global rank variants",
    1,
    qtf_global_rank_variants,
    qtf_global_rank_variants == 1,
)

model_fit_status_failures = int(
    (
        ~combined_model_fits[
            "Status"
        ].astype(str).eq(
            "PASS_MODEL_FIT"
        )
    ).sum()
)

add_check(
    validation,
    "Model-fit status failures",
    0,
    model_fit_status_failures,
    model_fit_status_failures == 0,
)

for frame, columns, label in [
    (
        combined_project_runs,
        [
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ],
        "Project metrics",
    ),
    (
        combined_build_metrics,
        [
            "APFDc",
            "APFD",
        ],
        "Build metrics",
    ),
]:
    values = frame[
        columns
    ].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(float)

    bad = int(
        (
            ~np.isfinite(
                values
            )
        ).sum()
        + (
            (
                values < 0
            )
            | (
                values > 1
            )
        ).sum()
    )

    add_check(
        validation,
        f"{label} non-finite/out-of-range values",
        0,
        bad,
        bad == 0,
    )

add_check(
    validation,
    "Shared smoke-equivalence rows",
    2,
    len(equivalence),
    len(equivalence) == 2,
)

add_check(
    validation,
    "Shared smoke-equivalence failures",
    0,
    int(
        (
            ~equivalence[
                "Pass"
            ].map(
                as_bool
            )
        ).sum()
    ),
    equivalence[
        "Pass"
    ].map(
        as_bool
    ).all(),
)

add_check(
    validation,
    "Registry SHA unchanged",
    EXPECTED_REGISTRY_SHA256,
    sha256_file(
        REGISTRY_PATH
    ),
    sha256_file(
        REGISTRY_PATH
    ) == EXPECTED_REGISTRY_SHA256,
)

add_check(
    validation,
    "Source root unchanged",
    EXPECTED_SOURCE_ROOT_SHA256,
    source_root_before,
    source_root_before
    == EXPECTED_SOURCE_ROOT_SHA256,
)

for cohort_path, expected_sha in cohort_hashes_before.items():
    actual_sha = sha256_file(
        Path(
            cohort_path
        )
    )

    add_check(
        validation,
        f"Frozen cohort unchanged: {Path(cohort_path).name}",
        expected_sha,
        actual_sha,
        actual_sha == expected_sha,
    )

add_check(
    validation,
    "Registry Project 24 rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Master finalizer model fits",
    0,
    0,
    True,
)

add_check(
    validation,
    "Master finalizer condition executions",
    0,
    0,
    True,
)

step5a_validation = pd.DataFrame(
    validation
)

failed_validation = step5a_validation.loc[
    ~step5a_validation[
        "Pass"
    ]
]

print(
    "\nProject 24 master Step 5A validation:"
)

display(
    step5a_validation
)

if not failed_validation.empty:
    print(
        "\nFAILED MASTER CHECKS:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 24 STEP 5A MASTER FINALIZATION VALIDATION FAILED. "
        "No official Step 5A freeze was written."
    )


# --------------------------------------------------------------------------------------------------
# 11. WRITE OFFICIAL STEP 5A AGGREGATES ONLY AFTER ALL CHECKS PASS
# --------------------------------------------------------------------------------------------------

FULL_EXPERIMENT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_csv(
    CONDITION_INVENTORY_PATH,
    condition_inventory,
)

atomic_csv(
    RAW_MANIFEST_PATH,
    raw_manifest,
)

atomic_csv(
    BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)

atomic_csv(
    COMBINED_CONDITION_AUDIT_PATH,
    combined_condition_audit,
)

atomic_csv(
    COMBINED_PROJECT_RUNS_PATH,
    combined_project_runs,
)

atomic_csv(
    COMBINED_BUILD_METRICS_PATH,
    combined_build_metrics,
)

atomic_csv(
    COMBINED_MODEL_FITS_PATH,
    combined_model_fits,
)

atomic_csv(
    WORKER_CHECKPOINT_AUDIT_PATH,
    worker_audit,
)

atomic_csv(
    STEP5A_VALIDATION_PATH,
    step5a_validation,
)

aggregate_output_paths = [
    CONDITION_INVENTORY_PATH,
    RAW_MANIFEST_PATH,
    BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH,
    COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH,
    COMBINED_MODEL_FITS_PATH,
    WORKER_CHECKPOINT_AUDIT_PATH,
    STEP5A_VALIDATION_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
]

aggregate_output_manifest = [
    {
        "Path":
            str(path),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in aggregate_output_paths
]

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

master_execution_seconds = (
    time.perf_counter()
    - master_started
)

worker_checkpoint_records = [
    {
        "WorkerTag":
            row.WorkerTag,

        "AssignedSeedsJSON":
            row.AssignedSeedsJSON,

        "Status":
            row.Status,

        "ActualCheckpointSHA256":
            row.ActualCheckpointSHA256,

        "ReportedCheckpointSHA256":
            row.ReportedCheckpointSHA256,

        "CheckpointMetadataRefreshed":
            bool(
                row.CheckpointMetadataRefreshed
            ),

        "WorkerRawFiles":
            int(
                row.WorkerRawFiles
            ),

        "WorkerRawBytes":
            int(
                row.WorkerRawBytes
            ),

        "WorkerRawRootSHA256":
            row.ActualWorkerRawRootSHA256,

        "WorkerReportSHA256":
            row.WorkerReportSHA256,

        "SharedSmokeEquivalenceSHA256":
            row.SharedSmokeEquivalenceSHA256,
    }
    for row in worker_audit.itertuples(
        index=False
    )
]

report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_STATUS,

    "Implementation":
        MASTER_FINALIZER_IMPLEMENTATION,

    "CompletedAtUTC":
        completed_at_utc,

    "ExecutionMode":
        "PARALLEL_SIX_WORKERS_THEN_ZERO_FIT_MASTER_FINALIZATION",

    "AcceleratedEngineVersion":
        ACCELERATED_ENGINE_VERSION,

    "SelectionCheckpointSHA256":
        EXPECTED_SELECTION_CHECKPOINT_SHA256,

    "RECCheckpointSHA256":
        EXPECTED_REC_CHECKPOINT_SHA256,

    "NoisePlanCheckpointSHA256":
        EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,

    "RuntimeCheckpointSHA256":
        EXPECTED_RUNTIME_CHECKPOINT_SHA256,

    "SmokeCheckpointSHA256":
        EXPECTED_SMOKE_CHECKPOINT_SHA256,

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA256,

    "RegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "WorkerCheckpoints":
        worker_checkpoint_records,

    "ActualWorkerCheckpointSHA256":
        worker_checkpoint_hashes_actual,

    "WorkerCheckpointMetadataRefreshes":
        worker_checkpoint_hash_refreshes,

    "WorkersVerified":
        len(worker_audit),

    "SeedsCoveredExactlyOnce":
        True,

    "Conditions":
        len(condition_inventory),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "MLFits":
        len(combined_model_fits),

    "RankingRows":
        int(
            condition_inventory[
                "RankingRows"
            ].sum()
        ),

    "BuildMetricRows":
        len(combined_build_metrics),

    "ProjectRunRows":
        len(combined_project_runs),

    "ConditionAuditRows":
        len(combined_condition_audit),

    "TrainingMedianRows":
        int(
            condition_inventory[
                "TrainingMedianRows"
            ].sum()
        ),

    "RawFiles":
        raw_files,

    "RawBytes":
        raw_bytes,

    "RawRootSHA256":
        raw_root_sha256,

    "BaselineInvarianceFailures":
        baseline_invariance_failures,

    "QTFGlobalKeyVariants":
        qtf_global_key_variants,

    "QTFGlobalScoreVariants":
        qtf_global_score_variants,

    "QTFGlobalRankVariants":
        qtf_global_rank_variants,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "SharedSmokeEquivalenceSHA256":
        sha256_file(
            ACCELERATED_EQUIVALENCE_PATH
        ),

    "ValidationChecks":
        len(step5a_validation),

    "FailedValidationChecks":
        len(failed_validation),

    "MasterFinalizerModelFits":
        0,

    "MasterFinalizerConditionExecutions":
        0,

    "SourceRestoredInThisRuntime":
        source_restored,

    "MasterExecutionSeconds":
        float(
            master_execution_seconds
        ),

    "AggregateOutputManifest":
        aggregate_output_manifest,

    "RegistryModified":
        False,

    "Projects1To23Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,
}

atomic_json(
    STEP5A_REPORT_PATH,
    report_payload,
)

checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_24_FULL_270_CONDITION_STEP5A_FREEZE",

    "Full270ConditionExperimentComplete":
        True,

    "RawResultRootFrozen":
        True,

    "RawResultRootReadOnlyDuringMasterFinalization":
        True,

    "AllSixWorkerScientificRawRootsVerified":
        True,

    "All270ConditionsIndependentlyRevalidated":
        True,

    "EvaluationCohortImmutable":
        True,

    "ReadyForStep5B":
        True,

    "NextRequiredStep":
        "PROJECT 24 STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION",
}

atomic_json(
    STEP5A_CHECKPOINT_PATH,
    checkpoint_payload,
)

step5a_checkpoint_sha256 = sha256_file(
    STEP5A_CHECKPOINT_PATH
)

status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "Conditions":
        EXPECTED_CONDITIONS,

    "MLFits":
        EXPECTED_TOTAL_MODEL_FITS,

    "RawFiles":
        EXPECTED_RAW_FILES,

    "RawBytes":
        EXPECTED_RAW_BYTES,

    "RawRootSHA256":
        raw_root_sha256,

    "Checkpoint":
        str(
            STEP5A_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        step5a_checkpoint_sha256,

    "MasterFinalizerModelFits":
        0,

    "MasterFinalizerConditionExecutions":
        0,

    "RegistryModified":
        False,

    "ReadyForStep5B":
        True,

    "NextRequiredStep":
        "PROJECT 24 STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION",
}

atomic_json(
    STEP5A_STATUS_PATH,
    status_payload,
)

atomic_json(
    RUN_PROGRESS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "Status":
            STEP5A_STATUS,

        "ConditionsComplete":
            EXPECTED_CONDITIONS,

        "ConditionsTotal":
            EXPECTED_CONDITIONS,

        "WorkerShardsComplete":
            6,

        "WorkerShardsTotal":
            6,

        "OfficialStep5AComplete":
            True,

        "Step5ACheckpointSHA256":
            step5a_checkpoint_sha256,
    },
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL READBACK + RAW/SOURCE/REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    STEP5A_CHECKPOINT_PATH
)

report_readback = load_json(
    STEP5A_REPORT_PATH
)

status_readback = load_json(
    STEP5A_STATUS_PATH
)

for label, payload in [
    (
        "Step 5A checkpoint",
        checkpoint_readback,
    ),
    (
        "Step 5A report",
        report_readback,
    ),
    (
        "Step 5A status",
        status_readback,
    ),
]:
    if payload.get(
        "Status"
    ) != STEP5A_STATUS:
        raise RuntimeError(
            f"{label} readback failed."
        )

if not bool(
    checkpoint_readback.get(
        "ReadyForStep5B",
        False,
    )
):
    raise RuntimeError(
        "Step 5A checkpoint does not authorize Step 5B."
    )

aggregate_manifest_failures = verify_manifest_entries(
    checkpoint_readback.get(
        "AggregateOutputManifest",
        [],
    ),
    "official Step 5A",
)

if aggregate_manifest_failures:
    display(
        pd.DataFrame(
            aggregate_manifest_failures
        )
    )

    raise RuntimeError(
        "Official Step 5A aggregate-output readback failed."
    )

raw_manifest_final = directory_manifest(
    FULL_RAW_RESULT_ROOT
)

if (
    len(raw_manifest_final)
    != EXPECTED_RAW_FILES
    or int(
        raw_manifest_final[
            "Bytes"
        ].sum()
    ) != EXPECTED_RAW_BYTES
    or directory_root_hash(
        raw_manifest_final
    ) != raw_root_sha256
):
    raise RuntimeError(
        "Project 24 raw root changed during official master writes."
    )

if sha256_file(
    REGISTRY_PATH
) != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry changed during master finalization."
    )

final_source_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    })

if source_root_hash(
    pd.DataFrame(
        final_source_records
    )
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Project 24 source changed during master finalization."
    )

for path_string, expected_sha in cohort_hashes_before.items():
    if sha256_file(
        Path(
            path_string
        )
    ) != expected_sha:
        raise RuntimeError(
            "A frozen Project 24 cohort/plan changed during master finalization:\n"
            f"{path_string}"
        )


# --------------------------------------------------------------------------------------------------
# 13. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 140
)

print(
    "=== PROJECT 24 CELL 9 / STEP 5A RESULT ==="
)

print(
    "=" * 140
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print()

print(
    "Parallel workers:"
)

print(
    "Workers verified:",
    len(worker_audit),
    "/ 6",
)

print(
    "Seeds covered exactly once:",
    sorted(worker_seed_union)
    == REPETITION_SEEDS
    and len(worker_seed_union)
    == len(set(worker_seed_union)),
)

print(
    "Worker checkpoint metadata refreshes:",
    worker_checkpoint_hash_refreshes,
)

print()

print(
    "Full experiment:"
)

print(
    "Conditions:",
    len(condition_inventory),
    "/",
    EXPECTED_CONDITIONS,
)

print(
    "ML fits:",
    len(combined_model_fits),
    "/",
    EXPECTED_TOTAL_MODEL_FITS,
)

print(
    "Ranking rows:",
    int(
        condition_inventory[
            "RankingRows"
        ].sum()
    ),
    "/",
    EXPECTED_TOTAL_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    len(combined_build_metrics),
    "/",
    EXPECTED_TOTAL_BUILD_METRIC_ROWS,
)

print(
    "Project-run rows:",
    len(combined_project_runs),
    "/",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)

print(
    "Condition-audit rows:",
    len(combined_condition_audit),
    "/",
    EXPECTED_TOTAL_CONDITION_AUDIT_ROWS,
)

print(
    "Training-median rows:",
    int(
        condition_inventory[
            "TrainingMedianRows"
        ].sum()
    ),
    "/",
    EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS,
)

print()

print(
    "Raw result freeze:"
)

print(
    "Raw files:",
    raw_files,
    "/",
    EXPECTED_RAW_FILES,
)

print(
    "Raw bytes:",
    raw_bytes,
    "/",
    EXPECTED_RAW_BYTES,
)

print(
    "Raw root SHA-256:",
    raw_root_sha256,
)

print()

print(
    "Baselines and metrics:"
)

print(
    "Baseline invariance failures:",
    baseline_invariance_failures,
)

print(
    "QTF global score variants:",
    qtf_global_score_variants,
)

print(
    "QTF global rank variants:",
    qtf_global_rank_variants,
)

print(
    "Primary / secondary metrics: APFDc / APFD"
)

print()

print(
    "Isolation:"
)

print(
    "Completion registry unchanged:",
    sha256_file(
        REGISTRY_PATH
    )
    == EXPECTED_REGISTRY_SHA256,
)

print(
    "Projects 1–23 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Master-finalizer model fits:",
    0,
)

print(
    "Master-finalizer condition executions:",
    0,
)

print()

print(
    "Validation:"
)

print(
    "Checks:",
    len(step5a_validation),
)

print(
    "Failed checks:",
    len(failed_validation),
)

print()

print(
    "Official Step 5A checkpoint:"
)

print(
    STEP5A_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    step5a_checkpoint_sha256,
)

print()

print(
    "Master finalization seconds:",
    round(
        master_execution_seconds,
        2,
    ),
)

print()

print(
    "Next required step:",
    "PROJECT 24 STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION",
)

print()

print(
    "STATUS:",
    STEP5A_STATUS,
)

print(
    "=" * 140
)


=== PROJECT 24 CELL 9 / STEP 5A: PARALLEL MASTER FINALIZATION — ZERO FIT ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Restoring only SonarSource@sonarqube from the frozen TCP-CI archive into this master runtime.
Project source files restored: 6


RuntimeError: Project 24 is unexpectedly already registered.

In [2]:
# ==================================================================================================
# PROJECT 24 — CELL 9 / STEP 5A PARALLEL MASTER FINALIZATION
# ZERO-FIT REVALIDATION, AGGREGATION, RAW-ROOT FREEZE, AND OFFICIAL CHECKPOINT
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS AS THE NEXT NEW CELL IN THE RECONNECTED MASTER Thesis_project_24.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the six completed Project 24 Step 5A worker shards;
# - prove the six seed shards are disjoint and cover seeds 1–30 exactly;
# - independently revalidate all 270 raw condition directories without fitting any model;
# - rebuild the official compact Step 5A aggregate outputs from the frozen raw condition files;
# - independently hash and freeze the complete 2,160-file Project 24 raw-result root;
# - create the official Project 24 Step 5A report/status/checkpoint required by Step 5B.
#
# IMPORTANT CHECKPOINT NOTE:
# - resume-safe worker re-finalization can legitimately rewrite only volatile worker metadata
#   (CompletedAtUTC / invocation runtime), which changes a worker-checkpoint SHA while leaving all
#   frozen scientific raw outputs unchanged;
# - therefore this master records the ACTUAL current SHA of every worker checkpoint, but gates the
#   scientific freeze on exact pre-recorded worker raw-root SHA-256, raw bytes, seed shard, counts,
#   worker report linkage, worker output manifests, and an independent master revalidation of every
#   condition directory;
# - the official Step 5A checkpoint is thus cryptographically bound to the exact six checkpoint files
#   that exist at master-finalization time, while remaining robust to the accidental resume-safe
#   seeds-1–5 rerun.
#
# SAFETY:
# - ZERO model fitting;
# - ZERO noise injection / REC reconstruction / condition execution;
# - no completion-registry write;
# - no access to prior-project condition outputs;
# - raw condition directories are READ ONLY;
# - official master outputs are written only after every validation gate passes.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gzip
import hashlib
import json
import os
import shutil
import tarfile
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive


print("=" * 140)
print("=== PROJECT 24 CELL 9 / STEP 5A: PARALLEL MASTER FINALIZATION — ZERO FIT ===")
print("=" * 140)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 23
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

STEP5A_MASTER_CODE_REVISION = (
    "PROJECT_24_STEP5A_MASTER_V2_PROJECTNUMBER_ONLY_REGISTRY_GATE"
)

EXPECTED_SELECTION_STATUS = "PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN"
EXPECTED_STEP2B_STATUS = "PASS_PROJECT_24_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
EXPECTED_STEP3A_STATUS = "PASS_PROJECT_24_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
EXPECTED_STEP4A_STATUS = "PASS_PROJECT_24_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
EXPECTED_STEP4B_STATUS = "PASS_PROJECT_24_TWO_CONDITION_END_TO_END_SMOKE_TEST"

STEP5A_STATUS = "PASS_PROJECT_24_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
CONDITION_STATUS = "PASS_FULL_CONDITION"

MASTER_FINALIZER_IMPLEMENTATION = (
    "PROJECT_24_PARALLEL_STEP5A_MASTER_FINALIZER_V1_ZERO_FIT_"
    "WORKER_RAW_ROOT_BOUND_RESUME_METADATA_TOLERANT"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
)
EXPECTED_REC_CHECKPOINT_SHA256 = (
    "80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803"
)
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "9dba84682eabdc1393219671e77aabaf1f2d4385f29239236e50615bf0180515"
)
EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "ff9ed031d21d67bf218d5eab70bd8502ca39953adedbe840348f382d5599314e"
)
EXPECTED_SMOKE_CHECKPOINT_SHA256 = (
    "3f5a0cc24260b251fd8cd43f7391ff7d3cc3d517e8f79a8ad443f4002cdc9ff2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)
EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 23
EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 374_503_252

EXPECTED_RAW_TRAIN_ROWS = 4_040_801
EXPECTED_RAW_EVAL_ROWS = 1_594_226
EXPECTED_MODEL_TRAIN_ROWS = 205_696
EXPECTED_MODEL_EVAL_ROWS = 18_854
EXPECTED_MODEL_TRAIN_FAILURES = 1_777
EXPECTED_MODEL_EVAL_FAILURES = 20
EXPECTED_FAILING_EVAL_BUILDS = 17
EXPECTED_EVALUATION_BUILDS = 1_072
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_RNG_ROWS = 121_224_030

EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 523_093_565

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES
INVARIANT_BASELINES = ["Random", "QTF-Avg"]

ACCELERATED_ENGINE_VERSION = (
    "PROJECT_24_FAST_DEPENDENT_REC_V1_"
    "SMOKE_EQUIVALENT_FROZEN_INFERRED_TEST_ORDER_ANCHOR_AWARE"
)

SMOKE_EQUIVALENCE_KEYS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_MODEL_FITS = (
    EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = (
    EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_CONDITION_AUDIT_ROWS = EXPECTED_CONDITIONS

EXPECTED_WORKER_CONDITIONS = 45
EXPECTED_WORKER_MODEL_FITS = 180
EXPECTED_WORKER_RAW_FILES = 360
EXPECTED_WORKER_RANKING_ROWS = 5_939_010
EXPECTED_WORKER_BUILD_METRIC_ROWS = 5_355
EXPECTED_WORKER_PROJECT_RUN_ROWS = 315
EXPECTED_WORKER_CONDITION_AUDIT_ROWS = 45
EXPECTED_WORKER_TRAINING_MEDIAN_ROWS = 6_795

# Scientific worker freezes reported by the successful Project 24 worker runs.
# ReportedCheckpointSHA256 is retained for provenance. It is NOT the scientific gate because a
# resume-safe rerun can refresh volatile checkpoint metadata. Actual checkpoint SHA-256 values are
# captured and frozen by this master after all scientific validations pass.
EXPECTED_WORKERS = {
    "seed_01_05": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_01_05_COMPLETE",
        "Seeds": list(range(1, 6)),
        "ReportedCheckpointSHA256":
            "c6e8d76fe71c6aedfba9c3a0632c5daa5e2a8ed7cd3598dc56330d68e19ac81b",
        "RawRootSHA256":
            "53c1767a911a5d52561e2e12777f18d74ef8386c151a60fdf75e1488c2ab5705",
        "RawBytes": 86_952_171,
    },
    "seed_06_10": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_06_10_COMPLETE",
        "Seeds": list(range(6, 11)),
        "ReportedCheckpointSHA256":
            "ce2a285795c0a2b7aa16a51026686646328e817c778e49bfd14992137449a43a",
        "RawRootSHA256":
            "1703eef52c16b247a43bd95b378b6e1feb3bc8f8da6fc8a5b8c395b4437572a7",
        "RawBytes": 87_379_614,
    },
    "seed_11_15": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_11_15_COMPLETE",
        "Seeds": list(range(11, 16)),
        "ReportedCheckpointSHA256":
            "a8af44492f79733b78955a4f26213f5abd56332874cecb7c32b2a15f21392270",
        "RawRootSHA256":
            "01fef4c6d37ac6ee491a415aee4a3eabf05551be372c09fdc1b997fb52f7b93d",
        "RawBytes": 87_582_299,
    },
    "seed_16_20": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_16_20_COMPLETE",
        "Seeds": list(range(16, 21)),
        "ReportedCheckpointSHA256":
            "3177e9c8d783df28d2f28f60c75eb69d239c885c4273c55f0ed702ca1d63fd9d",
        "RawRootSHA256":
            "910ef203274d1ea744ad92ef1ec7802777d38be713333b21f5191fa69d4c364a",
        "RawBytes": 87_224_279,
    },
    "seed_21_25": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_21_25_COMPLETE",
        "Seeds": list(range(21, 26)),
        "ReportedCheckpointSHA256":
            "7cc8475c5c42fbfd86ab92e4024c7e7d486d03d8d7f053955bc86bb0fdb0c3d1",
        "RawRootSHA256":
            "a311f4370ebe599c4d7cee8a6d865ed7857c1b8d2641998fbabac352e30d7e1e",
        "RawBytes": 87_220_469,
    },
    "seed_26_30": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_26_30_COMPLETE",
        "Seeds": list(range(26, 31)),
        "ReportedCheckpointSHA256":
            "d9cf039b99bdee7b502adf78497c028c8d8b918556d68b6cf0eca975c56661b1",
        "RawRootSHA256":
            "b09e26b9bc4087379eac076931f70cc141083f9e6d15584c40a433f31f071aac",
        "RawBytes": 86_734_733,
    },
}

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"

REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
ARCHIVE_PATH = THESIS_ROOT / "Data" / "Raw" / "TCP-CI-main-dataset.tar.gz"
SOURCE_DIR = Path("/content/datasets/datasets/SonarSource@sonarqube")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_24_selection"
FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT / "project_24_frozen_source_manifest.csv"
)
SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_selection_checkpoint.json"
)

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
REC_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)
RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)
MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)
MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)
RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
)
CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
)
NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_noise_plan_checkpoint.json"
)

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_runtime_contract_checkpoint.json"
)

SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"
SMOKE_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_smoke_test_checkpoint.json"
)

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"

ACCELERATED_EQUIVALENCE_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"
)

PARALLEL_WORKERS_ROOT = FULL_EXPERIMENT_ROOT / "parallel_workers"

WORKER_CHECKPOINT_PATHS = {
    tag: NOTES_ROOT / f"project_24_step5a_worker_{tag}_checkpoint.json"
    for tag in EXPECTED_WORKERS
}

CONDITION_INVENTORY_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
)
RAW_MANIFEST_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
)
BASELINE_INVARIANCE_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
)
COMBINED_CONDITION_AUDIT_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
)
COMBINED_PROJECT_RUNS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
)
COMBINED_BUILD_METRICS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
)
COMBINED_MODEL_FITS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
)
WORKER_CHECKPOINT_AUDIT_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_worker_checkpoint_audit.csv"
)
STEP5A_VALIDATION_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
)
STEP5A_REPORT_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
)
RUN_PROGRESS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
)

STEP5A_STATUS_PATH = (
    PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
)
STEP5A_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_step5a_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary, path)


def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
    )

    os.replace(temporary, path)


def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        size = getattr(
            row,
            "SizeBytes",
            getattr(row, "Bytes", None),
        )

        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(size)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


DIRECTORY_MANIFEST_COLUMNS = [
    "RelativePath",
    "Bytes",
    "SHA256",
]


def directory_manifest(root):
    root = Path(root)
    records = []

    if root.exists():
        for path in sorted(
            (
                candidate
                for candidate in root.rglob("*")
                if candidate.is_file()
            ),
            key=lambda candidate:
                candidate.relative_to(root).as_posix(),
        ):
            records.append({
                "RelativePath":
                    path.relative_to(root).as_posix(),

                "Bytes":
                    int(path.stat().st_size),

                "SHA256":
                    sha256_file(path),
            })

    return pd.DataFrame(
        records,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )


def directory_root_hash(manifest):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
        "y",
    }


def verify_manifest_entries(entries, label):
    failures = []

    if not isinstance(entries, list) or not entries:
        return [{
            "Label": label,
            "Path": "<manifest>",
            "Reason": "missing/empty manifest",
        }]

    for item in entries:
        path = Path(str(item.get("Path", "")))

        if not path.is_file():
            failures.append({
                "Label": label,
                "Path": str(path),
                "Reason": "missing",
            })
            continue

        expected_bytes = int(item.get("Bytes", -1))
        expected_sha = str(item.get("SHA256", "")).lower()
        actual_bytes = int(path.stat().st_size)
        actual_sha = sha256_file(path)

        if (
            actual_bytes != expected_bytes
            or actual_sha != expected_sha
        ):
            failures.append({
                "Label": label,
                "Path": str(path),
                "Reason": "size/hash mismatch",
            })

    return failures


def resolve_registry_columns(registry):
    lookup = {
        str(column).strip().lower(): column
        for column in registry.columns
    }

    def pick(names, label):
        for name in names:
            if name in lookup:
                return lookup[name]

        raise RuntimeError(
            f"Could not resolve registry {label}; "
            f"columns={list(registry.columns)}"
        )

    return (
        pick(
            [
                "projectnumber",
                "project_number",
                "project no",
                "projectno",
            ],
            "ProjectNumber",
        ),
        pick(
            [
                "project",
                "projectname",
                "project_name",
            ],
            "Project",
        ),
        pick(
            [
                "status",
                "projectstatus",
                "project_status",
            ],
            "Status",
        ),
    )


def count_csv_data_rows(path):
    """
    Count CSV data rows without materialising the full frame.
    rankings.csv.gz is the only large file; all generated ranking rows contain no embedded newlines.
    """
    path = Path(path)

    opener = gzip.open if path.suffix == ".gz" else open

    with opener(
        path,
        "rt",
        encoding="utf-8",
        newline="",
    ) as handle:
        count = -1  # remove header
        for _ in handle:
            count += 1

    return max(count, 0)


def restore_source_if_needed():
    required_files = {
        "builds.csv",
        "contributors.csv",
        "dataset.csv",
        "entity_change_history.csv",
        "exe.csv",
        "id_map.csv",
    }

    source_ready = bool(
        SOURCE_DIR.is_dir()
        and required_files.issubset({
            path.name
            for path in SOURCE_DIR.iterdir()
            if path.is_file()
        })
    )

    if source_ready:
        return False

    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(
            "Project 24 source is absent from this reconnected runtime "
            "and the frozen thesis archive is missing:\n"
            f"{ARCHIVE_PATH}"
        )

    print(
        "\nRestoring only SonarSource@sonarqube from the frozen TCP-CI archive "
        "into this master runtime."
    )

    local_dataset_root = Path("/content/datasets")
    local_dataset_root.mkdir(parents=True, exist_ok=True)

    member_prefix = "datasets/SonarSource@sonarqube/"
    resolved_root = local_dataset_root.resolve()
    extracted_files = 0

    with tarfile.open(
        ARCHIVE_PATH,
        mode="r:gz",
    ) as archive:
        for member in archive:
            member_name = (
                member.name
                .replace("\\", "/")
                .lstrip("/")
            )

            if not (
                member_name == "datasets/SonarSource@sonarqube"
                or member_name.startswith(member_prefix)
            ):
                continue

            target_path = (
                local_dataset_root
                / member_name
            )

            resolved_target = target_path.resolve()

            if (
                resolved_target != resolved_root
                and resolved_root not in resolved_target.parents
            ):
                raise RuntimeError(
                    "Unsafe archive member encountered:\n"
                    f"{member.name}"
                )

            if member.isdir():
                target_path.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            elif member.isfile():
                target_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                source_handle = archive.extractfile(
                    member
                )

                if source_handle is None:
                    raise RuntimeError(
                        "Could not read archive member:\n"
                        f"{member.name}"
                    )

                with (
                    source_handle,
                    target_path.open("wb") as output_handle,
                ):
                    shutil.copyfileobj(
                        source_handle,
                        output_handle,
                        length=8 * 1024 * 1024,
                    )

                extracted_files += 1

    source_names = (
        {
            path.name
            for path in SOURCE_DIR.iterdir()
            if path.is_file()
        }
        if SOURCE_DIR.is_dir()
        else set()
    )

    missing = sorted(
        required_files
        - source_names
    )

    if missing:
        raise FileNotFoundError(
            "Selective Project 24 source restoration is incomplete:\n"
            + "\n".join(missing)
        )

    print(
        "Project source files restored:",
        extracted_files,
    )

    return True


EXPECTED_CONDITION_FILES = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}


def validate_condition_read_only(condition_dir, plan_row):
    """
    Independent master-side revalidation.
    No condition output is modified.
    """
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)

    if not condition_dir.is_dir():
        return None, f"{condition_key}: condition directory missing"

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    if actual_files != EXPECTED_CONDITION_FILES:
        return (
            None,
            f"{condition_key}: file set differs: "
            f"{sorted(actual_files)}",
        )

    completion_path = (
        condition_dir / "COMPLETE.json"
    )
    summary_path = (
        condition_dir / "condition_summary.json"
    )

    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception as error:
        return (
            None,
            f"{condition_key}: JSON read failure: {error!r}",
        )

    identity_checks = [
        (
            completion.get("Status"),
            CONDITION_STATUS,
            "completion status",
        ),
        (
            summary.get("Status"),
            CONDITION_STATUS,
            "summary status",
        ),
        (
            completion.get("ConditionKey"),
            condition_key,
            "completion key",
        ),
        (
            summary.get("ConditionKey"),
            condition_key,
            "summary key",
        ),
        (
            summary.get("Project"),
            PROJECT_NAME,
            "project",
        ),
        (
            summary.get("ProjectSlug"),
            PROJECT_SLUG,
            "project slug",
        ),
        (
            summary.get("AcceleratedEngineVersion"),
            ACCELERATED_ENGINE_VERSION,
            "accelerated engine",
        ),
    ]

    for actual, expected, label in identity_checks:
        if actual != expected:
            return (
                None,
                f"{condition_key}: {label} differs; "
                f"expected={expected!r}, actual={actual!r}",
            )

    if int(
        summary.get("ConditionOrder", -1)
    ) != int(plan_row.ConditionOrder):
        return None, f"{condition_key}: ConditionOrder differs"

    if int(
        summary.get("NoisePercent", -1)
    ) != int(plan_row.NoisePercent):
        return None, f"{condition_key}: NoisePercent differs"

    if int(
        summary.get("RepetitionSeed", -1)
    ) != int(plan_row.RepetitionSeed):
        return None, f"{condition_key}: RepetitionSeed differs"

    if str(
        completion.get("ConditionSummaryPath")
    ) != str(summary_path):
        return None, f"{condition_key}: summary path linkage differs"

    if str(
        completion.get("ConditionSummarySHA256")
    ) != sha256_file(summary_path):
        return None, f"{condition_key}: summary SHA linkage differs"

    output_manifest = summary.get(
        "OutputManifest",
        [],
    )

    if (
        not isinstance(output_manifest, list)
        or len(output_manifest) != 6
    ):
        return None, f"{condition_key}: output manifest differs"

    output_manifest_failures = verify_manifest_entries(
        output_manifest,
        condition_key,
    )

    if output_manifest_failures:
        return (
            None,
            f"{condition_key}: output manifest validation failed",
        )

    for item in output_manifest:
        if Path(
            str(item.get("Path", ""))
        ).parent != condition_dir:
            return (
                None,
                f"{condition_key}: output-manifest path escaped condition directory",
            )

    expected_counts = {
        "MLFits":
            EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows":
            EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows":
            EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }

    for key, expected in expected_counts.items():
        if int(
            summary.get(key, -1)
        ) != expected:
            return (
                None,
                f"{condition_key}: summary {key} differs",
            )

    if int(
        summary.get("IndependentRECChanges", -1)
    ) != 0:
        return (
            None,
            f"{condition_key}: independent REC changes are nonzero",
        )

    fingerprints = summary.get(
        "BaselineFingerprints",
        {},
    )

    if sorted(
        fingerprints.keys()
    ) != [
        "QTF-Avg",
        "Random",
    ]:
        return (
            None,
            f"{condition_key}: baseline fingerprints differ",
        )

    # Independent content-level readback of all compact raw outputs.
    condition_audit = pd.read_csv(
        condition_dir / "condition_audit.csv",
        low_memory=False,
    )

    if len(condition_audit) != 1:
        return (
            None,
            f"{condition_key}: condition_audit row count differs",
        )

    audit = condition_audit.iloc[0]

    for column, expected in [
        ("ConditionKey", condition_key),
        ("NoisePercent", int(plan_row.NoisePercent)),
        ("RepetitionSeed", int(plan_row.RepetitionSeed)),
        ("Status", CONDITION_STATUS),
    ]:
        if column not in condition_audit.columns:
            return (
                None,
                f"{condition_key}: condition_audit missing {column}",
            )

        actual = audit[column]

        if column in {
            "NoisePercent",
            "RepetitionSeed",
        }:
            if int(actual) != int(expected):
                return (
                    None,
                    f"{condition_key}: audit {column} differs",
                )
        elif str(actual) != str(expected):
            return (
                None,
                f"{condition_key}: audit {column} differs",
            )

    for column in [
        "IndependentRECChanges",
        "IndependentReconstructionMismatches",
    ]:
        if (
            column not in condition_audit.columns
            or int(
                pd.to_numeric(
                    pd.Series([audit[column]]),
                    errors="raise",
                ).iloc[0]
            ) != 0
        ):
            return (
                None,
                f"{condition_key}: audit {column} differs",
            )

    for expected_column, actual_column in [
        (
            "ExpectedFlipMaskSHA256",
            "ActualFlipMaskSHA256",
        ),
        (
            "ExpectedNoisyRawVerdictSHA256",
            "ActualNoisyRawVerdictSHA256",
        ),
        (
            "ExpectedNoisyModelVerdictSHA256",
            "ActualNoisyModelVerdictSHA256",
        ),
    ]:
        if (
            expected_column not in condition_audit.columns
            or actual_column not in condition_audit.columns
            or str(audit[expected_column])
            != str(audit[actual_column])
        ):
            return (
                None,
                f"{condition_key}: noise-plan hash audit differs",
            )

    number_flipped = int(
        pd.to_numeric(
            pd.Series([
                audit["NumberFlipped"]
            ]),
            errors="raise",
        ).iloc[0]
    )

    model_label_changes = int(
        pd.to_numeric(
            pd.Series([
                audit["ModelLabelChanges"]
            ]),
            errors="raise",
        ).iloc[0]
    )

    dependent_rec_changes = int(
        pd.to_numeric(
            pd.Series([
                audit["DependentRECChanges"]
            ]),
            errors="raise",
        ).iloc[0]
    )

    if int(plan_row.NoisePercent) == 0:
        if (
            number_flipped != 0
            or model_label_changes != 0
            or dependent_rec_changes != 0
        ):
            return (
                None,
                f"{condition_key}: zero-noise scientific audit differs",
            )
    else:
        if (
            number_flipped <= 0
            or model_label_changes <= 0
            or dependent_rec_changes <= 0
        ):
            return (
                None,
                f"{condition_key}: positive-noise scientific audit differs",
            )

    model_fits = pd.read_csv(
        condition_dir / "model_fits.csv",
        low_memory=False,
    )

    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: model-fit row count differs",
        )

    if (
        "Technique" not in model_fits.columns
        or "Status" not in model_fits.columns
        or sorted(
            model_fits["Technique"]
            .astype(str)
            .tolist()
        ) != sorted(ML_TECHNIQUES)
        or not model_fits["Status"]
        .astype(str)
        .eq("PASS_MODEL_FIT")
        .all()
    ):
        return (
            None,
            f"{condition_key}: model-fit contract differs",
        )

    build_metrics = pd.read_csv(
        condition_dir / "build_metrics.csv",
        low_memory=False,
    )

    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: build-metric row count differs",
        )

    required_build_metric_columns = {
        "Technique",
        "Build",
        "APFDc",
        "APFD",
    }

    if not required_build_metric_columns.issubset(
        build_metrics.columns
    ):
        return (
            None,
            f"{condition_key}: build-metric columns differ",
        )

    if sorted(
        build_metrics["Technique"]
        .astype(str)
        .unique()
        .tolist()
    ) != sorted(ALL_TECHNIQUES):
        return (
            None,
            f"{condition_key}: build-metric technique set differs",
        )

    if not build_metrics.groupby(
        "Technique"
    ).size().eq(
        EXPECTED_FAILING_EVAL_BUILDS
    ).all():
        return (
            None,
            f"{condition_key}: build-metric rows per technique differ",
        )

    build_metric_values = build_metrics[
        ["APFDc", "APFD"]
    ].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(float)

    if (
        not np.isfinite(
            build_metric_values
        ).all()
        or (
            (
                build_metric_values < 0
            )
            | (
                build_metric_values > 1
            )
        ).any()
    ):
        return (
            None,
            f"{condition_key}: build metrics invalid",
        )

    project_runs = pd.read_csv(
        condition_dir / "project_runs.csv",
        low_memory=False,
    )

    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: project-run row count differs",
        )

    if (
        "Technique" not in project_runs.columns
        or sorted(
            project_runs["Technique"]
            .astype(str)
            .tolist()
        ) != sorted(ALL_TECHNIQUES)
    ):
        return (
            None,
            f"{condition_key}: project-run technique set differs",
        )

    project_metric_columns = [
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]

    if not set(
        project_metric_columns
    ).issubset(
        project_runs.columns
    ):
        return (
            None,
            f"{condition_key}: project metric columns differ",
        )

    project_metric_values = project_runs[
        project_metric_columns
    ].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(float)

    if (
        not np.isfinite(
            project_metric_values
        ).all()
        or (
            (
                project_metric_values < 0
            )
            | (
                project_metric_values > 1
            )
        ).any()
    ):
        return (
            None,
            f"{condition_key}: project metrics invalid",
        )

    training_medians = pd.read_csv(
        condition_dir / "training_medians.csv",
        low_memory=False,
    )

    if len(
        training_medians
    ) != EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: training-median row count differs",
        )

    if (
        "Predictor" not in training_medians.columns
        or training_medians[
            "Predictor"
        ].astype(str).nunique()
        != EXPECTED_PREDICTORS
    ):
        return (
            None,
            f"{condition_key}: training-median predictor contract differs",
        )

    ranking_path = (
        condition_dir / "rankings.csv.gz"
    )

    ranking_rows = count_csv_data_rows(
        ranking_path
    )

    if ranking_rows != EXPECTED_RANKING_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: ranking row count differs; "
            f"expected={EXPECTED_RANKING_ROWS_PER_CONDITION}, "
            f"actual={ranking_rows}",
        )

    condition_manifest = directory_manifest(
        condition_dir
    )

    if len(
        condition_manifest
    ) != EXPECTED_FILES_PER_CONDITION:
        return (
            None,
            f"{condition_key}: condition manifest file count differs",
        )

    inventory = {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionOrder":
            int(plan_row.ConditionOrder),

        "ConditionKey":
            condition_key,

        "NoisePercent":
            int(plan_row.NoisePercent),

        "RepetitionSeed":
            int(plan_row.RepetitionSeed),

        "ConditionDirectory":
            str(condition_dir),

        "Status":
            CONDITION_STATUS,

        "Files":
            int(len(condition_manifest)),

        "ConditionBytes":
            int(
                condition_manifest[
                    "Bytes"
                ].sum()
            ),

        "ConditionRootSHA256":
            directory_root_hash(
                condition_manifest
            ),

        "RankingRows":
            int(summary["RankingRows"]),

        "BuildMetricRows":
            int(summary["BuildMetricRows"]),

        "ProjectRunRows":
            int(summary["ProjectRunRows"]),

        "ModelFitRows":
            int(summary["MLFits"]),

        "TrainingMedianRows":
            int(summary["TrainingMedianRows"]),

        "NumberFlipped":
            int(summary.get(
                "NumberFlipped",
                -1,
            )),

        "ModelLabelChanges":
            int(summary.get(
                "ModelLabelChanges",
                -1,
            )),

        "DependentRECChanges":
            int(summary.get(
                "DependentRECChanges",
                -1,
            )),

        "IndependentRECChanges":
            int(summary.get(
                "IndependentRECChanges",
                -1,
            )),

        "TrainingFailures":
            int(summary.get(
                "TrainingFailures",
                -1,
            )),

        "ConditionSeconds":
            float(summary.get(
                "ConditionSeconds",
                np.nan,
            )),

        "RandomKeySHA256":
            str(
                fingerprints[
                    "Random"
                ][
                    "KeySHA256"
                ]
            ),

        "RandomScoreSHA256":
            str(
                fingerprints[
                    "Random"
                ][
                    "ScoreSHA256"
                ]
            ),

        "RandomRankSHA256":
            str(
                fingerprints[
                    "Random"
                ][
                    "RankSHA256"
                ]
            ),

        "QTFAvgKeySHA256":
            str(
                fingerprints[
                    "QTF-Avg"
                ][
                    "KeySHA256"
                ]
            ),

        "QTFAvgScoreSHA256":
            str(
                fingerprints[
                    "QTF-Avg"
                ][
                    "ScoreSHA256"
                ]
            ),

        "QTFAvgRankSHA256":
            str(
                fingerprints[
                    "QTF-Avg"
                ][
                    "RankSHA256"
                ]
            ),
    }

    return (
        {
            "Inventory":
                inventory,

            "ConditionAudit":
                condition_audit,

            "ProjectRuns":
                project_runs,

            "BuildMetrics":
                build_metrics,

            "ModelFits":
                model_fits,

            "Manifest":
                condition_manifest,

            "Fingerprints":
                fingerprints,
        },
        "",
    )


# --------------------------------------------------------------------------------------------------
# 4. MOUNT, RESTORE SOURCE IF NEEDED, VERIFY UPSTREAM FREEZES
# --------------------------------------------------------------------------------------------------

master_started = time.perf_counter()

drive.mount(
    "/content/drive",
    force_remount=False,
)

source_restored = restore_source_if_needed()

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RUNTIME_CHECKPOINT_PATH,
    SMOKE_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
    FULL_RAW_RESULT_ROOT,
] + list(
    WORKER_CHECKPOINT_PATHS.values()
)

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing Project 24 master-finalization inputs:\n"
        + "\n".join(missing_paths)
    )

# Protect against accidental official Step 5A re-finalization.
if STEP5A_CHECKPOINT_PATH.is_file():
    existing = load_json(
        STEP5A_CHECKPOINT_PATH
    )

    if (
        existing.get("Status")
        == STEP5A_STATUS
        and bool(
            existing.get(
                "Full270ConditionExperimentComplete",
                False,
            )
        )
    ):
        raise RuntimeError(
            "Official Project 24 Step 5A is already frozen successfully. "
            "Do not rerun this master finalizer."
        )

    raise RuntimeError(
        "An unexpected pre-existing Project 24 Step 5A checkpoint exists. "
        "Stop and inspect it before finalization."
    )

checkpoint_hash_expectations = {
    "selection": (
        SELECTION_CHECKPOINT_PATH,
        EXPECTED_SELECTION_CHECKPOINT_SHA256,
    ),
    "REC": (
        REC_CHECKPOINT_PATH,
        EXPECTED_REC_CHECKPOINT_SHA256,
    ),
    "noise plan": (
        NOISE_PLAN_CHECKPOINT_PATH,
        EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    ),
    "runtime": (
        RUNTIME_CHECKPOINT_PATH,
        EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    ),
    "smoke": (
        SMOKE_CHECKPOINT_PATH,
        EXPECTED_SMOKE_CHECKPOINT_SHA256,
    ),
}

for label, (
    path,
    expected_sha,
) in checkpoint_hash_expectations.items():
    actual_sha = sha256_file(path)

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Frozen Project 24 {label} checkpoint SHA-256 differs.\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}"
        )

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)
noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
smoke_checkpoint = load_json(
    SMOKE_CHECKPOINT_PATH
)

for label, payload, expected_status in [
    (
        "selection checkpoint",
        selection_checkpoint,
        EXPECTED_SELECTION_STATUS,
    ),
    (
        "REC checkpoint",
        rec_checkpoint,
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "noise-plan checkpoint",
        noise_checkpoint,
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "runtime checkpoint",
        runtime_checkpoint,
        EXPECTED_STEP4A_STATUS,
    ),
    (
        "smoke checkpoint",
        smoke_checkpoint,
        EXPECTED_STEP4B_STATUS,
    ),
]:
    if payload.get(
        "Status"
    ) != expected_status:
        raise RuntimeError(
            f"{label} does not contain the frozen PASS status."
        )

    if (
        payload.get("Project") != PROJECT_NAME
        or payload.get("ProjectSlug") != PROJECT_SLUG
    ):
        raise RuntimeError(
            f"{label} Project 24 identity differs."
        )

if runtime_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active reservations differ."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint priority rule differs."
    )

if smoke_checkpoint.get(
    "RuntimeCheckpointSHA256"
) != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Smoke checkpoint is not linked to the frozen runtime checkpoint."
    )

if not bool(
    smoke_checkpoint.get(
        "ReadyForFull270ConditionExperiment",
        False,
    )
):
    raise RuntimeError(
        "Smoke checkpoint does not authorise the full experiment."
    )

upstream_manifest_failures = []

# These checkpoint schemas contain frozen file manifests.
for label, payload, manifest_key in [
    (
        "noise-plan",
        noise_checkpoint,
        "OutputManifest",
    ),
    (
        "runtime",
        runtime_checkpoint,
        "RuntimeOutputManifest",
    ),
    (
        "smoke",
        smoke_checkpoint,
        "OutputManifest",
    ),
]:
    failures = verify_manifest_entries(
        payload.get(
            manifest_key,
            [],
        ),
        label,
    )

    upstream_manifest_failures.extend(
        failures
    )

if upstream_manifest_failures:
    print(
        "\nUpstream manifest failures:"
    )
    display(
        pd.DataFrame(
            upstream_manifest_failures
        )
    )
    raise RuntimeError(
        "One or more frozen Project 24 upstream outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 5. REGISTRY, SOURCE, COHORT, AND CONDITION-PLAN IMMUTABILITY
# --------------------------------------------------------------------------------------------------

registry_sha_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 5A master finalization."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

(
    registry_project_number_column,
    registry_project_column,
    registry_status_column,
) = resolve_registry_columns(
    registry
)

registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)

if (
    len(registry)
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–23."
    )

if not registry[
    registry_status_column
].astype(str).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )

for number, identity in REQUIRED_REGISTERED_IDENTITIES.items():
    matching = registry.loc[
        registry_project_numbers.eq(
            number
        )
    ]

    if (
        len(matching) != 1
        or str(
            matching.iloc[0][
                registry_project_column
            ]
        ) != identity
    ):
        raise RuntimeError(
            "A frozen predecessor identity differs.\n"
            f"Project {number}; expected={identity}"
        )

# Registration is keyed by ProjectNumber in the frozen completion registry.
#
# IMPORTANT:
# The registry itself has already been byte-for-byte authenticated above by
# EXPECTED_REGISTRY_SHA256 and independently constrained to exactly rows 1..23.
# Therefore the scientifically relevant pre-Step-5C condition is simply that
# ProjectNumber 24 does not exist.  Do NOT reject on a Project-name string match:
# that extra name-based predicate was not part of the authoritative Project-24
# Step-1A registration check and caused a false-positive in the master finalizer.
project_24_registry_number_rows = int(
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
)

project_name_match_rows_diagnostic = int(
    registry[
        registry_project_column
    ].astype(str).eq(
        PROJECT_NAME
    ).sum()
)

if project_24_registry_number_rows != 0:
    raise RuntimeError(
        "Project 24 is unexpectedly already registered by ProjectNumber."
    )

print(
    "Completion-registry ProjectNumber-24 rows:",
    project_24_registry_number_rows,
)

print(
    "Completion-registry exact project-name matches (diagnostic only):",
    project_name_match_rows_diagnostic,
)

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 24 source file is missing: {path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(path),
    })

current_source_manifest = pd.DataFrame(
    current_source_records
)

source_root_before = source_root_hash(
    current_source_manifest
)

if source_root_before != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 24 source root differs."
    )

if (
    len(current_source_manifest)
    != EXPECTED_SOURCE_FILES
    or int(
        current_source_manifest[
            "SizeBytes"
        ].sum()
    ) != EXPECTED_SOURCE_BYTES
):
    raise RuntimeError(
        "Frozen Project 24 source file/byte counts differ."
    )

cohort_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
]

cohort_hashes_before = {
    str(path):
        sha256_file(path)
    for path in cohort_paths
}

if pq.ParquetFile(
    RAW_TRAINING_COHORT_PATH
).metadata.num_rows != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError(
        "Frozen raw training cohort row count differs."
    )

if pq.ParquetFile(
    RAW_EVALUATION_COHORT_PATH
).metadata.num_rows != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError(
        "Frozen raw evaluation cohort row count differs."
    )

if pq.ParquetFile(
    MODEL_TRAINING_COHORT_PATH
).metadata.num_rows != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError(
        "Frozen model training cohort row count differs."
    )

if pq.ParquetFile(
    MODEL_EVALUATION_COHORT_PATH
).metadata.num_rows != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError(
        "Frozen model evaluation cohort row count differs."
    )

if pq.ParquetFile(
    RNG_MANIFEST_PATH
).metadata.num_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG manifest row count differs."
    )

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)

required_plan_columns = {
    "ConditionOrder",
    "ConditionID",
    "NoisePercent",
    "RepetitionSeed",
}

if not required_plan_columns.issubset(
    condition_plan.columns
):
    raise RuntimeError(
        "Condition plan is missing required columns:\n"
        + "\n".join(
            sorted(
                required_plan_columns
                - set(condition_plan.columns)
            )
        )
    )

condition_plan = (
    condition_plan.sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

expected_grid = [
    (
        seed,
        noise,
        f"noise_{noise:02d}__seed_{seed:02d}",
    )
    for seed in REPETITION_SEEDS
    for noise in NOISE_LEVELS
]

actual_grid = [
    (
        int(row.RepetitionSeed),
        int(row.NoisePercent),
        str(row.ConditionID),
    )
    for row in condition_plan.itertuples(
        index=False
    )
]

if (
    len(condition_plan)
    != EXPECTED_CONDITIONS
    or actual_grid != expected_grid
):
    raise RuntimeError(
        "Frozen Project 24 condition grid/order differs."
    )

if (
    condition_plan[
        "ConditionID"
    ].duplicated().any()
    or condition_plan.duplicated(
        [
            "NoisePercent",
            "RepetitionSeed",
        ]
    ).any()
):
    raise RuntimeError(
        "Frozen Project 24 condition plan contains duplicate coordinates."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY ALL SIX WORKER CHECKPOINTS AND SCIENTIFIC SHARD FREEZES
# --------------------------------------------------------------------------------------------------

print(
    "\nVerifying six worker checkpoints and exact scientific worker raw roots."
)

worker_records = []
worker_seed_union = []
worker_checkpoint_hashes_actual = {}
worker_checkpoint_hash_refreshes = 0
shared_equivalence_hashes = set()

for tag, expected in EXPECTED_WORKERS.items():
    checkpoint_path = (
        WORKER_CHECKPOINT_PATHS[tag]
    )

    actual_checkpoint_sha = sha256_file(
        checkpoint_path
    )

    worker_checkpoint_hashes_actual[
        tag
    ] = actual_checkpoint_sha

    reported_sha_still_current = bool(
        actual_checkpoint_sha
        == expected[
            "ReportedCheckpointSHA256"
        ]
    )

    if not reported_sha_still_current:
        worker_checkpoint_hash_refreshes += 1

    checkpoint = load_json(
        checkpoint_path
    )

    if (
        checkpoint.get("Project")
        != PROJECT_NAME
        or checkpoint.get("ProjectSlug")
        != PROJECT_SLUG
        or int(
            checkpoint.get(
                "ProjectNumber",
                -1,
            )
        ) != PROJECT_NUMBER
    ):
        raise RuntimeError(
            f"Worker {tag} project identity differs."
        )

    if checkpoint.get(
        "Status"
    ) != expected[
        "Status"
    ]:
        raise RuntimeError(
            f"Worker {tag} PASS status differs."
        )

    assigned_seeds = [
        int(value)
        for value in checkpoint.get(
            "AssignedSeeds",
            [],
        )
    ]

    if assigned_seeds != expected[
        "Seeds"
    ]:
        raise RuntimeError(
            f"Worker {tag} assigned seeds differ: "
            f"{assigned_seeds}"
        )

    worker_seed_union.extend(
        assigned_seeds
    )

    scalar_expectations = {
        "AssignedConditions":
            EXPECTED_WORKER_CONDITIONS,

        "CompletedConditions":
            EXPECTED_WORKER_CONDITIONS,

        "MLFits":
            EXPECTED_WORKER_MODEL_FITS,

        "RankingRows":
            EXPECTED_WORKER_RANKING_ROWS,

        "BuildMetricRows":
            EXPECTED_WORKER_BUILD_METRIC_ROWS,

        "ProjectRunRows":
            EXPECTED_WORKER_PROJECT_RUN_ROWS,

        "ConditionAuditRows":
            EXPECTED_WORKER_CONDITION_AUDIT_ROWS,

        "TrainingMedianRows":
            EXPECTED_WORKER_TRAINING_MEDIAN_ROWS,

        "WorkerRawFiles":
            EXPECTED_WORKER_RAW_FILES,

        "WorkerRawBytes":
            expected["RawBytes"],

        "FailedValidationChecks":
            0,

        "BaselineInvarianceFailures":
            0,

        "MasterStep5AWriteGuardFailures":
            0,
    }

    for key, expected_value in scalar_expectations.items():
        actual_value = int(
            checkpoint.get(
                key,
                -1,
            )
        )

        if actual_value != int(
            expected_value
        ):
            raise RuntimeError(
                f"Worker {tag} {key} differs. "
                f"Expected={expected_value}; "
                f"actual={actual_value}"
            )

    if checkpoint.get(
        "WorkerRawRootSHA256"
    ) != expected[
        "RawRootSHA256"
    ]:
        raise RuntimeError(
            f"Worker {tag} frozen raw-root SHA-256 differs."
        )

    if checkpoint.get(
        "SourceRootSHA256"
    ) != EXPECTED_SOURCE_ROOT_SHA256:
        raise RuntimeError(
            f"Worker {tag} source root differs."
        )

    if checkpoint.get(
        "RegistrySHA256"
    ) != EXPECTED_REGISTRY_SHA256:
        raise RuntimeError(
            f"Worker {tag} registry SHA differs."
        )

    if not bool(
        checkpoint.get(
            "WorkerShardComplete",
            False,
        )
    ):
        raise RuntimeError(
            f"Worker {tag} is not marked complete."
        )

    if bool(
        checkpoint.get(
            "OfficialStep5AComplete",
            True,
        )
    ):
        raise RuntimeError(
            f"Worker {tag} incorrectly marks official Step 5A complete."
        )

    worker_report_path = Path(
        str(
            checkpoint.get(
                "WorkerReportPath",
                "",
            )
        )
    )

    if (
        not worker_report_path.is_file()
        or sha256_file(
            worker_report_path
        )
        != str(
            checkpoint.get(
                "WorkerReportSHA256",
                "",
            )
        )
    ):
        raise RuntimeError(
            f"Worker {tag} report linkage failed."
        )

    worker_report = load_json(
        worker_report_path
    )

    if (
        worker_report.get("Status")
        != expected["Status"]
        or worker_report.get(
            "WorkerRawRootSHA256"
        )
        != expected["RawRootSHA256"]
        or int(
            worker_report.get(
                "WorkerRawBytes",
                -1,
            )
        )
        != expected["RawBytes"]
    ):
        raise RuntimeError(
            f"Worker {tag} report scientific freeze differs."
        )

    worker_output_manifest_failures = verify_manifest_entries(
        checkpoint.get(
            "WorkerOutputManifest",
            [],
        ),
        f"worker {tag}",
    )

    if worker_output_manifest_failures:
        display(
            pd.DataFrame(
                worker_output_manifest_failures
            )
        )

        raise RuntimeError(
            f"Worker {tag} private output manifest failed."
        )

    shared_equivalence_path = Path(
        str(
            checkpoint.get(
                "SharedSmokeEquivalencePath",
                "",
            )
        )
    )

    if shared_equivalence_path != ACCELERATED_EQUIVALENCE_PATH:
        raise RuntimeError(
            f"Worker {tag} shared equivalence path differs."
        )

    shared_equivalence_sha = str(
        checkpoint.get(
            "SharedSmokeEquivalenceSHA256",
            "",
        )
    )

    if (
        not ACCELERATED_EQUIVALENCE_PATH.is_file()
        or sha256_file(
            ACCELERATED_EQUIVALENCE_PATH
        )
        != shared_equivalence_sha
    ):
        raise RuntimeError(
            f"Worker {tag} shared equivalence linkage differs."
        )

    shared_equivalence_hashes.add(
        shared_equivalence_sha
    )

    # Independently recompute this worker's exact 360-file scientific raw root from the global raw root.
    worker_condition_keys = set(
        condition_plan.loc[
            condition_plan[
                "RepetitionSeed"
            ].isin(
                assigned_seeds
            ),
            "ConditionID",
        ].astype(str)
    )

    worker_raw_records = []

    for condition_key in sorted(
        worker_condition_keys
    ):
        condition_dir = (
            FULL_RAW_RESULT_ROOT
            / condition_key
        )

        condition_manifest = directory_manifest(
            condition_dir
        )

        for item in condition_manifest.itertuples(
            index=False
        ):
            worker_raw_records.append({
                "RelativePath":
                    f"{condition_key}/{item.RelativePath}",

                "Bytes":
                    int(item.Bytes),

                "SHA256":
                    str(item.SHA256),
            })

    worker_raw_manifest = pd.DataFrame(
        worker_raw_records,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    ).sort_values(
        "RelativePath",
        kind="mergesort",
    ).reset_index(
        drop=True
    )

    worker_raw_files_recomputed = int(
        len(
            worker_raw_manifest
        )
    )

    worker_raw_bytes_recomputed = int(
        worker_raw_manifest[
            "Bytes"
        ].sum()
    )

    worker_raw_root_recomputed = directory_root_hash(
        worker_raw_manifest
    )

    if (
        worker_raw_files_recomputed
        != EXPECTED_WORKER_RAW_FILES
        or worker_raw_bytes_recomputed
        != expected["RawBytes"]
        or worker_raw_root_recomputed
        != expected["RawRootSHA256"]
    ):
        raise RuntimeError(
            f"Worker {tag} independent raw-root recomputation differs."
        )

    worker_records.append({
        "WorkerTag":
            tag,

        "AssignedSeedsJSON":
            json.dumps(
                assigned_seeds
            ),

        "Status":
            checkpoint.get(
                "Status"
            ),

        "ReportedCheckpointSHA256":
            expected[
                "ReportedCheckpointSHA256"
            ],

        "ActualCheckpointSHA256":
            actual_checkpoint_sha,

        "ReportedCheckpointSHAStillCurrent":
            reported_sha_still_current,

        "CheckpointMetadataRefreshed":
            not reported_sha_still_current,

        "WorkerRawFiles":
            worker_raw_files_recomputed,

        "WorkerRawBytes":
            worker_raw_bytes_recomputed,

        "ExpectedWorkerRawRootSHA256":
            expected[
                "RawRootSHA256"
            ],

        "ActualWorkerRawRootSHA256":
            worker_raw_root_recomputed,

        "WorkerReportPath":
            str(
                worker_report_path
            ),

        "WorkerReportSHA256":
            sha256_file(
                worker_report_path
            ),

        "SharedSmokeEquivalenceSHA256":
            shared_equivalence_sha,

        "Pass":
            True,
    })

worker_audit = pd.DataFrame(
    worker_records
)

if len(
    shared_equivalence_hashes
) != 1:
    raise RuntimeError(
        "The six workers do not reference one identical accelerated-equivalence freeze."
    )

if (
    sorted(
        worker_seed_union
    ) != REPETITION_SEEDS
    or len(
        worker_seed_union
    ) != len(
        set(
            worker_seed_union
        )
    )
):
    raise RuntimeError(
        "Worker seed shards are not disjoint exact coverage of seeds 1–30."
    )

print(
    "Workers verified:",
    len(worker_audit),
    "/ 6",
)

print(
    "Worker checkpoint metadata refreshes:",
    worker_checkpoint_hash_refreshes,
)

if worker_checkpoint_hash_refreshes:
    print(
        "NOTE: refreshed worker checkpoint SHA values are acceptable only because "
        "their exact scientific raw roots, output manifests, report linkage, seed shards, "
        "and all 270 raw conditions are independently revalidated below."
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY SHARED ACCELERATED SMOKE-EQUIVALENCE FREEZE
# --------------------------------------------------------------------------------------------------

equivalence = pd.read_csv(
    ACCELERATED_EQUIVALENCE_PATH,
    low_memory=False,
)

if (
    len(equivalence) != 2
    or "ConditionKey" not in equivalence.columns
    or "Pass" not in equivalence.columns
    or sorted(
        equivalence[
            "ConditionKey"
        ].astype(str).tolist()
    ) != sorted(
        SMOKE_EQUIVALENCE_KEYS
    )
    or not equivalence[
        "Pass"
    ].map(
        as_bool
    ).all()
):
    raise RuntimeError(
        "Frozen accelerated smoke-equivalence audit differs."
    )


# --------------------------------------------------------------------------------------------------
# 8. INDEPENDENTLY REVALIDATE ALL 270 RAW CONDITIONS AND REBUILD MASTER AGGREGATES
# --------------------------------------------------------------------------------------------------

print(
    "\nIndependently revalidating all 270 raw condition directories."
)

inventory_records = []
condition_audit_frames = []
project_run_frames = []
build_metric_frames = []
model_fit_frames = []
raw_manifest_records = []
baseline_fingerprint_records = []
condition_validation_errors = []

validation_started = time.perf_counter()

for index, plan_row in enumerate(
    condition_plan.itertuples(
        index=False
    ),
    start=1,
):
    condition_key = str(
        plan_row.ConditionID
    )

    condition_dir = (
        FULL_RAW_RESULT_ROOT
        / condition_key
    )

    validated, error = validate_condition_read_only(
        condition_dir,
        plan_row,
    )

    if validated is None:
        condition_validation_errors.append({
            "ConditionKey":
                condition_key,

            "Error":
                error,
        })
        continue

    inventory_records.append(
        validated[
            "Inventory"
        ]
    )

    condition_audit_frames.append(
        validated[
            "ConditionAudit"
        ]
    )

    project_run_frames.append(
        validated[
            "ProjectRuns"
        ]
    )

    build_metric_frames.append(
        validated[
            "BuildMetrics"
        ]
    )

    model_fit_frames.append(
        validated[
            "ModelFits"
        ]
    )

    fingerprints = validated[
        "Fingerprints"
    ]

    for technique in INVARIANT_BASELINES:
        fingerprint = fingerprints[
            technique
        ]

        baseline_fingerprint_records.append({
            "ConditionKey":
                condition_key,

            "NoisePercent":
                int(
                    plan_row.NoisePercent
                ),

            "RepetitionSeed":
                int(
                    plan_row.RepetitionSeed
                ),

            "Technique":
                technique,

            "Rows":
                int(
                    fingerprint[
                        "Rows"
                    ]
                ),

            "KeySHA256":
                str(
                    fingerprint[
                        "KeySHA256"
                    ]
                ),

            "ScoreSHA256":
                str(
                    fingerprint[
                        "ScoreSHA256"
                    ]
                ),

            "RankSHA256":
                str(
                    fingerprint[
                        "RankSHA256"
                    ]
                ),
        })

    condition_manifest = validated[
        "Manifest"
    ]

    for item in condition_manifest.itertuples(
        index=False
    ):
        raw_manifest_records.append({
            "RelativePath":
                f"{condition_key}/{item.RelativePath}",

            "Bytes":
                int(
                    item.Bytes
                ),

            "SHA256":
                str(
                    item.SHA256
                ),
        })

    if (
        index % 25 == 0
        or index == EXPECTED_CONDITIONS
    ):
        print(
            f"  Conditions revalidated: {index}/{EXPECTED_CONDITIONS} "
            f"| elapsed seconds: "
            f"{round(time.perf_counter() - validation_started, 2)}"
        )

if condition_validation_errors:
    print(
        "\nCondition validation failures:"
    )

    display(
        pd.DataFrame(
            condition_validation_errors
        )
    )

    raise RuntimeError(
        "One or more Project 24 raw condition directories failed independent master validation."
    )

condition_inventory = (
    pd.DataFrame(
        inventory_records
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

combined_condition_audit = pd.concat(
    condition_audit_frames,
    ignore_index=True,
)

combined_project_runs = pd.concat(
    project_run_frames,
    ignore_index=True,
)

combined_build_metrics = pd.concat(
    build_metric_frames,
    ignore_index=True,
)

combined_model_fits = pd.concat(
    model_fit_frames,
    ignore_index=True,
)

baseline_fingerprints = pd.DataFrame(
    baseline_fingerprint_records
)

raw_manifest = (
    pd.DataFrame(
        raw_manifest_records,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )
    .sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

raw_files = int(
    len(
        raw_manifest
    )
)

raw_bytes = int(
    raw_manifest[
        "Bytes"
    ].sum()
)

raw_root_sha256 = directory_root_hash(
    raw_manifest
)


# --------------------------------------------------------------------------------------------------
# 9. MASTER BASELINE INVARIANCE + GLOBAL RAW-ROOT CROSS-CHECK
# --------------------------------------------------------------------------------------------------

baseline_invariance_records = []

for repetition_seed in REPETITION_SEEDS:
    for technique in INVARIANT_BASELINES:
        rows = baseline_fingerprints.loc[
            baseline_fingerprints[
                "RepetitionSeed"
            ].eq(
                repetition_seed
            )
            & baseline_fingerprints[
                "Technique"
            ].eq(
                technique
            )
        ]

        key_variants = int(
            rows[
                "KeySHA256"
            ].nunique()
        )

        score_variants = int(
            rows[
                "ScoreSHA256"
            ].nunique()
        )

        rank_variants = int(
            rows[
                "RankSHA256"
            ].nunique()
        )

        passed = bool(
            len(rows)
            == len(NOISE_LEVELS)
            and key_variants == 1
            and score_variants == 1
            and rank_variants == 1
        )

        baseline_invariance_records.append({
            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "Conditions":
                int(
                    len(rows)
                ),

            "KeyVariantsAcrossNoise":
                key_variants,

            "ScoreVariantsAcrossNoise":
                score_variants,

            "RankVariantsAcrossNoise":
                rank_variants,

            "Pass":
                passed,
        })

baseline_invariance = pd.DataFrame(
    baseline_invariance_records
)

baseline_invariance_failures = int(
    (
        ~baseline_invariance[
            "Pass"
        ]
    ).sum()
)

qtf_global_key_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints[
            "Technique"
        ].eq(
            "QTF-Avg"
        ),
        "KeySHA256",
    ].nunique()
)

qtf_global_score_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints[
            "Technique"
        ].eq(
            "QTF-Avg"
        ),
        "ScoreSHA256",
    ].nunique()
)

qtf_global_rank_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints[
            "Technique"
        ].eq(
            "QTF-Avg"
        ),
        "RankSHA256",
    ].nunique()
)

# Completely independent directory walk/hash over the raw root.
raw_manifest_direct = directory_manifest(
    FULL_RAW_RESULT_ROOT
)

raw_root_sha256_direct = directory_root_hash(
    raw_manifest_direct
)

raw_manifest_keyed = raw_manifest.set_index(
    "RelativePath"
).sort_index()

raw_manifest_direct_keyed = raw_manifest_direct.set_index(
    "RelativePath"
).sort_index()

raw_manifest_exact_match = bool(
    raw_manifest_keyed.equals(
        raw_manifest_direct_keyed
    )
)


# --------------------------------------------------------------------------------------------------
# 10. MASTER VALIDATION GATE
# --------------------------------------------------------------------------------------------------

validation = []

add_check(
    validation,
    "Workers verified",
    6,
    len(worker_audit),
    len(worker_audit) == 6,
)

add_check(
    validation,
    "Worker seeds cover 1–30 exactly once",
    REPETITION_SEEDS,
    sorted(worker_seed_union),
    (
        sorted(worker_seed_union)
        == REPETITION_SEEDS
        and len(worker_seed_union)
        == len(set(worker_seed_union))
    ),
)

add_check(
    validation,
    "Worker scientific raw-root mismatches",
    0,
    int(
        (
            worker_audit[
                "ExpectedWorkerRawRootSHA256"
            ]
            != worker_audit[
                "ActualWorkerRawRootSHA256"
            ]
        ).sum()
    ),
    bool(
        (
            worker_audit[
                "ExpectedWorkerRawRootSHA256"
            ]
            == worker_audit[
                "ActualWorkerRawRootSHA256"
            ]
        ).all()
    ),
)

add_check(
    validation,
    "Conditions",
    EXPECTED_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory)
    == EXPECTED_CONDITIONS,
)

add_check(
    validation,
    "Duplicate condition keys",
    0,
    int(
        condition_inventory[
            "ConditionKey"
        ].duplicated(
            keep=False
        ).sum()
    ),
    not condition_inventory[
        "ConditionKey"
    ].duplicated(
        keep=False
    ).any(),
)

add_check(
    validation,
    "Files per condition",
    EXPECTED_FILES_PER_CONDITION,
    sorted(
        condition_inventory[
            "Files"
        ].unique().tolist()
    ),
    condition_inventory[
        "Files"
    ].eq(
        EXPECTED_FILES_PER_CONDITION
    ).all(),
)

add_check(
    validation,
    "Raw files",
    EXPECTED_RAW_FILES,
    raw_files,
    raw_files == EXPECTED_RAW_FILES,
)

add_check(
    validation,
    "Raw bytes",
    EXPECTED_RAW_BYTES,
    raw_bytes,
    raw_bytes == EXPECTED_RAW_BYTES,
)

add_check(
    validation,
    "Direct raw-root file count",
    EXPECTED_RAW_FILES,
    len(raw_manifest_direct),
    len(raw_manifest_direct)
    == EXPECTED_RAW_FILES,
)

add_check(
    validation,
    "Direct raw-root bytes",
    EXPECTED_RAW_BYTES,
    int(
        raw_manifest_direct[
            "Bytes"
        ].sum()
    ),
    int(
        raw_manifest_direct[
            "Bytes"
        ].sum()
    ) == EXPECTED_RAW_BYTES,
)

add_check(
    validation,
    "Collected/direct raw manifests identical",
    True,
    raw_manifest_exact_match,
    raw_manifest_exact_match,
)

add_check(
    validation,
    "Collected/direct raw-root SHA identical",
    raw_root_sha256,
    raw_root_sha256_direct,
    raw_root_sha256
    == raw_root_sha256_direct,
)

add_check(
    validation,
    "Ranking rows",
    EXPECTED_TOTAL_RANKING_ROWS,
    int(
        condition_inventory[
            "RankingRows"
        ].sum()
    ),
    int(
        condition_inventory[
            "RankingRows"
        ].sum()
    ) == EXPECTED_TOTAL_RANKING_ROWS,
)

add_check(
    validation,
    "Build-metric rows",
    EXPECTED_TOTAL_BUILD_METRIC_ROWS,
    len(combined_build_metrics),
    len(combined_build_metrics)
    == EXPECTED_TOTAL_BUILD_METRIC_ROWS,
)

add_check(
    validation,
    "Project-run rows",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
    len(combined_project_runs),
    len(combined_project_runs)
    == EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)

add_check(
    validation,
    "Model fits",
    EXPECTED_TOTAL_MODEL_FITS,
    len(combined_model_fits),
    len(combined_model_fits)
    == EXPECTED_TOTAL_MODEL_FITS,
)

add_check(
    validation,
    "Condition-audit rows",
    EXPECTED_TOTAL_CONDITION_AUDIT_ROWS,
    len(combined_condition_audit),
    len(combined_condition_audit)
    == EXPECTED_TOTAL_CONDITION_AUDIT_ROWS,
)

add_check(
    validation,
    "Training-median rows",
    EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS,
    int(
        condition_inventory[
            "TrainingMedianRows"
        ].sum()
    ),
    int(
        condition_inventory[
            "TrainingMedianRows"
        ].sum()
    ) == EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS,
)

add_check(
    validation,
    "Zero-noise conditions",
    30,
    int(
        condition_inventory[
            "NoisePercent"
        ].eq(0).sum()
    ),
    int(
        condition_inventory[
            "NoisePercent"
        ].eq(0).sum()
    ) == 30,
)

zero_audit = combined_condition_audit.loc[
    pd.to_numeric(
        combined_condition_audit[
            "NoisePercent"
        ],
        errors="raise",
    ).eq(0)
]

positive_audit = combined_condition_audit.loc[
    pd.to_numeric(
        combined_condition_audit[
            "NoisePercent"
        ],
        errors="raise",
    ).gt(0)
]

add_check(
    validation,
    "Zero-noise raw flips",
    0,
    int(
        pd.to_numeric(
            zero_audit[
                "NumberFlipped"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            zero_audit[
                "NumberFlipped"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Zero-noise model-label changes",
    0,
    int(
        pd.to_numeric(
            zero_audit[
                "ModelLabelChanges"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            zero_audit[
                "ModelLabelChanges"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Zero-noise dependent REC changes",
    0,
    int(
        pd.to_numeric(
            zero_audit[
                "DependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            zero_audit[
                "DependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Positive-noise raw-change violations",
    0,
    int(
        pd.to_numeric(
            positive_audit[
                "NumberFlipped"
            ],
            errors="raise",
        ).le(0).sum()
    ),
    not pd.to_numeric(
        positive_audit[
            "NumberFlipped"
        ],
        errors="raise",
    ).le(0).any(),
)

add_check(
    validation,
    "Positive-noise model-change violations",
    0,
    int(
        pd.to_numeric(
            positive_audit[
                "ModelLabelChanges"
            ],
            errors="raise",
        ).le(0).sum()
    ),
    not pd.to_numeric(
        positive_audit[
            "ModelLabelChanges"
        ],
        errors="raise",
    ).le(0).any(),
)

add_check(
    validation,
    "Positive-noise dependent-REC violations",
    0,
    int(
        pd.to_numeric(
            positive_audit[
                "DependentRECChanges"
            ],
            errors="raise",
        ).le(0).sum()
    ),
    not pd.to_numeric(
        positive_audit[
            "DependentRECChanges"
        ],
        errors="raise",
    ).le(0).any(),
)

add_check(
    validation,
    "Independent REC changes",
    0,
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Independent reconstruction mismatches",
    0,
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentReconstructionMismatches"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentReconstructionMismatches"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

noise_plan_hash_mismatches = 0

for expected_column, actual_column in [
    (
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
    ),
    (
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
    ),
    (
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
    ),
]:
    noise_plan_hash_mismatches += int(
        combined_condition_audit[
            expected_column
        ].astype(str).ne(
            combined_condition_audit[
                actual_column
            ].astype(str)
        ).sum()
    )

add_check(
    validation,
    "Noise-plan hash mismatches",
    0,
    noise_plan_hash_mismatches,
    noise_plan_hash_mismatches == 0,
)

add_check(
    validation,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)

add_check(
    validation,
    "QTF global key variants",
    1,
    qtf_global_key_variants,
    qtf_global_key_variants == 1,
)

add_check(
    validation,
    "QTF global score variants",
    1,
    qtf_global_score_variants,
    qtf_global_score_variants == 1,
)

add_check(
    validation,
    "QTF global rank variants",
    1,
    qtf_global_rank_variants,
    qtf_global_rank_variants == 1,
)

model_fit_status_failures = int(
    (
        ~combined_model_fits[
            "Status"
        ].astype(str).eq(
            "PASS_MODEL_FIT"
        )
    ).sum()
)

add_check(
    validation,
    "Model-fit status failures",
    0,
    model_fit_status_failures,
    model_fit_status_failures == 0,
)

for frame, columns, label in [
    (
        combined_project_runs,
        [
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ],
        "Project metrics",
    ),
    (
        combined_build_metrics,
        [
            "APFDc",
            "APFD",
        ],
        "Build metrics",
    ),
]:
    values = frame[
        columns
    ].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(float)

    bad = int(
        (
            ~np.isfinite(
                values
            )
        ).sum()
        + (
            (
                values < 0
            )
            | (
                values > 1
            )
        ).sum()
    )

    add_check(
        validation,
        f"{label} non-finite/out-of-range values",
        0,
        bad,
        bad == 0,
    )

add_check(
    validation,
    "Shared smoke-equivalence rows",
    2,
    len(equivalence),
    len(equivalence) == 2,
)

add_check(
    validation,
    "Shared smoke-equivalence failures",
    0,
    int(
        (
            ~equivalence[
                "Pass"
            ].map(
                as_bool
            )
        ).sum()
    ),
    equivalence[
        "Pass"
    ].map(
        as_bool
    ).all(),
)

add_check(
    validation,
    "Registry SHA unchanged",
    EXPECTED_REGISTRY_SHA256,
    sha256_file(
        REGISTRY_PATH
    ),
    sha256_file(
        REGISTRY_PATH
    ) == EXPECTED_REGISTRY_SHA256,
)

add_check(
    validation,
    "Source root unchanged",
    EXPECTED_SOURCE_ROOT_SHA256,
    source_root_before,
    source_root_before
    == EXPECTED_SOURCE_ROOT_SHA256,
)

for cohort_path, expected_sha in cohort_hashes_before.items():
    actual_sha = sha256_file(
        Path(
            cohort_path
        )
    )

    add_check(
        validation,
        f"Frozen cohort unchanged: {Path(cohort_path).name}",
        expected_sha,
        actual_sha,
        actual_sha == expected_sha,
    )

add_check(
    validation,
    "Registry Project 24 rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Master finalizer model fits",
    0,
    0,
    True,
)

add_check(
    validation,
    "Master finalizer condition executions",
    0,
    0,
    True,
)

step5a_validation = pd.DataFrame(
    validation
)

failed_validation = step5a_validation.loc[
    ~step5a_validation[
        "Pass"
    ]
]

print(
    "\nProject 24 master Step 5A validation:"
)

display(
    step5a_validation
)

if not failed_validation.empty:
    print(
        "\nFAILED MASTER CHECKS:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 24 STEP 5A MASTER FINALIZATION VALIDATION FAILED. "
        "No official Step 5A freeze was written."
    )


# --------------------------------------------------------------------------------------------------
# 11. WRITE OFFICIAL STEP 5A AGGREGATES ONLY AFTER ALL CHECKS PASS
# --------------------------------------------------------------------------------------------------

FULL_EXPERIMENT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_csv(
    CONDITION_INVENTORY_PATH,
    condition_inventory,
)

atomic_csv(
    RAW_MANIFEST_PATH,
    raw_manifest,
)

atomic_csv(
    BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)

atomic_csv(
    COMBINED_CONDITION_AUDIT_PATH,
    combined_condition_audit,
)

atomic_csv(
    COMBINED_PROJECT_RUNS_PATH,
    combined_project_runs,
)

atomic_csv(
    COMBINED_BUILD_METRICS_PATH,
    combined_build_metrics,
)

atomic_csv(
    COMBINED_MODEL_FITS_PATH,
    combined_model_fits,
)

atomic_csv(
    WORKER_CHECKPOINT_AUDIT_PATH,
    worker_audit,
)

atomic_csv(
    STEP5A_VALIDATION_PATH,
    step5a_validation,
)

aggregate_output_paths = [
    CONDITION_INVENTORY_PATH,
    RAW_MANIFEST_PATH,
    BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH,
    COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH,
    COMBINED_MODEL_FITS_PATH,
    WORKER_CHECKPOINT_AUDIT_PATH,
    STEP5A_VALIDATION_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
]

aggregate_output_manifest = [
    {
        "Path":
            str(path),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in aggregate_output_paths
]

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

master_execution_seconds = (
    time.perf_counter()
    - master_started
)

worker_checkpoint_records = [
    {
        "WorkerTag":
            row.WorkerTag,

        "AssignedSeedsJSON":
            row.AssignedSeedsJSON,

        "Status":
            row.Status,

        "ActualCheckpointSHA256":
            row.ActualCheckpointSHA256,

        "ReportedCheckpointSHA256":
            row.ReportedCheckpointSHA256,

        "CheckpointMetadataRefreshed":
            bool(
                row.CheckpointMetadataRefreshed
            ),

        "WorkerRawFiles":
            int(
                row.WorkerRawFiles
            ),

        "WorkerRawBytes":
            int(
                row.WorkerRawBytes
            ),

        "WorkerRawRootSHA256":
            row.ActualWorkerRawRootSHA256,

        "WorkerReportSHA256":
            row.WorkerReportSHA256,

        "SharedSmokeEquivalenceSHA256":
            row.SharedSmokeEquivalenceSHA256,
    }
    for row in worker_audit.itertuples(
        index=False
    )
]

report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_STATUS,

    "Implementation":
        MASTER_FINALIZER_IMPLEMENTATION,

    "CompletedAtUTC":
        completed_at_utc,

    "ExecutionMode":
        "PARALLEL_SIX_WORKERS_THEN_ZERO_FIT_MASTER_FINALIZATION",

    "AcceleratedEngineVersion":
        ACCELERATED_ENGINE_VERSION,

    "SelectionCheckpointSHA256":
        EXPECTED_SELECTION_CHECKPOINT_SHA256,

    "RECCheckpointSHA256":
        EXPECTED_REC_CHECKPOINT_SHA256,

    "NoisePlanCheckpointSHA256":
        EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,

    "RuntimeCheckpointSHA256":
        EXPECTED_RUNTIME_CHECKPOINT_SHA256,

    "SmokeCheckpointSHA256":
        EXPECTED_SMOKE_CHECKPOINT_SHA256,

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA256,

    "RegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "WorkerCheckpoints":
        worker_checkpoint_records,

    "ActualWorkerCheckpointSHA256":
        worker_checkpoint_hashes_actual,

    "WorkerCheckpointMetadataRefreshes":
        worker_checkpoint_hash_refreshes,

    "WorkersVerified":
        len(worker_audit),

    "SeedsCoveredExactlyOnce":
        True,

    "Conditions":
        len(condition_inventory),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "MLFits":
        len(combined_model_fits),

    "RankingRows":
        int(
            condition_inventory[
                "RankingRows"
            ].sum()
        ),

    "BuildMetricRows":
        len(combined_build_metrics),

    "ProjectRunRows":
        len(combined_project_runs),

    "ConditionAuditRows":
        len(combined_condition_audit),

    "TrainingMedianRows":
        int(
            condition_inventory[
                "TrainingMedianRows"
            ].sum()
        ),

    "RawFiles":
        raw_files,

    "RawBytes":
        raw_bytes,

    "RawRootSHA256":
        raw_root_sha256,

    "BaselineInvarianceFailures":
        baseline_invariance_failures,

    "QTFGlobalKeyVariants":
        qtf_global_key_variants,

    "QTFGlobalScoreVariants":
        qtf_global_score_variants,

    "QTFGlobalRankVariants":
        qtf_global_rank_variants,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "SharedSmokeEquivalenceSHA256":
        sha256_file(
            ACCELERATED_EQUIVALENCE_PATH
        ),

    "ValidationChecks":
        len(step5a_validation),

    "FailedValidationChecks":
        len(failed_validation),

    "MasterFinalizerModelFits":
        0,

    "MasterFinalizerConditionExecutions":
        0,

    "SourceRestoredInThisRuntime":
        source_restored,

    "MasterExecutionSeconds":
        float(
            master_execution_seconds
        ),

    "AggregateOutputManifest":
        aggregate_output_manifest,

    "RegistryModified":
        False,

    "Projects1To23Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,
}

atomic_json(
    STEP5A_REPORT_PATH,
    report_payload,
)

checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_24_FULL_270_CONDITION_STEP5A_FREEZE",

    "Full270ConditionExperimentComplete":
        True,

    "RawResultRootFrozen":
        True,

    "RawResultRootReadOnlyDuringMasterFinalization":
        True,

    "AllSixWorkerScientificRawRootsVerified":
        True,

    "All270ConditionsIndependentlyRevalidated":
        True,

    "EvaluationCohortImmutable":
        True,

    "ReadyForStep5B":
        True,

    "NextRequiredStep":
        "PROJECT 24 STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION",
}

atomic_json(
    STEP5A_CHECKPOINT_PATH,
    checkpoint_payload,
)

step5a_checkpoint_sha256 = sha256_file(
    STEP5A_CHECKPOINT_PATH
)

status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "Conditions":
        EXPECTED_CONDITIONS,

    "MLFits":
        EXPECTED_TOTAL_MODEL_FITS,

    "RawFiles":
        EXPECTED_RAW_FILES,

    "RawBytes":
        EXPECTED_RAW_BYTES,

    "RawRootSHA256":
        raw_root_sha256,

    "Checkpoint":
        str(
            STEP5A_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        step5a_checkpoint_sha256,

    "MasterFinalizerModelFits":
        0,

    "MasterFinalizerConditionExecutions":
        0,

    "RegistryModified":
        False,

    "ReadyForStep5B":
        True,

    "NextRequiredStep":
        "PROJECT 24 STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION",
}

atomic_json(
    STEP5A_STATUS_PATH,
    status_payload,
)

atomic_json(
    RUN_PROGRESS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "Status":
            STEP5A_STATUS,

        "ConditionsComplete":
            EXPECTED_CONDITIONS,

        "ConditionsTotal":
            EXPECTED_CONDITIONS,

        "WorkerShardsComplete":
            6,

        "WorkerShardsTotal":
            6,

        "OfficialStep5AComplete":
            True,

        "Step5ACheckpointSHA256":
            step5a_checkpoint_sha256,
    },
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL READBACK + RAW/SOURCE/REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    STEP5A_CHECKPOINT_PATH
)

report_readback = load_json(
    STEP5A_REPORT_PATH
)

status_readback = load_json(
    STEP5A_STATUS_PATH
)

for label, payload in [
    (
        "Step 5A checkpoint",
        checkpoint_readback,
    ),
    (
        "Step 5A report",
        report_readback,
    ),
    (
        "Step 5A status",
        status_readback,
    ),
]:
    if payload.get(
        "Status"
    ) != STEP5A_STATUS:
        raise RuntimeError(
            f"{label} readback failed."
        )

if not bool(
    checkpoint_readback.get(
        "ReadyForStep5B",
        False,
    )
):
    raise RuntimeError(
        "Step 5A checkpoint does not authorize Step 5B."
    )

aggregate_manifest_failures = verify_manifest_entries(
    checkpoint_readback.get(
        "AggregateOutputManifest",
        [],
    ),
    "official Step 5A",
)

if aggregate_manifest_failures:
    display(
        pd.DataFrame(
            aggregate_manifest_failures
        )
    )

    raise RuntimeError(
        "Official Step 5A aggregate-output readback failed."
    )

raw_manifest_final = directory_manifest(
    FULL_RAW_RESULT_ROOT
)

if (
    len(raw_manifest_final)
    != EXPECTED_RAW_FILES
    or int(
        raw_manifest_final[
            "Bytes"
        ].sum()
    ) != EXPECTED_RAW_BYTES
    or directory_root_hash(
        raw_manifest_final
    ) != raw_root_sha256
):
    raise RuntimeError(
        "Project 24 raw root changed during official master writes."
    )

if sha256_file(
    REGISTRY_PATH
) != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry changed during master finalization."
    )

final_source_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    })

if source_root_hash(
    pd.DataFrame(
        final_source_records
    )
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Project 24 source changed during master finalization."
    )

for path_string, expected_sha in cohort_hashes_before.items():
    if sha256_file(
        Path(
            path_string
        )
    ) != expected_sha:
        raise RuntimeError(
            "A frozen Project 24 cohort/plan changed during master finalization:\n"
            f"{path_string}"
        )


# --------------------------------------------------------------------------------------------------
# 13. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 140
)

print(
    "=== PROJECT 24 CELL 9 / STEP 5A RESULT ==="
)

print(
    "=" * 140
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print()

print(
    "Parallel workers:"
)

print(
    "Workers verified:",
    len(worker_audit),
    "/ 6",
)

print(
    "Seeds covered exactly once:",
    sorted(worker_seed_union)
    == REPETITION_SEEDS
    and len(worker_seed_union)
    == len(set(worker_seed_union)),
)

print(
    "Worker checkpoint metadata refreshes:",
    worker_checkpoint_hash_refreshes,
)

print()

print(
    "Full experiment:"
)

print(
    "Conditions:",
    len(condition_inventory),
    "/",
    EXPECTED_CONDITIONS,
)

print(
    "ML fits:",
    len(combined_model_fits),
    "/",
    EXPECTED_TOTAL_MODEL_FITS,
)

print(
    "Ranking rows:",
    int(
        condition_inventory[
            "RankingRows"
        ].sum()
    ),
    "/",
    EXPECTED_TOTAL_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    len(combined_build_metrics),
    "/",
    EXPECTED_TOTAL_BUILD_METRIC_ROWS,
)

print(
    "Project-run rows:",
    len(combined_project_runs),
    "/",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)

print(
    "Condition-audit rows:",
    len(combined_condition_audit),
    "/",
    EXPECTED_TOTAL_CONDITION_AUDIT_ROWS,
)

print(
    "Training-median rows:",
    int(
        condition_inventory[
            "TrainingMedianRows"
        ].sum()
    ),
    "/",
    EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS,
)

print()

print(
    "Raw result freeze:"
)

print(
    "Raw files:",
    raw_files,
    "/",
    EXPECTED_RAW_FILES,
)

print(
    "Raw bytes:",
    raw_bytes,
    "/",
    EXPECTED_RAW_BYTES,
)

print(
    "Raw root SHA-256:",
    raw_root_sha256,
)

print()

print(
    "Baselines and metrics:"
)

print(
    "Baseline invariance failures:",
    baseline_invariance_failures,
)

print(
    "QTF global score variants:",
    qtf_global_score_variants,
)

print(
    "QTF global rank variants:",
    qtf_global_rank_variants,
)

print(
    "Primary / secondary metrics: APFDc / APFD"
)

print()

print(
    "Isolation:"
)

print(
    "Completion registry unchanged:",
    sha256_file(
        REGISTRY_PATH
    )
    == EXPECTED_REGISTRY_SHA256,
)

print(
    "Projects 1–23 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Master-finalizer model fits:",
    0,
)

print(
    "Master-finalizer condition executions:",
    0,
)

print()

print(
    "Validation:"
)

print(
    "Checks:",
    len(step5a_validation),
)

print(
    "Failed checks:",
    len(failed_validation),
)

print()

print(
    "Official Step 5A checkpoint:"
)

print(
    STEP5A_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    step5a_checkpoint_sha256,
)

print()

print(
    "Master finalization seconds:",
    round(
        master_execution_seconds,
        2,
    ),
)

print()

print(
    "Next required step:",
    "PROJECT 24 STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION",
)

print()

print(
    "STATUS:",
    STEP5A_STATUS,
)

print(
    "=" * 140
)


=== PROJECT 24 CELL 9 / STEP 5A: PARALLEL MASTER FINALIZATION — ZERO FIT ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


RuntimeError: Project 24 is unexpectedly already registered by ProjectNumber.

In [3]:
# ==================================================================================================
# PROJECT 24 — CELL 9 / STEP 5A PARALLEL MASTER FINALIZATION
# ZERO-FIT REVALIDATION, AGGREGATION, RAW-ROOT FREEZE, AND OFFICIAL CHECKPOINT
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS AS THE NEXT NEW CELL IN THE RECONNECTED MASTER Thesis_project_24.ipynb NOTEBOOK.
#
# PURPOSE:
# - verify the six completed Project 24 Step 5A worker shards;
# - prove the six seed shards are disjoint and cover seeds 1–30 exactly;
# - independently revalidate all 270 raw condition directories without fitting any model;
# - rebuild the official compact Step 5A aggregate outputs from the frozen raw condition files;
# - independently hash and freeze the complete 2,160-file Project 24 raw-result root;
# - create the official Project 24 Step 5A report/status/checkpoint required by Step 5B.
#
# IMPORTANT CHECKPOINT NOTE:
# - resume-safe worker re-finalization can legitimately rewrite only volatile worker metadata
#   (CompletedAtUTC / invocation runtime), which changes a worker-checkpoint SHA while leaving all
#   frozen scientific raw outputs unchanged;
# - therefore this master records the ACTUAL current SHA of every worker checkpoint, but gates the
#   scientific freeze on exact pre-recorded worker raw-root SHA-256, raw bytes, seed shard, counts,
#   worker report linkage, worker output manifests, and an independent master revalidation of every
#   condition directory;
# - the official Step 5A checkpoint is thus cryptographically bound to the exact six checkpoint files
#   that exist at master-finalization time, while remaining robust to the accidental resume-safe
#   seeds-1–5 rerun.
#
# SAFETY:
# - ZERO model fitting;
# - ZERO noise injection / REC reconstruction / condition execution;
# - no completion-registry write;
# - no access to prior-project condition outputs;
# - raw condition directories are READ ONLY;
# - official master outputs are written only after every validation gate passes.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gzip
import hashlib
import json
import os
import shutil
import tarfile
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive


print("=" * 140)
print("=== PROJECT 24 CELL 9 / STEP 5A: PARALLEL MASTER FINALIZATION — ZERO FIT ===")
print("=" * 140)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN CONSTANTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

STEP5A_MASTER_CODE_REVISION = (
    "PROJECT_24_STEP5A_MASTER_V3_CORRECT_PROJECT_IDENTITY_AND_REGISTRY_GATE"
)

EXPECTED_SELECTION_STATUS = "PASS_PROJECT_24_SELECTION_AND_SOURCE_FROZEN"
EXPECTED_STEP2B_STATUS = "PASS_PROJECT_24_CLEAN_REC_RECONSTRUCTION_AND_ANCHOR_FROZEN"
EXPECTED_STEP3A_STATUS = "PASS_PROJECT_24_DETERMINISTIC_NOISE_PLAN_AND_COHORTS_FROZEN"
EXPECTED_STEP4A_STATUS = "PASS_PROJECT_24_EXPERIMENT_RUNTIME_AND_MODEL_CONTRACT_FROZEN"
EXPECTED_STEP4B_STATUS = "PASS_PROJECT_24_TWO_CONDITION_END_TO_END_SMOKE_TEST"

STEP5A_STATUS = "PASS_PROJECT_24_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
CONDITION_STATUS = "PASS_FULL_CONDITION"

MASTER_FINALIZER_IMPLEMENTATION = (
    "PROJECT_24_PARALLEL_STEP5A_MASTER_FINALIZER_V1_ZERO_FIT_"
    "WORKER_RAW_ROOT_BOUND_RESUME_METADATA_TOLERANT"
)

EXPECTED_SELECTION_CHECKPOINT_SHA256 = (
    "d64a6f862d2303c250aab96d09a99d3d24bf6c5b73f16707e214ffb42185bcf2"
)
EXPECTED_REC_CHECKPOINT_SHA256 = (
    "80ac2e6986ae2c14d636c57c834cc47678b54cd05ba32395b8add03debca3803"
)
EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256 = (
    "9dba84682eabdc1393219671e77aabaf1f2d4385f29239236e50615bf0180515"
)
EXPECTED_RUNTIME_CHECKPOINT_SHA256 = (
    "ff9ed031d21d67bf218d5eab70bd8502ca39953adedbe840348f382d5599314e"
)
EXPECTED_SMOKE_CHECKPOINT_SHA256 = (
    "3f5a0cc24260b251fd8cd43f7391ff7d3cc3d517e8f79a8ad443f4002cdc9ff2"
)

EXPECTED_SOURCE_ROOT_SHA256 = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)
EXPECTED_REGISTRY_SHA256 = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

EXPECTED_COMPLETE_STATUS = "COMPLETE_AND_FROZEN"
EXPECTED_REGISTERED_PROJECTS = 23
EXPECTED_ACTIVE_RESERVATIONS = []

EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

EXPECTED_SOURCE_FILES = 6
EXPECTED_SOURCE_BYTES = 374_503_252

EXPECTED_RAW_TRAIN_ROWS = 4_040_801
EXPECTED_RAW_EVAL_ROWS = 1_594_226
EXPECTED_MODEL_TRAIN_ROWS = 205_696
EXPECTED_MODEL_EVAL_ROWS = 18_854
EXPECTED_MODEL_TRAIN_FAILURES = 1_777
EXPECTED_MODEL_EVAL_FAILURES = 20
EXPECTED_FAILING_EVAL_BUILDS = 17
EXPECTED_EVALUATION_BUILDS = 1_072
EXPECTED_PREDICTORS = 151
EXPECTED_REC_FEATURES = 19
EXPECTED_RNG_ROWS = 121_224_030

EXPECTED_TECHNIQUES = 7
EXPECTED_ML_TECHNIQUES = 4
EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 523_093_565

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
REPETITION_SEEDS = list(range(1, 31))

ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
BASELINE_TECHNIQUES = [
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ALL_TECHNIQUES = ML_TECHNIQUES + BASELINE_TECHNIQUES
INVARIANT_BASELINES = ["Random", "QTF-Avg"]

ACCELERATED_ENGINE_VERSION = (
    "PROJECT_24_FAST_DEPENDENT_REC_V1_"
    "SMOKE_EQUIVALENT_FROZEN_INFERRED_TEST_ORDER_ANCHOR_AWARE"
)

SMOKE_EQUIVALENCE_KEYS = [
    "noise_00__seed_01",
    "noise_50__seed_01",
]

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_MODEL_EVAL_ROWS * EXPECTED_TECHNIQUES
)
EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION = (
    EXPECTED_FAILING_EVAL_BUILDS * EXPECTED_TECHNIQUES
)
EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION = EXPECTED_TECHNIQUES
EXPECTED_MODEL_FIT_ROWS_PER_CONDITION = EXPECTED_ML_TECHNIQUES
EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS

EXPECTED_TOTAL_RANKING_ROWS = (
    EXPECTED_RANKING_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_BUILD_METRIC_ROWS = (
    EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_PROJECT_RUN_ROWS = (
    EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_MODEL_FITS = (
    EXPECTED_MODEL_FIT_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS = (
    EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION * EXPECTED_CONDITIONS
)
EXPECTED_TOTAL_CONDITION_AUDIT_ROWS = EXPECTED_CONDITIONS

EXPECTED_WORKER_CONDITIONS = 45
EXPECTED_WORKER_MODEL_FITS = 180
EXPECTED_WORKER_RAW_FILES = 360
EXPECTED_WORKER_RANKING_ROWS = 5_939_010
EXPECTED_WORKER_BUILD_METRIC_ROWS = 5_355
EXPECTED_WORKER_PROJECT_RUN_ROWS = 315
EXPECTED_WORKER_CONDITION_AUDIT_ROWS = 45
EXPECTED_WORKER_TRAINING_MEDIAN_ROWS = 6_795

# Scientific worker freezes reported by the successful Project 24 worker runs.
# ReportedCheckpointSHA256 is retained for provenance. It is NOT the scientific gate because a
# resume-safe rerun can refresh volatile checkpoint metadata. Actual checkpoint SHA-256 values are
# captured and frozen by this master after all scientific validations pass.
EXPECTED_WORKERS = {
    "seed_01_05": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_01_05_COMPLETE",
        "Seeds": list(range(1, 6)),
        "ReportedCheckpointSHA256":
            "c6e8d76fe71c6aedfba9c3a0632c5daa5e2a8ed7cd3598dc56330d68e19ac81b",
        "RawRootSHA256":
            "53c1767a911a5d52561e2e12777f18d74ef8386c151a60fdf75e1488c2ab5705",
        "RawBytes": 86_952_171,
    },
    "seed_06_10": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_06_10_COMPLETE",
        "Seeds": list(range(6, 11)),
        "ReportedCheckpointSHA256":
            "ce2a285795c0a2b7aa16a51026686646328e817c778e49bfd14992137449a43a",
        "RawRootSHA256":
            "1703eef52c16b247a43bd95b378b6e1feb3bc8f8da6fc8a5b8c395b4437572a7",
        "RawBytes": 87_379_614,
    },
    "seed_11_15": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_11_15_COMPLETE",
        "Seeds": list(range(11, 16)),
        "ReportedCheckpointSHA256":
            "a8af44492f79733b78955a4f26213f5abd56332874cecb7c32b2a15f21392270",
        "RawRootSHA256":
            "01fef4c6d37ac6ee491a415aee4a3eabf05551be372c09fdc1b997fb52f7b93d",
        "RawBytes": 87_582_299,
    },
    "seed_16_20": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_16_20_COMPLETE",
        "Seeds": list(range(16, 21)),
        "ReportedCheckpointSHA256":
            "3177e9c8d783df28d2f28f60c75eb69d239c885c4273c55f0ed702ca1d63fd9d",
        "RawRootSHA256":
            "910ef203274d1ea744ad92ef1ec7802777d38be713333b21f5191fa69d4c364a",
        "RawBytes": 87_224_279,
    },
    "seed_21_25": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_21_25_COMPLETE",
        "Seeds": list(range(21, 26)),
        "ReportedCheckpointSHA256":
            "7cc8475c5c42fbfd86ab92e4024c7e7d486d03d8d7f053955bc86bb0fdb0c3d1",
        "RawRootSHA256":
            "a311f4370ebe599c4d7cee8a6d865ed7857c1b8d2641998fbabac352e30d7e1e",
        "RawBytes": 87_220_469,
    },
    "seed_26_30": {
        "Status": "PASS_PROJECT_24_STEP5A_WORKER_SEEDS_26_30_COMPLETE",
        "Seeds": list(range(26, 31)),
        "ReportedCheckpointSHA256":
            "d9cf039b99bdee7b502adf78497c028c8d8b918556d68b6cf0eca975c56661b1",
        "RawRootSHA256":
            "b09e26b9bc4087379eac076931f70cc141083f9e6d15584c40a433f31f071aac",
        "RawBytes": 86_734_733,
    },
}

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis_Experiment")
NOTES_ROOT = THESIS_ROOT / "Notes"
RESULTS_ROOT = THESIS_ROOT / "Results"

REGISTRY_PATH = NOTES_ROOT / "completed_project_registry.csv"
ARCHIVE_PATH = THESIS_ROOT / "Data" / "Raw" / "TCP-CI-main-dataset.tar.gz"
SOURCE_DIR = Path("/content/datasets/datasets/SonarSource@sonarqube")

SELECTION_ROOT = RESULTS_ROOT / "Aggregated" / "project_24_selection"
FROZEN_SOURCE_MANIFEST_PATH = (
    SELECTION_ROOT / "project_24_frozen_source_manifest.csv"
)
SELECTION_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_selection_checkpoint.json"
)

PROJECT_ROOT = RESULTS_ROOT / "Aggregated" / PROJECT_SLUG
REC_PREFLIGHT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_rec_preflight"
REC_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_rec_reconstruction_checkpoint.json"
)

NOISE_PLAN_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_noise_plan"
RAW_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_training_cohort.parquet"
)
RAW_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_raw_evaluation_cohort.parquet"
)
MODEL_TRAINING_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_training_cohort.parquet"
)
MODEL_EVALUATION_COHORT_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_fixed_model_evaluation_cohort.parquet"
)
RNG_MANIFEST_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_rng_manifest.parquet"
)
CONDITION_PLAN_PATH = (
    NOISE_PLAN_ROOT / f"{PROJECT_SHORT}_condition_plan.csv"
)
NOISE_PLAN_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_noise_plan_checkpoint.json"
)

RUNTIME_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_runtime_contract"
RUNTIME_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_runtime_contract_checkpoint.json"
)

SMOKE_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_smoke_test"
SMOKE_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_smoke_test_checkpoint.json"
)

FULL_RAW_RESULT_ROOT = RESULTS_ROOT / "Raw" / PROJECT_SLUG
FULL_EXPERIMENT_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"

ACCELERATED_EQUIVALENCE_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_accelerated_engine_equivalence.csv"
)

PARALLEL_WORKERS_ROOT = FULL_EXPERIMENT_ROOT / "parallel_workers"

WORKER_CHECKPOINT_PATHS = {
    tag: NOTES_ROOT / f"project_24_step5a_worker_{tag}_checkpoint.json"
    for tag in EXPECTED_WORKERS
}

CONDITION_INVENTORY_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_condition_inventory.csv"
)
RAW_MANIFEST_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
)
BASELINE_INVARIANCE_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"
)
COMBINED_CONDITION_AUDIT_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_condition_audit.csv"
)
COMBINED_PROJECT_RUNS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_project_runs.csv"
)
COMBINED_BUILD_METRICS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_build_metrics.csv"
)
COMBINED_MODEL_FITS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_combined_model_fits.csv"
)
WORKER_CHECKPOINT_AUDIT_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_worker_checkpoint_audit.csv"
)
STEP5A_VALIDATION_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_validation.csv"
)
STEP5A_REPORT_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
)
RUN_PROGRESS_PATH = (
    FULL_EXPERIMENT_ROOT / f"{PROJECT_SHORT}_run_progress.json"
)

STEP5A_STATUS_PATH = (
    PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
)
STEP5A_CHECKPOINT_PATH = (
    NOTES_ROOT / "project_24_step5a_checkpoint.json"
)


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
        handle.write("\n")

    os.replace(temporary, path)


def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp_{os.getpid()}")

    frame.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
    )

    os.replace(temporary, path)


def source_root_hash(frame):
    digest = hashlib.sha256()

    for row in frame.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        size = getattr(
            row,
            "SizeBytes",
            getattr(row, "Bytes", None),
        )

        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(size)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


DIRECTORY_MANIFEST_COLUMNS = [
    "RelativePath",
    "Bytes",
    "SHA256",
]


def directory_manifest(root):
    root = Path(root)
    records = []

    if root.exists():
        for path in sorted(
            (
                candidate
                for candidate in root.rglob("*")
                if candidate.is_file()
            ),
            key=lambda candidate:
                candidate.relative_to(root).as_posix(),
        ):
            records.append({
                "RelativePath":
                    path.relative_to(root).as_posix(),

                "Bytes":
                    int(path.stat().st_size),

                "SHA256":
                    sha256_file(path),
            })

    return pd.DataFrame(
        records,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )


def directory_root_hash(manifest):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(index=False):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode("utf-8")
        )

    return digest.hexdigest()


def add_check(rows, check, expected, actual, passed):
    rows.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Pass": bool(passed),
    })


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
        "y",
    }


def verify_manifest_entries(entries, label):
    failures = []

    if not isinstance(entries, list) or not entries:
        return [{
            "Label": label,
            "Path": "<manifest>",
            "Reason": "missing/empty manifest",
        }]

    for item in entries:
        path = Path(str(item.get("Path", "")))

        if not path.is_file():
            failures.append({
                "Label": label,
                "Path": str(path),
                "Reason": "missing",
            })
            continue

        expected_bytes = int(item.get("Bytes", -1))
        expected_sha = str(item.get("SHA256", "")).lower()
        actual_bytes = int(path.stat().st_size)
        actual_sha = sha256_file(path)

        if (
            actual_bytes != expected_bytes
            or actual_sha != expected_sha
        ):
            failures.append({
                "Label": label,
                "Path": str(path),
                "Reason": "size/hash mismatch",
            })

    return failures


def resolve_registry_columns(registry):
    lookup = {
        str(column).strip().lower(): column
        for column in registry.columns
    }

    def pick(names, label):
        for name in names:
            if name in lookup:
                return lookup[name]

        raise RuntimeError(
            f"Could not resolve registry {label}; "
            f"columns={list(registry.columns)}"
        )

    return (
        pick(
            [
                "projectnumber",
                "project_number",
                "project no",
                "projectno",
            ],
            "ProjectNumber",
        ),
        pick(
            [
                "project",
                "projectname",
                "project_name",
            ],
            "Project",
        ),
        pick(
            [
                "status",
                "projectstatus",
                "project_status",
            ],
            "Status",
        ),
    )


def count_csv_data_rows(path):
    """
    Count CSV data rows without materialising the full frame.
    rankings.csv.gz is the only large file; all generated ranking rows contain no embedded newlines.
    """
    path = Path(path)

    opener = gzip.open if path.suffix == ".gz" else open

    with opener(
        path,
        "rt",
        encoding="utf-8",
        newline="",
    ) as handle:
        count = -1  # remove header
        for _ in handle:
            count += 1

    return max(count, 0)


def restore_source_if_needed():
    required_files = {
        "builds.csv",
        "contributors.csv",
        "dataset.csv",
        "entity_change_history.csv",
        "exe.csv",
        "id_map.csv",
    }

    source_ready = bool(
        SOURCE_DIR.is_dir()
        and required_files.issubset({
            path.name
            for path in SOURCE_DIR.iterdir()
            if path.is_file()
        })
    )

    if source_ready:
        return False

    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(
            "Project 24 source is absent from this reconnected runtime "
            "and the frozen thesis archive is missing:\n"
            f"{ARCHIVE_PATH}"
        )

    print(
        "\nRestoring only SonarSource@sonarqube from the frozen TCP-CI archive "
        "into this master runtime."
    )

    local_dataset_root = Path("/content/datasets")
    local_dataset_root.mkdir(parents=True, exist_ok=True)

    member_prefix = "datasets/SonarSource@sonarqube/"
    resolved_root = local_dataset_root.resolve()
    extracted_files = 0

    with tarfile.open(
        ARCHIVE_PATH,
        mode="r:gz",
    ) as archive:
        for member in archive:
            member_name = (
                member.name
                .replace("\\", "/")
                .lstrip("/")
            )

            if not (
                member_name == "datasets/SonarSource@sonarqube"
                or member_name.startswith(member_prefix)
            ):
                continue

            target_path = (
                local_dataset_root
                / member_name
            )

            resolved_target = target_path.resolve()

            if (
                resolved_target != resolved_root
                and resolved_root not in resolved_target.parents
            ):
                raise RuntimeError(
                    "Unsafe archive member encountered:\n"
                    f"{member.name}"
                )

            if member.isdir():
                target_path.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            elif member.isfile():
                target_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                source_handle = archive.extractfile(
                    member
                )

                if source_handle is None:
                    raise RuntimeError(
                        "Could not read archive member:\n"
                        f"{member.name}"
                    )

                with (
                    source_handle,
                    target_path.open("wb") as output_handle,
                ):
                    shutil.copyfileobj(
                        source_handle,
                        output_handle,
                        length=8 * 1024 * 1024,
                    )

                extracted_files += 1

    source_names = (
        {
            path.name
            for path in SOURCE_DIR.iterdir()
            if path.is_file()
        }
        if SOURCE_DIR.is_dir()
        else set()
    )

    missing = sorted(
        required_files
        - source_names
    )

    if missing:
        raise FileNotFoundError(
            "Selective Project 24 source restoration is incomplete:\n"
            + "\n".join(missing)
        )

    print(
        "Project source files restored:",
        extracted_files,
    )

    return True


EXPECTED_CONDITION_FILES = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}


def validate_condition_read_only(condition_dir, plan_row):
    """
    Independent master-side revalidation.
    No condition output is modified.
    """
    condition_dir = Path(condition_dir)
    condition_key = str(plan_row.ConditionID)

    if not condition_dir.is_dir():
        return None, f"{condition_key}: condition directory missing"

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    if actual_files != EXPECTED_CONDITION_FILES:
        return (
            None,
            f"{condition_key}: file set differs: "
            f"{sorted(actual_files)}",
        )

    completion_path = (
        condition_dir / "COMPLETE.json"
    )
    summary_path = (
        condition_dir / "condition_summary.json"
    )

    try:
        completion = load_json(completion_path)
        summary = load_json(summary_path)
    except Exception as error:
        return (
            None,
            f"{condition_key}: JSON read failure: {error!r}",
        )

    identity_checks = [
        (
            completion.get("Status"),
            CONDITION_STATUS,
            "completion status",
        ),
        (
            summary.get("Status"),
            CONDITION_STATUS,
            "summary status",
        ),
        (
            completion.get("ConditionKey"),
            condition_key,
            "completion key",
        ),
        (
            summary.get("ConditionKey"),
            condition_key,
            "summary key",
        ),
        (
            summary.get("Project"),
            PROJECT_NAME,
            "project",
        ),
        (
            summary.get("ProjectSlug"),
            PROJECT_SLUG,
            "project slug",
        ),
        (
            summary.get("AcceleratedEngineVersion"),
            ACCELERATED_ENGINE_VERSION,
            "accelerated engine",
        ),
    ]

    for actual, expected, label in identity_checks:
        if actual != expected:
            return (
                None,
                f"{condition_key}: {label} differs; "
                f"expected={expected!r}, actual={actual!r}",
            )

    if int(
        summary.get("ConditionOrder", -1)
    ) != int(plan_row.ConditionOrder):
        return None, f"{condition_key}: ConditionOrder differs"

    if int(
        summary.get("NoisePercent", -1)
    ) != int(plan_row.NoisePercent):
        return None, f"{condition_key}: NoisePercent differs"

    if int(
        summary.get("RepetitionSeed", -1)
    ) != int(plan_row.RepetitionSeed):
        return None, f"{condition_key}: RepetitionSeed differs"

    if str(
        completion.get("ConditionSummaryPath")
    ) != str(summary_path):
        return None, f"{condition_key}: summary path linkage differs"

    if str(
        completion.get("ConditionSummarySHA256")
    ) != sha256_file(summary_path):
        return None, f"{condition_key}: summary SHA linkage differs"

    output_manifest = summary.get(
        "OutputManifest",
        [],
    )

    if (
        not isinstance(output_manifest, list)
        or len(output_manifest) != 6
    ):
        return None, f"{condition_key}: output manifest differs"

    output_manifest_failures = verify_manifest_entries(
        output_manifest,
        condition_key,
    )

    if output_manifest_failures:
        return (
            None,
            f"{condition_key}: output manifest validation failed",
        )

    for item in output_manifest:
        if Path(
            str(item.get("Path", ""))
        ).parent != condition_dir:
            return (
                None,
                f"{condition_key}: output-manifest path escaped condition directory",
            )

    expected_counts = {
        "MLFits":
            EXPECTED_MODEL_FIT_ROWS_PER_CONDITION,
        "RankingRows":
            EXPECTED_RANKING_ROWS_PER_CONDITION,
        "BuildMetricRows":
            EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION,
        "ProjectRunRows":
            EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION,
        "TrainingMedianRows":
            EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION,
    }

    for key, expected in expected_counts.items():
        if int(
            summary.get(key, -1)
        ) != expected:
            return (
                None,
                f"{condition_key}: summary {key} differs",
            )

    if int(
        summary.get("IndependentRECChanges", -1)
    ) != 0:
        return (
            None,
            f"{condition_key}: independent REC changes are nonzero",
        )

    fingerprints = summary.get(
        "BaselineFingerprints",
        {},
    )

    if sorted(
        fingerprints.keys()
    ) != [
        "QTF-Avg",
        "Random",
    ]:
        return (
            None,
            f"{condition_key}: baseline fingerprints differ",
        )

    # Independent content-level readback of all compact raw outputs.
    condition_audit = pd.read_csv(
        condition_dir / "condition_audit.csv",
        low_memory=False,
    )

    if len(condition_audit) != 1:
        return (
            None,
            f"{condition_key}: condition_audit row count differs",
        )

    audit = condition_audit.iloc[0]

    for column, expected in [
        ("ConditionKey", condition_key),
        ("NoisePercent", int(plan_row.NoisePercent)),
        ("RepetitionSeed", int(plan_row.RepetitionSeed)),
        ("Status", CONDITION_STATUS),
    ]:
        if column not in condition_audit.columns:
            return (
                None,
                f"{condition_key}: condition_audit missing {column}",
            )

        actual = audit[column]

        if column in {
            "NoisePercent",
            "RepetitionSeed",
        }:
            if int(actual) != int(expected):
                return (
                    None,
                    f"{condition_key}: audit {column} differs",
                )
        elif str(actual) != str(expected):
            return (
                None,
                f"{condition_key}: audit {column} differs",
            )

    for column in [
        "IndependentRECChanges",
        "IndependentReconstructionMismatches",
    ]:
        if (
            column not in condition_audit.columns
            or int(
                pd.to_numeric(
                    pd.Series([audit[column]]),
                    errors="raise",
                ).iloc[0]
            ) != 0
        ):
            return (
                None,
                f"{condition_key}: audit {column} differs",
            )

    for expected_column, actual_column in [
        (
            "ExpectedFlipMaskSHA256",
            "ActualFlipMaskSHA256",
        ),
        (
            "ExpectedNoisyRawVerdictSHA256",
            "ActualNoisyRawVerdictSHA256",
        ),
        (
            "ExpectedNoisyModelVerdictSHA256",
            "ActualNoisyModelVerdictSHA256",
        ),
    ]:
        if (
            expected_column not in condition_audit.columns
            or actual_column not in condition_audit.columns
            or str(audit[expected_column])
            != str(audit[actual_column])
        ):
            return (
                None,
                f"{condition_key}: noise-plan hash audit differs",
            )

    number_flipped = int(
        pd.to_numeric(
            pd.Series([
                audit["NumberFlipped"]
            ]),
            errors="raise",
        ).iloc[0]
    )

    model_label_changes = int(
        pd.to_numeric(
            pd.Series([
                audit["ModelLabelChanges"]
            ]),
            errors="raise",
        ).iloc[0]
    )

    dependent_rec_changes = int(
        pd.to_numeric(
            pd.Series([
                audit["DependentRECChanges"]
            ]),
            errors="raise",
        ).iloc[0]
    )

    if int(plan_row.NoisePercent) == 0:
        if (
            number_flipped != 0
            or model_label_changes != 0
            or dependent_rec_changes != 0
        ):
            return (
                None,
                f"{condition_key}: zero-noise scientific audit differs",
            )
    else:
        if (
            number_flipped <= 0
            or model_label_changes <= 0
            or dependent_rec_changes <= 0
        ):
            return (
                None,
                f"{condition_key}: positive-noise scientific audit differs",
            )

    model_fits = pd.read_csv(
        condition_dir / "model_fits.csv",
        low_memory=False,
    )

    if len(model_fits) != EXPECTED_MODEL_FIT_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: model-fit row count differs",
        )

    if (
        "Technique" not in model_fits.columns
        or "Status" not in model_fits.columns
        or sorted(
            model_fits["Technique"]
            .astype(str)
            .tolist()
        ) != sorted(ML_TECHNIQUES)
        or not model_fits["Status"]
        .astype(str)
        .eq("PASS_MODEL_FIT")
        .all()
    ):
        return (
            None,
            f"{condition_key}: model-fit contract differs",
        )

    build_metrics = pd.read_csv(
        condition_dir / "build_metrics.csv",
        low_memory=False,
    )

    if len(build_metrics) != EXPECTED_BUILD_METRIC_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: build-metric row count differs",
        )

    required_build_metric_columns = {
        "Technique",
        "Build",
        "APFDc",
        "APFD",
    }

    if not required_build_metric_columns.issubset(
        build_metrics.columns
    ):
        return (
            None,
            f"{condition_key}: build-metric columns differ",
        )

    if sorted(
        build_metrics["Technique"]
        .astype(str)
        .unique()
        .tolist()
    ) != sorted(ALL_TECHNIQUES):
        return (
            None,
            f"{condition_key}: build-metric technique set differs",
        )

    if not build_metrics.groupby(
        "Technique"
    ).size().eq(
        EXPECTED_FAILING_EVAL_BUILDS
    ).all():
        return (
            None,
            f"{condition_key}: build-metric rows per technique differ",
        )

    build_metric_values = build_metrics[
        ["APFDc", "APFD"]
    ].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(float)

    if (
        not np.isfinite(
            build_metric_values
        ).all()
        or (
            (
                build_metric_values < 0
            )
            | (
                build_metric_values > 1
            )
        ).any()
    ):
        return (
            None,
            f"{condition_key}: build metrics invalid",
        )

    project_runs = pd.read_csv(
        condition_dir / "project_runs.csv",
        low_memory=False,
    )

    if len(project_runs) != EXPECTED_PROJECT_RUN_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: project-run row count differs",
        )

    if (
        "Technique" not in project_runs.columns
        or sorted(
            project_runs["Technique"]
            .astype(str)
            .tolist()
        ) != sorted(ALL_TECHNIQUES)
    ):
        return (
            None,
            f"{condition_key}: project-run technique set differs",
        )

    project_metric_columns = [
        "MeanAPFDc",
        "MedianAPFDc",
        "MeanAPFD",
        "MedianAPFD",
    ]

    if not set(
        project_metric_columns
    ).issubset(
        project_runs.columns
    ):
        return (
            None,
            f"{condition_key}: project metric columns differ",
        )

    project_metric_values = project_runs[
        project_metric_columns
    ].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(float)

    if (
        not np.isfinite(
            project_metric_values
        ).all()
        or (
            (
                project_metric_values < 0
            )
            | (
                project_metric_values > 1
            )
        ).any()
    ):
        return (
            None,
            f"{condition_key}: project metrics invalid",
        )

    training_medians = pd.read_csv(
        condition_dir / "training_medians.csv",
        low_memory=False,
    )

    if len(
        training_medians
    ) != EXPECTED_TRAINING_MEDIAN_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: training-median row count differs",
        )

    if (
        "Predictor" not in training_medians.columns
        or training_medians[
            "Predictor"
        ].astype(str).nunique()
        != EXPECTED_PREDICTORS
    ):
        return (
            None,
            f"{condition_key}: training-median predictor contract differs",
        )

    ranking_path = (
        condition_dir / "rankings.csv.gz"
    )

    ranking_rows = count_csv_data_rows(
        ranking_path
    )

    if ranking_rows != EXPECTED_RANKING_ROWS_PER_CONDITION:
        return (
            None,
            f"{condition_key}: ranking row count differs; "
            f"expected={EXPECTED_RANKING_ROWS_PER_CONDITION}, "
            f"actual={ranking_rows}",
        )

    condition_manifest = directory_manifest(
        condition_dir
    )

    if len(
        condition_manifest
    ) != EXPECTED_FILES_PER_CONDITION:
        return (
            None,
            f"{condition_key}: condition manifest file count differs",
        )

    inventory = {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionOrder":
            int(plan_row.ConditionOrder),

        "ConditionKey":
            condition_key,

        "NoisePercent":
            int(plan_row.NoisePercent),

        "RepetitionSeed":
            int(plan_row.RepetitionSeed),

        "ConditionDirectory":
            str(condition_dir),

        "Status":
            CONDITION_STATUS,

        "Files":
            int(len(condition_manifest)),

        "ConditionBytes":
            int(
                condition_manifest[
                    "Bytes"
                ].sum()
            ),

        "ConditionRootSHA256":
            directory_root_hash(
                condition_manifest
            ),

        "RankingRows":
            int(summary["RankingRows"]),

        "BuildMetricRows":
            int(summary["BuildMetricRows"]),

        "ProjectRunRows":
            int(summary["ProjectRunRows"]),

        "ModelFitRows":
            int(summary["MLFits"]),

        "TrainingMedianRows":
            int(summary["TrainingMedianRows"]),

        "NumberFlipped":
            int(summary.get(
                "NumberFlipped",
                -1,
            )),

        "ModelLabelChanges":
            int(summary.get(
                "ModelLabelChanges",
                -1,
            )),

        "DependentRECChanges":
            int(summary.get(
                "DependentRECChanges",
                -1,
            )),

        "IndependentRECChanges":
            int(summary.get(
                "IndependentRECChanges",
                -1,
            )),

        "TrainingFailures":
            int(summary.get(
                "TrainingFailures",
                -1,
            )),

        "ConditionSeconds":
            float(summary.get(
                "ConditionSeconds",
                np.nan,
            )),

        "RandomKeySHA256":
            str(
                fingerprints[
                    "Random"
                ][
                    "KeySHA256"
                ]
            ),

        "RandomScoreSHA256":
            str(
                fingerprints[
                    "Random"
                ][
                    "ScoreSHA256"
                ]
            ),

        "RandomRankSHA256":
            str(
                fingerprints[
                    "Random"
                ][
                    "RankSHA256"
                ]
            ),

        "QTFAvgKeySHA256":
            str(
                fingerprints[
                    "QTF-Avg"
                ][
                    "KeySHA256"
                ]
            ),

        "QTFAvgScoreSHA256":
            str(
                fingerprints[
                    "QTF-Avg"
                ][
                    "ScoreSHA256"
                ]
            ),

        "QTFAvgRankSHA256":
            str(
                fingerprints[
                    "QTF-Avg"
                ][
                    "RankSHA256"
                ]
            ),
    }

    return (
        {
            "Inventory":
                inventory,

            "ConditionAudit":
                condition_audit,

            "ProjectRuns":
                project_runs,

            "BuildMetrics":
                build_metrics,

            "ModelFits":
                model_fits,

            "Manifest":
                condition_manifest,

            "Fingerprints":
                fingerprints,
        },
        "",
    )


# --------------------------------------------------------------------------------------------------
# 4. MOUNT, RESTORE SOURCE IF NEEDED, VERIFY UPSTREAM FREEZES
# --------------------------------------------------------------------------------------------------

master_started = time.perf_counter()

drive.mount(
    "/content/drive",
    force_remount=False,
)

source_restored = restore_source_if_needed()

required_paths = [
    REGISTRY_PATH,
    FROZEN_SOURCE_MANIFEST_PATH,
    SELECTION_CHECKPOINT_PATH,
    REC_CHECKPOINT_PATH,
    NOISE_PLAN_CHECKPOINT_PATH,
    RUNTIME_CHECKPOINT_PATH,
    SMOKE_CHECKPOINT_PATH,
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
    FULL_RAW_RESULT_ROOT,
] + list(
    WORKER_CHECKPOINT_PATHS.values()
)

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing Project 24 master-finalization inputs:\n"
        + "\n".join(missing_paths)
    )

# Protect against accidental official Step 5A re-finalization.
if STEP5A_CHECKPOINT_PATH.is_file():
    existing = load_json(
        STEP5A_CHECKPOINT_PATH
    )

    if (
        existing.get("Status")
        == STEP5A_STATUS
        and bool(
            existing.get(
                "Full270ConditionExperimentComplete",
                False,
            )
        )
    ):
        raise RuntimeError(
            "Official Project 24 Step 5A is already frozen successfully. "
            "Do not rerun this master finalizer."
        )

    raise RuntimeError(
        "An unexpected pre-existing Project 24 Step 5A checkpoint exists. "
        "Stop and inspect it before finalization."
    )

checkpoint_hash_expectations = {
    "selection": (
        SELECTION_CHECKPOINT_PATH,
        EXPECTED_SELECTION_CHECKPOINT_SHA256,
    ),
    "REC": (
        REC_CHECKPOINT_PATH,
        EXPECTED_REC_CHECKPOINT_SHA256,
    ),
    "noise plan": (
        NOISE_PLAN_CHECKPOINT_PATH,
        EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,
    ),
    "runtime": (
        RUNTIME_CHECKPOINT_PATH,
        EXPECTED_RUNTIME_CHECKPOINT_SHA256,
    ),
    "smoke": (
        SMOKE_CHECKPOINT_PATH,
        EXPECTED_SMOKE_CHECKPOINT_SHA256,
    ),
}

for label, (
    path,
    expected_sha,
) in checkpoint_hash_expectations.items():
    actual_sha = sha256_file(path)

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Frozen Project 24 {label} checkpoint SHA-256 differs.\n"
            f"Expected: {expected_sha}\n"
            f"Actual:   {actual_sha}"
        )

selection_checkpoint = load_json(
    SELECTION_CHECKPOINT_PATH
)
rec_checkpoint = load_json(
    REC_CHECKPOINT_PATH
)
noise_checkpoint = load_json(
    NOISE_PLAN_CHECKPOINT_PATH
)
runtime_checkpoint = load_json(
    RUNTIME_CHECKPOINT_PATH
)
smoke_checkpoint = load_json(
    SMOKE_CHECKPOINT_PATH
)

for label, payload, expected_status in [
    (
        "selection checkpoint",
        selection_checkpoint,
        EXPECTED_SELECTION_STATUS,
    ),
    (
        "REC checkpoint",
        rec_checkpoint,
        EXPECTED_STEP2B_STATUS,
    ),
    (
        "noise-plan checkpoint",
        noise_checkpoint,
        EXPECTED_STEP3A_STATUS,
    ),
    (
        "runtime checkpoint",
        runtime_checkpoint,
        EXPECTED_STEP4A_STATUS,
    ),
    (
        "smoke checkpoint",
        smoke_checkpoint,
        EXPECTED_STEP4B_STATUS,
    ),
]:
    if payload.get(
        "Status"
    ) != expected_status:
        raise RuntimeError(
            f"{label} does not contain the frozen PASS status."
        )

    if (
        payload.get("Project") != PROJECT_NAME
        or payload.get("ProjectSlug") != PROJECT_SLUG
    ):
        raise RuntimeError(
            f"{label} Project 24 identity differs."
        )

if runtime_checkpoint.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Runtime checkpoint source root differs."
    )

if runtime_checkpoint.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Runtime checkpoint active reservations differ."
    )

if runtime_checkpoint.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Runtime checkpoint priority rule differs."
    )

if smoke_checkpoint.get(
    "RuntimeCheckpointSHA256"
) != EXPECTED_RUNTIME_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Smoke checkpoint is not linked to the frozen runtime checkpoint."
    )

if not bool(
    smoke_checkpoint.get(
        "ReadyForFull270ConditionExperiment",
        False,
    )
):
    raise RuntimeError(
        "Smoke checkpoint does not authorise the full experiment."
    )

upstream_manifest_failures = []

# These checkpoint schemas contain frozen file manifests.
for label, payload, manifest_key in [
    (
        "noise-plan",
        noise_checkpoint,
        "OutputManifest",
    ),
    (
        "runtime",
        runtime_checkpoint,
        "RuntimeOutputManifest",
    ),
    (
        "smoke",
        smoke_checkpoint,
        "OutputManifest",
    ),
]:
    failures = verify_manifest_entries(
        payload.get(
            manifest_key,
            [],
        ),
        label,
    )

    upstream_manifest_failures.extend(
        failures
    )

if upstream_manifest_failures:
    print(
        "\nUpstream manifest failures:"
    )
    display(
        pd.DataFrame(
            upstream_manifest_failures
        )
    )
    raise RuntimeError(
        "One or more frozen Project 24 upstream outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 5. REGISTRY, SOURCE, COHORT, AND CONDITION-PLAN IMMUTABILITY
# --------------------------------------------------------------------------------------------------

registry_sha_before = sha256_file(
    REGISTRY_PATH
)

if registry_sha_before != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry SHA-256 differs before Step 5A master finalization."
    )

registry = pd.read_csv(
    REGISTRY_PATH,
    low_memory=False,
)

(
    registry_project_number_column,
    registry_project_column,
    registry_status_column,
) = resolve_registry_columns(
    registry
)

registry_project_numbers = pd.to_numeric(
    registry[
        registry_project_number_column
    ],
    errors="raise",
).astype(int)

if (
    len(registry)
    != EXPECTED_REGISTERED_PROJECTS
    or sorted(
        registry_project_numbers.tolist()
    )
    != list(
        range(
            1,
            EXPECTED_REGISTERED_PROJECTS + 1,
        )
    )
):
    raise RuntimeError(
        "Completion registry does not contain exactly Projects 1–23."
    )

if not registry[
    registry_status_column
].astype(str).eq(
    EXPECTED_COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )

for number, identity in REQUIRED_REGISTERED_IDENTITIES.items():
    matching = registry.loc[
        registry_project_numbers.eq(
            number
        )
    ]

    if (
        len(matching) != 1
        or str(
            matching.iloc[0][
                registry_project_column
            ]
        ) != identity
    ):
        raise RuntimeError(
            "A frozen predecessor identity differs.\n"
            f"Project {number}; expected={identity}"
        )

# Registration is keyed by ProjectNumber in the frozen completion registry.
#
# IMPORTANT:
# The registry itself has already been byte-for-byte authenticated above by
# EXPECTED_REGISTRY_SHA256 and independently constrained to exactly rows 1..23.
# Therefore the scientifically relevant pre-Step-5C condition is simply that
# ProjectNumber 24 does not exist.  Do NOT reject on a Project-name string match:
# that extra name-based predicate was not part of the authoritative Project-24
# Step-1A registration check and caused a false-positive in the master finalizer.
project_24_registry_number_rows = int(
    registry_project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
)

project_name_match_rows_diagnostic = int(
    registry[
        registry_project_column
    ].astype(str).eq(
        PROJECT_NAME
    ).sum()
)

if project_24_registry_number_rows != 0:
    raise RuntimeError(
        "Project 24 is unexpectedly already registered by ProjectNumber."
    )

print(
    "Completion-registry ProjectNumber-24 rows:",
    project_24_registry_number_rows,
)

print(
    "Completion-registry exact project-name matches (diagnostic only):",
    project_name_match_rows_diagnostic,
)

frozen_source_manifest = pd.read_csv(
    FROZEN_SOURCE_MANIFEST_PATH,
    low_memory=False,
)

current_source_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Frozen Project 24 source file is missing: {path}"
        )

    current_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(path),
    })

current_source_manifest = pd.DataFrame(
    current_source_records
)

source_root_before = source_root_hash(
    current_source_manifest
)

if source_root_before != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Frozen Project 24 source root differs."
    )

if (
    len(current_source_manifest)
    != EXPECTED_SOURCE_FILES
    or int(
        current_source_manifest[
            "SizeBytes"
        ].sum()
    ) != EXPECTED_SOURCE_BYTES
):
    raise RuntimeError(
        "Frozen Project 24 source file/byte counts differ."
    )

cohort_paths = [
    RAW_TRAINING_COHORT_PATH,
    RAW_EVALUATION_COHORT_PATH,
    MODEL_TRAINING_COHORT_PATH,
    MODEL_EVALUATION_COHORT_PATH,
    RNG_MANIFEST_PATH,
    CONDITION_PLAN_PATH,
]

cohort_hashes_before = {
    str(path):
        sha256_file(path)
    for path in cohort_paths
}

if pq.ParquetFile(
    RAW_TRAINING_COHORT_PATH
).metadata.num_rows != EXPECTED_RAW_TRAIN_ROWS:
    raise RuntimeError(
        "Frozen raw training cohort row count differs."
    )

if pq.ParquetFile(
    RAW_EVALUATION_COHORT_PATH
).metadata.num_rows != EXPECTED_RAW_EVAL_ROWS:
    raise RuntimeError(
        "Frozen raw evaluation cohort row count differs."
    )

if pq.ParquetFile(
    MODEL_TRAINING_COHORT_PATH
).metadata.num_rows != EXPECTED_MODEL_TRAIN_ROWS:
    raise RuntimeError(
        "Frozen model training cohort row count differs."
    )

if pq.ParquetFile(
    MODEL_EVALUATION_COHORT_PATH
).metadata.num_rows != EXPECTED_MODEL_EVAL_ROWS:
    raise RuntimeError(
        "Frozen model evaluation cohort row count differs."
    )

if pq.ParquetFile(
    RNG_MANIFEST_PATH
).metadata.num_rows != EXPECTED_RNG_ROWS:
    raise RuntimeError(
        "Frozen RNG manifest row count differs."
    )

condition_plan = pd.read_csv(
    CONDITION_PLAN_PATH,
    low_memory=False,
)

required_plan_columns = {
    "ConditionOrder",
    "ConditionID",
    "NoisePercent",
    "RepetitionSeed",
}

if not required_plan_columns.issubset(
    condition_plan.columns
):
    raise RuntimeError(
        "Condition plan is missing required columns:\n"
        + "\n".join(
            sorted(
                required_plan_columns
                - set(condition_plan.columns)
            )
        )
    )

condition_plan = (
    condition_plan.sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

expected_grid = [
    (
        seed,
        noise,
        f"noise_{noise:02d}__seed_{seed:02d}",
    )
    for seed in REPETITION_SEEDS
    for noise in NOISE_LEVELS
]

actual_grid = [
    (
        int(row.RepetitionSeed),
        int(row.NoisePercent),
        str(row.ConditionID),
    )
    for row in condition_plan.itertuples(
        index=False
    )
]

if (
    len(condition_plan)
    != EXPECTED_CONDITIONS
    or actual_grid != expected_grid
):
    raise RuntimeError(
        "Frozen Project 24 condition grid/order differs."
    )

if (
    condition_plan[
        "ConditionID"
    ].duplicated().any()
    or condition_plan.duplicated(
        [
            "NoisePercent",
            "RepetitionSeed",
        ]
    ).any()
):
    raise RuntimeError(
        "Frozen Project 24 condition plan contains duplicate coordinates."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY ALL SIX WORKER CHECKPOINTS AND SCIENTIFIC SHARD FREEZES
# --------------------------------------------------------------------------------------------------

print(
    "\nVerifying six worker checkpoints and exact scientific worker raw roots."
)

worker_records = []
worker_seed_union = []
worker_checkpoint_hashes_actual = {}
worker_checkpoint_hash_refreshes = 0
shared_equivalence_hashes = set()

for tag, expected in EXPECTED_WORKERS.items():
    checkpoint_path = (
        WORKER_CHECKPOINT_PATHS[tag]
    )

    actual_checkpoint_sha = sha256_file(
        checkpoint_path
    )

    worker_checkpoint_hashes_actual[
        tag
    ] = actual_checkpoint_sha

    reported_sha_still_current = bool(
        actual_checkpoint_sha
        == expected[
            "ReportedCheckpointSHA256"
        ]
    )

    if not reported_sha_still_current:
        worker_checkpoint_hash_refreshes += 1

    checkpoint = load_json(
        checkpoint_path
    )

    if (
        checkpoint.get("Project")
        != PROJECT_NAME
        or checkpoint.get("ProjectSlug")
        != PROJECT_SLUG
        or int(
            checkpoint.get(
                "ProjectNumber",
                -1,
            )
        ) != PROJECT_NUMBER
    ):
        raise RuntimeError(
            f"Worker {tag} project identity differs."
        )

    if checkpoint.get(
        "Status"
    ) != expected[
        "Status"
    ]:
        raise RuntimeError(
            f"Worker {tag} PASS status differs."
        )

    assigned_seeds = [
        int(value)
        for value in checkpoint.get(
            "AssignedSeeds",
            [],
        )
    ]

    if assigned_seeds != expected[
        "Seeds"
    ]:
        raise RuntimeError(
            f"Worker {tag} assigned seeds differ: "
            f"{assigned_seeds}"
        )

    worker_seed_union.extend(
        assigned_seeds
    )

    scalar_expectations = {
        "AssignedConditions":
            EXPECTED_WORKER_CONDITIONS,

        "CompletedConditions":
            EXPECTED_WORKER_CONDITIONS,

        "MLFits":
            EXPECTED_WORKER_MODEL_FITS,

        "RankingRows":
            EXPECTED_WORKER_RANKING_ROWS,

        "BuildMetricRows":
            EXPECTED_WORKER_BUILD_METRIC_ROWS,

        "ProjectRunRows":
            EXPECTED_WORKER_PROJECT_RUN_ROWS,

        "ConditionAuditRows":
            EXPECTED_WORKER_CONDITION_AUDIT_ROWS,

        "TrainingMedianRows":
            EXPECTED_WORKER_TRAINING_MEDIAN_ROWS,

        "WorkerRawFiles":
            EXPECTED_WORKER_RAW_FILES,

        "WorkerRawBytes":
            expected["RawBytes"],

        "FailedValidationChecks":
            0,

        "BaselineInvarianceFailures":
            0,

        "MasterStep5AWriteGuardFailures":
            0,
    }

    for key, expected_value in scalar_expectations.items():
        actual_value = int(
            checkpoint.get(
                key,
                -1,
            )
        )

        if actual_value != int(
            expected_value
        ):
            raise RuntimeError(
                f"Worker {tag} {key} differs. "
                f"Expected={expected_value}; "
                f"actual={actual_value}"
            )

    if checkpoint.get(
        "WorkerRawRootSHA256"
    ) != expected[
        "RawRootSHA256"
    ]:
        raise RuntimeError(
            f"Worker {tag} frozen raw-root SHA-256 differs."
        )

    if checkpoint.get(
        "SourceRootSHA256"
    ) != EXPECTED_SOURCE_ROOT_SHA256:
        raise RuntimeError(
            f"Worker {tag} source root differs."
        )

    if checkpoint.get(
        "RegistrySHA256"
    ) != EXPECTED_REGISTRY_SHA256:
        raise RuntimeError(
            f"Worker {tag} registry SHA differs."
        )

    if not bool(
        checkpoint.get(
            "WorkerShardComplete",
            False,
        )
    ):
        raise RuntimeError(
            f"Worker {tag} is not marked complete."
        )

    if bool(
        checkpoint.get(
            "OfficialStep5AComplete",
            True,
        )
    ):
        raise RuntimeError(
            f"Worker {tag} incorrectly marks official Step 5A complete."
        )

    worker_report_path = Path(
        str(
            checkpoint.get(
                "WorkerReportPath",
                "",
            )
        )
    )

    if (
        not worker_report_path.is_file()
        or sha256_file(
            worker_report_path
        )
        != str(
            checkpoint.get(
                "WorkerReportSHA256",
                "",
            )
        )
    ):
        raise RuntimeError(
            f"Worker {tag} report linkage failed."
        )

    worker_report = load_json(
        worker_report_path
    )

    if (
        worker_report.get("Status")
        != expected["Status"]
        or worker_report.get(
            "WorkerRawRootSHA256"
        )
        != expected["RawRootSHA256"]
        or int(
            worker_report.get(
                "WorkerRawBytes",
                -1,
            )
        )
        != expected["RawBytes"]
    ):
        raise RuntimeError(
            f"Worker {tag} report scientific freeze differs."
        )

    worker_output_manifest_failures = verify_manifest_entries(
        checkpoint.get(
            "WorkerOutputManifest",
            [],
        ),
        f"worker {tag}",
    )

    if worker_output_manifest_failures:
        display(
            pd.DataFrame(
                worker_output_manifest_failures
            )
        )

        raise RuntimeError(
            f"Worker {tag} private output manifest failed."
        )

    shared_equivalence_path = Path(
        str(
            checkpoint.get(
                "SharedSmokeEquivalencePath",
                "",
            )
        )
    )

    if shared_equivalence_path != ACCELERATED_EQUIVALENCE_PATH:
        raise RuntimeError(
            f"Worker {tag} shared equivalence path differs."
        )

    shared_equivalence_sha = str(
        checkpoint.get(
            "SharedSmokeEquivalenceSHA256",
            "",
        )
    )

    if (
        not ACCELERATED_EQUIVALENCE_PATH.is_file()
        or sha256_file(
            ACCELERATED_EQUIVALENCE_PATH
        )
        != shared_equivalence_sha
    ):
        raise RuntimeError(
            f"Worker {tag} shared equivalence linkage differs."
        )

    shared_equivalence_hashes.add(
        shared_equivalence_sha
    )

    # Independently recompute this worker's exact 360-file scientific raw root from the global raw root.
    worker_condition_keys = set(
        condition_plan.loc[
            condition_plan[
                "RepetitionSeed"
            ].isin(
                assigned_seeds
            ),
            "ConditionID",
        ].astype(str)
    )

    worker_raw_records = []

    for condition_key in sorted(
        worker_condition_keys
    ):
        condition_dir = (
            FULL_RAW_RESULT_ROOT
            / condition_key
        )

        condition_manifest = directory_manifest(
            condition_dir
        )

        for item in condition_manifest.itertuples(
            index=False
        ):
            worker_raw_records.append({
                "RelativePath":
                    f"{condition_key}/{item.RelativePath}",

                "Bytes":
                    int(item.Bytes),

                "SHA256":
                    str(item.SHA256),
            })

    worker_raw_manifest = pd.DataFrame(
        worker_raw_records,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    ).sort_values(
        "RelativePath",
        kind="mergesort",
    ).reset_index(
        drop=True
    )

    worker_raw_files_recomputed = int(
        len(
            worker_raw_manifest
        )
    )

    worker_raw_bytes_recomputed = int(
        worker_raw_manifest[
            "Bytes"
        ].sum()
    )

    worker_raw_root_recomputed = directory_root_hash(
        worker_raw_manifest
    )

    if (
        worker_raw_files_recomputed
        != EXPECTED_WORKER_RAW_FILES
        or worker_raw_bytes_recomputed
        != expected["RawBytes"]
        or worker_raw_root_recomputed
        != expected["RawRootSHA256"]
    ):
        raise RuntimeError(
            f"Worker {tag} independent raw-root recomputation differs."
        )

    worker_records.append({
        "WorkerTag":
            tag,

        "AssignedSeedsJSON":
            json.dumps(
                assigned_seeds
            ),

        "Status":
            checkpoint.get(
                "Status"
            ),

        "ReportedCheckpointSHA256":
            expected[
                "ReportedCheckpointSHA256"
            ],

        "ActualCheckpointSHA256":
            actual_checkpoint_sha,

        "ReportedCheckpointSHAStillCurrent":
            reported_sha_still_current,

        "CheckpointMetadataRefreshed":
            not reported_sha_still_current,

        "WorkerRawFiles":
            worker_raw_files_recomputed,

        "WorkerRawBytes":
            worker_raw_bytes_recomputed,

        "ExpectedWorkerRawRootSHA256":
            expected[
                "RawRootSHA256"
            ],

        "ActualWorkerRawRootSHA256":
            worker_raw_root_recomputed,

        "WorkerReportPath":
            str(
                worker_report_path
            ),

        "WorkerReportSHA256":
            sha256_file(
                worker_report_path
            ),

        "SharedSmokeEquivalenceSHA256":
            shared_equivalence_sha,

        "Pass":
            True,
    })

worker_audit = pd.DataFrame(
    worker_records
)

if len(
    shared_equivalence_hashes
) != 1:
    raise RuntimeError(
        "The six workers do not reference one identical accelerated-equivalence freeze."
    )

if (
    sorted(
        worker_seed_union
    ) != REPETITION_SEEDS
    or len(
        worker_seed_union
    ) != len(
        set(
            worker_seed_union
        )
    )
):
    raise RuntimeError(
        "Worker seed shards are not disjoint exact coverage of seeds 1–30."
    )

print(
    "Workers verified:",
    len(worker_audit),
    "/ 6",
)

print(
    "Worker checkpoint metadata refreshes:",
    worker_checkpoint_hash_refreshes,
)

if worker_checkpoint_hash_refreshes:
    print(
        "NOTE: refreshed worker checkpoint SHA values are acceptable only because "
        "their exact scientific raw roots, output manifests, report linkage, seed shards, "
        "and all 270 raw conditions are independently revalidated below."
    )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY SHARED ACCELERATED SMOKE-EQUIVALENCE FREEZE
# --------------------------------------------------------------------------------------------------

equivalence = pd.read_csv(
    ACCELERATED_EQUIVALENCE_PATH,
    low_memory=False,
)

if (
    len(equivalence) != 2
    or "ConditionKey" not in equivalence.columns
    or "Pass" not in equivalence.columns
    or sorted(
        equivalence[
            "ConditionKey"
        ].astype(str).tolist()
    ) != sorted(
        SMOKE_EQUIVALENCE_KEYS
    )
    or not equivalence[
        "Pass"
    ].map(
        as_bool
    ).all()
):
    raise RuntimeError(
        "Frozen accelerated smoke-equivalence audit differs."
    )


# --------------------------------------------------------------------------------------------------
# 8. INDEPENDENTLY REVALIDATE ALL 270 RAW CONDITIONS AND REBUILD MASTER AGGREGATES
# --------------------------------------------------------------------------------------------------

print(
    "\nIndependently revalidating all 270 raw condition directories."
)

inventory_records = []
condition_audit_frames = []
project_run_frames = []
build_metric_frames = []
model_fit_frames = []
raw_manifest_records = []
baseline_fingerprint_records = []
condition_validation_errors = []

validation_started = time.perf_counter()

for index, plan_row in enumerate(
    condition_plan.itertuples(
        index=False
    ),
    start=1,
):
    condition_key = str(
        plan_row.ConditionID
    )

    condition_dir = (
        FULL_RAW_RESULT_ROOT
        / condition_key
    )

    validated, error = validate_condition_read_only(
        condition_dir,
        plan_row,
    )

    if validated is None:
        condition_validation_errors.append({
            "ConditionKey":
                condition_key,

            "Error":
                error,
        })
        continue

    inventory_records.append(
        validated[
            "Inventory"
        ]
    )

    condition_audit_frames.append(
        validated[
            "ConditionAudit"
        ]
    )

    project_run_frames.append(
        validated[
            "ProjectRuns"
        ]
    )

    build_metric_frames.append(
        validated[
            "BuildMetrics"
        ]
    )

    model_fit_frames.append(
        validated[
            "ModelFits"
        ]
    )

    fingerprints = validated[
        "Fingerprints"
    ]

    for technique in INVARIANT_BASELINES:
        fingerprint = fingerprints[
            technique
        ]

        baseline_fingerprint_records.append({
            "ConditionKey":
                condition_key,

            "NoisePercent":
                int(
                    plan_row.NoisePercent
                ),

            "RepetitionSeed":
                int(
                    plan_row.RepetitionSeed
                ),

            "Technique":
                technique,

            "Rows":
                int(
                    fingerprint[
                        "Rows"
                    ]
                ),

            "KeySHA256":
                str(
                    fingerprint[
                        "KeySHA256"
                    ]
                ),

            "ScoreSHA256":
                str(
                    fingerprint[
                        "ScoreSHA256"
                    ]
                ),

            "RankSHA256":
                str(
                    fingerprint[
                        "RankSHA256"
                    ]
                ),
        })

    condition_manifest = validated[
        "Manifest"
    ]

    for item in condition_manifest.itertuples(
        index=False
    ):
        raw_manifest_records.append({
            "RelativePath":
                f"{condition_key}/{item.RelativePath}",

            "Bytes":
                int(
                    item.Bytes
                ),

            "SHA256":
                str(
                    item.SHA256
                ),
        })

    if (
        index % 25 == 0
        or index == EXPECTED_CONDITIONS
    ):
        print(
            f"  Conditions revalidated: {index}/{EXPECTED_CONDITIONS} "
            f"| elapsed seconds: "
            f"{round(time.perf_counter() - validation_started, 2)}"
        )

if condition_validation_errors:
    print(
        "\nCondition validation failures:"
    )

    display(
        pd.DataFrame(
            condition_validation_errors
        )
    )

    raise RuntimeError(
        "One or more Project 24 raw condition directories failed independent master validation."
    )

condition_inventory = (
    pd.DataFrame(
        inventory_records
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

combined_condition_audit = pd.concat(
    condition_audit_frames,
    ignore_index=True,
)

combined_project_runs = pd.concat(
    project_run_frames,
    ignore_index=True,
)

combined_build_metrics = pd.concat(
    build_metric_frames,
    ignore_index=True,
)

combined_model_fits = pd.concat(
    model_fit_frames,
    ignore_index=True,
)

baseline_fingerprints = pd.DataFrame(
    baseline_fingerprint_records
)

raw_manifest = (
    pd.DataFrame(
        raw_manifest_records,
        columns=DIRECTORY_MANIFEST_COLUMNS,
    )
    .sort_values(
        "RelativePath",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

raw_files = int(
    len(
        raw_manifest
    )
)

raw_bytes = int(
    raw_manifest[
        "Bytes"
    ].sum()
)

raw_root_sha256 = directory_root_hash(
    raw_manifest
)


# --------------------------------------------------------------------------------------------------
# 9. MASTER BASELINE INVARIANCE + GLOBAL RAW-ROOT CROSS-CHECK
# --------------------------------------------------------------------------------------------------

baseline_invariance_records = []

for repetition_seed in REPETITION_SEEDS:
    for technique in INVARIANT_BASELINES:
        rows = baseline_fingerprints.loc[
            baseline_fingerprints[
                "RepetitionSeed"
            ].eq(
                repetition_seed
            )
            & baseline_fingerprints[
                "Technique"
            ].eq(
                technique
            )
        ]

        key_variants = int(
            rows[
                "KeySHA256"
            ].nunique()
        )

        score_variants = int(
            rows[
                "ScoreSHA256"
            ].nunique()
        )

        rank_variants = int(
            rows[
                "RankSHA256"
            ].nunique()
        )

        passed = bool(
            len(rows)
            == len(NOISE_LEVELS)
            and key_variants == 1
            and score_variants == 1
            and rank_variants == 1
        )

        baseline_invariance_records.append({
            "RepetitionSeed":
                repetition_seed,

            "Technique":
                technique,

            "Conditions":
                int(
                    len(rows)
                ),

            "KeyVariantsAcrossNoise":
                key_variants,

            "ScoreVariantsAcrossNoise":
                score_variants,

            "RankVariantsAcrossNoise":
                rank_variants,

            "Pass":
                passed,
        })

baseline_invariance = pd.DataFrame(
    baseline_invariance_records
)

baseline_invariance_failures = int(
    (
        ~baseline_invariance[
            "Pass"
        ]
    ).sum()
)

qtf_global_key_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints[
            "Technique"
        ].eq(
            "QTF-Avg"
        ),
        "KeySHA256",
    ].nunique()
)

qtf_global_score_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints[
            "Technique"
        ].eq(
            "QTF-Avg"
        ),
        "ScoreSHA256",
    ].nunique()
)

qtf_global_rank_variants = int(
    baseline_fingerprints.loc[
        baseline_fingerprints[
            "Technique"
        ].eq(
            "QTF-Avg"
        ),
        "RankSHA256",
    ].nunique()
)

# Completely independent directory walk/hash over the raw root.
raw_manifest_direct = directory_manifest(
    FULL_RAW_RESULT_ROOT
)

raw_root_sha256_direct = directory_root_hash(
    raw_manifest_direct
)

raw_manifest_keyed = raw_manifest.set_index(
    "RelativePath"
).sort_index()

raw_manifest_direct_keyed = raw_manifest_direct.set_index(
    "RelativePath"
).sort_index()

raw_manifest_exact_match = bool(
    raw_manifest_keyed.equals(
        raw_manifest_direct_keyed
    )
)


# --------------------------------------------------------------------------------------------------
# 10. MASTER VALIDATION GATE
# --------------------------------------------------------------------------------------------------

validation = []

add_check(
    validation,
    "Workers verified",
    6,
    len(worker_audit),
    len(worker_audit) == 6,
)

add_check(
    validation,
    "Worker seeds cover 1–30 exactly once",
    REPETITION_SEEDS,
    sorted(worker_seed_union),
    (
        sorted(worker_seed_union)
        == REPETITION_SEEDS
        and len(worker_seed_union)
        == len(set(worker_seed_union))
    ),
)

add_check(
    validation,
    "Worker scientific raw-root mismatches",
    0,
    int(
        (
            worker_audit[
                "ExpectedWorkerRawRootSHA256"
            ]
            != worker_audit[
                "ActualWorkerRawRootSHA256"
            ]
        ).sum()
    ),
    bool(
        (
            worker_audit[
                "ExpectedWorkerRawRootSHA256"
            ]
            == worker_audit[
                "ActualWorkerRawRootSHA256"
            ]
        ).all()
    ),
)

add_check(
    validation,
    "Conditions",
    EXPECTED_CONDITIONS,
    len(condition_inventory),
    len(condition_inventory)
    == EXPECTED_CONDITIONS,
)

add_check(
    validation,
    "Duplicate condition keys",
    0,
    int(
        condition_inventory[
            "ConditionKey"
        ].duplicated(
            keep=False
        ).sum()
    ),
    not condition_inventory[
        "ConditionKey"
    ].duplicated(
        keep=False
    ).any(),
)

add_check(
    validation,
    "Files per condition",
    EXPECTED_FILES_PER_CONDITION,
    sorted(
        condition_inventory[
            "Files"
        ].unique().tolist()
    ),
    condition_inventory[
        "Files"
    ].eq(
        EXPECTED_FILES_PER_CONDITION
    ).all(),
)

add_check(
    validation,
    "Raw files",
    EXPECTED_RAW_FILES,
    raw_files,
    raw_files == EXPECTED_RAW_FILES,
)

add_check(
    validation,
    "Raw bytes",
    EXPECTED_RAW_BYTES,
    raw_bytes,
    raw_bytes == EXPECTED_RAW_BYTES,
)

add_check(
    validation,
    "Direct raw-root file count",
    EXPECTED_RAW_FILES,
    len(raw_manifest_direct),
    len(raw_manifest_direct)
    == EXPECTED_RAW_FILES,
)

add_check(
    validation,
    "Direct raw-root bytes",
    EXPECTED_RAW_BYTES,
    int(
        raw_manifest_direct[
            "Bytes"
        ].sum()
    ),
    int(
        raw_manifest_direct[
            "Bytes"
        ].sum()
    ) == EXPECTED_RAW_BYTES,
)

add_check(
    validation,
    "Collected/direct raw manifests identical",
    True,
    raw_manifest_exact_match,
    raw_manifest_exact_match,
)

add_check(
    validation,
    "Collected/direct raw-root SHA identical",
    raw_root_sha256,
    raw_root_sha256_direct,
    raw_root_sha256
    == raw_root_sha256_direct,
)

add_check(
    validation,
    "Ranking rows",
    EXPECTED_TOTAL_RANKING_ROWS,
    int(
        condition_inventory[
            "RankingRows"
        ].sum()
    ),
    int(
        condition_inventory[
            "RankingRows"
        ].sum()
    ) == EXPECTED_TOTAL_RANKING_ROWS,
)

add_check(
    validation,
    "Build-metric rows",
    EXPECTED_TOTAL_BUILD_METRIC_ROWS,
    len(combined_build_metrics),
    len(combined_build_metrics)
    == EXPECTED_TOTAL_BUILD_METRIC_ROWS,
)

add_check(
    validation,
    "Project-run rows",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
    len(combined_project_runs),
    len(combined_project_runs)
    == EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)

add_check(
    validation,
    "Model fits",
    EXPECTED_TOTAL_MODEL_FITS,
    len(combined_model_fits),
    len(combined_model_fits)
    == EXPECTED_TOTAL_MODEL_FITS,
)

add_check(
    validation,
    "Condition-audit rows",
    EXPECTED_TOTAL_CONDITION_AUDIT_ROWS,
    len(combined_condition_audit),
    len(combined_condition_audit)
    == EXPECTED_TOTAL_CONDITION_AUDIT_ROWS,
)

add_check(
    validation,
    "Training-median rows",
    EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS,
    int(
        condition_inventory[
            "TrainingMedianRows"
        ].sum()
    ),
    int(
        condition_inventory[
            "TrainingMedianRows"
        ].sum()
    ) == EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS,
)

add_check(
    validation,
    "Zero-noise conditions",
    30,
    int(
        condition_inventory[
            "NoisePercent"
        ].eq(0).sum()
    ),
    int(
        condition_inventory[
            "NoisePercent"
        ].eq(0).sum()
    ) == 30,
)

zero_audit = combined_condition_audit.loc[
    pd.to_numeric(
        combined_condition_audit[
            "NoisePercent"
        ],
        errors="raise",
    ).eq(0)
]

positive_audit = combined_condition_audit.loc[
    pd.to_numeric(
        combined_condition_audit[
            "NoisePercent"
        ],
        errors="raise",
    ).gt(0)
]

add_check(
    validation,
    "Zero-noise raw flips",
    0,
    int(
        pd.to_numeric(
            zero_audit[
                "NumberFlipped"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            zero_audit[
                "NumberFlipped"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Zero-noise model-label changes",
    0,
    int(
        pd.to_numeric(
            zero_audit[
                "ModelLabelChanges"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            zero_audit[
                "ModelLabelChanges"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Zero-noise dependent REC changes",
    0,
    int(
        pd.to_numeric(
            zero_audit[
                "DependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            zero_audit[
                "DependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Positive-noise raw-change violations",
    0,
    int(
        pd.to_numeric(
            positive_audit[
                "NumberFlipped"
            ],
            errors="raise",
        ).le(0).sum()
    ),
    not pd.to_numeric(
        positive_audit[
            "NumberFlipped"
        ],
        errors="raise",
    ).le(0).any(),
)

add_check(
    validation,
    "Positive-noise model-change violations",
    0,
    int(
        pd.to_numeric(
            positive_audit[
                "ModelLabelChanges"
            ],
            errors="raise",
        ).le(0).sum()
    ),
    not pd.to_numeric(
        positive_audit[
            "ModelLabelChanges"
        ],
        errors="raise",
    ).le(0).any(),
)

add_check(
    validation,
    "Positive-noise dependent-REC violations",
    0,
    int(
        pd.to_numeric(
            positive_audit[
                "DependentRECChanges"
            ],
            errors="raise",
        ).le(0).sum()
    ),
    not pd.to_numeric(
        positive_audit[
            "DependentRECChanges"
        ],
        errors="raise",
    ).le(0).any(),
)

add_check(
    validation,
    "Independent REC changes",
    0,
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentRECChanges"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Independent reconstruction mismatches",
    0,
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentReconstructionMismatches"
            ],
            errors="raise",
        ).sum()
    ),
    int(
        pd.to_numeric(
            combined_condition_audit[
                "IndependentReconstructionMismatches"
            ],
            errors="raise",
        ).sum()
    ) == 0,
)

noise_plan_hash_mismatches = 0

for expected_column, actual_column in [
    (
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
    ),
    (
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
    ),
    (
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
    ),
]:
    noise_plan_hash_mismatches += int(
        combined_condition_audit[
            expected_column
        ].astype(str).ne(
            combined_condition_audit[
                actual_column
            ].astype(str)
        ).sum()
    )

add_check(
    validation,
    "Noise-plan hash mismatches",
    0,
    noise_plan_hash_mismatches,
    noise_plan_hash_mismatches == 0,
)

add_check(
    validation,
    "Baseline invariance failures",
    0,
    baseline_invariance_failures,
    baseline_invariance_failures == 0,
)

add_check(
    validation,
    "QTF global key variants",
    1,
    qtf_global_key_variants,
    qtf_global_key_variants == 1,
)

add_check(
    validation,
    "QTF global score variants",
    1,
    qtf_global_score_variants,
    qtf_global_score_variants == 1,
)

add_check(
    validation,
    "QTF global rank variants",
    1,
    qtf_global_rank_variants,
    qtf_global_rank_variants == 1,
)

model_fit_status_failures = int(
    (
        ~combined_model_fits[
            "Status"
        ].astype(str).eq(
            "PASS_MODEL_FIT"
        )
    ).sum()
)

add_check(
    validation,
    "Model-fit status failures",
    0,
    model_fit_status_failures,
    model_fit_status_failures == 0,
)

for frame, columns, label in [
    (
        combined_project_runs,
        [
            "MeanAPFDc",
            "MedianAPFDc",
            "MeanAPFD",
            "MedianAPFD",
        ],
        "Project metrics",
    ),
    (
        combined_build_metrics,
        [
            "APFDc",
            "APFD",
        ],
        "Build metrics",
    ),
]:
    values = frame[
        columns
    ].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(float)

    bad = int(
        (
            ~np.isfinite(
                values
            )
        ).sum()
        + (
            (
                values < 0
            )
            | (
                values > 1
            )
        ).sum()
    )

    add_check(
        validation,
        f"{label} non-finite/out-of-range values",
        0,
        bad,
        bad == 0,
    )

add_check(
    validation,
    "Shared smoke-equivalence rows",
    2,
    len(equivalence),
    len(equivalence) == 2,
)

add_check(
    validation,
    "Shared smoke-equivalence failures",
    0,
    int(
        (
            ~equivalence[
                "Pass"
            ].map(
                as_bool
            )
        ).sum()
    ),
    equivalence[
        "Pass"
    ].map(
        as_bool
    ).all(),
)

add_check(
    validation,
    "Registry SHA unchanged",
    EXPECTED_REGISTRY_SHA256,
    sha256_file(
        REGISTRY_PATH
    ),
    sha256_file(
        REGISTRY_PATH
    ) == EXPECTED_REGISTRY_SHA256,
)

add_check(
    validation,
    "Source root unchanged",
    EXPECTED_SOURCE_ROOT_SHA256,
    source_root_before,
    source_root_before
    == EXPECTED_SOURCE_ROOT_SHA256,
)

for cohort_path, expected_sha in cohort_hashes_before.items():
    actual_sha = sha256_file(
        Path(
            cohort_path
        )
    )

    add_check(
        validation,
        f"Frozen cohort unchanged: {Path(cohort_path).name}",
        expected_sha,
        actual_sha,
        actual_sha == expected_sha,
    )

add_check(
    validation,
    "Registry Project 24 rows",
    0,
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        registry_project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ) == 0,
)

add_check(
    validation,
    "Master finalizer model fits",
    0,
    0,
    True,
)

add_check(
    validation,
    "Master finalizer condition executions",
    0,
    0,
    True,
)

step5a_validation = pd.DataFrame(
    validation
)

failed_validation = step5a_validation.loc[
    ~step5a_validation[
        "Pass"
    ]
]

print(
    "\nProject 24 master Step 5A validation:"
)

display(
    step5a_validation
)

if not failed_validation.empty:
    print(
        "\nFAILED MASTER CHECKS:"
    )

    display(
        failed_validation
    )

    raise RuntimeError(
        "PROJECT 24 STEP 5A MASTER FINALIZATION VALIDATION FAILED. "
        "No official Step 5A freeze was written."
    )


# --------------------------------------------------------------------------------------------------
# 11. WRITE OFFICIAL STEP 5A AGGREGATES ONLY AFTER ALL CHECKS PASS
# --------------------------------------------------------------------------------------------------

FULL_EXPERIMENT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_csv(
    CONDITION_INVENTORY_PATH,
    condition_inventory,
)

atomic_csv(
    RAW_MANIFEST_PATH,
    raw_manifest,
)

atomic_csv(
    BASELINE_INVARIANCE_PATH,
    baseline_invariance,
)

atomic_csv(
    COMBINED_CONDITION_AUDIT_PATH,
    combined_condition_audit,
)

atomic_csv(
    COMBINED_PROJECT_RUNS_PATH,
    combined_project_runs,
)

atomic_csv(
    COMBINED_BUILD_METRICS_PATH,
    combined_build_metrics,
)

atomic_csv(
    COMBINED_MODEL_FITS_PATH,
    combined_model_fits,
)

atomic_csv(
    WORKER_CHECKPOINT_AUDIT_PATH,
    worker_audit,
)

atomic_csv(
    STEP5A_VALIDATION_PATH,
    step5a_validation,
)

aggregate_output_paths = [
    CONDITION_INVENTORY_PATH,
    RAW_MANIFEST_PATH,
    BASELINE_INVARIANCE_PATH,
    COMBINED_CONDITION_AUDIT_PATH,
    COMBINED_PROJECT_RUNS_PATH,
    COMBINED_BUILD_METRICS_PATH,
    COMBINED_MODEL_FITS_PATH,
    WORKER_CHECKPOINT_AUDIT_PATH,
    STEP5A_VALIDATION_PATH,
    ACCELERATED_EQUIVALENCE_PATH,
]

aggregate_output_manifest = [
    {
        "Path":
            str(path),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in aggregate_output_paths
]

completed_at_utc = datetime.now(
    timezone.utc
).isoformat()

master_execution_seconds = (
    time.perf_counter()
    - master_started
)

worker_checkpoint_records = [
    {
        "WorkerTag":
            row.WorkerTag,

        "AssignedSeedsJSON":
            row.AssignedSeedsJSON,

        "Status":
            row.Status,

        "ActualCheckpointSHA256":
            row.ActualCheckpointSHA256,

        "ReportedCheckpointSHA256":
            row.ReportedCheckpointSHA256,

        "CheckpointMetadataRefreshed":
            bool(
                row.CheckpointMetadataRefreshed
            ),

        "WorkerRawFiles":
            int(
                row.WorkerRawFiles
            ),

        "WorkerRawBytes":
            int(
                row.WorkerRawBytes
            ),

        "WorkerRawRootSHA256":
            row.ActualWorkerRawRootSHA256,

        "WorkerReportSHA256":
            row.WorkerReportSHA256,

        "SharedSmokeEquivalenceSHA256":
            row.SharedSmokeEquivalenceSHA256,
    }
    for row in worker_audit.itertuples(
        index=False
    )
]

report_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_STATUS,

    "Implementation":
        MASTER_FINALIZER_IMPLEMENTATION,

    "CompletedAtUTC":
        completed_at_utc,

    "ExecutionMode":
        "PARALLEL_SIX_WORKERS_THEN_ZERO_FIT_MASTER_FINALIZATION",

    "AcceleratedEngineVersion":
        ACCELERATED_ENGINE_VERSION,

    "SelectionCheckpointSHA256":
        EXPECTED_SELECTION_CHECKPOINT_SHA256,

    "RECCheckpointSHA256":
        EXPECTED_REC_CHECKPOINT_SHA256,

    "NoisePlanCheckpointSHA256":
        EXPECTED_NOISE_PLAN_CHECKPOINT_SHA256,

    "RuntimeCheckpointSHA256":
        EXPECTED_RUNTIME_CHECKPOINT_SHA256,

    "SmokeCheckpointSHA256":
        EXPECTED_SMOKE_CHECKPOINT_SHA256,

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA256,

    "RegistrySHA256":
        EXPECTED_REGISTRY_SHA256,

    "WorkerCheckpoints":
        worker_checkpoint_records,

    "ActualWorkerCheckpointSHA256":
        worker_checkpoint_hashes_actual,

    "WorkerCheckpointMetadataRefreshes":
        worker_checkpoint_hash_refreshes,

    "WorkersVerified":
        len(worker_audit),

    "SeedsCoveredExactlyOnce":
        True,

    "Conditions":
        len(condition_inventory),

    "NoiseLevels":
        NOISE_LEVELS,

    "RepetitionSeeds":
        REPETITION_SEEDS,

    "MLFits":
        len(combined_model_fits),

    "RankingRows":
        int(
            condition_inventory[
                "RankingRows"
            ].sum()
        ),

    "BuildMetricRows":
        len(combined_build_metrics),

    "ProjectRunRows":
        len(combined_project_runs),

    "ConditionAuditRows":
        len(combined_condition_audit),

    "TrainingMedianRows":
        int(
            condition_inventory[
                "TrainingMedianRows"
            ].sum()
        ),

    "RawFiles":
        raw_files,

    "RawBytes":
        raw_bytes,

    "RawRootSHA256":
        raw_root_sha256,

    "BaselineInvarianceFailures":
        baseline_invariance_failures,

    "QTFGlobalKeyVariants":
        qtf_global_key_variants,

    "QTFGlobalScoreVariants":
        qtf_global_score_variants,

    "QTFGlobalRankVariants":
        qtf_global_rank_variants,

    "PrimaryMetric":
        "APFDc",

    "SecondaryMetric":
        "APFD",

    "SharedSmokeEquivalenceSHA256":
        sha256_file(
            ACCELERATED_EQUIVALENCE_PATH
        ),

    "ValidationChecks":
        len(step5a_validation),

    "FailedValidationChecks":
        len(failed_validation),

    "MasterFinalizerModelFits":
        0,

    "MasterFinalizerConditionExecutions":
        0,

    "SourceRestoredInThisRuntime":
        source_restored,

    "MasterExecutionSeconds":
        float(
            master_execution_seconds
        ),

    "AggregateOutputManifest":
        aggregate_output_manifest,

    "RegistryModified":
        False,

    "Projects1To23Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,
}

atomic_json(
    STEP5A_REPORT_PATH,
    report_payload,
)

checkpoint_payload = {
    **report_payload,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_24_FULL_270_CONDITION_STEP5A_FREEZE",

    "Full270ConditionExperimentComplete":
        True,

    "RawResultRootFrozen":
        True,

    "RawResultRootReadOnlyDuringMasterFinalization":
        True,

    "AllSixWorkerScientificRawRootsVerified":
        True,

    "All270ConditionsIndependentlyRevalidated":
        True,

    "EvaluationCohortImmutable":
        True,

    "ReadyForStep5B":
        True,

    "NextRequiredStep":
        "PROJECT 24 STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION",
}

atomic_json(
    STEP5A_CHECKPOINT_PATH,
    checkpoint_payload,
)

step5a_checkpoint_sha256 = sha256_file(
    STEP5A_CHECKPOINT_PATH
)

status_payload = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5A_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "Conditions":
        EXPECTED_CONDITIONS,

    "MLFits":
        EXPECTED_TOTAL_MODEL_FITS,

    "RawFiles":
        EXPECTED_RAW_FILES,

    "RawBytes":
        EXPECTED_RAW_BYTES,

    "RawRootSHA256":
        raw_root_sha256,

    "Checkpoint":
        str(
            STEP5A_CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        step5a_checkpoint_sha256,

    "MasterFinalizerModelFits":
        0,

    "MasterFinalizerConditionExecutions":
        0,

    "RegistryModified":
        False,

    "ReadyForStep5B":
        True,

    "NextRequiredStep":
        "PROJECT 24 STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION",
}

atomic_json(
    STEP5A_STATUS_PATH,
    status_payload,
)

atomic_json(
    RUN_PROGRESS_PATH,
    {
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "Status":
            STEP5A_STATUS,

        "ConditionsComplete":
            EXPECTED_CONDITIONS,

        "ConditionsTotal":
            EXPECTED_CONDITIONS,

        "WorkerShardsComplete":
            6,

        "WorkerShardsTotal":
            6,

        "OfficialStep5AComplete":
            True,

        "Step5ACheckpointSHA256":
            step5a_checkpoint_sha256,
    },
)


# --------------------------------------------------------------------------------------------------
# 12. FINAL READBACK + RAW/SOURCE/REGISTRY IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    STEP5A_CHECKPOINT_PATH
)

report_readback = load_json(
    STEP5A_REPORT_PATH
)

status_readback = load_json(
    STEP5A_STATUS_PATH
)

for label, payload in [
    (
        "Step 5A checkpoint",
        checkpoint_readback,
    ),
    (
        "Step 5A report",
        report_readback,
    ),
    (
        "Step 5A status",
        status_readback,
    ),
]:
    if payload.get(
        "Status"
    ) != STEP5A_STATUS:
        raise RuntimeError(
            f"{label} readback failed."
        )

if not bool(
    checkpoint_readback.get(
        "ReadyForStep5B",
        False,
    )
):
    raise RuntimeError(
        "Step 5A checkpoint does not authorize Step 5B."
    )

aggregate_manifest_failures = verify_manifest_entries(
    checkpoint_readback.get(
        "AggregateOutputManifest",
        [],
    ),
    "official Step 5A",
)

if aggregate_manifest_failures:
    display(
        pd.DataFrame(
            aggregate_manifest_failures
        )
    )

    raise RuntimeError(
        "Official Step 5A aggregate-output readback failed."
    )

raw_manifest_final = directory_manifest(
    FULL_RAW_RESULT_ROOT
)

if (
    len(raw_manifest_final)
    != EXPECTED_RAW_FILES
    or int(
        raw_manifest_final[
            "Bytes"
        ].sum()
    ) != EXPECTED_RAW_BYTES
    or directory_root_hash(
        raw_manifest_final
    ) != raw_root_sha256
):
    raise RuntimeError(
        "Project 24 raw root changed during official master writes."
    )

if sha256_file(
    REGISTRY_PATH
) != EXPECTED_REGISTRY_SHA256:
    raise RuntimeError(
        "Completion registry changed during master finalization."
    )

final_source_records = []

for row in frozen_source_manifest.itertuples(
    index=False
):
    path = (
        SOURCE_DIR
        / str(
            row.RelativePath
        )
    )

    final_source_records.append({
        "RelativePath":
            str(
                row.RelativePath
            ),

        "SizeBytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    })

if source_root_hash(
    pd.DataFrame(
        final_source_records
    )
) != EXPECTED_SOURCE_ROOT_SHA256:
    raise RuntimeError(
        "Project 24 source changed during master finalization."
    )

for path_string, expected_sha in cohort_hashes_before.items():
    if sha256_file(
        Path(
            path_string
        )
    ) != expected_sha:
        raise RuntimeError(
            "A frozen Project 24 cohort/plan changed during master finalization:\n"
            f"{path_string}"
        )


# --------------------------------------------------------------------------------------------------
# 13. FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 140
)

print(
    "=== PROJECT 24 CELL 9 / STEP 5A RESULT ==="
)

print(
    "=" * 140
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print()

print(
    "Parallel workers:"
)

print(
    "Workers verified:",
    len(worker_audit),
    "/ 6",
)

print(
    "Seeds covered exactly once:",
    sorted(worker_seed_union)
    == REPETITION_SEEDS
    and len(worker_seed_union)
    == len(set(worker_seed_union)),
)

print(
    "Worker checkpoint metadata refreshes:",
    worker_checkpoint_hash_refreshes,
)

print()

print(
    "Full experiment:"
)

print(
    "Conditions:",
    len(condition_inventory),
    "/",
    EXPECTED_CONDITIONS,
)

print(
    "ML fits:",
    len(combined_model_fits),
    "/",
    EXPECTED_TOTAL_MODEL_FITS,
)

print(
    "Ranking rows:",
    int(
        condition_inventory[
            "RankingRows"
        ].sum()
    ),
    "/",
    EXPECTED_TOTAL_RANKING_ROWS,
)

print(
    "Build-metric rows:",
    len(combined_build_metrics),
    "/",
    EXPECTED_TOTAL_BUILD_METRIC_ROWS,
)

print(
    "Project-run rows:",
    len(combined_project_runs),
    "/",
    EXPECTED_TOTAL_PROJECT_RUN_ROWS,
)

print(
    "Condition-audit rows:",
    len(combined_condition_audit),
    "/",
    EXPECTED_TOTAL_CONDITION_AUDIT_ROWS,
)

print(
    "Training-median rows:",
    int(
        condition_inventory[
            "TrainingMedianRows"
        ].sum()
    ),
    "/",
    EXPECTED_TOTAL_TRAINING_MEDIAN_ROWS,
)

print()

print(
    "Raw result freeze:"
)

print(
    "Raw files:",
    raw_files,
    "/",
    EXPECTED_RAW_FILES,
)

print(
    "Raw bytes:",
    raw_bytes,
    "/",
    EXPECTED_RAW_BYTES,
)

print(
    "Raw root SHA-256:",
    raw_root_sha256,
)

print()

print(
    "Baselines and metrics:"
)

print(
    "Baseline invariance failures:",
    baseline_invariance_failures,
)

print(
    "QTF global score variants:",
    qtf_global_score_variants,
)

print(
    "QTF global rank variants:",
    qtf_global_rank_variants,
)

print(
    "Primary / secondary metrics: APFDc / APFD"
)

print()

print(
    "Isolation:"
)

print(
    "Completion registry unchanged:",
    sha256_file(
        REGISTRY_PATH
    )
    == EXPECTED_REGISTRY_SHA256,
)

print(
    "Projects 1–23 modified:",
    0,
)

print(
    "Prior project condition outputs accessed:",
    False,
)

print(
    "Prior project condition outputs modified:",
    False,
)

print(
    "Master-finalizer model fits:",
    0,
)

print(
    "Master-finalizer condition executions:",
    0,
)

print()

print(
    "Validation:"
)

print(
    "Checks:",
    len(step5a_validation),
)

print(
    "Failed checks:",
    len(failed_validation),
)

print()

print(
    "Official Step 5A checkpoint:"
)

print(
    STEP5A_CHECKPOINT_PATH
)

print(
    "Checkpoint SHA-256:",
    step5a_checkpoint_sha256,
)

print()

print(
    "Master finalization seconds:",
    round(
        master_execution_seconds,
        2,
    ),
)

print()

print(
    "Next required step:",
    "PROJECT 24 STEP 5B — RAW REVALIDATION AND COMPACT AGGREGATION",
)

print()

print(
    "STATUS:",
    STEP5A_STATUS,
)

print(
    "=" * 140
)


=== PROJECT 24 CELL 9 / STEP 5A: PARALLEL MASTER FINALIZATION — ZERO FIT ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Completion-registry ProjectNumber-24 rows: 0
Completion-registry exact project-name matches (diagnostic only): 0

Verifying six worker checkpoints and exact scientific worker raw roots.
Workers verified: 6 / 6
Worker checkpoint metadata refreshes: 0

Independently revalidating all 270 raw condition directories.
  Conditions revalidated: 25/270 | elapsed seconds: 6.84
  Conditions revalidated: 50/270 | elapsed seconds: 18.81
  Conditions revalidated: 75/270 | elapsed seconds: 25.06
  Conditions revalidated: 100/270 | elapsed seconds: 32.67
  Conditions revalidated: 125/270 | elapsed seconds: 38.42
  Conditions revalidated: 150/270 | elapsed seconds: 45.31
  Conditions revalidated: 175/270 | elapsed seconds: 51.41
  Conditions revalidated: 200/270 | elapsed seconds: 57.06
  Conditions 

,Check,Expected,Actual,Pass
0,Workers verified,6,6,True
1,Worker seeds cover 1–30 exactly once,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",True
2,Worker scientific raw-root mismatches,0,0,True
3,Conditions,270,270,True
4,Duplicate condition keys,0,0,True
5,Files per condition,8,[8],True
6,Raw files,2160,2160,True
7,Raw bytes,523093565,523093565,True
8,Direct raw-root file count,2160,2160,True
9,Direct raw-root bytes,523093565,523093565,True



=== PROJECT 24 CELL 9 / STEP 5A RESULT ===
Project: SonarSource@sonarqube
Project slug: SonarSource__sonarqube

Parallel workers:
Workers verified: 6 / 6
Seeds covered exactly once: True
Worker checkpoint metadata refreshes: 0

Full experiment:
Conditions: 270 / 270
ML fits: 1080 / 1080
Ranking rows: 35634060 / 35634060
Build-metric rows: 32130 / 32130
Project-run rows: 1890 / 1890
Condition-audit rows: 270 / 270
Training-median rows: 40770 / 40770

Raw result freeze:
Raw files: 2160 / 2160
Raw bytes: 523093565 / 523093565
Raw root SHA-256: 6968c094a564405fe23e02f9a5b1ed01cce130ea000dd387f7a3f6f12b421d09

Baselines and metrics:
Baseline invariance failures: 0
QTF global score variants: 1
QTF global rank variants: 1
Primary / secondary metrics: APFDc / APFD

Isolation:
Completion registry unchanged: True
Projects 1–23 modified: 0
Prior project condition outputs accessed: False
Prior project condition outputs modified: False
Master-finalizer model fits: 0
Master-finalizer condition exec

In [4]:
# ==================================================================================================
# PROJECT 24 — CELL 10 / STEP 5B
# INDEPENDENT RAW REVALIDATION AND COMPACT AGGREGATION
#
# PROJECT:
#   apache@sling
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_24.ipynb.
#
# THIS CELL:
# - verifies the official frozen Project 24 Step 5A checkpoint;
# - independently hashes all 2,160 raw condition files and reproduces the frozen raw-root SHA-256;
# - independently revalidates every condition marker, summary, embedded output manifest, and compact output;
# - physically recounts all 11,374,020 rows in the 270 compressed ranking CSVs;
# - revalidates noise hashes, REC invariance, model-fit status, APFDc/APFD ranges, and baselines;
# - rebuilds compact analysis-ready summaries across all 30 repetition seeds;
# - creates seed-level deltas relative to the same-technique 0%-noise condition;
# - freezes the Project 24 Step 5B checkpoint;
# - does NOT rerun a condition, reconstruct REC, inject noise, or fit a model;
# - does NOT register Project 24;
# - does NOT access prior-project condition outputs.
#
# BASELINE-INVARIANCE CONTRACT:
# - Random and QTF-Avg must be invariant across noise for the same repetition seed;
# - APFDc and APFD are evaluated independently and are never compared against one another.
#
# STATISTICAL CONTRACT:
# - all standard deviations in compact summaries are sample SD across 30 seeds (pandas std, ddof=1).
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import gzip
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd


print("=" * 140)
print("=== PROJECT 24 CELL 10 / STEP 5B: RAW REVALIDATION AND COMPACT AGGREGATION ===")
print("=" * 140)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PROJECT 24 CONTRACT
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

STEP5A_STATUS = "PASS_PROJECT_24_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
CONDITION_STATUS = "PASS_FULL_CONDITION"
STEP5B_STATUS = "PASS_PROJECT_24_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"

STEP5B_CODE_REVISION = (
    "PROJECT_24_STEP5B_V1_INDEPENDENT_RAW_REVALIDATION_AND_COMPACT_AGGREGATION"
)

EXPECTED_STEP5A_SHA = (
    "d7abb32056cc47a4b304794dcc2d59477a18c7d42effdacfcf1e58c97056d1a1"
)
EXPECTED_RAW_ROOT_SHA = (
    "6968c094a564405fe23e02f9a5b1ed01cce130ea000dd387f7a3f6f12b421d09"
)
EXPECTED_REGISTRY_SHA = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)
EXPECTED_SOURCE_ROOT_SHA = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)

EXPECTED_ACTIVE_RESERVATIONS = []
EXPECTED_RUNTIME_PRIORITY_RULE = [
    "ModelTrainingRows ascending",
    "ModelEvaluationRows ascending",
    "RawExecutionRows ascending",
    "Project ascending",
]

NOISE_LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]
SEEDS = list(range(1, 31))

TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
    "Random",
    "LatestFail",
    "QTF-Avg",
]
ML_TECHNIQUES = [
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "NaiveBayes",
]
INVARIANT_BASELINES = [
    "Random",
    "QTF-Avg",
]

PROJECT_METRICS = [
    "MeanAPFDc",
    "MedianAPFDc",
    "MeanAPFD",
    "MedianAPFD",
]
BUILD_METRICS = [
    "APFDc",
    "APFD",
]

EXPECTED_CONDITIONS = 270
EXPECTED_FILES_PER_CONDITION = 8
EXPECTED_RAW_FILES = 2_160
EXPECTED_RAW_BYTES = 523_093_565

EXPECTED_EVALUATION_ROWS = 18_854
EXPECTED_SCORED_FAILING_BUILDS = 17
EXPECTED_EVALUATION_BUILDS = 1_072
EXPECTED_EVALUATION_FAILURES = 20
EXPECTED_PREDICTORS = 151

EXPECTED_RANKING_ROWS_PER_CONDITION = (
    EXPECTED_EVALUATION_ROWS * len(TECHNIQUES)
)  # 42,126
EXPECTED_BUILD_ROWS_PER_CONDITION = (
    EXPECTED_SCORED_FAILING_BUILDS * len(TECHNIQUES)
)  # 336
EXPECTED_PROJECT_ROWS_PER_CONDITION = len(TECHNIQUES)  # 7
EXPECTED_FIT_ROWS_PER_CONDITION = len(ML_TECHNIQUES)   # 4
EXPECTED_MEDIAN_ROWS_PER_CONDITION = EXPECTED_PREDICTORS  # 151

EXPECTED_TOTAL_RANKING_ROWS = 35_634_060
EXPECTED_TOTAL_BUILD_ROWS = 32_130
EXPECTED_TOTAL_PROJECT_ROWS = 1_890
EXPECTED_TOTAL_FIT_ROWS = 1_080
EXPECTED_TOTAL_AUDIT_ROWS = 270
EXPECTED_TOTAL_MEDIAN_ROWS = 40_770

EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS = len(NOISE_LEVELS) * len(TECHNIQUES)  # 63
EXPECTED_SEED_DELTA_ROWS = EXPECTED_TOTAL_PROJECT_ROWS                        # 1,890
EXPECTED_NOISE_DELTA_SUMMARY_ROWS = EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS     # 63

EXPECTED_CONDITION_FILES = {
    "rankings.csv.gz",
    "build_metrics.csv",
    "project_runs.csv",
    "model_fits.csv",
    "training_medians.csv",
    "condition_audit.csv",
    "condition_summary.json",
    "COMPLETE.json",
}
CONDITION_OUTPUT_FILES = EXPECTED_CONDITION_FILES - {
    "condition_summary.json",
    "COMPLETE.json",
}

REQUIRED_REGISTERED_IDENTITIES = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}


# --------------------------------------------------------------------------------------------------
# 2. DRIVE PATHS
# --------------------------------------------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False,
)

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)
NOTES = ROOT / "Notes"
RESULTS = ROOT / "Results"

REGISTRY = NOTES / "completed_project_registry.csv"

PROJECT_ROOT = RESULTS / "Aggregated" / PROJECT_SLUG
RAW_ROOT = RESULTS / "Raw" / PROJECT_SLUG
FULL_ROOT = PROJECT_ROOT / f"{PROJECT_SHORT}_full_experiment"

PLAN = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_noise_plan"
    / f"{PROJECT_SHORT}_condition_plan.csv"
)

STEP5A_CHECKPOINT = NOTES / "project_24_step5a_checkpoint.json"
STEP5A_STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5a_status.json"
STEP5A_REPORT = FULL_ROOT / f"{PROJECT_SHORT}_step5a_report.json"
STEP5A_RAW_MANIFEST = FULL_ROOT / f"{PROJECT_SHORT}_raw_manifest.csv"
STEP5A_BASELINE = FULL_ROOT / f"{PROJECT_SHORT}_baseline_invariance.csv"

OUT = PROJECT_ROOT / f"{PROJECT_SHORT}_step5b"

CURRENT_MANIFEST = OUT / f"{PROJECT_SHORT}_independent_raw_manifest.csv"
CONDITION_INVENTORY = OUT / f"{PROJECT_SHORT}_independent_condition_inventory.csv"
REVALIDATED_PROJECT_RUNS = OUT / f"{PROJECT_SHORT}_revalidated_project_runs.csv"
REVALIDATED_BUILD_METRICS = OUT / f"{PROJECT_SHORT}_revalidated_build_metrics.csv"
REVALIDATED_MODEL_FITS = OUT / f"{PROJECT_SHORT}_revalidated_model_fits.csv"
REVALIDATED_CONDITION_AUDIT = OUT / f"{PROJECT_SHORT}_revalidated_condition_audit.csv"
REVALIDATED_MEDIANS = OUT / f"{PROJECT_SHORT}_revalidated_training_medians.csv"
NOISE_SUMMARY = OUT / f"{PROJECT_SHORT}_noise_technique_summary.csv"
SEED_DELTAS = OUT / f"{PROJECT_SHORT}_seed_level_noise_deltas.csv"
DELTA_SUMMARY = OUT / f"{PROJECT_SHORT}_noise_delta_summary.csv"
VALIDATION_PATH = OUT / f"{PROJECT_SHORT}_step5b_validation.csv"
REPORT_PATH = OUT / f"{PROJECT_SHORT}_step5b_report.json"

STATUS_PATH = PROJECT_ROOT / f"{PROJECT_SHORT}_step5b_status.json"
CHECKPOINT_PATH = NOTES / "project_24_step5b_checkpoint.json"


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def load_json(
    path,
):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(
            handle
        )


def atomic_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        handle.write(
            "\n"
        )

    os.replace(
        temporary,
        path,
    )


def atomic_csv(
    path,
    frame,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    frame.to_csv(
        temporary,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        temporary,
        path,
    )


def resolve_col(
    columns,
    names,
    label,
):
    lookup = {
        str(column).strip().lower():
            column
        for column in columns
    }

    for name in names:
        key = str(name).strip().lower()

        if key in lookup:
            return lookup[
                key
            ]

    raise RuntimeError(
        f"Could not resolve {label}; "
        f"columns={list(columns)}"
    )


def normalize_manifest(
    frame,
    label,
):
    path_col = resolve_col(
        frame.columns,
        [
            "RelativePath",
        ],
        f"{label} path",
    )

    bytes_col = resolve_col(
        frame.columns,
        [
            "Bytes",
            "SizeBytes",
        ],
        f"{label} bytes",
    )

    sha_col = resolve_col(
        frame.columns,
        [
            "SHA256",
        ],
        f"{label} SHA256",
    )

    out = frame[
        [
            path_col,
            bytes_col,
            sha_col,
        ]
    ].copy()

    out.columns = [
        "RelativePath",
        "Bytes",
        "SHA256",
    ]

    out[
        "RelativePath"
    ] = (
        out[
            "RelativePath"
        ]
        .astype(str)
        .str.replace(
            "\\",
            "/",
            regex=False,
        )
    )

    out[
        "Bytes"
    ] = pd.to_numeric(
        out[
            "Bytes"
        ],
        errors="raise",
    ).astype(
        "int64"
    )

    out[
        "SHA256"
    ] = (
        out[
            "SHA256"
        ]
        .astype(str)
        .str.lower()
    )

    return (
        out.sort_values(
            "RelativePath",
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )


def root_hash(
    manifest,
):
    digest = hashlib.sha256()

    for row in manifest.sort_values(
        "RelativePath",
        kind="mergesort",
    ).itertuples(
        index=False
    ):
        digest.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode(
                "utf-8"
            )
        )

    return digest.hexdigest()


def gzip_rows(
    path,
):
    """
    Independent physical data-row recount of a compressed ranking CSV.
    Generated ranking CSV rows contain no embedded newlines.
    """
    rows = 0

    with gzip.open(
        path,
        "rb",
    ) as handle:
        for _ in handle:
            rows += 1

    return max(
        0,
        rows - 1,
    )


def add_check(
    rows,
    name,
    expected,
    actual,
    passed,
):
    rows.append({
        "Check":
            name,

        "Expected":
            expected,

        "Actual":
            actual,

        "Pass":
            bool(
                passed
            ),
    })


def metric_nonfinite(
    frame,
    columns,
):
    values = (
        frame[
            columns
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(
            dtype=float
        )
    )

    return int(
        (
            ~np.isfinite(
                values
            )
        ).sum()
    )


def metric_outside(
    frame,
    columns,
):
    values = (
        frame[
            columns
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .to_numpy(
            dtype=float
        )
    )

    return int(
        (
            (
                values < 0
            )
            | (
                values > 1
            )
        ).sum()
    )


def pass_series(
    values,
):
    if values.dtype == bool:
        return values.astype(
            bool
        )

    parsed = (
        values.astype(
            str
        )
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )

    if parsed.isna().any():
        raise RuntimeError(
            "Could not parse frozen baseline Pass values."
        )

    return parsed.astype(
        bool
    )


# --------------------------------------------------------------------------------------------------
# 4. REQUIRED INPUTS + SAFE RERUN GUARD
# --------------------------------------------------------------------------------------------------

required_paths = [
    REGISTRY,
    PLAN,
    STEP5A_CHECKPOINT,
    STEP5A_STATUS_PATH,
    STEP5A_REPORT,
    STEP5A_RAW_MANIFEST,
    STEP5A_BASELINE,
    RAW_ROOT,
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing Project 24 Step 5B inputs:\n"
        + "\n".join(
            missing
        )
    )


if CHECKPOINT_PATH.is_file():
    existing_step5b = load_json(
        CHECKPOINT_PATH
    )

    if (
        existing_step5b.get(
            "Status"
        )
        == STEP5B_STATUS
        and bool(
            existing_step5b.get(
                "RawResultsRevalidated",
                False,
            )
        )
        and bool(
            existing_step5b.get(
                "CompactAggregatesFrozen",
                False,
            )
        )
    ):
        raise RuntimeError(
            "Project 24 Step 5B is already frozen successfully. "
            "Do not rerun this cell."
        )

    raise RuntimeError(
        "An unexpected pre-existing Project 24 Step 5B checkpoint exists. "
        "Inspect it before rerunning."
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY THE OFFICIAL STEP 5A FREEZE AND REGISTRY
# --------------------------------------------------------------------------------------------------

step5a_sha = sha256_file(
    STEP5A_CHECKPOINT
)

step5a = load_json(
    STEP5A_CHECKPOINT
)

step5a_status = load_json(
    STEP5A_STATUS_PATH
)

step5a_report = load_json(
    STEP5A_REPORT
)


if step5a_sha != EXPECTED_STEP5A_SHA:
    raise RuntimeError(
        "Step 5A checkpoint SHA-256 differs.\n"
        f"Expected: {EXPECTED_STEP5A_SHA}\n"
        f"Actual:   {step5a_sha}"
    )


for label, payload in [
    (
        "checkpoint",
        step5a,
    ),
    (
        "status",
        step5a_status,
    ),
    (
        "report",
        step5a_report,
    ),
]:
    if payload.get(
        "Status"
    ) != STEP5A_STATUS:
        raise RuntimeError(
            f"Step 5A {label} is not in the frozen PASS state."
        )


if (
    step5a.get(
        "Project"
    ) != PROJECT_NAME
    or step5a.get(
        "ProjectSlug"
    ) != PROJECT_SLUG
):
    raise RuntimeError(
        "Step 5A Project 24 identity differs."
    )


if step5a.get(
    "SourceRootSHA256"
) != EXPECTED_SOURCE_ROOT_SHA:
    raise RuntimeError(
        "Step 5A source-root SHA-256 differs."
    )


if step5a.get(
    "RawRootSHA256"
) != EXPECTED_RAW_ROOT_SHA:
    raise RuntimeError(
        "Step 5A frozen raw-root SHA-256 differs."
    )


for key, expected in [
    (
        "RawFiles",
        EXPECTED_RAW_FILES,
    ),
    (
        "RawBytes",
        EXPECTED_RAW_BYTES,
    ),
    (
        "Conditions",
        EXPECTED_CONDITIONS,
    ),
    (
        "MLFits",
        EXPECTED_TOTAL_FIT_ROWS,
    ),
    (
        "RankingRows",
        EXPECTED_TOTAL_RANKING_ROWS,
    ),
    (
        "BuildMetricRows",
        EXPECTED_TOTAL_BUILD_ROWS,
    ),
    (
        "ProjectRunRows",
        EXPECTED_TOTAL_PROJECT_ROWS,
    ),
    (
        "ConditionAuditRows",
        EXPECTED_TOTAL_AUDIT_ROWS,
    ),
    (
        "TrainingMedianRows",
        EXPECTED_TOTAL_MEDIAN_ROWS,
    ),
]:
    if int(
        step5a.get(
            key,
            -1,
        )
    ) != int(
        expected
    ):
        raise RuntimeError(
            f"Step 5A {key} differs. "
            f"Expected={expected}; "
            f"actual={step5a.get(key)!r}"
        )


if step5a.get(
    "ActiveReservations"
) != EXPECTED_ACTIVE_RESERVATIONS:
    raise RuntimeError(
        "Step 5A active-reservation contract differs."
    )


if step5a.get(
    "RuntimePriorityRule"
) != EXPECTED_RUNTIME_PRIORITY_RULE:
    raise RuntimeError(
        "Step 5A runtime-priority rule differs."
    )


for flag in [
    "Full270ConditionExperimentComplete",
    "RawResultRootFrozen",
    "AllSixWorkerScientificRawRootsVerified",
    "All270ConditionsIndependentlyRevalidated",
    "ReadyForStep5B",
]:
    if not bool(
        step5a.get(
            flag,
            False,
        )
    ):
        raise RuntimeError(
            f"Step 5A checkpoint flag {flag} is not True."
        )


if int(
    step5a.get(
        "MasterFinalizerModelFits",
        -1,
    )
) != 0:
    raise RuntimeError(
        "Step 5A master-finalizer model-fit count differs."
    )


if int(
    step5a.get(
        "MasterFinalizerConditionExecutions",
        -1,
    )
) != 0:
    raise RuntimeError(
        "Step 5A master-finalizer condition-execution count differs."
    )


registry_sha_before = sha256_file(
    REGISTRY
)

if registry_sha_before != EXPECTED_REGISTRY_SHA:
    raise RuntimeError(
        "Registry SHA-256 differs before Project 24 Step 5B.\n"
        f"Expected: {EXPECTED_REGISTRY_SHA}\n"
        f"Actual:   {registry_sha_before}"
    )


registry = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna(
    ""
)


project_number_col = resolve_col(
    registry.columns,
    [
        "ProjectNumber",
        "project_number",
        "ProjectNo",
        "Project No",
    ],
    "registry ProjectNumber",
)

project_col = resolve_col(
    registry.columns,
    [
        "Project",
        "ProjectName",
        "project_name",
    ],
    "registry Project",
)

status_col = resolve_col(
    registry.columns,
    [
        "Status",
        "ProjectStatus",
        "project_status",
    ],
    "registry Status",
)


project_numbers = pd.to_numeric(
    registry[
        project_number_col
    ],
    errors="raise",
).astype(
    int
)


if (
    len(
        registry
    ) != 23
    or sorted(
        project_numbers.tolist()
    )
    != list(
        range(
            1,
            24,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–23 "
        "before Project 24 Step 5B."
    )


if not registry[
    status_col
].astype(
    str
).eq(
    "COMPLETE_AND_FROZEN"
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )


for required_number, required_project in REQUIRED_REGISTERED_IDENTITIES.items():
    matching = registry.loc[
        project_numbers.eq(
            required_number
        )
    ]

    if (
        len(
            matching
        ) != 1
        or str(
            matching.iloc[
                0
            ][
                project_col
            ]
        )
        != required_project
    ):
        raise RuntimeError(
            "A frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )


project_24_registry_rows = int(
    project_numbers.eq(
        PROJECT_NUMBER
    ).sum()
)

project_name_match_rows_diagnostic = int(
    registry[
        project_col
    ].astype(
        str
    ).eq(
        PROJECT_NAME
    ).sum()
)

if project_24_registry_rows != 0:
    raise RuntimeError(
        "Project 24 is unexpectedly already present in the completion registry by ProjectNumber."
    )

print(
    "Completion-registry ProjectNumber-24 rows:",
    project_24_registry_rows,
)

print(
    "Completion-registry exact project-name matches (diagnostic only):",
    project_name_match_rows_diagnostic,
)


# --------------------------------------------------------------------------------------------------
# 6. VERIFY STEP 5A AGGREGATE MANIFEST
# --------------------------------------------------------------------------------------------------

aggregate_manifest = step5a.get(
    "AggregateOutputManifest",
    [],
)


if (
    not isinstance(
        aggregate_manifest,
        list,
    )
    or not aggregate_manifest
):
    raise RuntimeError(
        "Step 5A checkpoint has no AggregateOutputManifest."
    )


aggregate_manifest_failures = 0


for item in aggregate_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    passed = bool(
        path.is_file()
        and int(
            path.stat().st_size
        )
        == int(
            item[
                "Bytes"
            ]
        )
        and sha256_file(
            path
        )
        == str(
            item[
                "SHA256"
            ]
        ).lower()
    )

    aggregate_manifest_failures += int(
        not passed
    )


if aggregate_manifest_failures:
    raise RuntimeError(
        f"{aggregate_manifest_failures} frozen "
        "Step 5A aggregate outputs changed."
    )


# --------------------------------------------------------------------------------------------------
# 7. INDEPENDENTLY HASH ALL 2,160 RAW FILES
# --------------------------------------------------------------------------------------------------

print(
    "\nIndependently hashing all 2,160 raw files."
)


hash_start = time.perf_counter()


raw_paths = sorted(
    [
        path
        for path in RAW_ROOT.rglob(
            "*"
        )
        if path.is_file()
    ],
    key=lambda path:
        path.relative_to(
            RAW_ROOT
        ).as_posix(),
)


manifest_rows = []


for index, path in enumerate(
    raw_paths,
    start=1,
):
    manifest_rows.append({
        "RelativePath":
            path.relative_to(
                RAW_ROOT
            ).as_posix(),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    })

    if (
        index % 200 == 0
        or index == len(
            raw_paths
        )
    ):
        print(
            "  Raw hashing progress:",
            index,
            "/",
            len(
                raw_paths
            ),
            "files",
        )


current_manifest = normalize_manifest(
    pd.DataFrame(
        manifest_rows
    ),
    "current manifest",
)


hash_seconds = (
    time.perf_counter()
    - hash_start
)


frozen_manifest = normalize_manifest(
    pd.read_csv(
        STEP5A_RAW_MANIFEST,
        low_memory=False,
    ),
    "frozen Step 5A manifest",
)


current_raw_sha = root_hash(
    current_manifest
)


current_raw_bytes = int(
    current_manifest[
        "Bytes"
    ].sum()
)


manifest_compare = frozen_manifest.merge(
    current_manifest,
    on="RelativePath",
    how="outer",
    suffixes=(
        "_frozen",
        "_current",
    ),
    indicator=True,
)


missing_raw = int(
    manifest_compare[
        "_merge"
    ].eq(
        "left_only"
    ).sum()
)


unexpected_raw = int(
    manifest_compare[
        "_merge"
    ].eq(
        "right_only"
    ).sum()
)


both = manifest_compare[
    "_merge"
].eq(
    "both"
)


size_mismatch = int(
    (
        both
        & manifest_compare[
            "Bytes_frozen"
        ].ne(
            manifest_compare[
                "Bytes_current"
            ]
        )
    ).sum()
)


hash_mismatch = int(
    (
        both
        & manifest_compare[
            "SHA256_frozen"
        ].ne(
            manifest_compare[
                "SHA256_current"
            ]
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 8. CONDITION-BY-CONDITION INDEPENDENT REVALIDATION
# --------------------------------------------------------------------------------------------------

condition_plan = pd.read_csv(
    PLAN,
    low_memory=False,
)


condition_id_col = resolve_col(
    condition_plan.columns,
    [
        "ConditionID",
        "ConditionKey",
    ],
    "condition identifier",
)

condition_order_col = resolve_col(
    condition_plan.columns,
    [
        "ConditionOrder",
    ],
    "condition order",
)

noise_col = resolve_col(
    condition_plan.columns,
    [
        "NoisePercent",
    ],
    "noise percent",
)

seed_col = resolve_col(
    condition_plan.columns,
    [
        "RepetitionSeed",
    ],
    "repetition seed",
)


for column in [
    condition_order_col,
    noise_col,
    seed_col,
]:
    condition_plan[
        column
    ] = pd.to_numeric(
        condition_plan[
            column
        ],
        errors="raise",
    ).astype(
        int
    )


condition_plan = (
    condition_plan.sort_values(
        condition_order_col,
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# Exact frozen design-grid order: seed-major, then the nine noise levels.
expected_grid = [
    (
        seed,
        noise,
        f"noise_{noise:02d}__seed_{seed:02d}",
    )
    for seed in SEEDS
    for noise in NOISE_LEVELS
]

actual_grid = [
    (
        int(
            row[
                seed_col
            ]
        ),
        int(
            row[
                noise_col
            ]
        ),
        str(
            row[
                condition_id_col
            ]
        ),
    )
    for _, row in condition_plan.iterrows()
]

if actual_grid != expected_grid:
    raise RuntimeError(
        "Project 24 condition plan/grid order differs from the frozen design."
    )


inventory_rows = []
project_frames = []
build_frames = []
fit_frames = []
audit_frames = []
median_frames = []

marker_failures = 0
summary_failures = 0
file_set_failures = 0
embedded_manifest_failures = 0
ranking_count_failures = 0


print(
    "\nRevalidating all 270 condition directories."
)


condition_start = time.perf_counter()


for index, (_, row) in enumerate(
    condition_plan.iterrows(),
    start=1,
):
    condition_key = str(
        row[
            condition_id_col
        ]
    )

    condition_order = int(
        row[
            condition_order_col
        ]
    )

    noise_percent = int(
        row[
            noise_col
        ]
    )

    repetition_seed = int(
        row[
            seed_col
        ]
    )

    condition_dir = (
        RAW_ROOT
        / condition_key
    )

    if not condition_dir.is_dir():
        raise FileNotFoundError(
            f"Missing condition directory: {condition_dir}"
        )

    actual_files = {
        path.name
        for path in condition_dir.iterdir()
        if path.is_file()
    }

    file_set_pass = (
        actual_files
        == EXPECTED_CONDITION_FILES
    )

    file_set_failures += int(
        not file_set_pass
    )

    complete_path = (
        condition_dir
        / "COMPLETE.json"
    )

    summary_path = (
        condition_dir
        / "condition_summary.json"
    )

    complete = load_json(
        complete_path
    )

    summary = load_json(
        summary_path
    )

    complete_pass = bool(
        complete.get(
            "Status"
        )
        == CONDITION_STATUS
        and complete.get(
            "ConditionKey"
        )
        == condition_key
        and str(
            complete.get(
                "ConditionSummaryPath"
            )
        )
        == str(
            summary_path
        )
        and str(
            complete.get(
                "ConditionSummarySHA256"
            )
        ).lower()
        == sha256_file(
            summary_path
        )
    )

    summary_pass = bool(
        summary.get(
            "Status"
        )
        == CONDITION_STATUS
        and summary.get(
            "ConditionKey"
        )
        == condition_key
        and int(
            summary.get(
                "ConditionOrder",
                -1,
            )
        )
        == condition_order
        and int(
            summary.get(
                "NoisePercent",
                -1,
            )
        )
        == noise_percent
        and int(
            summary.get(
                "RepetitionSeed",
                -1,
            )
        )
        == repetition_seed
    )

    marker_failures += int(
        not complete_pass
    )

    summary_failures += int(
        not summary_pass
    )

    output_manifest = summary.get(
        "OutputManifest",
        [],
    )

    local_embedded_failures = 0
    output_names = set()

    if (
        not isinstance(
            output_manifest,
            list,
        )
        or len(
            output_manifest
        )
        != 6
    ):
        local_embedded_failures += 1

    else:
        for item in output_manifest:
            path = Path(
                item.get(
                    "Path",
                    "",
                )
            )

            output_names.add(
                path.name
            )

            passed = bool(
                path.parent
                == condition_dir
                and path.is_file()
                and int(
                    path.stat().st_size
                )
                == int(
                    item.get(
                        "Bytes",
                        -1,
                    )
                )
                and sha256_file(
                    path
                )
                == str(
                    item.get(
                        "SHA256",
                        "",
                    )
                ).lower()
            )

            local_embedded_failures += int(
                not passed
            )

        local_embedded_failures += int(
            output_names
            != CONDITION_OUTPUT_FILES
        )

    embedded_manifest_failures += (
        local_embedded_failures
    )

    ranking_rows = gzip_rows(
        condition_dir
        / "rankings.csv.gz"
    )

    ranking_count_failures += int(
        ranking_rows
        != EXPECTED_RANKING_ROWS_PER_CONDITION
    )

    build_metrics = pd.read_csv(
        condition_dir
        / "build_metrics.csv",
        low_memory=False,
    )

    project_runs = pd.read_csv(
        condition_dir
        / "project_runs.csv",
        low_memory=False,
    )

    model_fits = pd.read_csv(
        condition_dir
        / "model_fits.csv",
        low_memory=False,
    )

    condition_audit = pd.read_csv(
        condition_dir
        / "condition_audit.csv",
        low_memory=False,
    )

    training_medians = pd.read_csv(
        condition_dir
        / "training_medians.csv",
        low_memory=False,
    )

    expected_counts = [
        EXPECTED_BUILD_ROWS_PER_CONDITION,
        EXPECTED_PROJECT_ROWS_PER_CONDITION,
        EXPECTED_FIT_ROWS_PER_CONDITION,
        1,
        EXPECTED_MEDIAN_ROWS_PER_CONDITION,
    ]

    actual_counts = [
        len(
            build_metrics
        ),
        len(
            project_runs
        ),
        len(
            model_fits
        ),
        len(
            condition_audit
        ),
        len(
            training_medians
        ),
    ]

    if actual_counts != expected_counts:
        raise RuntimeError(
            f"{condition_key}: compact output counts differ.\n"
            f"Expected: {expected_counts}\n"
            f"Actual:   {actual_counts}"
        )

    for field, count in [
        (
            "RankingRows",
            ranking_rows,
        ),
        (
            "BuildMetricRows",
            len(
                build_metrics
            ),
        ),
        (
            "ProjectRunRows",
            len(
                project_runs
            ),
        ),
        (
            "MLFits",
            len(
                model_fits
            ),
        ),
        (
            "TrainingMedianRows",
            len(
                training_medians
            ),
        ),
    ]:
        if int(
            summary.get(
                field,
                -1,
            )
        ) != int(
            count
        ):
            raise RuntimeError(
                f"{condition_key}: "
                f"condition_summary {field} differs."
            )

    project_frames.append(
        project_runs
    )

    build_frames.append(
        build_metrics
    )

    fit_frames.append(
        model_fits
    )

    audit_frames.append(
        condition_audit
    )

    median_frames.append(
        training_medians
    )

    inventory_rows.append({
        "ProjectNumber":
            PROJECT_NUMBER,

        "Project":
            PROJECT_NAME,

        "ProjectSlug":
            PROJECT_SLUG,

        "ConditionOrder":
            condition_order,

        "ConditionKey":
            condition_key,

        "NoisePercent":
            noise_percent,

        "RepetitionSeed":
            repetition_seed,

        "ConditionDirectory":
            str(
                condition_dir
            ),

        "CompletionStatus":
            complete.get(
                "Status"
            ),

        "SummaryStatus":
            summary.get(
                "Status"
            ),

        "Files":
            len(
                actual_files
            ),

        "ConditionBytes":
            int(
                sum(
                    path.stat().st_size
                    for path in condition_dir.iterdir()
                    if path.is_file()
                )
            ),

        "RankingRows":
            ranking_rows,

        "BuildMetricRows":
            len(
                build_metrics
            ),

        "ProjectRunRows":
            len(
                project_runs
            ),

        "ModelFits":
            len(
                model_fits
            ),

        "ConditionAuditRows":
            len(
                condition_audit
            ),

        "TrainingMedianRows":
            len(
                training_medians
            ),

        "FileSetPass":
            file_set_pass,

        "CompletionMarkerPass":
            complete_pass,

        "ConditionSummaryPass":
            summary_pass,

        "EmbeddedManifestFailures":
            local_embedded_failures,

        "CompletionMarkerSHA256":
            sha256_file(
                complete_path
            ),

        "ConditionSummarySHA256":
            sha256_file(
                summary_path
            ),
    })

    if (
        index % 30 == 0
        or index == len(
            condition_plan
        )
    ):
        print(
            "  Condition revalidation progress:",
            index,
            "/",
            len(
                condition_plan
            ),
        )


condition_seconds = (
    time.perf_counter()
    - condition_start
)


inventory = (
    pd.DataFrame(
        inventory_rows
    )
    .sort_values(
        "ConditionOrder",
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


project_runs = pd.concat(
    project_frames,
    ignore_index=True,
)


build_metrics = pd.concat(
    build_frames,
    ignore_index=True,
)


model_fits = pd.concat(
    fit_frames,
    ignore_index=True,
)


condition_audit = pd.concat(
    audit_frames,
    ignore_index=True,
)


training_medians = pd.concat(
    median_frames,
    ignore_index=True,
)


# --------------------------------------------------------------------------------------------------
# 9. INDEPENDENT CONTRACT AUDITS
# --------------------------------------------------------------------------------------------------

coordinate_count = len(
    inventory[
        [
            "NoisePercent",
            "RepetitionSeed",
        ]
    ].drop_duplicates()
)


duplicate_condition_keys = int(
    inventory.duplicated(
        [
            "ConditionKey",
        ],
        keep=False,
    ).sum()
)


duplicate_coordinates = int(
    inventory.duplicated(
        [
            "NoisePercent",
            "RepetitionSeed",
        ],
        keep=False,
    ).sum()
)


condition_order_violations = int(
    (
        inventory[
            "ConditionOrder"
        ].to_numpy(
            dtype=int
        )
        != np.arange(
            1,
            EXPECTED_CONDITIONS + 1,
        )
    ).sum()
)


files_per_condition_violations = int(
    inventory[
        "Files"
    ].ne(
        EXPECTED_FILES_PER_CONDITION
    ).sum()
)


ranking_rows_per_condition_violations = int(
    inventory[
        "RankingRows"
    ].ne(
        EXPECTED_RANKING_ROWS_PER_CONDITION
    ).sum()
)


small_rows_per_condition_violations = int(
    inventory[
        "BuildMetricRows"
    ].ne(
        EXPECTED_BUILD_ROWS_PER_CONDITION
    ).sum()
    + inventory[
        "ProjectRunRows"
    ].ne(
        EXPECTED_PROJECT_ROWS_PER_CONDITION
    ).sum()
    + inventory[
        "ModelFits"
    ].ne(
        EXPECTED_FIT_ROWS_PER_CONDITION
    ).sum()
    + inventory[
        "ConditionAuditRows"
    ].ne(
        1
    ).sum()
    + inventory[
        "TrainingMedianRows"
    ].ne(
        EXPECTED_MEDIAN_ROWS_PER_CONDITION
    ).sum()
)


project_techniques = sorted(
    project_runs[
        "Technique"
    ].astype(
        str
    ).unique().tolist()
)


fit_techniques = sorted(
    model_fits[
        "Technique"
    ].astype(
        str
    ).unique().tolist()
)


duplicate_project_rows = int(
    project_runs.duplicated(
        [
            "ConditionKey",
            "Technique",
        ],
        keep=False,
    ).sum()
)


duplicate_build_rows = int(
    build_metrics.duplicated(
        [
            "ConditionKey",
            "Technique",
            "Build",
        ],
        keep=False,
    ).sum()
)


duplicate_fit_rows = int(
    model_fits.duplicated(
        [
            "ConditionKey",
            "Technique",
        ],
        keep=False,
    ).sum()
)


duplicate_audit_rows = int(
    condition_audit.duplicated(
        [
            "ConditionKey",
        ],
        keep=False,
    ).sum()
)


duplicate_median_rows = int(
    training_medians.duplicated(
        [
            "ConditionKey",
            "PredictorOrder",
        ],
        keep=False,
    ).sum()
)


model_fit_failures = int(
    (
        ~model_fits[
            "Status"
        ].astype(
            str
        ).eq(
            "PASS_MODEL_FIT"
        )
    ).sum()
)


model_fit_errors = int(
    model_fits[
        "Error"
    ].fillna(
        ""
    ).astype(
        str
    ).str.len().gt(
        0
    ).sum()
)


project_nonfinite = metric_nonfinite(
    project_runs,
    PROJECT_METRICS,
)


project_outside = metric_outside(
    project_runs,
    PROJECT_METRICS,
)


build_nonfinite = metric_nonfinite(
    build_metrics,
    BUILD_METRICS,
)


build_outside = metric_outside(
    build_metrics,
    BUILD_METRICS,
)


median_values = pd.to_numeric(
    training_medians[
        "TrainingMedian"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


median_nonfinite = int(
    (
        ~np.isfinite(
            median_values
        )
    ).sum()
)


median_predictor_violations = int(
    training_medians.groupby(
        "ConditionKey"
    )[
        "Predictor"
    ].nunique().ne(
        EXPECTED_PREDICTORS
    ).sum()
)


scored_build_violations = int(
    project_runs[
        "ScoredFailingBuilds"
    ].ne(
        EXPECTED_SCORED_FAILING_BUILDS
    ).sum()
)


evaluation_build_violations = int(
    project_runs[
        "EvaluationBuilds"
    ].ne(
        EXPECTED_EVALUATION_BUILDS
    ).sum()
)


evaluation_row_violations = int(
    project_runs[
        "EvaluationRows"
    ].ne(
        EXPECTED_EVALUATION_ROWS
    ).sum()
)


evaluation_failure_violations = int(
    project_runs[
        "EvaluationFailures"
    ].ne(
        EXPECTED_EVALUATION_FAILURES
    ).sum()
)


zero_noise = condition_audit.loc[
    pd.to_numeric(
        condition_audit[
            "NoisePercent"
        ],
        errors="raise",
    ).eq(
        0
    )
].copy()


positive_noise = condition_audit.loc[
    pd.to_numeric(
        condition_audit[
            "NoisePercent"
        ],
        errors="raise",
    ).gt(
        0
    )
].copy()


zero_flip_violations = int(
    pd.to_numeric(
        zero_noise[
            "NumberFlipped"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


zero_model_violations = int(
    pd.to_numeric(
        zero_noise[
            "ModelLabelChanges"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


zero_rec_violations = int(
    pd.to_numeric(
        zero_noise[
            "DependentRECChanges"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


positive_raw_violations = int(
    pd.to_numeric(
        positive_noise[
            "NumberFlipped"
        ],
        errors="raise",
    ).le(
        0
    ).sum()
)


positive_model_violations = int(
    pd.to_numeric(
        positive_noise[
            "ModelLabelChanges"
        ],
        errors="raise",
    ).le(
        0
    ).sum()
)


positive_rec_violations = int(
    pd.to_numeric(
        positive_noise[
            "DependentRECChanges"
        ],
        errors="raise",
    ).le(
        0
    ).sum()
)


independent_rec_violations = int(
    pd.to_numeric(
        condition_audit[
            "IndependentRECChanges"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


independent_reconstruction_violations = int(
    pd.to_numeric(
        condition_audit[
            "IndependentReconstructionMismatches"
        ],
        errors="raise",
    ).ne(
        0
    ).sum()
)


noise_hash_mismatches = 0


for expected_column, actual_column in [
    (
        "ExpectedFlipMaskSHA256",
        "ActualFlipMaskSHA256",
    ),
    (
        "ExpectedNoisyRawVerdictSHA256",
        "ActualNoisyRawVerdictSHA256",
    ),
    (
        "ExpectedNoisyModelVerdictSHA256",
        "ActualNoisyModelVerdictSHA256",
    ),
]:
    noise_hash_mismatches += int(
        condition_audit[
            expected_column
        ].astype(
            str
        ).ne(
            condition_audit[
                actual_column
            ].astype(
                str
            )
        ).sum()
    )


# Frozen ranking-level baseline-invariance audit from the zero-fit Step 5A master.
baseline = pd.read_csv(
    STEP5A_BASELINE,
    low_memory=False,
)


if "Pass" in baseline.columns:
    ranking_baseline_failures = int(
        (
            ~pass_series(
                baseline[
                    "Pass"
                ]
            )
        ).sum()
    )

else:
    mismatch_columns = [
        column
        for column in baseline.columns
        if "mismatch" in column.lower()
    ]

    if not mismatch_columns:
        raise RuntimeError(
            "Could not resolve baseline-invariance Pass/mismatch fields."
        )

    ranking_baseline_failures = int(
        baseline[
            mismatch_columns
        ].apply(
            pd.to_numeric,
            errors="coerce",
        ).fillna(
            0
        ).to_numpy(
            dtype=float
        ).sum()
    )


# Independently re-evaluate project-metric invariance for Random/QTF-Avg.
# Each metric is tested independently; APFDc and APFD are NEVER compared to one another.
metric_baseline_failures = 0
metric_baseline_maximum_range = 0.0


for _, group in project_runs.loc[
    project_runs[
        "Technique"
    ].isin(
        INVARIANT_BASELINES
    )
].groupby(
    [
        "RepetitionSeed",
        "Technique",
    ],
    sort=False,
):
    values = group[
        PROJECT_METRICS
    ].to_numpy(
        dtype=float
    )

    per_metric_ranges = (
        np.max(
            values,
            axis=0,
        )
        - np.min(
            values,
            axis=0,
        )
    )

    metric_baseline_maximum_range = max(
        metric_baseline_maximum_range,
        float(
            np.max(
                per_metric_ranges
            )
        ),
    )

    metric_baseline_failures += int(
        (
            per_metric_ranges
            > 1e-15
        ).any()
    )


# --------------------------------------------------------------------------------------------------
# 10. COMPACT ANALYSIS-READY AGGREGATES
# --------------------------------------------------------------------------------------------------

aggregation_start = time.perf_counter()


noise_summary = (
    project_runs.groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        Runs=(
            "ConditionKey",
            "count",
        ),
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        Mean_MeanAPFDc=(
            "MeanAPFDc",
            "mean",
        ),
        SD_MeanAPFDc=(
            "MeanAPFDc",
            "std",
        ),
        Median_MeanAPFDc=(
            "MeanAPFDc",
            "median",
        ),

        Mean_MedianAPFDc=(
            "MedianAPFDc",
            "mean",
        ),
        SD_MedianAPFDc=(
            "MedianAPFDc",
            "std",
        ),
        Median_MedianAPFDc=(
            "MedianAPFDc",
            "median",
        ),

        Mean_MeanAPFD=(
            "MeanAPFD",
            "mean",
        ),
        SD_MeanAPFD=(
            "MeanAPFD",
            "std",
        ),
        Median_MeanAPFD=(
            "MeanAPFD",
            "median",
        ),

        Mean_MedianAPFD=(
            "MedianAPFD",
            "mean",
        ),
        SD_MedianAPFD=(
            "MedianAPFD",
            "std",
        ),
        Median_MedianAPFD=(
            "MedianAPFD",
            "median",
        ),
    )
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


clean_reference = (
    project_runs.loc[
        project_runs[
            "NoisePercent"
        ].eq(
            0
        ),
        [
            "RepetitionSeed",
            "Technique",
        ]
        + PROJECT_METRICS,
    ]
    .rename(
        columns={
            metric:
                f"Clean_{metric}"
            for metric in PROJECT_METRICS
        }
    )
)


if len(
    clean_reference
) != (
    len(SEEDS)
    * len(TECHNIQUES)
):
    raise RuntimeError(
        "Clean per-seed/per-technique reference is incomplete."
    )


seed_deltas = project_runs.merge(
    clean_reference,
    on=[
        "RepetitionSeed",
        "Technique",
    ],
    how="left",
    validate="many_to_one",
)


for metric in PROJECT_METRICS:
    seed_deltas[
        f"Delta_{metric}"
    ] = (
        seed_deltas[
            metric
        ]
        - seed_deltas[
            f"Clean_{metric}"
        ]
    )


delta_columns = [
    f"Delta_{metric}"
    for metric in PROJECT_METRICS
]


seed_deltas = (
    seed_deltas[
        [
            "ProjectNumber",
            "Project",
            "ProjectSlug",
            "ConditionKey",
            "NoisePercent",
            "RepetitionSeed",
            "Technique",
        ]
        + PROJECT_METRICS
        + [
            f"Clean_{metric}"
            for metric in PROJECT_METRICS
        ]
        + delta_columns
    ]
    .sort_values(
        [
            "NoisePercent",
            "Technique",
            "RepetitionSeed",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


delta_summary = (
    seed_deltas.groupby(
        [
            "NoisePercent",
            "Technique",
        ],
        as_index=False,
        sort=True,
    )
    .agg(
        Seeds=(
            "RepetitionSeed",
            "nunique",
        ),

        Mean_Delta_MeanAPFDc=(
            "Delta_MeanAPFDc",
            "mean",
        ),
        SD_Delta_MeanAPFDc=(
            "Delta_MeanAPFDc",
            "std",
        ),
        Median_Delta_MeanAPFDc=(
            "Delta_MeanAPFDc",
            "median",
        ),

        Mean_Delta_MedianAPFDc=(
            "Delta_MedianAPFDc",
            "mean",
        ),
        SD_Delta_MedianAPFDc=(
            "Delta_MedianAPFDc",
            "std",
        ),
        Median_Delta_MedianAPFDc=(
            "Delta_MedianAPFDc",
            "median",
        ),

        Mean_Delta_MeanAPFD=(
            "Delta_MeanAPFD",
            "mean",
        ),
        SD_Delta_MeanAPFD=(
            "Delta_MeanAPFD",
            "std",
        ),
        Median_Delta_MeanAPFD=(
            "Delta_MeanAPFD",
            "median",
        ),

        Mean_Delta_MedianAPFD=(
            "Delta_MedianAPFD",
            "mean",
        ),
        SD_Delta_MedianAPFD=(
            "Delta_MedianAPFD",
            "std",
        ),
        Median_Delta_MedianAPFD=(
            "Delta_MedianAPFD",
            "median",
        ),
    )
    .sort_values(
        [
            "NoisePercent",
            "Technique",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


aggregation_seconds = (
    time.perf_counter()
    - aggregation_start
)


noise_summary_numeric_columns = [
    column
    for column in noise_summary.columns
    if column not in {
        "NoisePercent",
        "Technique",
    }
]


delta_summary_numeric_columns = [
    column
    for column in delta_summary.columns
    if column not in {
        "NoisePercent",
        "Technique",
    }
]


summary_nonfinite = metric_nonfinite(
    noise_summary,
    noise_summary_numeric_columns,
)


delta_nonfinite = metric_nonfinite(
    delta_summary,
    delta_summary_numeric_columns,
)


clean_delta_nonzero = int(
    (
        np.abs(
            seed_deltas.loc[
                seed_deltas[
                    "NoisePercent"
                ].eq(
                    0
                ),
                delta_columns,
            ].to_numpy(
                dtype=float
            )
        )
        > 1e-15
    ).sum()
)


invariant_baseline_delta_nonzero = int(
    (
        np.abs(
            seed_deltas.loc[
                seed_deltas[
                    "Technique"
                ].isin(
                    INVARIANT_BASELINES
                ),
                delta_columns,
            ].to_numpy(
                dtype=float
            )
        )
        > 1e-15
    ).sum()
)


# Explicitly prove pandas used sample SD (ddof=1) on an example group.
example_group = (
    project_runs.loc[
        (
            project_runs[
                "NoisePercent"
            ].eq(
                0
            )
            & project_runs[
                "Technique"
            ].eq(
                "RandomForest"
            )
        ),
        "MeanAPFDc",
    ]
    .to_numpy(
        dtype=float
    )
)


sample_sd_contract_pass = bool(
    len(
        example_group
    )
    == 30
    and np.isclose(
        noise_summary.loc[
            (
                noise_summary[
                    "NoisePercent"
                ].eq(
                    0
                )
                & noise_summary[
                    "Technique"
                ].eq(
                    "RandomForest"
                )
            ),
            "SD_MeanAPFDc",
        ].iloc[
            0
        ],
        np.std(
            example_group,
            ddof=1,
        ),
        rtol=0,
        atol=1e-15,
    )
)


# --------------------------------------------------------------------------------------------------
# 11. VALIDATION TABLE
# --------------------------------------------------------------------------------------------------

checks = []


add_check(
    checks,
    "Step 5A status",
    STEP5A_STATUS,
    step5a.get(
        "Status"
    ),
    step5a.get(
        "Status"
    )
    == STEP5A_STATUS,
)


add_check(
    checks,
    "Step 5A checkpoint SHA-256",
    EXPECTED_STEP5A_SHA,
    step5a_sha,
    step5a_sha
    == EXPECTED_STEP5A_SHA,
)


add_check(
    checks,
    "Frozen raw-root SHA-256",
    EXPECTED_RAW_ROOT_SHA,
    step5a.get(
        "RawRootSHA256"
    ),
    step5a.get(
        "RawRootSHA256"
    )
    == EXPECTED_RAW_ROOT_SHA,
)


add_check(
    checks,
    "Independent current raw-root SHA-256",
    EXPECTED_RAW_ROOT_SHA,
    current_raw_sha,
    current_raw_sha
    == EXPECTED_RAW_ROOT_SHA,
)


add_check(
    checks,
    "Step 5A aggregate-manifest failures",
    0,
    aggregate_manifest_failures,
    aggregate_manifest_failures
    == 0,
)


add_check(
    checks,
    "Condition marker failures",
    0,
    marker_failures,
    marker_failures
    == 0,
)


add_check(
    checks,
    "Condition summary failures",
    0,
    summary_failures,
    summary_failures
    == 0,
)


add_check(
    checks,
    "Condition file-set failures",
    0,
    file_set_failures,
    file_set_failures
    == 0,
)


add_check(
    checks,
    "Embedded output-manifest failures",
    0,
    embedded_manifest_failures,
    embedded_manifest_failures
    == 0,
)


add_check(
    checks,
    "Conditions",
    EXPECTED_CONDITIONS,
    len(
        inventory
    ),
    len(
        inventory
    )
    == EXPECTED_CONDITIONS,
)


add_check(
    checks,
    "Condition coordinates",
    EXPECTED_CONDITIONS,
    coordinate_count,
    coordinate_count
    == EXPECTED_CONDITIONS,
)


add_check(
    checks,
    "Duplicate condition keys",
    0,
    duplicate_condition_keys,
    duplicate_condition_keys
    == 0,
)


add_check(
    checks,
    "Duplicate condition coordinates",
    0,
    duplicate_coordinates,
    duplicate_coordinates
    == 0,
)


add_check(
    checks,
    "Condition-order violations",
    0,
    condition_order_violations,
    condition_order_violations
    == 0,
)


add_check(
    checks,
    "Files-per-condition violations",
    0,
    files_per_condition_violations,
    files_per_condition_violations
    == 0,
)


add_check(
    checks,
    "Raw files",
    EXPECTED_RAW_FILES,
    len(
        current_manifest
    ),
    len(
        current_manifest
    )
    == EXPECTED_RAW_FILES,
)


add_check(
    checks,
    "Raw bytes",
    EXPECTED_RAW_BYTES,
    current_raw_bytes,
    current_raw_bytes
    == EXPECTED_RAW_BYTES,
)


add_check(
    checks,
    "Missing raw files",
    0,
    missing_raw,
    missing_raw
    == 0,
)


add_check(
    checks,
    "Unexpected raw files",
    0,
    unexpected_raw,
    unexpected_raw
    == 0,
)


add_check(
    checks,
    "Raw size mismatches",
    0,
    size_mismatch,
    size_mismatch
    == 0,
)


add_check(
    checks,
    "Raw SHA-256 mismatches",
    0,
    hash_mismatch,
    hash_mismatch
    == 0,
)


add_check(
    checks,
    "Ranking rows",
    EXPECTED_TOTAL_RANKING_ROWS,
    int(
        inventory[
            "RankingRows"
        ].sum()
    ),
    int(
        inventory[
            "RankingRows"
        ].sum()
    )
    == EXPECTED_TOTAL_RANKING_ROWS,
)


add_check(
    checks,
    "Ranking row-count failures",
    0,
    (
        ranking_count_failures
        + ranking_rows_per_condition_violations
    ),
    (
        ranking_count_failures
        + ranking_rows_per_condition_violations
    )
    == 0,
)


add_check(
    checks,
    "Project-run rows",
    EXPECTED_TOTAL_PROJECT_ROWS,
    len(
        project_runs
    ),
    len(
        project_runs
    )
    == EXPECTED_TOTAL_PROJECT_ROWS,
)


add_check(
    checks,
    "Build-metric rows",
    EXPECTED_TOTAL_BUILD_ROWS,
    len(
        build_metrics
    ),
    len(
        build_metrics
    )
    == EXPECTED_TOTAL_BUILD_ROWS,
)


add_check(
    checks,
    "Model-fit rows",
    EXPECTED_TOTAL_FIT_ROWS,
    len(
        model_fits
    ),
    len(
        model_fits
    )
    == EXPECTED_TOTAL_FIT_ROWS,
)


add_check(
    checks,
    "Condition-audit rows",
    EXPECTED_TOTAL_AUDIT_ROWS,
    len(
        condition_audit
    ),
    len(
        condition_audit
    )
    == EXPECTED_TOTAL_AUDIT_ROWS,
)


add_check(
    checks,
    "Training-median rows",
    EXPECTED_TOTAL_MEDIAN_ROWS,
    len(
        training_medians
    ),
    len(
        training_medians
    )
    == EXPECTED_TOTAL_MEDIAN_ROWS,
)


add_check(
    checks,
    "Small rows-per-condition violations",
    0,
    small_rows_per_condition_violations,
    small_rows_per_condition_violations
    == 0,
)


add_check(
    checks,
    "Project-run technique set",
    sorted(
        TECHNIQUES
    ),
    project_techniques,
    project_techniques
    == sorted(
        TECHNIQUES
    ),
)


add_check(
    checks,
    "Model-fit technique set",
    sorted(
        ML_TECHNIQUES
    ),
    fit_techniques,
    fit_techniques
    == sorted(
        ML_TECHNIQUES
    ),
)


duplicate_compact_rows = (
    duplicate_project_rows
    + duplicate_build_rows
    + duplicate_fit_rows
    + duplicate_audit_rows
    + duplicate_median_rows
)


add_check(
    checks,
    "Duplicate project/build/fit/audit/median rows",
    0,
    duplicate_compact_rows,
    duplicate_compact_rows
    == 0,
)


add_check(
    checks,
    "Model-fit status failures",
    0,
    model_fit_failures,
    model_fit_failures
    == 0,
)


add_check(
    checks,
    "Model-fit error rows",
    0,
    model_fit_errors,
    model_fit_errors
    == 0,
)


add_check(
    checks,
    "Project metrics non-finite",
    0,
    project_nonfinite,
    project_nonfinite
    == 0,
)


add_check(
    checks,
    "Project metrics outside [0,1]",
    0,
    project_outside,
    project_outside
    == 0,
)


add_check(
    checks,
    "Build metrics non-finite",
    0,
    build_nonfinite,
    build_nonfinite
    == 0,
)


add_check(
    checks,
    "Build metrics outside [0,1]",
    0,
    build_outside,
    build_outside
    == 0,
)


add_check(
    checks,
    "Training-median invalid values",
    0,
    (
        median_nonfinite
        + median_predictor_violations
    ),
    (
        median_nonfinite
        + median_predictor_violations
    )
    == 0,
)


add_check(
    checks,
    "Scored failing-build violations",
    0,
    scored_build_violations,
    scored_build_violations
    == 0,
)


add_check(
    checks,
    "Evaluation-build violations",
    0,
    evaluation_build_violations,
    evaluation_build_violations
    == 0,
)


add_check(
    checks,
    "Evaluation-row violations",
    0,
    evaluation_row_violations,
    evaluation_row_violations
    == 0,
)


add_check(
    checks,
    "Evaluation-failure violations",
    0,
    evaluation_failure_violations,
    evaluation_failure_violations
    == 0,
)


add_check(
    checks,
    "Zero-noise conditions",
    30,
    len(
        zero_noise
    ),
    len(
        zero_noise
    )
    == 30,
)


add_check(
    checks,
    "Zero-noise raw flips",
    0,
    zero_flip_violations,
    zero_flip_violations
    == 0,
)


add_check(
    checks,
    "Zero-noise model-label changes",
    0,
    zero_model_violations,
    zero_model_violations
    == 0,
)


add_check(
    checks,
    "Zero-noise dependent REC changes",
    0,
    zero_rec_violations,
    zero_rec_violations
    == 0,
)


add_check(
    checks,
    "Positive-noise raw-change violations",
    0,
    positive_raw_violations,
    positive_raw_violations
    == 0,
)


add_check(
    checks,
    "Positive-noise model-change violations",
    0,
    positive_model_violations,
    positive_model_violations
    == 0,
)


add_check(
    checks,
    "Positive-noise dependent-REC violations",
    0,
    positive_rec_violations,
    positive_rec_violations
    == 0,
)


add_check(
    checks,
    "Independent REC violations",
    0,
    independent_rec_violations,
    independent_rec_violations
    == 0,
)


add_check(
    checks,
    "Independent reconstruction violations",
    0,
    independent_reconstruction_violations,
    independent_reconstruction_violations
    == 0,
)


add_check(
    checks,
    "Noise-plan hash mismatches",
    0,
    noise_hash_mismatches,
    noise_hash_mismatches
    == 0,
)


add_check(
    checks,
    "Ranking-level baseline-invariance failures",
    0,
    ranking_baseline_failures,
    ranking_baseline_failures
    == 0,
)


add_check(
    checks,
    "Project-metric baseline-invariance failures",
    0,
    metric_baseline_failures,
    metric_baseline_failures
    == 0,
)


add_check(
    checks,
    "Maximum within-metric baseline range",
    0.0,
    metric_baseline_maximum_range,
    metric_baseline_maximum_range
    <= 1e-15,
)


add_check(
    checks,
    "Noise-technique summary rows",
    EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS,
    len(
        noise_summary
    ),
    len(
        noise_summary
    )
    == EXPECTED_NOISE_TECHNIQUE_SUMMARY_ROWS,
)


noise_summary_count_violations = int(
    noise_summary[
        "Runs"
    ].ne(
        30
    ).sum()
    + noise_summary[
        "Seeds"
    ].ne(
        30
    ).sum()
)


add_check(
    checks,
    "Noise-technique summary count/nonfinite violations",
    0,
    (
        noise_summary_count_violations
        + summary_nonfinite
    ),
    (
        noise_summary_count_violations
        + summary_nonfinite
    )
    == 0,
)


add_check(
    checks,
    "Sample SD contract ddof=1",
    True,
    sample_sd_contract_pass,
    sample_sd_contract_pass,
)


add_check(
    checks,
    "Seed-level delta rows",
    EXPECTED_SEED_DELTA_ROWS,
    len(
        seed_deltas
    ),
    len(
        seed_deltas
    )
    == EXPECTED_SEED_DELTA_ROWS,
)


add_check(
    checks,
    "Clean delta non-zero values",
    0,
    clean_delta_nonzero,
    clean_delta_nonzero
    == 0,
)


add_check(
    checks,
    "Invariant-baseline delta non-zero values",
    0,
    invariant_baseline_delta_nonzero,
    invariant_baseline_delta_nonzero
    == 0,
)


add_check(
    checks,
    "Noise-delta summary rows",
    EXPECTED_NOISE_DELTA_SUMMARY_ROWS,
    len(
        delta_summary
    ),
    len(
        delta_summary
    )
    == EXPECTED_NOISE_DELTA_SUMMARY_ROWS,
)


delta_summary_count_violations = int(
    delta_summary[
        "Seeds"
    ].ne(
        30
    ).sum()
)


add_check(
    checks,
    "Noise-delta summary count/nonfinite violations",
    0,
    (
        delta_summary_count_violations
        + delta_nonfinite
    ),
    (
        delta_summary_count_violations
        + delta_nonfinite
    )
    == 0,
)


add_check(
    checks,
    "Registry rows",
    23,
    len(
        registry
    ),
    len(
        registry
    )
    == 23,
)


for number, identity in REQUIRED_REGISTERED_IDENTITIES.items():
    actual_identity = str(
        registry.loc[
            project_numbers.eq(
                number
            ),
            project_col,
        ].iloc[
            0
        ]
    )

    add_check(
        checks,
        f"Project {number} frozen identity",
        identity,
        actual_identity,
        actual_identity
        == identity,
    )


add_check(
    checks,
    "Registry Project 24 rows",
    0,
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    )
    == 0,
)


add_check(
    checks,
    "Models fitted in Step 5B",
    False,
    False,
    True,
)


add_check(
    checks,
    "Conditions rerun in Step 5B",
    False,
    False,
    True,
)


validation = pd.DataFrame(
    checks
)


failed = validation.loc[
    ~validation[
        "Pass"
    ]
]


print(
    "\nProject 24 Step 5B validation:"
)


display(
    validation
)


if not failed.empty:
    print(
        "\nFailed Project 24 Step 5B checks:"
    )

    display(
        failed
    )

    raise RuntimeError(
        "PROJECT 24 STEP 5B VALIDATION FAILED. "
        "No PASS checkpoint was written."
    )


# --------------------------------------------------------------------------------------------------
# 12. FREEZE STEP 5B OUTPUTS
# --------------------------------------------------------------------------------------------------

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


for path, frame in [
    (
        CURRENT_MANIFEST,
        current_manifest,
    ),
    (
        CONDITION_INVENTORY,
        inventory,
    ),
    (
        REVALIDATED_PROJECT_RUNS,
        project_runs,
    ),
    (
        REVALIDATED_BUILD_METRICS,
        build_metrics,
    ),
    (
        REVALIDATED_MODEL_FITS,
        model_fits,
    ),
    (
        REVALIDATED_CONDITION_AUDIT,
        condition_audit,
    ),
    (
        REVALIDATED_MEDIANS,
        training_medians,
    ),
    (
        NOISE_SUMMARY,
        noise_summary,
    ),
    (
        SEED_DELTAS,
        seed_deltas,
    ),
    (
        DELTA_SUMMARY,
        delta_summary,
    ),
    (
        VALIDATION_PATH,
        validation,
    ),
]:
    atomic_csv(
        path,
        frame,
    )


output_paths = [
    CURRENT_MANIFEST,
    CONDITION_INVENTORY,
    REVALIDATED_PROJECT_RUNS,
    REVALIDATED_BUILD_METRICS,
    REVALIDATED_MODEL_FITS,
    REVALIDATED_CONDITION_AUDIT,
    REVALIDATED_MEDIANS,
    NOISE_SUMMARY,
    SEED_DELTAS,
    DELTA_SUMMARY,
    VALIDATION_PATH,
]


output_manifest = [
    {
        "Path":
            str(
                path
            ),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in output_paths
]


completed_at_utc = datetime.now(
    timezone.utc
).isoformat()


report = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5B_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "Step5ACheckpointSHA256":
        step5a_sha,

    "FrozenRawRootSHA256":
        EXPECTED_RAW_ROOT_SHA,

    "IndependentRawRootSHA256":
        current_raw_sha,

    "RawFiles":
        len(
            current_manifest
        ),

    "RawBytes":
        current_raw_bytes,

    "MissingRawFiles":
        missing_raw,

    "UnexpectedRawFiles":
        unexpected_raw,

    "RawSizeMismatches":
        size_mismatch,

    "RawSHA256Mismatches":
        hash_mismatch,

    "Conditions":
        len(
            inventory
        ),

    "MLFits":
        len(
            model_fits
        ),

    "RankingRows":
        int(
            inventory[
                "RankingRows"
            ].sum()
        ),

    "BuildMetricRows":
        len(
            build_metrics
        ),

    "ProjectRunRows":
        len(
            project_runs
        ),

    "ConditionAuditRows":
        len(
            condition_audit
        ),

    "TrainingMedianRows":
        len(
            training_medians
        ),

    "NoiseTechniqueSummaryRows":
        len(
            noise_summary
        ),

    "SeedLevelNoiseDeltaRows":
        len(
            seed_deltas
        ),

    "NoiseDeltaSummaryRows":
        len(
            delta_summary
        ),

    "StandardDeviationDefinition":
        "Sample SD across 30 seeds; pandas std, ddof=1",

    "RankingLevelBaselineInvarianceFailures":
        int(
            ranking_baseline_failures
        ),

    "ProjectMetricBaselineInvarianceFailures":
        int(
            metric_baseline_failures
        ),

    "ProjectMetricBaselineMaximumWithinMetricRange":
        float(
            metric_baseline_maximum_range
        ),

    "RawHashingSeconds":
        float(
            hash_seconds
        ),

    "ConditionRevalidationSeconds":
        float(
            condition_seconds
        ),

    "AggregationSeconds":
        float(
            aggregation_seconds
        ),

    "OutputManifest":
        output_manifest,

    "ValidationChecks":
        len(
            validation
        ),

    "FailedValidationChecks":
        len(
            failed
        ),

    "RegistrySHA256":
        registry_sha_before,

    "RegistryModified":
        False,

    "Projects1To22Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "PriorProjectWriteAttempted":
        False,

    "ModelsFitted":
        False,

    "ConditionsRerun":
        False,

    "SourceRootSHA256":
        EXPECTED_SOURCE_ROOT_SHA,

    "ActiveReservations":
        EXPECTED_ACTIVE_RESERVATIONS,

    "RuntimePriorityRule":
        EXPECTED_RUNTIME_PRIORITY_RULE,
}


atomic_json(
    REPORT_PATH,
    report,
)


checkpoint = {
    **report,

    "CheckpointVersion":
        1,

    "CheckpointType":
        "PROJECT_23_RAW_REVALIDATION_AND_COMPACT_AGGREGATION",

    "RawResultsRevalidated":
        True,

    "CompactAggregatesFrozen":
        True,

    "ReadyForFinalPackageAndRegistration":
        True,

    "NextRequiredStep":
        "PROJECT 24 STEP 5C — FINAL PACKAGE, FREEZE, AND REGISTRATION",
}


atomic_json(
    CHECKPOINT_PATH,
    checkpoint,
)


checkpoint_sha = sha256_file(
    CHECKPOINT_PATH
)


status = {
    "ProjectNumber":
        PROJECT_NUMBER,

    "Project":
        PROJECT_NAME,

    "ProjectSlug":
        PROJECT_SLUG,

    "Status":
        STEP5B_STATUS,

    "CompletedAtUTC":
        completed_at_utc,

    "Step5ACheckpointSHA256":
        step5a_sha,

    "RawRootSHA256":
        current_raw_sha,

    "RawFiles":
        len(
            current_manifest
        ),

    "RawBytes":
        current_raw_bytes,

    "Conditions":
        len(
            inventory
        ),

    "MLFits":
        len(
            model_fits
        ),

    "Checkpoint":
        str(
            CHECKPOINT_PATH
        ),

    "CheckpointSHA256":
        checkpoint_sha,

    "ReadyForFinalPackageAndRegistration":
        True,

    "RegistryModified":
        False,

    "Projects1To22Modified":
        False,

    "PriorProjectConditionOutputsAccessed":
        False,

    "PriorProjectConditionOutputsModified":
        False,

    "ModelsFitted":
        False,

    "ConditionsRerun":
        False,

    "NextRequiredStep":
        "PROJECT 24 STEP 5C — FINAL PACKAGE, FREEZE, AND REGISTRATION",
}


atomic_json(
    STATUS_PATH,
    status,
)


# --------------------------------------------------------------------------------------------------
# 13. READBACK AND IMMUTABILITY
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    CHECKPOINT_PATH
)

status_readback = load_json(
    STATUS_PATH
)

report_readback = load_json(
    REPORT_PATH
)


for label, payload in [
    (
        "checkpoint",
        checkpoint_readback,
    ),
    (
        "status",
        status_readback,
    ),
    (
        "report",
        report_readback,
    ),
]:
    if payload.get(
        "Status"
    ) != STEP5B_STATUS:
        raise RuntimeError(
            f"Project 24 Step 5B {label} readback failed."
        )


if not bool(
    checkpoint_readback.get(
        "ReadyForFinalPackageAndRegistration",
        False,
    )
):
    raise RuntimeError(
        "Step 5B checkpoint does not authorize Step 5C."
    )


for item in output_manifest:
    path = Path(
        item[
            "Path"
        ]
    )

    if (
        not path.is_file()
        or int(
            path.stat().st_size
        )
        != int(
            item[
                "Bytes"
            ]
        )
        or sha256_file(
            path
        )
        != str(
            item[
                "SHA256"
            ]
        )
    ):
        raise RuntimeError(
            f"Step 5B output readback failed: {path}"
        )


registry_sha_after = sha256_file(
    REGISTRY
)


if registry_sha_after != registry_sha_before:
    raise RuntimeError(
        "Completion registry changed during Project 24 Step 5B."
    )


if sha256_file(
    STEP5A_CHECKPOINT
) != EXPECTED_STEP5A_SHA:
    raise RuntimeError(
        "Frozen Step 5A checkpoint changed during Step 5B."
    )


# Final independent raw-root immutability check after all Step 5B writes.
final_raw_paths = sorted(
    [
        path
        for path in RAW_ROOT.rglob(
            "*"
        )
        if path.is_file()
    ],
    key=lambda path:
        path.relative_to(
            RAW_ROOT
        ).as_posix(),
)


if len(
    final_raw_paths
) != EXPECTED_RAW_FILES:
    raise RuntimeError(
        "Raw result file count changed during Step 5B."
    )


final_manifest_rows = [
    {
        "RelativePath":
            path.relative_to(
                RAW_ROOT
            ).as_posix(),

        "Bytes":
            int(
                path.stat().st_size
            ),

        "SHA256":
            sha256_file(
                path
            ),
    }
    for path in final_raw_paths
]


final_raw_manifest = normalize_manifest(
    pd.DataFrame(
        final_manifest_rows
    ),
    "final raw manifest",
)


if (
    int(
        final_raw_manifest[
            "Bytes"
        ].sum()
    )
    != EXPECTED_RAW_BYTES
    or root_hash(
        final_raw_manifest
    )
    != EXPECTED_RAW_ROOT_SHA
):
    raise RuntimeError(
        "Raw result root changed during Project 24 Step 5B."
    )


# --------------------------------------------------------------------------------------------------
# 14. DISPLAY + FINAL RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\nNoise-technique summary:"
)

display(
    noise_summary
)


print(
    "\nNoise-delta summary:"
)

display(
    delta_summary
)


print(
    "\n"
    + "=" * 140
)


print(
    "=== PROJECT 24 CELL 10 / STEP 5B RESULT ==="
)


print(
    "=" * 140
)


print(
    "Project:",
    PROJECT_NAME,
)


print(
    "Project slug:",
    PROJECT_SLUG,
)


print(
    "Step 5A checkpoint SHA-256:",
    step5a_sha,
)


print(
    "Frozen raw-root SHA-256:",
    EXPECTED_RAW_ROOT_SHA,
)


print(
    "Independent current raw-root SHA-256:",
    current_raw_sha,
)


print(
    "\nRaw-output revalidation:"
)


print(
    "Conditions:",
    len(
        inventory
    ),
    "/",
    EXPECTED_CONDITIONS,
)


print(
    "Raw files:",
    len(
        current_manifest
    ),
    "/",
    EXPECTED_RAW_FILES,
)


print(
    "Raw bytes:",
    current_raw_bytes,
    "/",
    EXPECTED_RAW_BYTES,
)


print(
    "Missing / unexpected / size / SHA mismatches:",
    missing_raw,
    "/",
    unexpected_raw,
    "/",
    size_mismatch,
    "/",
    hash_mismatch,
)


print(
    "Embedded output-manifest failures:",
    embedded_manifest_failures,
)


print(
    "\nExperiment totals:"
)


print(
    "ML fits:",
    len(
        model_fits
    ),
    "/",
    EXPECTED_TOTAL_FIT_ROWS,
)


print(
    "Ranking rows:",
    int(
        inventory[
            "RankingRows"
        ].sum()
    ),
    "/",
    EXPECTED_TOTAL_RANKING_ROWS,
)


print(
    "Build-metric rows:",
    len(
        build_metrics
    ),
    "/",
    EXPECTED_TOTAL_BUILD_ROWS,
)


print(
    "Project-run rows:",
    len(
        project_runs
    ),
    "/",
    EXPECTED_TOTAL_PROJECT_ROWS,
)


print(
    "Condition-audit rows:",
    len(
        condition_audit
    ),
    "/",
    EXPECTED_TOTAL_AUDIT_ROWS,
)


print(
    "Training-median rows:",
    len(
        training_medians
    ),
    "/",
    EXPECTED_TOTAL_MEDIAN_ROWS,
)


print(
    "\nAnalysis-ready aggregates:"
)


print(
    "Noise-technique summary rows:",
    len(
        noise_summary
    ),
)


print(
    "Seed-level noise-delta rows:",
    len(
        seed_deltas
    ),
)


print(
    "Noise-delta summary rows:",
    len(
        delta_summary
    ),
)


print(
    "Sample SD calculated with ddof=1:",
    sample_sd_contract_pass,
)


print(
    "Ranking-level baseline-invariance failures:",
    ranking_baseline_failures,
)


print(
    "Project-metric baseline-invariance failures:",
    metric_baseline_failures,
)


print(
    "Maximum within-metric baseline range:",
    metric_baseline_maximum_range,
)


print(
    "\nImmutability and isolation:"
)


print(
    "Completion registry unchanged:",
    registry_sha_after
    == registry_sha_before,
)


print(
    "Registry Project 24 rows:",
    int(
        project_numbers.eq(
            PROJECT_NUMBER
        ).sum()
    ),
)


print(
    "Projects 1–23 modified:",
    0,
)


print(
    "Prior project condition outputs accessed:",
    False,
)


print(
    "Prior project condition outputs modified:",
    False,
)


print(
    "Conditions rerun:",
    False,
)


print(
    "Models fitted:",
    False,
)


print(
    "\nRuntime:"
)


print(
    "Raw hashing seconds:",
    round(
        hash_seconds,
        2,
    ),
)


print(
    "Condition revalidation seconds:",
    round(
        condition_seconds,
        2,
    ),
)


print(
    "Compact aggregation seconds:",
    round(
        aggregation_seconds,
        2,
    ),
)


print(
    "\nValidation:"
)


print(
    "Checks:",
    len(
        validation
    ),
)


print(
    "Failed checks:",
    len(
        failed
    ),
)


print(
    "\nProject 24 Step 5B checkpoint:"
)


print(
    CHECKPOINT_PATH
)


print(
    "Checkpoint SHA-256:",
    checkpoint_sha,
)


print(
    "\nNext required step:",
    "PROJECT 24 STEP 5C — FINAL PACKAGE, FREEZE, AND REGISTRATION",
)


print(
    "\nSTATUS:",
    STEP5B_STATUS,
)


print(
    "=" * 140
)


=== PROJECT 24 CELL 10 / STEP 5B: RAW REVALIDATION AND COMPACT AGGREGATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Completion-registry ProjectNumber-24 rows: 0
Completion-registry exact project-name matches (diagnostic only): 0

Independently hashing all 2,160 raw files.
  Raw hashing progress: 200 / 2160 files
  Raw hashing progress: 400 / 2160 files
  Raw hashing progress: 600 / 2160 files
  Raw hashing progress: 800 / 2160 files
  Raw hashing progress: 1000 / 2160 files
  Raw hashing progress: 1200 / 2160 files
  Raw hashing progress: 1400 / 2160 files
  Raw hashing progress: 1600 / 2160 files
  Raw hashing progress: 1800 / 2160 files
  Raw hashing progress: 2000 / 2160 files
  Raw hashing progress: 2160 / 2160 files

Revalidating all 270 condition directories.
  Condition revalidation progress: 30 / 270
  Condition revalidation progress: 60 / 270
  Condition revalidation progress: 90 / 270


,Check,Expected,Actual,Pass
0,Step 5A status,PASS_PROJECT_24_FULL_270_CONDITION_EXPERIMENT_...,PASS_PROJECT_24_FULL_270_CONDITION_EXPERIMENT_...,True
1,Step 5A checkpoint SHA-256,d7abb32056cc47a4b304794dcc2d59477a18c7d42effda...,d7abb32056cc47a4b304794dcc2d59477a18c7d42effda...,True
2,Frozen raw-root SHA-256,6968c094a564405fe23e02f9a5b1ed01cce130ea000dd3...,6968c094a564405fe23e02f9a5b1ed01cce130ea000dd3...,True
3,Independent current raw-root SHA-256,6968c094a564405fe23e02f9a5b1ed01cce130ea000dd3...,6968c094a564405fe23e02f9a5b1ed01cce130ea000dd3...,True
4,Step 5A aggregate-manifest failures,0,0,True
...,...,...,...,...
76,Project 22 frozen identity,apache@logging-log4j2,apache@logging-log4j2,True
77,Project 23 frozen identity,apache@sling,apache@sling,True
78,Registry Project 24 rows,0,0,True
79,Models fitted in Step 5B,False,False,True



Noise-technique summary:


,NoisePercent,Technique,Runs,Seeds,Mean_MeanAPFDc,SD_MeanAPFDc,Median_MeanAPFDc,Mean_MedianAPFDc,SD_MedianAPFDc,Median_MedianAPFDc,Mean_MeanAPFD,SD_MeanAPFD,Median_MeanAPFD,Mean_MedianAPFD,SD_MedianAPFD,Median_MedianAPFD
0,0,LatestFail,30,30,0.528929,0.000000,0.528929,0.554260,0.000000,0.554260,0.137540,0.000000,0.137540,0.061533,0.000000,0.061533
1,0,LightGBM,30,30,0.660968,0.037534,0.662759,0.699993,0.075543,0.720063,0.923647,0.013294,0.926419,0.982553,0.006327,0.984522
2,0,NaiveBayes,30,30,0.232200,0.000000,0.232200,0.160492,0.000000,0.160492,0.546040,0.000000,0.546040,0.565542,0.000000,0.565542
3,0,QTF-Avg,30,30,0.803867,0.000000,0.803867,0.875686,0.000000,0.875686,0.319966,0.000000,0.319966,0.104788,0.000000,0.104788
4,0,Random,30,30,0.497806,0.063769,0.486153,0.493544,0.083302,0.502048,0.497524,0.072509,0.494125,0.495016,0.105704,0.501588
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,30,0.461239,0.252153,0.269706,0.449908,0.379962,0.222052,0.481470,0.270973,0.298044,0.474386,0.429997,0.133987
59,50,QTF-Avg,30,30,0.803867,0.000000,0.803867,0.875686,0.000000,0.875686,0.319966,0.000000,0.319966,0.104788,0.000000,0.104788
60,50,Random,30,30,0.497806,0.063769,0.486153,0.493544,0.083302,0.502048,0.497524,0.072509,0.494125,0.495016,0.105704,0.501588
61,50,RandomForest,30,30,0.526894,0.055121,0.534225,0.563906,0.083278,0.555907,0.488845,0.067786,0.482164,0.515465,0.109329,0.501549



Noise-delta summary:


,NoisePercent,Technique,Seeds,Mean_Delta_MeanAPFDc,SD_Delta_MeanAPFDc,Median_Delta_MeanAPFDc,Mean_Delta_MedianAPFDc,SD_Delta_MedianAPFDc,Median_Delta_MedianAPFDc,Mean_Delta_MeanAPFD,SD_Delta_MeanAPFD,Median_Delta_MeanAPFD,Mean_Delta_MedianAPFD,SD_Delta_MedianAPFD,Median_Delta_MedianAPFD
0,0,LatestFail,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,LightGBM,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,NaiveBayes,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,50,NaiveBayes,30,0.229039,0.252153,0.037506,0.289416,0.379962,0.061560,-0.064571,0.270973,-0.247996,-0.091155,0.429997,-0.431555
59,50,QTF-Avg,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
60,50,Random,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
61,50,RandomForest,30,-0.051239,0.068074,-0.039293,-0.023057,0.109706,-0.023073,-0.314470,0.075072,-0.318157,-0.463459,0.108174,-0.474251



=== PROJECT 24 CELL 10 / STEP 5B RESULT ===
Project: SonarSource@sonarqube
Project slug: SonarSource__sonarqube
Step 5A checkpoint SHA-256: d7abb32056cc47a4b304794dcc2d59477a18c7d42effdacfcf1e58c97056d1a1
Frozen raw-root SHA-256: 6968c094a564405fe23e02f9a5b1ed01cce130ea000dd387f7a3f6f12b421d09
Independent current raw-root SHA-256: 6968c094a564405fe23e02f9a5b1ed01cce130ea000dd387f7a3f6f12b421d09

Raw-output revalidation:
Conditions: 270 / 270
Raw files: 2160 / 2160
Raw bytes: 523093565 / 523093565
Missing / unexpected / size / SHA mismatches: 0 / 0 / 0 / 0
Embedded output-manifest failures: 0

Experiment totals:
ML fits: 1080 / 1080
Ranking rows: 35634060 / 35634060
Build-metric rows: 32130 / 32130
Project-run rows: 1890 / 1890
Condition-audit rows: 270 / 270
Training-median rows: 40770 / 40770

Analysis-ready aggregates:
Noise-technique summary rows: 63
Seed-level noise-delta rows: 1890
Noise-delta summary rows: 63
Sample SD calculated with ddof=1: True
Ranking-level baseline-invarian

In [5]:
# ==================================================================================================
# PROJECT 24 — CELL 11 / STEP 5C
# REGISTRY-SCHEMA-COMPLETE, CROSS-FILESYSTEM-SAFE FINAL PACKAGE AND REGISTRATION
#
# PROJECT:
#   apache@sling
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_24.ipynb.
#
# REQUIRED FROZEN INPUTS:
# - Project 24 Step 5A checkpoint SHA-256:
#   600035cfbf9f8b2dfd0fd83d49050466e3c71e6187c1d631b24cfe57a6107a26
# - Project 24 Step 5B checkpoint SHA-256:
#   9259c486debcda1794423bd16d4333083e14d203216df596327ec4658bfb04fb
# - Project 24 raw-root SHA-256:
#   374e4e1eaab266451539c9c7a4751fa6fe50250dccf57b19a5bc19cf8fad3f11
# - Project 24 source-root SHA-256:
#   281c3d80cab88595c49b2d98c0ce9074d06f5b96872b64f842964c896c6ac334
# - Registry before registration:
#   exactly Projects 1–21, all COMPLETE_AND_FROZEN
# - Registry SHA-256 before registration:
#   79cd6ecb595c5e8ae91a9494e469792716338d144308560a62caf1b9342306b2
#
# SAFETY:
# - no model fitting;
# - no condition reruns;
# - no raw-result modification or deletion;
# - no prior-project condition-output access or write;
# - registry write only after package and candidate-row validation;
# - cross-filesystem-safe Google Drive staging and readback;
# - six frozen Project 24 Step 5A worker checkpoints are preserved and revalidated.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from IPython.display import display
import hashlib
import json
import os
import re
import shutil
import tempfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 24 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN IDENTITY, HASHES, AND COUNTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

STEP5C_STATUS = "PASS_PROJECT_24_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
STEP5A_STATUS = "PASS_PROJECT_24_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
STEP5B_STATUS = "PASS_PROJECT_24_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"
STEP5C_CODE_REVISION = "PROJECT_24_STEP5C_V1_REGISTRY_24_ROW_COMPLETE_CROSS_FILESYSTEM_SAFE"

REGISTRY_SHA_BEFORE_EXPECTED = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

STEP5A_SHA_EXPECTED = (
    "d7abb32056cc47a4b304794dcc2d59477a18c7d42effdacfcf1e58c97056d1a1"
)

STEP5B_SHA_EXPECTED = (
    "3117c9637893b030d67ef4e7743d08a37f0787ec7037cbd27b8f8ca519ddc6c8"
)

SOURCE_ROOT_SHA = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)

RAW_ROOT_SHA = (
    "6968c094a564405fe23e02f9a5b1ed01cce130ea000dd387f7a3f6f12b421d09"
)

COUNTS = {
    "RawFiles": 2160,
    "RawBytes": 523093565,
    "Conditions": 270,
    "MLFits": 1080,
    "RankingRows": 35634060,
    "BuildMetricRows": 32130,
    "ProjectRunRows": 1890,
    "ConditionAuditRows": 270,
    "TrainingMedianRows": 40770,

    "Builds": 4286,
    "TrainingBuilds": 3214,
    "EvaluationBuilds": 1072,

    "RawRows": 5635027,
    "RawTrainingRows": 4040801,
    "RawEvaluationRows": 1594226,
    "RawTrainingFailures": 1778,
    "RawEvaluationFailures": 20,

    "ModelRows": 224550,
    "ModelTrainingRows": 205696,
    "ModelEvaluationRows": 18854,
    "ModelTrainingFailures": 1777,
    "ModelEvaluationFailures": 20,
    "ModelFailingEvaluationBuilds": 17,

    "Predictors": 151,
    "RECFeatures": 19,
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False,
)

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = (
    ROOT
    / "Notes"
)

RESULTS = (
    ROOT
    / "Results"
)

REGISTRY = (
    NOTES
    / "completed_project_registry.csv"
)

PROJECT_ROOT = (
    RESULTS
    / "Aggregated"
    / PROJECT_SLUG
)

RAW_ROOT = (
    RESULTS
    / "Raw"
    / PROJECT_SLUG
)

FINAL_ROOT = (
    RESULTS
    / "Final"
    / PROJECT_SLUG
)

MANIFEST_PATH = (
    FINAL_ROOT
    / "final_package_manifest.csv"
)

SUMMARY_PATH = (
    FINAL_ROOT
    / "final_package_summary.json"
)

README_PATH = (
    FINAL_ROOT
    / "README.txt"
)

STEP5C_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5c"
)

VALIDATION_PATH = (
    STEP5C_ROOT
    / f"{PROJECT_SHORT}_step5c_validation.csv"
)

REPORT_PATH = (
    STEP5C_ROOT
    / f"{PROJECT_SHORT}_step5c_report.json"
)

STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5c_status.json"
)

CHECKPOINT_PATH = (
    NOTES
    / "project_24_step5c_checkpoint.json"
)

BACKUP_PATH = (
    NOTES
    / "completed_project_registry_before_project_24.csv"
)

STEP5B_RAW_MANIFEST_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b"
    / f"{PROJECT_SHORT}_independent_raw_manifest.csv"
)

STEP5A_CP = (
    NOTES
    / "project_24_step5a_checkpoint.json"
)

STEP5B_CP = (
    NOTES
    / "project_24_step5b_checkpoint.json"
)

STEP5A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5a_status.json"
)

STEP5B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b_status.json"
)

UPSTREAM_CPS = [
    NOTES / "project_24_selection_checkpoint.json",
    NOTES / "project_24_rec_reconstruction_checkpoint.json",
    NOTES / "project_24_noise_plan_checkpoint.json",
    NOTES / "project_24_runtime_contract_checkpoint.json",
    NOTES / "project_24_smoke_test_checkpoint.json",
    STEP5A_CP,
    STEP5B_CP,
]

WORKER_CHECKPOINT_PATHS = {
    "seed_01_05":
        NOTES / "project_24_step5a_worker_seed_01_05_checkpoint.json",
    "seed_06_10":
        NOTES / "project_24_step5a_worker_seed_06_10_checkpoint.json",
    "seed_11_15":
        NOTES / "project_24_step5a_worker_seed_11_15_checkpoint.json",
    "seed_16_20":
        NOTES / "project_24_step5a_worker_seed_16_20_checkpoint.json",
    "seed_21_25":
        NOTES / "project_24_step5a_worker_seed_21_25_checkpoint.json",
    "seed_26_30":
        NOTES / "project_24_step5a_worker_seed_26_30_checkpoint.json",
}



# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha(
    path,
    chunk=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:
        for block in iter(
            lambda: f.read(
                chunk
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(
            f
        )


def atomic_json(
    path,
    obj,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        f.write(
            "\n"
        )

    os.replace(
        tmp,
        path,
    )


def atomic_csv(
    path,
    df,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    df.to_csv(
        tmp,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        tmp,
        path,
    )


def atomic_text(
    path,
    text,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    tmp.write_text(
        text,
        encoding="utf-8",
    )

    os.replace(
        tmp,
        path,
    )


def resolve(
    cols,
    expected,
):
    matches = [
        c
        for c in cols
        if str(
            c
        ).strip().lower()
        == expected.lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Could not resolve registry column {expected!r}; "
            f"matches={matches}; columns={list(cols)}"
        )

    return matches[
        0
    ]


def norm(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value
        ).lower(),
    )


def manifest(
    root,
    exclude=(),
):
    root = Path(
        root
    )

    exclude = set(
        exclude
    )

    rows = []

    files = sorted(
        (
            path
            for path in root.rglob(
                "*"
            )
            if path.is_file()
        ),
        key=lambda path:
            path.relative_to(
                root
            ).as_posix(),
    )

    for path in files:
        rel = path.relative_to(
            root
        ).as_posix()

        if rel in exclude:
            continue

        rows.append(
            {
                "RelativePath": rel,
                "Bytes": int(
                    path.stat().st_size
                ),
                "SHA256": sha(
                    path
                ),
            }
        )

    return pd.DataFrame(
        rows,
        columns=[
            "RelativePath",
            "Bytes",
            "SHA256",
        ],
    )


def root_hash(
    df,
):
    h = hashlib.sha256()

    ordered = df.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        h.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode()
        )

    return h.hexdigest()


def verify_manifest(
    items,
    label,
):
    if (
        not isinstance(
            items,
            list,
        )
        or not items
    ):
        raise RuntimeError(
            f"{label} has no output manifest."
        )

    rows = []

    for item in items:
        path = Path(
            item[
                "Path"
            ]
        )

        exists = path.is_file()

        expected_bytes = int(
            item[
                "Bytes"
            ]
        )

        expected_sha = str(
            item[
                "SHA256"
            ]
        ).lower()

        if exists:
            actual_bytes = int(
                path.stat().st_size
            )

            actual_sha = sha(
                path
            )
        else:
            actual_bytes = -1
            actual_sha = "MISSING"

        rows.append(
            {
                "Path": str(
                    path
                ),
                "ExpectedBytes": expected_bytes,
                "ActualBytes": actual_bytes,
                "ExpectedSHA256": expected_sha,
                "ActualSHA256": actual_sha,
                "Pass": bool(
                    exists
                    and expected_bytes
                    == actual_bytes
                    and expected_sha
                    == actual_sha
                ),
            }
        )

    out = pd.DataFrame(
        rows
    )

    if not out[
        "Pass"
    ].all():
        display(
            out.loc[
                ~out[
                    "Pass"
                ]
            ]
        )

        raise RuntimeError(
            f"{label} manifest verification failed."
        )

    return out


def check(
    rows,
    name,
    expected,
    actual,
    passed,
):
    rows.append(
        {
            "Check": name,
            "Expected": expected,
            "Actual": actual,
            "Pass": bool(
                passed
            ),
        }
    )


# --------------------------------------------------------------------------------------------------
# 4. VERIFY ALL REQUIRED INPUTS BEFORE ANY PACKAGE OR REGISTRY WRITE
# --------------------------------------------------------------------------------------------------

required = [
    REGISTRY,
    RAW_ROOT,
    STEP5A_CP,
    STEP5B_CP,
    STEP5A_STATUS_PATH,
    STEP5B_STATUS_PATH,
    *UPSTREAM_CPS,
    *WORKER_CHECKPOINT_PATHS.values(),
]

missing = [
    str(
        path
    )
    for path in required
    if not Path(
        path
    ).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing Project 24 Step 5C inputs:\n"
        + "\n".join(
            missing
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 5A / STEP 5B FREEZES
# --------------------------------------------------------------------------------------------------

step5a_sha = sha(
    STEP5A_CP
)

step5b_sha = sha(
    STEP5B_CP
)

if step5a_sha != STEP5A_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5A checkpoint SHA differs.\n"
        f"Expected: {STEP5A_SHA_EXPECTED}\n"
        f"Actual:   {step5a_sha}"
    )

if step5b_sha != STEP5B_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5B checkpoint SHA differs.\n"
        f"Expected: {STEP5B_SHA_EXPECTED}\n"
        f"Actual:   {step5b_sha}"
    )

step5a = load_json(
    STEP5A_CP
)

step5b = load_json(
    STEP5B_CP
)

step5a_status_payload = load_json(
    STEP5A_STATUS_PATH
)

step5b_status_payload = load_json(
    STEP5B_STATUS_PATH
)

for label, payload, expected in [
    (
        "Step 5A checkpoint",
        step5a,
        STEP5A_STATUS,
    ),
    (
        "Step 5A status",
        step5a_status_payload,
        STEP5A_STATUS,
    ),
    (
        "Step 5B checkpoint",
        step5b,
        STEP5B_STATUS,
    ),
    (
        "Step 5B status",
        step5b_status_payload,
        STEP5B_STATUS,
    ),
]:
    if payload.get(
        "Status"
    ) != expected:
        raise RuntimeError(
            f"{label} is not in expected PASS state."
        )

if (
    step5a.get(
        "SourceRootSHA256"
    )
    != SOURCE_ROOT_SHA
):
    raise RuntimeError(
        "Step 5A source-root SHA differs."
    )

if (
    step5a.get(
        "RawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5A raw-root SHA differs."
    )

if (
    step5b.get(
        "SourceRootSHA256"
    )
    != SOURCE_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B source-root SHA differs."
    )

if (
    step5b.get(
        "FrozenRawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B frozen raw-root SHA differs."
    )

if (
    step5b.get(
        "IndependentRawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B independent raw-root SHA differs."
    )

if (
    step5b.get(
        "Step5ACheckpointSHA256"
    )
    != STEP5A_SHA_EXPECTED
):
    raise RuntimeError(
        "Step 5B does not link to the frozen Step 5A checkpoint."
    )

if not bool(
    step5b.get(
        "ReadyForFinalPackageAndRegistration",
        False,
    )
):
    raise RuntimeError(
        "Step 5B is not marked ready for final package and registration."
    )

if bool(
    step5b.get(
        "RegistryModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports that the completion registry was modified."
    )

if bool(
    step5b.get(
        "PriorProjectConditionOutputsAccessed",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports access to prior-project condition outputs."
    )

if bool(
    step5b.get(
        "PriorProjectConditionOutputsModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports modification of prior-project condition outputs."
    )

if bool(
    step5b.get(
        "PriorProjectWriteAttempted",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports a prior-project write attempt."
    )

if bool(
    step5b.get(
        "ModelsFitted",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports model fitting."
    )

if bool(
    step5b.get(
        "ConditionsRerun",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports condition reruns."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY SIX FROZEN PARALLEL WORKER CHECKPOINTS
# --------------------------------------------------------------------------------------------------
#
# Project 24 had one resume-safe worker metadata refresh before official Step 5A.
# Therefore the authoritative current worker checkpoint SHA map is the ACTUAL map frozen
# in the official Step 5A master checkpoint.
# --------------------------------------------------------------------------------------------------

expected_worker_hashes_by_tag = step5a.get(
    "ActualWorkerCheckpointSHA256",
    {},
)

if (
    not isinstance(
        expected_worker_hashes_by_tag,
        dict,
    )
    or sorted(
        expected_worker_hashes_by_tag
    )
    != sorted(
        WORKER_CHECKPOINT_PATHS
    )
):
    raise RuntimeError(
        "Step 5A ActualWorkerCheckpointSHA256 does not contain "
        "the exact six Project 24 worker tags."
    )

step5a_worker_records = step5a.get(
    "WorkerCheckpoints",
    [],
)

if (
    not isinstance(
        step5a_worker_records,
        list,
    )
    or len(
        step5a_worker_records
    )
    != 6
):
    raise RuntimeError(
        "Step 5A WorkerCheckpoints does not contain exactly six records."
    )

worker_records_by_tag = {
    str(
        record.get(
            "WorkerTag",
            "",
        )
    ):
        record
    for record in step5a_worker_records
}

if sorted(
    worker_records_by_tag
) != sorted(
    WORKER_CHECKPOINT_PATHS
):
    raise RuntimeError(
        "Step 5A WorkerCheckpoints tags differ from the expected six worker shards."
    )

worker_checkpoint_sha_mismatches = 0
worker_scientific_freeze_failures = 0

for tag, worker_path in WORKER_CHECKPOINT_PATHS.items():
    expected_worker_sha = str(
        expected_worker_hashes_by_tag[
            tag
        ]
    )

    actual_worker_sha = sha(
        worker_path
    )

    worker_checkpoint_sha_mismatches += int(
        actual_worker_sha
        != expected_worker_sha
    )

    worker_payload = load_json(
        worker_path
    )

    master_record = worker_records_by_tag[
        tag
    ]

    worker_scientific_freeze_failures += int(
        worker_payload.get(
            "Status"
        )
        != master_record.get(
            "Status"
        )
    )

    worker_scientific_freeze_failures += int(
        str(
            worker_payload.get(
                "WorkerRawRootSHA256",
                "",
            )
        )
        != str(
            master_record.get(
                "WorkerRawRootSHA256",
                "",
            )
        )
    )

    worker_scientific_freeze_failures += int(
        int(
            worker_payload.get(
                "WorkerRawFiles",
                -1,
            )
        )
        != int(
            master_record.get(
                "WorkerRawFiles",
                -2,
            )
        )
    )

    worker_scientific_freeze_failures += int(
        int(
            worker_payload.get(
                "WorkerRawBytes",
                -1,
            )
        )
        != int(
            master_record.get(
                "WorkerRawBytes",
                -2,
            )
        )
    )

if worker_checkpoint_sha_mismatches:
    raise RuntimeError(
        f"{worker_checkpoint_sha_mismatches} frozen Project 24 worker "
        "checkpoint SHA values changed after official Step 5A."
    )

if worker_scientific_freeze_failures:
    raise RuntimeError(
        f"{worker_scientific_freeze_failures} Project 24 worker scientific "
        "freeze fields differ from official Step 5A."
    )

# --------------------------------------------------------------------------------------------------
# 7. VERIFY STEP 5A / STEP 5B OUTPUT MANIFESTS
# --------------------------------------------------------------------------------------------------

step5a_audit = verify_manifest(
    step5a.get(
        "AggregateOutputManifest",
        [],
    ),
    "Step 5A aggregate",
)

step5b_audit = verify_manifest(
    step5b.get(
        "OutputManifest",
        [],
    ),
    "Step 5B",
)


# --------------------------------------------------------------------------------------------------
# 8. VERIFY COMPLETION REGISTRY PRE-STATE
# --------------------------------------------------------------------------------------------------

registry_sha_before = sha(
    REGISTRY
)

if (
    registry_sha_before
    != REGISTRY_SHA_BEFORE_EXPECTED
):
    raise RuntimeError(
        "Registry SHA differs before Project 24 registration:\n"
        f"Expected: {REGISTRY_SHA_BEFORE_EXPECTED}\n"
        f"Actual:   {registry_sha_before}"
    )

if (
    step5b.get(
        "RegistrySHA256"
    )
    != REGISTRY_SHA_BEFORE_EXPECTED
):
    raise RuntimeError(
        "The frozen Step 5B checkpoint does not reference "
        "the expected pre-Project-24 registry SHA."
    )

reg_before = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna(
    ""
)

pn_col = resolve(
    reg_before.columns,
    "ProjectNumber",
)

project_col = resolve(
    reg_before.columns,
    "Project",
)

status_col = resolve(
    reg_before.columns,
    "Status",
)

pnums = pd.to_numeric(
    reg_before[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        reg_before
    )
    != 23
    or sorted(
        pnums.tolist()
    )
    != list(
        range(
            1,
            24,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–23."
    )

if not reg_before[
    status_col
].eq(
    COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}

for (
    required_number,
    required_project,
) in required_registered_identities.items():
    matching_rows = reg_before.loc[
        pnums.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_col
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

project_24_registry_rows_before = int(
    pnums.eq(
        PROJECT_NUMBER
    ).sum()
)

project_name_match_rows_diagnostic = int(
    reg_before[
        project_col
    ].eq(
        PROJECT_NAME
    ).sum()
)

if project_24_registry_rows_before != 0:
    raise RuntimeError(
        "Project 24 is already present in the completion registry by ProjectNumber."
    )

print(
    "Completion-registry ProjectNumber-24 rows:",
    project_24_registry_rows_before,
)

print(
    "Completion-registry exact project-name matches (diagnostic only):",
    project_name_match_rows_diagnostic,
)


# --------------------------------------------------------------------------------------------------
# 8B. FINAL DIRECT RAW-ROOT IMMUTABILITY VERIFICATION
# --------------------------------------------------------------------------------------------------

if not STEP5B_RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The independently frozen Step 5B raw-results manifest is missing:\n"
        f"{STEP5B_RAW_MANIFEST_PATH}"
    )

step5b_raw_manifest = pd.read_csv(
    STEP5B_RAW_MANIFEST_PATH,
    low_memory=False,
)

required_raw_manifest_columns = {
    "RelativePath",
    "Bytes",
    "SHA256",
}

if not required_raw_manifest_columns.issubset(
    step5b_raw_manifest.columns
):
    raise RuntimeError(
        "Step 5B raw manifest is missing required columns."
    )

step5b_raw_manifest = (
    step5b_raw_manifest[
        [
            "RelativePath",
            "Bytes",
            "SHA256",
        ]
    ]
    .copy()
)

step5b_raw_manifest[
    "RelativePath"
] = step5b_raw_manifest[
    "RelativePath"
].astype(
    str
)

step5b_raw_manifest[
    "Bytes"
] = pd.to_numeric(
    step5b_raw_manifest[
        "Bytes"
    ],
    errors="raise",
).astype(
    "int64"
)

step5b_raw_manifest[
    "SHA256"
] = (
    step5b_raw_manifest[
        "SHA256"
    ]
    .astype(
        str
    )
    .str.lower()
)

raw_file_failures = 0

for raw_row in step5b_raw_manifest.itertuples(
    index=False
):
    raw_path = (
        RAW_ROOT
        / str(
            raw_row.RelativePath
        )
    )

    if not raw_path.is_file():
        raw_file_failures += 1
        continue

    if int(
        raw_path.stat().st_size
    ) != int(
        raw_row.Bytes
    ):
        raw_file_failures += 1
        continue

    if sha(
        raw_path
    ) != str(
        raw_row.SHA256
    ):
        raw_file_failures += 1

if (
    len(
        step5b_raw_manifest
    )
    != COUNTS[
        "RawFiles"
    ]
    or int(
        step5b_raw_manifest[
            "Bytes"
        ].sum()
    )
    != COUNTS[
        "RawBytes"
    ]
    or root_hash(
        step5b_raw_manifest
    )
    != RAW_ROOT_SHA
    or raw_file_failures
    != 0
):
    raise RuntimeError(
        "Project 24 raw result root failed final pre-registration "
        "immutability verification."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE A PRE-PROJECT-23 REGISTRY BACKUP
# --------------------------------------------------------------------------------------------------

if not BACKUP_PATH.exists():
    shutil.copy2(
        REGISTRY,
        BACKUP_PATH,
    )

if sha(
    BACKUP_PATH
) != registry_sha_before:
    raise RuntimeError(
        "Pre-Project-24 registry backup does not match the live registry."
    )


# --------------------------------------------------------------------------------------------------
# 10. ASSEMBLE FINAL COMPACT PACKAGE
# --------------------------------------------------------------------------------------------------

sources = set(
    Path(
        path
    )
    for path in UPSTREAM_CPS
)

sources.update(
    WORKER_CHECKPOINT_PATHS.values()
)

sources.update(
    [
        STEP5A_STATUS_PATH,
        STEP5B_STATUS_PATH,
    ]
)

sources.update(
    Path(
        item[
            "Path"
        ]
    )
    for item in step5a.get(
        "AggregateOutputManifest",
        [],
    )
)

sources.update(
    Path(
        item[
            "Path"
        ]
    )
    for item in step5b.get(
        "OutputManifest",
        [],
    )
)

sources = sorted(
    sources,
    key=str,
)

missing_sources = [
    str(
        path
    )
    for path in sources
    if not path.is_file()
]

if missing_sources:
    raise FileNotFoundError(
        "Missing compact-package sources:\n"
        + "\n".join(
            missing_sources
        )
    )

created_at = str(
    step5b.get(
        "CompletedAtUTC",
        "",
    )
).strip()

if not created_at:
    raise RuntimeError(
        "The frozen Step 5B checkpoint contains no CompletedAtUTC timestamp."
    )

tmp_root = Path(
    tempfile.mkdtemp(
        prefix="project24_package_",
        dir="/content",
    )
)

drive_staging_root = None

try:
    for src in sources:
        try:
            rel = src.relative_to(
                ROOT
            )
        except ValueError as exc:
            raise RuntimeError(
                f"Package source is outside thesis root: {src}"
            ) from exc

        dst = (
            tmp_root
            / rel
        )

        dst.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            src,
            dst,
        )

    atomic_text(
        tmp_root
        / "README.txt",
        f"""PROJECT 24 FINAL COMPACT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}
Status: {COMPLETE_STATUS}
Created at UTC: {created_at}

The 2,160 raw condition-output files are not duplicated here.
Raw results: {RAW_ROOT}
Raw-root SHA-256: {RAW_ROOT_SHA}
Source-root SHA-256: {SOURCE_ROOT_SHA}
Primary metric: APFDc
Secondary metric: APFD
Parallel Step 5A worker checkpoints preserved in this package: 6
""",
    )

    before_summary = manifest(
        tmp_root
    )

    atomic_json(
        tmp_root
        / "final_package_summary.json",
        {
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "Status": COMPLETE_STATUS,
            "CreatedAtUTC": created_at,
            "SourceRootSHA256": SOURCE_ROOT_SHA,
            "RawRootSHA256": RAW_ROOT_SHA,
            **COUNTS,
            "Step5ACheckpointSHA256": step5a_sha,
            "Step5BCheckpointSHA256": step5b_sha,
            "PayloadRootSHA256BeforeSummary": root_hash(
                before_summary
            ),
            "RawResultsDuplicatedIntoPackage": False,
            "ParallelWorkerCheckpointSHA256":
                expected_worker_hashes_by_tag,
            "PriorProjectConditionOutputsAccessed": False,
            "PriorProjectConditionOutputsModified": False,
            "PriorProjectWriteAttempted": False,
        },
    )

    candidate_manifest = manifest(
        tmp_root,
        {
            "final_package_manifest.csv",
        },
    )

    package_root_sha = root_hash(
        candidate_manifest
    )

    atomic_csv(
        tmp_root
        / "final_package_manifest.csv",
        candidate_manifest,
    )

    package_files = (
        len(
            candidate_manifest
        )
        + 1
    )

    package_bytes = int(
        candidate_manifest[
            "Bytes"
        ].sum()
        + (
            tmp_root
            / "final_package_manifest.csv"
        ).stat().st_size
    )

    if FINAL_ROOT.exists():
        if not MANIFEST_PATH.is_file():
            raise RuntimeError(
                "An existing Project 24 final-package directory has no manifest "
                "and was not modified."
            )

        existing = pd.read_csv(
            MANIFEST_PATH,
            low_memory=False,
        )

        if root_hash(
            existing
        ) != package_root_sha:
            raise RuntimeError(
                "A different Project 24 final package already exists "
                "and was not modified."
            )

        shutil.rmtree(
            tmp_root
        )

        package_already_frozen = True

    else:
        # /content and Google Drive are different filesystems.
        # Publish in two stages:
        #   1. copy completed local package to a sibling Drive staging directory;
        #   2. verify the staged package exactly;
        #   3. rename staging -> FINAL_ROOT within Google Drive.
        FINAL_ROOT.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        drive_staging_root = FINAL_ROOT.with_name(
            f"{FINAL_ROOT.name}__staging_{os.getpid()}"
        )

        if drive_staging_root.exists():
            shutil.rmtree(
                drive_staging_root
            )

        shutil.copytree(
            tmp_root,
            drive_staging_root,
            copy_function=shutil.copy2,
        )

        staged_manifest_path = (
            drive_staging_root
            / "final_package_manifest.csv"
        )

        if not staged_manifest_path.is_file():
            raise RuntimeError(
                "The Google Drive staging package has no manifest."
            )

        staged_manifest = pd.read_csv(
            staged_manifest_path,
            low_memory=False,
        )

        staged_root_sha = root_hash(
            staged_manifest
        )

        if staged_root_sha != package_root_sha:
            raise RuntimeError(
                "The Google Drive staging package root SHA-256 differs.\n"
                f"Expected: {package_root_sha}\n"
                f"Actual:   {staged_root_sha}"
            )

        staged_payload_manifest = manifest(
            drive_staging_root,
            {
                "final_package_manifest.csv",
            },
        )

        candidate_payload_manifest = (
            candidate_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        staged_payload_manifest = (
            staged_payload_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        if not staged_payload_manifest.equals(
            candidate_payload_manifest
        ):
            comparison = candidate_payload_manifest.merge(
                staged_payload_manifest,
                on="RelativePath",
                how="outer",
                suffixes=(
                    "_candidate",
                    "_staged",
                ),
                indicator=True,
            )

            failed_comparison = comparison.loc[
                (
                    comparison[
                        "_merge"
                    ].ne(
                        "both"
                    )
                    | comparison[
                        "Bytes_candidate"
                    ].ne(
                        comparison[
                            "Bytes_staged"
                        ]
                    )
                    | comparison[
                        "SHA256_candidate"
                    ].ne(
                        comparison[
                            "SHA256_staged"
                        ]
                    )
                )
            ]

            raise RuntimeError(
                "The Google Drive staging package failed exact file-level "
                "verification.\n"
                + failed_comparison.head(
                    20
                ).to_string(
                    index=False
                )
            )

        os.replace(
            drive_staging_root,
            FINAL_ROOT,
        )

        drive_staging_root = None

        shutil.rmtree(
            tmp_root
        )

        package_already_frozen = False

except Exception:
    if tmp_root.exists():
        shutil.rmtree(
            tmp_root,
            ignore_errors=True,
        )

    if (
        drive_staging_root is not None
        and drive_staging_root.exists()
    ):
        shutil.rmtree(
            drive_staging_root,
            ignore_errors=True,
        )

    raise


# --------------------------------------------------------------------------------------------------
# 11. READ BACK EVERY FINAL-PACKAGE FILE
# --------------------------------------------------------------------------------------------------

pkg_manifest = pd.read_csv(
    MANIFEST_PATH,
    low_memory=False,
)

manifest_paths = set(
    pkg_manifest[
        "RelativePath"
    ].astype(
        str
    )
)

actual_paths = {
    path.relative_to(
        FINAL_ROOT
    ).as_posix()
    for path in FINAL_ROOT.rglob(
        "*"
    )
    if path.is_file()
}

expected_paths = (
    manifest_paths
    | {
        "final_package_manifest.csv",
    }
)

missing_pkg = len(
    expected_paths
    - actual_paths
)

unexpected_pkg = len(
    actual_paths
    - expected_paths
)

size_bad = 0
hash_bad = 0

for row in pkg_manifest.itertuples(
    index=False
):
    path = (
        FINAL_ROOT
        / str(
            row.RelativePath
        )
    )

    if path.is_file():
        size_bad += int(
            path.stat().st_size
            != int(
                row.Bytes
            )
        )

        hash_bad += int(
            sha(
                path
            )
            != str(
                row.SHA256
            )
        )

package_root_readback = root_hash(
    pkg_manifest
)

if (
    package_root_readback
    != package_root_sha
    or any(
        [
            missing_pkg,
            unexpected_pkg,
            size_bad,
            hash_bad,
        ]
    )
):
    raise RuntimeError(
        "Final Project 24 package failed readback validation."
    )


# --------------------------------------------------------------------------------------------------
# 12. BUILD A COMPLETE PROJECT 24 REGISTRY ROW
# --------------------------------------------------------------------------------------------------
#
# The registry schema has evolved across projects.
# Use Project 23's exact formatting for protocol fields whose textual
# representation may have varied historically, while filling Project 24's
# project-specific values explicitly.
# --------------------------------------------------------------------------------------------------

project_23_template_rows = reg_before.loc[
    pd.to_numeric(
        reg_before[
            pn_col
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        23
    )
]

if len(
    project_23_template_rows
) != 1:
    raise RuntimeError(
        "Could not resolve exactly one Project 23 registry template row."
    )

project_23_template = project_23_template_rows.iloc[
    0
]

protocol_template_values = {}

for registry_column in reg_before.columns:
    normalised_column = norm(
        registry_column
    )

    if normalised_column in {
        "seeds",
        "noiselevels",
        "techniques",
        "donotrerun",
    }:
        protocol_template_values[
            normalised_column
        ] = str(
            project_23_template[
                registry_column
            ]
        ).strip()

protocol_fallback_values = {
    "seeds":
        json.dumps(
            list(
                range(
                    1,
                    31,
                )
            ),
            separators=(
                ",",
                ":",
            ),
        ),

    "noiselevels":
        json.dumps(
            [
                0,
                5,
                10,
                15,
                20,
                25,
                30,
                40,
                50,
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "techniques":
        json.dumps(
            [
                "RandomForest",
                "XGBoost",
                "LightGBM",
                "NaiveBayes",
                "Random",
                "LatestFail",
                "QTF-Avg",
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "donotrerun":
        "True",
}

for (
    protocol_key,
    fallback_value,
) in protocol_fallback_values.items():
    if not protocol_template_values.get(
        protocol_key,
        "",
    ):
        protocol_template_values[
            protocol_key
        ] = fallback_value


values = {
    "projectnumber": PROJECT_NUMBER,
    "projectno": PROJECT_NUMBER,
    "project": PROJECT_NAME,
    "projectname": PROJECT_NAME,
    "projectslug": PROJECT_SLUG,
    "slug": PROJECT_SLUG,

    "status": COMPLETE_STATUS,
    "completionstatus": COMPLETE_STATUS,

    "completedatutc": created_at,
    "completedat": created_at,
    "frozenatutc": created_at,
    "frozenat": created_at,
    "registeredatutc": created_at,
    "registeredat": created_at,

    "sourcerootsha256": SOURCE_ROOT_SHA,

    "rawrootsha256": RAW_ROOT_SHA,
    "rawresultrootsha256": RAW_ROOT_SHA,
    "rawresultsrootsha256": RAW_ROOT_SHA,

    "rawroot": str(
        RAW_ROOT
    ),
    "rawresultroot": str(
        RAW_ROOT
    ),

    "finalpackagepath": str(
        FINAL_ROOT
    ),
    "packagepath": str(
        FINAL_ROOT
    ),
    "finalpackageroot": str(
        FINAL_ROOT
    ),

    "finalpackagerootsha256": package_root_sha,
    "packagerootsha256": package_root_sha,
    "packagesha256": package_root_sha,

    "packagefiles": package_files,
    "packagefilecount": package_files,
    "packagebytes": package_bytes,

    "step5acheckpointsha256": step5a_sha,
    "step5bcheckpointsha256": step5b_sha,

    # Complete observed registry schema.
    "seeds":
        protocol_template_values[
            "seeds"
        ],

    "noiselevels":
        protocol_template_values[
            "noiselevels"
        ],

    "techniques":
        protocol_template_values[
            "techniques"
        ],

    "evaluationrows":
        COUNTS[
            "ModelEvaluationRows"
        ],

    "evaluationfailures":
        COUNTS[
            "ModelEvaluationFailures"
        ],

    "finaldirectory":
        str(
            FINAL_ROOT
        ),

    "finalauditreport":
        str(
            REPORT_PATH
        ),

    "donotrerun":
        protocol_template_values[
            "donotrerun"
        ],

    "freezerecord":
        str(
            CHECKPOINT_PATH
        ),

    "rawresultsmanifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "finalpackagemanifest":
        str(
            MANIFEST_PATH
        ),

    "finalauditstatus":
        STEP5C_STATUS,
}

for (
    key,
    value,
) in COUNTS.items():
    values[
        norm(
            key
        )
    ] = value

values.update(
    {
        "rawfilecount":
            COUNTS[
                "RawFiles"
            ],

        "conditioncount":
            COUNTS[
                "Conditions"
            ],

        "modelfits":
            COUNTS[
                "MLFits"
            ],

        "modelreadyrows":
            COUNTS[
                "ModelRows"
            ],

        "predictorcount":
            COUNTS[
                "Predictors"
            ],

        "recfeaturecount":
            COUNTS[
                "RECFeatures"
            ],

        "rawexecutionrows":
            COUNTS[
                "RawRows"
            ],
    }
)

if not STEP5B_RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The independently frozen Step 5B raw-results manifest is missing:\n"
        f"{STEP5B_RAW_MANIFEST_PATH}"
    )

new_row = {}
unresolved = []

for col in reg_before.columns:
    normalised = norm(
        col
    )

    if normalised in values:
        new_row[
            col
        ] = str(
            values[
                normalised
            ]
        )

    else:
        unique_nonempty = sorted(
            set(
                value
                for value in reg_before[
                    col
                ].astype(
                    str
                ).str.strip()
                if value
            )
        )

        if len(
            unique_nonempty
        ) == 1:
            # Preserve a global protocol constant.
            new_row[
                col
            ] = unique_nonempty[
                0
            ]

        elif reg_before[
            col
        ].astype(
            str
        ).str.strip().eq(
            ""
        ).all():
            new_row[
                col
            ] = ""

        else:
            new_row[
                col
            ] = ""

            unresolved.append(
                col
            )

new_row[
    pn_col
] = str(
    PROJECT_NUMBER
)

new_row[
    project_col
] = PROJECT_NAME

new_row[
    status_col
] = COMPLETE_STATUS

if unresolved:
    raise RuntimeError(
        "Unexpected unmapped registry columns remain; "
        "no registry write was attempted:\n"
        + "\n".join(
            unresolved
        )
    )

reg_candidate = pd.concat(
    [
        reg_before,
        pd.DataFrame(
            [
                new_row
            ]
        ),
    ],
    ignore_index=True,
)

reg_candidate[
    pn_col
] = pd.to_numeric(
    reg_candidate[
        pn_col
    ],
    errors="raise",
).astype(
    int
).astype(
    str
)

candidate_nums = pd.to_numeric(
    reg_candidate[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

project24_candidate = reg_candidate.loc[
    candidate_nums.eq(
        24
    )
]

if (
    len(
        reg_candidate
    )
    != 24
    or sorted(
        candidate_nums.tolist()
    )
    != list(
        range(
            1,
            25,
        )
    )
):
    raise RuntimeError(
        "Candidate registry does not contain exactly Projects 1–24."
    )

if (
    not reg_candidate[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or len(
        project24_candidate
    )
    != 1
    or project24_candidate.iloc[
        0
    ][
        project_col
    ]
    != PROJECT_NAME
):
    raise RuntimeError(
        "Candidate Project 24 registry row failed status/identity validation."
    )


# --------------------------------------------------------------------------------------------------
# 13. PRE-WRITE VALIDATION
# --------------------------------------------------------------------------------------------------

rows = []

check(
    rows,
    "Step 5A checkpoint SHA-256",
    STEP5A_SHA_EXPECTED,
    step5a_sha,
    step5a_sha
    == STEP5A_SHA_EXPECTED,
)

check(
    rows,
    "Step 5B checkpoint SHA-256",
    STEP5B_SHA_EXPECTED,
    step5b_sha,
    step5b_sha
    == STEP5B_SHA_EXPECTED,
)

check(
    rows,
    "Source root SHA-256",
    SOURCE_ROOT_SHA,
    step5a.get(
        "SourceRootSHA256"
    ),
    step5a.get(
        "SourceRootSHA256"
    )
    == SOURCE_ROOT_SHA,
)

check(
    rows,
    "Raw root SHA-256",
    RAW_ROOT_SHA,
    step5b.get(
        "IndependentRawRootSHA256"
    ),
    step5b.get(
        "IndependentRawRootSHA256"
    )
    == RAW_ROOT_SHA,
)

check(
    rows,
    "Step 5A manifest failures",
    0,
    int(
        (
            ~step5a_audit[
                "Pass"
            ]
        ).sum()
    ),
    step5a_audit[
        "Pass"
    ].all(),
)

check(
    rows,
    "Step 5B manifest failures",
    0,
    int(
        (
            ~step5b_audit[
                "Pass"
            ]
        ).sum()
    ),
    step5b_audit[
        "Pass"
    ].all(),
)

check(
    rows,
    "Parallel worker checkpoint SHA mismatches",
    0,
    worker_checkpoint_sha_mismatches,
    worker_checkpoint_sha_mismatches
    == 0,
)

check(
    rows,
    "Parallel worker checkpoints",
    6,
    len(
        expected_worker_hashes_by_tag
    ),
    len(
        expected_worker_hashes_by_tag
    )
    == 6,
)

check(
    rows,
    "Parallel worker scientific-freeze failures",
    0,
    worker_scientific_freeze_failures,
    worker_scientific_freeze_failures
    == 0,
)

check(
    rows,
    "Step 5B prior-project output access",
    False,
    bool(
        step5b.get(
            "PriorProjectConditionOutputsAccessed",
            True,
        )
    ),
    not bool(
        step5b.get(
            "PriorProjectConditionOutputsAccessed",
            True,
        )
    ),
)

check(
    rows,
    "Step 5B prior-project output modification",
    False,
    bool(
        step5b.get(
            "PriorProjectConditionOutputsModified",
            True,
        )
    ),
    not bool(
        step5b.get(
            "PriorProjectConditionOutputsModified",
            True,
        )
    ),
)

check(
    rows,
    "Raw result file verification failures",
    0,
    raw_file_failures,
    raw_file_failures
    == 0,
)

check(
    rows,
    "Package missing files",
    0,
    missing_pkg,
    missing_pkg
    == 0,
)

check(
    rows,
    "Package unexpected files",
    0,
    unexpected_pkg,
    unexpected_pkg
    == 0,
)

check(
    rows,
    "Package size mismatches",
    0,
    size_bad,
    size_bad
    == 0,
)

check(
    rows,
    "Package SHA-256 mismatches",
    0,
    hash_bad,
    hash_bad
    == 0,
)

check(
    rows,
    "Registry rows before",
    23,
    len(
        reg_before
    ),
    len(
        reg_before
    )
    == 24,
)

check(
    rows,
    "Registry rows candidate",
    24,
    len(
        reg_candidate
    ),
    len(
        reg_candidate
    )
    == 24,
)

check(
    rows,
    "Candidate Project 24 rows",
    1,
    len(
        project24_candidate
    ),
    len(
        project24_candidate
    )
    == 1,
)

check(
    rows,
    "Unresolved variable registry columns",
    0,
    len(
        unresolved
    ),
    len(
        unresolved
    )
    == 0,
)

required_registry_field_expectations = {
    "Seeds":
        protocol_template_values[
            "seeds"
        ],

    "NoiseLevels":
        protocol_template_values[
            "noiselevels"
        ],

    "Techniques":
        protocol_template_values[
            "techniques"
        ],

    "EvaluationRows":
        str(
            COUNTS[
                "ModelEvaluationRows"
            ]
        ),

    "EvaluationFailures":
        str(
            COUNTS[
                "ModelEvaluationFailures"
            ]
        ),

    "FinalDirectory":
        str(
            FINAL_ROOT
        ),

    "FinalAuditReport":
        str(
            REPORT_PATH
        ),

    "DoNotRerun":
        protocol_template_values[
            "donotrerun"
        ],

    "FreezeRecord":
        str(
            CHECKPOINT_PATH
        ),

    "RawResultsManifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "FinalPackageManifest":
        str(
            MANIFEST_PATH
        ),

    "RawResultsRootSHA256":
        RAW_ROOT_SHA,

    "FinalAuditStatus":
        STEP5C_STATUS,
}

registry_field_validation_failures = 0

for (
    expected_column_name,
    expected_value,
) in required_registry_field_expectations.items():
    matching_columns = [
        column
        for column in reg_before.columns
        if norm(
            column
        )
        == norm(
            expected_column_name
        )
    ]

    if len(
        matching_columns
    ) != 1:
        registry_field_validation_failures += 1
        continue

    actual_value = str(
        project24_candidate.iloc[
            0
        ][
            matching_columns[
                0
            ]
        ]
    )

    registry_field_validation_failures += int(
        actual_value
        != str(
            expected_value
        )
    )

check(
    rows,
    "Explicit Project 24 registry-field failures",
    0,
    registry_field_validation_failures,
    registry_field_validation_failures
    == 0,
)

pre = pd.DataFrame(
    rows
)

print(
    "\nProject 24 Step 5C pre-write validation:"
)

display(
    pre
)

print(
    "\nProject 24 registry row candidate:"
)

display(
    project24_candidate
)

if not pre[
    "Pass"
].all():
    raise RuntimeError(
        "PROJECT 24 STEP 5C PRE-WRITE VALIDATION FAILED. "
        "Registry not modified."
    )


# --------------------------------------------------------------------------------------------------
# 14. ATOMIC COMPLETION-REGISTRY WRITE
# --------------------------------------------------------------------------------------------------

tmp_reg = REGISTRY.with_name(
    f".{REGISTRY.name}.project24_{os.getpid()}"
)

reg_candidate.to_csv(
    tmp_reg,
    index=False,
    lineterminator="\n",
)

tmp_read = pd.read_csv(
    tmp_reg,
    dtype=str,
).fillna(
    ""
)

tmp_nums = pd.to_numeric(
    tmp_read[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        tmp_read
    )
    != 24
    or sorted(
        tmp_nums.tolist()
    )
    != list(
        range(
            1,
            25,
        )
    )
    or not tmp_read[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or int(
        tmp_nums.eq(
            24
        ).sum()
    )
    != 1
):
    tmp_reg.unlink(
        missing_ok=True
    )

    raise RuntimeError(
        "Temporary Project 24 registry failed readback; "
        "live registry unchanged."
    )

os.replace(
    tmp_reg,
    REGISTRY,
)


# --------------------------------------------------------------------------------------------------
# 15. POST-WRITE REGISTRY VALIDATION
# --------------------------------------------------------------------------------------------------

reg_after = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna(
    ""
)

after_nums = pd.to_numeric(
    reg_after[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

project24_after = reg_after.loc[
    after_nums.eq(
        24
    )
]

registry_sha_after = sha(
    REGISTRY
)

if (
    len(
        reg_after
    )
    != 24
    or sorted(
        after_nums.tolist()
    )
    != list(
        range(
            1,
            25,
        )
    )
    or not reg_after[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or len(
        project24_after
    )
    != 1
    or project24_after.iloc[
        0
    ][
        project_col
    ]
    != PROJECT_NAME
):
    raise RuntimeError(
        "Live registry failed post-write validation.\n"
        f"Backup: {BACKUP_PATH}"
    )

for (
    required_number,
    required_project,
) in required_registered_identities.items():
    matching_rows = reg_after.loc[
        after_nums.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_col
        ]
        != required_project
    ):
        raise RuntimeError(
            "A predecessor registry identity changed after "
            "Project 24 registration."
        )

check(
    rows,
    "Registry rows after",
    24,
    len(
        reg_after
    ),
    len(
        reg_after
    )
    == 24,
)

check(
    rows,
    "COMPLETE_AND_FROZEN projects after",
    24,
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    ),
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    )
    == 23,
)

check(
    rows,
    "Registry Project 24 rows after",
    1,
    len(
        project24_after
    ),
    len(
        project24_after
    )
    == 1,
)

check(
    rows,
    "Registry SHA changed",
    True,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    != registry_sha_before,
)

for number in range(
    11,
    24,
):
    check(
        rows,
        f"Registry Project {number} rows after",
        1,
        int(
            after_nums.eq(
                number
            ).sum()
        ),
        int(
            after_nums.eq(
                number
            ).sum()
        )
        == 1,
    )

validation = pd.DataFrame(
    rows
)

failed = validation.loc[
    ~validation[
        "Pass"
    ]
]

if not failed.empty:
    display(
        failed
    )

    raise RuntimeError(
        "PROJECT 24 STEP 5C POST-WRITE VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 16. WRITE STEP 5C AUDIT / CHECKPOINT / STATUS
# --------------------------------------------------------------------------------------------------

STEP5C_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)

registry_rows_after_by_project = {
    f"Project{number}RegistryRowsAfter":
        int(
            after_nums.eq(
                number
            ).sum()
        )
    for number in range(
        11,
        25,
    )
}

report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS,
    "CompletedAtUTC": created_at,

    "SourceRootSHA256": SOURCE_ROOT_SHA,
    "RawRootSHA256": RAW_ROOT_SHA,

    **COUNTS,

    "FinalPackageRoot": str(
        FINAL_ROOT
    ),
    "FinalPackageFiles": package_files,
    "FinalPackageBytes": package_bytes,
    "FinalPackageRootSHA256": package_root_sha,
    "PackageAlreadyFrozenBeforeThisCell":
        package_already_frozen,

    "PackageMissingFiles": missing_pkg,
    "PackageUnexpectedFiles": unexpected_pkg,
    "PackageSizeMismatches": size_bad,
    "PackageSHA256Mismatches": hash_bad,

    "RegistrySHA256Before": registry_sha_before,
    "RegistrySHA256After": registry_sha_after,
    "RegistryRowsBefore": len(
        reg_before
    ),
    "RegistryRowsAfter": len(
        reg_after
    ),

    **registry_rows_after_by_project,

    "RegistryBackup": str(
        BACKUP_PATH
    ),

    "Step5ACheckpointSHA256": step5a_sha,
    "Step5BCheckpointSHA256": step5b_sha,

    "ParallelWorkerCheckpointSHA256":
        expected_worker_hashes_by_tag,

    "ValidationChecks": len(
        validation
    ),
    "FailedValidationChecks": len(
        failed
    ),

    "ConditionsRerun": False,
    "ModelsFitted": False,
    "RawResultsModified": False,

    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "PriorProjectWriteAttempted": False,
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    CHECKPOINT_PATH,
    {
        **report,
        "CheckpointVersion": 1,
        "CheckpointType":
            "PROJECT_24_FINAL_PACKAGE_AND_REGISTRY",
        "FinalPackageFrozen": True,
        "CompletionRegistryUpdated": True,
        "ProjectCompleteAndFrozen": True,
        "NextRequiredStep":
            "PROJECT_25_MAY_START_IN_A_NEW_NOTEBOOK",
    },
)

step5c_sha = sha(
    CHECKPOINT_PATH
)

atomic_json(
    STATUS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "Status": STEP5C_STATUS,
        "CompletedAtUTC": created_at,

        "FinalPackageRoot": str(
            FINAL_ROOT
        ),
        "FinalPackageRootSHA256":
            package_root_sha,

        "RegistryRows": len(
            reg_after
        ),
        "RegistrySHA256":
            registry_sha_after,

        "Checkpoint": str(
            CHECKPOINT_PATH
        ),
        "CheckpointSHA256":
            step5c_sha,

        "ProjectCompleteAndFrozen": True,

        "ParallelWorkerCheckpointSHA256":
            expected_worker_hashes_by_tag,

        "PriorProjectConditionOutputsAccessed": False,
        "PriorProjectConditionOutputsModified": False,

        "NextRequiredStep":
            "PROJECT_25_MAY_START_IN_A_NEW_NOTEBOOK",
    },
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL READBACK AND IMMUTABILITY PROOF
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    CHECKPOINT_PATH
)

status_readback = load_json(
    STATUS_PATH
)

if (
    checkpoint_readback.get(
        "Status"
    )
    != STEP5C_STATUS
    or status_readback.get(
        "Status"
    )
    != STEP5C_STATUS
):
    raise RuntimeError(
        "Project 24 Step 5C checkpoint/status readback failed."
    )

if not bool(
    checkpoint_readback.get(
        "ProjectCompleteAndFrozen",
        False,
    )
):
    raise RuntimeError(
        "Project 24 checkpoint is not marked complete and frozen."
    )

if not bool(
    checkpoint_readback.get(
        "CompletionRegistryUpdated",
        False,
    )
):
    raise RuntimeError(
        "Project 24 checkpoint is not marked registry-updated."
    )

if not bool(
    checkpoint_readback.get(
        "FinalPackageFrozen",
        False,
    )
):
    raise RuntimeError(
        "Project 24 checkpoint is not marked final-package frozen."
    )

if sha(
    REGISTRY
) != registry_sha_after:
    raise RuntimeError(
        "Completion registry changed after Project 24 finalisation."
    )

if root_hash(
    pd.read_csv(
        MANIFEST_PATH,
        low_memory=False,
    )
) != package_root_sha:
    raise RuntimeError(
        "Final package changed after Project 24 finalisation."
    )

if sha(
    STEP5A_CP
) != STEP5A_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5A checkpoint changed during Project 24 Step 5C."
    )

if sha(
    STEP5B_CP
) != STEP5B_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5B checkpoint changed during Project 24 Step 5C."
    )

for tag, worker_path in WORKER_CHECKPOINT_PATHS.items():
    if sha(
        worker_path
    ) != str(
        expected_worker_hashes_by_tag[
            tag
        ]
    ):
        raise RuntimeError(
            "A frozen Project 24 worker checkpoint changed "
            "during Step 5C."
        )


# --------------------------------------------------------------------------------------------------
# 18. RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 136
)

print(
    "=== PROJECT 24 CELL 11 / STEP 5C RESULT ==="
)

print(
    "=" * 136
)

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "\nRaw result freeze:"
)

print(
    "Conditions:",
    COUNTS[
        "Conditions"
    ],
)

print(
    "ML fits:",
    COUNTS[
        "MLFits"
    ],
)

print(
    "Raw files:",
    COUNTS[
        "RawFiles"
    ],
)

print(
    "Raw bytes:",
    COUNTS[
        "RawBytes"
    ],
)

print(
    "Raw root SHA-256:",
    RAW_ROOT_SHA,
)

print(
    "\nParallel execution freeze:"
)

print(
    "Worker checkpoints:",
    len(
        expected_worker_hashes_by_tag
    ),
)

print(
    "Worker checkpoint SHA mismatches:",
    worker_checkpoint_sha_mismatches,
)

print(
    "\nFinal package freeze:"
)

print(
    "Package root:",
    FINAL_ROOT,
)

print(
    "Package files:",
    package_files,
)

print(
    "Package bytes:",
    package_bytes,
)

print(
    "Missing package files:",
    missing_pkg,
)

print(
    "Unexpected package files:",
    unexpected_pkg,
)

print(
    "Package size mismatches:",
    size_bad,
)

print(
    "Package SHA-256 mismatches:",
    hash_bad,
)

print(
    "Final package root SHA-256:",
    package_root_sha,
)

print(
    "\nCompletion registry:"
)

print(
    "Registry rows:",
    len(
        reg_after
    ),
)

print(
    "COMPLETE_AND_FROZEN projects:",
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    ),
)

print(
    "Project 24 registry rows:",
    len(
        project24_after
    ),
)

print(
    "Registry SHA-256 before:",
    registry_sha_before,
)

print(
    "Registry SHA-256 after:",
    registry_sha_after,
)

print(
    "Package already frozen before this cell:",
    package_already_frozen,
)

print(
    "\nStep 5C freeze:"
)

print(
    "Validation checks:",
    len(
        validation
    ),
)

print(
    "Failed validation checks:",
    len(
        failed
    ),
)

print(
    "Step 5C checkpoint:",
    CHECKPOINT_PATH,
)

print(
    "Step 5C checkpoint SHA-256:",
    step5c_sha,
)

print(
    "Project complete and frozen:",
    True,
)

print(
    "Next required step:",
    "PROJECT 24 MAY START IN A NEW NOTEBOOK",
)

print(
    "STATUS:",
    STEP5C_STATUS,
)

print(
    "=" * 136
)


=== PROJECT 24 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Completion-registry ProjectNumber-24 rows: 0
Completion-registry exact project-name matches (diagnostic only): 0

Project 24 Step 5C pre-write validation:


,Check,Expected,Actual,Pass
0,Step 5A checkpoint SHA-256,d7abb32056cc47a4b304794dcc2d59477a18c7d42effda...,d7abb32056cc47a4b304794dcc2d59477a18c7d42effda...,True
1,Step 5B checkpoint SHA-256,3117c9637893b030d67ef4e7743d08a37f0787ec7037cb...,3117c9637893b030d67ef4e7743d08a37f0787ec7037cb...,True
2,Source root SHA-256,2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21...,2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21...,True
3,Raw root SHA-256,6968c094a564405fe23e02f9a5b1ed01cce130ea000dd3...,6968c094a564405fe23e02f9a5b1ed01cce130ea000dd3...,True
4,Step 5A manifest failures,0,0,True
5,Step 5B manifest failures,0,0,True
6,Parallel worker checkpoint SHA mismatches,0,0,True
7,Parallel worker checkpoints,6,6,True
8,Parallel worker scientific-freeze failures,0,0,True
9,Step 5B prior-project output access,False,False,True



Project 24 registry row candidate:


,ProjectNumber,Project,ProjectSlug,Status,Conditions,Seeds,NoiseLevels,Techniques,EvaluationBuilds,EvaluationRows,...,FreezeRecord,ChecksumManifest,LastFreezeValidationAtUTC,RawResultsManifest,FinalPackageManifest,RawResultsRootSHA256,FinalPackageRootSHA256,ModelFits,ManifestRowsAudited,FinalAuditStatus
23,24,SonarSource@sonarqube,SonarSource__sonarqube,COMPLETE_AND_FROZEN,270,30,9,7,1072,18854,...,/content/drive/MyDrive/Thesis_Experiment/Notes...,/content/drive/MyDrive/Thesis_Experiment/Resul...,2026-07-25T03:49:02.436302+00:00,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,6968c094a564405fe23e02f9a5b1ed01cce130ea000dd3...,ed2e84213389ad8102554ab0e68740a8d893b8b4281729...,1080,5358150.0,PASS_PROJECT_24_FINAL_PACKAGE_FROZEN_AND_REGIS...


RuntimeError: PROJECT 24 STEP 5C PRE-WRITE VALIDATION FAILED. Registry not modified.

In [6]:
# ==================================================================================================
# PROJECT 24 — CELL 11 / STEP 5C
# REGISTRY-SCHEMA-COMPLETE, CROSS-FILESYSTEM-SAFE FINAL PACKAGE AND REGISTRATION
#
# PROJECT:
#   SonarSource@sonarqube
#
# RUN THIS AS THE NEXT NEW CELL IN Thesis_project_24.ipynb.
#
# REQUIRED FROZEN INPUTS:
# - Project 24 Step 5A checkpoint SHA-256:
#   d7abb32056cc47a4b304794dcc2d59477a18c7d42effdacfcf1e58c97056d1a1
# - Project 24 Step 5B checkpoint SHA-256:
#   3117c9637893b030d67ef4e7743d08a37f0787ec7037cbd27b8f8ca519ddc6c8
# - Project 24 raw-root SHA-256:
#   6968c094a564405fe23e02f9a5b1ed01cce130ea000dd387f7a3f6f12b421d09
# - Project 24 source-root SHA-256:
#   2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70
# - Registry before registration:
#   exactly Projects 1–23, all COMPLETE_AND_FROZEN
# - Registry SHA-256 before registration:
#   c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274
#
# SAFETY:
# - no model fitting;
# - no condition reruns;
# - no raw-result modification or deletion;
# - no prior-project condition-output access or write;
# - registry write only after package and candidate-row validation;
# - cross-filesystem-safe Google Drive staging and readback;
# - six frozen Project 24 Step 5A worker checkpoints are preserved and revalidated.
# ==================================================================================================

from google.colab import drive
from pathlib import Path
from IPython.display import display
import hashlib
import json
import os
import re
import shutil
import tempfile
import pandas as pd

print("=" * 136)
print("=== PROJECT 24 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===")
print("=" * 136)


# --------------------------------------------------------------------------------------------------
# 1. FROZEN IDENTITY, HASHES, AND COUNTS
# --------------------------------------------------------------------------------------------------

PROJECT_NUMBER = 24
PROJECT_NAME = "SonarSource@sonarqube"
PROJECT_SLUG = "SonarSource__sonarqube"
PROJECT_SHORT = "SONARQUBE"

COMPLETE_STATUS = "COMPLETE_AND_FROZEN"

STEP5C_STATUS = "PASS_PROJECT_24_FINAL_PACKAGE_FROZEN_AND_REGISTERED"
STEP5A_STATUS = "PASS_PROJECT_24_FULL_270_CONDITION_EXPERIMENT_COMPLETE"
STEP5B_STATUS = "PASS_PROJECT_24_RAW_RESULTS_REVALIDATED_AND_COMPACT_AGGREGATES_FROZEN"
STEP5C_CODE_REVISION = "PROJECT_24_STEP5C_V2_CORRECT_PRE_POST_REGISTRY_COUNT_PREDICATES"

REGISTRY_SHA_BEFORE_EXPECTED = (
    "c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274"
)

STEP5A_SHA_EXPECTED = (
    "d7abb32056cc47a4b304794dcc2d59477a18c7d42effdacfcf1e58c97056d1a1"
)

STEP5B_SHA_EXPECTED = (
    "3117c9637893b030d67ef4e7743d08a37f0787ec7037cbd27b8f8ca519ddc6c8"
)

SOURCE_ROOT_SHA = (
    "2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21ffb4525f5625000b70"
)

RAW_ROOT_SHA = (
    "6968c094a564405fe23e02f9a5b1ed01cce130ea000dd387f7a3f6f12b421d09"
)

COUNTS = {
    "RawFiles": 2160,
    "RawBytes": 523093565,
    "Conditions": 270,
    "MLFits": 1080,
    "RankingRows": 35634060,
    "BuildMetricRows": 32130,
    "ProjectRunRows": 1890,
    "ConditionAuditRows": 270,
    "TrainingMedianRows": 40770,

    "Builds": 4286,
    "TrainingBuilds": 3214,
    "EvaluationBuilds": 1072,

    "RawRows": 5635027,
    "RawTrainingRows": 4040801,
    "RawEvaluationRows": 1594226,
    "RawTrainingFailures": 1778,
    "RawEvaluationFailures": 20,

    "ModelRows": 224550,
    "ModelTrainingRows": 205696,
    "ModelEvaluationRows": 18854,
    "ModelTrainingFailures": 1777,
    "ModelEvaluationFailures": 20,
    "ModelFailingEvaluationBuilds": 17,

    "Predictors": 151,
    "RECFeatures": 19,
}


# --------------------------------------------------------------------------------------------------
# 2. PATHS
# --------------------------------------------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False,
)

ROOT = Path(
    "/content/drive/MyDrive/Thesis_Experiment"
)

NOTES = (
    ROOT
    / "Notes"
)

RESULTS = (
    ROOT
    / "Results"
)

REGISTRY = (
    NOTES
    / "completed_project_registry.csv"
)

PROJECT_ROOT = (
    RESULTS
    / "Aggregated"
    / PROJECT_SLUG
)

RAW_ROOT = (
    RESULTS
    / "Raw"
    / PROJECT_SLUG
)

FINAL_ROOT = (
    RESULTS
    / "Final"
    / PROJECT_SLUG
)

MANIFEST_PATH = (
    FINAL_ROOT
    / "final_package_manifest.csv"
)

SUMMARY_PATH = (
    FINAL_ROOT
    / "final_package_summary.json"
)

README_PATH = (
    FINAL_ROOT
    / "README.txt"
)

STEP5C_ROOT = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5c"
)

VALIDATION_PATH = (
    STEP5C_ROOT
    / f"{PROJECT_SHORT}_step5c_validation.csv"
)

REPORT_PATH = (
    STEP5C_ROOT
    / f"{PROJECT_SHORT}_step5c_report.json"
)

STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5c_status.json"
)

CHECKPOINT_PATH = (
    NOTES
    / "project_24_step5c_checkpoint.json"
)

BACKUP_PATH = (
    NOTES
    / "completed_project_registry_before_project_24.csv"
)

STEP5B_RAW_MANIFEST_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b"
    / f"{PROJECT_SHORT}_independent_raw_manifest.csv"
)

STEP5A_CP = (
    NOTES
    / "project_24_step5a_checkpoint.json"
)

STEP5B_CP = (
    NOTES
    / "project_24_step5b_checkpoint.json"
)

STEP5A_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5a_status.json"
)

STEP5B_STATUS_PATH = (
    PROJECT_ROOT
    / f"{PROJECT_SHORT}_step5b_status.json"
)

UPSTREAM_CPS = [
    NOTES / "project_24_selection_checkpoint.json",
    NOTES / "project_24_rec_reconstruction_checkpoint.json",
    NOTES / "project_24_noise_plan_checkpoint.json",
    NOTES / "project_24_runtime_contract_checkpoint.json",
    NOTES / "project_24_smoke_test_checkpoint.json",
    STEP5A_CP,
    STEP5B_CP,
]

WORKER_CHECKPOINT_PATHS = {
    "seed_01_05":
        NOTES / "project_24_step5a_worker_seed_01_05_checkpoint.json",
    "seed_06_10":
        NOTES / "project_24_step5a_worker_seed_06_10_checkpoint.json",
    "seed_11_15":
        NOTES / "project_24_step5a_worker_seed_11_15_checkpoint.json",
    "seed_16_20":
        NOTES / "project_24_step5a_worker_seed_16_20_checkpoint.json",
    "seed_21_25":
        NOTES / "project_24_step5a_worker_seed_21_25_checkpoint.json",
    "seed_26_30":
        NOTES / "project_24_step5a_worker_seed_26_30_checkpoint.json",
}



# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha(
    path,
    chunk=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:
        for block in iter(
            lambda: f.read(
                chunk
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path,
):
    with Path(
        path
    ).open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(
            f
        )


def atomic_json(
    path,
    obj,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    with tmp.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )

        f.write(
            "\n"
        )

    os.replace(
        tmp,
        path,
    )


def atomic_csv(
    path,
    df,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    df.to_csv(
        tmp,
        index=False,
        lineterminator="\n",
    )

    os.replace(
        tmp,
        path,
    )


def atomic_text(
    path,
    text,
):
    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        f".{path.name}.tmp_{os.getpid()}"
    )

    tmp.write_text(
        text,
        encoding="utf-8",
    )

    os.replace(
        tmp,
        path,
    )


def resolve(
    cols,
    expected,
):
    matches = [
        c
        for c in cols
        if str(
            c
        ).strip().lower()
        == expected.lower()
    ]

    if len(
        matches
    ) != 1:
        raise RuntimeError(
            f"Could not resolve registry column {expected!r}; "
            f"matches={matches}; columns={list(cols)}"
        )

    return matches[
        0
    ]


def norm(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value
        ).lower(),
    )


def manifest(
    root,
    exclude=(),
):
    root = Path(
        root
    )

    exclude = set(
        exclude
    )

    rows = []

    files = sorted(
        (
            path
            for path in root.rglob(
                "*"
            )
            if path.is_file()
        ),
        key=lambda path:
            path.relative_to(
                root
            ).as_posix(),
    )

    for path in files:
        rel = path.relative_to(
            root
        ).as_posix()

        if rel in exclude:
            continue

        rows.append(
            {
                "RelativePath": rel,
                "Bytes": int(
                    path.stat().st_size
                ),
                "SHA256": sha(
                    path
                ),
            }
        )

    return pd.DataFrame(
        rows,
        columns=[
            "RelativePath",
            "Bytes",
            "SHA256",
        ],
    )


def root_hash(
    df,
):
    h = hashlib.sha256()

    ordered = df.sort_values(
        "RelativePath",
        kind="mergesort",
    )

    for row in ordered.itertuples(
        index=False
    ):
        h.update(
            (
                f"{row.RelativePath}\0"
                f"{int(row.Bytes)}\0"
                f"{str(row.SHA256).lower()}\n"
            ).encode()
        )

    return h.hexdigest()


def verify_manifest(
    items,
    label,
):
    if (
        not isinstance(
            items,
            list,
        )
        or not items
    ):
        raise RuntimeError(
            f"{label} has no output manifest."
        )

    rows = []

    for item in items:
        path = Path(
            item[
                "Path"
            ]
        )

        exists = path.is_file()

        expected_bytes = int(
            item[
                "Bytes"
            ]
        )

        expected_sha = str(
            item[
                "SHA256"
            ]
        ).lower()

        if exists:
            actual_bytes = int(
                path.stat().st_size
            )

            actual_sha = sha(
                path
            )
        else:
            actual_bytes = -1
            actual_sha = "MISSING"

        rows.append(
            {
                "Path": str(
                    path
                ),
                "ExpectedBytes": expected_bytes,
                "ActualBytes": actual_bytes,
                "ExpectedSHA256": expected_sha,
                "ActualSHA256": actual_sha,
                "Pass": bool(
                    exists
                    and expected_bytes
                    == actual_bytes
                    and expected_sha
                    == actual_sha
                ),
            }
        )

    out = pd.DataFrame(
        rows
    )

    if not out[
        "Pass"
    ].all():
        display(
            out.loc[
                ~out[
                    "Pass"
                ]
            ]
        )

        raise RuntimeError(
            f"{label} manifest verification failed."
        )

    return out


def check(
    rows,
    name,
    expected,
    actual,
    passed,
):
    rows.append(
        {
            "Check": name,
            "Expected": expected,
            "Actual": actual,
            "Pass": bool(
                passed
            ),
        }
    )


# --------------------------------------------------------------------------------------------------
# 4. VERIFY ALL REQUIRED INPUTS BEFORE ANY PACKAGE OR REGISTRY WRITE
# --------------------------------------------------------------------------------------------------

required = [
    REGISTRY,
    RAW_ROOT,
    STEP5A_CP,
    STEP5B_CP,
    STEP5A_STATUS_PATH,
    STEP5B_STATUS_PATH,
    *UPSTREAM_CPS,
    *WORKER_CHECKPOINT_PATHS.values(),
]

missing = [
    str(
        path
    )
    for path in required
    if not Path(
        path
    ).exists()
]

if missing:
    raise FileNotFoundError(
        "Missing Project 24 Step 5C inputs:\n"
        + "\n".join(
            missing
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. VERIFY STEP 5A / STEP 5B FREEZES
# --------------------------------------------------------------------------------------------------

step5a_sha = sha(
    STEP5A_CP
)

step5b_sha = sha(
    STEP5B_CP
)

if step5a_sha != STEP5A_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5A checkpoint SHA differs.\n"
        f"Expected: {STEP5A_SHA_EXPECTED}\n"
        f"Actual:   {step5a_sha}"
    )

if step5b_sha != STEP5B_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5B checkpoint SHA differs.\n"
        f"Expected: {STEP5B_SHA_EXPECTED}\n"
        f"Actual:   {step5b_sha}"
    )

step5a = load_json(
    STEP5A_CP
)

step5b = load_json(
    STEP5B_CP
)

step5a_status_payload = load_json(
    STEP5A_STATUS_PATH
)

step5b_status_payload = load_json(
    STEP5B_STATUS_PATH
)

for label, payload, expected in [
    (
        "Step 5A checkpoint",
        step5a,
        STEP5A_STATUS,
    ),
    (
        "Step 5A status",
        step5a_status_payload,
        STEP5A_STATUS,
    ),
    (
        "Step 5B checkpoint",
        step5b,
        STEP5B_STATUS,
    ),
    (
        "Step 5B status",
        step5b_status_payload,
        STEP5B_STATUS,
    ),
]:
    if payload.get(
        "Status"
    ) != expected:
        raise RuntimeError(
            f"{label} is not in expected PASS state."
        )

if (
    step5a.get(
        "SourceRootSHA256"
    )
    != SOURCE_ROOT_SHA
):
    raise RuntimeError(
        "Step 5A source-root SHA differs."
    )

if (
    step5a.get(
        "RawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5A raw-root SHA differs."
    )

if (
    step5b.get(
        "SourceRootSHA256"
    )
    != SOURCE_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B source-root SHA differs."
    )

if (
    step5b.get(
        "FrozenRawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B frozen raw-root SHA differs."
    )

if (
    step5b.get(
        "IndependentRawRootSHA256"
    )
    != RAW_ROOT_SHA
):
    raise RuntimeError(
        "Step 5B independent raw-root SHA differs."
    )

if (
    step5b.get(
        "Step5ACheckpointSHA256"
    )
    != STEP5A_SHA_EXPECTED
):
    raise RuntimeError(
        "Step 5B does not link to the frozen Step 5A checkpoint."
    )

if not bool(
    step5b.get(
        "ReadyForFinalPackageAndRegistration",
        False,
    )
):
    raise RuntimeError(
        "Step 5B is not marked ready for final package and registration."
    )

if bool(
    step5b.get(
        "RegistryModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports that the completion registry was modified."
    )

if bool(
    step5b.get(
        "PriorProjectConditionOutputsAccessed",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports access to prior-project condition outputs."
    )

if bool(
    step5b.get(
        "PriorProjectConditionOutputsModified",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports modification of prior-project condition outputs."
    )

if bool(
    step5b.get(
        "PriorProjectWriteAttempted",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports a prior-project write attempt."
    )

if bool(
    step5b.get(
        "ModelsFitted",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports model fitting."
    )

if bool(
    step5b.get(
        "ConditionsRerun",
        True,
    )
):
    raise RuntimeError(
        "Step 5B reports condition reruns."
    )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY SIX FROZEN PARALLEL WORKER CHECKPOINTS
# --------------------------------------------------------------------------------------------------
#
# Project 24 Step 5A froze the authoritative current worker-checkpoint SHA map.
# The authoritative current worker checkpoint SHA map is the ACTUAL map frozen
# in the official Step 5A master checkpoint.
# --------------------------------------------------------------------------------------------------

expected_worker_hashes_by_tag = step5a.get(
    "ActualWorkerCheckpointSHA256",
    {},
)

if (
    not isinstance(
        expected_worker_hashes_by_tag,
        dict,
    )
    or sorted(
        expected_worker_hashes_by_tag
    )
    != sorted(
        WORKER_CHECKPOINT_PATHS
    )
):
    raise RuntimeError(
        "Step 5A ActualWorkerCheckpointSHA256 does not contain "
        "the exact six Project 24 worker tags."
    )

step5a_worker_records = step5a.get(
    "WorkerCheckpoints",
    [],
)

if (
    not isinstance(
        step5a_worker_records,
        list,
    )
    or len(
        step5a_worker_records
    )
    != 6
):
    raise RuntimeError(
        "Step 5A WorkerCheckpoints does not contain exactly six records."
    )

worker_records_by_tag = {
    str(
        record.get(
            "WorkerTag",
            "",
        )
    ):
        record
    for record in step5a_worker_records
}

if sorted(
    worker_records_by_tag
) != sorted(
    WORKER_CHECKPOINT_PATHS
):
    raise RuntimeError(
        "Step 5A WorkerCheckpoints tags differ from the expected six worker shards."
    )

worker_checkpoint_sha_mismatches = 0
worker_scientific_freeze_failures = 0

for tag, worker_path in WORKER_CHECKPOINT_PATHS.items():
    expected_worker_sha = str(
        expected_worker_hashes_by_tag[
            tag
        ]
    )

    actual_worker_sha = sha(
        worker_path
    )

    worker_checkpoint_sha_mismatches += int(
        actual_worker_sha
        != expected_worker_sha
    )

    worker_payload = load_json(
        worker_path
    )

    master_record = worker_records_by_tag[
        tag
    ]

    worker_scientific_freeze_failures += int(
        worker_payload.get(
            "Status"
        )
        != master_record.get(
            "Status"
        )
    )

    worker_scientific_freeze_failures += int(
        str(
            worker_payload.get(
                "WorkerRawRootSHA256",
                "",
            )
        )
        != str(
            master_record.get(
                "WorkerRawRootSHA256",
                "",
            )
        )
    )

    worker_scientific_freeze_failures += int(
        int(
            worker_payload.get(
                "WorkerRawFiles",
                -1,
            )
        )
        != int(
            master_record.get(
                "WorkerRawFiles",
                -2,
            )
        )
    )

    worker_scientific_freeze_failures += int(
        int(
            worker_payload.get(
                "WorkerRawBytes",
                -1,
            )
        )
        != int(
            master_record.get(
                "WorkerRawBytes",
                -2,
            )
        )
    )

if worker_checkpoint_sha_mismatches:
    raise RuntimeError(
        f"{worker_checkpoint_sha_mismatches} frozen Project 24 worker "
        "checkpoint SHA values changed after official Step 5A."
    )

if worker_scientific_freeze_failures:
    raise RuntimeError(
        f"{worker_scientific_freeze_failures} Project 24 worker scientific "
        "freeze fields differ from official Step 5A."
    )

# --------------------------------------------------------------------------------------------------
# 7. VERIFY STEP 5A / STEP 5B OUTPUT MANIFESTS
# --------------------------------------------------------------------------------------------------

step5a_audit = verify_manifest(
    step5a.get(
        "AggregateOutputManifest",
        [],
    ),
    "Step 5A aggregate",
)

step5b_audit = verify_manifest(
    step5b.get(
        "OutputManifest",
        [],
    ),
    "Step 5B",
)


# --------------------------------------------------------------------------------------------------
# 8. VERIFY COMPLETION REGISTRY PRE-STATE
# --------------------------------------------------------------------------------------------------

registry_sha_before = sha(
    REGISTRY
)

if (
    registry_sha_before
    != REGISTRY_SHA_BEFORE_EXPECTED
):
    raise RuntimeError(
        "Registry SHA differs before Project 24 registration:\n"
        f"Expected: {REGISTRY_SHA_BEFORE_EXPECTED}\n"
        f"Actual:   {registry_sha_before}"
    )

if (
    step5b.get(
        "RegistrySHA256"
    )
    != REGISTRY_SHA_BEFORE_EXPECTED
):
    raise RuntimeError(
        "The frozen Step 5B checkpoint does not reference "
        "the expected pre-Project-24 registry SHA."
    )

reg_before = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna(
    ""
)

pn_col = resolve(
    reg_before.columns,
    "ProjectNumber",
)

project_col = resolve(
    reg_before.columns,
    "Project",
)

status_col = resolve(
    reg_before.columns,
    "Status",
)

pnums = pd.to_numeric(
    reg_before[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        reg_before
    )
    != 23
    or sorted(
        pnums.tolist()
    )
    != list(
        range(
            1,
            24,
        )
    )
):
    raise RuntimeError(
        "Registry must contain exactly Projects 1–23."
    )

if not reg_before[
    status_col
].eq(
    COMPLETE_STATUS
).all():
    raise RuntimeError(
        "Projects 1–23 are not all COMPLETE_AND_FROZEN."
    )

required_registered_identities = {
    11: "apache@shardingsphere",
    12: "zolyfarkas@spf4j",
    13: "jcabi@jcabi-github",
    14: "JMRI@JMRI",
    15: "eclipse@steady",
    16: "apache@rocketmq",
    17: "yamcs@Yamcs",
    18: "cantaloupe-project@cantaloupe",
    19: "EMResearch@EvoMaster",
    20: "apache@curator",
    21: "facebook@buck",
    22: "apache@logging-log4j2",
    23: "apache@sling",
}

for (
    required_number,
    required_project,
) in required_registered_identities.items():
    matching_rows = reg_before.loc[
        pnums.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_col
        ]
        != required_project
    ):
        raise RuntimeError(
            "A required frozen predecessor has a different registry identity.\n"
            f"Project number: {required_number}\n"
            f"Expected project: {required_project}"
        )

project_24_registry_rows_before = int(
    pnums.eq(
        PROJECT_NUMBER
    ).sum()
)

project_name_match_rows_diagnostic = int(
    reg_before[
        project_col
    ].eq(
        PROJECT_NAME
    ).sum()
)

if project_24_registry_rows_before != 0:
    raise RuntimeError(
        "Project 24 is already present in the completion registry by ProjectNumber."
    )

print(
    "Completion-registry ProjectNumber-24 rows:",
    project_24_registry_rows_before,
)

print(
    "Completion-registry exact project-name matches (diagnostic only):",
    project_name_match_rows_diagnostic,
)


# --------------------------------------------------------------------------------------------------
# 8B. FINAL DIRECT RAW-ROOT IMMUTABILITY VERIFICATION
# --------------------------------------------------------------------------------------------------

if not STEP5B_RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The independently frozen Step 5B raw-results manifest is missing:\n"
        f"{STEP5B_RAW_MANIFEST_PATH}"
    )

step5b_raw_manifest = pd.read_csv(
    STEP5B_RAW_MANIFEST_PATH,
    low_memory=False,
)

required_raw_manifest_columns = {
    "RelativePath",
    "Bytes",
    "SHA256",
}

if not required_raw_manifest_columns.issubset(
    step5b_raw_manifest.columns
):
    raise RuntimeError(
        "Step 5B raw manifest is missing required columns."
    )

step5b_raw_manifest = (
    step5b_raw_manifest[
        [
            "RelativePath",
            "Bytes",
            "SHA256",
        ]
    ]
    .copy()
)

step5b_raw_manifest[
    "RelativePath"
] = step5b_raw_manifest[
    "RelativePath"
].astype(
    str
)

step5b_raw_manifest[
    "Bytes"
] = pd.to_numeric(
    step5b_raw_manifest[
        "Bytes"
    ],
    errors="raise",
).astype(
    "int64"
)

step5b_raw_manifest[
    "SHA256"
] = (
    step5b_raw_manifest[
        "SHA256"
    ]
    .astype(
        str
    )
    .str.lower()
)

raw_file_failures = 0

for raw_row in step5b_raw_manifest.itertuples(
    index=False
):
    raw_path = (
        RAW_ROOT
        / str(
            raw_row.RelativePath
        )
    )

    if not raw_path.is_file():
        raw_file_failures += 1
        continue

    if int(
        raw_path.stat().st_size
    ) != int(
        raw_row.Bytes
    ):
        raw_file_failures += 1
        continue

    if sha(
        raw_path
    ) != str(
        raw_row.SHA256
    ):
        raw_file_failures += 1

if (
    len(
        step5b_raw_manifest
    )
    != COUNTS[
        "RawFiles"
    ]
    or int(
        step5b_raw_manifest[
            "Bytes"
        ].sum()
    )
    != COUNTS[
        "RawBytes"
    ]
    or root_hash(
        step5b_raw_manifest
    )
    != RAW_ROOT_SHA
    or raw_file_failures
    != 0
):
    raise RuntimeError(
        "Project 24 raw result root failed final pre-registration "
        "immutability verification."
    )


# --------------------------------------------------------------------------------------------------
# 9. FREEZE A PRE-PROJECT-24 REGISTRY BACKUP
# --------------------------------------------------------------------------------------------------

if not BACKUP_PATH.exists():
    shutil.copy2(
        REGISTRY,
        BACKUP_PATH,
    )

if sha(
    BACKUP_PATH
) != registry_sha_before:
    raise RuntimeError(
        "Pre-Project-24 registry backup does not match the live registry."
    )


# --------------------------------------------------------------------------------------------------
# 10. ASSEMBLE FINAL COMPACT PACKAGE
# --------------------------------------------------------------------------------------------------

sources = set(
    Path(
        path
    )
    for path in UPSTREAM_CPS
)

sources.update(
    WORKER_CHECKPOINT_PATHS.values()
)

sources.update(
    [
        STEP5A_STATUS_PATH,
        STEP5B_STATUS_PATH,
    ]
)

sources.update(
    Path(
        item[
            "Path"
        ]
    )
    for item in step5a.get(
        "AggregateOutputManifest",
        [],
    )
)

sources.update(
    Path(
        item[
            "Path"
        ]
    )
    for item in step5b.get(
        "OutputManifest",
        [],
    )
)

sources = sorted(
    sources,
    key=str,
)

missing_sources = [
    str(
        path
    )
    for path in sources
    if not path.is_file()
]

if missing_sources:
    raise FileNotFoundError(
        "Missing compact-package sources:\n"
        + "\n".join(
            missing_sources
        )
    )

created_at = str(
    step5b.get(
        "CompletedAtUTC",
        "",
    )
).strip()

if not created_at:
    raise RuntimeError(
        "The frozen Step 5B checkpoint contains no CompletedAtUTC timestamp."
    )

tmp_root = Path(
    tempfile.mkdtemp(
        prefix="project24_package_",
        dir="/content",
    )
)

drive_staging_root = None

try:
    for src in sources:
        try:
            rel = src.relative_to(
                ROOT
            )
        except ValueError as exc:
            raise RuntimeError(
                f"Package source is outside thesis root: {src}"
            ) from exc

        dst = (
            tmp_root
            / rel
        )

        dst.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            src,
            dst,
        )

    atomic_text(
        tmp_root
        / "README.txt",
        f"""PROJECT 24 FINAL COMPACT PACKAGE

Project number: {PROJECT_NUMBER}
Project: {PROJECT_NAME}
Project slug: {PROJECT_SLUG}
Status: {COMPLETE_STATUS}
Created at UTC: {created_at}

The 2,160 raw condition-output files are not duplicated here.
Raw results: {RAW_ROOT}
Raw-root SHA-256: {RAW_ROOT_SHA}
Source-root SHA-256: {SOURCE_ROOT_SHA}
Primary metric: APFDc
Secondary metric: APFD
Parallel Step 5A worker checkpoints preserved in this package: 6
""",
    )

    before_summary = manifest(
        tmp_root
    )

    atomic_json(
        tmp_root
        / "final_package_summary.json",
        {
            "ProjectNumber": PROJECT_NUMBER,
            "Project": PROJECT_NAME,
            "ProjectSlug": PROJECT_SLUG,
            "Status": COMPLETE_STATUS,
            "CreatedAtUTC": created_at,
            "SourceRootSHA256": SOURCE_ROOT_SHA,
            "RawRootSHA256": RAW_ROOT_SHA,
            **COUNTS,
            "Step5ACheckpointSHA256": step5a_sha,
            "Step5BCheckpointSHA256": step5b_sha,
            "PayloadRootSHA256BeforeSummary": root_hash(
                before_summary
            ),
            "RawResultsDuplicatedIntoPackage": False,
            "ParallelWorkerCheckpointSHA256":
                expected_worker_hashes_by_tag,
            "PriorProjectConditionOutputsAccessed": False,
            "PriorProjectConditionOutputsModified": False,
            "PriorProjectWriteAttempted": False,
        },
    )

    candidate_manifest = manifest(
        tmp_root,
        {
            "final_package_manifest.csv",
        },
    )

    package_root_sha = root_hash(
        candidate_manifest
    )

    atomic_csv(
        tmp_root
        / "final_package_manifest.csv",
        candidate_manifest,
    )

    package_files = (
        len(
            candidate_manifest
        )
        + 1
    )

    package_bytes = int(
        candidate_manifest[
            "Bytes"
        ].sum()
        + (
            tmp_root
            / "final_package_manifest.csv"
        ).stat().st_size
    )

    if FINAL_ROOT.exists():
        if not MANIFEST_PATH.is_file():
            raise RuntimeError(
                "An existing Project 24 final-package directory has no manifest "
                "and was not modified."
            )

        existing = pd.read_csv(
            MANIFEST_PATH,
            low_memory=False,
        )

        if root_hash(
            existing
        ) != package_root_sha:
            raise RuntimeError(
                "A different Project 24 final package already exists "
                "and was not modified."
            )

        shutil.rmtree(
            tmp_root
        )

        package_already_frozen = True

    else:
        # /content and Google Drive are different filesystems.
        # Publish in two stages:
        #   1. copy completed local package to a sibling Drive staging directory;
        #   2. verify the staged package exactly;
        #   3. rename staging -> FINAL_ROOT within Google Drive.
        FINAL_ROOT.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        drive_staging_root = FINAL_ROOT.with_name(
            f"{FINAL_ROOT.name}__staging_{os.getpid()}"
        )

        if drive_staging_root.exists():
            shutil.rmtree(
                drive_staging_root
            )

        shutil.copytree(
            tmp_root,
            drive_staging_root,
            copy_function=shutil.copy2,
        )

        staged_manifest_path = (
            drive_staging_root
            / "final_package_manifest.csv"
        )

        if not staged_manifest_path.is_file():
            raise RuntimeError(
                "The Google Drive staging package has no manifest."
            )

        staged_manifest = pd.read_csv(
            staged_manifest_path,
            low_memory=False,
        )

        staged_root_sha = root_hash(
            staged_manifest
        )

        if staged_root_sha != package_root_sha:
            raise RuntimeError(
                "The Google Drive staging package root SHA-256 differs.\n"
                f"Expected: {package_root_sha}\n"
                f"Actual:   {staged_root_sha}"
            )

        staged_payload_manifest = manifest(
            drive_staging_root,
            {
                "final_package_manifest.csv",
            },
        )

        candidate_payload_manifest = (
            candidate_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        staged_payload_manifest = (
            staged_payload_manifest.sort_values(
                "RelativePath",
                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )

        if not staged_payload_manifest.equals(
            candidate_payload_manifest
        ):
            comparison = candidate_payload_manifest.merge(
                staged_payload_manifest,
                on="RelativePath",
                how="outer",
                suffixes=(
                    "_candidate",
                    "_staged",
                ),
                indicator=True,
            )

            failed_comparison = comparison.loc[
                (
                    comparison[
                        "_merge"
                    ].ne(
                        "both"
                    )
                    | comparison[
                        "Bytes_candidate"
                    ].ne(
                        comparison[
                            "Bytes_staged"
                        ]
                    )
                    | comparison[
                        "SHA256_candidate"
                    ].ne(
                        comparison[
                            "SHA256_staged"
                        ]
                    )
                )
            ]

            raise RuntimeError(
                "The Google Drive staging package failed exact file-level "
                "verification.\n"
                + failed_comparison.head(
                    20
                ).to_string(
                    index=False
                )
            )

        os.replace(
            drive_staging_root,
            FINAL_ROOT,
        )

        drive_staging_root = None

        shutil.rmtree(
            tmp_root
        )

        package_already_frozen = False

except Exception:
    if tmp_root.exists():
        shutil.rmtree(
            tmp_root,
            ignore_errors=True,
        )

    if (
        drive_staging_root is not None
        and drive_staging_root.exists()
    ):
        shutil.rmtree(
            drive_staging_root,
            ignore_errors=True,
        )

    raise


# --------------------------------------------------------------------------------------------------
# 11. READ BACK EVERY FINAL-PACKAGE FILE
# --------------------------------------------------------------------------------------------------

pkg_manifest = pd.read_csv(
    MANIFEST_PATH,
    low_memory=False,
)

manifest_paths = set(
    pkg_manifest[
        "RelativePath"
    ].astype(
        str
    )
)

actual_paths = {
    path.relative_to(
        FINAL_ROOT
    ).as_posix()
    for path in FINAL_ROOT.rglob(
        "*"
    )
    if path.is_file()
}

expected_paths = (
    manifest_paths
    | {
        "final_package_manifest.csv",
    }
)

missing_pkg = len(
    expected_paths
    - actual_paths
)

unexpected_pkg = len(
    actual_paths
    - expected_paths
)

size_bad = 0
hash_bad = 0

for row in pkg_manifest.itertuples(
    index=False
):
    path = (
        FINAL_ROOT
        / str(
            row.RelativePath
        )
    )

    if path.is_file():
        size_bad += int(
            path.stat().st_size
            != int(
                row.Bytes
            )
        )

        hash_bad += int(
            sha(
                path
            )
            != str(
                row.SHA256
            )
        )

package_root_readback = root_hash(
    pkg_manifest
)

if (
    package_root_readback
    != package_root_sha
    or any(
        [
            missing_pkg,
            unexpected_pkg,
            size_bad,
            hash_bad,
        ]
    )
):
    raise RuntimeError(
        "Final Project 24 package failed readback validation."
    )


# --------------------------------------------------------------------------------------------------
# 12. BUILD A COMPLETE PROJECT 24 REGISTRY ROW
# --------------------------------------------------------------------------------------------------
#
# The registry schema has evolved across projects.
# Use Project 23's exact formatting for protocol fields whose textual
# representation may have varied historically, while filling Project 24's
# project-specific values explicitly.
# --------------------------------------------------------------------------------------------------

project_23_template_rows = reg_before.loc[
    pd.to_numeric(
        reg_before[
            pn_col
        ],
        errors="raise",
    ).astype(
        int
    ).eq(
        23
    )
]

if len(
    project_23_template_rows
) != 1:
    raise RuntimeError(
        "Could not resolve exactly one Project 23 registry template row."
    )

project_23_template = project_23_template_rows.iloc[
    0
]

protocol_template_values = {}

for registry_column in reg_before.columns:
    normalised_column = norm(
        registry_column
    )

    if normalised_column in {
        "seeds",
        "noiselevels",
        "techniques",
        "donotrerun",
    }:
        protocol_template_values[
            normalised_column
        ] = str(
            project_23_template[
                registry_column
            ]
        ).strip()

protocol_fallback_values = {
    "seeds":
        json.dumps(
            list(
                range(
                    1,
                    31,
                )
            ),
            separators=(
                ",",
                ":",
            ),
        ),

    "noiselevels":
        json.dumps(
            [
                0,
                5,
                10,
                15,
                20,
                25,
                30,
                40,
                50,
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "techniques":
        json.dumps(
            [
                "RandomForest",
                "XGBoost",
                "LightGBM",
                "NaiveBayes",
                "Random",
                "LatestFail",
                "QTF-Avg",
            ],
            separators=(
                ",",
                ":",
            ),
        ),

    "donotrerun":
        "True",
}

for (
    protocol_key,
    fallback_value,
) in protocol_fallback_values.items():
    if not protocol_template_values.get(
        protocol_key,
        "",
    ):
        protocol_template_values[
            protocol_key
        ] = fallback_value


values = {
    "projectnumber": PROJECT_NUMBER,
    "projectno": PROJECT_NUMBER,
    "project": PROJECT_NAME,
    "projectname": PROJECT_NAME,
    "projectslug": PROJECT_SLUG,
    "slug": PROJECT_SLUG,

    "status": COMPLETE_STATUS,
    "completionstatus": COMPLETE_STATUS,

    "completedatutc": created_at,
    "completedat": created_at,
    "frozenatutc": created_at,
    "frozenat": created_at,
    "registeredatutc": created_at,
    "registeredat": created_at,

    "sourcerootsha256": SOURCE_ROOT_SHA,

    "rawrootsha256": RAW_ROOT_SHA,
    "rawresultrootsha256": RAW_ROOT_SHA,
    "rawresultsrootsha256": RAW_ROOT_SHA,

    "rawroot": str(
        RAW_ROOT
    ),
    "rawresultroot": str(
        RAW_ROOT
    ),

    "finalpackagepath": str(
        FINAL_ROOT
    ),
    "packagepath": str(
        FINAL_ROOT
    ),
    "finalpackageroot": str(
        FINAL_ROOT
    ),

    "finalpackagerootsha256": package_root_sha,
    "packagerootsha256": package_root_sha,
    "packagesha256": package_root_sha,

    "packagefiles": package_files,
    "packagefilecount": package_files,
    "packagebytes": package_bytes,

    "step5acheckpointsha256": step5a_sha,
    "step5bcheckpointsha256": step5b_sha,

    # Complete observed registry schema.
    "seeds":
        protocol_template_values[
            "seeds"
        ],

    "noiselevels":
        protocol_template_values[
            "noiselevels"
        ],

    "techniques":
        protocol_template_values[
            "techniques"
        ],

    "evaluationrows":
        COUNTS[
            "ModelEvaluationRows"
        ],

    "evaluationfailures":
        COUNTS[
            "ModelEvaluationFailures"
        ],

    "finaldirectory":
        str(
            FINAL_ROOT
        ),

    "finalauditreport":
        str(
            REPORT_PATH
        ),

    "donotrerun":
        protocol_template_values[
            "donotrerun"
        ],

    "freezerecord":
        str(
            CHECKPOINT_PATH
        ),

    "rawresultsmanifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "finalpackagemanifest":
        str(
            MANIFEST_PATH
        ),

    "finalauditstatus":
        STEP5C_STATUS,
}

for (
    key,
    value,
) in COUNTS.items():
    values[
        norm(
            key
        )
    ] = value

values.update(
    {
        "rawfilecount":
            COUNTS[
                "RawFiles"
            ],

        "conditioncount":
            COUNTS[
                "Conditions"
            ],

        "modelfits":
            COUNTS[
                "MLFits"
            ],

        "modelreadyrows":
            COUNTS[
                "ModelRows"
            ],

        "predictorcount":
            COUNTS[
                "Predictors"
            ],

        "recfeaturecount":
            COUNTS[
                "RECFeatures"
            ],

        "rawexecutionrows":
            COUNTS[
                "RawRows"
            ],
    }
)

if not STEP5B_RAW_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "The independently frozen Step 5B raw-results manifest is missing:\n"
        f"{STEP5B_RAW_MANIFEST_PATH}"
    )

new_row = {}
unresolved = []

for col in reg_before.columns:
    normalised = norm(
        col
    )

    if normalised in values:
        new_row[
            col
        ] = str(
            values[
                normalised
            ]
        )

    else:
        unique_nonempty = sorted(
            set(
                value
                for value in reg_before[
                    col
                ].astype(
                    str
                ).str.strip()
                if value
            )
        )

        if len(
            unique_nonempty
        ) == 1:
            # Preserve a global protocol constant.
            new_row[
                col
            ] = unique_nonempty[
                0
            ]

        elif reg_before[
            col
        ].astype(
            str
        ).str.strip().eq(
            ""
        ).all():
            new_row[
                col
            ] = ""

        else:
            new_row[
                col
            ] = ""

            unresolved.append(
                col
            )

new_row[
    pn_col
] = str(
    PROJECT_NUMBER
)

new_row[
    project_col
] = PROJECT_NAME

new_row[
    status_col
] = COMPLETE_STATUS

if unresolved:
    raise RuntimeError(
        "Unexpected unmapped registry columns remain; "
        "no registry write was attempted:\n"
        + "\n".join(
            unresolved
        )
    )

reg_candidate = pd.concat(
    [
        reg_before,
        pd.DataFrame(
            [
                new_row
            ]
        ),
    ],
    ignore_index=True,
)

reg_candidate[
    pn_col
] = pd.to_numeric(
    reg_candidate[
        pn_col
    ],
    errors="raise",
).astype(
    int
).astype(
    str
)

candidate_nums = pd.to_numeric(
    reg_candidate[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

project24_candidate = reg_candidate.loc[
    candidate_nums.eq(
        24
    )
]

if (
    len(
        reg_candidate
    )
    != 24
    or sorted(
        candidate_nums.tolist()
    )
    != list(
        range(
            1,
            25,
        )
    )
):
    raise RuntimeError(
        "Candidate registry does not contain exactly Projects 1–24."
    )

if (
    not reg_candidate[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or len(
        project24_candidate
    )
    != 1
    or project24_candidate.iloc[
        0
    ][
        project_col
    ]
    != PROJECT_NAME
):
    raise RuntimeError(
        "Candidate Project 24 registry row failed status/identity validation."
    )


# --------------------------------------------------------------------------------------------------
# 13. PRE-WRITE VALIDATION
# --------------------------------------------------------------------------------------------------

rows = []

check(
    rows,
    "Step 5A checkpoint SHA-256",
    STEP5A_SHA_EXPECTED,
    step5a_sha,
    step5a_sha
    == STEP5A_SHA_EXPECTED,
)

check(
    rows,
    "Step 5B checkpoint SHA-256",
    STEP5B_SHA_EXPECTED,
    step5b_sha,
    step5b_sha
    == STEP5B_SHA_EXPECTED,
)

check(
    rows,
    "Source root SHA-256",
    SOURCE_ROOT_SHA,
    step5a.get(
        "SourceRootSHA256"
    ),
    step5a.get(
        "SourceRootSHA256"
    )
    == SOURCE_ROOT_SHA,
)

check(
    rows,
    "Raw root SHA-256",
    RAW_ROOT_SHA,
    step5b.get(
        "IndependentRawRootSHA256"
    ),
    step5b.get(
        "IndependentRawRootSHA256"
    )
    == RAW_ROOT_SHA,
)

check(
    rows,
    "Step 5A manifest failures",
    0,
    int(
        (
            ~step5a_audit[
                "Pass"
            ]
        ).sum()
    ),
    step5a_audit[
        "Pass"
    ].all(),
)

check(
    rows,
    "Step 5B manifest failures",
    0,
    int(
        (
            ~step5b_audit[
                "Pass"
            ]
        ).sum()
    ),
    step5b_audit[
        "Pass"
    ].all(),
)

check(
    rows,
    "Parallel worker checkpoint SHA mismatches",
    0,
    worker_checkpoint_sha_mismatches,
    worker_checkpoint_sha_mismatches
    == 0,
)

check(
    rows,
    "Parallel worker checkpoints",
    6,
    len(
        expected_worker_hashes_by_tag
    ),
    len(
        expected_worker_hashes_by_tag
    )
    == 6,
)

check(
    rows,
    "Parallel worker scientific-freeze failures",
    0,
    worker_scientific_freeze_failures,
    worker_scientific_freeze_failures
    == 0,
)

check(
    rows,
    "Step 5B prior-project output access",
    False,
    bool(
        step5b.get(
            "PriorProjectConditionOutputsAccessed",
            True,
        )
    ),
    not bool(
        step5b.get(
            "PriorProjectConditionOutputsAccessed",
            True,
        )
    ),
)

check(
    rows,
    "Step 5B prior-project output modification",
    False,
    bool(
        step5b.get(
            "PriorProjectConditionOutputsModified",
            True,
        )
    ),
    not bool(
        step5b.get(
            "PriorProjectConditionOutputsModified",
            True,
        )
    ),
)

check(
    rows,
    "Raw result file verification failures",
    0,
    raw_file_failures,
    raw_file_failures
    == 0,
)

check(
    rows,
    "Package missing files",
    0,
    missing_pkg,
    missing_pkg
    == 0,
)

check(
    rows,
    "Package unexpected files",
    0,
    unexpected_pkg,
    unexpected_pkg
    == 0,
)

check(
    rows,
    "Package size mismatches",
    0,
    size_bad,
    size_bad
    == 0,
)

check(
    rows,
    "Package SHA-256 mismatches",
    0,
    hash_bad,
    hash_bad
    == 0,
)

check(
    rows,
    "Registry rows before",
    23,
    len(
        reg_before
    ),
    len(
        reg_before
    )
    == 23,
)

check(
    rows,
    "Registry rows candidate",
    24,
    len(
        reg_candidate
    ),
    len(
        reg_candidate
    )
    == 24,
)

check(
    rows,
    "Candidate Project 24 rows",
    1,
    len(
        project24_candidate
    ),
    len(
        project24_candidate
    )
    == 1,
)

check(
    rows,
    "Unresolved variable registry columns",
    0,
    len(
        unresolved
    ),
    len(
        unresolved
    )
    == 0,
)

required_registry_field_expectations = {
    "Seeds":
        protocol_template_values[
            "seeds"
        ],

    "NoiseLevels":
        protocol_template_values[
            "noiselevels"
        ],

    "Techniques":
        protocol_template_values[
            "techniques"
        ],

    "EvaluationRows":
        str(
            COUNTS[
                "ModelEvaluationRows"
            ]
        ),

    "EvaluationFailures":
        str(
            COUNTS[
                "ModelEvaluationFailures"
            ]
        ),

    "FinalDirectory":
        str(
            FINAL_ROOT
        ),

    "FinalAuditReport":
        str(
            REPORT_PATH
        ),

    "DoNotRerun":
        protocol_template_values[
            "donotrerun"
        ],

    "FreezeRecord":
        str(
            CHECKPOINT_PATH
        ),

    "RawResultsManifest":
        str(
            STEP5B_RAW_MANIFEST_PATH
        ),

    "FinalPackageManifest":
        str(
            MANIFEST_PATH
        ),

    "RawResultsRootSHA256":
        RAW_ROOT_SHA,

    "FinalAuditStatus":
        STEP5C_STATUS,
}

registry_field_validation_failures = 0

for (
    expected_column_name,
    expected_value,
) in required_registry_field_expectations.items():
    matching_columns = [
        column
        for column in reg_before.columns
        if norm(
            column
        )
        == norm(
            expected_column_name
        )
    ]

    if len(
        matching_columns
    ) != 1:
        registry_field_validation_failures += 1
        continue

    actual_value = str(
        project24_candidate.iloc[
            0
        ][
            matching_columns[
                0
            ]
        ]
    )

    registry_field_validation_failures += int(
        actual_value
        != str(
            expected_value
        )
    )

check(
    rows,
    "Explicit Project 24 registry-field failures",
    0,
    registry_field_validation_failures,
    registry_field_validation_failures
    == 0,
)

pre = pd.DataFrame(
    rows
)

print(
    "\nProject 24 Step 5C pre-write validation:"
)

display(
    pre
)

print(
    "\nProject 24 registry row candidate:"
)

display(
    project24_candidate
)

if not pre[
    "Pass"
].all():
    raise RuntimeError(
        "PROJECT 24 STEP 5C PRE-WRITE VALIDATION FAILED. "
        "Registry not modified."
    )


# --------------------------------------------------------------------------------------------------
# 14. ATOMIC COMPLETION-REGISTRY WRITE
# --------------------------------------------------------------------------------------------------

tmp_reg = REGISTRY.with_name(
    f".{REGISTRY.name}.project24_{os.getpid()}"
)

reg_candidate.to_csv(
    tmp_reg,
    index=False,
    lineterminator="\n",
)

tmp_read = pd.read_csv(
    tmp_reg,
    dtype=str,
).fillna(
    ""
)

tmp_nums = pd.to_numeric(
    tmp_read[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

if (
    len(
        tmp_read
    )
    != 24
    or sorted(
        tmp_nums.tolist()
    )
    != list(
        range(
            1,
            25,
        )
    )
    or not tmp_read[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or int(
        tmp_nums.eq(
            24
        ).sum()
    )
    != 1
):
    tmp_reg.unlink(
        missing_ok=True
    )

    raise RuntimeError(
        "Temporary Project 24 registry failed readback; "
        "live registry unchanged."
    )

os.replace(
    tmp_reg,
    REGISTRY,
)


# --------------------------------------------------------------------------------------------------
# 15. POST-WRITE REGISTRY VALIDATION
# --------------------------------------------------------------------------------------------------

reg_after = pd.read_csv(
    REGISTRY,
    dtype=str,
).fillna(
    ""
)

after_nums = pd.to_numeric(
    reg_after[
        pn_col
    ],
    errors="raise",
).astype(
    int
)

project24_after = reg_after.loc[
    after_nums.eq(
        24
    )
]

registry_sha_after = sha(
    REGISTRY
)

if (
    len(
        reg_after
    )
    != 24
    or sorted(
        after_nums.tolist()
    )
    != list(
        range(
            1,
            25,
        )
    )
    or not reg_after[
        status_col
    ].eq(
        COMPLETE_STATUS
    ).all()
    or len(
        project24_after
    )
    != 1
    or project24_after.iloc[
        0
    ][
        project_col
    ]
    != PROJECT_NAME
):
    raise RuntimeError(
        "Live registry failed post-write validation.\n"
        f"Backup: {BACKUP_PATH}"
    )

for (
    required_number,
    required_project,
) in required_registered_identities.items():
    matching_rows = reg_after.loc[
        after_nums.eq(
            required_number
        )
    ]

    if (
        len(
            matching_rows
        )
        != 1
        or matching_rows.iloc[
            0
        ][
            project_col
        ]
        != required_project
    ):
        raise RuntimeError(
            "A predecessor registry identity changed after "
            "Project 24 registration."
        )

check(
    rows,
    "Registry rows after",
    24,
    len(
        reg_after
    ),
    len(
        reg_after
    )
    == 24,
)

check(
    rows,
    "COMPLETE_AND_FROZEN projects after",
    24,
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    ),
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    )
    == 24,
)

check(
    rows,
    "Registry Project 24 rows after",
    1,
    len(
        project24_after
    ),
    len(
        project24_after
    )
    == 1,
)

check(
    rows,
    "Registry SHA changed",
    True,
    registry_sha_after
    != registry_sha_before,
    registry_sha_after
    != registry_sha_before,
)

for number in range(
    11,
    24,
):
    check(
        rows,
        f"Registry Project {number} rows after",
        1,
        int(
            after_nums.eq(
                number
            ).sum()
        ),
        int(
            after_nums.eq(
                number
            ).sum()
        )
        == 1,
    )

validation = pd.DataFrame(
    rows
)

failed = validation.loc[
    ~validation[
        "Pass"
    ]
]

if not failed.empty:
    display(
        failed
    )

    raise RuntimeError(
        "PROJECT 24 STEP 5C POST-WRITE VALIDATION FAILED."
    )


# --------------------------------------------------------------------------------------------------
# 16. WRITE STEP 5C AUDIT / CHECKPOINT / STATUS
# --------------------------------------------------------------------------------------------------

STEP5C_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

atomic_csv(
    VALIDATION_PATH,
    validation,
)

registry_rows_after_by_project = {
    f"Project{number}RegistryRowsAfter":
        int(
            after_nums.eq(
                number
            ).sum()
        )
    for number in range(
        11,
        25,
    )
}

report = {
    "ProjectNumber": PROJECT_NUMBER,
    "Project": PROJECT_NAME,
    "ProjectSlug": PROJECT_SLUG,
    "Status": STEP5C_STATUS,
    "CompletedAtUTC": created_at,

    "SourceRootSHA256": SOURCE_ROOT_SHA,
    "RawRootSHA256": RAW_ROOT_SHA,

    **COUNTS,

    "FinalPackageRoot": str(
        FINAL_ROOT
    ),
    "FinalPackageFiles": package_files,
    "FinalPackageBytes": package_bytes,
    "FinalPackageRootSHA256": package_root_sha,
    "PackageAlreadyFrozenBeforeThisCell":
        package_already_frozen,

    "PackageMissingFiles": missing_pkg,
    "PackageUnexpectedFiles": unexpected_pkg,
    "PackageSizeMismatches": size_bad,
    "PackageSHA256Mismatches": hash_bad,

    "RegistrySHA256Before": registry_sha_before,
    "RegistrySHA256After": registry_sha_after,
    "RegistryRowsBefore": len(
        reg_before
    ),
    "RegistryRowsAfter": len(
        reg_after
    ),

    **registry_rows_after_by_project,

    "RegistryBackup": str(
        BACKUP_PATH
    ),

    "Step5ACheckpointSHA256": step5a_sha,
    "Step5BCheckpointSHA256": step5b_sha,

    "ParallelWorkerCheckpointSHA256":
        expected_worker_hashes_by_tag,

    "ValidationChecks": len(
        validation
    ),
    "FailedValidationChecks": len(
        failed
    ),

    "ConditionsRerun": False,
    "ModelsFitted": False,
    "RawResultsModified": False,

    "PriorProjectConditionOutputsAccessed": False,
    "PriorProjectConditionOutputsModified": False,
    "PriorProjectWriteAttempted": False,
}

atomic_json(
    REPORT_PATH,
    report,
)

atomic_json(
    CHECKPOINT_PATH,
    {
        **report,
        "CheckpointVersion": 1,
        "CheckpointType":
            "PROJECT_24_FINAL_PACKAGE_AND_REGISTRY",
        "FinalPackageFrozen": True,
        "CompletionRegistryUpdated": True,
        "ProjectCompleteAndFrozen": True,
        "NextRequiredStep":
            "PROJECT_25_MAY_START_IN_A_NEW_NOTEBOOK",
    },
)

step5c_sha = sha(
    CHECKPOINT_PATH
)

atomic_json(
    STATUS_PATH,
    {
        "ProjectNumber": PROJECT_NUMBER,
        "Project": PROJECT_NAME,
        "ProjectSlug": PROJECT_SLUG,
        "Status": STEP5C_STATUS,
        "CompletedAtUTC": created_at,

        "FinalPackageRoot": str(
            FINAL_ROOT
        ),
        "FinalPackageRootSHA256":
            package_root_sha,

        "RegistryRows": len(
            reg_after
        ),
        "RegistrySHA256":
            registry_sha_after,

        "Checkpoint": str(
            CHECKPOINT_PATH
        ),
        "CheckpointSHA256":
            step5c_sha,

        "ProjectCompleteAndFrozen": True,

        "ParallelWorkerCheckpointSHA256":
            expected_worker_hashes_by_tag,

        "PriorProjectConditionOutputsAccessed": False,
        "PriorProjectConditionOutputsModified": False,

        "NextRequiredStep":
            "PROJECT_25_MAY_START_IN_A_NEW_NOTEBOOK",
    },
)


# --------------------------------------------------------------------------------------------------
# 17. FINAL READBACK AND IMMUTABILITY PROOF
# --------------------------------------------------------------------------------------------------

checkpoint_readback = load_json(
    CHECKPOINT_PATH
)

status_readback = load_json(
    STATUS_PATH
)

if (
    checkpoint_readback.get(
        "Status"
    )
    != STEP5C_STATUS
    or status_readback.get(
        "Status"
    )
    != STEP5C_STATUS
):
    raise RuntimeError(
        "Project 24 Step 5C checkpoint/status readback failed."
    )

if not bool(
    checkpoint_readback.get(
        "ProjectCompleteAndFrozen",
        False,
    )
):
    raise RuntimeError(
        "Project 24 checkpoint is not marked complete and frozen."
    )

if not bool(
    checkpoint_readback.get(
        "CompletionRegistryUpdated",
        False,
    )
):
    raise RuntimeError(
        "Project 24 checkpoint is not marked registry-updated."
    )

if not bool(
    checkpoint_readback.get(
        "FinalPackageFrozen",
        False,
    )
):
    raise RuntimeError(
        "Project 24 checkpoint is not marked final-package frozen."
    )

if sha(
    REGISTRY
) != registry_sha_after:
    raise RuntimeError(
        "Completion registry changed after Project 24 finalisation."
    )

if root_hash(
    pd.read_csv(
        MANIFEST_PATH,
        low_memory=False,
    )
) != package_root_sha:
    raise RuntimeError(
        "Final package changed after Project 24 finalisation."
    )

if sha(
    STEP5A_CP
) != STEP5A_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5A checkpoint changed during Project 24 Step 5C."
    )

if sha(
    STEP5B_CP
) != STEP5B_SHA_EXPECTED:
    raise RuntimeError(
        "Step 5B checkpoint changed during Project 24 Step 5C."
    )

for tag, worker_path in WORKER_CHECKPOINT_PATHS.items():
    if sha(
        worker_path
    ) != str(
        expected_worker_hashes_by_tag[
            tag
        ]
    ):
        raise RuntimeError(
            "A frozen Project 24 worker checkpoint changed "
            "during Step 5C."
        )


# --------------------------------------------------------------------------------------------------
# 18. RESULT
# --------------------------------------------------------------------------------------------------

print(
    "\n"
    + "=" * 136
)

print(
    "=== PROJECT 24 CELL 11 / STEP 5C RESULT ==="
)

print(
    "=" * 136
)

print(
    "Project number:",
    PROJECT_NUMBER,
)

print(
    "Project:",
    PROJECT_NAME,
)

print(
    "Project slug:",
    PROJECT_SLUG,
)

print(
    "\nRaw result freeze:"
)

print(
    "Conditions:",
    COUNTS[
        "Conditions"
    ],
)

print(
    "ML fits:",
    COUNTS[
        "MLFits"
    ],
)

print(
    "Raw files:",
    COUNTS[
        "RawFiles"
    ],
)

print(
    "Raw bytes:",
    COUNTS[
        "RawBytes"
    ],
)

print(
    "Raw root SHA-256:",
    RAW_ROOT_SHA,
)

print(
    "\nParallel execution freeze:"
)

print(
    "Worker checkpoints:",
    len(
        expected_worker_hashes_by_tag
    ),
)

print(
    "Worker checkpoint SHA mismatches:",
    worker_checkpoint_sha_mismatches,
)

print(
    "\nFinal package freeze:"
)

print(
    "Package root:",
    FINAL_ROOT,
)

print(
    "Package files:",
    package_files,
)

print(
    "Package bytes:",
    package_bytes,
)

print(
    "Missing package files:",
    missing_pkg,
)

print(
    "Unexpected package files:",
    unexpected_pkg,
)

print(
    "Package size mismatches:",
    size_bad,
)

print(
    "Package SHA-256 mismatches:",
    hash_bad,
)

print(
    "Final package root SHA-256:",
    package_root_sha,
)

print(
    "\nCompletion registry:"
)

print(
    "Registry rows:",
    len(
        reg_after
    ),
)

print(
    "COMPLETE_AND_FROZEN projects:",
    int(
        reg_after[
            status_col
        ].eq(
            COMPLETE_STATUS
        ).sum()
    ),
)

print(
    "Project 24 registry rows:",
    len(
        project24_after
    ),
)

print(
    "Registry SHA-256 before:",
    registry_sha_before,
)

print(
    "Registry SHA-256 after:",
    registry_sha_after,
)

print(
    "Package already frozen before this cell:",
    package_already_frozen,
)

print(
    "\nStep 5C freeze:"
)

print(
    "Validation checks:",
    len(
        validation
    ),
)

print(
    "Failed validation checks:",
    len(
        failed
    ),
)

print(
    "Step 5C checkpoint:",
    CHECKPOINT_PATH,
)

print(
    "Step 5C checkpoint SHA-256:",
    step5c_sha,
)

print(
    "Project complete and frozen:",
    True,
)

print(
    "Next required step:",
    "PROJECT 24 MAY START IN A NEW NOTEBOOK",
)

print(
    "STATUS:",
    STEP5C_STATUS,
)

print(
    "=" * 136
)


=== PROJECT 24 CELL 11 / STEP 5C: FINAL PACKAGE FREEZE AND REGISTRY REGISTRATION ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Completion-registry ProjectNumber-24 rows: 0
Completion-registry exact project-name matches (diagnostic only): 0

Project 24 Step 5C pre-write validation:


,Check,Expected,Actual,Pass
0,Step 5A checkpoint SHA-256,d7abb32056cc47a4b304794dcc2d59477a18c7d42effda...,d7abb32056cc47a4b304794dcc2d59477a18c7d42effda...,True
1,Step 5B checkpoint SHA-256,3117c9637893b030d67ef4e7743d08a37f0787ec7037cb...,3117c9637893b030d67ef4e7743d08a37f0787ec7037cb...,True
2,Source root SHA-256,2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21...,2259d700ed4f3fae1e8dfc27da4ce44e198e02d0eeca21...,True
3,Raw root SHA-256,6968c094a564405fe23e02f9a5b1ed01cce130ea000dd3...,6968c094a564405fe23e02f9a5b1ed01cce130ea000dd3...,True
4,Step 5A manifest failures,0,0,True
5,Step 5B manifest failures,0,0,True
6,Parallel worker checkpoint SHA mismatches,0,0,True
7,Parallel worker checkpoints,6,6,True
8,Parallel worker scientific-freeze failures,0,0,True
9,Step 5B prior-project output access,False,False,True



Project 24 registry row candidate:


,ProjectNumber,Project,ProjectSlug,Status,Conditions,Seeds,NoiseLevels,Techniques,EvaluationBuilds,EvaluationRows,...,FreezeRecord,ChecksumManifest,LastFreezeValidationAtUTC,RawResultsManifest,FinalPackageManifest,RawResultsRootSHA256,FinalPackageRootSHA256,ModelFits,ManifestRowsAudited,FinalAuditStatus
23,24,SonarSource@sonarqube,SonarSource__sonarqube,COMPLETE_AND_FROZEN,270,30,9,7,1072,18854,...,/content/drive/MyDrive/Thesis_Experiment/Notes...,/content/drive/MyDrive/Thesis_Experiment/Resul...,2026-07-25T03:49:02.436302+00:00,/content/drive/MyDrive/Thesis_Experiment/Resul...,/content/drive/MyDrive/Thesis_Experiment/Resul...,6968c094a564405fe23e02f9a5b1ed01cce130ea000dd3...,ed2e84213389ad8102554ab0e68740a8d893b8b4281729...,1080,5358150.0,PASS_PROJECT_24_FINAL_PACKAGE_FROZEN_AND_REGIS...



=== PROJECT 24 CELL 11 / STEP 5C RESULT ===
Project number: 24
Project: SonarSource@sonarqube
Project slug: SonarSource__sonarqube

Raw result freeze:
Conditions: 270
ML fits: 1080
Raw files: 2160
Raw bytes: 523093565
Raw root SHA-256: 6968c094a564405fe23e02f9a5b1ed01cce130ea000dd387f7a3f6f12b421d09

Parallel execution freeze:
Worker checkpoints: 6
Worker checkpoint SHA mismatches: 0

Final package freeze:
Package root: /content/drive/MyDrive/Thesis_Experiment/Results/Final/SonarSource__sonarqube
Package files: 39
Package bytes: 16434698
Missing package files: 0
Unexpected package files: 0
Package size mismatches: 0
Package SHA-256 mismatches: 0
Final package root SHA-256: ed2e84213389ad8102554ab0e68740a8d893b8b4281729525d82210a25b12cfc

Completion registry:
Registry rows: 24
COMPLETE_AND_FROZEN projects: 24
Project 24 registry rows: 1
Registry SHA-256 before: c67aad9d6e6a34cdc06d75adf82c1b25a69e751e0d179db29d21e8c5598f9274
Registry SHA-256 after: dc5cdc752d89661c0b41adc5680509034ded1